# Contact LP Continuation Lab

**Verified local score: 22.249882014187.** Both the exact exported file and an ordinary pandas read/write pass the unchanged V20 metric with zero rejected packings. The previous official competition result is V29, 22.2498853779. This version's score is a local measurement until its own submission completes.

**What changed.** Select 65 whole layouts from the fixed public portfolio, guard each with positive clearance, and continue n=991 on CPU. The inherited reduction is 3.321726276e-06. Three stages of our CPU search contribute a further 1.756441881e-09. The final exact CSV beats the independently checked Dataset V2 raw benchmark by 6.799112384e-10.

**Run All.** Turn on Internet, keep `BACKEND="cpu"` and leave the optional switches off. The notebook verifies both full input hashes, restores the exact layouts, renders the audit, and checks the CPU plus both complete CSV representations. The default run is deterministic. Timed optional controls depend on the CPU and solver.

The final CSV SHA-256 is `b412097a8264bb2ae4d9b48dbce24fcfb30814ba85e82ddd25b2df5aa296fe5e`. Adapted data follow [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). See DATA_LICENSE.md and SOURCE_NOTICES.md.


## 1. Choose a device

Default replay does not launch an expensive search. To experiment, set
`RUN_SEARCH=True`: tensor proposals use your chosen device, while LP contact repair
and official certification always run on CPU. An unavailable requested device raises
an explicit error instead of silently switching devices.

| Backend | Intended environment | Hardware evidence |
|---|---|---|
| `cpu` | Mac, Linux, Kaggle CPU | Fresh full replay, geometry gates and CPU optimizer/export check |
| `cuda` | NVIDIA GPU with PyTorch | Prior real T4 smoke test of the same portable algorithm; no fresh full GPU benchmark |
| `mps` | Apple silicon with PyTorch | Implemented, but hardware-untested in this work |

Kaggle publication should run on CPU. Local MPS is reserved for another task.

In [ ]:
import base64
import csv
import gzip
import hashlib
import json
import lzma
import math
import os
import shutil
import subprocess
import sys
from pathlib import Path

for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[variable] = "1"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

BACKEND = "cpu"
RUN_SEARCH = False
REPLAY_MOVEMENT_STUDY = False
SEARCH_SIZES = [910, 991, 994]
SEARCH_STEPS = 240
SEARCH_BATCH = 6
ROOT = Path.cwd()
CACHE = ROOT / "_closure_inputs"
CACHE.mkdir(exist_ok=True)
Path("reference_sources").mkdir(exist_ok=True)
assert BACKEND in {"cpu", "cuda", "mps"}
EXPECTED_SCORE = 22.249882014187367
EXPECTED_CSV_SHA256 = "b412097a8264bb2ae4d9b48dbce24fcfb30814ba85e82ddd25b2df5aa296fe5e"
BASELINE_SOURCE = {
    "id": 140109525,
    "author": "kirilltym",
    "slug": "22-2499-cpu-cuda-mps-eda-budget-lp",
    "version": 30,
    "native_version": 366028783,
    "source_sha256": "cda60d87a329c6fc8797b37e387a43b1cca61322004979d25db16026bb4a1e08",
    "csv_sha256": "e0c9104aae54695732a93d53c8e857bb5992888a7728a6eb6bd7e21e7a1bbe3a",
}
OFFICIAL_METRIC_SHA256 = (
    "d269f24b02d5f33fbd2149f76b83414b3b446be4d02776560e04631af3365ccf"
)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(262144), b""):
            digest.update(block)
    return digest.hexdigest()


print(f"Backend: {BACKEND}; optional search: {RUN_SEARCH}; sequential certification")


## 2. Choose whole layouts and continue the tightest case

A public layout enters only as a complete packing. For each selected size, re-encode it with at least the positive 2e-12 clearance and verify both exact decimal rows and an ordinary pandas round trip. The fixed public source has a valid exact raw score. Its ordinary round trip rejects 19 sizes, so certify each adopted layout rather than assuming the source's decimals will survive parsing.

On CPU, linearize the contact gaps and enclosing sides in a trust region. First compare a small L1 movement penalty with the unpenalized LP from identical starts. Continue the one improving case, n=991, from the saved guarded result and compare trust radii and variable scaling. A final longer budget starts from the time-limited winner.


### `packing_geometry.py`

Rigid-square geometry and decimal CSV loading. Square dimensions are fixed.

In [ ]:
%%writefile packing_geometry.py
"""CPU-only geometry and file utilities for the Squares in Squares experiment."""

import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
import math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree, ConvexHull
from scipy.optimize import minimize_scalar


def grid(n):
    k = math.ceil(math.sqrt(n))
    return np.array([(i % k + 0.5, i // k + 0.5, 0.0) for i in range(n)])


def aabbs(p):
    h = (np.abs(np.cos(p[:, 2])) + np.abs(np.sin(p[:, 2]))) / 2
    return p[:, :2] - h[:, None], p[:, :2] + h[:, None]


def side(p):
    lo, hi = aabbs(p)
    return float(np.max(hi.max(0) - lo.min(0)))


def normalize(p):
    p = np.array(p, dtype=float, copy=True)
    p[:, :2] -= aabbs(p)[0].min(0)
    return p


def rotate(p, angle):
    p = p.copy()
    c, s = math.cos(angle), math.sin(angle)
    p[:, :2] = p[:, :2] @ np.array([[c, s], [-s, c]])
    p[:, 2] += angle
    return normalize(p)


def vertices(p):
    u = np.array([[-0.5, -0.5], [0.5, -0.5], [0.5, 0.5], [-0.5, 0.5]])
    c, s = np.cos(p[:, 2]), np.sin(p[:, 2])
    return p[:, None, :2] + np.stack(
        (
            c[:, None] * u[:, 0] - s[:, None] * u[:, 1],
            s[:, None] * u[:, 0] + c[:, None] * u[:, 1],
        ),
        2,
    )


def pair_data(p, radius=1.55):
    pairs = cKDTree(p[:, :2]).query_pairs(radius, output_type="ndarray")
    if not len(pairs):
        return pairs, np.empty((0, 4)), np.empty(0)
    i, j = pairs.T
    d = p[j, :2] - p[i, :2]
    a, b = p[i, 2], p[j, 2]
    projections = np.column_stack(
        (
            d[:, 0] * np.cos(a) + d[:, 1] * np.sin(a),
            -d[:, 0] * np.sin(a) + d[:, 1] * np.cos(a),
            d[:, 0] * np.cos(b) + d[:, 1] * np.sin(b),
            -d[:, 0] * np.sin(b) + d[:, 1] * np.cos(b),
        )
    )
    support = (1 + np.abs(np.cos(a - b)) + np.abs(np.sin(a - b))) / 2
    return pairs, projections, support


def min_gap(p):
    _, projection, support = pair_data(p)
    return (
        float(np.min(np.abs(projection).max(1) - support)) if len(support) else math.inf
    )


def repair(p, clearance=3e-10):
    """Separate centres slightly without ever changing unit-square dimensions."""
    p = normalize(p)
    _, projection, support = pair_data(p)
    if len(support):
        ratio = float(
            np.max((support + clearance) / np.maximum(np.abs(projection).max(1), 1e-30))
        )
        if ratio > 1.005:
            return None
        p[:, :2] *= max(1.0, ratio)
    return normalize(p)


def load_bank(path="inputs/submission.csv"):
    data = pd.read_csv(path, float_precision="round_trip")
    bank = {n: grid(n) for n in range(1, 1001)}
    for n, group in data.groupby("n"):
        n = int(n)
        assert len(group) == n
        bank[n] = group.sort_values("square_id")[["x", "y", "angle"]].to_numpy(float)
    return bank


def save_bank(path, bank):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **{str(n): p for n, p in bank.items()})


def total_score(bank):
    return float(sum(side(bank[n]) / math.sqrt(n) - 1 for n in range(1, 1001)))


def optimal_rotation(p):
    """Minimize the enclosing square of the fixed full layout over global angle."""
    points = vertices(p).reshape(-1, 2)
    hull = points[ConvexHull(points).vertices]
    angles = np.linspace(-math.pi / 4, math.pi / 4, 721)
    c, s = np.cos(angles), np.sin(angles)
    xx = hull[:, 0, None] * c - hull[:, 1, None] * s
    yy = hull[:, 0, None] * s + hull[:, 1, None] * c
    lengths = np.maximum(np.ptp(xx, axis=0), np.ptp(yy, axis=0))
    indices = (
        np.flatnonzero((lengths[1:-1] <= lengths[:-2]) & (lengths[1:-1] <= lengths[2:]))
        + 1
    )
    winner = (side(p), 0.0)

    def objective(a):
        c, s = math.cos(a), math.sin(a)
        return max(
            np.ptp(hull[:, 0] * c - hull[:, 1] * s),
            np.ptp(hull[:, 0] * s + hull[:, 1] * c),
        )

    for k in indices:
        fit = minimize_scalar(
            objective,
            bounds=(angles[k - 1], angles[k + 1]),
            method="bounded",
            options={"xatol": 1e-14},
        )
        if fit.fun < winner[0]:
            winner = (float(fit.fun), float(fit.x))
    return rotate(p, winner[1]), winner[1]


### `cpu_axis_reclosure.py`

The visible LP now accepts an explicit clearance and side-gain threshold. Historical
defaults remain unchanged. The new small-margin variants are tested on CPU only.


In [ ]:
%%writefile cpu_axis_reclosure.py
"""CPU fixed-angle LP with randomized valid separation-axis choices.

The optional clearance sets a separating-axis gap; it never changes unit sides.
The historical defaults retain the previous algorithm behavior.
Only square centres move. All admitted layouts retain exact unit-square sizes.
The radius covers every pair that could interact under bounded centre moves.
"""

import os

for key in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(key, "1")
import time, math
import numpy as np
from scipy.optimize import linprog
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from packing_geometry import normalize, side, repair


def reclose(
    p,
    seed=0,
    trust=0.22,
    tie=0.015,
    rounds=6,
    seconds=12,
    clearance=3e-10,
    minimum_side_gain=1e-8,
):
    p = normalize(p)
    rng = np.random.default_rng(seed)
    start = time.perf_counter()
    history = []
    n = len(p)
    for it in range(rounds):
        if time.perf_counter() - start > seconds:
            break
        L = side(p)
        a = p[:, 2]
        c = np.cos(a)
        s = np.sin(a)
        h = (abs(c) + abs(s)) / 2
        radius = math.sqrt(2) + 2 * math.sqrt(2) * trust + 1e-4
        pairs = cKDTree(p[:, :2]).query_pairs(radius, output_type="ndarray")
        i, j = pairs.T
        d = p[j, :2] - p[i, :2]
        ux = np.column_stack((c[i], -s[i], c[j], -s[j]))
        uy = np.column_stack((s[i], c[i], s[j], c[j]))
        pr = d[:, 0, None] * ux + d[:, 1, None] * uy
        support = (1 + abs(np.cos(a[i] - a[j])) + abs(np.sin(a[i] - a[j]))) / 2
        rank = abs(pr)
        valid = (rank >= support[:, None] - 1e-9) & (rank >= rank.max(1)[:, None] - tie)
        random = rng.random(rank.shape)
        random[~valid] = -1
        chosen = random.argmax(1)
        bad = ~valid.any(1)
        chosen[bad] = rank[bad].argmax(1)
        rr = np.arange(len(i))
        sgn = np.sign(pr[rr, chosen])
        vx = sgn * ux[rr, chosen]
        vy = sgn * uy[rr, chosen]
        dist = abs(pr[rr, chosen])
        cols = [np.column_stack((2 * i, 2 * i + 1, 2 * j, 2 * j + 1))]
        vals = [np.column_stack((vx, vy, -vx, -vy))]
        rhs = [dist - support - clearance]
        rows = [np.repeat(np.arange(len(i)), 4)]
        offset = len(i)
        k = np.arange(n)
        for dim in [0, 1]:
            for upper in [False, True]:
                cc = np.column_stack((2 * k + dim, np.full(n, 2 * n)))
                vv = np.tile([1.0 if upper else -1.0, -1.0 if upper else 0.0], (n, 1))
                bb = L - p[:, dim] - h if upper else p[:, dim] - h
                cols.append(cc)
                vals.append(vv)
                rhs.append(bb)
                rows.append(np.repeat(np.arange(offset, offset + n), 2))
                offset += n
        A = coo_matrix(
            (
                np.concatenate([v.ravel() for v in vals]),
                (np.concatenate(rows), np.concatenate([v.ravel() for v in cols])),
            ),
            shape=(offset, 2 * n + 1),
        ).tocsr()
        obj = np.zeros(2 * n + 1)
        obj[-1] = 1
        fit = linprog(
            obj,
            A_ub=A,
            b_ub=np.concatenate(rhs),
            bounds=[(-trust, trust)] * (2 * n) + [(-trust, trust)],
            method="highs",
            options={
                "primal_feasibility_tolerance": 1e-9,
                "dual_feasibility_tolerance": 1e-9,
                "time_limit": max(
                    0.1, min(4.0, seconds - (time.perf_counter() - start))
                ),
            },
        )
        if not fit.success:
            continue
        q = p.copy()
        q[:, :2] += fit.x[: 2 * n].reshape(n, 2)
        q = repair(q, clearance=clearance)
        if q is not None and side(q) < L - minimum_side_gain:
            p = q
            history.append(
                dict(
                    iteration=it,
                    before=L,
                    after=side(q),
                    seed=seed,
                    trust=trust,
                    tie=tie,
                )
            )
    return p, history, time.perf_counter() - start


### `cpu_contact_polish.py`

Free-angle local contact polishing, adapted from credited public contact-search work.

In [ ]:
%%writefile cpu_contact_polish.py
# Adapted from lightningv08 CUDA Packing Search contact_polish.py (Apache 2.0).
# Changes: CPU-only imports, configurable regularization/trust, local experiment runner.
"""Trust-region sequential LP with free centers/angles and exact SAT admission.

Background: coupled sparse LP refinement in prvsiyan's A Square Deal notebook.
This vectorized constraint builder and experiment runner are implemented here.
"""

import os

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
import argparse, json, math, time
from pathlib import Path
import numpy as np
from scipy.optimize import linprog
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from packing_geometry import side, normalize, repair


def polish(p, rounds=45, seed=0, max_seconds=25, regularization=0.0, trust_start=0.04):
    p = normalize(p)
    n = len(p)
    rng = np.random.default_rng(seed)
    at = time.perf_counter()
    step = trust_start
    trace = []
    for it in range(rounds):
        if time.perf_counter() - at > max_seconds:
            break
        L = side(p)
        theta = p[:, 2]
        c = np.cos(theta)
        s = np.sin(theta)
        h = (abs(c) + abs(s)) / 2
        edges = cKDTree(p[:, :2]).query_pairs(1.75, output_type="ndarray")
        i, j = edges.T
        d = p[j, :2] - p[i, :2]
        ux = np.column_stack((c[i], -s[i], c[j], -s[j]))
        uy = np.column_stack((s[i], c[i], s[j], c[j]))
        proj = d[:, 0, None] * ux + d[:, 1, None] * uy
        ranks = abs(proj)
        rank = ranks + rng.uniform(-2e-12, 2e-12, ranks.shape)
        chosen = rank.argmax(1)
        rr = np.arange(len(i))
        sign = np.sign(proj[rr, chosen])
        vx = sign * ux[rr, chosen]
        vy = sign * uy[rr, chosen]
        rotation = -d[:, 0] * vy + d[:, 1] * vx
        delta = theta[j] - theta[i]
        cd = np.cos(delta)
        sd = np.sin(delta)
        rad = (1 + abs(cd) + abs(sd)) / 2
        dr = (-sd * np.sign(cd) + cd * np.sign(sd)) / 2
        dist = abs(proj[rr, chosen])
        own = chosen < 2
        columns = np.column_stack(
            (3 * i, 3 * i + 1, 3 * j, 3 * j + 1, 3 * i + 2, 3 * j + 2)
        )
        values = np.column_stack(
            (
                vx,
                vy,
                -vx,
                -vy,
                -dr - np.where(own, rotation, 0),
                dr - np.where(own, 0, rotation),
            )
        )
        rows = []
        cols = []
        vals = []
        bounds = []
        offset = 0

        def add(cc, vv, bb):
            nonlocal offset
            bb = np.asarray(bb)
            count = len(bb)
            rows.append(np.repeat(np.arange(offset, offset + count), cc.shape[1]))
            cols.append(cc.ravel())
            vals.append(vv.ravel())
            bounds.append(bb)
            offset += count

        add(columns, values, dist - rad - 4e-11)
        # At relative right angles, both one-sided radius derivatives are needed.
        kink = np.flatnonzero(np.minimum(abs(cd), abs(sd)) < 1e-8)
        for slope in [-0.5, 0.5]:
            vv = values[kink].copy()
            vv[:, 4] = -slope - np.where(own[kink], rotation[kink], 0)
            vv[:, 5] = slope - np.where(own[kink], 0, rotation[kink])
            add(columns[kink], vv, dist[kink] - rad[kink] - 4e-11)
        k = np.arange(n)
        dh = (-s * np.sign(c) + c * np.sign(s)) / 2
        aligned = np.minimum(abs(c), abs(s)) < 1e-8
        for dim in [0, 1]:
            for upper in [False, True]:
                ids = k
                extra = k[aligned]
                for ids, deriv in [
                    (ids, dh),
                    (extra, np.full(len(extra), -0.5)),
                    (extra, np.full(len(extra), 0.5)),
                ]:
                    cc = np.column_stack(
                        (3 * ids + dim, 3 * ids + 2, np.full(len(ids), 3 * n))
                    )
                    vv = np.column_stack(
                        (
                            np.full(len(ids), 1.0 if upper else -1.0),
                            deriv,
                            np.full(len(ids), -1.0 if upper else 0.0),
                        )
                    )
                    bb = L - p[ids, dim] - h[ids] if upper else p[ids, dim] - h[ids]
                    add(cc, vv, bb)
        k = np.arange(3 * n)
        u = 3 * n + 1 + k
        for sign in [-1.0, 1.0]:
            add(
                np.column_stack((k, u)),
                np.tile([sign, -1.0], (3 * n, 1)),
                np.zeros(3 * n),
            )
        A = coo_matrix(
            (np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
            shape=(offset, 6 * n + 1),
        ).tocsr()
        rhs = np.concatenate(bounds)
        objective = np.zeros(6 * n + 1)
        objective[3 * n] = 1
        objective[3 * n + 1 :] = regularization
        trust = np.tile([step, step, step * 0.4], n)
        limits = list(zip(-trust, trust)) + [(-step, step)] + [(0, None)] * (3 * n)
        fit = linprog(
            objective,
            A_ub=A,
            b_ub=rhs,
            bounds=limits,
            method="highs",
            options={
                "primal_feasibility_tolerance": 1e-10,
                "dual_feasibility_tolerance": 1e-9,
                "time_limit": 2,
            },
        )
        if not fit.success:
            step *= 0.4
            if step < 2e-7:
                break
            continue
        move = fit.x[: 3 * n].reshape(n, 3)
        winner = None
        for alpha in [1.0, 0.5, 0.25, 0.125, 0.0625, 0.03125]:
            trial = repair(p + alpha * move)
            if (
                trial is not None
                and side(trial) < L - 2e-10
                and (winner is None or side(trial) < side(winner))
            ):
                winner = trial
        if winner is None:
            step *= 0.4
            if step < 2e-7:
                break
        else:
            p = winner
            trace.append(dict(iteration=it, before=L, after=side(p), trust=step))
            step = min(0.06, step * 1.3)
    return p, trace, time.perf_counter() - at


### `resolve_public_baseline.py`

Resolve the public source by stable notebook ID, even if its title changes.

In [ ]:
%%writefile resolve_public_baseline.py
"""Resolve a public notebook's current slug by stable ID before pinned replay.

Only anonymous public competition listings are read. Notebook renaming changes
slugs, so the version number and file SHA256 are still checked independently.
"""

import requests


def resolve_public_slug(notebook_id, expected_author="kirilltym"):
    page = "https://www.kaggle.com/competitions/squares-in-squares/code"
    s = requests.Session()
    s.get(page, timeout=30).raise_for_status()
    headers = {
        "Content-Type": "application/json",
        "Origin": "https://www.kaggle.com",
        "Referer": page,
    }
    token = s.cookies.get("XSRF-TOKEN")
    if token:
        headers["X-XSRF-TOKEN"] = token
    for number in range(1, 21):
        payload = {
            "kernelFilterCriteria": {
                "listRequest": {
                    "competitionId": 166661,
                    "sortBy": "SCORE_ASCENDING",
                    "group": "EVERYONE",
                    "pageSize": 100,
                    "page": number,
                }
            },
            "pageSize": 100,
        }
        r = s.post(
            "https://www.kaggle.com/api/i/kernels.KernelsService/ListKernels",
            json=payload,
            headers=headers,
            timeout=40,
        )
        r.raise_for_status()
        rows = r.json()["kernels"]
        for row in rows:
            if row["id"] == notebook_id:
                assert row["author"]["userName"] == expected_author
                return row["currentUrlSlug"]
        if len(rows) < 100:
            break
    raise LookupError(
        "Pinned public baseline notebook ID was not found; keep the frozen hash and version, do not substitute another input."
    )


### `public_download.py`

A single streaming HTTP transfer writes the fixed-version numerical CSV to disk.
The stable notebook ID, historical version and complete SHA256 remain independent guards.


In [ ]:
%%writefile public_download.py
"""Download public numerical CSV data sequentially without holding it in RAM."""

from pathlib import Path
import re
import requests


def download_public_csv(url, destination):
    destination = Path(destination)
    partial = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(3):
        offset = partial.stat().st_size if partial.exists() else 0
        headers = {"Range": f"bytes={offset}-"} if offset else {}
        try:
            with requests.get(
                url, headers=headers, stream=True, timeout=(20, 60)
            ) as response:
                response.raise_for_status()
                if offset:
                    assert response.status_code == 206
                    assert response.headers["Content-Range"].startswith(
                        f"bytes {offset}-"
                    )
                else:
                    assert response.status_code == 200
                with partial.open("ab" if offset else "wb") as stream:
                    for block in response.iter_content(262144):
                        stream.write(block)
                        assert stream.tell() < 80 * 1024 * 1024
            native = re.search(r"/kf/(\d+)/", response.url)
            assert native, "Expected a versioned public Kaggle kernel output"
            partial.replace(destination)
            return {
                "download_native": int(native.group(1)),
                "streamed": True,
                "network_workers": 1,
            }
        except requests.HTTPError:
            raise
        except requests.RequestException:
            if attempt == 2:
                raise


## 3. Immutable inputs and source credits

The export parent is our previous public V30 (stable notebook 140109525, native 366028783), CSV SHA-256 `e0c9104aae54695732a93d53c8e857bb5992888a7728a6eb6bd7e21e7a1bbe3a`. Its full Kaggle CPU checks measured 22.249885337670. V30 was never submitted because a stronger reproducible public result arrived.

The numerical reference is [Prvsiyan's fixed Dataset V2](https://www.kaggle.com/datasets/prvsiyan/squares-verified-best-of-1105), CSV SHA-256 `00f11d13ec771fc050a2eef78ab3c93ef420c7b2ac983e85d92392b35719cf47`. It matches [A Square Deal V29](https://www.kaggle.com/code/prvsiyan/a-square-deal-research-to-verified-packings?scriptVersionId=366027470). The published CSV is the numerical source. The notebook, earlier blend and source authors are credited separately. Prvsiyan's reported local contact-polish experiments are historical source evidence. Our CPU controls begin from the independently guarded layouts.

DATA_LICENSE.md marks the adapted data as CC BY-SA 4.0 and SOURCE_NOTICES.md preserves prior source-specific notices. The stable-ID resolver can follow a title change while retaining the pinned parent version and exact hash.


In [ ]:
from resolve_public_baseline import resolve_public_slug
from public_download import download_public_csv
from packing_geometry import load_bank, side

baseline_path = CACHE / ("public_v" + str(BASELINE_SOURCE["version"]) + ".csv")
if not baseline_path.exists():
    resolved_slug = resolve_public_slug(
        BASELINE_SOURCE["id"], expected_author=BASELINE_SOURCE["author"]
    )
    url = (
        "https://www.kaggle.com/api/v1/kernels/output/download/"
        f"{BASELINE_SOURCE['author']}/{resolved_slug}/submission.csv?versionNumber={BASELINE_SOURCE['version']}"
    )
    transfer = download_public_csv(url, baseline_path)
    assert transfer["download_native"] == BASELINE_SOURCE["native_version"]
    Path("baseline_download.json").write_text(
        json.dumps({**transfer, "resolved_slug": resolved_slug}, indent=2)
    )
assert sha256_file(baseline_path) == BASELINE_SOURCE["csv_sha256"]
Path("replay_baseline_source.json").write_text(json.dumps(BASELINE_SOURCE, indent=2))
print("Pinned previous V30 full CSV hash verified.")


### Lossless numerical replay

Restore each checked packing by adding its stored modular uint64 offsets to the corresponding float64 bit patterns of the pinned Dataset V2. The decoder reverses byte shuffling, checks each restored layout's exact hash, and keeps executable code visible. The packet also contains measured trial tables, fixed numerical starts, and license notices.


In [ ]:
REPLAY_DATA_BASE85 = '{Wp48S^xk9=GL@E0stWa761SMbT8$j<49lc{#^hGfP@?rAuPbhQ}ZYh9WO-<nZefEF@1{j?b;Tuje=CZFU(ti2rk*$os|h?a$*62pH69dpWqNbMIV^5Jo{nW(%ZAR>}#DY1w%oD64|+;(q#i%vy;e=G7_@aeqSAXZU&synxX6ch2M1ZpM#}A1#*G9gp=9Se&5;zlaYb;&^*fLg)OF)xeZ5~1DW`u)l)o?0KMK?@KxeNE%}}!y9>=%F)s{OE;i~E_~Jc65O|r+JCvBC;b;R8ey0uSG)gJMM_!qOmKke8j=4uscOe&CxX)+hCGnUl1vRM#c>{o5>?}KU=Z1|KmuC_GaJR|!@qWDsL^U>@os5=qwhm2YM7s3^%e@>l+r7v|=Ga5NEQk`HQ^9O+scco?zp0#MJ=_^6voA16po+(f{c&Zma{1q6-o(Km+%dF`WkNH9332~vW3{(J$9m)`xYlik!lz_b_TqkSI8)N-B7er48bI@oFcC`0R6bU2kFDqeuPKv_&Cf%OTdGC=w;3R}sN$Y90=hy@e^XuUI+I9{r)umN8Gg8@9~%1!^HZb+4*gyQ++dnjiarzcTA)>o3z5dDgf-NoNzCfI$MS=ngmQkG-@{*Ei69wJrEYZ8{WS9xy`DRHu%Fdr4QS@axqZ5y8kNI5hz&PBXv3NM2-LiJb6_??1C?NAZyUHHnxbQpkD3|zm8K@0^TyZPm=#U)CL^lnazjBF{#_;Ni_Z(kzg}-OpYN02g-Fruo*t$<DrV9iP^Dt(@;N@=8U<s@INyAu<z_+a4$*jGMkT?y&j?1ia(JzxC(om8t}bEP1<U{tk_4_-6z+3*kzMQ!`9fO$Z>Ak#xpu#NCwk&h_&my<ER%*0RrWmJ)dQ{%sBjB^W?g45yoPlYUPc9R=^A0-+Z8{9l9o&cE%(Bx^saR?galQu7Ji=m4lkdS{>urt6*_h|P=7F&T1qK)g^STzXem;6w1((Vc}wrM^yn7+{Aqe@YpD^!#v(eP_)@X~G;JdGXyrOKT8$Ci%h_t-|3{^?CT5mVXzpErp^$S#o$*<SL$MA^H?8j!0h0=l9aeAl^ypjPqIaYq2xoiy)IkLPzn1nOAwvOV_2gc|!8|5-5};LN-XX5Ry0*of5e8_zV<rztRsPR7QrA4UBs*k1%c+vdGkJx>!!?KP4cqdYnHHLCu+Z!MbUMM$05SsR3&t+3F+u?EWN@BOq>3Ef9w}FLzZ*Dc|EvDI!X`xr;gjxBk$>ncRHl&Eg>e_nMc{=1bk__<EiE!5v!!SvYahJkBsSvF7rB`Q;P@<b>Dp7?1$&@#v=jv~7PT?mf74~I@Zu4^AMv7<cn!o+xsF7(zUz`^mM_$=2KZnnx^7AnG@1)ZiC>ozFs$LtrV{BcAmY5EKSTSQoJa4iq_v~0GXlr9HYqv5N@%}vxILgn$6Ifs2*`G~w6JUv6LK&Dx8PaY*_quvgj-I8lyOZ%{}b^FykV>l7qhip4oZ!4NLz&H#A5Ty_Sh#8P$T{OrvF|GCx<`|WB5k~Ik&N50b5+1n@Ka8OsQ-MD-wO7SmH<OuNtA7DMR8RRhS!adPc}9KJfeHA1?q+BXYWlcR2ofFIc{Fx2Vv*QT_9W!NH)~49@MMvz612#sG6~WVxe7ev48k&kuM>Seb@g1`BEX{((TZmyDmLODL7nIeunfFE~X;=_#Y2U8?^9ZsVux&9o2RXal+*kX-V{38fdc=xShYvxNZ@GsP-{U{5+x;l4lY<_EmyFhvkjACEc_Y2UkZqn^-s&Dq!L(6AUgsz7^4KuUzpqtSwk{s;*-`e3toRZ>f^%!5-9iA4y}iW_uPc69N^y5jD^9bPYx`M5qhi^-W`|Cm8!y`x=?)yThPJfXb`Y<stj#%p=p=ZbQXea#00x?8W9)mhUR9ET;f{J-KH&;1r~HyHxs$v7&JLg(DFZ(rWv871L%ZwZWvJ<#bS5a~*H;98pc3wkQ~%nN7Q8_CErl;|dfh-&-F1o5tiL`1tHm80)&orKm{M6mG4=prVbkxU|6^^(YxabIpdo>X849leH_cd0yzNum@`#f3Oj#+UQ(W%NQ#>`oxn%BO(%aeie;OIXc@T2nH9WZdW-V2G&C*Zg7Fi!ArX!5~wWIl>ZkoFddGxD^)uzQ;vBWDyl&#>qtHTQFhzh}Ea%a;PMgg|h(3{2l9BpL6a{xnJ0>)GK&A|Lo(9!ogU!pMv~IdpLm{(bH5_sj8<o(?E33^KR~|8`x=vLi*j=7%3jwy)AhKSoi-VmbQ0{&A0g+PB0}ku?C^9??skEdj&!sZe5O26g*0bf95XybdyHm9_eF<n5o$Cp~QE{Xla`Lt^Ky<>H!|9KI*VTlMGwX=3F~K=H6k~$B$%vTnI}k7Rpsr&_62<vW@EaXG!_z^i(qB{qWC?z<dR>*9AW3SKmA4cYd<;{6OB)VpwVVa~0*ZY-!nggHpwSTG&72x8Ze<zgdgX7#+}`rY}{DIXG;g{(_<mj=t9mY;ISzv({hL+1c@|)fLix>qvoh2T6tdK~(&t&znadQ8gEZf2`f#AE2Yq&0Y62oKkv90W^H?@{VT!jOJnNMY*38YK&b{&+TMf!Fp!QmIAwW(0mSj?HWRg|0wKOAjE_H?LY3o;x3r$-HzM`B<n?vhC3mIfh`CaVe2!6ODh(jJ6n7sgE|sZYAstBi(Vj*J7mCKUUfJ^R~&73Ah2n3X%Uy^f+*skUJ$S^8Bpnda?FoH8Dc_A8sT0+MyUE2JXDQxg}Q?d3oU_y)NGr>v#_p7W7jPMRgC~4D0-#JlU^zrn-Mv=4SBX1;{;R6q(|y2=7J9h4VOSB4ry9EErR1eD|Nco9MF$9%PUMjG4HdAykt>=44XTex>)ZQA?fHLx;R_@0~E9bOE;r(*KV7oD1Nzi+5b(C!|Zp4qhj2UsHx%WPBsax{mmeqVo5}sT60SRhm*6DuMe=u+D}^mpnx1kBialKCUvHS(<^^I{UNXT)X|=~FqVf6p(lt$Dn4s#y;Ptfj*61Qt&fv)`+q8(u6&p0eQjD<^|L2p+aAcn7g!_3$=3GsKA2~=UEzUq%@dTxYmR$QRS=A&wtYA&5KO;z9=!OZ@SY<Wb7!cF`xqJ6jF(E0eEW<%7ym&|ZY&MKMEPw|@S8wY{Skk)d;`$qqCT}QN-ke`J4CE-zQ*yI(2>s+LsaqAY7ZzGsgTh#V$TuG>9dYg#e;p#Y(`*b0Bqcq-)RBt6&blI+@UntDlDXgQPCOE6;tlj1>+l8y{*{)0w=Z~->>S5RlGVzk}BPz0XUs!2Ck!L$~M`n?N0OF1+E-DCFm7Un!w%K90e?Z(e;cvr^-x~M+-zP(D{yH<VfcIJgCfF_Y&gNOO#OEj}xB-FgQ>~=9lQ&Q*&gc3KLo4Acm1z0b%v=bkg;ZjdW^=vxjG`q{7NP{c?HYl0nh2=o`(=mH>-1HP<GJg+d_RAA|>IXR;#D8PrK7#NT$Ea)tQTI~X*W)C0-p$Dqt(AJ7Cn)@>>v+nua2wu{S^NpTYZ+9a6Gh)_Sw>8|#w1|x_MM@koob83T4QOPK~);h(!10ID7(Rh+f@5d7YV(}ppcM7IsQ?SIPhU_}-%-T1?kWr51?cj}hUp^8B0Xt`wR4#8Ue(ll+MS!=J7^-u5O}Y*L2@Omkr)TH%N^D=7-8&%LDVfdWT~+Y@yhj>nnoqm;sV@3Y$m6u3x#|4IKeYajO3JC&rAzYUu8DKiHpMu_#B$661#y3_8Jbn4+k<LAN(3QZQ)xOBa-PcrJm3L`!$k-$Mm>2boFTF1vY0`hDffU13i{lroT}%4zPs+~3QDk$@Ku<T5)C#s;~2lTCkivM#Xyr|7I<MmS&I011om(9Yb2%gy(`vpYkDppU=XqA>T{y2WglrtCOm;I%}90&J60|jAa;6tE0<|{->2=wm?JxpB2?;In)+y%1@qrd6ZA6NK|$pA`5>NftmvyCDNw3jG4AJa{&K4rU!UgP`2&f~Hp58%VGx6LMYE;VNX|KY57g@^B%PlZI4{s(nE%-8gWwjfJS-;z&>g99d+~0aM!9-Rkz!J<90ZRIrzWfKME6%u+e~kYyn`yx&Kke7TCrH^fq2B`s$fa1u40?ag_t${WpJoq!=6Q=e=zwv@txoZ5uV6_WoI0L*eekzGvJs>oR`gX_3)5{xBt&5(2(1ZbXgYMf2!uY($EZ56<RUqUks|8fK>0smR~D72;nCxn_ha})i{!I{@u?+b?$oRt|pBqqNhc-P_0WD4|N#Q;6HVc3C*A~vw9L>itwnRT>ZJUM(8Scd?FIPJvu0iR*Dk|%fG50dohs&L~~spCF(u>6Urs^i~Ng89EsqUW#p(;F6(fH09IyZr?MkUd|IKg(_P#rd1e~PnnUz^V*3lH{i4`XhJ*E#^V5Ur+gHUaXqvOfiGZNTWa`|%b6AUsJ+i>rC`=G15BUwTvEsSkcSiP^ja%7RA}yDcRcbmeAX#xopR~1PTozirPHl3~6MIJ?-%0A&m*GqcK6G)_+Dh)+W*}VnFL#Z2%;<Nux0WRe#C`fYp$Ajp9GQU0$}z40tls!+g(%-?uykX&h*m@W3Q!={^vVkZLW^9e>IL*3BiDUXEMl8BMAom7uQ}kh<#;9GsI;k<^b}hYz(cje-rs0UVcEGEj_lj@c{Y|FIB8WtlTyBG29}U}<T<hswHY$s)H<X;d%bDmP+zG7YIG>WS1dBr4JYZK`E~B$K7pf^a$!Tp|FdYONFsuRT`IcrVw(@O<xvKY#=&%Ti&Grd>B?H9R1&2tQcNW@Uea;x2P$(*Vg@(Rwl)FMd#CKMQc&nI>uSe;d$pf>(YPf`4Fa9vNf3lK(^4?K?*VXIW1apWpiaaWnimldqInCVcdSd3)DC7=^9LZNF@r}>2GwI7JXgtsZX*Sy1V|aIu;>*b_5(+>9OFD4Ji$f+4G@;3keXevWI)6QeOZtns4`z`7BxN1d0?t8g~xmeXW%R#45ly?ZpAf;TH7%wJQV$Dw!?JXN=0JbJ{YZ#l<XZ)8NEwi_#VfN1?Us}*>2PU0~TEzp_ETuOELc{9XnkKm@s^gN1>*;Yh4AmC_k46(GQRsNkwQxr3{c<XcBrV3|DCy9gL5TA0-6u3DZuSTaxSTFn&YU#XPvDYPMZ*^&dVh`DBn`{MQ(=HhZOuQ3>#4XK`5y@3k9vH6Y)8N=Z0OhAPqvy!liDx=Up>I7p5Y0cD=ScG`}T5`e5v`~gKCARULThF;HBSZcQNi0@LPK|ct!Yq<3pv69_CEOhkxjuju*`YTJkQ=!r6_@4)bt~ICt_kuf&U2)ki^0*mljBE$KQk#N3=Rs`MBeyZLcNfFQKIUnp!Pj95I?R4<3{I_ujB`nqpp=3YbwkAszgVM|5{o#2y`S`J{9o2%Us0WG_s1m3>Erwl_r=%h2A`DnKb<{$i%iYVZd>8y2_aPzP>*wp83eo$CbFRc%MA#VLLbS={vKmkfE0MBa)|FkjyT|OHFyGw@-b$KEtxZ}a2;!i9PFfs16T8KimtXA2?r4rXoUYR?=oOpG?>_@0nv%JoBd;_#u?$Viq_xixv8xTiXnu}O<SP|B4zT&lK<oSbnaC+WdYyF!Cdgrx_40yt`il`Y^D9&Nrw5{4IW+v2{hqVB?KxgX>S^)o{N`#oZ_?RvkLa}>aQSr&Xs?{tNi>lscNtAbDOY;01)|*=HvIRww00cN=$^P`^~PM;^_JMcgd@y=$t6d#oQ2{mjjggm0AD~{*^5u3NWq?>^Rz*X$W~hAKXuW4dY+hjPTkk&^dksr&HQv=@*|U3f91OlH2B5Y|eS3X=ke0fxf;l#Z08keCpbT?z$mQ*vAlUBao~>gEjS)q<gC=lf?ft0)-vCXtwEJiml*%hKl=L5{s;;bTC6RQV$jUi<WUO>hN8SDXUqwi}SNeR$>Mbb8YoUY*-aUf|_0e0`MDT{<I-^di@kmvRfP_A>EW-yLK*T^L;z(_ad;d7MLbg<sOB`!mOBn$cGn6l3P&c`6+X1?zeHqKWB@LIoexOKagTyH!ad+&JKY-7sJv{M`?=a{PppNnN{Bm@4TWo(LoRUp-+K7xczSXkuYi&bR$u~j@#%e@y$6{fa-k8UzIgXSU1NGW}eGvtu8Z6x^6G6K?!z})aoNp(*}J&y2SIFee>DOyT*gKV@T*LCIJ@n6{l@rDn0uO)T1CubUAH?Y*ZQdFFJRl>~i(3vi2WY3m4jg$<~I<HVa@qGosZVOsu0VJvL2Zs+SLcYrn<Dp|uBsoMQWloU-Uf&4RJD=_2ZiXL>e{VP<ZgUvg|PaA{W=o>N?iLg=uAOZTx@(p2}gK}ei^s=o;OCi1H6vS~bPCu2U-j>=MFrS5KIOcF7so##!q@Rg2EDqA7yBLIGH2bB(W#mYN4zr`&8HcWElb!1;g2Hc#P{$@IjzD$&dMJ_-3as0S4RZn`&mMSX|e~JS6_fuChHh5D0?Tj<GE&YU9u>!!_(&tX7%XmJlnTCQH^J2k+Q0_O&10+Jmz|c6QC_fu|#%I5MIH#5L@Y9zdMQU5o5!B>{hSB{zQ`T$~D@aAdFa{`S##=LeD(8F4BHT(nGKKdV<Sn5A>CRa&lA?RZz^7kZhTGppDER66(iLz7+Mn4X@`;2y-u$Is>D<K_Wc2|UTN-ql8n+qEyN=B?7A>((`0KCW%#UJ(9o4R*+&_WKIPC4uR#K~onP1XPp+!?V>e#^XlQvlHM^V<(Y<V)lQSb6lnCY$~IS&NVQj!{gkY;F7Ap+Ou!)RBp<bO*!56eGET8Z9SP?zArLB5I16|XtRL)g|~0st0<6fPME&N~VHu&h;UbT%TF#;+$^e}tFQ8VI8s{JNZpEIOf<&O^+(N)n&lz(_CV3@rO{4-#sZT)zul$ee8OAQ|}MPmhSKS^2hqvWyi)on*KdR}74j_SQWvS;?7BsD?9GRIPiaPNLS;8(kxOksIm2Rq(4_%r22n0|%tGG~Gz)Ja2u^u*~2iI23?!yxS#;p<bngta++=&20e^3L-4LO2VR@O~Fz3UokUc3YpEH*{%TuI4lmB#z_;x5`LEa&RWJg#{C0KjmJFOo^u@XWV(p6)#(d9>VX#FrrY7fQP`lFLbYPB@1{qw9zg;n)!=t+nx<O@nTA<aX6Xb6v)GINB&-dnbf{S6B?!ANCh!8t$@vaKUrq|I9%Gs^c};M_|CPvOlb`!bbl1e^ZWSxm-IGbbC?}b@pkp8SazrBolxRxG3G{6O23a}0k##S|zp=Pfbr<A<B`w#Ab4yg;$E(9*qm54%*v5qxD$D=ALu^l833xM4!$hK_XE=@jWS5Jc;z4+FTYoQ+iA-(2XeU+_T|Nutp{EMjH$lblu~>Lt*J^f?QY+-HR&v~BV;?iac~WdU@yJ^mIQ1styP~tNHMj0%2VyVo0Bs+;TKo>6S#>VFmSjPRF`GUsbD>r-?76$TE9<~M9J%!(L<n!_H4O19j0ZL+6u0~&o2wM}R3>pANJd(Zg4oO0+}-UYwI74Z^>#k(JTp>>`=gM4t`ZrDy@BxJa9ZlwT$B83Y^N<@y|!oHO1ZIWq7+#AF9=9pe!xB(0#&6e+;@mg=MbK*i-b_bR#nh$sxNb4)GV?rzxBNFT=<Uh>13@z#-rOGG}y0;Y@v~-7AeTSDTB*M=|j?e8wK^F#>hU0M9ub(-_y&EiHj%Q3cI>U(16#+v$_FYQxSrlE9rW+G>jT6KxpUHV2G9T*Vl}LC4qPxw!cbHZ*I(pbG*4W#QxVIU))#c=DC-_hQqiiI3e^XeBZ%RRJ(-%dPfri`TuJ;7|^=~l}e6NL_dAH;1sqWNm6xZJfY*g5}TJ-4!}dpqFJyWCIs>z!aN2!gHo>X3r`Wuh8BJXufJGA>i=MSgb+H_RDEw=0Q3qK593=j>pSaXERKw`_p~qTxWOeQw`rqW9-7QrRKlx>{{$pIfipl<c^Yc6NDe+Nw3628Ke<OMO~VKdcmKv5Mha$AX&9)ACK<{u29!ciRY^@QD-m-)Sr(S&^m=8H>VP)>f^3+dp6H~`Gdl+Qrx>d~=Pxb_*;$b5s*nK}M)~-U9H3zDd63;8ANN2smZ^MXNYw@qd1X<Hhik9kWE#4k2M2Lc%Jax9@W6?SQkPcdu+Mk+m+gTfc?LbaUYoN<4_z0zaj=dQJES%HC@-RI#og5X20Vf4mq#;oFO{bjfI~~e<w`11HJ_GjVnK+807sVISr!Dv-cifzFyDy3D}#q{1#g}W-Z+VuDfuIgq(_|%4!sN#TeP1;`3_H=`=Q(Ipj5(0ac>5d=qBK<8S@b0Mr2!cLFv9!$TD9%bzRg@H)NCXggG!5Npq&UR`*k%SDrtYgQRxT#_}(izc$wjyV#`pt#%&=NPrM7k{QJrK2{_8NcgFtV$7L6fX=ChceTG4f3Jw=qmm(|Fxgr@<av`EGFHdX1ARmZsm~4UZu1bPFv!`uhg~CKFgGUH!!KMt!v%J(t65SWGPWy0oY#*#>yBPRYTSOBmi2j$Yo}AGJl<QW211k{-$KRWQTA{<FUd|njJa0q`D$<0e&K>oNJmkE+y)f!%)OnKaiBH4{P;P~Lm=~N|9Fx{2943Z${GK4s*Gx#duKsl=<w8Bxgq6elvuo<*V7^ds9=D<@=jw?m6ULxii@?9msV|x*q<@MZ&^v{%^6xa&YPQ0;Ej+jy&VDxppFw|(FfsNaye|~H0_FM?EO{E(g9u06lOid3NKjm(S?^7wA}>UKBgloJmF;77XKWQoTr^v)x+P)#dhdgGvq3_a>7Yz+D5^I3p8B%iGZKfWXr$r;5>KZytgL@+GSL7O!Fsn6iH!l0HvN6J{vU@pK|hb?&3Q=!4&6}vSPWRU${5g{fPt-V(;2?W!AeouY#pV*FS5i9=9suVM{V0NQUExYJAukyG@dnKOg{8f+9HN?kKVk2RA^7&@)*@Hh|)d2qFf?(x*34!5`zDJN4+<{x7WF?DUO;4y$7GgzwRybY?9xz09QGu>8$lqpE}p0#&<|MC%K>N|t~OId5idfr)GR{)$PLlAs>Y_YZEh*ExFr$RriUw*u-;^R)??C&IwnBn2M*j=IC@7W^pRvczDqO8UO6FD2a9EK}BP7r0)$5a4)n!sv?jrxJ6s$M#yYn*L5XLw^`k4A?18qz}V+op%y}LZr^6S7@9}i@<MnNE5Qpo?+DcL)rp-1unEaxbbF(&fgPm;a(p8Gg*nQ$IaI>8M{<idKmuhKKt$kmlTX4ENq`8<>49d#%ODOPN!~w;wv=jvixvPwgJ3vAz%zVuD|<s6Kr!d%(q$R44)UadqoJnBiPT81?_`!;BfRo_Fk~~z;xDk+<1Ks$~UB}csn4&-j_39>x*y{`HiItCWiL+(oZ9vkaIU8hw)%Q(i7QooxT15Mv)l5A@x`vwi4bRR$HxWB`c>DB=5v-^mqi(s5S<mko_|wbdqKP%Z&y9=zQ#FQ{8Rf{8I$D&}#!oyweTBX)sDj7`?n-^Dnm}w&hv);=!69HHFPjm-838mO!*(>^cyWzRx<NEpjzcG5NMT1+Ae03+*EwZ$p;w5<`Re=ul<!NmNnL3=w%s{Ck#Yi(6k1dy!(ilAFY)`tnnCX`j0}Sq;ydmQ#O}v^E62tPln5RbGuVgs6*-FKUxiJgcwnUZkl_U?M5u@QUJ;+AlE5s4Q4X!5Y+m@WU#MBZe__1ree@2r0%X8w#rLe)NS2GDC$6Jqf7cbKRUNy<_;6luFx0j^)Xpt~!3tv?EW?uVsLoi{<c77Z(+l4jiN8-a`|T(T7_bfq4T)oyONON;vJmGpF_%CD7Kcj{qO9Z)kwz&kxNjMOCc>5BDH{)*DxjX>$kn0wRQ2mGVP<nRCu(mm-GCu4V?V-&`1(Gs9KC#jrITkJ93ISCmwj9u;{@H_Qhc!p@m>(8ET?_ErJwv;p3dHW$%RTKsQbH;`f`9%!c*G<G}SeEJTL<n$s_uzHe~%+@1vs)~-U2jQ^4qzSnh)xN<bVI5iEn|YtM7>+b)bbVxvcJ(AlyhECbdO(?nag6YV{*^kqZF=F`4YbAnaf_vqFVV}X9hIFrMzN)A{QmKJoy4q(PgrRj9abaEl;5G6>l<YgM)Fn`Or5ma-O-VIlA>~y5Z|YD*wKmC)6e;BwT>;1T759WRXL=Sl6}Pqrb>EPhT9PEvd(3s1c6A$sl$DO0mQR&i)$S}RRiFh^Be_R4t7*w&$9_zK_M!f)eTOHf)wo*<TY20yV7*2i%cO5f&>i3|JbXgs$RF(c2QO$QZ*PK)ee-9J_<XC+UE>|XiirR=K|VfAfUVtE<1m+&18FJ(jH!jN8F6s)tw~RJZ4*#UIOz-BtFD!5qgDE?5DBSx3Q+i!~fEM1`6bQ6Jk<x_&^Y4N?OCXMRz#0iWs@n(Zd33*Qcw3B?HEsXpi{P8ifVW#?gyD8LV(Z*5O1BDQ4^&6uOHBJnJ_#y)~i_HCN>yK)!auOU*6~68tUe0ara?TPb5(v_Y;w$%oL*aJObj)c{Q_vC5I3ts=I{Dg2QHhW@>^3ad0t=l67fq*bBfTrQB7_pER+fExe*{v~Ied6`ca3^kqjBTfnA%$Kvh?v&8PwzV_|gD3eP3_STIM*Hz$1T2<~6x4f5TVKAAoL<Q41NWcQ2#OK2Dxbb@EpQ`DeF&E2YB^I^J<Uzus8BlN)z_ifp!zhrvzw7y@w~(xIe9BscXeU=S|sN@6qdVx*uznfS;WtOxQ2~<d#7-Zje7w7tPaQ|bFbSE2ictJGX2Lf?<u5^=F!Fao{6%h;G`2-&HBMOhohGs!&NHRP~E?iXn1fuvZI9R|5t5x=j<gza4Ej+ctkT9$LDC;UI`Pd`RIHa$!+HeIGoYD8pBKpW2Bds(K|i!L2BRDG2_O;WjL+xT)l|c$?Tz;7Lwz3^JGlH^}W}p219Y)WtwxKW-XxiG$!?S-}fQy=f?_-cg7=FWX<Pl^xrSkoMHo(<{CQC^d$A0VzyHI{2Wl<xe-oqP9`uk1ZUubx6asfhisZ*b!y`#i8V*0_HWroOWxTYf|-nHq0xc_p$m9Fxhhn>rKXhF`nIX3LFpH9?RIX-7uXJkO~i&ZKn~Cn^CMC3?#KA@#-K3~Q^w5BL8d54<@5He2cQ+hBX{0-lJxfXf`m_Bt_dIFWa@ySMWEtugDTtw8G#@IA81LhZP1Byf_arx9_@C}(7ODcFSF{6U|o|79d8U$)q{_c+z!R9cW|j39_H)ZU2I6(f)N-`;EWeZeDa}Yf`?l4Rb5`Pd?j$VLG9NXK5x{Up^Ud1)a8Nc^)r`?0%JA-ydVI4&1fgM1qu+@#$~R$fD@EGa`ZRoC@`gM`*h7Jd<Q~Npar&B&rWf%tn#jpa{!NFh6Dlzt1JYhu+Iy}`eVq1bise_=@~DW@P@2%^aRV)J6{(Xd6=w=iO@~o8J33lF*i7CJVyk)QutcT2aEBA-p3Ykz|nw+3y^Nob3yNzF+QAXK=XDsIF^4~+7F}Xkn^;2<rPc|?%U(B=%PyY_CnuXw0J-yl+o=Qz#pL766uy3+ug)9cg(+^W!~3S3w~JsryOqT%AxUX?Fsw&gs|*SGcGT`50Qtk>f}ezBuIc8#!T5;&USi$&`q#lTinY&G>hy~bm6{NHD?D)6FKeCxttw0zb3rJ)&sx1jiFlrX-=qgm!893m@>!W&$iv|e`UI$KZ22Kv7_l~E?U?-D2L0VB%TKA-^_>~fRJEkKW(g(k)>ylaFfwNCU&aC8UIzLYKkJzP{evXe*Xb@Mc~{4hEqA1z#e<Bf?vEYhg3_I{92lUQ*^XAEZ$zr=(%;Gv!RJUP8n7dA_txFB;){U5JRYQa+8F3k)F9O!oE$sWXM>7tMMwqN?@)5NpJPa!>Ii#c%;CTdM3_10LUDZr+>%5ce&t$7>ElXQf&zu$v1MbIdafZ1wr48Zv4$%AR0KTQ8f9CNsyF(=l2HW&&ftz!-op^BOk*BYMuPkjwiw~c5*p}TvWL>MpXn12pzSvwQ%wr`x|eD(YY@THAmdfSvkY@_cqtec_!U*wWc~CH@BY&v@e#7OQc_*Ra2J?(5&2@C}NR)rm;T13$VE`2G1Hg{C4(P9v2in`TC!-y}<1<TG|O78bGvN1KqQDhG3Fn8nSU4QGFdfv5>i+{LENfcq_1e8JM)h2G66+l<=J{*+)CD${c&w>SDqDp}4<`O+|E7*RgBPpw*{9lb(~U5UMg0U0;yI-u@CzPXo<3j7|-*?>GQ4w?vC3f%8c4&c$VyK<L57cO&=THcI8e>p73+(h(s&1vF|zww`c?Ic%NH269ZKIS}=<Y0Do6BN8<!w#}x$)^0wYafAL@wnraA5W(TK&e6$qPM+D-V^xvb$zDqe^P>E?AYB-ZSYQOfdwc(ry?t2(l{o5j85(l{)fqu-bgB;j+8(8i_@yZ-f-An$8hTe2sbube2k+D);*WREbimrJ6tGso=ZM|VR6Vk1Rn&i%15PYCe1KGbm~vI0)=Zz4U9X_eG#t?{g3w5C5X+@OR1vZ~8h2pf(_h1K@p*_Wo-*e3^n6)G<#z;0rbWwjZ)pi(f-43WH(m{ZsWntTkVx8kD_?U<4b>YG-yM4TS<W(tGso*YS@8MoZXvNO5^l4xzK9TRY7_(fV5Ou(+e1UQG0_`5Fa%?pT|iPCy1d#z!aZTJPePYy)VC#RKxWqLc`L(@K1gvg0P7BM^#{ugJ!5MtFwWZK2bdUT%8WK_xYxW1Uyms+JoJSU6REu5>?9-ZV!*^Qwl0KSe=-%VTu|V%2oX*-M#B4S4%Fk9HDuE8`w}Rf6w}=iY_F%#A~I>_bi0nDKMC=>ZwQNy?!$_bnY9J?y<d!6p*{rD8H`;^KPo!2#j7$<j<)q(v>HXeXa?}-{Vfo8&F3H7oU0JLmcUU73jGQzjuvc=4vz4iXAgd2=V7>wRxI=+)8Rq1+=3WCG`{2?A)jR)slZx%=wL3<9f>uuKt0LCw_S`&BRV_f&)U+>-9kXMNX|Dc86~vcD)MU5xBHD;<$V4)lBo-#mMUb}@Z^$bj9I(Y{SYzvi};tpM-<&`NM-JiY?oj+{UpbGAubApi)`rXL6_j2nyDQ@S(p?|DVg#^DEew2K;gYgl_FO7DP2ZhG{Gp%N+36YIZ-IL(GSQI0vllv2n)gRIcuE!?8FXM_8CPOVik8aEVOWpYO%F*_LU-X)jXKWAdVfQYS;o5Q@&U^O?z9>6rttc_?H?=T@=_mz^kbi_?O~_T}Wx&GdX>wUGrFlz1@W%^Tl{|aTjLpPY^VtI2QP!rM_%U7l7RI@Zs^S=osj=RGsj>ZI3A&942*n?exLAZ4WdZw&z0{^jyq^4JQG5d^&N7MO}`}!+gf5;mreb3t-u@eV_4G-2ZlUA8)@JFmzmbivsSzK-dg)GWN+Tp_Iu8EJK|^_V`(+2SWu=?tz|XNVp)AEQLQ`PfU+K*B{VK#~@}`>Ht%+nJE$XGz$XS6TfD^wL5Qqf@3%TuRf(fb5qdlrX+!ch;N%)L*>0f8104T^v-IO{^6T(?Y_#?F0mhw%mytM(^T0y^w_lP6XoHU%A@eD7bpjudDZ@Lr0Bx%LD6Jy^stWP#)*A~ri`@d!SkOS1MJTYK7q8_W(zwAUEivi*#5ZMa+&GJCO_f~l|;na(8t#rv8Unq#(uXFN=yx_>QC*IZMZ>fJ5Pp&2x*#4*zcYwH%N!%TBu-%wjKIs7LgIchAJao94kaOo4C_^IcNStg-Kf;;GQ?j*rU>bSX`esKwHc;n?l*E4Gk6cNoaJ;vPI!Hb|B*}lXzQ5wl((+fPa*o^DCIqRr1t8+Bmv0|A>Y2tV9JtP|$@cm@xDM0MrQ(4x8@Nz6bc;?G*zcBtlJ?h`7gp4>7laj)aKjm*zvT1l%23jR=5}=4p{GJzi@s|B2qsmU@2n^#OHsPo1fA%FodI1@OR`$?iZ!#fTJ1$L<ctQoqqSgxUA?$7d`uexnpr4W}ojXG9eO<)ktc*dL@<5w&Hif0oBgT|O1UjtY0co!X_n;x3;|80dC=P|*c&-Q1CRB~~q15bf8;jygm|Z5#$2;+gW<yRNVk%=6TRS=~Ou(H_>Tcp7p841rN^7`)HZ)n&xOB!{BH+;qcOg``Wn$q16QDxhKyNR;jSviUCSkl`Dty37iQy3LTZy_%-bgXDr6fta<tVp{Ku&_$L*7yZV~^V%IG3=-?!Ts*KbXcL2iwTaS1dG=&>I-3mwVcw#~`zCL>)tRBDu^ycA*LOdgW&}H6?q2HleY#EsWF~evg&{RY&D38%2^8(@760wooM!bJ+7&GtTXjbzl}Jx>*)W$mw~+bHX#&6G$jEtDB)VV*fmt2alA1<pbu5jut(IX4hZT~Eae3Z?K;Y${mCK~<JE~2ON7H{QI(3y*AwOKzJ~j&cdIxyN5sBfBVj!WTCsGzuJ!9%WA9HJV;lTH8@`At3;=hn%JnN%$#{b?o++f@#$DF^xP`wFUpxzQ0Ro0YoE(r}jTGMqF5Xe_N0n8ydftoq#=7r0X7d3XrwSz0zY&|ENxsDtTV%aF#HZ5WKV?UBuZW=y%2!wImGg$CJaSPzF$c++0@|_sQvY&%H9r-Xx`{1&y{>A63A8)d9a-$&&u;>9NnMRS+?9#~P3$3{QgccNH@FDN+2XyE!y?~1G&~FkFb-z@l+gjT&b(@shzhuN1J3(v##H&31XyAanj{ig(RUlQlHzY3Fk8qfi3W>%umvos0R9D}GgT?5)(fn`iiGVw_@Ojg;)Nx9cHhc>ur&TQb<lG+S^Qk<3ZB=0kPV$Wb^V)}q$3A0>kJ!93;z7mJF%8y?CHZkAgp-BR_u6D8Vgkh*lA~hmxg#NO_updeTJzj_6%RTv8*T#L$2}zc#JHRtN$r9}QlzS>o+I?x(g%OMxs^6L8_}L2!?OG5-Xwd7eIGd@{K`T9)=XnyGPFTYLxxJK%@77M3~(?S{T~J}Ow?K|jB2)9nNoY?Wi5RxB$Q`0>0Vq$sK!&Vab52F^xS=#uhH(2a{Qmng;U_Nzjw`kX(}mGUak3g-0z<&24U@Y6%sU+Y*_=nx0bkq)4K*G-@O1o$QR{WlRfFp9H(nFj*+;G9W$Amd_bK#6DJ_M^BsTZBZE{6Ct)_+nH0Ox&mn?y!TEwV29`Z|fnAk*ph(n+|CJmC5Cf|5#9J-zROU5&<svE76b~#o7j!Qr1=uc3Diadr`U@!`4+CZ~$%(;>kC!crW*AsIuVWjim{{PU8@v5exOetz6HfpZUX+zJxjAe&pYfCm>qqA1^p&1zZOvjP>vxF43kCqsOI<E^;B(INQ=6(}*uR-F0xFLL<KGT|6oxiW!{pQWI=vWV8jj`9LqM!82eyCHJgs?@-6lbmUhWcYB1E`Z1Clb>l^H|T=#CH@lNpM56X>6G%?9+qf=KvTT~|Vh02Am;rmm#DQUa6Ik15`Ttm;SHyg32?Ljz$(UjwDKamE-|`di8fT1Fk)1<)9rOyTbp!DtTe%<Jodbk2h%GEdtx>H#$TsP3eWr2^;Z{HJQVNKiR`d?xpbIhgPK<J<I-Bh6d3pW-<XOX*R*&%v}biwM$O)ktQl!6zeKLuP`QU}woyXR6LnI7Wnrh3#^#6NBQ65DjIhjcIUWFC$5=LYu&73vM{It+s|`#M-}sYSfkoV-7Q+@&}R_f7$kPYUwhp#WPSAbwu~I5x#>0?KL$YW6pEU9@GL0PP&>WlQ!+9*r5i;n0|0{bMS2`#~^H&;$d;}>;*FTVY4rDSQATK{knD~ieTcX?{Sx7>0r@yOUc)e|E@pPG!VaPe!xF;fj}LIPh;sf1+%Lu<g=UoK$2zW_Q`w<&CrrV)*CMog4lp$We4qy)bh)WAxF(Z29&7E);sjJ9YvdL5mQ~z;HtV@-;*9)A{5_%o<-+j<n`lc-{hh+y0LvR_FK+Y1W%XW1l3n0yX8Df{k$jJmr4sNRk_C^OysMIJph*62(xKH|477i@wO)BqAY-@`stgyqp2XY;y4VU<63DUmxtFnUg2b-4C|T&nr!@5#I#6KzD)2_M)MF@uFjEfb#9Cd+ug*(X88gVC1;PnoeV1sC*I8XJ|mEN<&7j-JG;2<&WNaB>|BB;*`#*7G}G<CyJF6VbF?jUhnHt~RI~TQq=Gk*2r5w43I|eSNYqw!V->!`f9l-gPpi^`RbG(N7rZcHRCCdjVRP);(p?{17s^va<nPreg|0BlcaQ+Ua6p32%AE?R@^MEYSz2G;U?Pm6qG1g|#(wLpQazZKi(Pw}jCiU=!lzsGu~$td6OK>^Y$GwMUr2u!t@X?{cv$MdQEhvVP#4nAjhiww?s|=9D)T%V{N`R0OOLS{V_5%al_q3KkE<Zxs>+P7xJc>^Od$pH)x{C^pV<)RC?B6}q>plP^ps`s8{NxT^D-JflI*=(VEP|0sen}RX(<%6dg+hqwA|7D6Ryh-<6rKVung--Tzax{o#UJ@h!`L(P2EHjc6C`+lA%XqD=+U%7XtL_AvdDmMr=V>!UHlU1RI*h3Q=dMa7G33_wp7oPpqZ*vH@u4IRnP=WtF!Q8!tR!s%)Y`jHptz9rNxh#xWzNB<LY>0{x=TTi$59)WG$hGoiFlrx$dk^_=M{@~FVp0fzX(SS^%S4?<;p!v1^~Kv(#ULB4j2Oe09<(649$H{vs`HGNfX#yKNnZB>bTuGh@D>{>E(Bw+~DR-(J=+Vy*K7Z|60?Rog)f37g{HzEXkPJ9#|(P$XkkTQsf2Ff$rLvG7aEY?p0QO$r%2MZ5}@1G6k%MYZ@eXko@7Sjze=|DYYkLf1xAw#&;Wzr4p#?6xzI^xcG<I#u4RLY~6ESt5PkY2sN_J`x%gM%m|-tY`Wa?&isHmBJfL8`KX{gP{y9Dr1tz3!XgVJr;tMma-D!IO3DA0r0HVX&xAi{vqhRri%l=4Pc7K0Pv#9oe|d73E}S6`jwih@oLJs`qon)s+sy5M3iREnRW!gt@KBClTh}>WzHCq%}%ogRy<T*{ilKr}oqc%Ox4PsxMsXgT61MybzO<V!5Tu_`j89NM61e3bi^DibGFdQ7;J1{+tywG)@O}5|>`QsT(q*4mlVoHd%ECKXoTl`kdc^PWn*3`iDAkW2|Ta9T=&KLGpf}VKMf3FgUeniLo*{Ay0{=;6mxTyKe+8NK4-gx1M%EyACxu*YVO;e#xWR{0uUj?)OAHQO;BvO5+Y;$_|`Qj(r1cs`sSm5N08nf|_~Fh@Upsv83JJN5~Iw)7|DXGg(Pc3K_=Zr8a%`6|x<4^*q#*U*VV{7=x9tT~6-I^WeV+s(olSBFTHO$+u>YcNV*Ehl;vNW;fOLUEt-ea6p7J9yIYKVS)-?a}4t&>1BRvJ|k7=!Yo>O{<b+OJnZs0$KD~i5iy*4QkGbaV&b<hzla%t>FgaJLI);n|3$QQ1b^4^hfZ!MXg2~=Z5z)CjeL53g+sq$mFjabn?Kaoi{m=S*$5bn&TQN0IUsTHF_?RT>IAC$T+!S09Sbx}P=5l4dU%Y&FnnMxny|tbsiCq{&YctpAoNawBXvAX;89Uus_=toyL?eX66GC%$wL}<;Q(w>Kr{=Nr$uoQ;Rg2B)*@)7LvkoM)Sv@ufu_vadtDAkdr|C|a?(3;&H=m;(FM8$vQYXy*z=PEnw3uZ2h?KI+?Rfj4avpP_)GH(X07wx(ggHxoOHeRsbylU#qhpBI4Mik<i?_ATWs!RYb1U%k9I$i;{dz8cGUX!cox$A>Vw8Idf(}BGs4Y$Y5O8X(#fxHlGj$SJ8R72M=Af~9z*eqU1IMx7zCp`&MGQ+C^vVqd+6$C6C&#iK85sqv8m;bj@bWoBrfcQX|)=v7>eaXm7Y(st<)%PD`|EzTLswyDtnkb*ossMdAj^K4;*esVHFL8C)*W2kt0C%3S}!8qZl-+PaC<b@5wn%i^AmWb-*R#9W!Ne?#PLt3L%-Aq23{0mu`Ix7D6_`w^Q|Jw~)maj@sqa(P`~SZqJC9QaiVf3VVYH`RKCKc{Dd&OK~I`yMf!&r)|r$>sb%_Y}6q>Ava7sbGYHvThsg&3}0v2XB(w^u(9r&5162iIsCwF4t;S?dKYYRCDj(avn^vG+)J(N*!B40Yhf5Q)bxJDQM0f_SI)Q(;^{aJ8o*2ds0I`d*&r5riuTuB_eUMulM4fx5J7sMg5C0u2txV0jY+_?EBobJG4Paj!G{hT;(8RPT(9`$z#B0JWL&fr>Eaeu<D{N|OiZpWZeUPd^tvPBy?GBaxn7Dv#Oa#=q%pTsC@?GB3t))qVe9)F(Y9R^S?UtpaoWxZV%)(ye>}aY$)_MFsbC1fuTfuwDMrIGc9IC%;mF9jQ05}RiX>W9{Qf?qawEa{OR0@Dh2lt(4YX?08hH8|x#R9tR4e`0m-mRa>sEqY5ujvB9rN9tmy45%OPwP5rX^MntY5>H*7+W7Izx7Z)p*NJnb<keNT8qRenu?S7-2xDv5Yu_6{N}L!ayRPu}IM2Hu`mAaR;_fDF2(eADfv$%q;#RD8r;3_#n*-;)=?7%brvw=vG4{bRf#vbm$G(DLmEnxJ(>;aY(L}rQ0%pWv4!~#j=ccJful4o<r#v)?9qoDB=)q>U=tLb_FAIGXUH^JwhbfDrkTCcO;TAaA$$Q*UGlsuYul5EFl&v*pW(R)6gE}oA?VFp#HO8A_dPL{df-B7Hm<Tx{`F{v!D4wx2)&_B^m{%ALkLD)8|Nb0cZd+%sql<F^v(>RW&VELO;7rY4<OKQM1Ys_^KqeX}!V_dUkb65Sy;`IGpoHl-gU9Ajel6{!?iWb-Vf=^@^D08)4gRz}Ma<HNguKPzb`zQ-x)esc*>$1Mrs;w6un&da^>J1vXJi>MT?$=OW2v(?;)5L%yDv0r^OoN&~dJ<7`RcItccHrJDTiFtt&FvE0z>S7HjOIw?^_djzjmS#|L^PR?7&g^rvdykibC4&1*Umz$7p*JlZM*Y2M*#EAwncF_8A=c$f~dtT;|#C6t=E1#~Z{}{1?D?MH9wkE3<dE16s8ooed`TP{`T}Pz$Qi<L}g<r>`Ow*zof-QU0?e3c4O+yOb_{TYJ&;jD0S;FwSmYaW;qGj!q>YW$ED)~-}%};>$=KIsEPQ#d$f7Ym|mHfQ^+$3~VtaGZ|=>M<64<7U~B|o3K4WYzUda(q9$37z?=m?xJj4gIDYQCpr*UuRF*C*V$*CX^k;H<EeNGFHTl26xTW72=0gO>NFLO=vKKp+nu1w<8hsVy*I8_9^2N;ko#hOsG>rHtkGabrQzgn*$d5jmVB$Fq-r`rwPQZ&OnxohlCKJ^$n}T6?}unyVU@yn#mIs|z$WYy)FKN+X3XN^jo-q#c3zQ3uRFQ44O8AOD=d9JnhQ7OM;RyeH#Mx2+t;^5N*_M7W?~J1#ry@6OTv{TM@<Ix|0F%brvk-16#k-ExHyQf+^N6RBpRP9~oQP=#z&HJpVM_g^%As158~wan1k3oH>JlkXfpN24l^JAz6KBG{Gc(MBfHLEAB^NV)ZE!k!b50SBeyZb{UpT(<tUKfRs^%AerEuiatay#Kwp2VXjKze!NprJKQu+W5XTF@oeMMR%Mo%|<N5PTfvmOex08wLPhIo<tOx*IQE1{(lFC7oNU4l5X3Wl~Ya~CA=__EGqy`mJw&RSsm)ZUs6%?8vl+@YuQmn&tR)T2OtLw-*eXo0c|YGobIY6AC$4tNklpy6R$OZ*7*>vuFl-g@%Yh*2*{0(qAGStAT+PNGBPeNSMDco6gX09xO+VlfnSEyVFFELmmsQS|0jpU3;gE163UzD74QGl<6#?ejMy^3FsSuYBg10gwGM?nnEBD9Dtvd$kjzs8kK5Bp-e6t-A}Oc?`eUCx2yyKJkjkVZi<RQm@lb-+;|Ztz!Ga`iAdRyLkt%V;&TZXfm5=={&z3wM5w5T3N-41hej}LEdj)G*LE_O7z;NqKeA(eA+JK%6*YcYFN~&!0P~sO`$U(A>W7CX*nc%XbL^pAZEoaLo52SlPfbTZ}+!y^6AwaIcioXS^9Ks`4vJwhDw#$%AQnxS>hlHW=G9B~w)L~f=1ZlTVDuA;Wu=Osn@Vh83RdMX&kcr8XAM-R|VkeEEyXVnSaE<z4PlebEv1sn(wIJyt#5o-Is|zDPW@6U745L}7>Bkt@r8F(=*z{F)2JvsxiE#X@-9bHNO_)E_&GQ%;^L%kg`_5r5*H&ga^Yc+R&WsSqb?k}lZFRY`opv14H@DI%hgzf}cP+W#qfgc%Jtln}F-#pxomt>7dGG-|ncO|)ZtOeook>3k3#_#lj|aoL*U5i25k&-r?)T4FcRNp2l$zRp-`f)-o0u|zlVtig@X73czdu&~1B)dfnBzyM{H{Yk;I@7JjJSrBdM_Hs;|%Xw3#-FqLoI^3^m@0JQAI(o3ZCe@<xS9fT>_3olG8`Dp{*9L(rp6e)PEN>$?MGPdA(P>t|t;a?DuJg+rXhxvcu{}PCP9R|FTurt4`1xT|8egP05X{GE_Vv4l1QFtm*@04<u@aRFYkePIB#zHh37@X)mMmGi`Nw4>Jqrt#xKK&kRfA@ec(*$Kzz|Zi|=@ksga!%fJ9Kyfk2v-aK1k?M#xxrXHd2xP^45ugj$@dpjzRHAe->sMtV`6{=ul!n))J;BW`ztki3BG|Ud=O5N0wqlYL?5Iv^2v9)$}=i$1Lx@jjR>xkt2jUhAxKmzx(M$tKPNtq^JE6<9htV^aHsYf52+AN7Cb@Z{j29N^=vGFr}fDXL*K>1xdLE85h>{?N{;i}to#ZTQ!`Cg+S{Ry>voB{zjpf?PhRW|l!H>)a&rhUIV%X?HT81!E+FTUC<doh5J3YWQU89jdEV`F6ZZxh4u0@%R8#jg@Y*5R*=lTRfPo61_Y?QnkRMOspXN5|G&KoRJ0IW&$#`gOhmk#U=zp>4Fv+|>Fu{`09OtrJJ&>9>|Bb)f}%Y+$n8b?lH?8TJfw`c^_AOCWw*ZhMFk7OQ9<30S|4G#;f$Sj7cX2O@je0jqpsON9raKF(Q<*xbb-{wc@jq$=nWfQt$g$|#@Zykiv(`_#Dc_>mu0P7h0o<U$hc!?K@Yj@!Ox;uJWurj_A4kXRP86_FObZ6?-k$~-V3Iq4;3M2xExtSaHXOr>6qBnO&3ObHAt*hX*=m|63|2XEvq)8539t3aI{L`F$cA(1MpF|Vut-E7{>VkHj9dVp@PtAa^tYj{<D_tVEkD>lyv+^pSur6DMoU~3a)+=us-d)T^v^VoBs8|Xf*t6jh{5?(n#A15F~`Uyts0w>l5@rTCLt%}{_pp#AI?E)9_U;<NZN#X#5l@ZJxMaSYS%nra}k)#7*@IG>zr)>iysPI$E0SB!qqT<CFgr>Y%eK6b!r|X$};MDP&=od|!GTxHDR$j^Q$M8olg-@;SQ*LrBV?BOsd=C>p&DKM-;mXT7oqOV(p;A$JaSp3Y4Uw65+Uv}%6vmeXUF<t2SS>1ZF+UdnUJm$wx7bA7ZC{cnJE>SQ6a{kGVnBba-}}-F&))6vD?1sllP!`v;F}_!?OO0@TZi2!uImX7C%s%_?&VZp*Vszo6J`kq`^2!K5GcpK!DR+2%8yeebTlAz7?|I675>er>5Ss#9F!Qx4_)`V(t)BWH1|&;k6*t8nd?uydC^9$d%h?D4Kd6r=z$l6{PWmmprmi41+(T}=PRd>5FRuizp4(?bgIzmODPH@V{uj2Z{GQQ@I#`>{{i!f7TzjZceF-$^_WoPEDf0Gn%;R?lBxc-j#`i6?aVY`p$OpdoxtIrqjq6#Svc4xMVyU4kj|$PV~^51HLHIl$<9@KCCAKkpPm%d2$j_Ck%S`^C>qz?b-2pjpQ29{I92=605Lp}u!a<9lX2z6v%{bTEN+pX=5S!O*Gye78MWn$I7eE1zM#9Z{Yhot_r0EZ4CZmT-K_2(#9C^eVe--t634d1;o54%=oVhwafuMXrw^i&01}R7>KPg)DqZ6pGC}(wnB%BRCj+vOSyp6c&>=q{T%Q!4Fx;GN5Y+Hs>bn_M((lZF9>EAgkPjkY-4%=27k-@mdTH~*OiFCwc&UZmChfmEY07__N;``b017H{K=VeSM<&Dg6<JX*eZzvL=v&C@j@&t<e6)tCYG`;usdkH=9Fn@d=G96)N>eoS)7+$lRy0A!=ZqPYW_&<)!xNG`H>zm)PTT=9h2#XJNQ#}0#TS?Oqu2Y&H@JbK8DYfY%{W<m#~?;So=jsWz}NGTJyk|{=B7LOsQH>kVlB-AsS@Wk8I%O`DK%`W7B_xk)lOTNM~Ub+l1tI-TjJ}=b?QySRJVTcX+==B?sP!Q;fxEq)ln(~Z3m56Qx|o~ObBpASkk2zd|*zAy(`mEl|5pz!7kS>a@f8Fgh|_pofov0I-|oduA&3W=!DzQfY88XH2uOOi*Y8`?mH+%#>-QttX`v+6T_iHS(WQ1rvI^M*<+e@ZF9SORI(pe`zw=s=~A~%OjAGu+Hn612=s<hUTu#Oi9fY;iQ}JgQnR+hA{qAtX-56$KVw(sJFDB$D9)_|uzWP3v)!-JR9^+o1gEO2*Wj~;5VNP48GW;hcQ167W?GeLEv9ppyC4<SAl;Jb{W4GYFk7>CU4`!R--$2F(D|L$$Wr>MjNDSeVCR`aAAImvx<{m$X#SV~-F^X&O6HCXsP0Ozf2f@N@rt~OG3vHBgwXOMg@>;inESV$2#ie=CaPTBL2f_l0M!}{H^r_}PBMk(Zs2Zl*-5DcbNy@gXT4B|N@qjkfO>Wp%#ZR!?_eOE?0O5cUdyxLPY&F2Ive_Pt(6+Ah*20$@f|nzFedGb+G~RB>s@UvAQz6PIA}-PjSQ|0-ltk4r%zok`d>#>Bc$MBZlq4-_Z4eQ?Zmg?h5OSG*ZZ&*grfWQ_XiT73i>-bZX7lT55&`LF6wPlRQE6hv$l+yh4JGVUHUGb&uqQKdsa9rV}tU!4)-N_6N4tCSO@vMj$yy&wA698SuT0G1<+&TU_lw23s&*PnwNYIc$yI!HCiGXWCQ&tF&e`T6IJVXB~F+_r0<X5FO??J1r@xPm>5?RIjoi=0P7d1^cFKxovj^jvbmCS*v@`o8j_@H>a+FO;&56&nLm!m=LmOHT;1Uyvux!JbOG~kl5mVc+!nNED)CBWF>EDw!<=2Rl)SEiIJk0Mbl(5kqyN;aQT>jP_aN6Nk)Xo(LJk*z+v+86U*5XfQ)FOP?*5BR%O%S2X~+hu>&g_3!|ZW0B;@mRK+_xr-6APPYAA=SGUo024cX47z&}a<Z$ywV@70io3CS_j5L2WSPj6sgyNktf{uo>BdB-7f>d8*M@kfTt%X~w?Beef0o`wW()ncTP)%zloX_Ai!1?mc#W?KYncZou4YLouI%}qE;CVknI1UncYvj%-4q?hbpo9IH0&(r{{bfR*OTT#*>@rP?*ucIArK>W6&jVj^~p$D=k3q8yTu1BoV@Tc-naf^FUA0D)%$B|#yELiakD4fL)HoXWTI;!7v%QDM;X`<TQ)dz7i7Xcjiqw_Hya!#D6LJ3xCgRdTK?qemHk)WXUiulujpk6#H<{n1XiPFgHx&(0#Qy;EBuER2{y$M{G1}u=X`SN(v?n(5u4|)48sU)rDwP=A*B-Q+z{~L}#f$6^NHfFPV>QzM(mShe=v$$_Dsw8$wmAK}!bwIWZkv-aCd=_m7t9G3@{bd$>ZVae>J&x|LPXzmz@(c<!ApJ}ZPOnUVba-lVTE{$)DQ>6I?;Fd70*=fUxqFlQTCFUdF7^;1RXWE;0<}lAWt;xtFA^5S^F=L)v+uHW^t&cZ(8{yU*LE@*%JHYC57ErI3IO8Uq4<Z*Ct`V}RH!8fRsRa0*0z@wX1BdIaMp*FA#zTQMA7Y%2Zf(2Qx}DMK=hu|$hMslNhrBzu=)ahLs7b`?WYF`;Z3w7thF2D!%rWQ-RI>3w%WncHrY1&h8l%5GzZ1FWv*kCK1l<xb9fo;HDqu-MHJn{_TC-iob4BHT&XrtuqI;*U*Cz;=jXw#YHlZ0wo0-YEnA8?=0A#>hsE1B_;)i{-70ozX$Q+L+c>w!iLj?etnjWBmx?7L4}P>@^}P8O1y=&wd8P%fVAbv(O-C;nN_+a_UXHpn?X|ys7)7;KIgKjmOALLOx1V{QK;e+duVZp;nxqnUTeXWUc&7?-|JK3IUBsTQ01K?dmZz1weG3mXWrbeWTWuF}YoY6oU?e#kw~E&E^#9@?d-PCZA|$6pFZE54g~>GmxNZ<~_4I(BCgclp;Jc6RL@!2({-@diGPmDH3XWLKKu$4=<}(94DO}wj)?=J<(i{4s%-J@_s3G8^!4B(<9f;HIW36jNoz`w3{Pi65{KHve|K?|>bD@^*@E#XW%`>C~u;b7T!=U<EP-J0iG<O&})EBXtbz;8n2EdcRi|ig$u%E-?hizdP%UiYphG9>%z(10#AsDXTZ|q~bXzemu`e&B*3mU3ka(}5W|BH`+%uH+fo4L2>B5zvN``OK?sc!M8ZJdW&S)`Yr?{==S-OpupSL4R>KSg~0+~5++@`M1<hd6xw6(3O*TdR`FhQH;X=N)cGo&He6C7VuHw`f}T@Rt)Q(ICR}=sFr;^oZ&dOTan?lg4~#xFl#V&`Gh)iR*{VAvc<NI`i8HbPrZ{79oNb;3s!5D8ha>JBD*(*1&YMNH&<IA4uIw<+D|^NogH0oQTDptz7|YjrMk<!rgzzCAq1yzywMvSdA>UtiSw~`%Nox@=5gT_8=k%`7#Hj8tPDMmWsgPDlMi_R8M2X>)lOMw7?sk?^G5k9H}M*1N#G|SI5sxJLlb{z4@Lhp!=anK$eSmq7!^6c-(GROE}&Ss1t?>wL)g09lAa$k|ufHJb-)JaIfW_lzbSDSxRN&+a!jfo!74R9%kPC)sX{Jdr3)b%}m|<o-J2qND-i6-m9Jy5D8!$Z3(LxFl&1*<>~w`S`L8pzJ)vlvI8>`Um!9b(}f!KkBR^l!O|ZtSyRI&Is!UtQ)=q5$IJ#gTOnPXe?Ib1f1%MBXOs!t#t1-KAz|b5Mq1e}b=lcK){fx{+TBg89@Lao{<?HsWnTV??<o^5o|g@4KnnTa0v0h?Z*I0gcW#iRR9iC5MFw&vbjjWsp6iGT&Dm@s7XrtaE+uW-Cd%S15oc%@C3U6zo0CDRBhuan{x8Y^tA7}4=c1S?+HjpZSQ<x+j63jnx)#?op2{r$QM?@(P6&G(DA|ZHTio+akVrNJPCP7R88s3I#C6?`x2k=$qVk$|F3#Y>0z&>uBR=-+e+h5+w7#IiB^0U8_FORGmwKc`ku@1z>K9if;Hn9S7!4CglN}Gr#E#VXH&H<yd2=CZ?s@zgo0XqDnp%rAa8FGn`m|7f(kLg4fkE(~A{@g2!D<H&f`w$1#?0XUp|S1Q^CVt3)DB2`af#H7WTA%1%-GQYF<O&=hQFcSh2UOljmaoO!8AB~i{p&Eb}*+-tCuUxZg)j89R|7z(t+TA$QYa4Fp5m!-xkoIJK-;hWxWI}h{P}{y1J_miM*%r0;k9D0%gDWNss)<9ql09rhR2E7;`M}i{ak`LU(5!RXGiy>D0jrRP;Mvxi~Qhu;jevg{|FBO3r9Mw>y<dZR=6hMFhFd(V>JQ;$?QC4$&GujOj$%ZpmwABa@1pOtj|L*<jQS>|;AQZGF=NF9<fp-RGcH0TouGmT@r$AK|DkK53-{lw~?kK(p~W3N$H$ZBpT(@Js(;AK2ZCQ0b2>M@_+%nX8yj)r;v@>UdQrhe=vzeW!FHeXKyIY$GYi%<-ikEjlY$K|#ttKW77Uad0_UMEEdkc>nnGQSdaQz{oK!IGyOEkX3l^^AEo#`}&Jk!s81^Up*+xpEY^3Rymlc>13Hu2o?1&|3B~vTHrm#08ycPv>D|cbbhr(6M+>0?bGs<yBara9%E%Pwza#UWV{M3pHkN|qp9nH#3kMK-iHR=Rv}6YXUW4xa$<4OYk}wHI^MF!-5iLL!9hw+Lm#zCr+}czgV_=I?hERLLZmX)qJBOv++-}2=5wdCkuD7twm(|AGEP<k<TkQ3ZRM#*f4O1=^+wVqFp5WvcAMBa4|H~JU#_>|n8ozXj?3SKMg}+M#_GeD9vwheQ`aRVzv7CEaer)FiSQ>cXd2<f8E}+X>fJ6SxuM{?XCw@tj~^=vFVcu@)1Xl%4EDF;IV-rF<-ggkC&$9i3H+I~I>-w$FiTi~{k1r6{qcl~Od$KdAO9)uVD-WbuM+RrXj`MX6Sf?5cIqQW-ePc5Woi4!8d@N}SJLbpj5bp@E<pH8Q?AL%5I)j;Hhu7qY8WbggcAVW1q4Z}FnH2={}p=d5z)VxyHS)=<;VNaY8g@q-h$_ue>A@Uv--q_e;fh~_fi?AB3TChG!`>evfZnL-rST?r<{6lABDrhkx?Qt@=M{p=Oc%SRg330(05OadCktR4BlBx)=m*g-kzF@?M>-M6L)=7`g}5cCI;ReBp;!A*hv|v>u-eXs1z_^;QKnHDrgvl?hcm7*OP^5{U3wbL5X6Rt0P!w*!C7vFhIVmm?04mU|+GDpxlV2&oj)jbVv{oz<a&&)L+TPY7gE>@K@a$8eA+7?aR+L#M(*}N?6F~ELfYpAlk<AXjWjZG8Inrjy^oM3GEp9gm7!<lh7axl971yBQhmru(M&V!>_bmNWzL&;${H1+S#FvMH0(7m(zwDt{iJZ804OnLWn|LU3&vKLFd*q{%ff4%sJRmbu>r-bouuq&yyT1a{#g4^y;QRyiSe~skM1!o5CWOl~pHkls1cn;Eu-mfa1*U0DpXrEW(XFX<;(ku3P(9R?2<wr6Lil4t;RRT=_S+GLCx0ru!LH6`yG!Gnx!uh-)TvfEhk8r*d74k{UMhFcNj6Xap%#W3hW03n1p*!-<;L;k5_!Aid!sRSI|{HZb_z#x;1@wH<;EKl^6$^2VOfPZ)lX4Tv*9lAVgp?FY}pD1y2$P@nL!EyT`G&F)`2iU}&|(^5I5Lg4EW6}tqW556nBcE4dW2{-;D<u4g+LmuL4dJEeL62d{cYE_{OEzKdRAa;bcY>9Z#*&OBvhgp7?@yud1$VGZ9e5f)=WMj@~+NGj~@%IleYNsHubC6lfm+g6mW3&$t8R+@*!a+w4?_G+tCK}2jyklG?7W$!gU@fI((6mjcEOOj1=X7Y~u{gHM%QNnH^qP)2kAG3DHOH2u`VSrzX|YwHbAs1cqQT-q%6hL}-aHZ>Cp>Rb;F+&~c-9-FMvVXa;9@9n$gFp9r`Z=*8!`mre&l!9^u&Q|H%DIh4p&nuLYXX6pA0)JuG{F?%D90Bzn6W<Hzs=tHg+<>7L(gC5q}vKH14dQbxzh*IF#AFLnZYQOLY`?K-DXuWNAy@mvv&K*9|#scbKx$qJGI#DGzViM`Ni_FZORT2GpG*bliE(dkoj$Ec#1Ay>A5>yHWy?j0cdR8BKXL9K9pgJ4Ep5^}`QwSj>vg=ISPl?NtK9xmsR!<gXh!ae^IOXC?tWJr`$oqQTh!C3gxJZfO4sBxoVjnDq8P2s;!I5o;S1j!s51*{W(~)40rWrxK;z9KOP8DA^<C0TTandQ5S$Q?0p5grAIrK?DD3UzL-S(&lJDMxGoh5He)^AVdA&?3yfC&>ODT`vuV$f0RX>>L4J(2{!2&AxgZEb31QyY|zW&|IzHAmF9amo>Z~mgK6_Ij|F9_jxUsHDPV?Pkx51DO$NP|w%20aTCEngHN`9prtfm{`>(i|b>W$NUTJjXJNA~s{mf#fiu+N^9AwfN>_qUUB*BTdfoMy3GjGPZigdh51kmNz1MuZogOk68?taqYSnhX6d?bh@m4nQj;-YpjrmG!IySt_3Z{M4e=8%X_?$7F^w0*#u*WSXTV~`ge0B){yQ9hJo!MH6V1BU>T^PNdty#(jAjSLC<Iy{pXq(>DX6JvqU*4Wp_1?m;mc~<v>^E%R@w-uma%Z|cP0gtldmbPQ;@3>V2U0&?x(6XBh(_pAqC$Yfh>f9Xd4DVhAMdePjlIt=*>IDdOi%lRW8pyoK-n^F<8RxPvs7mPKnAg;{&XyU2OUX%MyEuAU^N~)tbb$oi)^lA;u{((pm+y?AchOSr*jy|eKXzZDQ!xu3Zs<LFzX-b8Oyl4kR^rBp$86}#6<1q|biHT>f<PvF;{%v@5Tl?SmKe9k#$Wrr@%1=FIU~jj8=J_ob6$xq;(kDxi%zaSlM2~|m8t&gIx91RJRFwJJ;kTITj<$D<aT@}+@uRn+jBM&QPu}}uw(vARXd*|SK`6Rw>7U+Qc>_|yP{{9-gd&J{^M$n%Ir8dWx3km!WB&~%Z>}Qj5+tV0-DyiVnw_77@DGlp{)4KwPb?&TDZfB&W&(YM(wHT%L%#4dw^j1-}b3`j+}nyYiFSAqd;2b;i%GMX78J2VFqIH%1!`xOH*S**QP!6@tP=Lhj&&SAHl_LU|YTfkFs}eJ#gvd4>A0A15gsf<6bW)i4tf~W*h;7s4=_cs|Z0OxoXEeRHz=``cdw&6_YEfkowq~cvY?kB<u)t9lu+>(8bA+GaZ9y%C!2F-+%y(19cwI57mD}eBS>?*shLvz-2lyMmu^IU{FA$8jky)%WU-0#n3nR#u`nP@9u0y3exi{)tL_vKWnd1#7)$`A}JgVqe`3d!Ob=1niMX?Z+FoqI40J_$(4~3T@67ZBL%MD(pi`Tk+h;qaoB;j9`ILL(;awyeOYNDZkxRwiMp@6A`R{2_D`CRMUkJ^@G^{^E1KiXL>G@a^NgK8uuMK0duZDTq0?L?gT==)e~>PU+f3-hl*DvC>Q%oK(?HXe)o&J#R2j9c%-b+7r*?4}G8lmax`NMxO51=Y9+`F4z;Ob97Bo3EAA3UUv~PY6-O}#~Zw&|lLvc>)wB>jey=3m#w}<BWkP;}Q$+V+#&cg4NBGf*43|~ot9a@!xs(6CxyYhQj^wI#qa%P1I_IBeoM^(P;*q*=B@InWy@y}8+)oD12U?DHGDUn87tTPPZKNG+TQ@@y7&Wt8-X8nSlmtx}v-^H4*2$gYhW%K1_l4}n^+{K9FrjcGeV@@;8!BC%!bU!t)s02m6-)2k1V92f}f`XKYa2T5;)?L95A)?7k!e1FSIkb8gsi};mtbIm|#rru)XQ558nS8_(LAx?N+@miR52OA>l<#dzME4G#?O(P6Cvj_)d6MhVGQZ0FB}_39yj7LDRjww>7Td=ss|#g%$Tn)6`*gQ_3_<bv$0GLlRkh=mK0b+@KU69J0}){dX(|$DmkU2EJr~LywJNMbXjYQc*$y-7@l;$3Z>4e5p(S5HgEDUDsed7naNkj?IX2A%@0B8rb5&U?_jyjUua%s0<=Uup7OvS-@#SJ`@2wp15@fH_Cf1c@)qX^!!WL_%Po24z_`dGM^vuy<^1R2v5x37_yKE%LstUb=7mZgG#3o*gjAo<2PXAkeqhz3B?5I)Jx-Y>k*+~RhX_|l;3+c$GJ<?h8-8?I2q>KB+)NnoY9Gk&j`@aXeETi%yI<v5#>cX*IUZncgqxm?ht;n*Dd>&yf6Ors(8UpBLIO8UBv>jJ6CG_&gC*Sot!?aIRpKkbJB34xF<>9n5jZD&kUg-94bhr?&W{OefJmqlGDfl#ogm>DS)M---5Ggu!{W2N9{M~p+y3XFN-+*NFeONJo@}dGW5?tV<zTh2K)OM8&ek|a+rmai;(B}&UZr+EgDN)ewy9*|13(k4f=3X%cA|~*~=TPLHYF(&QZMnEk!0PS$NB7(Er<#7stW*@Uc0?cGUZ58TA^NL655_w)Q06&vD!7{&Zdmlde}9;oQ1;lko<`h0=?{j<k?b#tf`<VV?GW(ybG;D$Dpx3rz1mc@6sT1|ZeT6y5CszRhB>>G#<?9o_-;O;+U$iUkK&`!3Z~q$Cg(UIVP?-8DIztaXnU1xacX2uYr)dCLypa)K~oqw^mlfFL!8W{wlkI{5aY+wGv-s{33)4%c6Hb)meU$~A#+ORwQK?r;fRKx!>ozv^TeaCSWr7Cq7H+}6f1rqD$)J$fF5L+twI0T`ZAnX$(Ktd@d{p-mf^{bXZySWjXkYcv_s#^Dy8+dC|PWGIvEpWfv&pJc|r}1iOEKabcDkK6c}r%WNN#BtUcNGI}QDmRrEnU>#MQwUD+$;ACsZy_pX&YsG<C++NpR*#{q@aUkhG}t={6OO_O=V7P2-Hi3<Wl74Yln9hFZO)++i?NvzRIlX@UIgAn?KOM0V!5&<bd;QI9gLxlP}<C9`f(o%u2MT<pPC&sqiXAz!c5cn^*NqnP`s77K|M2$7x$Dg39#pg9=zYS4EHJi{9lXsW$;vqf%S7Av`;EHmQ=l@qtEH9`We(GCr4X5Zh1)qh_r`yt@yk>f03rL2!*A}Z0W6-nn2JubSaDE7yZXbFz%&;F?j`7!PXfa6;HVE>nwAn3<G?)v()E>zUFC8uYYpMQoKwG<05AW7Y6-Lwuw}+7hEaA{kOobIqJK0*;Jp>?G`mw1ZjBc3x)yrJb%rj0~tcUHq?yM7<xk?doici9hX^a1+S@41!ou?o2dr68U31@92LOQ=^n%kqGZ!s4^>XzW@<PiOOpnA@AAZQhk_X)D`wmj+7zuTW_Tq}^%`-LgrmP>Le$M+>CYTmB!0^kbrl&mej+I8f-aXn{8y-MB+)px*zSYSjy1}8;mal-w|XAbIim!AvLVJ15JAv;Ad8kW70*pdY31k;MXB#M=C6AQjta#5L-2jkWgoReQ|*_rx=#rAtU97z>z!N$;K#%+oU$q>L2Ja8DJx&(#v9C;JM)WY8Z5RFGc?Ez6ZsYYI$$z+jjj6q9)h5?XB7Tm;xPn+X2!#Quxv0eWC@FCkfq@oyohi)<Nu{2?qF0k{@r#Go@Cu#VbF!7|~BxZSLyb;oFJ|{v)B9b%#;d^_x9DQUQ+Dpe%)c{s%&Pep~&hLQ{h6L_D=BP)YCu`r9?;RN+{+U+`Vi;4C3+cnHR#MNJcNb!{n@jvmIRhmlCyn5T-g9qy^hHd{NYb0xQxPS^JAsrpFd>;8N5q9Y&T^?)2z@y;h--bg4el?BFWhBOuU=7$NId)-x~$~_RwE=i06|O}m&(FiOImIvv)X`mj2Z*cw%;7LRM>c?!xK7K8EB3sc>__@Tr2`n%aKpBgJk0L_A@+s3w?<PpE#?XdXi{pw@SKemTFz=x*4RQCUUWjK#=FZo-gzOfi+eD5;`CUXE8@!=D4|R;dqPONXtX}-+tVdqjOq&<M$`YQ~L@Z|5NaIym9jQB3F2}VPx%ev{eQBVS<0w-~}>ng^}F3I*T|Bg#`E&v$L_k?uB2CCATcAIOkWs<(4?tvk{1$qxqo{iAD>Mn&#YseGkU?n4vZ`bKOOaMRP!cz?!o@(x}cHn}%Cvy-9~DAg;ler6{Q>VpP<^<X^YP^Ixnz*x?#hb`JB*u@ZVRaK-UZPrFaHXq>dccvOg+PHb;S8kzE{<^=o(sHR!Cv*i5_`T+0H`@`~ZsERbg4SCmj2r+0R$IV$=iluUg#PhqqGF$lPxlr9W#iO;!aB778Vboa_N+W7FRis_wfwrmRmyx3!T5zG(=Y*<7lRr7-&}uxCZ^7(NWcO78t1PHeKv?Q}JKW1qa|S4<CCoV&rGUigp73z2;MY%a|K`j`p(6rfFRN)gXrJ}E6=1SVoM%(p=7br$vYjW6ge(Cvu_)uyn6_cAz!$QmMp-BbQC`_wJUx*9#(l-2K+KHLCq#^I-$6*W+d(k9nC`I3EAg*=s=Mn*2y}W<tyZhGMsmpDp8V%NcyrM!tyYE*2j>3@xmx_>Y)iXp7uV5~i8_D##H}~}@8WM!r=xa_m3;Wt$T}|^Z4%p`Sl*AI^*AFI0nAkvQT>aD`FvEuv(v*d+N{|c>ZGipgmJ-z|NDsSF#-WLn4`e1CyG5I%^_1Asu_yo7kg$~nPjD|GAWm<=74y%+O-h5BP|Dck{f6Ls@MpWuPu+$6cPi*sVypvw=AS0>|~0rx7b{v{4FPec=wLAgE7>jJP}ZD72B9bfD+{NkSKMFVev^khAUc$ApK&LMtUKIXXn{da_lkYelombVHaySm}F-l;Ku9w<<XHgUlAU`lO&Oo68}gelEV>(KKQhME}wOdAZLSkSIok0ZlllHV$Eq1n)cwo_@GgkVyMLvi)8P$k4!pG_z>bcMS`U0IZnQ4nbEvzDC>VCJFNB?KqEdsN`bl{2=-o$oM>Fr35wjgD$#EA?d}4|Z%X?y8|jj93mgJmQKB3*N=7l~F4ccr0repWgIe5U#?zc~a5f&V@x<Vjn*K;p_V7%$wdj<-{V|kdoDsEbcQgn$^rFH-1f5*4+oChk_MP&r;`Tu<k+{P_13IwHc9dscVNbHG4{Z<<L<=KMZ6{-6DJTUWuIx%f&`jL=ivt?zrl1@&;9iWu{sz`1UHYDF!0<CoQN?CCZ`_<0U7sxL?)Vv{rCY3CwN$q&htTc7oL};7XzV<oJDa4iU;05oiiQzm{@tSNMuMH?x^^6VRE!+qFaR=o8Z?(8k1PxN2$srt<HvduVjQF2tz7+wSbf%p+g^B3@9|okJM<}DL1qU8Pg5k?tB6%N6z6NIq%TzM5g;={Jcd3i6b=B7_yg+ttA3gxTV}7nekqA~OEl)L+Vv1ff}df!8LS|Tb$&2wF$XenWay1!SABal>n5YF%mMb6p@B3d{rq!9Y~HP2z5FC+-E(u;mA2uhzO`fi#)0Plxg1Fb)M8+kCUP%O9RI(9ph|Yia!-;z!g;B>IO`00-K29I$>2k&klU-)Tv-s@iaA+xc%Hi@pUydJsN*81;*(k>5W2qW8D3><J!j}-$)JGj_FwNiiD(a5y=jJHTt%b8h?3|t-~UzPu24|`(gz^86jp)!Ri3l;R|kKTYe^yeVHpSpA?N!R5u3FX=tCZek+zD!(vdE4aiMKDDS76*UiYTtU<uxMN>UtSdS5TM6J_Ug{tZPV_0I3P`A)bD#YJrKlJJjQu7c|t>=WoMp5iaVPO`M8cyr>3i{^B^ydD==_kk=d!@@lwAfIQ+{;G9lp0cW=aRopgtp!=ai?i>=*zB19NQ-A{0PIGG`=QN9-YBws&Dq#K*<2m%>?|sn(8f6?u3pCG;=!|6$wPYAH_d5Qm4-{!Sz=CCeN&FHZCIDK6c==)1n*l^;d=-u09B-2#PEn>f?(d(DNzwxiW<YFCP16*#^b@n^#=u4!|c5iAYFTC&OpLTLL?Fq5Gd}|lX-@?WznC#gb!I;3T7h5=Sz4H!~@AZ+`Hbuf@CD4^vPxv-L*1!ptczw-~CBD$B1_oa|>vOFN0*x*2Uz-t8gWi#vCw{Z(mY#IHpv4J+_!jggPRZi^H+RDgbzTX}BE?wmy{s1lmjBhX7Cyov?iA007^A$!$#1?_3TeLq^6nD=7Q1@>XzM4|bm1Iy1Fwp4z8mk8<NokOFYAsr}8Slh+o6o8`w$r@OXpI`V?8(W$h8?jsoft=(otjS73gr%Js-d`NQapRtR*r66l+sfE|zv&OqUyWx9Ido-Ft_5l$de{8OpqlpLGCkDjQjZR5!+VVreZo(o@2s^|59r0MqE0(b>9%1)$0_km%*xG|&&wab_Rp8x%BqgzI$B3aAi6l8t1I)I)0+{U=!5#~yB-zl1&5+D~sp_bQ0k?aakC=>cG?(Lt`{dX;%((N1`AZ)@eqH_)T#cV$2sRty4F;28uZCx6>*LpBzO!s+NJIuS(u5>hd=X4TB1rK>%aIrSt{dy`;ID&KCE|2F3<x6xl;PD@`;_nC^gpqfSHJ}zm0nARQDa}H%W>h0Di%r2HI1X-f+-*QBO*`3=`&l9b-jI{X=pGC_s>(@cO1=|uN01O75G;!<FG|1s5QL|g&~}X87c8R0a$Ev`A*AEegqPNj%0U++mF;;XVzbu0;&2sN4txoh+!q)%`%kg0%7j2k14DbBkAemC+K)$$L>lHYUtW38UPo3sxhUWEp&E>P<!#d#_~TgJ}I6h^L;`JOjjc|Jp;V+oGHNXGc~XX1Eqru<0F!)LJH(3z!2hQ`NNvo#a-yj$$k?b`E<H5jG`tXl|SwKR3lp{UiccP^+%)Rhsi`XdW@U<zx7!imJip>8Bn6V-o<vjxDe-JYcB;vLgP;ITmquFC=`AM+CUny2OcciTdtf+F>bYcW`!A{hh8IPx!xX1A!OVgcx;MYw?Z;$r{5Jv+=)1%^`e4nDy;%3`-t~HtB*{OG;)qDZDUm>oHnJF0R}e+BtKlroWh|pI^G1|qG^g0)+)HYS*YXy<)eS4MSmqTyr&3&IOYB9BzL`#K$`@)uAw%I5!s(Xw!7x%p)mIWn!)CK(l68Ahg3S8tBc@#PXt3m760BsH-!nkQ)R~D^6!~|pOfD<(J;x!frTD?_!%a^)2QGUC!ozWf_J1DET+ILGVmi}u580=y{mNe{C((>p)fjENnLw8@NFU<@e%;w#IH_1%w?G%<qy;us*$!xa$Zj6Y8edbB<)uLpBGx1-nU-G7i+Qs_gVj@tf;fLniT7@H!1{&Sfsf@*}dt%yP&4Jx#`*MUc$@E^ZBw0nCE3eJF*D}BKCK&cZyo+Hjyfk;()94=&<PD=v{Z+aURTO;}0<sloK%Zy7<U;c~HGv-u#wd6|Cr&u<6HeaaaEj!A@6~8`Pu>IJgfunx*I_=p&heW;d!uNVG2Rj7}r0^aJ$6YNs?*F3p+Ba|2XC)K-sRGjP$J{psdRW0knsRJ~_QJHwc#6UU+DYHH&&>GfEbxQZd^{avfT^9H{)9pe{6`B;baU=ps419~t{HR=13k0<1F3-SzQl=CzJ1}wqrF=EJH2~lS=H<+K`OQUgK4b2~yrC=bTX??a6kSXfX7!!{%x*1s<X!jB1Q6B394H=MgRS6J(6j0?q|FvlDzpw4bs3psNpTXYnxmVf4eX|5&7#N812s-;v6teBNx9D`v*FYkUOR_#p9k|%Ex;*tZtCp$syE-t*nvl*8Y%N$y4Rk*GYJcq3>BH#Fn_3Zp0Y0;x5_YgZ<*R(!jSA;JU~7Y(zry5yGHX!>;AC>nh<fiOMXgKs_<Upk_yOa%U#e^8smM>G_@a5FJqPjSpmagSy4e6}?U5CK!BZ>>x5GN+G1m3(X;^RFcYFN+bxLNNM*2jYE7Y=)t?5%P*k>zSB|MC3?{8<vU8=vufAs=4*&t~pumcBma$k~q(|LE!Y5w=z(v?X8f$x2a1gXy+CoO8V6oIcXu~G}4^VgEN*7Jea`uLrzzCvw=H0u7Vy?uCW+kYST2sdY(%DxP1^JuE!%-y~P59J(xLvE2FH#+A5XSn4jn9eL!=R8YtNOmfEyb~{Mk8kh;Vz~x3;%|aTqoBwc|4jTDccA$_jw&;QlszZ<Lq_hzz=|;g;rgjPwt|EI&Q?fDeaqmE*ia`Wnv375z1sl?w$O5Ya6WNR5R|++qtW1_m!+n5ezNyaZ!iW{f&)bFDY;O_O@MB9A<|l?uOZrS0sor8Rc~@4B)<|EiWnzHl;FR3u;)Qh`rlYo6=5#J8)&^p^y+c1olYWACe>|<SwA+RTrQKxBq>b~?YV+{jJw^!SEK1!+Lo0!HWy;ch`{An`*DS~<_r0o>fVeVXW9=51>MFt+J9YVsKijP5jTXR@1k2L>87p92{<a&3fC8$p9R)>WlswqXHX2JD$*8sdmj!i4t8ga%ipTbfUKX@J_cWQoAnJQw%MTUWlrnaiCdl+rYESznRIv6NkSL2u%_6D1R`EGUKm#A?#B6siwbXaq9qV3bAvtqK2KU`2`Z&!<iB|f7d;Qb*5#C|KfV=e{ZM<1>5cC)@pV&h#^&fxDsbSsqYf~MC}BFvjUYIf<#oT7J9m+N{wu>Q3lsFIhBRFjt5FdX8FyL`5ltEje%FuioY7HFC&D8}{E`J=Uf~Ra@xIp;5-?D}FFvXn>B?C7F?4E+YBymsd@yH-EEQ1@#7%8U;tZ(mAK5xw41V!qrz`KYs<>08Ppn_EG!#k*OGBL|0HH_gx^5I~qF7r82f617r{BlC2~dKuNGk2S!2Y5J&L5Hc%bYd;IZ;hN+2JWaxNe&F`e9DFvA?K>CwT1!kIkibc1M5T)Z#w_W!pwFOfm^tcI;dHwmbmdCr+*1WAh4Vd+}NwjgImeRu&{J((3m7O3eg=8_iTB6-DXYPHo$LNag`xM3V?8F<XWh2V-B=Wz8p&BF(E<V)@YcH(G%aQbKox{t`Cg3>3QVoQYWUa5<O!t>I~XAwDgMD3_?JMxsPs>#lW|;bH}0QpYgt|IE@-G<~?tR)p_zup=*)iyrV1{c;HA2P+gdSFm^2B94-#<(rsm&s4TXoJH`#D{7cbtS%TazrV!DSkeD(dLfUr@ZU=)M0yc$OFOd$g`vgWm2pl2*-_NDn*=2;v4G*Cuji?mFPq!ttD(#N(af~^kcL5)CeHh`Qs-n4s#wsnYYB{gY1>}RiKYY|V2^GB(mln|bz*j0G(Gjk3mO1zK%wF<1a;m*Dv7%jz>^J_M#h(75cPc#AI5IX7L4qE%+Q3DO$v!4bgtIvZ`rk;0^V8BoNmB(b^tLyDJElL+Ql#2i^l=&i1y+M&z7oQ*Kc&Ga6PS^my<w=q|!Gx%kDJ$Zod5@$`qUTykKyFzCz|j0oIipxLR}H&L@>&sOFy~dYM$b`)7WG^hz}P*SkF6(d%|;!BJQ3^f->0MY{BuctM0$F#^owTQd*Ncxs&Xx@yPFw;mz|-ROw{+7AthGyQK!V0nhnD5`vP<>m0H#>EDb;?2|=XL|V;ylNxfxfac97EIpDMV?91+>gBW6bcc4o8uDgQ=_3q;AWgX3eoeo!&u4NF)ZZ^A(1pspb4#WL#fF7ersuQn6a8J<`$CYgCrOh;@gHF35zd_CtuHiifsrPxtv#RC>#Wi^)NPJ+M&`!9s~})37=itj5n|W&t&>GH^`C=-U_XQ$82UJ;v>_)p5fOMk%e7KHfJc{nBv;1Mf5rlY_yrt=sV>cmAizdNe<xQPXl%J&nb$I_5nZZf<HV7`^k|o^&A0De4rtaqT(e{Ce7%bR}H2Ij>|5dG0p?_GPkMY6O$tt6vOYNsb@@k%-N5CpX&%$prZfD{MMU?{w)m;Gw4#s-iHV{>7Tx(39G4vEX#a+zI|E$iH;m>#{63z=7HLCHI4zyxQu#e)K&-EDB+d?-|Z0ViV?F&h@t@ZzT}%Gliaj${yw&n$QG42XW>;EK&U8_fE@tr%nkSwK^eT9-q=CiuFOgkvOJbo%L+H#nO9u3STtgyq1{NBkZ<Botnp{eHf<Tv!&VRDBJeH^J)pU1wLzUw_*I*&oPdUwT@d4V)y4lv>^I47virhb=KYUiv?64f7~B`htSwsjTqPA91%!}t4rn<<Cr;(ioAM`=P%{$JQ$dz`Cg))i>RTy9Q>w!?HH9~xmpXoUy=PG6@^ptvVkbVTD2B?OwN+TiXXb|Q{sFa68n$CJZbLI;{k~P}n_v4a`GP~h>}P1dFU;1e!$j&In1Ru?#!HOkt=gAT8B7^MV(X6PH4uVA@@WP#WDa{tFU22u=3(oT1lR@7GtCK1UICCE>PpPDMER`QsOyx05%)=w$zgw$EtMoJ;NbFz`4a`ic7(-?vN(8LuYm4B*qC)n$3{Ib%9q_3ztC5Tj_W4-b&xIHC8!(@z<|iXVBOj1zL^-a&`idv$kK(grhgCmb!se_pGbg9vJn1@a#@v=qRS!COcE<ru7Z&Rd|I!e5b_ih(BYF~9RM|2-6V)d;VdkEpf~A7EH7zU0NIyE42vF6lO0W>K6#b&_IEuk(czN05Skv;vSzxO9GD($0i*=q)oVCt57W^M)Q!>vmzmZm-vv6s%~Q$@_{W`&7*WnCV~)_Hk2U$`w$l;rK)dYLT@4*~|4TW?Y>EoT0~Pg;t&%kVz|`RcmNxsvn9v2oH*%r?$~pIze<3i?6U6&v3%U^3!~gaXm5Y$n#AvRBdE}?#meFSPDCQ$V>dC{3Lh2|AY2~m?%`#>D2AYNy3`B;kF<zMxA{^74#ayZ{4LdYoAn9Y8=H*)LAE8?0UbuJo*`c}ST^4m}`0@WFD@4XI*(sQT?DGcSXTE`Do6J=juUO+_i3dXE8T=b~tH`eE7$`K(M=Wr~+?`0OV+%UnN?1Bd<t~i6RFn;LYVwUDI+zJB)DW^NLQ*28JjGcA;L?${*&*c#r?>%i{@ETip*nEBLk%>DS(1aoiU3p*1FCS{UP5uT8tv5I9=WfZMyS4-=8PZM_XXYW_I;)9PIvU9ugw?FzUejw;GCTvMb&gzM<m^gLE$2k2$mxt*JABRYIk~g)L+;VyeGnv5b+=tHye8j7`Fw=z#o;OLmquwy$_#pl4y3V>^8jNC(+8?MovW#7C>wHk{i0lO!0mV`k5^9P{$h1%6{JP3Bi)5zo?0``nwdpDPFSwela?wojk{e2f`<8)-ZS<SCXD{8ja9jb;mXW+F$aMU<LouRL4nIZI6+5F;5)Rz!-<fm4`J(HNc}Wg48yNV@14&<T84DI4OgO9q8)ET86t_X+hS*E|>?PwS9$y%iUs>e3d}|Ig^xROT@P4=~sv|f4!iqoSM%jGKLwcu9T}fUG@w}bb!AABbV8KPLSF9P}zWn{`Z_Qsus34qK&UoI@xY$<4XPmMCy=&d-x)rb9rQ;L*{ZQI_a*mmcjfQw;GuGSrWm-hookRf$nDfhO$vkCf8Kv_;Hq%C>*S-r!>RNJhy7jSipjg@}`w~%pUL5H+#QAMiWd-XRrf+hUZhCm*v(qvZE;WK#+w_beon9lUYMSL*}0!)FDQG`kQy5CrEU3LN#D|nJb&kNZPJEtt!I2-%r}^e0*L`qA+5Ab4cxma4+K;HeTV9B;>oT=TaVV2PDpF3PCmhL;dtu?$qhxWCxwk;SzoW7>+!a-q2#CDw-5%0_So~EV)LU!n6*F*w7xj`tLqx|Cr#jO#6g{s3ZmD7fPxxSJX^R9}RwHltEj5h`eo;{L%#E32&@)X2u|A<_<66KaF(@a!x?crgO41S-Y*PL8F=u|6Jt)O8{FGy$eAFNBq4Vf?$Q!Lj-P2SibVY1n-YQiub*5Yst3Kcr~>O<i8=73cApQ+G`Hv$D=iPyDywi;$o=cZXR@PZR*<J9+0YlU_26rSVg2)lpfTkjeW6X!E7>M?@J(aJ|7+iog2;OuRG7ee+?;l9{{dqG3=g-8y`rPY4yMLc7URNPXOqnrdTjJWFw~OZ{)pmr2~4?5l5a(GhmYA2(NI~F1K_lSNw17-I@QzId);bFPR?34w3M^!IW<NDWPHwI8ylS5T{7Yt4D9TGgZ2c{DjZ1%+TQ+?nKqAdrlE+28d%H%9uKziao;QeWI6<onRRh{$rhMRn2Vkv%McQWr^;%2XRrZb#+~tm+t(1a%J^<%tYL-!nhb6B8-H%^<_70T*up|t4VPk=!M!t5xwccy?xIv5xYAKnaANLCLl$F5yH}(%*aL<VQu(mntm2a;U%B6h@Td89j$=CGr#N1V?9_Qj}9dwX2k}BI&NoGvxC$ygzxd5&&MH#=+<dXf2toq@sL{bh$^I9!hF}+#ud>TJPIK7>gLiB+2UNds@z$P9rj0)qE>cdHsUCS#+CP*_1#3K?@$`QB1UrGO-rrQ4S%`{QJgmEjw36E-ob_Mq*ZUrzx;;;M?dVD$l!d#4a27=9)nk%V1LtdMY3R7CIA5SCeyP{-Mf@mwh3m)$=$(@7bgI^wa;W0=30<ulI2U(20P!f$!A(-ZoHUa)atoCTT?}C>?Ldst-9Q&4z(#$B70SJ`%(`sMNvi9t_qrj2b+;2l%U@l;bIu;a3R<E)Cx;stBT4PI`9t^Uaxtn(ao)%P2jWMs&8z|{WX8c${saab|O7iN;xa5I1eF!BcUD02apN!ocLX4P1x?5npPNBBDc3&1f9P=!lW<~GZg(?#B17ZkBn**QA5Cp@EWF*3dM`lQzbs_+&RFD&9G`+F<LSoR;0i3=mrweF(?QCRo+Ig-JOHjR=_W!hB>t*<??1sP+{n3y|!zqs^2!jc`@k8Rz36uJdgb%FVC=f?z)LnI8ja((yMgL0-7`t5XHM4ecMYntEW?&sQDpsMS(8NI%QqGz~`1K*XI_1sUPKmj1?p5LR=?2gf$u2NgSWC6!Iwip_*8q-N77MnRK^!(oWxgw;W!Pq03$`QebfMBPr0)5XPyjtCGoM+10F_mzr+C0~$uK9FG%6KTnT6;Mo~b91s6|rJ|CtwJ}1mu3{Dnv)#vmbw#E#qDmP4NqyX1z-G|*i=Z+DWDl4T<N+!#>PEcus_At2(k-`7x)?{ek0Ed!FLE&u1G(i>&QrG+>X^>StUnY+{5Z&5#rE`Pn+A<PSuCHsAm%A2V+4nCK=9Yr-Rzf^|HN$aoybZ)4mnXu5_pdBX&~k@L=Xph^853Q;u`qmGS)Yc$Y{~a>0bEs9n+8ilk1#7IxNlz7fT=sv!qUK)tl_j0-kfjNC;o}7V2J{O}*h|hrf@fR)mhyVK|&^9$MkMm-I#~0xw+uFohikCQ(POu%d`4wpT_%JEJ*O(n4&lIsJ5+Xv@E2RMa`w$-vO|XJI5{appol!#-zo2hxu=#w{jhBMIxr25r|mABMk@^kkSKigPe!($xnw$w(VJVpC~y1Bc;8RO-3#XZ2mMgSII>UpExI^}ug32~Yo4q1wJ?8*YS@Csx8f#%7(^(y8snOWnYKbTnvm_hx`N&XE<sx!{{>o7kWBDVL|}Bqpq2SwM9*)klJouN1T^$vFRlWaR!0F+9X(Z6@0QvDCz7<+=)U=nZ%*UnbCl(Dj<1;|}}x=Ck2h<~uHE6PEe4OnMT_cXDFp|LF=NkEe`M?)%&*rKUfKZx9|L5Lnyn#G)lL(3litDB2jek@0v|c>i7#7L0xJ%{nLD!iExewv}t}cETPrIj~@^5MsU(`2i&WRyHAkOo)heCK7Nrg+w+X&i8T!Mw1kqnq;Qpy2dt1&TnPALv(W_t+sHV8zevLfzu^vhXkBdZTaTSo`U2tr{i48n&Op|DRf9R4mG<5o>dwRr?B_6_RIkkDKb-<uUk096745YbA`sQIoucacDi{N`<0E!u|HZ)nD<NWqXI>$Y}J_4<9G9i4d&bb$}c}V`tmWZQ?W~0!zbva`;07)B03K2MA3uL6sXdMu@a4SrWvS2;#{3G_d;_S_Ld5EUYW<V`G?@6oB=Pph~wvqCwG?*1mOM*M!%hogEU!8*)VC7kEh-Rs(&5y6}$daRYg$M%7M?J?tZ(RxQfa~eO5wq#Tul(uH)#7xg&im!~JV>cd}L92n`@^q_|K=kLJc9(Rr&wKESZhhmGg#d#B%xM0Wks244TA$b5G%;*NCcCv&moZt@_7b?=*q?vR8#7ATu!xX~-ZNZ6E~EJLM1!`=minLV-r;-JZQhb34?U()K}iSq*6gB2(>_3j8D6tvz}e?)|SJ?IZH5%;HL<-Z#n5#?^-oO;(pDa2(ajol-lfhWWGi=5F5d;GOa?~NLWTjN%5>t4Sge65J|ex7}x9<%@?<qL!7%)un;^}4oSL)D(!>V|+>d%|`X0yp@%u_l2|!U$ov99xr)_-fWfo(>Wu_h+kpA;Id|W5oFXpuoYyLH54qJh$q*hll310?O>1E*XP*x&6@f>>k5L9-+q)>g>1_@U`_HpyLs2z9r7!Dka)^kK>$hHvonBF0-XM3mlgAa7s#YxM=p+Ah5YLq=9Jin+esF%a0i;g0DM)e~INN_s1NR<yWz<<#P_I*5%($6+i_<tLvwWX(x{!&vs3o2U~@TbY}yr?Pt(X<vpY}>JYaEu&JQaxP^rU@l(+loWqv~ODeJNxVVQXI!ckQ#<LD(k%jlPY1d*=zvqGy_<;F_0@D`op-cgbK(<Xv)zoj?@(P^3+1_J$kA3~eX**}O85s8j%L(-R<ZqcHxDP*0fF_j0uAOqwmYF#?MS}A=rbqp}kO}(inQyy8p$7s7A*3@r=90P$w+Q|~*_$23sYYL~+&eLT1z)g@oe1T{B01VT*>GVodF`e+3pBCc0K{)e$EW>18=}%twN5zktu-B&`d<Q4$y5>&lH>m7JRKCGJth)@68|1UIrOcFKhh1nY9sOi`~4_b5QC`5Xt?}HLKy8P{C7GzGM{xwf1s`dd#(kLe^x@P;PMekMsLQ7zP3Ej06&pp5QREVW4v`*Prajrq^-}foe<fd7vLV$qwx(lumJW2`~O1Q%3<C6X7jU&%>@QE4<6tNp#ZIBTq&R85g`ebo3Lzx5<7oLCe}@)9SM6>NvLwrTbok*JP<CcRH$LtRM~tx%Y4N0p2GNPKmn5l(j*5)+zIXp%;d#_G^@X)oHw?nn=f9|X3q+!>6kNa=a7RRDX-DJ8NiD*Oj|#h`}CMu5M+$=*V}2~-?PKO^MEY)ZKT2xYjnfJZv@zw28{o|mjA>dz?Cil0(BeiO~%Zs6$Tx{({B|$F{quZZr!gOouERG+}KJ_)m;*^FZ^N!4{7Q`NZ)#AF#yAJK%d{<*jVw~!CgHD^DJMY)K<zjFK0M%@m(;a22Id3mhDX*12cf~i$ZY!9^<sH6m1__nGO8a20%$wznuW~A^}U6$gp0N<NH0nxvkbpdJ$iS2ezIKuXA0Xgnh46{%HDT)*{=m<%}gWeVc3p`O>l~f70T{JSFK&40{0A`SpGfZVxe=N8@vTVja}e{3(iVpJ#fqS@`Vgv2L$JjIT-A{SN@Z@pG<cp8H8J7!|C#dkX{1xzuVnVVO$B;lgFwlaKw%B9@-{AU2--L8$l|Eu6WDY_>AP#)?seFszEmvWLwgqUGF=P7Kx`C^4hNO&oa)2k1XJ7`Vc1Q9DamRXn|JvzhT1O5WIMhgVT~&MD{ATYf`|it`VxBiNe~WNWKs2Jj%<sfMXCax$E}N<@%I`B%=}+1%P8wGAWH;xv7~gU=nx4RgENL?EYA+NE`QG+CIN6pRzGdjB>aC~{O~?JZe=Y+A5$ZwDb;X1d<7L)el{(Jp_fRnc~{NTE&FI5U6klgx?wwyXH*BZnMjRvd6CXXb%z<SDJsMqS#E=Jwrzx*TO$YEqF~b>@AL^Z&u~rJrf3nQ9UP1r0F1qdQCb+80Y!b8Ht`oywM152W^`9zK8qE04Pl9Uw|7-l}@@Ub&`FdbG~qv2xo@dfJ~BVXtg4b9xI`e`$o>{1;e0t71zPRh&7QqJiRK@G|w&)|D(DwJgnq%p7(8%M{n)v2<YPnPz?r&4}(7>o*Nj8fj$A8Mo1_{p*~k{F`^PCz`~CUrbD!8U1-uqp=BZC+uEwoesx{KV1EV6-F+(;PE${apF3F&CKadhEA{-iY*jGXn&Up#LF;HS03qroe=>g#oATvHj*p@8k-u!qYh5W^9rAqW=~;{anxQGK&EYsfvvBx|AM!21~t2r-CapzFIg#Q%GlGP<EnBTv)_VwNIhX|P&31Iq8vP3C{qHH&%w!(X^vt&_*CAaog1AwE#<px!eahbqG(lnX0^=q^>*#Z%0pBLNkHx{abIe+@S>Aqt{;b=EFr$}zyt9M;YhuR=nM-gN@iv%bLvnunyx^<Zst@V+L%W0Hc2Amkbpc5se}53)e;t39tH*EF9^{fP&KUeTRNZusG7XS+j!_Vp)q_A9x&&hwSFc~;Q@Rv3h9iPQez)2787HLXHt^Ms<5)nyq@k1|CE<^@0khp&2dofU+T$6Cxo=*5#)Jg#>uL{Of)t#x}1@-mvSDEKaxvvVSnS03~_-!9r_rZY@EK}NqlM-?<eol6BHEz!^tcG4#AZnR2!Z>_Y{h`Z2`vBQnSz~>rg#ghK##+0lhxjdTYBnYI1gz${QN%5F+f1s`7#*zoKz4UZ5?93stKkIKm0J72`8F0*4C8e6WY(IwNLRV=Yu6NNri2Ace4H%Z2ZXD=XONWp*GV7g`_{+nzSa^ph|K+ZU1ymfH?hz_A?8A4#=qSWAe@6>zkWf;leNA#uq{q6+5}i^1UaKcS)#R+PZ0rVYa+abz*6`XTx=te0P5<$ps^q~I+jI0L-8hQwXn=pMzL^JOzg6Iww5?EvvDBz25`%0lnYh-xvknjR?b;Ei(JgN#fIOn%!C4Fr}%9BL_r#20f8{{A(toK$T^CC!&w6zr<aUTaxVm#q75EGA+ThZ?6|`PufHl~LyWXWgd7fPu?VKGiHR(MEWl+n^5D#l!KvN`qia4m*2M(Qq19D3mERgL2}CI>I0@<>;*Q;sk0qBjtmy6)%Acei6dM9p10PhteDhHr5B?X#Z+1?1;~Z*1t@Z-G8fWQO+IDAb<K@f+HEis3uSMtgD9K1H5DvTqSdRh>-63xU=w!4(b!%SFFCK^#+=(M*o_2#(Dr_Au7C!A~g|Aw$Ukr$2n}Q9mjbXis)wk)&1g&$jk(6`0-D4q1EhMb6GhM0#p=chaDQf7kb2*j4Tiq;M9SxB^zwa2xE^!`nNc8auxIQ8fzX>l0+{cehf7TUuu2cBDLQ&jJdkn1WVY1)nDT|hD^dQx$WOCqayxxtx-1aXnTI4==?zD%HkSYe=aQ*xL)!TV-Z}Mt0ua)GO$cduBUN(+*y`I)Ol2(@Ap!?p!qXn%%6Zu5M8ZrR4VbR&(DLb9qay^hp9$pkz`cKB3mWSASGtz`1|6V^~NRylw<UZs=;|Liz<(IY22;!ZFNPg7Hb^G;?Kz#)q%%;Htu>CeUYAC7ff1RD#9gW&LK+_*B5%c<6Zevyt&W*23-E7W9a+{eIL_)=y%0L52=+9Q)=|)E?kZvk!TlR)gI|ppiW&QLpF&M5r70wjW#=>cZ#rQ=!V(zc&&S}o3?={*gaz$_DDOcTl<#_<e$|Vja1YU?NA+y_H$J_X_A6UCIdFTl&xqaTW5@8c&x+IXm+skSGun_=&kKpJqv>VP#~Qq(%~(@yh81wn-AV%*kwgbs^>gPdFi0rQYMIKjNyK5c(U)4XASSUFs8q)B!{t$(uio^EG=A!9U|1uNy^#?I(t1HEJfY#q`9InPB<|XNS-MV3WpsLIj@c<ZiaaXB<AC*TWS1uO#*#j4+BGwi5llaO+hPh{vB-_TYucW_ssBnrnv#^YOB7qoN{K|{CNBr9c@oWx_?CihgaiMvu6)JufYG46uTjr)_3cx#jCB$hMaKE>|(Y-c=}38NeWn(l$h_`mn%fQ$}uVOZ;u45R;3GIRYrckO|R)B&>-g*rL{7GY_=Az17a(twmzoDw0))B>~pqcdo+qqI+SwCisC)$TgW1d@`-l71`<vK`GpPh1pK<zZZ_rpmR2AAy(SkiY0D3ksM-w}9mq~E?okny2bBs;92K|gj~VHShB|Hvf@VsfvT>!i%H5Gn4`aQpMD<k@OP;$GqPp@d;W=<5Ez-cw5M?fD$l)|pLXUxmx~m-Kaq5gz4~3|q88Aq?wG&Yf-(zBQ`?d>OtT#rH#(S;e#f02RTvTMXx9H`@Fe%86!LNveFENgYWISM(1GkSMM#|-jWg%n10vc!H8>}CnovS>ww2bmPeIwsZn}B?XVN;hi2&I>+qbQ&%H1Hxp5{*DAj`<BCH&WvgZ)KE_!779!a&c&<49u^r&Y-3ppdBfIr0|}&2UVsTk&>_flgdTPfHEg}dWWDQnaf_PkaD;|>o>Egt`$lfOUFiZQ%}y+m4;~?F<I@Qw$w8rjlc7s(fo(JP}4i8nwTWghjYr8IXby?vJ2#4ZX>1txw!Q)^{a6i{B-tBny1LDA7`_9Z=9+q)wMb3Q%RBK^I*5)q(HL+fejYXg45Y&lmhC!{UKbpV~09HZRD%MN0{7DmV}9N0vU0i5~&hibL7i7BNM<HC0?BT_@m60Ko_FluzykX4LGkDLioG#cLbEkQZRThs)n|UT|8oDNdTu0UCbj~T?V<dt@2Rh(6jPGkAc%x4P4XT11{9_SG2Y9WziJs3WQmmJs^WLvaH3esB=Ob{<h;=$Esbr6?SNsEEj@HoHF}p4%A$~VrgYLX4=()6@IYxSJA{1VY70JV=X2JkvI7?stsU^?r*QgxCFR6d)H#oYI5-uglCd=^8Z3JoUL+|eD*ka$R&rP4VyX4%ZIQ+lPTDfpybaFI7u7p!j9ZB_!08m6DyU@3W|t&#RA^Z0Zk`^IiZuXkOl!5KK#Oka2uxEUHBTdZbmf+Vyw!!ZG)=sp&Tw?Zi(+Mq~5v@JzFB*|2W{f00{pfzIk&<uGudb$WTA-$gndWh|iP>qMAh1>hmMDd}E6&)(%Y5ec7+{m?6vxTYt1G*nhKjS|{dYhb(TYchZ@s?ZS?5Td#G1tp5haF1Jet(oCChYG@VC4kC{b(Z0`|AUIQQeBQO&(5+=I`Tm|^$M3J5bO;@1K$jJF0cZxMEh8G8OyzfNFEr<<YnX*~h!U!$*3AEnPhD6+XqPfqGK3R3-Kr~K+$`|r`j&nOg5=Wyo*~Y*c|H_wz*Y9zZ;LtLKX1`1N%Sd^5o(kQq~B5XEZaQ-PjxNOjVJ3EnYeV8*nohh-vyo=ylRy-&erzGg@~)lgUuaW=|>dkI$j)EoOUX~Wsoq6|5&b-yiy_VE1!;5n81VHZaVS(Cd3(%spSSD(ff>8m&DNNwbSpjd?lT7ZTK3*$$XICl@Rv>9rLP4%|eX%$Az-fQz6TpgPlR};^Reoxs^0U{X0xJokwe1lU7VPHzXJ<0*Xt4p|lI(^Qmnqt6QVs0t#n2USesUx_p!UK8fG}Cys}+I-lw7#(uPIt>;g;vYT5k5<JAoev^aAbcSOU4E5QExU&E`AWSoKJ{M~M86JORw75Y53O6xo`WUPA4v5XGq{G%Qf}k$8J{WRCFv!ZSFCdfNVI{S4pnY?n91O&pz#=Hhd)vTIKw;oDtCPE$f+k|6d6V-no2r1FPpSxDe9%rJ2&K-hLoKI$*@!%fheU5D?pGwN`8hiL(Rp(o+xy>2J1PvE4j0lh#4cYO-p-W7W~lHe6MSSo)xIqS&K#Z?UNkC_p#%&}?Qq)fZg;jwq8pB~eui<&NK&3Y>u2!L5fnTI!5teSsUcF_@FxnU4LzYRhcV;sxgE79=jZ~8E9Srp{g+$Aq=qcWr;E~vKrP38$mJ#b!_4QRHmq@=t524@UEO`|-`y{!T80YrUgwU(4|TTI;1VFDz4GMT(K;w#7=d%%_%r3-0er3HBuZS{Vk$RsKYladKzOmmE@$i2`%24IVIy^EN1L(WwL&IBYo3uaJhooM1Kg;qS`@wuSX{Jvq^QN6^gii;9DSZ4I89=C+=Z)nriVOqG)j{s{%|!@7SL%$yC4{L-RSo(R*9nvQQDS#la@OAbPQk+q=y>RsturD8lrX{y_O*!_8p5yU3dn@&zcL!4s;{*NapcSTFjOPcY(}0>r1mPdA>SJE6Dq5)Fj>%gFOw^vu=`=`*N(0^SN;Ku13Gvh>h4%cU!R;qS*YS%+XcR6HEmanxq#>wZSe4;@h8aMG|oPD{oWUjbgpG#n=4|TT$2O$j*ZQz;bmc5MCqKhem5YU88f2t_fg+(k$?aPS*JI*7XDe=*ajEEtJ%Pru%MM6+dXvw}Nk(5Pi~C49w4`L@#CT^nY72%^uOrod}Lwjif~lOhTumTJ`CDpw2&o%|;Y}T<2rXlWJ0Oi-?o{^JfXR{d|7V16P&N6m)d?L(9|QZJ${lU`Nu@iu$Wkes8#S{nD8YxqNUP^m^A3kR_k>dC!~E$0N#O;LADn#mwUZ*iTt&F1(R%(!0VHHN9^9Q0IGw=LE!-l~2|N4cBP<t#(jStp4rY7WufIYx~7%C>;cfmbv5Bt+?I@M{5d3Pz5e91JG7Zw;<Li>1l!6+}Rci_w1Y>?|o@Jqva}#C!Q1uZ_OFvxvg0fE{8v?-%J$-%`uyYeStARzFI*V(!&NW5sTkt<D{tKPIytX@npz__!|Nu<Opqa^m}0aKt;>agQRsRG`-btAS-W8Ur631R_}8+lC0p5j?9-(g5HK|y_}XDhj(eRmfaT$?83%gkZ@m<^iqh5S7xz0nV|jFl5NB-*zjF1mAG?+mX3F5DCD}CJgRsWT9^B@RsB$zN}~H->-o_hsHdAo@%KGl^TdN;GLUY?2~gZUTX`)U36s%+Sk!s%J&J;w?c~O^a^8F!psxF-D~M+)w0MkWe=>8YeF1OqO$N#yxKg2Id~_r50(GNC@et&dtF~n~LVo{R2wGNvnrF3>f4M<PL$_$MSoWR=kk7k5Wg!#&`~fv5;iywU?OE1X*=Z=p5<dB#xKa%K^~H4puJUG%m&DJV+PmHQUDK;yY<AUL`PP*_FaPXECVpY0IZp6y8|U<tCH>1Wu$P{PMb22x_rPGrizN&-EYN%jN*-99sL{96?oK8bbj*##*oQo1{Uw9)LKcXZ)s^4mj&j5SbTc`CU+z1OD0wK_3^>kURP!ioZ!<l~Poy?nOkyaF(@+x$lpW~z1)udr#T7d$D6wL`$rFjV4V^sP7`+OGN0eiH7W2!}S>vy$K=skw3DZna_$@;d;Da=vo59i$Bqq^IFqoTaQ(iAeNonYKu}5+RCthGi&W@R#SHFQ_J3}TR956FQxJXx*VPz^}s+LFwNfiu!fnS1*6x(NR5spe;4i=t3Xg<G8SNnLtF)&Ox>c&S0d2giWjIvT*%Ah~C6dd2HvN+md_#$Rm*@*>pUKXSRciazmpx4i~xMu<bdzDgY@Q?7F40U>tZmihTmp`&(*f<PzJX^hR*(7jD6jgHBJ5d^8u*3it_rpHb5dWeAqIzz$^@5cHs)qVfSxdG86{yH2x4RuKOL3=f=akc?_uqVhsupI<wnCvcg0tisUQTA@I<CUdzY|`jpzAO;{E!{Ff`HcOzOB`>Pifzpr|5kteS|p%`JUyB(gIFWyE|`^cl9%j_X*paqfNIzc_deGQp})^E>6#CBqpW?)w4g&{t1XWtLB0o-5q%Qn@t_*_@U@ovC~X0UVkdEna9BVTyc0Fqw)s{s}I7PMq@&B@0IWx^tG;35lmWwhKju7pd|xz{opaKm34&Vp$T$gPJ@>|w}kUB82Wp7+(rXP*_4?>jX?g(f}}i1Bmf*VYjNF{SZ`qM3r0E&rv<|eFx$OAM<ra)E%}my<yL%{O7@IaY8H1vw5|9V*Y`DTwK<)YB)?@TDTdebDzY1cG}P!tUg;U!bZ!%MAe?-jx4DH2^nddXC9?*Rs5$n@Ge5VQ|B817;Jy&`--Blj3gFes8J`KCiiodj#G@Y9K}nKBhQe3TDuR)1_`_vw>7cSl9(>@>D!+nT@zXV-i|5>SFrUKa9*3kwDzf<=X8x?*s1PSEu}NTR&C@?Ym$}__?K9C&p7D2t+#r?PjbP6-=Obiz0cYz&)AK;w)QN?>MW!p-3)EPVSwKHQ4`cvVr${MQ3Ojss7rnttsJl4{_qdTDulz}4txwS_AhF=?vDLF;YI)2r><b2~UX~^Dxlt5bXSaW%C6ER4w=qe%d5#Y*W$Oi|xZ?(-gQ&OR#@+kifF8e)(QgTyuTS)_g9I3cqZgj2p`s&HLuYJ}+V7w8cRoz{74i86F51`DE!v8^zgiIP@wYYxU?t$Ox{F}!jt_C>CAs&Ipa#TuQWgjJ0s4-}Zj~r=aG-p#hGM6F20lSXvUxB!-@0*Xvlg-3UxQ$vzuAH{n0ma?30i@v_tMa2hyYZjJu(N$=kepTxgfcqv2bO1g_;VcwnZ38XM~nu#ozgeNK3uyTh0*)+8_LbO1*+17fN?09AJ5-@y73O1qqU5*bh2W?J8o4S&`Uj2Rs>zDwy7!v?+IE=2uUd@B*(RR0*WK?pk8I5kTIE$>UE_AqNT_M~|csRA*;h;~_5#BKrB{Z!1HG^XQJBN{6HHS<%uwT*l|jeQHw_o%Pe<=b2D~FQTRn3F&JEo%Jho2aG@^z^Cc$(kVsE8RcL&4$O^86wv=76jtODgO;z3ity5^eco26$M@>V>H!)I#gr2#x!B<FAQDr>SgReDjFc1)bf$Z0p|s_AlJbN;N#FW;b<d<d;R&9r#i4EO76H<k-|oWYi4fc$;(th9sI2U#FY5l=predodJc5$7wCf6%G-K>=Nc$B==W!~4i)QkYPFt8^hffnaDv6qRn_bIdy{Jre2~}!FAOa~?>Waha2V!v#}Ml?Bde0=5-Hl^IntU0Z<g8qCrf%}t*e}AZj`APk-2VRX?V1_Uot^f4vYuIboRJQ<)ithLkZC2g_nGO%<Q0Nq$kHgku#Z2bs^S@3CerFv8#_K8{K@pH+otw`s$ZB3)Tl{H*6An5Eg{~LBolTLjxiWuJ3GhL8h=u2P2-M=M-b~w{StSv=1q0Re2Tnjp_RgUsAu*;jQKZ`YUblEwMkG1JmQFFHUR6Hr^CJ+8cpfUD^cOUCd>>{$rjWBUt(fRTs2aILX8KQ`hNdLo|U%!Imla?-O#9rc@CRn%DSyRI5}4O+tdYjLuiOp@zkm>B^P&Ll0DW*kPlErRqus6kzV*%g8I#hwe<w=D*t@S1x@(D3289Na#fkfR^$UNjV-;C3leaLp}ZVHbh}1&1J0FjKfGVa3JOX*lGrkTYJ;Pt)gOA8;)0hl02&!+vIsLTwnhm=bvqms)28w{UvZ##JjBh$2NZez_TN`N^$miYNw%qlo|h3c0@f7kNjN;GI1l!WHA}j3LOzw!;Vx0^I*kBv||r5zQ$7Cg8wXr%^C0k$AAP2!Mh&B=s+zJ+S0^fLOU<;BroZ#ktno`XI*nlVH>Sa%}<q?<0)iosqey<4opCIgMod>HmJRQPLn-q6vi^RA(u*jQ#<xqxl9Fx)QH7&5jmc7Syso`<@3B??=L*%_6H-p<b<9k<f5&4>-ll(J;r?fI)|LZD?t9}#VU;1k;V=?mau=UjP+mPV>!*4U3xEOk~%UrNbi+`^=AEP2;1S*{3y1#4JkBd@|jhKe5a}dUgL59E;y~=_-)`N5ZWzp#EU@VI0{`j*fJiaQg|i~HW6GaC44@5XLqZlRDPm&qX*!GeOX^z{L?SThSU+PE9050t=#dUsL#nuwlrf4J&AA40+4T(LNXT<g%M?4RjeKZu|HO6<*yBfABTfqQU1g@)qL1@Rg%}B@Df})h3%g3ES5=g(>k7XWKd}F-qKLOLjY>Vx!V!N)+-Z-_sY9Q|Ll*TRvT@%BJBKh4>F|5-xo`c6K_~`9=!bRh7|FpsG$Ptk<ooNJT>LGREOC@p0N+}fKIrG{{@V8Csw>Z9dPlQy4Dc}t@-7sRFB<0SqV~hnYGaoXGdobAJN3W*<1GVG@b||dAMzb5oChw0hs=DM9O=dH)$^X*+UJa0x@IyK!%dwXoBOvm8P&4^JAF|&(B?9cP(3t>U1A2)W{&H;95d1W{JXPwit%tKov_0--5{HeJI96Pa7v7vK4C8!Fjx)*INO_XaN5jU^m@&UK_R(IOrU>F;+7A3#bX&s#_v+9T%9srWyXiUI&SAS^DXeIQFepCY?%Ypdf1_1ecGz=QPq)-my~8b|GpCiV1D;L*U+>3WCQw-eNY!wTR%B0{cR{1#)|1#PBb@#mai~5rhagB8&k$zqGu$4y?&SjFbvoDm^>P4&n#<3(0RHJ}b}phuMZ#;;y+SBkdZYoZDn9Uxh$F_^CK3_wQMkVpwb+3Eof>Dmdv-dcTBTZ7_zfQrs0J^iY^%(3cMD8Ftrq`}1H)$=fQFgDZxws`wPoz9KWqLuQ&~dwu^Xps^CDA=mt@`=RWGgjvMr6kXvLrY&w<*(@UKylt5Vrl8-GH9?%DgZuqm;*5qUayph_4m$-}g?({A)F93hp)h)(Bl^!#aA^9-;{kt2lP2jhA(m?TZP@@I(p(O2y4)g9LqxT{b1rf|S-n<79P$;358PHj%GI-Ys52IE{zHB!OyA9o)hX;JW6$+rYGK9BuVe^I<NJE?8o<D=T`pCKLwur<0|X|86H&fRe8JghFoFrwz~hX=Wc5)qOdTu{jy$K(m~WO1%2<eCHwOGgaqYB={%rpbBPkWYY=~ouSNBaCGPIS5#&PUt%c4%%<Py%v`(Y(HCSRi;h5I^S4OPsu=z`<ncvJ=VsT7M^b`O5n^8F);(D)?pBg1GHi~CT)V3;;M++<9nV!QtiemE)LD?QKSorSR?Tz^-i$`0nvDqm#KVD_-uam)^L(VOOkf<1R6d8Sj?kYqY?oE5V_gzcBzv2#vhrJa$m1BrbNo->Vd9>+=Mc6_~MyRScLK&oE`(0wh|F`Sp#Nt{9wDR|A%34)Z=+<)3wmP@k|!TL{|nIrf!)`8=5?t()jD2{^qUBU#uJq!1527G|VGPbgLTq(?>u^xjUow8JPWS*_Cr;ZyU?a?H_P4aJrMmkgv>2A-$GCeY}{*7P!1Y&^l>P>{=aCC0n_+P;O{5Krm9}jz|pW~fsiN!^-DX-5xVg||tVh6AtUX+%nAV=8z4jO6D#jwmxy7IOW^iv$HsR<skAJcw~IbxJ-7c=rmnIb-~lAJ!jP(TL6A`W^ecN05cD27~K=H5e~<0IjWf&k6Mf*SDnwJE8^I|(_@wE+>*d6UJl&}83DS>^-7Itjqq7iSGxw%yLx4;r;tm&6xp;>mUNBQ@9g=3wF5d6$N71F7({j&sNF_wJ6&|3s{-rnO-`(~p$M(k~VzLjR`C{xQ(YzP&}=%WFlSbb!z0Jh1Fm-UJDts2<GArPVuV_(o<)J=f5gUq?`m%1Xu@K8UN`+sw=_2dNs$e+o2ltB0qx(&8lgt1aPMJmjQ2(M!m1v<0)z?DNxI;xUv=m|xR!gcyES;+-+Tj91|ZCt!d!Kkp0|rF4}PP8lnQ4Jw&+;AxdW<GyG=*9(7R{B)dbnDa8QG!ZmSvmx_<hF25Hkkr;PhCTMD6}Jw5zr!T|#kaIo>iEr_D-=q+jobqx2~S(;R1zR-4B^VexgG$(P3VH8@G@v4yi}n3{>8R1n6lbB9(A=w;Y7J1gz&uUhh#7U(LH6~`2IGJxso~wOu4y8YWWGj_Ewx9i2k?}pB1~h0fDpvS=^m<awpBd(e3CsDQbkAQzVqoqUtTLtA&PJIJ@jNZf2;xFp)x;(_}L3cWe)E3!Pw`8LZ<Wm32l}6H1VGLz}1^Ui{gC>FeXerI(hhMsg^2Q0@=KL+m{tk+Y}bTj*RUW{s2Z8-`qU;`$o~m^gdjpn~b=yX`Y3in(%TlY-Vq8l|1NRPS#xdZcMba=ce2Di28490vF7z=S7w^H)}OgD4$w<?-^xKr;(vOSkEH7#@6rwd`@Fdef7zHbJ3K;u~r_!fNFxd9xV`A={4!5BH%Mu_45+g_QR*{sF%wo(>9=&BAoqfrLThg7`Iz;eL;ptwA3AiY?Y8=>2Ts6x7$O_X*kWppBc{P7Z<Cn*ej1-^nOK<$Jsmu<TnOWl+<kaXZU-%+7hg0B0)ooE%M3$#-6<t}y|~?jGBvx@l~JbEV9W^whx?8;0o}94Vgu)zOSvTrR>Fg{KCvWX~XN<x$<s_YBUoAR5zhKdxrhIAXkPkHp<gj%G+{4oQ+~UxeHL`mT>-L0yX6HkhyD1su(BPJS4%2hMh&z4VnsVz5An{BlAu1{rv(94($QokgOSrr|=(f9k%xF50NPNFk9DWzI)=diA3!muKzgDzm1H%N>Bi$E9C6WEod*>b@4~Sg|laoo=q6*)JTPfA0H)1H+QL@coovS(naGLLVk`vTzdhc1rA?bLgjZoaTOCxArxdaJ^Y&t9VxYs!#F_028!g`Oo3DSuWn|3Syw8Y||Rek>$Yh?j6opE@3IGb}18sfo`i%D0gz5J97h!<2PK>GIM@)g64lLOJrB*rh?kx!SENmRhj!0sEwoBK&_gl1gtPaG8k6mU9si&|DdY~B|BI^krU!+6t2W=i{B~&n+(N=NybW}#KNV+Bf>uIy)Z3S11!L`p?N^>`<C?cisB}e!Cw|ARI^;_f?~!8IFR;nyBAPr!?QTsSkCcFj;QI7z6>6?O-j(J0nFm%c82)xh`^{={mLIR>@VZ4D5C1G=8GI4O}N+BS}$`Uzc&-|Q(nd$>Z4SU9q1e+(oIXcLoY}q<dQO(PCJq=Sy7#4NZjl_O$JkxnVA`d_&070`_>nTWcSTWk3lEU&|}yLP2!h&@b%2EIq>Q1jzawJ-<I-oKu+UlV?@sVAx{vyCp|jqIuYm|_xi=i0H9*9>3G}JC1|&_8|5G;>zl~|z?@AoD(oi_eg%3$>-Pr3P5boaMg2Hy-^*w`iaDpxa0U3sx2=Srrp7I{6%_ZfK`zd_SvC-;KtvH`hhKlv3ftF({;}>%M4fDws{Z)-9D8E?vfi}=C6iZf+Ous{2{2|+B%hr!_(`W2EW})5=Sk=%!5O2izHjjq3y4*18-b1r^_;@#cOT#alZP*vmEY1ZyuT8uHrko<eBA6#UUTNm#MEbp%1K>rPD-(F0@uW0x}Gx|T|B?%q`Rn}p%EksTkV$C%>VOVjJ=Vlk}AsY1=*hbC`?OI(9gJRnHJ)ji-rbQ1ppfK&2raS%G9!Vl9y{Yu)({#uTK}9N3?Y7ZR0QA(dVF1lT|d)b=3ciK!1f*Neb60W94|A=hm_6L974o9gdcP+16qr`Aa`QjdYNUzkF2(my4s`{o=~M>-_sIzCGTZ$#qd-o}YqP8B}`{(FO!<OKA`|PGSPRQ`!P9MLGp}4>S;IwZ&E=!+Qo;MTLMe*~}v59yuoX<axd`=&Y)&Nc9UFhU70pyM?A0sToA&l`#peYQ-2uNappJ!81@=l@upWQSx6NB%Bj0Abt4k$5FH~oT=(APt6{ReH#Q%BGnQQg<SBp43p+QsD9*_tE_;)`V`Gre{psA7ZLN{{p*V(=rpRc*2edw59`84zkVP-I%0pGEm^7ygq$MCHV409cm^~qH~51h-Q@q((gFb{5?!?G$V%r@D_yw@lAgapsfj3e_!n|=kr>b>^vLjLX6_qXK3+g{KeKr^H%hmX)yhf+bKKC2D_6m7r_IS)3hgmmy*RkQ1=(XvzgYyGm{~h0JvA-G<BjPha2bG*nX-<9B<HT|Nic3kuJ7a2<f@$tl%F;h?nsYCHh730I9Oq>GNCEl9l-FlQL8Hcn7^d);=gO8mE5)ZVU}oxaCf_(XZS@(db22-qp7i=LkKUsloO^LngB26;?GZdzc{gv)l@pN9jWUYOT2Q;b0W1@ARG@=<(F|U6f#|pA><ClG<E2c&QwYYMgqjR@_@xxJeL4e73OWp(MsbT@3-TY<`l|9`kTf^N9>@jPTQC)gae#u31|?QxuGRYIqq>kI0CUl2#MYIDBXw`_GFQ)({->z&nqQkxUvn#>D1f|o>JHmP^>5A2Vb$zo=nXtjIibcR=<3Cz0~s={s!{?P@RtBt3Ec<c%}D4eiKsAXd;>%wc#2>lPdejJx6H_j)kOmG<h*{re{Ro-J-fAdjbD#yMjT%JTyp!gMpl9^ixg!8uPz1;?BsPGRdl3le$N?qu2Pd{pltK`$^woB`@gigVrQcx9EGe{8G*0VJ|HXZFGP)qydNnE78-D*T5mV&ux_hU=TL0dL<3q>3A~4zX8$6RFa{Zc(B)nRj#zhoD#$O=OJPFR5Q>ZI=L`Q3Xs{4W<e=x-z|I%>QK<GC49rtdc9ke!4fl&FT#(zEulJPTlsVCU7Ii?L)=x)n8%A8OimpIYUdfrz_zfuvcO%Qf&Im;spj+MjH^oqEM_&hS{(R?x^~xO@VX&+u}tb7;13%Paj+@0P<%4v%eDnWlgsmz#IJE5be+OLLIWg($b7u+cM5oyvY>J6{^2{~SgJQ<Bg9q9tQFy&Q_AHf@vrnpnc(mmD#@QJoa0&BF|m9A)BeY8EuSVfE6}nSNJEZ;EsN^KTER1}=LP713hEq{lk=T~;yxYXPUB4bQ=l9}&W4_ia7J!wzMZS_(Lng_2kql|$9#%n(Qks@LI?txg`F+g87(FaTawi3LDE{Ds3%*CBzf#?Vz@QniqIg^YlCH%tn>u5+%$Ie5}a7wVjB<?C&@V&`b52Km@GJ8L%RB`!@$e!tFYI;q1>cNwg(zhy}i-*=X?`8sH`3$EhibV91$63OSm3yJa(jP9Tbj5)z;~dGAIUfA;<Nl4TY`WDl|_sVgn~j!**f}|0CC~>8?t<g9W42E#d}4++GKEPQ67glQ~RQnil|F7de7<6n|k8h4;VmjEPc8#rRu(1PONRKLOa3bxUexRn%GKpNG=q=PI>^5p(P8|3z>0Uq#L3-a>&G4I<?56cfkz6ozh~ZFUwNex}(_=+H@2Jd%%-o0qIiH}g-o4AaX!K>|7<CBp-=7qZMhIbKkE1FA!t39n%r%$;lGL>uCv=7y~zNpQA!ZYOLBg)}J@9C#zU8UcKchDf<~FNJPG`K!7Kp>M72l<Chv9>-QQyKl)r6JGNhF{NBL^vOb^cQcb=^nKo^*;4K|lr8p;S=7&N9E|;pf$qGM(1Gb$OK@&G(6+(GpiO^{4U6_wBxCkUA+<-!fHVgx9~;nHq<=r(ch$^_Ch(~89gGzL+LgyNl`SX@Trx{wW;H9geWS200-5TE2-98T%cJpzj~B{bN`S&$VQ=82-~P$J_GDKcXz2kfdT+i3dmh`h{@bL^lYn-|8V7jGc&B%@=6WLBi-N14%Ku`B6OubGE;L?)8J@5;0&9mj$Iz0Qp7137`zkz36rMuAGby#fmzptpp7V#Mllf_y<=_fDV!3=x*|~;M@So1A)M1Jq3*3$0BsXx0!YJ-cq{%Vk2CuF#VU^6Cl4xw#tPiy*V7jeRF+?H&9QK}_Oek~FmDmZs733-GWaD2Hd(P|XpqE}!=Qd9eD5bOfO1z;Xa27865uBtxYXNH(X0on(WCQ9^(k&vOF=LAZJ4y}@zK)ofG(B2t;LrA3@vG`*^SK|xq162qN&+s!&kozWOp9h$h}y@GjQR2aOlR|CQ$i3bt|I@^;_ue;ZekJ|I?!;@eeP;}44Q#6N>=f(0U06PGK%rYp+P(>(4Z@N`}}S#A8b#iOc7wrn%}}_To}Wp2CWND*k0@yoQQZ<5S_TX3zV0S>X$%eflK?7xZI3uEhHH;!g(stn&g@D`aTCfjl7iy)>_>xN1D1!{4RkqBEr@XMJO(Z%zt`Zg5NP|QX*NZ*$5U$RFIL5ee%{>V+LeQSc?}yXThd&SCA7Q&v^g%n1xy$z<$VBbH>%l7A<+$x+)w3gNQ8|lsyEe+bXpER{gr0LJQnk+_TAp7<_F48k4T}PHX5U*lqqWYH@8-WF2{PBr(CfaT?Cm`Cq*2I~h6V_2XKa{_<KANlvI7|MgU-lP+T=%}A_8#%-~uN9rHCCeqEQD}hiUW}r@{kwhc3;t#|m6K0*4k0HYbu-|!4nV+w9U`MxND)S>1obS=XF5PkGy~X70ZTB-V#KSa$x$+<kTp{?icohV1qKDXt_=PK1EDgtg<MZ)Yg%kPt&=P@Fh#0i8*Q5Gkx{^|VAsNF=C2ze5>ZKWr2%*?k1%gOK>zxX?s>NOt))vh}(B%-%%=)BzQ+)ioj%ojKUJgH+U`8cA2bruQTwM}oQrK|<7f|8ukH-y}IzaIw_Y_F_g2cs*BDt5m=s@S3nkMt}qkjjeaMAcmx=MD?gsd7`ty@{P+;8J1a19hGwDxy+WzNV7dGMG<k1V(1ob>UHTv>&WEhwOL*UMD#G^ftfS*fqbF8oI%eyK{PNQ#_YfbmBsi`u}x`>7`eN-hnKVwzw%4xQMwYeE||;t_yW?Mn+JxTyRtq5#G9r~+pV&`blZf5Lw&HIPlIx{X*KfV3<Lt!J4L?}Jm|@_VovUTz%o^-PZZle9uz;w?^y7<1GxPU1}g<#WCMcINZC9l_ipcI8@6*xr%Ojd;-z)TEz#MV4UCn3W_2#VTAx>U4OxPu{kek;h@#Y^iPVCJ2#*(`cr^r{_;y!=UuIQC>a<GIuB5bEJx|o|KP8_6%k?L6F>h8tuxcfqgW}u^(q#UXbW~m%%TJ6E3?&44)*`*Qt(T7O?%ytqaO1W6IO7+K#`R5)=oOXd%xn$foItdU-Rk`M1>=#YqXjSpFFlz}J&z->0UgF`r$CL}ye-a-bTsf`TV8dh+1>QB}=X2<gv$4c9gbzm3a*VB6z;EGt&}LvuO9%CeiWTQ-~yY=DnDbvrDVDQGo}EJo)$Zk`F%x;w=tN}zlS{l(DEk8*)!{&Xe)<{?%76H)#i$2ma{6!>43rgi(wCi+z(h&7n5G)?xH$T`{)fMws;q@)h*OUPmd$;k3Zwo}&h!o52)uE`hiex<DDq6LagU6x-!rQ6Tv2hg(pGSs>_1%%2o0NG3p7%+wYt1nBYI9qws0rd%rTBcj!q0;nfVquL>3;$RoJCCg)j%VjNk1=!(V8nDo=R~7X&cSu4ve0F`aH`#MjJL8ydH%~2cyR<F`4(`$?fQc+gez~}W|HFj-pGCsG=iPn3$u}^3iH-6+-{hi-LC7)9ORn&WiK1g58UGUWB9;AYOp^1oaF0u$7^l6G%h<8N<F<+G)&ej^mrzagghDHVAh>5@Z+RvP-nxj9v-V)zU=08hHeZTa_zH<%vvtzSs)K;lv-FBBCMTejQd7_vm4EI*{2x8kln>))4GAFep5YBwQ~}Kcqn*c%2Bw!sU^~{rXN@x(*tfA1gcMIe?0%=J&2AKK)KZU=Zi3?h^a&DPs(<#4*c`VmH?%tyv<{P<Zdh&%3iSh8QC!8su7y7ely|043w5s#gca<U;t@&@dz0TboI+)7`I@Ir^elUqWzoKHuYdLOv+lRNbEOLO1uZ@F!5#X-f|MVUCU``3|PhXWO0!7KMxp<B_ku*jS8!wu9J1l((DziQg~OSQCm_NUu9xWo`cSoYOj{zfpaRPhWe-#H@k^OPQOnAU)2CN^Ulk%9C&FU)v@qIpW!s_9|YF>9*3|SN*&2EO?yI>1c*{&rDw&<+1!U!c3WFXks?3-(meA-KjhFPZEQH(W?UX(TFZS?cobe)fRrcM&}9&7*(|tf1eLa0wo(45n{{GqacrzC-;erkIgL51$6V5$yg;^&<1Q6?sICOF58YOq_}CFfdCORAk(Vcp`-Vadb<X`AaYIf!#+qo(Bne#^i1CS3^y&8{hRdR7j8ZCAH_c@B!j2vG3tZz=HWgu-g?%~^5NcX_Vx$8Kt(l#~Hwk0e1#K1KPqTYAtMEh>ps%7uu(XXkK?yYetd#MgO^Vr%jljLO?18%v)pGwIyJ)GqW@+h;sGU{<lA|9BFUKo5BPgJ;H_8E0;;@Gt-5)Wmil3)FK%46LLTs|fl_S8{1u!LALR5J1_p@3@3+r84Cl9nTtRciPfW4J#>o3Aj3Vml@h+o33E1Ky6S;7}QBR2jRLyyuowu3qe_0)f>%0aYN=3IWfb`fy7heSd`X2))a^}tLuiwdDS=fQm4<k1~OpWZ7>d^aA2(I=Wl-Dda<&75I{M4n&0!vRyfDSZIkW@vjee0W?~2MWc*HFZQZ^Y3)0T3p8W+6V1e1B?HSF5+)^6izVGdTA7{ngd5amS}5Q(`k=hykb^BB*iVU-{>ua8OVsp?A^Xbro<r@<~EL=#fRg`69CXw6r#UWtE5qjJ}QnuO<dLn*{;7ftTyshWsWT{?gy&o%O8si+72ek<z^nA1=BdHf$o9S#-eTj^kP8Eq@3Mvo>KJL4D)>|S4m8t$y1bMaqCN0R0BQU%URwTGpJUVt;kC!4TbxCaV?)MUI~q-^@AmcT5sOf6L*L+3P-?-gDpY3Q|P(X77n9${GdD*o4r8+wDLC3nA*|Z$ppDQ7#V}7^Ygz39QpV`$!Iy)g;1XL@szH*Kv!l_fD4B1hQ*1}S;lda#Rr++6znXeX|=&PlO^F6G4W6@P<vg=+P7;X=UK@eKvY%|5Ly4_{339h<w5n}QATb=!(TiH!O7jFH*K>46|al)$hvvq08M6y$C)OQ(ygB@>{-SIJcq2m0vc~9u`dciBEN3TH;FEzfX;<}o)$~I4WuYm;avWT>k>VctCZiUd*I$v75RQ{EX%lE#^d;e%Lrr;-#!I8=F+(|Y#hK%kk|p6--&A%<5)Iel!sqD1t}R+MZ^gfp!~@n&Ve0Sr3`C{s(A!MH!v|xX`9tbJb_xv1C)N&PfIVg@#tyG=S}`REEh1ATn$eeQQcO3+{J)60##3cGuv4GZF!ATumyg!aV?!oGk#o<^O$?*+J2<0Y@N&o!Be!R#rn3rNGOCzvK4c9OVT!>;C3kX44CH-BD^sNZ0Ua{e)~V)n;@Crym7=^{V%G0@)tK*6bkhHaKBqHI8TWb7M3~xoHu=t+|c~r$qV+(c1!Ss;(M)^JiDy8YFwYhEt!D;D%BZ9={CG-0)|0a_AkMUtM?J?sAH3JC<aq4b>tDim(1F}O4v&~L+rg%JMAE<7sV0zM0Ds_m|ojnb~V<jrr^9A8P_N{`#u*Xk@^vs$jB5!peqZGx}L(X)gzb>5u`*?;n+sD@XO5hWo;lf{Mg-MjZ@_)s1cEmM*0#$#A>#YGM=pu7df+S*Wk89#FXXX)j^wf@zyh&b7^22?o(;=*cT<IxOxWN)ya@~HRpXjVzTr6$BkaQ+8VxG>WJ66*RX!rR&=Iwh>F16kj9~wA|NDOL;-yE7RYq}PW#PQgaP6>-0je-)~Ei$M-h1q`&Yvr3Q*)I%^RdB0ET;@#^I7`WnlQi=^;uJN~2xX=MzeYd*<cGLm7qj`asIA6U0Q)0}W5bl$#raGhWwr3{di3m9qJLF%VL}B&L*2zA<AFHr&inB-moO#DKtTUi2|=marw<64MEVd^rWQ#(cl)OB9p9&$KB)(W&&;BynfZ&PUD&(TA_~aBsF1tmMdwrIu*9g52)~)!5nGgtE9Ja70I=NF%UR?LwA@6Ee^zFypDy;}kso>+gR|5*sY#6>{te3kO~Z&KVaezXWJu)aAN8Z1YYFOhY&5hl__}^X{&L<);g>XgaRCf<%qUfk5>13R;keLnwcVii+pX9@Xqq%5|9ECuG3=h>XV@fon^hB%PWduA@ZRI*3~N8vnZtmEvwpZp2xky_xFM&bae26b#T=Qejoh>C#nI^w`#a(3B02OR)098%S~nB(5~y1d0og6o<0}MstXQ)uC=Xu#121PT^R@MGC(Ae63pqcNL>cCJdycO;Fm87OOXYQsiS-*^m1{y(U7nZU%wL0cF@wL@@OtH};YI1)t=9{vtmGh;4GJ^|Mm^l<ac(N{R!;H_YF@nX}u1Rs5egcVES7eB2!bZ@Mt9WPZ=cA{hecv2%nWpJ=C=)sQVw{xS$Pde8>Xi5aV}L@k$X$jm6mTf+n2KlMi`urp-KG&+A#C0`*Wthr^Q8{Ib1VSryL_PSo5Jk$Y!0m^9sCG`KK6=vkPvcIKr7Gt;Gtt*q|JXy0$H`|~+@~Yb&&#{GDh~@`b{Dqk}xiQD1s(n(*jd3L>>TObA&=i0OeC9<JT%MvRTbW>h>75A0rvCumn$I$E+~*vrrH0ChNH&Zbkzho?f6OUp9iO$9^=JRkMWoQ{Z=>EfapNobPpi%79@=}5{5g{X*M$>)eld1mQiHN~X;^;p$$WUs_wHnH2h}QeKv4YXr)WsPE>wOD0g~03LQuJ6Zm8Dl=?uZc{-(aExm6%{T(N7i-pZ(wjHu_rrjx*U!*2C(Zp+=_e@)lf+`I2hj?(p6Meg<4O>2C=I>a}9Q7SAb0g!t2As@@AWTTu&>8P)=f>uI$|1EV32|u?xSfq$l5od4G;Zgrz_$>}KaqK1jKWT7jbjJ98+rOE(Vz*^0beSEar~#jkVDYo)u@yWS|KsZUuT~F=mL}|-zA2q>ER5a(2exkYsrB(MHKvC2L8yuutgq^Nd8`L6T$Ely*88zVoLDbsH_LRxb1oK-!p3R*2TwBH5)fC*5ra0%6)$7e%Mmdw>wX?_7SiXURCO7#cLq?%0uq%MlLv#(2=;Q)n{NwKnFMuVF)DyYuMZOorr-)wV@>yo)cG0=G+Fy46AzH$hNFFdB0pdDjxp^B0ks-BicdY)z?j$a7UpycIB+>4vJM-*l-hxmRfhuRyw>xQaS&9GPqLlF<jMlI05se0kr?>%j|$n<S(W$JK3-}UldB;;mf+OYK4a#@q_l*!%vv#1^)Mt^9GhqE!6f1{Q?0c%S5U2za@&=G6V@#NFQvM;ii^Hx7WU76O?YZ86Wu8;XJhA0aB=#_W1f!<QZ%0|rCW-X?sXTO$B{br@=_o*PXg;8i}w3|hns%;?p>y=Pp|C>1P%@Ku7&g;0IOi}TDp0pgsgU(=*4Akg*PeSeBkM7Tp6+O<aDAq^M%#Hw7N43G_{=ps3@MLXS}?A=Gx&~Sz4UC^I#Xt?-pbHudll6Vi&%G7!LX2EY&rX(O_TpH0RNGC>v5E&(}`uIaC0;RoXK5?$k0pj@T7Om42j@6?1_yL0>C}mkSJaK0^fxZbom#<y}cKL%&nl>gYJeYqm87*Z>Di0S^6nz{*2cV4dd_HVI(jRbql4o^ch>RSqm9qlA8aJ5o~Xw2w^JX?oef6-mCT;&=pN^|jL+o+QV+!H3FMmBahJ*{2*2@S^{Q8B>cJk<lMm_5BDB`}^KQ0@t`z2-`>>f7CmEAB)#Y#3ejk3h-doD3E;vz6fQWBY%o<KW|#{7S5Y8f@$YBpwLz6`C!xMKK@uY>|DuPqA<&yITYt;2PL%(4VGfz%Q}gDPs?RoP)hq{2zDX^jt*N2#J{>XfWMXpp!%cnC6BG|{VkxmWqb&8Tpc$fWgP7?hR1n}h4p`#K&|Y}J>*{_31mCdajX`3_;x%fp8~%b4>Z4D7$$EV$*5`iRLliTTE?8OR5bBw`FQF5A~uhdfSC%mC*_ysDusG{vdgUu_R%Yit82uHFA?tSy@5Y4UweDxsNx5AfVXTT(UeN059#>XohEg~sK;L<e03x?DwB`SkmhWzOV$8yox6@<CpO%8QNpMlFs>{LWgky^SSR+_R-rF(WH|tDZ5C0>@!~E8CuHD(Mx<hHJnUM_Ti0I*#lzK(C8$AsMjK;FpzC}8&}ch>f!$uKe@_q10`pP0@!OImjh%TXuoU*o6>Z`EL9LU8VJVtOPh-?^7G5CbNO`I;K7Tg(RRknqK!{eF%v%dk(^^F~+;S=o3lGyVFHktTdBK0gpcLcTYNhp3)?j~vN!hs&X=b~?^1Wd^ibf`xUw5HB__?zB9IaQj@F2y~0BX%GnC(kueykjnu}1{g&tQRCwGI7aooXIti6x+Ma;V&(<@#-B|4R%77|PIy?ba}~aPBxk`A^yBD?R@RL_Dr4)z3Y(Ug*_4VvMM_Q+@COP26VeeI52pKGd32AkvW=exC*s6n5=P#=)@OZED@JlFx0cs?`_f^0xu@J+H0tFV($iWX<5w!II?~!0!ZKKC@?hysCj;!Vr_}Jn(qek<(MyBg-Mn&fOVhvC>;1oq?5b<q~p5&44gi5X{lN;0(lU22q>MN#XS<mHNkCme%{KU=v;~A8&C|XsjfDC~1-cjBYu;2*O$ij~0^{kN$w8WFPcU3KQgyA+3fdeu$%ILPycRKchv8EyRp&PrPhUbuc7RCAhCE+72F>vbI5A^(-0+h19<}zZT&TK1Y7x*4lHQwPtiemUPM;YAAB+lMn4R5q9_*d9Ih`H-2hG#``1r^5T^~)aow`doH;h$$7k=ezyVrnA6byf{1RqI~MVesj!nH&zcDZt^_YW7&VBLdWx^_+*Z<KTjJO!U!UrS*E`}zydkcrd<g!*tKrE|l<)Z>u1RnkdJG5LAIw>Df6dN&AekCrAQ%2X>CoZ(_>taq$%hS)-vEns<`gB1DX`zbt^Z}+_<RkYeT!8Ap&mW{NnD}(<y(?BQngDM@R~{_U4dgo24GX<V2d%9Dgt+h5Ep^_i9?fb6Hs#+JPS=!WfH-!s%fH>QBc+v+zlFvj<C=ScV-bH0;sv9{IX0!=--A3_U8_!pQ^}`z#PH5Py{Kx9T*WI7n!{RQ*!ld49O{+brWKC;0B?J$A&`2Y{gXS;KVAh3(V%@S}7If(G}uOkY~Z304NEg<C(MX#gYW@_cMmnkRI+}9x5on#JVt9H9*{a6cR=n(9;XM9p5y5AL8Y&F&96QELU|c87Ct$WIhAmj!Ac0I^PaMCtDNTV}P>B3&<VH!4WOvQ0VEzCw>>%)3G=I+W75e-?X*nn46wWkv=zh&KO7w;R#e+t56&((a)O@tJjd71Z>Vw=PVluDjZg9-og2=IVTZ>_!+>9NGt1aJmAfyi!!<x56BQ~rfrF;2I5XSu~dbogzSBWLRJ#~ks!b-kJi92)EA3)QPtK$Bgw`0qFQF`UNksQk*o|Cl|semf4!b##)VU)M+~L@R_O>DRqoBYYWiMfZ%K$8IHzvDqqXlW(LK*{4fatw31)XUKu_?SUfz_j-1d7?v~qjRof4lgTNRMLgRC=HY8fm$%RVB-vcU^A-1{8=fwv?ZG}gP~hA|#7RH+~v->;ANsIkFlxCy5`k9g}+7d$NtexKjSk$9a_h~V`IO8ipA<1VC|I<W&rqExzyB*UCht-7KKcp8{hQU;1ig!+XG=wBLm`dIjoq;p22mxT7C&!>WuL}lAx#aGr_z5^kk9d<%HkmuUlWy&i^S!@%x?QL2Iy_oQQNC7L@)q-pA7upsa_qLaX!<l;CgCXc3{Yp^VJFOg1wdh@nVGw+GAU64{?$0is&1i9QJFCA#`~^12nCG>6LI#mmYlMT<RHAerVjQ%M`T;YE3Ru}QwosQP5J1TwTM5_;qH$r|=JB0qJl~qo;UTP?=M9l8Q}{8R$uG8}1MEkCiYYi<?%BbKpEp(8e6sbE)(>(Z#KhgrIPx~3u1_NZ16Zff<ZIDNQ48Rd{m9Cg^vFVhXaAQFZ6kc)L^848Jyk<77Iesy6w+yNaZR?SAw41s#yDWivB&oq#XQmW{8f6p>R>j2L_Tp#76x!@)aM|{=V9#TaH+NhQ)8?*N;Dka=TC_?YiuMxn*|~M0|SCxc_Vzjw_c(X*Sh&+<$-jDfiN$T+*N@L81*73AJRQ#3R+*m_HR9M=35p*plD9sz{{!7MaVjA8IA@Go*x&k#?E3yHzu3U<v0uMea{~3e|!LvGO>9`ILp5Sx@UMV>VV}MxIi~ji1|2g-V4{nZ}oc!v_(`o>F{{hkCDkYeFQ4}9c=dY>IO4h=^XY@POjfG{OsZz5*bb37#`qJBz2?R#si<-4$$%B@ks$R%$sN=6kylNHV42-Mqx=L#4AmCH9I;%>~do&u-b35Y|CflHTwf$8#74=yEZ#Y;8E`^xv~&-V7N~=e?n^+_DApX=9X9A;6NLX=q{94^msN%b7)`b6%<Zn4bIV_t?l40u3fMP`1VW|&3G-8|DS@6#~5H9dEn;uPSrPBGBJm3i7}Q;&FJ2xBnN<10jI!*ARFCth&-f}{zQLP41?-7IM9Q2ohYf$6XYsxp@^6F%OrF28A43DA}0?@Zm|9*UH5|RJ!^z_PHTyI0*E)755n|E26(^q{$kz`+Q!^iCy!r9K8E&pgd+<15T>6CWDZJEH`<Nt!X|?v)Moj*D@&b15PM3O{RfYcDX4(_(~mQuF~q}i8&I?d)cH~&rP}<=L;r^(4?hf;dWB`;lPK^b$=|L=Ycopu7N!;E)G2M|-iVsUIz?yPL|tkJx)z*Ks`W4Z+s`z$t^)Nile7=!=B#M&NT;6FBmJJhXPVc$w`xa|YQIb%9khGAo@m^ACG_j$-IO*J8*$_Q7~5+6QP2$wg3x9S`{zloPA}lLIUM7M|H`O@umKf|)aWXp&_}n3g?UJCUgZk6U&&p<X+1?fUz!+>Uq29G))Ubu5+jW)U#eFMJJW8Q@#`r`N(FQv+t8SLiBv{m#qtN-M;e5y1fw~sycE&@SEdl*lI6k(yNj!M_lx0wAtKK@Mb`Bdf?QHv5<ac`a>)d#IAW!<uaq@Cs+<KIWz!cR1xugRmGZo;wCoo-y(iqg;q(;kP7ZRcI-OpDxd-yCQH{o|Z9^%V$O>2+H#(wVMf+&?Qvtt@FX)jsC09k8K%qRmFkO?bz-V|_)hP$dC0r*bQH7#=<V9I-;+wE-3}gYnB=F?^B9&zf+V9+NpcAI&B&$CsaM$(I4%+^RwSVs?rZV=J3Cv}LlWv1kyx)9KGn<Yi#2f%q)*B?HZ?zrMnJl$hS&UW7AGP)&og~^ow;$xXpMXt-H|8VpUftWV<>Y0=9RDQ3>DJgt>ij1Gun~Jv&kDfM0iN*K7va;5z1HRX4V9@=f}KioLJ>wfNJ4W4<qBn`vF8bBIOaspm1i%C!~Js=XhwiVg}-^^Bi{0Zz~+1?jlvaauHGHn$iniQdiu{x9vxK0aQ{tV$oZae3YCgE$(K+aASXE3LW1-${PcA$g9mc1n9IzzFtem-<Za%J!j-^#W6+k%^ge(*xIi+x9@)+@q;Z_U4-3hyh8|ku%}kHYCq6BK^txDjq7@8v&o(rOo|gwQ=gGk(ND;wUF|^p4axq=lc-TrgC{_NDT#<q*7aD|6xnP_mYr+wg>sw><RboruHclF+KTmA!wc_|_0tiGQV&K@KYeT@2UAgInK_>FMN8qn^>0`j*`YmoG@IEVvezq?v-rNwbIsumkR~i5!J2#xsBP@G>;EK+gfo<*HYqytxKl2I;ll1AjjdN>r6P?-^xXRLvH3QMC7;o1AamfSjlh#;!80)V=|6F3610;1V$`f@S<oES_$*j4UTp$WJJCZs@&|h4Yf~H`EI3ib{|KwqosoUq7z6p=6p^aeMIAs-Y!>OFw4Qf3Stt-kO&&NKrNX}<r^y^9(0auqAQXxFUfk4*96{cdy$R%Ka&*dXeLDjrWcJ_FX@<W&VmU+pV31dQeXjbWcnT*1Z&^s5gpl+xu^^Cf+haX&X!2e5epA}d{M`mYf^S^3hrr6^bceUsUxd)j;zD2$LpW+xo3T?P*Wh%UOWn9MxxJG-%$U+W}hq`$ReJ6?aRo|Zz<U}0|5~q*5SVuqKE80njy7|2btIS@xyR_|}kY=2JV>$dQyY@IJm{`MuH?SxQS(>skp7jlY*7|96(wQp}nT0+>)m;7G9>%f=WaM*cOIU#$exQdS{jDAk`v3t$<I|&A1K6bRhfZuQPA*t;o!kdCPsbK87@I&*C|9da10qSR=owN2@pnjAvWE=OMOoyK4qzn09|8+xKIZU=M~@1Us7HwpPuuSbOK7J^2X5X|M?<x-uTSb_)odMDfKln*|DB3_G$^cdUdY1%t~Q-EW>|UVzkvlt5sB$Ran(#DG0u5OP#VEIBw@}%N;97a+%j3qre6<+){}hD9}1(cW5;c(mO2m$7DZ}^R7c%504pqub*CzNH3t2;kv?ZO3?&3?e@-gMTgiX}>{D{%ckyln-1%jW#K=`AE~7%iOWM%n-zee2y;Aw#$w=(L;X}5EPEI{i5d0CyqB|q9n#~Lo|CGtH83qcnC<YfF{B~uAW&g)dqW{UW>+;+odf0GmSIayy&fquEWjdzFQHGv0Z03&lwRD2;+$}e_5u4^&(@Mrcm9TsEfhmn&lG^yv0D<1v{M8W*N0-puf&`u&>HpA?jd(3A;H*HnvVJ9u%FA1Z7IbQ#`)>!N6v7zTZoXt>NO^=1&}47DI3-Wj`E83m65kx#%3m3lN}eZ!suVP{tS6Z;(joSr5OX_pAcNxX8l8qK4CPai4r<%QMr0Q6inMy0i+~N?);FIN6}A{cm~t{GhR+$O=F8k^lebp|edNOkry`Ub?oJzJN4xwv+Her$eS9GLwIgt8l%oKlaofp!jXu$mjiP|R#!$x>Lq4Go!q>I;B(H5Fcxb5YgU81i#r(E0*ajStw>9-VB@hN<6|Ng8`&YL7m{!Pdvb;%Qtjy#WNUbh4d2yeKx-AH?bQ^`|DdY2(dpn-gCA8Yq0tvrd;pfMN6Mv_DHmr0Oe~yz*)&33m#T1pkts~R~Qv|B~z0aAp5gRtgLc*EY+=V5<`yb3xW#$+>!(6N2YQc)8{5F@=wU&>I5Ml(ebw9cVSKb9my;~!(`l$5|wW+z!oZMWg)vribB62dJ_D_7}$FM~jyw(NlD3HKi8@{2IH`je6r7^01hE?DKT5<7M<2QWj!Mxxqkr2!*Ea>%o-Sb{RkeD^7E~(v(QdV4x!|#u~dZazQHcxIC)lJ#Tq*w6up-0XMG*p4941Mio@G5$+PdMSC%&G->`qiP_S--I>;KbTRZV<?ik`JAUWAuf>#NTr1>OGp}yy$g+g#AJJbbp~hF-(I{QXT$bINj6u$j>Z}@z8-kSsOkb@(ou{FRPLPe?ukrzeWkbc48i2c3o`H7J61=RY!g|%{^(tkRzpwl#;-K8PWrIk9}aK+w)^ohu&-L_QeT<`<UgF=&UzCEkL^cRG9}d%jzZD_M>uEMe1oj-{$gnz7KM(ITdLo>L*w8d_6$fhs43;i57Btq;3Lr9@#?&eqYTH@ItXE4J6Srx?S&+U~lz^BwT1N0d8b*a6pU@+wK(@JjajVT7bI9Ukp_xTU;bpj>Mf0p7U--z0N}nV}Y{a!MdKnlk!L3w@3tCjdJ{5#7!~l`J-OkzX9p*E=sPpQ_9C$YugUq_Ox^F+@UCf_t^Sh7#>r4C7~&G%6Wws5p$l1JLmbE#JsmYDlzW1Uo$t%*J1$=+rZ0w!S}EI79OgoOZwQ!93Wo=6qQD7YFkC8@8s0WihZzMFbRChy`WWo6J_6trZ^j2;6?vwgfWpILrJ+&n;$vh`WmF&Ei8*AgKWNR#!2`536Nc}Y_M|{Fg0iXU@fsP@BoXqPWw@5X&2_i)ZYnXa3qCR-)_R?qjwL)@M!l(?&A(=6#45&`XQ<gt2(5zmabF%wu4|{ObFz-1gxN<2MCZJEJr$~w}(w=3r~vrhm`Z7$!xIV)>s9A3rZ)vu18)YG7vhd4kr1I+5KrryWnH`bKF$umb&q+pVr8FvbOI|`yw=Gv%mxTe!!t6S)hmg-&-ASqKdh;F3~2H8%lXzDP@TJk{)E$M%EzMQ?~=8;qGB+2f2FuoC;>UW%sIeNa(iiFt6kUsPR64$X{)&=^K!<?=J6tAzTN}j{!I*VgVm$IB(#=5|AOBdpZk05Pu%X<SMbU)zu;5Pf^$w96uNB^R`C@WjbkMs9WuM-SMWL!W?kXwS)v;#g$wk8a~JTaDC{y3O@G!|4mgE0PfaO>1Wr_{~wlI{J}o6+m`d}DU`E=45+9YjrOqO`+|D4$#mya^^^l82Q9Qnyj3_wq!>NQ<DPK53M7D+5jnYnP;7{C3}GSp>h9g$B?xznx&8UCH?rL>O(aB7as`uR7uEqk?#&a)jLo1;RgkZYk4Xf$1``+@GP6bESr|}+#nP}Y?FJ0qR7t<~jWVCC;o>o^WUT1>_ahyjXYP8U0=@WsNt!9F*8?Pq-A=~FQV7ftdnYb(6-)*0rwMX%625N&$L>{*KaH}y?_vdw2cG;ck>gG~`mNn99u5aqj#AMqUQBTFO(N0G<Q;Jf|5m~O>7iD<%9z$OiGC(!Wq?gqrCQ>(7=2&LfqYOEAA5ngDDbnA8kl>K1H#37LE$fF{=EN$yl#(&<8(9OtJOs_z1pjol?U7;Rl>((i@j0O^z~ojdn~#j&Ue-&Jg%%wn*#TfKaT@c0G8VbA(>>TK2@G&XavS}MYysm&>-ol%AanJ-3OWG$m`#W7SDi6O)Jx#+w<@=;0P+F5D+?jRpUT+`_6otu553<^pHtXAhe`n|6#oqD$?`IW0Ljf22+fnXIn+*0L803>2fB@^3)#45A02|gE8~%j%lppLn->7$K=Vn%QWoH^y=XsC*GT`OO+(cJ%^GtX)7xH)y9KGfUQ|%)6m}ZZY4qdq%l_C_{$)2|9n7jv%rm&Qe&HrugMZ8?+oqhogkK^OmXik$xjYxyU6~@dG<uhggCe<kC5LL!m;HD&c&i3&bF$bbaOa7;1#=#sS!-n*<<lCXgl9WSzW}J3Q-!PO&8hBQTW$f1jGLM=}y}$p-|WRCkQXoaX(IUQ`@NbgW`^N{9tF?EkD5^M{z^8?qv-QNf9duF}VXBWFt=9EZxp^r4DWzT``i6`5m+3pS;yV%F<7t#-v_gL)+%j`Js3Ncv|S_?86sKS2|e`6>r;4`f~v)_qz~k6O?fW<YQ~*ypVPx3qUG6)}A8m^I3%Jw6fBzCjqv_y(XhL8)QwdVNNibkC>m^2-d|Cuy-Rk1|+ueHH<8Cba+PD#Awg>X0+dsFIDp<BLQ1Q`LH<fs!=k-a~s+c@pAF#ikNyk^+UW#0#7C*tOn`QUKS%Bwe+Kltl$f{-+DhR1iMA2v@tp(v1%bp$<{$C4$^H@Z=#iHZ701n?SNE?2d__jh>$0CFriNfz798JGF_lF5}%2H*e0>o4YLXE2a_Z@#2vYEw2{ZBHk^Ft08_80ORvofzKJmO{xADhx5ERSE}tf=kX<d8^`DaD!9M&h%$h;T_tCp3lKO>8T0{>lbZABqvNmGN{Xn<Xi}NdYP6S)cF`?$;eCknY5w!VLpcwUXDp<MG^>cX&@9d0tA!>2k`<uVdf3S=;E*gg?F;2?v9vY7mVo;ynGPBmC`8+%lzb$d(wFysyqqkW%Djvz3`fj(K<0+gJjA1ujR^1mfJNNi^B7Jf)N7~aQ5!N;GCsE;TxgSQyRNxm>HD~gjoU$(RNh}ZBe=MQvMAo^`VQz3YHA;n*<v7RgDu3yzndl5`U~?Wj2@WNxSgC`va1TL7osII%<0H8DQ(rJkRm#BFupRwy8r+?WR6YPMaZ2Qatv}r;HV@?i&46Zl{)r2|YJ?F2Oj5f;I9!)wn>N3eKpuIbSdHcO+X|T|(Hot=kDJfhG@<4xiYx1sj`4WZJm#;O83;{(s{!;q`o9ul-Q4o*vsyuR?6*v<AwV&tdRw|ovIebgG5G#3&ScKs_x<+kiL62<&{t&U-=tyYR>?^+b_ifa1URP$951ppq@a(Bc^<lpw6g~x5m>*zFn=*m)76)J_Rmfx_^mybVE!9l5Bwa7dI4x^6=Bi%xO>4a7;9H*OWic)&mm(rywm8NJ_S^4;39~iA|~+i@c1l+X=_tFg@d3?Ff}psy}1>iIfd$CMM!!dDs$j~v!p>hM^s`6cHGggG<e_1!Juns!w3~c>dIL3q9ns6n-K5bOF+TXyf42+sFvDU4xZCmyPcE8&nq03i%JD^x1e$1U!{cTT_UY?Zp9(Ou=->MC>Y<r%pTY%XD0k9^|`Dtra@@xBj`s%2^`6YlK}dBq&Ys`z=rVWwJN??b)x&~+bVI%J=Hnu6BGk#A~9VZLY0Wv#>6YAbaXW1dDWq*j7BQnO%K{Pit2z+0P_ISG&8eXBCP_J(R;)giestUnJ~stpzu~7FWQD3U&sHGjgMOzUX|6+CT+cNoA54zf-T$mx}s{5(akw;!B9>K0a2G-!D=l~J};#tllYV;2~YF`Sw%(_#=~!u5r=72f36rS=~!AD_oP}7mFJ-rKfcTnx^W@qh=%T=goG9-=lYYv*87zc;oO^daL3mq7))zpoII3E6pM25^3;{SBB0K^8r~kXA$=-m6aEl<UE87hqDH47?8y3h?`omUpwEl<`VT{~;jAys10ic~23V`eh?SPfh+?=9=mE^C1G(tunc-ORTxuB}iKXhV1{Jq!(Z<e%01va08gUEf_1IfGIx0Mq+5{2lAXSAJEX?orP?gHHIQjSFqoeC3^L#zw;*jyMuec`+KH^^}5T-B8ykAGV1Z#jV!mF*{@u`%9<M)%HeW~(M8)UtQhgKf!mi>xcmOkgeKY;G8HV4$(a(<i{iV~<1Ct>DZ?HKf(fULw-y$^w5TlylZdaf`azgDCZ%AD~*R+8GC9sJ$MYTe|$P*ff1Oj9?9ff?Hm5+R|<^dilH|9iG+o0-2sxkT^Ch4*!z&?z-EZBk=U{fRY9S47E!WpmCA+PKP>6$q%?OBCzpXrnU!_@ep)KUTtEf8QoQ+X7-%VvxH^kD~|vH(y{LxMI9E<h4GVGjUQ9C^hTZj@-?I?f>)=w3kqAXM`Su#mE8Av|~_se1eN|LSh;=dDDncP<e?Wrj9Z3z$fe>F%Q-Q*@HPz+&sl9WRz+I;@Lq2HJPlhon6<y<llC_eHaz>w5dF55rx3+X7@F2I&?NZHTm-bUSb)qk|A#s<bb5Nl%kHhT?~*H`9P_jG`6~<#HbV%B-GDcyId_EMvg40_c_H~*r^fq_`Rn{tPBKAqVJ;?OYR$hFqgp8%aQz)YKRhh(;aL<t|t03Y&pi~3&PWi*p*^`|8ZmQPcb(q@MASHptjh}6MQmfyrYgdK(D}VdkbCr>MtDKFsen(9_AIicPZ0J^Oqpug)gE6CfIE&KxF}sPS0VmdVJmi{&u-|s06A%c`Pj5%{ZNC!Ul%p1Wh-BCL_{knv{ML*YV8}uCCo5G&yUl$x|n@j;vuXd-Y#rWae6>^(-B`{_fCK2I{iMj+HKSF9+Am9Pdv@h0i$QLm|2KRx?M?Q6AzsFO}VC8ghpNRO`rfngfq=q5o-XjzS{B&;xdYv0JOycrv?$Kyp#)EdmskU@<qz-!b6+qFsnr?`5oQ#h#d2v<zw>u-}?TeWy4&xF#31!gEt$2JNknO$na8yb+*|+J@?DscO*IWKo-)BL`+!=qEteSFG2d1%`fdMwP%CW#c;b3LmEsCkFl4zl=vq8UBaXNZ0P4Or2zC_U4#yFld0ABOoE>Xu`4+r2?bWR^6>b)UpL^ID$_A>x;O|XHF)5uB%c1yB3|$R<3UOW}g$6MY2PdPx9I~U1gZLtW<Ayc4S3V5q7Uao>9ObwaW(Db4z<d)v{DRaL?QvbZ`V+(kPRjnaD_#RB@YsaRQH)TNzvC+jm+(_`_mr>F=WQY6^+(3<T-Zhl<8%PJ>*-eVH6{GA{)t7zAeTV^Hy)Z&^SR=eIW{MG>h)OvivltN&`WIe=wMdY0bEV<HrB*(L3&y6Z`M0Uk(OB!dTv)tGcRvpD*n`>`wOua2`3Sl&#-2jhsV=OUD^6z}{*wb)!d96GL>6$vxZPeJRaP*>IzR#4~+^|Z#najuOW*GuI<YEl`prA{|as5by+#GZaS(iEHxBYS0eL3mAoI^QC};C1g`tkbEb=;@Z%a1A}%c4i+Ew}q39Mw+BCo9|kL+j*aUO@Hi-tI#}&k5_#zdi}tcNj-pk=~-z0$3ME{xlyeMW5XBB6@23SgG?Kb4=DLA7BTLhYI%5^eXp>@*Qy%C570?^2@|tZlB~RN@*$axEk#BbVHWN=kGHMHGf`y=6s7G|0(%h|oSI;hZrZI>s?gr{cI&edJ9(bJhoB|I`m`#s&)vW%?y@>hv=3pvKR!#V#;;hnmYAc15KBEWE`JF_P6iK;rd>wcg;fOm8aaR?c5+wl7{+oVd0zd9EX%u!2luRJ(tgn-zT6P+^;`)xt6_>#xw?t4vCtf(lVUCp7KK<mm>zis)D<TaCMi}d&%ff_5i~(jNyUPLK<6p|dsOVOMhSNAWjqgmmp&UBekk(jzHem(QHyC`Q=6vpSnx}mT{2+;MDCy*{|xbP{!Gz#_;9{x+pmt(*cM?kqiVFVtt;jCTBIQ%(_<oX`red~D4j-z*4K@UrckB+uZ`lCaY~Lv7L>07BFY}BC0VRJaE|vr(_iAJHLG>-4(NfAklK#IgAmBpR&%h$V9QlH=cmw@b(w3vq(M`CRS}rEr{S3P%SZyqITMkxYIrj~kdwOyH(Bis5GEx5%qX=dYN<kT)yI$AA~PP^f6Ph<r{3Vqsyu)>dbC6NzZ4(d;)BkCUHqT#O(st3I^lRuoSKh!@^Y_l6*JWB9Ge_Nu#V@fDwS@JcQ`GkAt}`rF7ONA@djg?=_XyDI0RvW1OwA)SBkqS84xr2Zin*hT6N#mKo5BzBYY5JR#%uUxq`T)$PlBSuSeXN2Vvl7erOnZak+#L6;+<3{CvDAHfORo2c-Fe5P{#yFafl`PsllfyDg*{{*sbL8`EB9ZL6^yY+<~&CiLz3xx-<QjjzAm;h+5Wr=~1)Wu_V=JvBe!Ul^KwvLDi(Y?C0x%~MZ8PE_HEYG87u{;c56C9JBH=~%6@Jl{A71?@u%MW0$RBnJ7PhvYgHa0HpP+G~r>k#mQA6VuP}AXKS?j=I42{k$+!^%A!JVu5i*>Vp<?e-!~|b&MoT^UyCuqP$SR{0Mpul}JohAkLUOC{(tHpRKERhnDC)?b_`WdmS|C&*mT_*eAJe2+#79Rbaj_U<!YD!+sF;Im;$~M;!q%J5vp5-w+7=E#;gfw(^$e0~m{ik3-K=lMz?NBUBus{IC0<AN8K%=?0i_#}|~O03t6iwTU#dw?96xsKihCn~CdG%W%Q2og}s_xfFKV?cb8bdptJR`JM%<zs0+kINg0Nb9Sv4Z+*DNfjOq9abl7tg*lo=<5NJ5H8?~Y0Si;Szv0}H6%g8?FWGxuJ;jP~sieW**((JqPz|EQMK<H%@dlK~ycctI8&j_8*2n^z4sf;I<Fxr?PFt)nZpLZ8Ccm116`mzmey%DZbqa&h2b8ix1zQ_a45MlGLQBI%W_1M6yA-`9`e)=X3E_ss{`j`mr-}`0fl?G~R_Y<)*Xz7iTs1Q@{Z?emmN0N?<TBc%tQP22hgokRc<|?Dgk^*@JR}<bCCIiM9a10Lx^0m9hA!b^`!vo`0OsxjmOpHAu?`P5*L?7ahN<9sv!u(KH2*V5pP%_%$tdhgu7*?lhj0!}awO^j(9GrIrd!|$DdHdmO;Rv0r2Ki{1C*lqN2qE-X`V;LX2?)tyv&G%q$Z%PV1&C<C%>MhkV;yLhYwGJ&0LQnZT0bkA|Ery_uc4`>4I>rFfOF^U@8HW<6`7OGVtIL+gTbLT>M6?7F)n*%ESWa;Sb1Y)d;e-T09WI;xnrJ)`Tfxfzs6iUMtny!P;Y7obp;2SQm>|y3LDRtw}ZL*Ot1+O_YOVbeL|JU40{yO7N>1|LA>s$G{9_Q{B1-r4o#Fz^II;y;)9Um`HYCXHj8=!Ae89zqfm_Q-%|JcbV(Zg6kDwUC-r<2A+{NY<Zzg7Xn;R_?`%k%@Nv7oU0aL5<;HTnx@FYUa*C!(ip<IC%ywb{}3n`#kCVEVMm0ug(%@4Mkn1VKYULZ_uUjT7{vVs!8dL_8)D8l0J?~07?|_YPfqr2Dfk2}+DkXjXs|WdO$APSgkOR66m?@gPx-<s8H{VNsyJR=PPS)*=`o@Rx%%VK$X3OH_0&EPF;n&#S*IBsRu29;yhcM(v~^2=Y22<WQ?%TRiV7uRQ5SMbSwF}^Fm<7tjyO}t<;JMnUB)h@R2VkGhb=8xvN3LHyzAS4bahUdxR>Ur6QHeGhkk6ytM~r{pe0Ko6HOrLE5kmHE-+F}p^T+<)+=)wH}ir9#1z}ujq~l-V}U%MQpjuKHv=j5pu>H5&SI<r_xpWB(keL!vi|dPtQ7jk`;EVB7C0Km=i76j*O>{ov!91)_sFJ5BQU6}L>0@wWrDh5{3}VpK8#?$z^g6{##(!H8s>4b;IQIvHgu3pk&^Wk;XYy{pLTyBTib$FG6U-|$wSnIY7%^Nd{!zC;^AftvY1x}g#ZQDTi5+IZLy_mlxhM$TxSkeU}p37$^ZZ1#35Jn%xNJ@vMa=&db(I{V*&OqI7aRJQ0;*ykBv$&`=24H<l=llzc5C3ARtrz&eG>fGRXB0s%YgHg0=INqgOB2NXaoyxwOu4_j5Zf&0zkGV-cY{ThA#1S05F1N<6)4vEu(JvFet5>0K<ZnK98RJQsAYT%>N?;Z`MERR2gp>Q)dPWuhys%}qsQQlWS0Pf_3#lbF;<5{*FDG;3-WUp;rL!!tqWAaTeyGdF{;pL9HW<&o+b=m6E41t4gsE%Xr?4b5C)-UvtM481VGw?>4<(tsWHdn+I{0@ZTB?_IIJ@brFX|Mi9#wC~=Ux>$3|tpN<WC`S<i2KX3u9LgqE&18I#a?5pM>f|&H%T~^P@O6K~XcAS$toWH&<j9YDPu{2OGcdb=8UxuU67EsmGyCK4u~v!9TnIA(6l(XIHQbbpGA$3uhv<zr_?{wQ`{3z~P{=O4?(F{;wh3UdP9VF$31aBNXkjtfbPq-3KDGxYKAI2N^Tp+4_}b$RPXA;Mr^7e?0ph^9dVPeMrE#L130FtQ5M97(x-jVR_P>^awY(WC59mBrAii-qxHS#rBC^n=j;j<UMYP*-p!0@}*W_k4(zanYJV<=^S2$zK`%<oEt`(Q^DzUb4{*p4fG2b7Mhgo2gY4#;|g0>uVM4fxOnbq{7<jrQt4zk8!G%anZUTMJt{-*8~#$dD^B*nRiGme!}&fDdnReHYK=P6~US%Q}?<Q`-F7m-X_=<0-(U5kn;Py}Jv;w;WO_XG`Za2-}oJ#Y+>r5P4!4v)Ie>rV$wKAHYC^!nPzvE526pla!f$1=}TdNV3f{8O+R$sh`2D8In<IB@(-kNEg>BecJVNYl^g1w#8Nrz*Ku3l%^eB1xR1Q8%WT<GIn0C>hQI6I#<MEO+;0iikYRH)OA+RD|C?3x;uxAO1R*<X_j-LI=D*k)RH18B=*}^u>&<|MG$V$+i~;9w6YLMJXeL;?rI5w(z_Fn<{J+V1E>f&I>g@Y}P=59Sx+Ba0!gF9e-W<RF@bGj^(HFNafx+QXX;SPJDT^r7DZ2_@Eon%mDf9qW5&ZzzAEW>aWrwx>A-CsYVzqV48lTI6RsH&d206BNdZ)Q_~6ctBMuE{Zr3v($8H!&nHHq6atlA&z}0zXt$Jle};_ia|3nA2rDp<3~36&bh(%&a04*G-uIMBae2F~Y^dF*Oeh{Xr^+Lkr}aTwwWffDAV5&KC1~o38adpWNnFmvjNTyY7I}#0x(L@dpVWJ4<r6NcK>TwR{pHD`q0|PPFzK+yOy6=?uTel22@S`S2h>&S&R}Vb_PCMYxxI3v^+EKUJ1sogk|OFpy+AvNbcou3<=F1!SI?{sH>9Y%abh6iO<Tpg7Se>aEQVyY<&8KA)DsKQ;&%&`cBPcQSYzraQSFF}C`#@*Q^Y_};I$o1M1Y%W3~H2b@IX(n<X8D2&ujrVJD?eHCm3K_CJ<d}=kL9h7A;smp4tfPc`v!^XzOyvj&vn{2AC3tZCfbCnDJ>gl${B11qrh(i@z$>NvSE$&Ll_EZQ4i=^R312RmYGaE(7D-jet5lZd|1N!NJA9O6>)+36}Jfh@$>f5rfrIll)<yXcgFghf2y5`0Ed5Omp9v9q5&;o11R~c<&o5SSig|`XK(YU^0<p%@It=xiH01L|D!X6(dR?Qr$vcm^id+^K>Ix7+yVImsN`8*8S}ChYl0!J?mJtf_GKOAc_IO2F|VSKGw&ny#WOn29B2CH|MTKqkCQtu2|;2+QU&N6=rANNo8m!F(e=HL9Gv_p)|1Yx?do(z~vL%A3i8pYw)Bsm<0mWG&w)>6B{O|(A?W-jJmY^n!aUErU;UL7^Mj!c@23rXsO}k*nLUOq_96k()FhK)hW9cl&Z#Y0ld+e8J4X|O~MGljhZD!q#|BK%THDkYE2$ieu4FaQ`CUzxrm$8&PXyJzfm67*(*A6!iU+8lCJ9WA`MecL>nE4n1g4)h!YT!uA`;x<mJ}bnEAQyZ@P{CrBj81&iH-Deo9yAl|f01h`geyo{l3Z&$nf$5iE1u0Q<vlVH<@x!D6y-W4ZIQh`o~U@zB924H227+@|qT^;_`FV<dR+e(`)bRMM7aA~3M7y}UKNq+qO6e_EX6Jt}*_)S%_3r~u(x<~_W_gJgD#=U^$-gSVkiZhd*>UK%fKZO&iXtFQ8?(hxDL=wdhy!h#qxduw0o5P^FfWpyl|*=}|X4YXJBva6@f4E~+mH!J?>u2qgPO!a3H@3=6TX`ER-ngu3@ICQvwpKi(~Ud)uvd6wp$ve=sclyIv$(v&4h#t`WxjDMjH04nEg(k>`2Uq=vDe9nr))*w)sbr|Oct||{<O^XS(-<cb|+8BM>)Kkrt#;z%}J#?pEM<p@k_g}S80A^7FZ>^~A<G)?gwmSzC!m&|;LULe8%(l%|o(~YTcpjvD2-7iDZHqFcLeuGU*$#*Q#p%a<m<PH<z9tiq{e~Zz!u=3?>s)bP;ot3hh0ul-NJfyE9p|wj{KqN{I+o2csqaNIvdg-I)5T;C00KOcB@;dNwB5B!9+O6`^cNw!W}ck3`QwVy5B9+JFS(KeVT%#;v{+IF2K0<CeV>_nCXW?L!H*;$=soIVvAEJ?K6>LT0hiw?BHN(k)c*jwuHa5BZ2Hf%odiwyA{r%KE-E;We;^`-l^$GcQf)-b9^L${aiCCY1(F(Tj=!;^Wf1S5Ywi-y)OW3tFl;=}uKoZ8#zEX&LBpl8@-@r#>V6g1$p?ZU>1e))Z@MEeh;ehUXuWvK<m|NR29+$roa4^Ls0u^0h-+=gy$RlJv!cPmJb3Pz&?5I~%_1sG!G=e}vXOg&CrJmZ+|11t1AN&MCq-0JXM*4ETCpOpD-DoD)0)GsPHb5nRcGu#qYt*OxqdCxg13aPwX35^JpARVq+!n)hI1xIB%%fs-fh8giG^)n*~bC1%OQ&7TRk9pKLer1B9D)8QAmMk+)n&!!z`Ma7w?(h0{Ezs$t@|_eMLl0f8co?o)bFDc+zbk{~YIvC?yGRR{3n-4yZz?E50pe>$IKsrv31bI=&RS&5CTd6C`EMUOr>=H(F}N)o00ou>*K}4%%REg%fNBYwo6DcD|x5ILs*ysy%pjnOE2jH&B5~?eMpqUXlG@Tx|e6U?6!5it3?aNd(io*}G$!u<@t}@Nbawxd!X>)5&_DuTn=agX?9K?^*JOxcy372Ws-?j9JIZYNcmSRp$kZ9QkW6aFi`feB>MoM}Z4L3PBqo(UXk*$|*9K)y&PtiORVQWFsBAaSTl{3VEN{3Lj6Qv2TN|GrDSqZyz3BP{td~dWR$bPVL6%aq}RX`FL!K59CL}IHb+D8DsfhH9jn=vy1XIMSdntqSJ(1X5Io!X*#8GZ5{^3^17K&2t?7~@x8a=bM&*(>br|ndut3xVmk}Cj&83L8Rhohf1!*OahXaORui8l%781>jbsye5+LwdJp3SVVmASY*Z15v(ik5Jw;DuS=Ns5qjS|6_6xDC{bSb1fVh~Q<_Po{qi!T0;Zx)AyasGZOIr}T92rC5#GeijHsMTn$Vj;Fc+#Xp#M#m<D=__hL;W(sR3Mp09H-%AF@BRQpTj#thPP<Vx2;g9_zeo(4c`Awi1|uUSFGO31*{za9TvJ&hlnm12xfEfxSTK_NfRDMkJ0wQVZy&05!jbg6063fQ`l7=}?)Y*yEM_O2Tv8O@l4?VDzD`;2Ix3=XV;M8RUyaxAGkTPo9;OPek23A=h}q8IRHtHV=lXqmAy2|RqKCLfEZxKE#ooHN9HO?M&a!2bcufo(fJb=vHxM|D;&|dbs3h~NMM4TpH4%*eWBB)Bkw=~bu{GWIHwu0-qYlCdJCH<Q&A*awX>iYlNM$1vxTeU&6$}Eo0OUbmE^0@0G=G;4USXE*{E>4q#v=E*s6Eu|#zM~GT4h)q>40M6@Kf$jZ^0hmOl|gjxoO-MvO&IBZhZ}JF+hs43$7{+t@be|&u{(vuj&({wOwjOaFcp4dOskzl-(-EmFg3yYvk&1T^MG5QY}btrjGKZOObC%hTt5ihCQx2SDU7W86=V#U_n4|e}1|2@qDKP*}FQuzGc8S#_mEiW3TD#FTLXam0a9L%tbxP8ORZmzKXqUpD<`wQ?P8q_8*h|`o~6c0#h(D0h_C1u-EoA9gX5(nA9<!L7?~{fXG4>HFWjz;6q|$qMU!GN_5f@4){BcdA&@y!!@D}Wv-ef1N)#Lz)p&}q4qPNVoW{zR=nQ3HpgZ-h5;1rzpXRAPDV>rmdSZ`abD>BKj_y}`w=JgMuuL-=V9GoBlfkElf?=#kgAlKp7$f#Z7&}kgP>U&*lvJ{em0VjZyP3mRFhy(ALA(C4LyHygiwTLYp&oszw=zVG&3lMwYAbP@zP~C_1*WjkF)wez6?44vVG#L>@WVIbr#*cvOY#zm%6i_3h<-z><-uzPN$k57uRHyfO^8U5q;To)z8ZX0INw}x`hA6K<YE|pXu<Lc}XIX<`1!RTOzMgQ0R{lXcunUz{TwR17#3+p|Fi2y7WtcnYr8T4Y*<>UBaLswM&)RH-thPrb0x%axTw(w8=*j;6MtUH+T&T7W(j0KD9W0I3}g?+I{eg4kN)0Nn^yIsJt<xVHsD+1E|=6(H9@Cd?~(K5J@1{=XM%H)6{t|X-#nP>x*bZFkUT{bq=vs^}{_sf^GC!M1h~e?0;0?J(&N?(<R?MUoq|!HV#|WbJlnMoEfEFE`oR(+931I_xU~tmGciAmzXbE2QzQ9-)>BC@7I2%@Xo6s^N6qMxxa@(Tj6^@8k{Q6`Yaz7K|^Xy=X{S+AUtYgY3y9ij{!uBVnXVK=_gChj5Kn`&GN+_?^W)D!$e}(B4guyR=CG%Yxj70iYOk#tAydWIgAUxnDr!`e=W~fu~rLMGie%a*}jUT!S3?wp88u5_dc;c#q~t^v!b&unXcb*7_yH~vLQLmB=$NO3F6o86l|6fwT6YXvY3U8GL3h>p08kHT8%%@!(PHuGT}HXXI^N2cMH+Yabrc_w>pkKxP6U4#5v+C0pv@t;%zO?%&JCqvr7~8Fd-cMaN_#DHs#cMFq1x&Af-z%$!am<hq?<sB3!o=_OzyFr6s>)Q#w;4k^eht3OWEnM>5HJ3jX=dvT^^rkBL6JzyGlGI(k|ScCSXDinhvk#aYyub1wAun16SWN)Y<~FO*!4r0<G)tjMF18#|R4*SK2&mG)8m%nFt|*sS~E_8ER<ru!R-#_JW5nPvPvKT85)M3$)5Yw3{5-A0)rDzhN#@L@e*%O*e_Bwg7w=kh5RMcZjs-@21BY=JBi&N5T{D-}0>5uNjf9K}w9F#>3LcFVitoeh`3y5Bj!clty0QN1)F391!o-Ydcb#8o)RUEAVmF5+lhQB7eaI$5LsNT}cwk~&SGKivfB72l?P{cgAgjRI*y5KPPt2(bSAFW?5#PrAdsb7BKvTeWo7FPVk7{lW^9OBR{kI?c(KaTp(=Jwz-hJr>V4-J9cCj_h;aQHBty<gQVN#h(=ytC1PH(2^s6iT4Z760vm@l>f&lkY}FB1HGfiE&RHs;fqwKP=gX>=elH~%g6i2x^JA3k!ZXG@8W6tdzD`=(9Cdg;x@|0OsF$DUA5c$AucP8)UH|Ks$}4p+1*_OPU15kJHhlc>VV+2?u_=&ZJn1ih<f)|5H!XLqj~*EiPxWolM^5ZUI0S=bg{afe7jdUv0e!XAGOF_z@#+0y*Qc#bpvGO5h`2%Fk~KmkEHFnbwFQ-vB0>dxKf5&=2r-tveAo`)Z-x!Z}}2$SPWe98+2;*Xr(D#9e{|m8{huvFbRDOUjZI6Pxrc&vbp1COz_&qqA(iNwz&(}F#KLt2UrM{)D>ah$rK?C!I^P`hCpxMt+`s2wVU?s`Mq0uP|QZ}o>lDnrix*tMw8R0vXabt%fk0B4`X{g;{2P>R^5vS91a1f6=`jn?QeVau41WBsv_ANcwps%ro+Vl;83<@`covk?L50)X+&(Cq$<!BN}5Fd&)_=RQ<hs--j2{8Y~$hwd=m&#J5CKVd-``Z<bJ|E=_2h0!p&1j>VoP~;9_gh0(LCT7y?IB+A8}eYs|a2A3%?fjvIXZntl6J9^wLhh2;;O2o^W2!|-gtWsEhji$~umvdMv!&qq1m#kbPhZ7IiYJBzCVy#Y|Nxs0~AuY&jX_&O)w^p(KYSMTOr^{rC7y?7S^bBdJyn-A(uHRyww^?p;_-}y^5(S|q2+Z7bn)Tl8;0-91~<`gjqCs6lX4fDQ8hUPRtZ2%>Fh^=y4QBz-pGKf>q@8A)-;$MDaT!TB(P|5i>gWMgf%d>$jb|o?eWDWEjqN-v=meckeKl(VEQon1WKTrqIG1#<UKN|%8?94vp=f`lc7f&pH&nEe>ciK!^f2vPkEz;%ZzRzjUlOVN%2SS|9?52Bb++OsU>C1I3TPxSBag?FQki2p!2&s2rDgmyjbeNbZ!$P)yU;BKN_H$c8!LLfS!}^M7n*QI51J`=?>C8VY+SoGs&BB8u`&;gaVc@PDoO{Rq9ucki@a*Vm$ci;MDc%fo_@?BrVp46UIdy^mn~SFz+{wI7KCRWmKxV%=lQbZc--0h*wv-RydE#o8duZOvdX-PdmDr=SLe_rKYyZ%eB;n#2`19DJfq0+!PU`>}Kq}M&k|B2xtz}yJp@!2^dZtRP{V=7djjXR=SI^7;onf11eBn*Ka+#+I4yg={YK_bnv+|_xq{-0NbPpUM%N6(%)NJrd2klU}MuqV<d4YZL^&IqB=-G0<jjU7sx<C8~Y3Sl~T=AB@nY$-(8@C}b{$*i8<JZ)pf=Wo`t47@9M`ia%ym;IT%-4o1ln!=}X@>1fLR@iz6G=qn-Gah>p-eqrTUPZ5xfV7UlUOAL6B}&iJ-?s#0CT7Wf0dxOZa)xEZS6i8z4grg;*zFR_2L^)eTJj|Z5~0z-yT+N?GaV;1cxh%h81GGQQqVfi=!su)VzFts;gogXs-}G;gKJUXbSgWw=luqxXyV@M)!1h$U@)n$|rP#UyF<A(WU8UNO!1Lj<??A?gMtE9{Ks`5YM6q8UNB6j=Z6+y@{|Pqp9OLn>FTT5gR*2o~^uNt*L@-(uWB2A_z{T*K{1`LQ%5S(w2~y&(_hudDwLdS%#q<=mdOxlNtBw0*^5F#~|G4GuOc72JFulkDO|G$zG~`VWikXJ<SvYXt`~QpAU4sCv>C@Yct{62XvU3{rVTGC+npJWb{z#17jY0ul3pcdAbO$Q+GmSSqY=%OA`6MlbD_7^_fLPOC)si@kYh6);`GD*Pc<T<aTa83-sOWkE<a7!(2weEF_J&St8sgkiX*lY~$lf(KH4;k3Wt>?3QBRcmEiux4*@8@WW{rYqa<>b4;K(f<qF<&l<o%l7A9F=BnA$R}@!f3fZ(KAUvW<zbu5_FOWxv;yJ5ec{qsTF_ij;_r2D8f`|a;2io~CIeW=*U#fAHA{z8S6nsw|L%7l6{vfR%=4C-a)f*&Pm37FyS7wgD(_dmmS{LzX2W5Al>IPkqiZXp6i|}BuS@?Z5WVJe4>!W<4vxOKVx-c`4)jZOaZJS&rvDeUdq0q4w>F0DT5vK{-fW7E(9tEa(oau$Wqt7AA+C(Vey^5szc^E>x#TfN)_pm%(N`ARRy>xa5<+xX)3Ww5gF<h)fUYgM1=u;h<CAo%)*%Lqp6~?ARr=$Vc0dTCa&3`+9ydXcyeos&qzn|+^NQ6VhvU?a*f)$l-QlOUlBDQr+k~quE9+_2E8xYcWpFwR0OrbFjU0<SK&fwROBW{?A&P1Za8aLx{9b`N)`la`d<0B&@y^b?HmsW7Ow2MHk55Qkd)gBTXDz*5<m9H2MF8qR2>XL}Vko9@>xS*)?5F?97QY5Yt58uVAy$=}oK#bAJaO*iE2?iur?ps7f5wknF82~dhh9K>X&a6;Z_F~WIb%Q$Vm>1zqSUC)qs2OOj4RX{0b<LttY#^Mh+gPw-%qTPl#O({9C5RycKw@Q1pCPQpG`0dSYMGYQn`)MHB5oe!<d*H$bB|p2mdE(VkJ&^n|3bm4rFs>^k(6Kh#q<jGn;q@wsv(*g2P9+&v-LA@&19qFzyCqbiN9w@vs-B!s+Q)$d*!rw1L6F*Fu#bJ+-g{XHUolQJHB_;e}=S>Z2Mh9u$`B<S*OD0^cm*c;1xSK)F=$&qyeiK!w3LGRbD{Rj&zQE%yh1dMN`N(wA%8Rn7vq{8*d?cHx)DZB*z4i9HG**yH&}(NzY&?h%H20-ZIwjWEZ!#yo76)kjc2eP*)f<&>5ihqOc8m>TSvmW>S3uTR7CmP13;7{e?|z|8M*#-R@YTp<arPrm#?6Ya-Fi!JvLJHU?jQXA_|q({Ll{&v(C7_B(6mw4A_?rQA5H5WknPNSsJOT4o<UAWoU>qhb0oTzAZanXo``!);KBEZVovujAGMu1Hh|-MJHV%Ucq!wPmJr=q87lA#VwLX}gFd?<D3VGIBl@Fomeupm`H&MX6}NuqXL$gZ&QVs7{7MlXpuGK<|Q{Y1O#)#-9QDS<YDrNBgnPS0cB~*6D09@2LZ5jR{Pi45CsY^hTpbIMwB?@%N8kNbZGC&n0E0MwECn0?0zFMW{^vRY+v-7DlJNzrfrV1#iHZpQP1~3f3SGzW7@8cU|ouF1E$`_Bd)JK@O(RnTyoS5okdF$Z4v~{Xg!1=S0U`;lRH%KiEAa-M!W*NaG4^{hlHkP548Rx3@Ni6g2C!F^HTlhEi;aD1vZY!d8_6#--mtN}Ne|2q`UmpgufgQvl`^?JT3b=|=wlMoMgh#E$KDYq$tJO$3L`WHbYbp-dka4KJzltFlR&rlJAJb4I>8yshtMxJTeshy`xeN)xB6k_#1M!}l5Ux1AEaZQ%vlZ#B!#wOg`>D&joC{YHdM5&q`cIuAvebxz1nq*T57zlm^wtJ%39J8PmfJxpz9MP8??ozv=Tt|UTa(BT$7l+?TQ4DSQ<7{w=+^c>B<0E5xQ>Nou1y3>I$$+~GjT@a7zH~13D{&^>4mwuaH0JaUfT;?`yCJ(y7W4ZABc^iwDShV);q1*EQF*?58t0H1ugF}Z1B=*o3ewL&!iOFL%LvyAB2K-CI2g+WLj!nkSgsk7<5(vPDO094U15vQG(N-h+(2Ap?obj6T|1x0F3HVSm>;#?wEbeP(>j6uag|Rv?n#eX?ZBybMXi$17s|kb+ty+Vrp#nWfJ+QZm3B%U|5{vbAJo{Ozhq7u_=?cP6WO64CT4;Egi#B=T4uNlWhKxgf4z^&=)BxGe%WSHVx&lEZY0)C}-&W=4k1fx+YOhmsZLUBs=;mq}np8m#pGicAG4!{_QB?pBc2v0crd?LY-aY7n0mU$-cLLeqD#}(v_Wg5XL-(R>Mb$F1)lhIjE8^@|XF9)|=G8<wVy&)WRh+pruyiNN-YfBCoTOlN=&A!>Y1^4{a)!gw0!?SBI#)#VV>VYrx)d3>;FnI_pEw5T)r+cr<AoRO&G*netez`LovJZ5_Xdw94B_kIacOQ*LoL*hs@rye$hOl*5h~OllBCUnRa7|U$KF;1%+D%lAtPlOcPmakT0XGY-p7bvjbQlFrL>*7@Wl!B$Ut$Y`s=WThbi)tqlvG9%~^?XbIm}@=78)U+5{^~mT>S@k^YgiW!Mj!&XuT!%)S}|+C^xz;srvzh4jpy6<#?K%Ri#|o?$I2tEhY`3pmex%lJg$9^ShCKNu+F{Q1wsUcnX4gfA4?MZ>Hc^reCec#|<&L~j?FVAkyNseuSW<m2KLn~<>|1C~6rY+f9`ECc=EG0Ls8o8jFL*6c>RdmlZh9Y?0FVoP|cq5!*qvJLZ0z$jXH_4UeggTScwA_|1gM26}TMC_PP>E3fPO$30Q<#{N$J^N~iVoyqNS>LBx4MUAP&S!60PqmP{GE_D&Q{*!Ogrt5^sQQ_-^$-9Qf&csPNIvHBpZcV~jPSfm=e#3fmRd7q-+nA+yBju9OjWT%#Lg}Os?(c?F>&jsC2v4Nd9B^x(l)-3y-K*CR2%U7j$m?Zy#M=?{RWW-Z)yFA6OeFp4Tjcu>mdHL$Dg4BIGn$OnLLwpwwQwEe7r%DZo<R@%?8Fyq@_&6d2v%*|Fm<qw$9x4Eg@7#mBq02jZ-@8%RKg^8=0ih?SzE_KaPLWi?ZIi%K=bh2nV>MHG0(H!C@q6(HcX`y-Nr&HwD_9&87`IJKR5?*{l|LQ{@C*<Fldz7n79b%7FhL3RFc|A^a9Y9e=RGhYLFI2wo>IH(f0+*>uY`q`bo$r#zt~i@ZXic5~ouiT*>Pqwih^Ln~*CIKiTPRt&=TYck|&q6;r$HMnDmPgdqK?4@k((0w0=4*@e9B$UGal^=b@ZNEUlX@q(_C^z$dZ8d$;ucUI~8!U(P1wJMgj%E)N@FuHj2SI=PkymbK%b6_>8p`q6X>aRCy|1JOFzJoZGt9*61<xn@r>WlZG(cd3y>zol-kl@??@*zf_2hKBPqHRN;EdV}I%{FDmkSqf`xejM%FcL8Hp{NFdjbS%P4m2=WqB^p>H5t&&+qtV5^=fHkvvDq&5YvPAng*>^PnnaG(EtyJty3YxO<sOTS|f#<7I4An>7LW^K4N|$w==}QL|-sBsY;u0sx9)XZC()vtSj+Mi0^8I!LbBm&*#-L2|hT*ApwiHp3mp$68ua0%G*weZ4?~<%Eccl<rhE*`6b7+nB7i4$jsu@LOE`8fWxD1z_}w<&7n|3eXYNW;E++pAh9gswuBzRHST+7n2b>^qv_{aL)OUQt#M$5$xUBX@BWB4lhMP2H}l5_y$rAdh8F>0fMQ}upcZ?Y?!<vk+2b@*ttb|f4@gi*@`~>5mU)~S#F98_-G13ra3DC9sTfJu%p7J;HshA+{ZGjA#wE=K7-!NMmcNE&>s}tm9@W!tV^B~8NKmPEJ<^AKm91o6+8)(O@9Wb1Ai~$7&yqZ{zJl@luQ}qRdEj#rqh`UD)Ml&3)Qgp0ixj_9aLsX<Tn{XsGePkDD1Eie2L052wNsEA07YDy3e`LD|o76<@45Y$7*dhO}pq5zciq?q{h)pljmf1{ZM}}^#X1#(hLEFxF!7vZyV{e^QPAT+1$n!MI`~Hi}4g5DiaETP}(c9Mxa$BM5-+WrseJ1@Hk0s#8GjL;Oj?w@T<O<qJB;e1_F1uD;1BRT@gSTnb}lZHzz>y*8FW0=keZ6y}H3u9RaJewkwG__^ARbSQek4vO**sG#*9^`|`R4*F=~Wt(nrJARa%I;3IJXA-VKhaSIVR-Y5G+ex8}>m$;Q+^w>S2yzTIc8mHL3rWkpEJ8n61>@%`Y_j{sZPF>zFE=-9tiz~0l<onDlY>gDdxwdq_^_y0hcuQW66j3Ox?Lnh^GRs4yG<2GltwK}YfFt3S<c2z%skgwT^0H#LAJN-zaT24h9MF7=5a!SUiNf)k0Kc5s&#WiyZr_I~g+UX_2W~@eqrXPIhN7V`erc5_uD)|-BWKnpS}bJyX5A#dRm##O=~2KUSB^oMUz0|}DUQ9$Go%9J>S0ICNN&EqN68^e9gPyAcQN~b>@(>VF9^^_{9M~A9K3mpQr7@+j_Uh8B?<@DcD<08mpU{HyLXxAz3r8d+*k`YCx+s_*TRJcvP!*YU$EL~%Mv3SQ@|JoOvoXKDPM!J@dSiJF41sHBP;q}vcoKbicJ-+NK_vpISMf0@=785dvyz&O?`6Q+Iudw^e1)8!(fKk?ZZA*FGZma{hOXoQ2L;`CzWeO<YQN}2X1!THO;#3NZTl@xs7*vXGgh=SsOUB7ELYx1%SZ#vrJo}-_mPqQvunVWUQ~~Aj=R)&*+$j>CUQ{-gcALyVXf_lB4IjV{U7|u=xLT{Sm>M`RhiDYa5e>NM8O@eIzg$w|iJSOk6CLu9|c58+&Dd1!G7SwD^H2LC!D?VsNBDmjXo^+G{dae_o4_>R5ke4Ie@JAqB`G*5n(sL(YVbb<}?Da@tB^XL2DBry1o_Ve?%lm6ke>PYIM<efQ?)Exuzhg6A`G+N{q-^VFJC#YD^*W4rd_u&|XwA`z#zJKR#{gkEFY<)g*;C83`oD2Ec*lSNQvu7L99-s8h~9WE^`WZP`lEIQ!`8bvK9CYun+{%+Yv7_ln3J3+{P<-)z2!ydy|7i$Tonsa4k`;Q2saorZ~*}vCU709B$TAPLp_H(`soKqW~(x$`J1IZ8ldg!gFd>KY&m;}aZE~rM29L^`*LXX0IIeL4od&eDZLJe!^oe`Fs3pBqcF&{Gv_~qChua8Sh)#ZuQ|8wh@gfOl_GjMGQpRgrC&~YU+b9&p-IwcZ0Gn=Rsj=*Es_)UJ&=cm-zJrYDYdjP=qY9~a!VgW%??4W!rsvIv>v`SHDlqPfli783KG`rW)j6;T^+uZBs4<N=XWf>FvZsMaD;PMo-Gsw30EYS)rf8buN-<%w=LNX+9AxI&8n!`0i_GoQboR}rnD{oFAR5ktvXSPXwpt7Vk5phuV`qSX?<}W-V9V1k{x$eATEMk@kkz}l{LZ^8Iu3gLX-$C(q(nc&$TL?9}kV}xBz|oalsE7l^00(%=95`FCdMtKFD*&}_DhtV@_=bR0_W(J1!)J$v&;WIVQVPp%7;t?nJhhjMdFt5Y^WF3t@$H!ky0Se_g}+8WlOJFGd9*rdkpsT)kw9&xL4L=zcUtnVx^l>XDWi7mLN>H;B>a-l$1q<u>htsQ!b%-5!Q0OP0SJL~W1ku7f^PNwZrnvRRbei>$`1UZa~yit{ueb`J<}QvyI{<qtAS9jDD}a>jij21P;0tLH_<Yz5Y0tGEabGS-*ERd9TFVDVq|UqJ1MiNYvKkOQbQFGST&TsNGs)mP~VtNWB5fgKkC%P6i4Nw9kZQ8-g4rg;q%Pp>00Gb$T{iqn~Zpac)iQUDFlj+#(-0I%|G?Z%!e_1mq^GzZO26RR9oeEGsf8(jxsImDf!z7xSx_Eh<L2&Cd#2_b|UJ8dIOkBZsl^udzdo%$;8j&mO(!HUcGZ6Afpt3ZQ|wgcqY%a!=DZX&yHg@Op4V^vkAf0Ac-@7|MFn8ch2^71FS?HHr;N~&G@C~c}C@q-}VRe2*&fs?gOe}LO3=~8gde%UTuzjQN7b7OA4v_T(v2Xogzsn93ZpXlBysQa^KKp(7%yv&s~sfgs!7(;P$0;5S}3s*E?=h;XDTDnhp-imibV<ipewc!0uO9v4tG_27S5x-tDYsO(TJ4O9qWciAFmfQqQgx%}mu&E?Zm@=ApguLi9@UidV*)I&_5v0OOP#$x{~yPbbyQ6G&iT4G?70oHh|O7@;xuE<J%Xsn)mDioYDNbGZ`ygw4E-GY3?eYi+%wsW^f^2D;3Wrg8zS;=(-S@*KyJR5BoDi9Y)%>s@hd)PD_t%J(9>JW=}$@TCQ1zoh};cPrwA&*rSBBl$(HaNEoWOpAhEjx4-PCp+nQv?K4SZx!?$T#p{z78oZGlDY>Z*|rMkbIXD4kn9E(ay1adW(IC4T5lFfBNt_;#$nm^1Z8l1A6=n9T2U@GiTEcFfahqg@ZA2zOY69?54=gY_6c{zxT!zt5Fy?LCNxB?7YZ-5=AP~aF=K@`n;sYJAi*PAUx;T*Kn;;@np{H}1m)*iIqkDfirG+AF|5g+Su9D!Q#l#3%LGPb6-aeELjM$y=ZC;T&<0YG8qadR&GTJW^Eqlv+iZ6sW}KVsPMYRxC?+=^wNbAo_V8$AkO8`?K*JI?iE9g<WjWNx9o6U;hrlnGA0ObU2I_SldqZI)obs`D`*{$0ALKW=NJ+b8x_Bs`<#Zysf=|N0Mu3d6QL`Oh&;jg#CtQ6&oiE$=7l1OQu25h$$h(xB!$cVnV)<1poX+P2&`c78tY-sH^ud6)#%sz)seh4Sq$c`h{@e8}pK7HIS653Pm`f(D$DZl)qxlp6PAWwBzU&Ant5SzH51>yio3;*awq$<s+vwn~aSSnJp2=J`1n1cffFPIaD-;B(tRGfwy2AgnfwidMjd0I&6$Xr7F7afYQ?DNTmG*)-?<AY7lyGjK3IR0hI{1J3;Kj&T=n%UdOt||m%>+s#kPo;_=Nf18CZ3Z>g$5BZK)Q(j7@6@J4H*lqI;;7_VlxaC*|F}h4NyLyL!@^mM$;-&_`hShXUYE;;4T@zCG1bbW>!+alnmZPFzaP*yn*y>!O{6P+nmFNiVmF`x2{Wd4Y?h6HU?a9bAafChwa~D6)gi96R}Xxwp9x$5vR2xDfyIZM&FVGy(4AaSLmzMiJd{16xaAPj{0#9y}U^-yryX#dq919Jg_yto1xB!ZHVn8_Jd=0w@`!ZZdC&VV@COexlRzh_c_U$X(u8EayvLqtiC#^fQcOnjcWOnV6MSdvrQe(@&PM)h=QvJ$=Rdux^6Ffy#}~^T-rWCwGH!7aP)OVtY<02NPcfqon|#uto2CtPpVbmmFLrbgl-DQcCW89c28`|Gv9=P65PKR=StbG)jps}0f;FWQ=9+wD>Sc+;kN^9vKw2|O1<vLhS|H;y09wdHEb)RIw)k#{qQ>^_(N?@(Z{x)TM?KEbwA%k@$2Cr;`|!!SL=f%uz#%4nUMQBs3_1<uU;<&ALEeQ!70pM=Z5-TR{)^*Wax)dx{_8ah&NMbmGuCxzVap@ApV}KpFkC;_k{5{Uy`HDD((HKl3~e<V?Uf=sUQKGc@V7Cfym-pkiFy{S2wqxvBQP{{t@a!j;v@q=kVkx@66ZH8vZ>(>5L|BON2D?O_NpoMY|S#rq~1kIa~F_EVi=a_XY-!V3mOHy_%Vjta0!J@<$XpQ@X?aqarRMN%)&8dsz?|d#E`m6W`)A6b2vC8QR|%4pP;^hi%-kS$=9r-7Wf8eUs*9-sQpUmORT(otw62Hj8<%g!Z3>cw^q~9a&<Vytsb;<RE)H=LRx#UpTtfmHxqNF%)g<QETkW1OYJ+dGyqXC$c3Mc;AHVzM#C3Fn81Pku~;r)ek(l4oQRD0jEBSnlO`D(Qvo44{$~y^=%+yHR6)fHFmV}wK4UjdK5L)T8T~YOHpu9>yuNwHoEVM$FbjaBh9OFM`|dCV=lx>*$-11|6Rr8zMx-cr_d<{Ggzz`<|A>Kq&!ZzQSw@9vgYiEOV$l<3c0j<QGxy1AB@^THV_DPBG8JfA`-~3&0;w4Rry1P%#aArRkLyV&HBM<G^rl@1I+tl!mIpjnqWa`$L{Zjt{;b?)L{=va1xbxc-X}p)JTd*cGXn!VC#>aieVcyqEf?nn&7p&)H_|3;q<3+diVO*4-&8xtv0P@;j-B-rUg^SMa#N85NQ}GzT<z|BmaHaQjaxh>2fc59@7u;Q<j}0kfDC5JNB!+F}?77`fT6J{~*$r{HL80qIl!gyA!PJ)IEkb2Uj|D7ztyinc*NejH8Eop1_=V!+gfbBA8`T{1EtYM2-IV)s(Kr=gjkOrzQqx_+b`pnoiebWvgXldLc`9WQ{l)0vrBtq{WDkFatd<clPPA$KB)%49h!O!@Y=8rT<nWugGjy<bjo$8<P|!`MS3+Rk<D%wNCM{AxBPKprO6NGv;#s#NDw?_#l@*A#`^7+wPok>@WsuvpXg@VMSi1KvWV=)K}terA%SCe#svH>zsog!>I;hIN&w`0Ry2Joz-&1Qv$Kc&P6cABBr-*La;|IN$cb%YwikqZq@v~OH1gR+FtAaYT*fy__2;#alb*PLp4NIRUgi3y|@2|L>D36KeI7RsR{3A`C58Syteomnfm!CU)dkA-N~S0Q|RS%hn|GDz<EMdSRxG1DNPu}Bml}Ri-@<px&OQ}HeRl%$9(4<?&S(|)Agm&jYf@o{C)CI+L6Vo1=C6g>0rnf5DPF<maUdVew`r17oPuj#uf=jK|7Ar?6TW|1VGE$J#)`jnxb`H_VF|4jF>SlzP-V2Z@p?;>am@dd?`^ksx~E8<1!zTH@mhH5)98&<cy0H#dN5z<{aKBK}9t1&*IHGI$JL3fun@a|2FT?x}i0AacR*SGr$26I0?DSaDn_lj?LF8qBV{@kLY>WuAP{Ms9BwwxK1-U1jrI$F6nrIoMu0)rm_;VcVDn!2Co*e-lsWD-+N62gC)sUFt=BP4epLqJ9fB)uP(qx--5<?Uzn+cICl@Hy>_o-K6t;ymd$E+C{*&<==k_0qFEgos1)r{)16-Z%AuMltfa2Y>%3rVSq-hi4=1~gD9&ecg@9B*LR_`QO^yF(*u$WOq?X}uwd5>$kW1}bKc;LFykLJVr2)xXi}hQk6vt4=?PTxsuV+%)*)V%+Lr21f&``SR2TM=XCRAZ_36_bU)(P%u1+MpfFJ@y{TStS@Wqx7K$xH*|x{@4*8qC--^7>XKgvu7n2ohuyiD|-Pf)vI@A)Km!BTBfLbyP%<#*0y4{llxp+3i}smO{zTrK8rh0ZO0)V-O|C>VR#tpUKcH$+SbeyhuC7e+}7lar|}L$Qh0&s*Ez8ZYg3;?9lUCJ#hK*D|FN%n5~hLGM5=&Mv474p`zNquHp0vkxnjdSP7{iW+7WdqAB>dDV=V!iKol||2l7cCdzHcrYosIQxBv`S~&Rm2UDcW9-HKnD$tSH;!yQf9g2jRpK8l8uwrT=%57yvfI4oBZb{ldILqPi8jT5C&ym^VLW-5LT6gUdvIGyb1gAI83i)$6izLR!#FCO|w&kV^*Y^U4v;9US#@T`=GrH8?7?E|3zePCvPnrB(4c@9VP+4}zZ%|F9yUfY3H_I(H#*5U6WP}EOAuN~I59ce;PH+v)sZ3n_$McsYrXFD$r0+ZQ{!v#2+6KMQX}_xIY9=}%Zl7<u2MBL5Fo!rg*{~2eNxv94r!wUa=UM4*_3_-+_KA8V)RFhC`yaC)O%O^)i9JLhMa7)1Tb6?h^_(kx!H`>)=J%MaoVgthO5yHEhzdYhJ50tH@L@${kHW(sXiIZy1A6v#fUUsM>P<<Y5mYHbay?fF5VZ-LwnX+J8o?AtXpa><K;U*R-RiG^w1V0}csVe#x5vrOag+UG(_8Ssggc?!G!k<x=;EN8CPK;$>BUAmk(;E*3h}NR)~k6D>0v(tBxif@t%QHM;W(E-3U36iktndTSYUD=+V{u|{-a%fjvTQ?+b;DZ!AY!eF3$s#&PrK4Nx1RF^rOx=9OV{jt)>>{T@S1FXKu2uLp?*Q8+Jdix)NQ~)+mR}C)9tODp{N9jD@&XnZHpZMh~xTL^Fe|l4*KW#=iv0<(`yCp-R{t7$x5%CsFuJ3bD|!1rjB=UPkE(4b;hL7M(Wz%I~}lyY8`sGEw%k%>Azxvjb@a*$O4TeapJH?mS{ZNa<ePBY#S+7MH9tB~1s&k{X|o!5AhaF|gD)v{qW<=?`o^k3q$gJb{G4X#i@sM5-^*oZC;VsTdX6qQWFi2o*6n`$Lkiug7%LdAY8S?fb3IpA8*g5M+<am!C4VHyHOe-~yK=0DBXjq4J#IjZ9CmpHbaP4KRK_hr--HP~tfJZuPgp6mJp0TnNe9I1Ie_*)xbQpu&-<i(ZHQ|4_1%eyW{PK3S%Asw$}9KZ)9H#Wjh!f$I<Db+3$mR56cnR*f0rS5VTp(FqVn+V8I|ym$(=*vLX$4a|Qb>mcLM?qLuw;nU0m0-em{%g9Ilot2T_@sDf6wj1`|P#E|&oN0qIBi%!NZCK==on1T1IiZ>{YG6Ft@vifXt$~S(yDj*nOgEg8ba&T)tVEtzR+HZV>md)IAC)1p!(VBxO1>KuZPr3+PX+4cP8gv0PP<xrb~RJW^6F)nk><tP#42+Pvqw3`Xf>0HsK_P#7nX5}qUq;`5ZQ;Io9hK8bX}YxW)6Y8-qV4X7B;F}I-;So3!XS%5fht^-|_41G{Pp5s-LhfnwlH$d`A+xrPjJkm!Dz3AI`!Xs@VGLnLNB$N%rpslz_Ie&;4v@5c}?B1W;8Rgbke&>+rF%qAyDQ#_bZ)YVT}dp9sxLXjrJt9dyA*f$gs+2N5LclO9@3oypLf8P&{LfFyFFV<v0>wQMo^vpUZ6M+8TD#zjDgZG0vYW3wpl;!WFJo``3$j~%j0h+38H96*?~wo&hT+iUHcE|DOTTD(wO-4zZ|WN&+gio5)LDqMt4C5g0F612_0N3~RoV?W#ldfkcW;1iQ$`3Azi2D3-a1b0ab8?9#+wai3}r6FF6Hm@jUnp`0rR2}+dkCYQUzxSVwD&^H)S9-Sz@+SkXQ@f50{^+HOylxjrhKBtfsvyS)gmhYcs8;kcA>cm*pDDGftU<uQZ<TEv>B6-Cz#k|O9iZycO#)79KxfB;4U^Pms~L68B*fn58tW{Q`qM9wvI5N5OR4q9Ni-eV-|dKc9)FYSO%$dQ)UnbCS7S?wtg>KGBJ<vxnwmKc(av0Hd|`-aN$Y0LNX6!VL4`DhRwR9L$Fr#F9Bi?n29G23`+L=?`8{SO6B`Ur*#-67FKQS%-IZ-HRcBDC6lZ9e6_G+(&?O){UAhGeX^#2My&?Dwt}tvN$fP~!#YYAH2!Gvx$_^~A6yv+nOyuy^#FC6gqN_HG7vq*{!NDY!YC>b$Q~@WY3ml=0AZZsg7yKRjxsInJvtwjGiq!h!FM`(dJS8#pcjE+oPtb&Wp9j<SHOikG+cSHS(YloH(pOI?0{LL_kP%0!$K9($e)8taB~~Ye-av2H*B;+_ZnTGRhN7|DS%osUj<$|;$~RwrJ@yoW9EjaA?uNCn+_E~yMxXsElCy)AHLbq$z99g4(Qr(6r5hc^X8B1GI7Vq@yFl1$a)e0869lqLU?Z)GEBz{c!<1yc@hXghhn_D_q8W}a%*~vPkStLqSpu8@pnDw^Xq`J|T+62)s(f^4Y5J+uO4TeL`y>(#eSw5&psAPxuoKMJ|AD!-xF>q$+A~15X~jZpKv1&H{m;I76S}s`8-J~_X(i}0xn%A75JXP&{{sh}wUy2Yd(JUkx;$K|EYqw8ad*AS)hnlB&X``~@eU@>cRTa34AUND4oTQKc_$?C$FpwgP91OKBTpO|vR8kE-xPH`6c1yi*Fmz}@^lTG`B-e7ehYf4M0YS8(?YpTpO1xw<A|_i`N*|fAo+no$Q(#Jq>+yjDGCCE&`p@?sI2Wr2@TWBS+)z53#MMccWJE4GVsHQGBWY1>};{d)w8b}Cz5ie#SCIxpR;93Jbg@5P(dUr_rOVr#^#J6{Nh4eM+q6r0H+%P7gL&!7}k7u)RejRgqHkU{X6s~e1C`@Z#{|!>4k8{o*chg?rBm#mPnC{$!HrLCu<znzl>m4)vFRmZ0it(TKb!fl?Dlsk6_OA!h{&@L$m-}$5weLvQt?dS5DAom&sX-68_)Y8MA9vK{i;Fy$JRSo&|Fr_gv-4r4l(qpmxCLs(35ja1^nwH=H=30t}I5Jtd_a$nc{KEdo4^*RLDgYq-PW#ip&a|Bx4jN_vh-G4vw}kyp*-KQ%J0)W;R=n;}h$b+DJpi-a<LH&q0hW-u+%!f>x%Jb0_<7oO@5F107ANz+Ee){m*sr!n_SB+zqWY1!Bo1fX9Ee>6M;TMe+OcDADQ6vfj_&G|OIgk`3|c>4b}+cmv->waUPOS2*Mceh#2zGR<VNlg$Tt0j40b{YysgBc5cnqP80G}9P9=s#(=t^W2#4gX1vz1AGsS`&3o(u*eZM(GjPF<yYjJ`Q4WQZ-%a1h}Ip{rZ?L<(U{(!seY`NIp#&o9aQclX^c+_-rRV7swF%q|f+=f6rrGN0ezWEFwaq+j_kfhrcb#BCMJ}S?L6|!NMTG^L4d=xqRE82E&8{YmT~uwIAT}{dm}TalkKeF60y;UAyAphaB}+yN=)k5$(MnG><booUN<@nkqf(HjBkv?oNs7EiXtJgC%73Vg=tP-SIzB&MR_gEeC+YmA`AA8`0;4i<1Cyw)=W*OPTj8Jy}6EWy!9UeO3reHnI|DB!)l;n1rBO0uZOX*e0*%lq!Xk3|wD#Jq{@h!cU|Q04TAB5fwcml5W3B@2bqhHnb*`5Xz)}V#7dopcw5vwrL5RRUiRa%%g6e@*({OD=M+!+>_Fn`_TcAWJO;EXg+2lxny1_mh%INvLY>Ph9Fc{!AS(@NPqt<-jT@~t?aiI9mDH?&=eCDP>WD4g|*rUWNf?&qbCEx@l-%TXnXEXa|MXZWRo#G=^zP-J|*8Fz^390(Bd0;{d|nTHC}jjIFqgaQ}tt%y<-oQRY+v76Si8l3Hb0;-IC8_+tCfm&35FI?Aa<rAvPCZa2SN$%)KB*Gin+na-%|U?OvqDPPvr3!SjgVf`c277V@(I*>vrMchnAkLy<rt`Fo9%<3Kl2JiL0yqGl$J8#V(O$R`mksp5|(p(HHz_e3|l*CVHc7Q<`ta7fofV75W(l_(Y&StwL?<}40;St9$dNJwJkdA%Nr^G6}XJUA-+iR>e+W%5@w70Mda9?+(R1+-fxV>4F(`7`dl1#rn{x}HPrBsl6S(%-^F0&&wkhjKW${D-|$(i9r2FpxImqOQL~mpIwavX@bihY0*9&VNGx01WfzWN;mycbphMsR|9aJobD3&#p6y8wHpMfaaO#Y1Z^x3C)b!#)GClqOB}<<*0eSM0&%d#>J=7KdB#V1x>|}E*qH8sKh#KZh{Iq{5x7Sm<N8I+pw=+!_;kKfV1F{FYIldS^X*{ezv@Oq73X8a{N{SnAt!2#HrK|Q?B#e&N)g{5fGW=;rBhBouwG_%K4Y(!u^k3&jlO3@(Tdq2x9GOYE~9XYQM<lG)3H_Eu*8=s!ThutAmj_S(f%|75;Y5RCohi2Jaqd7XSFlVB~}djS`qh?6H#+2jZh-RlNmTvSl8e&Sa|tg|2{x(8IW{kczikLDh&3IZL0{o)71Q>>*#vkJQ-VWL6iDJ$j?(@ZiIYFo@CiqbwgPU`0F?QK-PIi{mhCOupDnIrAYE%>>wvw1H;ZtWE^w>~upksKpkDCA<9N0%~sNyi;G(Z-9pKWumvFErIY3h82lG>t}N&%%YBDeN+jrsf}zu4_#>(8cSytfdvvYOmwI2mn?N*H-*P`iztEKH;J=|KV0sXtGI^#z>JvURr5bhx<2tkO>zBgmZ&vDWaUaJg)kGCP_V+N^w+6e*%I;;$tLlkSznvUwaN1IPVf#mpvBwAN)*>K4s@b>K}Z|ag7B}3-NIqL^;&+23hIy4ASw3<WmS7+(ZmpP5e71Ch=?{|U2{A!cs1b&#Fek*gx|&wi~^Sfcxu^HPkKIQ<1k;l=%_UN$pewd_N;)mppj=69*AG^5r=n);j+qZ!v7JB1faW#>REuA4I7ErhATsHPIGqOgN0pSGv3c&qA;eZbeN2udpDh1fCv0>oM$cdDBcvl%1Dd_EvvQ^-f-lSE6+RXm|<%ypPqZ8)#*Qm-nR@lRxRq8^Xg0*lE_T^N!!!x=Ou=+yP^k=&@=t<9wPEeUv{CH%mU0UR~_M-JTWAQQgCWYUBi+NX5<%6hcny>Qn|=mI4X7GvJr6|;F`A-kjRd?5>&1cb5d(3rBMR<8F#J{-TsC(nYpPn7Fhy3^oueEngg(0fCYev%p6$7_-?>G=I{~;G^y{u2YNP`z#=QKxkBC=K7&OIuL=J86Yy7#Hb!HeQ_1iql@3pTiub%lo-8=mT@xDo94p#mlPcmmDQ5$2R$>yV3R*_>tl9%juwlE&CZ52ECbNfTKy-Q;o^Vs<$oWuCqh0KJ9_jx_^jDfWXG!)L74BH@_<^^c2YGS$C?C7l-R0dW(((z!2D2<=8O1IlU~IylZ)ghk-vH>{MIVB|xz4B=k$_kJslxhcMh3kgK>}H14AdPM7&re$l!(ZxBLiYiiO}c~wxw*d!+iu5%~G+8h(=na(mtidHcZWf618ZNL7yil^I=Pp;Y2@h7*xE7-~rFfafQREyC}OnmTk!!Cnmt!QX)`1tyBQk&@?j@#rP!4FPvQgs0eLJKhM+@h`7i8vYkZ|G-h!ofkSbc3OtqZO)|jWhX}lbWAybwx+B8^%4j|H3jNch^J~hb3a>kqL<qx3Ua4ll;uN<Nmo-P(w?U4|P<toP3yiFI6Z6`z!cWt~vhpJp3UT_0NLC5AwWXbJ9e3}uq;v{lhw$x>Gh+rW2W7CAOOkTy56UWxCZ@2JEDRMAe4pFwD}6Yi-yO&YO$vZZr#k1@zUOeUi&1IX;z@i##vtZd5Lu`~i&MyTBO2ZEF$8ZewlS@J7A>;_PsbW}_p?g&A$)WPU7c2uaOSE;Qbc(4^blExw}iJfL(LGOK}^a>E2&e&Q2x>velL~y-)mIo`&1(!Q~XM_ISA=J;;!MFDs&GxF?mi`Ocp*G65a!yd+^6y%Om$4$N<Eahy7RY9TInibAk{4H3P%Ad7(OG+v}z%Zuqn7y~9E#G4k6AMpVOoaqgxylsBvm^b!ufx_bMtbDeC|W5uss&Ft`CW_9lCkE+uUMVra+H1S}c3iOY8vz4gugv&7iVa%)?UBtjjseNr7Vq6WE=C4onfAD_r2LhM=j42V->gOg8N_A;aV3P(yhISF{`I1mkdQO}!OC~m|Xrzsx|IO4-iTP7T;qq?;zpV{>-G5CcaR>S0Xu&M}5i@}wLC#{T&`_Gms!Ppg0oG2sdvxUpzq?j=_p-hvW});)iOk!J1;W6=*gb)Lg!t&}$+UAGQ=*f=-BqXdtv~UnXU(wqptoYSCA~pCl$6%)@O>0IuGToTSf^oOSu1!*$oC%HA>#h4B6_lo41Q6J^6B(#SM?_P5fcTH<jjmD8q^xPz(9AemoaGUTYhFNftoft7dD!7xE!SC!X49LhO};y#T_qL?b!j=nhV+)p2f&)?+7^ShaM@-8H7!T+3ljAA7JoT1VvP9+N-G~FxyeFgr#HxK{eY&zB_HzLnyfUwTB$8!a8A4$snn<s!$o)8e{WqPQYk4{tSz9tOY_!*|N>}vYYIvgC_pLJP!O;-d{iYCk(6SF*_bh<869vWN_!bCfIC|UbN2Tll~(O^H(f25d&<{d-YNl7jRu_rVVYyVy($IFQom&!Z*BGFTVtnldgu2VfA1ar7rM{HV#o^`wx6i4U*FkV{DIz#Y-3hK+?S$;DqXO6&NpF`F7g1<1@=@y(`KucFFJ_eF@viAgQ7>Er_j?+t{#erUzCg`jMw|AG*`xjc~4_f#W;8$i|T`JrXYeDj_?DFnMnC4G?-Bjm6Xn10rWRAkQ~2Zx#X&pe^=p8ILYmO*S0kV{-P|Z;4Swgj5rWnTs<LkMPMO8xNMFHUED^NmA0->(|LU=k#S@nXi7gCz*!sO>zY+d}?*0u*KK_UQ*b(p)5>jc2{Y|6woX=YI~s!De15G(9TxTU0z>%4VUDQ9saWy-RO!4COBsA5#6T-^Y^jaYN9&_Q&dwyaxI3id3(AYeSZ1bnD!lbMfB<&Z6>OBI{JkV02H5XE_CjB7U19KAL<pojbA7+95}{Dhg_9*O+RZgR^@w+#*B?02vN{Z!`|$=l$pzxHYSL(|LxjM!g<tJU$m&C0|E386NZ~u>LsMvCcb=?Y8<&7^C(ovIIZS%)3wyLpiHv|H7)#N3j;55fYz$*S&f`LxFT20GKX*QvL*@UOYCyoGc6~aaa@fDJW_hsozjgP>%u5Ee%OV<7&)QAg>#LoL&Z4NS_OT%_Eq=pe!Xds&>jFED?T-Tr1${UJ>O-6v=xRUYb<FL7&#naxDL}a*0+eik$g|y;=2rRf5B&`W!t?<Cu>re*on?<TKWt`qsRQELa`lUuLq7n8JCIIK{ItXqYkB4VjCu{EndtuqHxNZ1)t_e{AFBbf^}cPhYIjDN{x^OxPtADa+J#*T$GjZ8Q^Kq0f2d=O8*`}zJ+~ZsbwR|`i-v=*oT9ps(Fs(|9UnwfF=z7hb~<!es8g9${d6SLAu9~wo<yf0Ns%E;91=DHlTqURdE??2;@~ZNKGszD|$%=HngL5!M1+oFg7sIC`6)Q-02C~2>!P$0<BtMeciTUaYV^p*Ktlt*L_{CHP17uBX-I$wHriH>4YevtKtAW+plBtYc5*KFLp;Z2$%XK;hJIF+Ek~GQ;F!B-9~nLiTjo~0Ix8coYVx1jB=ce0ZI+Cx7p*XN;~;jPbGCN2qibp0NiTA3_^`W2`32w_lM0+R(@A|A3FH7R$l(SOp~v4>Z3_S%w*9~80D6~paRcP<U%(Z9NVd@i(&>Fh<ZGmKq9}PQJ)yO2~yYY%el^sYGe%@s~ixx^+O$)3svPpvrx9bk)Z)$j5xKt^?BN2*{V^ZHEq1un$ONLyv?7VNZVq=4cMG1L!DTuEo_*8Y|9Q#^;Ha<iH^@Pd#DiM%U2c2q3Jxo%o1zv8vbf9USsc0xuU1Ft3^x5siUaK7WA_|RU!v++;GYyEExDa0J5DK0J!-T{T^jN<?c#Ga$5he&k!A2v^6^h#*o13$tP1sb?b{iu?Q2sscYf&TQb+f1FIs)JG&0dbzEpg{i0$r`ysDJTJ)0;VUC&+(ixD98vH^LZdfC$%)P2EH}|OMdHF7554!6G*D_A+Y&DlBp{Z|m_Bz7RKcnHv?edCq)e;0A1d?0l`1mRs`z7QT{!=n;LfWAJh7fnG;pCmSc_9!eXu6#ts85$C1b-V#Fg5o^<@yB~f5ep@MOq)ENT0B@TtiL#&VC;6$Jt1qFCq<vhZ8SIjyTG?opumyg4^5~_OR#P37)bknO4Of(C+(T-1ydgSj&w%rpc&C8oT?pwaJ<^)B`-7)5d%8`S*~Y<XPTPN2#^RikwJzM3v78<<{syYJd6a7bCY``+10LA}VMwjO_wZ>g3{DOv9H-PySN##fkdOpnjn~S(cOawGn1IA6n%-81j~A33CF~FBHjC!K7|FH-^`j)Scz0fEZEP4u&Ws;eI%N-8|3<*4+Q<FMxByk=rsv;-MENOJ2f;PU9eHJC5|pdWQ^xB(8{J=I(y}-0-{fr>{syy2xurMv1LL_W?b?<4rYp0BH1v>N&DhI2?>keXfnQsQ4gg@m#IZ%*xvV`rP&WGfS7?S#-EZzlTk0EqI2RL{!H{?NQ+PtzZl$_wLxh&O{P>gc8r)mC!{jj~jCJCGoIJ=9svH&)q}bJ=70G*oo{k-j+?WZU)F|iQ1q&q&Sr`BkNTU3!npR!HY#?Y^foYSJ%Bba#vGi<#p`>+-vJDZ{lHjOlt3D<2aaf_>Wg4MD6cjW2L@8dAS9i1gzBtVEhTwuprlWC;fr~<AojjkszV$dcZCQZs0YZ>#EbZ8Ic{gVL2Za#(eb#cf&&^q|LTSPY|6a|9Dv&?#@Q?M6B~j^=9=gy4O-nsQ7AEl8lY;qAfx8IaZyW1|e(hgkXOCwwVGaTH?ZQE0leHkO8&B2nP|u1R6q`MFv3&t4d1HF$<egUikLt0A!D{0Xy|R@*FWde}w2&Ek#pbffI|Op<#)@gr7`|A)&v+({VqEK_22iBnV&vzi(a$N3Kd&OM77(lt2K!vp1LPTv97a`-#?*lIE2VS1hV^QX(G)mtz25j@?hAf7PL=3!;%zoKmRr_gHBG5eq723M5+26L*Do5;0Me#e?GlU+Au^8OH~Mmv_Rt)Rs}~-0GkQxatT=XWwuE@<b1(KqTM+1l?;v!+pI+*IYN$yCS$dSwU5>Nxk7}Pw_b;gWTiGe~HRZl#temxEr=H^K=StN)d(SFp}+Fm1`y#vchnqfU*ifmo9G1c(sUw^Uwu0!0iW~d;|d62P#<vbAE6@o}Zhbs(jn{{_O-q^1yngK`O@;>A8~0`bUMyjg<NIio#M4tksi5I6K70Y+7%gbY#ObwG4qDUUk`;vaHkUWjP?2Hv@e7FC-*`PPEa20~Poa^1Td5lR5DSs9+F0-#4`@MQ0Kr`y03p)68_M^dj;TZ~UyPBQkIl4u)%=2b(VojxDMqwipcdz-kRZF(b?f7sJB&1qYMpYU{?YHQ1FVHXBidcET=dXPr@5Q@RFOFMnb*SchnMRDoDIYGbU8B>{rpAoZG7`)*&7-euYAYBs}kA|l9@UdVlx{TU^pEytEP>#Vt3C&RFiqr{0Mp++Dy$)TZt_}^Xo{Hy7SpHP`(6WvCX2a>7QLp2VvuZYw3r_e^$)?C|B%hH$!M|doIXoYrmvP7Ykz^9P^jt9CT<1n_Dl4z6(AVX4a^StwB6`cSqlZVKT>Gd&DWFrRrNUoCOvkm5$5A)shJR9{TD%NB%9r!n3gF;hx%<td5S{_3ptlHGy+G0Q`zTh=LP)wo^>x4ri;rrzeE~@*Qgp*NRs8a<`moq2WXq9I)2RY8TFbDGvzm1)^KR5LjP1J~6LvD}M9ewI;TZ)qTGA3^Qv(}ZIRb~TQl1wXnXui7#jyW%u853)QZ_|?#O>;1LR1vojfN`w$+O&%Vot{TZo+V;lExbjwKp6g2-H@5Yc*;)pOtgOL@p{}fm3DZzVGshWC{XB02W|#iOxJuAx8rp8{xgw@-MGL;e^*^G&_j9&HSLG(^k^KIL1v|W1qRHtrY+8-8N768HQUQ|RTN=|ciPAIzxVk8Y*%eilcK58q=#292;2c^&zU_>G)qDs&y#+fWcU?h__zV>(8fM+ZTG-f)-2*l2dI_^7E8{mLWt!lxbj3Yw{Ouv_E%Qp(%j-|OAYahTi7ehSyXJ#l%}Z<U>}X|vBjfI*UaRM`TT<me@8Ib8%<%u6Bi4HpZp`HDkkc$vy({{+6fyUwcOs4Ch~Paf;5RcAnags9Ai79ddiM^F9&__hQ&OA{(=)yUclmhNU#|K^0g-~pMtXeZsu8ie%A|@z8x8uEJBI}r%>!Z{Srcqw7oQqf`5`K@c+2!@$^mi+~U4MCpBUKS*W>yf-9^MdB3<JYhGHUQZHU$sHnAXAIqIZ_3a+Dsn4O>fj8W#Van*lV}CB^uTGGqns8ogKnX4MD+KtrP(;DHZg<W(G*8>%b(qpe{`z6?n)J@&U@(rZa^veco6!sNx%|PDj7#>M=;}lx1#k{>?vjKP0l+L5zg|khKiHqjf3yg{vukT0-%5!Zwq3Bz55chx$120RWRq}ti5Iv{090d+c7ceO?3Y$nDo~<Sl5DCXN&IApRe3^nXw|GI#w~?<ta&BJPXL_`rC#Hn*vbZXrR8WrVaXVlVWj#zkfs{!$6n{ZJ<|OKnqhZJqWi2%ggdI8Jla5*wuBuVdg4cA{6fWh&yfg}LpuxYJ}}P3>WWp*Fz_ekeuh*bHR<7hGw5psn9SAyf(oYdx4lrClza3?Z|{=vms|WLf^EUz8KnjVjn6tcMDjnd9bvG1*{mOG6(<H^C25<Y1sKo=q@!a69+NS|;e1j3MlDi+r2mOeYjb~uc@=oDP7ycAVo?MB!A#V2jzy=N+)7r8E0Kh;65B4QZR5AqPRi$x*R<5axG9FsJb*p0lDj)Frl%p@s?L12ai_3V@__(hL;aL$!Wy;!D$+b93?eE|K1SA%OO?>kW!`7D&#Ks)%i5##IAaK>_pRt@W&*5D*6BoN7-K*jKwByDSl<tFd2fLW=s&P7+tNS+An|c!oEq9&Xf7ytKfZSF`F5919EdngFZ_=~j-jWtK(jrSCx0i->cB=afR3Vo_TAATxbrZ`|B?oevH_vVH6%T1ku5$u+@T0Ouq<1I&2a}-C~DNnUKun%rtMM9z%CWejna14I4W&Nl$`&RseYJ52pw=CSWX>2;4%MwX~XLTnkZnA(2dC^**Mb&F1-*(DgrL-<!^t`U*L%V>hpioyy+9!IdUO6ogs~NMt=r)A$ZN^MZja;Q8p9KX9>Pu#Yy^eO=muoY1MFkaee1zq7`O52_#K??7P7B`fX#-@@GDvH%-B*d$4jb=t6fZpyPye;&l{?*K3d|?1>E~yj^&9Egwu}p~PZ=I~R9yBrBi<UCfSzsH`ct^qS_cesP_ZGUdBlLnF>DxL<)ID4(BKI7`oPBD{V-cd#f=^g*<`2RR(+t+!WNSWv>(i9a6ctxRa<Q5S0D0zugUQjk&xn6^mkh+GdsLfv8sTkNEUkuAnZ<Pdls<7s+#t~>(Pa|=bidf827K$sgE4pVB0J9^xYDf|f{PVpjY&GPKkcC?AI<e!iUZnoo4PIl%#TrZ;0%X}N#XVYhGU!+^$c<rPdA!7Q2ByZoslN!)Bqay)=f2Wj!`LBSAIL^mRIuDv>q4c2ugvt=yd{SgN@eEH!Dqk7{SUg3oWl^;b))D(A@!K3dWSO1GGQw{a=B(*!FR$!%&>BozcB@yoo$^MW&<61%r~nR&B`_0Uh^h^K`4q&G&VtAm*F?VLcNc9(syMx}T&A|r#rm^|*cSSc)BJ0DQ60kr%gqXdEL?=$xrKvpt=*pzHvC3Zl#_c#7ND^q(~{d`p{e9B-?h7Qk%z@QbG-9G1s+&Y{g-7;OEHTFMR76N&g}M^b;!a3NQZSCT6v{o<30ReZ9AYjxd#mP8#$<D$1&=~W7Z2yM8<=T%&^AmxA-bF8br4F|64SNC@rovjwriFEFuyuK300hGIBy1%6ma~M}^&I`uHg>?FrqTYE&)P{wDt%cIv$bu1)g|^VM*<+HeJrKn_KKl+#$LrQW(=pJkB~VX}R|1vPK%*MX4;<oMb{HUNBzkaY;PN$@7ORJa<I5;1|qMKFi#8W9-x5of+30%lhGI73LvKj$q=?FbPs5(1y9@jHmmSde*;@a1ul5yGe8DL>hG&(oGT*&r?)KW4XoM-iCZNw;J}FXtFTEs@zyML3UcETAE~jGOOAOy1HlMj<ZXj$RW;$jL5DirV)l3;~063cLb!jM7}0(pS@?@bRWoLHqgB)I5eTtg<~W0$mvq+hnP?D+h-+HE5N(q(Om?hq&KRKiNy4iDYT9v-s;dA?wZ|F=ymbS#ncZLMFNP(p`Bp|B@Ev)Yh+fM-RyRA8CXty)qmAJ!_n7A2n^uFy(vUaSmTVz^4(;hYBS^rjZ)Hu%dR+dp_<7eipl?L##pGm;p{!z6-QgR61O0)fN7>;(YlVvbj*eIbKhmbP-i6fQ0fg$oo~D|4h-|O6Gr!9)0$#4qcflXSo8SxoMM{zkDrD(*Oq6L0;VEDF<Rhp)II$FLVxx3)5|}-j(-etia;2U?qt?W5U3HcKAeF31N5q1AIS2Q={WS72bF{bnGq8PMfW0=-p0|E$X)cwL5gugTLiOa5z^3is@i3x!$zbwGG}NRxK4Pj+Y(M@m0CNehW1pH|uXQj8S>YghKe?bX7jy9W}`n=|DI(*YMOysr3r)|JBzBv8B5g27}xn5=^-fpt41YAftS%)?Je<tdV9)d!col8<sx=RDJqki<!%2054iVpmAUot29(JZ~(pzu;PPse3HjbjpkMfi%hpR@_TxBC%m`QVrW<+&u=dDZ6exa{sbA+4Ao3F$LNuJo62iMQBbgzmCJB*yjPjgb7bEh#wFOBZS#ldC!m3TkN(uZ-EAl<wqug2&ax^C^+)T`!mBhjB~F;e3Gm4HZ!sbq%TB^al@(tWaPb@(N(gP$DQUNCQuy!IZiv?5`O=91F`oYVY2BP$A@y+guD|sKTuJLz^4c=q%A64L9J$p!-GECeb^ZRc8m5mUSJ|M*B<6)MD(9Ga(!Yo6=$GFnm9BfrHd6AiQURW`VTDM~uo*uC5JbB^^j}qqf7aeSS{8C*A0XGlQ!r+8qz)Zsgdr-EFo0^wDi^I-n(x^z&2vZ8z$h`NrkUY7)vGoKSx3yA-mC71RB%-?`K*$9M?^YG>Bi@r)s(0<RK$$x6Iu++!N)lp|F+l>sc7wX3f^qC(`6o<1h3=s%4&A#s2=lT{e%<=ztloNlF)xzWkN7ig}wHxnD1fOX+t3p>O^?@_JOo~*eF&}Hh?eTcE+LNrx3bcBU7u%-dG`f2pFwb)emCVse(90_`s0klYhU(8~<<$5%Q#RhR|%$*1c3=L3zLi_6|rrrgkS_sbH#ERHbVEs)euS{+4LO`4ZtfHPM9Qgsy|TK^50Bkg$Hk1oJs<@r`Z4!T;}-QNyq^YRt)jlxB+iF8Z0Ya{DhlM_B5XO17rlQZ%<zpgM!mc)J1Qezwb#wo=qieCxMn)86MJE5;y&hk;UXoOVln1Tk1*rX%8$TxeO0iYi$#-UFb0kc_DzgagsQxR9T!vDnp=PBGVLA=L0mvs<LIDV4*+W&>u3Uo@teS~=k-e6Q`yfg#<&j4~OME)7omPD}Nl)U1#tk#MvhE~)d1?gLakmEVp3VX!2b2Vn2oLZxgCevvn6`%?<KflSGb4C;;$w>>UH4SoUtMjpHooE(GEK+r+2);deX(USL7f`iOcJ(+l=ZI8R0+WMa!R@qrsZP)jPts)?5ca!vRPf^+C&_XgY>LKi41*3^jmyA@SZwZ~V030Rmuh$p)B!45BmmB#t;<wIa@>WgvlMzdJn|aK>7q*}WZ}E^)*5T=2Ylgy0s=9U#V-2hrl!cKR_fw!Zxbf8OvNqC*0>bDoW`&ODZe&@;fGF=S#vJ{%-e745jXAk-p)fvcs5q*a#FvAvfMabpbJd@zs8}rpz3n*uNCc%S6NcD*A%|p+eD~|kUn&gPPlMDVr}Yy#mOHS2OFGY{v<I_t)R|7qivSb&GVA5y_T*<f<k#o3`sWtN!x2vAVC}?f7T^rV{Jh&}Cu8}Xabc@29wkQ0j5Qifyz;Rgq(3$qJicL0<&}XO$$?u=1OtG4Vw4z^R(c|jegV${9eT~YYiLd5m`*U$p<jjzo~zAs1bsbWq8_ef&s!OBubD$bNNmB=Fg*-F5{K$~D8Ra!=l&7@D+Hc|xa55vdGR(7)HwcHSRSlL)ZKfbqun$4;V-Lr8*?2<6G&KB#g+S(H~&EnpLON7MN%Yko=Vn;ruj;ysmgeQ+z_U5avX0N$^<;eNq|(RH%|mkfg46CmlH#wJvmE~VX-;i9n7<?6;IyF$Txd72|eV9OX&h%gPmF$DlHhbT0=I8U&!35D_j)xd}M6feLrhU&elFT9ie-?j@l4~;r7g5$pk(oSyQ@Q-@lQCDS)j9!%vGND^Y;iB!BG(L)DE(VeSGL9WdZOu^NVsISToT6AAG)HE}nEA_Z^D#V!veL5Gtwn}_O0_<8N;V-;Mi*b|zqvQNV_x2oVDKZ?m;D?3mjbw@s&#|9>D1!eq8Yq~1Uc)qqwFjM5dU|;#>HiSz6U>V?8P|Ry2J)qt9RpvW7lPDuodVub6b{Dr)+$Y1vcq~A#%J9J0&tEy+Crk^&9aq3HXtB)v&{c8p6`V$oh|%tW5NZ{AUd0es{aql{TOo?pX&Dw3VDkJ4_!*wJUMxz(4g*y0*Dg`i;AKSt&WDoAH?q2{nJ$vOs=@ZFDG1LdKIBPto~)u*ac&-U|I8re+jGMibzjOE5%7?+kt<|4QYD3HZbMRiPzZ!<WC=cgKw2#fC-hcGN0hILlvpHtL|RZ?V$d#)#7zipoR4cM`e~f3#9_^gCuXCm=^Ff{XH*WU$V)60^i+tky&1%jB@TkS)1@6Mf20uKN{lAWO78i4i(M!!6ajzhU*xr{ohhC)C`hd&Rsr%9SU)mI30c^9X}F?Zo2jLgKof8zL9w~m1T769R#I_d-D|iEPsCh#RzeG@_Wm1<a5SGJjtg;$NA>8(^;U&h^6<<uqS~-d@rp=C+`kI1b0U^dw)XsUt9au0&J3mq$8xEwZHmxhO^dGSTpQ1es%+niXvSm&R!)hm0Cav4DL)QG9R+h;=$P$Na{cs3H7wj|i}OWpO`=+irKG}86<{V<1z_Ou+U-&<d}_4EDfh2DDyBDkaXCvH%3zrk^zRL4uFB@#0E*2r1;gL(FROfmW98SV3^gobH_J-^`hB|0i)~%p`)R7`OFzw^I1Dr0xwHAi1oaG0|7|JsA;0{Z&gNrPIeIup)jx>&$`^K|r+_PQ7fKMEy~V4j;xw5|*KrH++*9$+Y%V|aRjGQWJLPLNf?y8@TSz-gwsj{LIooNQdNb&RIk6auZ;FrG8+X9j(ZovI8Xu$rJK<wF8z_mgz@yd?CLG|Pp*P}a8CN(bQS{t`s$M4674E<G@w7>xvW<|Pau^hJOa%jjTXY3Pp!>i0jur+BNzyYs%z)V5s5i#d*QOjIZJ%-Ae{IDsY9jCXdjqQRqd%jXVY5_N((4#X74@0IS-52=0U&`R3t_nlzi3!zo(s%J)QcdlHxO$3&mnA1I)L(}E!!6eA6QLHYCqfHK>XrYY?6Up*#Ksm-HL#TNj@#ntljaLZ@ta=NM;sfGqMX1WkDhnG#+YOpP1Mw^_!e37ZxNVl?es;dLHeH>ph{e$+6`T$1B%S{IuT6Q!|n+GQW2|t!{Q|@YV%zpc9f{6tq`QXMEJiRcMqI3m7L2q?u$1b0EtnAMZ}nRd-MO>T)-)h@*f#kfS|I15AyuR4Z+hlkq((IyMIV#MrnaSL&vR)0Uw~wORnXl+($;(akA5qynKAp*O$y$SXp+Gyn)1{swRiL+ZQR7wtQfr0+A0GSKU`>xRETNf-i!6#tGhA~@tOjQi_7&51feE`cs}Pkm2C<L1L4ioM^3vP1fZ6*b4p!dQ>F!LiBHFOs60o4bI(7_JbHwg`_h<&woUPXg}CN?1h}x~J%{flQa<8q{PInB>`s`$N$2AXz<$chkHiA0fHAQ}`-S-QI;)+7GV@AsE}BdcIrI^0?%(e%C$TzPTg3jCxL~WT`ieu^7*wY8JsysL!E^Rf7dS!;7ii`bwv>sdE%d`?q6L4j9_=6xZ^cu0wxSATx%akZaesG$W`%ni||hB7H`vWwtu>BMfHb!H`6mGH_wP*c!JjejVga{2_tPe_7k_U0S2BMdsRHwAWMLH4VMJFk^movPbe2_-6>WUDJ@rA@$+AKj=oxKY=0C4ebs%m}6{|4nJCW3}t`gTfn&=svJ97jm&v2xr6)2@L(Z^w}D75P6E>3Lp6TClt6n8?heQuU!w;Koao)jpnTU<^s5(@H7u_7{>Bt{<$uuSRjj2qnr34pK()i7N6gPc`VK?s&nq?$WVs+B>fb;D)*S}IP%ac<@kCWNv>!E>QC=SEUxxa<GVNPgR%2iR)cOW-nsFh%x+BFX)X5aRHXUY;B9fE(cbn3PK=8oDS~3G&#mS2@z&jX1#8$VPNkH&rVol^*=PUq{xCzz`9{WLDo`&JolnAnGHAq*i<7Q0ZFPI2@w;|f%tOwzdem5X#qljPPqNNa>y76Ic!mvyv_go~pyBHIl;$>ymafT%n2*RN9F+r##@g%9LDX*ouj{2h_y~_i;7=cPrX=0y4$t%eeCq+IMmJ{r}GN96Zx_?b%HNwuXSU=SO-`Mq$>;<PuvnDF{;^Tn}*bcvt)@aY_W%p;fuu(Y&!oGHT8j_A_eK`;GU=vkH)0Pces|x}j9be+^VNqj$s|UT1M(YE=;sz<2T;V=q`Ozc8kW!@Y@(~g7Iy%fm{%I_y+J;iJAa``eJM=C0({i-vw%i6IwY_mQI2?ATn6FiPZvV)v8Z{@26`M}DIH%8e;B2<|&Ty-`eSgXs=`O1L%16m*csGX`Tzs{}j;Lm{7&n<CpY|!?ZF1nL1aPQ`Sxvqq#^0R-&RdEa60ylB?)$8SdV$8nj_}E5jJL)l*X5WXzNhT4aW|k{=)&is91@DX?pSe(9{7Y%U(vWeHQWixRQm8x1&yj?(V0pu%77Lu+-{tm!xjh>=~n6Ip^tb{*<F!P(5kB+LWN*yM@FJ@u)J7t)5N-@^SB#D|HDWM*dM=gG2azirQ2oTivyKhwYA6B`oi#0MvW#i;Lg2@VkA&a=Eni9bm&X@?%p=8KTk;ewEG+IrX))N;psM0^V=Yv=GW}3kpfgkvOdF-F$Czimw5cnULN|Q&gfH3NA02Cx=@akUO|)`P+S$nE~U&G3dk0Qsn&A`>Je|^fsaK~;a=?h%J`{$Tt<jt6duiT8JR|5(4d3Mb@!qMzFa&|Y4dw2R_DK@Qf>^UXs;ymv2i|jJD|odw<G$V3ZxyVKH34&KJF@l)EdB_>T!$%gO@=BFc;JzrQqB?%zG!8{xW8ktjJt?=7ybt%f_gX&zps}_dc5_uJK25aDBmHS8V*%;Tz**Hd2;(W0p^orFGb#VRap#n|#<5!@ZDDM_bg?#ADI;qxuQjJtVPsnvmH`N;hPs)+w!=Qlszc7eha2z^>A!GzTLO1=xxPjr>sDh>USR4uuf&S68lqe@F@}Bz*r0E>zX8iIIXkpa%Zb&|M39)cyi=psZeiI;b`Sbumb$=+eFppdP6#bQUOxvOSoFuN^iEh4Ey4)}}D5fK6|*>a}DuQL=Lvo91#tu7o$~#l%!2FON-8crY|Q9dJZO4=CIbs+e)Wj%wdrE7N4G$?O6lk81uu)7%8iK-la5#SPVHqgdHqhI6jJhzj$sF;ZIfcHJcY1Rj4QG4~k{LzLm)?iGxEIzLu2MUKVVsbl^<GG7~avN}*$M-#}pGiO42Mg+@;RY^Z(%Y6Nv_r`b7SOCcz<hhwo;Pa~XDOXyO{v)#f4vuHg6<8DV-c61dL&P2t%pp*e2LAd;OSC>TE%D3{5f1;--eHcV43@MSlcC~jF)zVsi)uq?QqN3>^@_YTjqkwrB6VD<nR$3SPY%zYa_xpDMOM@hoqx444<;$lf{Kse-#we&MKK%Mw#YB^k^xU-eMVx^cloQ10?h>-m-*d3Yl6R100vvmxJV*|lLodV-O}Fw3=cZF9Rr}P8~uz}Bc&Sk%tRB+t6U2yUom7gkWFV!Z7&2`U#HN@N0Me$1Cvp`WZ5W*k28cqmiIw$m`qtRt#dV*i$*<WgXB}0rAnFCQ|}XQaI6lB>BDPpk+7#7Hl9h9r$+h84>G=U9P&os6xS*SFZ^DC1bsJ3mtyyqbk`lVQ0X|x$9bL53pUnnslDFH-oHr(-evZinL`pgTJ65ROR$8|ENx%(zW_+~+Z<IUwYa+qhMdrr>4_RY@Z`S(dwOJmmt`Cd`n2w1DehV;>3&z40#z?Y#atIG<|ep@^Glmov!%sHZ7mC=fa7B)=ukiCM|MJ-(Lz&+4GOE7USXVPKF!44sMqGF%3*8IP;SxSPsKyv;VeP5_ZhjIQK@RVFiD~77)qX30ifT_ATIkFNz!ec5m<Of5Vab-@$z+Ya{8*B9Vw3J_aSr6F@`;L4i%Y?&mBf7+=%WK-F4b;bD+tr$xp>e6LA?}U(L}y;am+=?oOpf03y}7^iFPondWCi1J5W%xU%fFEcAc9NWj93+7<GK8<FD`)E_~ToabTJ4LR1&)OR6|ZhRm*Xh>}gwML<f&2o5=gQ4hv>P)<M+BoQ~1GTwGLqe~jO9iNo;yK#IYwK8qpBjDAMz22cUF$W{Kd`@C|Ah9|$LSI-e|QqE=g;7B593{N6gbJY2Q;loBeo3Yax1?V%*KBDK*8hn*3Kiyh@Lmhd($1#Q6}XncTFLb)`23A{}`7LRG5s-F^7pBAJ;Fu?=@Vh_G2-Jg3QWCmyTFf-tYA#6e}f;nWfn&azBw#r+Md?q@ckKceR4QiD9WTac_f?>K9&;Ry^>}1yEvtFf{M2gOSOCb!B?Niv{(QzChJiMe^0!9_v3<SVD+(AmHoSmYT+nU+lkq|C2iXO+qIjbVxiPZ#&+`-9Qe)D6t7~P8Kr<ZV23NKSWeXI{e^Fp(@f#bA`~3n?mI;Xn{z`KpF6Yi3;y}4B?y;{0O1bJd9LHhQFDHOoFy2i=!2<w1D<Z8(9$QPfu%Bb5ydbmLX-ThP7ZEATN55j^L8sCCd#f{iR!S?AS$Kbq`8%Bel056;s78nF?AwCLMB@mo{thO3+X#WaUl9q5Yq-eXI>Q!02`fe{0<#VRnM5J;|xTTanPg0n+KeVzAnJF`-wEFPA4uyslq0F&mZ+(s(uhisTD;mFnh?0`OG2s7!5*Ze!Vioa`1WTi{>oPJU;>6#7?~sz@*u?^8~de?DZaQ_vdT>~~AlCAnB=dlzpf#f5d`aY?d&k>;y1Y1)be;hXN02WH-Wx`}AAcLTViIT?nO0F7qn+IBariK|sYA@wZIJ333y)JQgCDrgz+Z#m}yJhXFeR8v6ZrEi0alRmv1Q|d8m_7l&CCkoZpQ4Gzgb9to2e5<v&lnaFP{N@XJH14o@HjYxRQ1>>%&5)>t=7Y5<zJZF(oTJ%VlgHQu!|Yh8>}OOUBiU4e>1Jq1X+o>M`_lS6W#=TdOqtIvHs8R2;6}a)wcDjgGPzPIXQy^q+f6!m0PhUKEjYUVNr+xSAQaWRjUGdw;h3Hk^aTnd6IhB*r#Ra8S3bec&R$-|E{O7u1tlDOl^7whaF0y#gE*{T<BgJE6VZf&9vUT+B{m`}|KdlhI`1+PYQ3#wD6qFX(dA?i&xl}|vjd9Y3J~6-%9yT^r9!@0=mu&jo5zUTHSRg3oJA`A#-9qnnjJOUzPpqhOD|Bo8n(2v9~H2X<C?o4xPbDLQitF;EU68rxfB23>iwU<{_mpETx1PMmwn{YW_;yhq?FSwsB*^aF3rc3HEABRCwXt>u*LRts9NNkG1IKjfcXsyMBUNZv}QyE2X+^m<mU68vTun26zl9k6B{BOq8|k;*a>5m_PIHt3-HylpKygA5Eis%L^qB&$!dmC)uZ|3;7Ddut2>SVEG|u*2}gEzOjCw|hBlXvk6&n<b6z7H@MYwhT!@w#uE3wGSRIMEst+*H+RJjUmI1iC84L$S$Yj59Kf?8@o*TCzC-bY4GRRu|=UTaS$#<WeA@DrY%G!yVVP5i+py~AEm{r&@fC~#q?{dGf?lMkrPDGqff9&Qx8ej*D`9M1QIDDA4@*jEg)5qwj2K;0nUYk3lsPWFW#!nXD^oSeT+xOt+2tO`W%Gpf*OPzrVA|TJg&w>!`!ppZAfA2duI26xB=$DEvcTPB;eQx~AM=dN_NJSbcc7UJ8ao|TOKXrd%?E<KM$BT542bEjBqE-5o3-4NS1{CA#D=NpbM7`8X4q}X?gcH`)91!+w$`KK+-a&P|=sLfHgAW%Z)yFUD)7!FWsl1D(Lu;n^j#%68`Ou~#i4(0*UO1}lrtF}U4(D)K27>vajq|GqsM8c!Ri~c!zxvq>9)V@ij97|-oOI^bHCL~;4{nhGcp`}GJ9ks^AmO6O`x|s|FL6bxSOJjj)IK%|hHG}2RK|@su!~@aE`nT0AWpcX{MzI;L8Z~iT34985mn~$q;s1+k{bI3TbXQ+o+ORxMoa9jVWP(4fK*Of(DkJjVcZ*aUpn*wguRJ}>S1JQey{VS0K~ItHlBW2<=6Kfo8bnAn|E~8?Uvg7cTQ*Yv<7*2TW_^qZOnWhBf6FA_-7(zm`AP0sLC^PePGp5YAw;EDHwHS7<C&f<?`n-v4up8j1bk`vFa&=*VBBBFzbFP^s)Wfl0}zkluiAr3x{%$pvklYOA{HRebV|<azTmX8Ji>!|D>`Dvwe)szXU0l<||b(FRR+eGbb|XS^8`9TkT|(<1f&TOYm{b1otR*ON>B~aU3|5=v#IQA(0)~ThDn>x$pydheZWvZ$`lFO#fG_x40UDRcAguB}jT?=#J=y7Lh}vC^@xB9XfRTUgvnaR93af<>$0l^3dc5c<s&30jA6N2drJXo4RREX)WNk7ZGKH$2?D<hNoyS>kYKB^=Gr}93#RN+FRu078g5EO%N#oE(1|W#ZkS=QdbpOu+JGh(8^QB@52Msw=0T?9<@sEsePblv%1o@vOVmuYY?jRGxm-52@Ax=miCs{>+rOhGtlP<o08YbB#HtY(ZTq<%bbjs6|Foy2j-6^q0!t_QPsVIOGwhNLDwKH*Pl}|vOF%+oJft3C8Eod4_n|(tC=~$=mL>V&Ddzk!M3MqUaM)TDk$ICJJ=77|4>VqoSmtlY3GJ;qE?ptHR#eHjt`0tUI#KZXVean%$>^8Lp4y?`);btJ5uk(SZV25ne&Skzmx0BJdNM?KMrvO;&<HK)Wm^5@r4~TDC)AgiEAdly`_PZ=(R+(Ep&C}H@pM(U`?o6nIJgQpL9%1(%uQMTe6v<39_S&V*%C6EuPIEy%Dq%hH7F0{w=5}+k2pEtgSqBVCG|4C&eSM+RF`qz9iX%1*@JmyMRZ)im=)T!k}Z9iZJJ(@t&NIYrq5WzXD&0a!CiuBkT+EaNkn5?&_XzdyP9pzebX@F}Pt>vl9>A1k8O~j6%hFVW7Xg%<+5{TTjd#{?nd~i76joZ?xlUO(Y@DiqtFWl_R|mJJEXv`#kclZ5XOE!Ksi<LTS$p<o83PCzgM&x)yJ^#4cZD47X4KZC}g>8pKJbXbu;H44dsjdS)ec=S<Yr=Oj*!O!W0gaCnl5lwtpce6XBLj2%tye}o*+>wV6*97LB97Gw=Y(XHrk7fke9!o`Th^Y>P&3te?m69|s=!2LLW|4eH0qq2bbd4ht!YbTqKIsjC=T^2^F6fynKsFE`y>HNM@rWpB_8Jscph;`C;eo~yxHb@-?c^Kc(FoHnoxqUGyfBB%1!gD86g4m`&R?W&Ku&@o<4mxJ{!1Wxv`~|8<QN3Ry=^`T=jyS4O6QjE1PEPboqB0Z}J9&vO-Jt}R8WOYiwcmsPT?`LR(eNi!B8Ce&G0!Ea_GYt)sZ^?-{19+aVc4w14Rk9Uq0LAqo}oiJX`=S=4;Z4*o-Q~`g1?;E0hLd^y-VJ}E)9#<5G<aww=UJ0D0d`1$Ki!5vwYCo87ovs>{+Igh=n2i4yhklWxW_+Kn^$Y<j~niexBNw)hs8nuK@<?RlD!6V9IxokbN*Yx8?Fvb2dAzqkU3vCb|NPWrWOcri<M922xwwQpSN$zt$L((&K78e%f4<wsv4B+}-*qa6v%|)90%ZIdHoAq^>c5zQ!TMn;_#WcUNnNLJeE5fk#2kle&;x2a=VBq<@FYzKl3&X|IRGU<7IG(`w#)M^B^9?Tv?Hkjo4mY=5q|-slv$y9B~RRK?MK>`jXtmXVQ+XK18=_7Ev@L;JPLxxau~9AjhL@qKjfV3YqpJgENzFZ@wA9CP_^90tR0*_?eg0hx5mj1M-P@e9yfqzCOrezGGU+Zql&0E@?-dc9^nMx$N3_Xg|WVn8TTJIf9qVR!Qs?l&w2ua2uW!GbH@@Y79<);nH^gAz#~Q$$HiO|#qCwW-_GwD>Z@>T^IFXav*4O(i*@g<w@)5JvrG&bq5?0(Se|p4yZiE7mj~pgMCB6VF0_yM4(Or!5BN04xbiRqG9-eS#l-1DJ?(fUW%dkg{?|Hy9}sJ*rU25CN?EGb$%$7JTJj#LNhm^xpAp^HQ+_d4nQ`w85@h*;0tds6M_f=vK(7Xa_0fowv$<vcTA2F7VhWl5}l9hZtiN7_4yJV-=5aNj^AIOe6P~!^DBVzO<|a?dxm488MLMK@810wx^atqJL}0$C2a0hbIMuO;neB@+RakeDpCfN`QRVk;s(C6k5DA@SGB@d?OlmApshcP1MfTahBvyk7Br3UBZ#t2Xe)-xbLd1%WZOyWAZwVv?0O%i3Kv(Ti{mmE$_|bYy2*~C;tQM85g?B-l|xoJZQ6~X?ZJM;L_P!DxjMpRm0Ni0TY?kmb_|vak<FT=adKJV11G`Hp>^Xgpq?_fkjQbya_vi>|Im|z8e#UR06jrML<0h<FB!4vaFtL{}NM`eprY1ubLiJBYR^wW8XcMOdmdO@GYNeP0?|t3@v8wj@AAqRw~-vkeJp>F_zaTy_}(0&y7?_k0>9OraiSkUQmn(!=#{29vs0TdZ%sClRd@XwU|v6L{mUSsoQ+y;lPni{sId{R687>`jQv7hQj}3X*GCkrFEbycCY#5fEtgR%R%;87EGeDuailJsN+8D;q>kLGgl(#n^m->5=6B2XtGShV81R%mNmO<qPBgpb1fCf{{8t$YkTbzo4W2IPPtopN=8ZXdlxhcnTL!}#Xo8VtP34+xHMd4g6Yy<8F#gKkyG@Qf1k{v^xRm;o?DnLfpi!Nbid(RedG{PnKsU<XNYZVpob{iV)XM5{x4WomKS;s>NWV1KM?nu6tI!cH3($(C<sM^?^H)cpNjUEwlW7{18H7%+7z5}>H+PU-co5XlDiX?^9NH9A!8+?AwvPwT(E}cCpYwq9vMJmnY-YKuLsQH8?}C&1b2}xG%YRdl{0PGN971-A{YZm04N^`&d|jStwX-1h&zsrQXcNRUHkQMVKvAm6RNkV44L&e{rg=MBuHWK>2@pVbHx1w^n-}mOxN#IV98RvkVmMH;W(VYN!Ma2K~j^F79t-Xj&WL{-3o8yTIlE*W{dc?P)N!A{lK$HHd&N6Rt+<+x6C%ONggRzO2^Pmx7qeK@;Me_NtY5n(Q8;RJClwrurs&k0<TdGK7O?Acz^f{iMozXaqf^VX0JO3kYX7WPMkL6KM{s~xL8@0MhZ%tgrL_vYqLBO!~hv`P`v+6r2Z>HRG^3$l{LJ_I(&Mt@f^bvn#r|!bV*I^MT+B{Yzq4O?Y|8<G7+E)47+xhCh@EwSSC1J`}V`Uhzd6JPk(@7U`-xqAO%|I$ikK|DtTO%#|aIO=PYg`O_!L5sQglm^0E0Tb|!rsPmDU<1*nWVJ<1-T%irJtBh`^;c}xu;BPHYi+SIRmf}Q^s5=u}!s|4pqjIYI{9?KXt%#r6dC$Ed<jJe;1?m(Var|B}q7FS4i{A_F|HP)^I-eRueUC2x*j8TCT)dd&PF4cQlc3T>Gw`Z>+RhQ33rh}rYkav7V`WlA&Wb=1-YAj5$$sEZ*$x-pit5U3x*eA!zw3IDuPr0kc^ojNQB5hp5r!X0h-=U?BYusnI1#DFq7|K*aR6&)MF0u5sPPtx<P}q^YJ$Irz2XuSw-ayVg{>yq9-*KR4o)2mS3R*y7;8clXg6*u)@~=7J7BS2p9MFM_7?yh3+BvZV(~5Cw{dY+Idcfz$IT>0~4r9%{|9T|jTA;|_K@@$fRR}?)n>RF=1%O8F00@nl-7d+7(@kap(|=f|3~Eq5u9pC&_BJtD^-*q)xK-$0i?(c_HUs+Oo_w_qdLpLy_NO`Vw1%YO6FvaT8QqnwkthA8wg1I%R?u~y%+e8~t%0P)^GS0RzWAeWys{OX2U!;B=_$&9Wc`Ul0Nat(k!p)MMBylh;}njN)Tc(486n+*joGQ0w0k<pU^Q|T*1w6puWwB8v~!l1#we$sNSlPhgIo2&TWHSg{rJqJtE^0ui&3{6dl4VdR-lJ#;^*vm%t|703;BJC5N&m+9DD7YXc|V&-63Uec;@|Raw@$$0}?W;l7HTNKm77$*^T%FOpNP_z8B|LbN%%-n#sEZ)KB;+c4Tz|7l&DGg3N^cg{AHE23y#+S$-$}OUkU=nyp@dQH%sPn=yVWT=`7Wn6?&7H>>3oqqI08Up8MeXqsV5Scx)(?w!_#K@+wVLHurV3qETa3gZkAIZ)dpI_S~FiJt;{&CW10m^lmS)Ewt<f5<+z9I&Y;WK?jVk<4D!iUM%-GKuNDcO82UE{m|Mw8@%dmcW-((>qO}MUA`}NmWF4^iY>{D^=+EwvTrU0<h*wh#Go|pjx|mA~dy{LR{Tq;EcE~<#0^mi=(FyBTEF1?z}MqzL?k4p^=Ax!;|{FW+){f{aIS89UUyX;PZ(JC<zqTlbMNM=5E&@`S|Ve*Wa9bS5sl=p?5F09e6Q=9a1ylwD!;$4JM8e$P>d%=<_~Ygv(im*|Q~Xv@)*QU?R|1#!@p0O9U>oOYy7S3q>IKnk(wu-6uCgJO+qxpe?wR%4PyMKNSQ4ay;vLTyKM@p2=hlFtk2oPcL!c-otyb-FE0Y=n4I54VR3iLR6wGSFLCuX2jzsc>()=&}3r+anU8Pz-A*n@L$OHt915jCSd*PKQva9q6$@M^BV;C!{7mhzLC(A3o%BV_HALi3Z6?9Ung91G`Ik{XPz8j--cXYP-k3tF^nDuW=U;qu~UjH9pui+SHI+6kb_(raZ4$F!3yNrM?UB%c&!tb^G(A#ns}xkC2?<=5>yq~$l;O5<5wb$g#it-dScd9azl5KmA4;-t9NBcS-wVtfpWSo#wAGca?xW+=Ao>?oJg2zUsqwW;~Wd5{yuCk!?leIdy>C7--UtEE+Ndk9NfuyHe;?G7YK~wp?*1ytj(MzzMh4fD-wNoL;|p;uUQ!I&x1O}W4?5{Z2Rr`*`S*~a;>u^SMw(3sZ_VGnc#x5AGMSd+tqU)BwcQBPym8Ai7rm1SzGcpn1$<zesw98uN})&$TuJy)Yn^Hj=Qxx%EVFhz5o;3yh$}&tVtQw1Q|d5t*V3{mI}(sYj51}g0)iK4J73F!hMN5V{@DhL&}}!){F%}$e(T-Rb*e)-kEa=|H}+*ey<ngZ8<tGeJyId%S@&M&>|$2wl^`LUS*IyR=XWeeUFBAL%h%<;#5C$OI145-*7!!h8!wT#kOjFmr?(h&+CSgHIfr9uUJfEnxn(J`y{&c<T{R2MxKkjB~g;{lfd3pP_1VWePPTs#oyb)`8)k!{{ON!{|94NsFnR^d&Lr-VkwrK1>_Q8yaO5E@=7>GB45_NDO4Fj#Xq&s0lDjEv&1`sLG^)Iec#Ibeo88XMSH)PMEX$erZm>cpIid6EEXr;GS>S^V!k6%;DQPpGl|-ELl}V8f`5X@S+0M;W#E`nXI$%cf2VyQI{hdWdkK8TC(SMvjF?5zvnXp}3GDtM6kRD%De+XXew-}hP{4x#R{nHO(*Ko2e`8z!DEJmu(jHMu=l(ToZbBOMQUw|ijT*v;k?ORpmCFNwOO(IhzPVE!Bwe%+`^$q}us~E};pCiK`c&C48(p*kCIZqfM!9PDxxfHR?u8LdtED_tRkKOEiSCL)LMj<-%lhogNjQC3jY^Y6#St<8C%rnCY!e{b)mtwyB7&vTC5T-!1U;g(diT}bSw`CD-DI0UKt(H3@E|E9MgoXEBf<vjprn<$l-lVu!)Es-*r&lkNZ~eSquSJn_<Q&s`)^ZO&wF|5sUIi;g@Y>XZSarZAV)o9;Al3k^13^lC<<71H$UfY%Z}TN*rW%|R4C7DViE4GSQt%}sL+ZQydlaXo*<$(fQ6jinY7&lSdg=?et*F&85wms6A3w*Dx}isihg8!GC8<>p@p0JO0;b#&efE&GCW-%wdjtD#3&mMU(;Sj)QRnAl4ie=A0+2rZ!ilkxJTm2vhtCkyK<c}%CWT>njh%vb`Sff#+M>Et!^jRET>Q+MFz-BzBUznaluaXU<$T=(C+6vnk=G|a?JtP_w483s20e1C-!+J*A`FR<nt4^l4dL$=WRb?{8>HU!8aQzdU4MZwd_mw&)%yr#j;bxD+q*^rOMgiZA(azB_(eIE1~e`2=f~RVu*^fWkTyrzlpt7?f2fDueeFPwd_X62(jf^#{FQS8^3wtiUPj_*c*OF;~-C6T0)*3E$I3JxZ;U;?kpcD!Vy<t`tS&!14KTYQ*s+U-{&p?wQy&@x*msXxIorlmT)wLjQNo$25<BilmHDfssOds9)kga$oNz>Vbb}v$@H|XEFQvO-u$po4LAP-5pDEPQbtc{BL1qLCai?p30M#Vnq-t7RebXAHyXax26v$DNO0FNQL=o`MW7?*)pR4WSVSHbT{s6)lDa|o(z8F+)m9WCfTh*UwCDA=ZwT(6;xy9STA8SvuM4-^zu|qe3Nproj+bck?-=>l$a9i-5F$5}{<s?sk>pgD!tCk;+Byol8dthjQHC-?b}OFX91c8kkoi+EjzK+yq-qBGde*_FQ*Wd5-<=k#MQTAKhMNY{M=Iuj!REDt)CYPnYs4=gsd`ihXTykuXGoSxRbrcxZ^B%oxgZ@H-*7h0Q1?>XQ4j~oFu+xV3%QC8>BdJ3bQ`=f4m7YIqV71D-ydxwL@-MsFpP5*Zx#t0i(Mr-<l*^r`FnW`32nUWrKZ{mhGU?4G=_D}Z3DU8m&&E+Vmv~+U-6w4Jk0Q{*$M8#B|97z-q7b=LA-BdTxcJ(GZC)&b1%c0|Ifbfr^ZBhUHA}WV3aDcg*2Vo^32+}=3A5g0Cv0lj(Nw}S3cSdvVKloxQLX5lO}CFpT|wH*U<ZbQACYFoww=Jqn-0x{3T5|57U!QMrBx*FJE&-($@!9J(6I@n&7%w0jVzAKMe|zJ_=NT!eo8+w5l4h<R4zMK6Fpi4RCCtv?+Wv=K-ZqSB&DS)TpSjLTUf=k1@xKd<xc<2C(7l;jRPinHY`ZY9;IfmUxbV=0{hwYqefbkU-Iq{8EPbE&5tk;G!W-#U~oz$^amV2W8E2Nnyl+LE!9@m^j@2<~EIK$)$4+>H9<9%=1!#t$xl^TzOYm+f|snfgQiu{Cv2&C>%h1XxkQ#&~S>}D0xK2{S2RjPt`u)lK6v0e>@?ze{EQo>;-jobPiv=;OOwytl~cBFk0q|`b~ND%9{sQS;*3U97QV6-%yUTJ(cpSV+khTe4tVVeJgGe=U^a=pkE}knp&p~vCMiK*;xxOy@88mnC;+emeu_J$2Xzk7s1bU0dEzURFC8E&Cy#H@ZMIj+sWuYIPSo(vpr1M8v?YA0BNQPs5B<%{PtWa9)Wy*-lgwl*4DLEw-KINKk*$0apCO)HiZw(ILOTLfr?An;=Fw5R9CtI0fTgORunbaT%I)OhT<d!c+2hlI1@iSPUfp$HBTK}x{?tbq>r}WH2L2MLMInFlvg^PfL&SqCR6%5@^eBdqL}at`=SvNq5c+PSr&m9yhh4DIxJeSHpaQbKa3_oNdKuq2!-I$I&OWcpiDt-1O)pYekbUcTW;7OnjPC=smfpDPdbk#(e`yVWPfnQSLr(`{Rom{cMlU(wCc)YFb-s`AXb+62Yw4?`zQG0Rbxn;MxBfyM4WG{stY`83O@Iix@>bKSqV7HY-V{ry^+DGTTq1SV*Wz1acA?O-838aC$80&MZuClYEACRK(-6uUo}W3@c!zU==Fs}rP(8$lPb_M^eV;?5y*f0cW=Y7nase?waC-b^S7K=3pa!VGb)pV{L}~WZ?zO*?$zNd&&);6Q0C}h#>mU@MRa;EdKOu{?rYd=Cs7=Q2*isM<F5|_KaJNst2@uQ!8swB`6_3&Ux!;-ohd+)vy)H7g7apBn5(3bkE#an`igq_Q3h#ougWBbQv&>^bzj9lDVC;TEROA$TSkLU@P)bVTIt3DotIw@Nmx1*5!NTYy@Vp$%*CL!Qz&Ef6U({d7F3~R)~7}E(ukj~l|TNGY)lspRp=PBDAk7eb}^hQk3xxYU%C%*!mb7n=&^DP`ZWPZZpi!PL;MHPNT<~5fiagymmU$d#lBQ1H_I!ng0?4I!C>QRHda#N1Y=0>oHodv%X>&LiTtDV9hTo&nw76ptazn^2Qaqh`V;DIdABeeFkrX9aM|l+pJEOP5cU=!hd8%dH~E%Q2KG4|JxZxstH>y{>!{SrL2yEiXgon@X*`!{NYnzA?(BSRz3nK3-kb1pT`0o^IIbV387nz$NMRa~QKXQFe)?eFs;b?dxCq)9jR!(S>$V5)6v3p4E~#cse1`b#DfhaIFBS)dM0m7cn7HxUOzGy9-01CmK@lR(OVgz4<E$%GtsaLROQmSQ9ZS!jeJAeXWWd2!uRjjNiG6U(r;*l=)b43j&GaaIa3g+%F>eJW6z^kAj3<I$iHP3br+6=k-G{6m^NHyvNBW^l{f}{zKh~!xL=j6Z{WT*30$n*p<$x~xN1^KTTIfF{uMk}kTAU5ke~r$xkl|7vsxWO6B^qSx?3wH$UOymh(7%eYm!qR6?q=LBh56gsJOSsadTvw0FU0S##~;vmJOgkAA-4jGD)es+mh1!Xb0KGw`Ib_KZd}w0>HA!yZh<@Zeq)W37Q?iZ?Zx<|R%A3qQ4GKCjwQ}&p8lNQdb}=7-sh|}DzRh0QJ^08=OUAO$|#YTL!=~`4OK6IVXn^-$EpMGk^xYktF}gUkdjb%8~`eA;AGK_w%Jl&CXrxE;23P|mW4hUxtp+lzod<;2kcLZmGaL;JTv;Bc4opeq`-Hc?p_+ThT|+rE)Zim8CVo1v7UlyjW?XB&!p{IH3nC7fgkW}b=9X{Y)txdjU%YYL0NSt6#DzxJJ?zcG5<hK6#I$PDgWrUWRi5_bG;p~iT!Buwi>LnJpGi)nzt|1;4*wr=!61o8*gWX+y^FidS9tk4Bz{MC#lt=Hy@kXct=Esq=yA3Xei@^AaA|r|D_%~9eNEO>#XdB9m<B85ly|xA0Q!q|0v%Cf)iHys49yY*zX;f7Bw$HGV<1bf_+^vs?Rd~JcS`ixgv|<*u*w=&Z|vb%;qu{qYZGjwaRe1WoHH3v;IQ@Phn~^WlmS1?GO^Nz#FEusfEY$f#g#NuYh0Q9`hU5NTMAz$u3J78uFeH49h+K2Z})P<Y_zhEtoF5O&dpgroJzYt(Ab#dpXK726r~eBUP%vL78c5@UMSa<w)r%Qtoz_o<##NbERC0{8beRQ=h~rn2=s@fm!DhRaof^8QJy=+YBvyI`{_bAPoci`Rk-eQhWNkpOQ4v_}~5?`7<<Db6_#!rX?u{ONBzp;w~%-jATX<OQq)Z7Ue02X8kk6bPAv(62b{gJlleA^PFX!miLP4!D3xWAFx66qlbp!=tj3*ZnF@=ofO_8k?Vk^1^wGxzzpPo^6{NscWd4n44x8Zb>`EF$_FNq)=XL%6+8jwj;?L~_SU<%rbKey4VHp7lKnxsQJM${av+z<)t_I<op|{zrNk=*T%m#KSOqo7AfNh`9^L8L8O_yC&yw^poz3U%JWFE7od7|oHe4(A9O=P0KlJ)N3)p0U&T#&qPNM$fu^V1wUUZu8sGVVW=bO!5h?asqgd};FXzam&C}hyqPEmI6^J2YAfw-cVdiIY!ah3MLoU9;RT1l0YKU>yerC~5cfCIDZs-*E=(U-~6aJx8)nhb5dWgg{FT|rh>DmQUvA6CuKnQ1R~*4E(eG$^yfx;g|ZJTkPctJsu{+->?42v3T~c`>P-8j{H({@pQ>krajy<uh2jRv2A$12^g@2W>=ggPvtWnv!3Zy5Vk0NAmY(B5)v7e=#Tf@%8v+ypG^Z8SxBe1gK@qm*f=67vy4JS$?P}15muYSQXcv3TFHc;qu<6OcPXm(qhtsUBSZ;P|X$BHM8FPEFT*dZ{qgU#JBEnhQLjyb=@40QKKS=7NdpO@&3eymf6}qP=Q_$et+zU`g!^;c{t(7c@1|;Fvsc&c^1w4%FC_}2&+O0A9#=cZDh}Aeks!&b|D2yP^~#WLn+ntilwfq@?%l9$h<TudCK;iXJG9PVr058LuB>(_A$r56lwug|2>p|mm6R3CUl0+;|5RK<`|run(9ij)*ZL#=HXN}qD=MNfF&jv7@1pM$r)b_xizg^9MIB<HN^u9#Y0ciWgf6YS_g8^ACwULpkvx23#gKl5en-Q+zT9Nwe74MC_lNxW!-*Yw+iAiIF?vbg4@^^W$^#$#f>-eBrzz#=6G{KZ>Cag+r%HAhG*ygdpE@m`v)+Sd@P{(<%FY3;DX4F+bvoDxlh{~wydv2)1m`2&q-kT)iS%!Xd=Up^x`7{aCTvpgF7_0Jesz<|Bey5sZdd!x@D{7?)|TWC4mg@$op#>OaEC4RXvyrxLD4f-W6BQF9u6!1ntYLEv2O)?caUSe6gzb`r;3((9ddho*w~>r;py|t!GafECQ}aG(9Lj*Jo3Om0*&+fhs8JXSR&^41WNKy<oe|lc|z=$UO=8&VN#&sPc3u?sm?4DM6z?T0<@p5lRaqu<h)v1FBH>vMhW`V1?jTz|oYDa=wrix7eG64c<gaH?&bvd6xT>6(`JBMcVLzA?fJmQBMox8Cmn{<dN>}nvyTwi<$guPY^@L1Qng&-b+DRRuV(tiijoxEt}*qi+YmgrznJke{$H2|3ABND{L`^AG^$%1!_R*rzs4a=%@9efIwUIJ6V%oiTk9-8hHZFE?A@B#Ab;!DF^3*-w-siWt0t-rSA0%2>VN!mEye@R$ocrBP3E}O$Mk`rY>PDT*AwHtPV@Tp-)ye^y9s=SaCA9A*(oU42CJbY6Y#qp1zzF4_Dw7)q}FF6YQzX<DJwV2XKBQ5uvoIms+1ip#)G*xLWgtz_L>3rSMbg>&2}CdbxI=9NzG3vv^C8yF5W`STw@erSCFyBPSBwu}v(_AXV*tbMABA#2s?E4fvFDoxR$bkOVj3QWM$}+&IY*Nc|4&Z<|OWN*2wTPm&x|nyOcM!eH>n@wX2pg!)I(QOj(6gqj%_Lu;h-yxxgF2fcRw9#T}}4?MLcF&*;tTFwBdHZBmWZLP_wR0C@yEt$V!oTq50(G{(Y87r0$MHm+9%*bWO3JMDv|1GD)6BFNw-oun^;FCH2PjgQ9DD_jdxhKGkv;UrFYJ=e#Ex{6F>nvm0pTHC%&Uc77l82VYTc0j!T~7mfidIBFTUBKBxrVg3*(KA*UnteM2BF-kJ6j%YpE#KT3SY^yqwXm<R14It!(prX+V070&Mc(j?h>4g-@vB|Uk_@wZpm!&Ya91lOgI|BMlj+y$f4RN>CfuqtV+DIe%Whxe6^uEfg;5wJki8Xf{W6mkqZYml2~V)^r_8jv`+y2`6!8MN!B9J(>JVTe(q3+cz2VFpofA9)`tx1*F2Gz)JaGi<0Td4rq*MB#h{6QJ>NxQQPcsfh4_ktv#WT!Kz=|DSBp$-gnPObblUvPZParqdKRya^*~$>?cCx>c9V&sNl3PhD*%aTEC1yiC<(6_>!Q~msN1-fFYpwc>dB82px%OTzX&%xJc`6c$9_A8J6!ifVONGcOnamdiYSk+rvy^%EQ#TNdOo!Mg*SUJ-=#0vmA>ltcER;=q$E0VWwGdd|3j+3yGpaAJZ*+JqW}~fG$|Gx-P7n-kQ59knt`X6anea|dfrjH$!Dl-{6`@n0jftlRJXqRSsGS$oL7rT<ZY@p^eg8vyC*?z+_NH`zkNb*>Ll9fMoy!@`8ZX2L<eb5RpT$!RAHRom=nSkb*Y&c34R%C;99vH5B!FTV0-vec)Me(cY2O<AD8LW+E3Dq7n9qNaxm{?EjE($_o<zCMK6LgTBu0}V!WN;2Csz9q<*s|3V{N0mHLfL7haWtQ{ZsIS$_P|T_qFjq@6PpD!1}tib|jMZQzk|3&-v;lf-)+vT`UD7wMmzu6r~Kmq83b5BS*529E2yZE9Y{JU-|g?dnKAS$3aF8tK<B`SekH%4QXDBgPR#q(YHSp=HI^){R81!g-k}p2Q0_d<(1mxL6oq$)j7R6e4^3Tlkzu6H?aQ?lv6Jh{+X15KrHrne;NUT0!lStk<^|zonmLGFN!Z%G!ZxAkN2AnsP7@i1MK`2*T8|TL^uvK+@NKZKtkzpLC7^*Cv88S8$#3T95hcyj`o<u{wVbGOUw$AwZ|4&8qyLJw#R5JcxMD>urn6-_eE*BQGCt$5_NpgU2P~I)jstWK86kFgf4GA%zi!8L25ueYO~s6JK>f-P<3R7c?_Yph}7ewy8-E4i4qOD6_q59p`vC$-)VCIj-H5hsT?7m!K@tWHeqKEA+~i2G5j_He(p|G#&R989^6e5Dt$P=GdQVhm--8yTCLno?*d<8F0w24FR0u`1P?dSI#RITCz;OGmNy?Up_+1zwaKXvA)u()sTQ#*BI#o7#0N-Wu=_4v7QUEpolz27VxUy+iHI;?D~;pfn|iU3;#(N*%ZRpNV8bXn3yhg5hBPK19Q0`!Y9=?lZ-^pdLsaN9(J``ylIB^!NL8Ldl_`yxMPxfyz3M#-$J(rf8+RUQO+=D=b<Ev(r}NQamQCe)%!NGeG8}j1Wdo<KYWHHK!gI+uP2F<)y9%#bv8u7OKMc2p-cT5rmQd{tcu7VUe9cjtv}8i@_{~FLr2K7wJ;tRCOk+kE#Q5Hcj1>knpnG-1Q%BwqBc#7YthuI*lB57bzh!WBHfFzE?|UHGK+}gg1TnzTei4svhb*FVusHmE8K*#&J$y$g@{$fDtJT6Iw11U4E+3nDigo+iK+d}`CXIe)ytorboC-+pZ2+PsaqQ0?5cnfLkF<1?;wF3oD{okSM7#PD=)!bj8SG(3;aeO0Q+c2CKdQ9dXverCXGsd-N_>b28IuZ<C~qj@EEs%?E<d)qRHqZMzKOddkCSeu8PCedsL#=woZ2#E;^g!a`cIlt@J<}#V&6Zm0H$7G(xg|-9a4Y;iuGPEp;H$-@jRp^n73}A+C^#oy2@HBG$=0mgPXFoqw4e>NAQ;QEjisa8M}dD6Z@DHr%(}HQ%A|-W!l;0M6rMk3cC{$%eARE1bnMxD&_d%`aadR)cc^3$=t$x@fk?2s6U;7$|Q;qXUM1Ho%KjKOxb-)zHasw?;M7%=>QG<eUJ4C|_}CodY2IHFralZE@H{7?gh8X+(@)+b+3RE|=Z)+EDGy&wTeJ&5xneFaA#bFs4XySHYR!2wU;_l&z=Kr)<x*=Js0$!WJuq8G9Mjthln!tuPeNxZl;zm={TW@h<XHd61##9p(F@{HOm@oDn?&PkQ3@0~Ky*DVGBhN^f%z=q0wXg%R_({CY-MdP@=54+(9eHvT*`vw5Yrt2=Lz-brF4O%#&dF!)E7V={g)Uj!2y+CKgNu=m`Si6IkhUoR_$+8W~18Z7n~`I8Z4U!rIQIKNBWk)4c6vsqPaI@0N_z0)U}|E!B<jLPSLQIzD<;LTRw;vx;v1+j@lQ8Q;x#-QGOt6j?H#BY-Q(L@qvqDT6xZ3P!mXYwPb?s?ieInmdNG*8wr9eVJCAhAg?_3kX0+t^<+*#Tyz5w6M!1%o_KCs_9nYd=gesm8*Lu~W^<izGXc8fB8=3gKYMadUmEtH>QLRZP$@-5@v;D`Fsfy}lI18Cy>PdY{tJ%WwI{0nZG*&g*7pK$OyIcjbX(iNx@ABvM9|`e~TZ4GbC<AtV~N&+dpeL~0&R@^yoOINc`X(5LU>tPWgog9axpE-a<qcceQ{^&Sg~1KhmUaejBa-21}W5cR=h>9y)Kss9NYetB5%jU~ye$+3<DZ__MZ=GrS+IKV);(K~~-QrJbHJVkDfMz(a{GvMd{e>L((S7Zy#+aw8M(m5(0QLO`BP0z970vlcghk$E_sR8x@{@VyG3p_R&&FxhdgZ)OXJ9B2^#ONzd567r*i(>xl@0aC#3yVKAOi+tt(MnvhX(C5$z@BlrVbjg9s(tb=9|6})Ej25)T}1F~uLTlvlu}IDk)~4&77-tUTc0v{|06X$|E0gyuwitTAkBXHf~ptxvphcN*e^EA1MRRCtf+J{DG<;Wpn+tUk^wo=7sP<qkKT1q(Z9ARP!0>wre${k5K+)U%*0h5UlE9|;20cHsxVtE4DdR2;jG0Nq?(N=P(wJv$c!3~9~VHoqIZH+e=>ujB^eo6VSOyVA{_U0SGugd3&r<_QL(qHM19{(Nu=uzCZABWIRe7WFq#}aCUvR!Wd0i4%k%(Ww-+}20v=&Qne34Qa74S(!dwm8z9dbAOq5CvybU=s^i*T|r2rMb$X~S@rVFHE>T9mI?T60a%>I+=ez18K1ozsHQAk4>c#Kq2p_Wf>DI$8BS(Wxl1g8qb02g6CO}*}`hObS=%u~wa7~H;%UVAPZp2+^Abl`M%56K;U4n@M$yOHgI!iL3zbca0sHeBEWk>4*{5coil?O7JA4a&+c6;4@4?Mz?Uu0pvb4%~uy(AF%Z4Y&MbLD`V`*OcH<7bM&lDg1*tYHCQ71K9xA$?-3|u>M-GeP9jd$z1Da)w6tLHp|{b=TzK~(EmF&RYx)%B!7@*d`7Mm{t!luPp&DxOgsJL1pHve22g!PuZ1UT0`Swb>I&HW==7i<U728;;9|#O_q1vg@MdUL&m-_*j68iY)j(>DHPxB1{)>w`tlBSwgl7UEI4z5zi!vNM53KVSo|X#zt%!*+FGB4_w3Qs&1jb=@zaggq@NBIzz-Lv=LzZEmkt1C!>QiF-NrKInOak9mN2y2xBb)KxbGr9QlqjmaIM|Q9Nk#%%%dx^coquvY(apJ!VUV01En>l@D$&#aCj|6RE#Y_r(9b=i7F@EXoCRazzdeiN<;}~{M(Gj=&@H43d7Wfx&q_7W|L(zU;({mV2b^Gke}5q@sE<!!WAvuJRhboM{8Cnxjrh<MK_e-Am9WVjSw~{?GEyEfxJw->?P^iEVuka2aXztt04rsms$Cs+g_b?}`(l-%6nNrL{SsU^gIoj&E2N7t0uO#PCby9tnvU`DRKWs>7Zi>PajfRK@VSA#gMUi0u9Xi!wrA7?LX4r}kQ@vm?)St(Xgl9sO7oNbxaIAz+yEpykMkja!U{ymyN3{coS6w@9h_M%upk_yr~DHcFDwho3pS6RDZdYqmJEqc+i9L>Ej`$PH}NI}0Z~O%Y_fLpIthSt(-+IY2AUDanQ(<5_io+CsO50~0$-S4Gj*rq7yyFiW10jNPz&)ZG^-2~rkj2;ijzw4v!5@^rNVmf`{gSzMIsF?2ASF9Y|v=>-Q0~{v6-PUX{hhVe@f#p-a~o7c=@eMSVczjU5$!0^?y?WBs%SSB5V!Xj|&lD2b7Fg_n@x;%<5N3Jz5Ah=-)2{?^AUgIrr9$la#9q@|y9ky+K$#yyk1M>U^d5^Ca5nZs90&it>b-RA|;HxKEVd_!l0Ycyzw!^dTpq8DIV`n~Uc5QGLK(9^0$G+8EbZrcDCBl!IZ>#0cwqwDj&KLqcylByEk&m($89qJyUBO{?_K4`45;Owr;T=mOXbMOf%?&tN~V!j*IlTx33cy<iWU0_FyUnU!*mGA41QKU(<VD{odA1SGBYdzPKvxC@~l9#+CU_CKHdFzL&(kB{sKS$^vK%F9_fmKQMwF6oadZr3VwKXjSXepY^34`%Zb+Er95$sF5UYP()$JL9~+3Ao*5^uT248UXZAtXh~#b51NR?apKz8kxTYlmV@n7P`P06^}V9ufot8_&vu)TD&~SH7G!a<>Mh4tzOftrTmnj<O@krwQ|bDoVQw9%)ieD&FJfA;O%jNhc5AXakzx22WwnOWcR}>Y2jQUe)8;sWbyX3DnX%Gh`x>HTR^=937D-VVSulTWc72P8Uho!UX0zww45ZLONJn=x8%cu^}|O>OJ<lX@bMH48+td`1Klf0hp&JR$BVWcW@A>zlcKN_<Elo4*`N64VD96JDm^!h3LksYE$T~@3y|p9@r0LgwC2<Ok}*vNgKBoH-H?IIeDcwsPO`ZP&G>?$-mx0HU<2aUhJovV|D3bYLz9WIu*lvST6IU9NteM3uNdz+z<d40%b;CLC}Kg+ENIZwTh`LEql6z4K=PHh!)g2xV!<=O*uOAj!V9ae5OsNRB1;Re<@<WqPmiLU*eXC>mnqMetMD|~t#IAz4@oWyE<M<UtYrizu|ipnXfMX6e7RXrY1~SNk!=_3<A9YcSi#r7XpzGY{<M!!CyjJy2RWbI!Ew&@?~X*9(CBs|4R3Gzwl1R2p(c8`ocyV@Jaw|A%f0L(XN*f^UgYRNoAY~uitL?oQg3<n0h>_0KhO8k<iyYq+<azUAI7ZinNE(CZ%(Zi3<c1C$bF8eL?h3<m>>}arZ!sXEC=1slO6NeQoVVCQc;0KKxZFx72j{_6@7@Q19~URj<WX^Awz91%oDB2_`1&lc773Esc_Bslq%8A#}l3G-D_t0F;S-hx1mmD<$oyN{#DJio;=f22UN`Lhim!n<U^4{-ikgX9tUw3GZUH+@5t|meV)vZcZA&f>z;MVzim&aQL~_|wIn;GrzGOEe-_=obLXfmE2eSU8fAW&6asf5ngPt98aH!a*c{+^JDdFA=b|cI^yD5)xYFWgUJ(|(XY6d6?G56Ue&9WjY1Q<bLxSeTwSXhxzOc*Ynz889Y^YuNVMMDCUyfN*XaLY%h04I8XaNP}A^ewVxFN=ypk4hTcoEW=3_yY29Y=9?94Co;<Jam~!{np#m*4F*ccR~TV@G0J0K&kXxkqfO@i0`vAm_tf{&+c-H{Z{FX$7WfsoULWxb0iTZ`FnnO;>idfW2l`PXxjLUa!hU-DsA-goNJQZ8OH5R@Jcr!(WLJM=2YpwNn5=z@0N_wy;86VqGrU%CZlKI{M7DSOwo(6kB&P&1E&d2}(oxGrtpmbj((@nNJ5v@cF4A&LmtYZGVs%r$ZJ_L3#$|*~gQPs8MAbL;ThMbY*AN96UK_v!C>{6E5sKpz)_5h0jurVnQqBrp|lMx~)vI?F>2^zof^@d_G5b{v6kw*=@50QJKw<>h4pcRYi){{0k@i_VosOonkv(AQm7TAzKrSs;^(06`ZXOqpIbM9xq!tKC#{Kic?5N2iWcXEqcl61j0c?;4>|cDb<Y(XTt;A4~-0`84$Afb}<82HTOVGc}v6Qlh@!|CS}@Mg6L<U@r=bhS#yp#Mdi^B(J42+-Cjb`)Cv!NT?l8YmnBA$-*N9Y(<WY?YRw2_0<En6mdi+=;+r?W25RmpDq%`^xtYIL;Ei>SaMLx(na5t_ki>#G%~;BE?CQ{C#kCA^5IUR<KwKS9ZNK49G^xk}#cQMp5^&r*l0&(l;kP2(b0$J^Qzhk9sIU&mrV=oaxI`8zm+G}fL!5$X7UABnb0hgP@l31n>*p3wi9f(M_4A{wXm<6A6-=!oC9Y=+ZX%v}%=t<PIsP=Ap)KM;aYVNOOX?yEab38iHHjCx(LB?c$hkBD3`3TpaWVHHeTlaWROe$iP}}v$P27KkzBDn-{5Y2IHi%L93deadPXXBqhP<aj+3Vk*nJn80u>g?+6yxmwJPCL|QDy=B>Nyz-RfdxTiX^cbluBVd)KDR^p|R%TO~%ck_T}2vLLHLw+$ZS`?{vm1ebUO5-3~imF-WSzB6uNcNIXO_vIIeiOV{!DpCTEjpRWIBWxPs$YccONeQ#HM`;Bh0xaQUnY^&3bvv&VvYlSgI@NqmLk7U7v^Va9%G%{i9!yVr)kdp41>iDA2@+xhR`*ZyK2nTuwxoOV~?KEUKGj9a#-{DJyT~oD%eGyIDY?{n&%L?xExhc3Qa&+63XY2={`aYWd8m^w$ISoVFVUzjSqZ``rsEt2Rz<tE(ZC`N!yx55eG(^1GF%?=aYT9M`g;3lke6R^V=tV41%ehLvm`E7it84wqSM*I=ZYDnZ0KEEyX8c37qyu}~i3011$AxX>&EQuympDOE=9pu+KXuwsYNhMkrI`qPTp7hGJ+RDQt%;PcAgb=WcA4ed%rAmpG0i>KP^oKMI|5FxyZ4dj`s07<U>-{_!(wykG1>^QvUmEs1kZ({CM=o6MlwO=#gq;mM=50kVU`9eEXaBxk)W++VTOcH0V$n@px@rkO@AVEaA8YO-B6LAYyR>U=)WjR6fX7F&mXy!XuRv&Cn5XLi57?OEiBdL@mhuD7*p)90U?Yhu)&RrU?PHvGLB9-{bix!h|bV9Qm2ND^xTbZ@9&^8`E(>r=Bb6c@?VApl>+AfkA_ay!b-wPJLuEBF}xKW;Do9=P0`Yw|ADRH)srMVmOlO1Gk3QZTz>J*T{I;$9m5~x5f|m<7*(UIvFVvt8cdNWnj=~nR2o4(z>G|vSOS{QaO*?Qi%A(7HDY}mqRrm8kvl|Yh!(BNooj5tQr)?|{>D5~qa1{Gqbv0TOYDg^#^5d-n}c`$<@m3Ot4Rp<8{3+XnhFU^qx2@qb=xp>cm}-i!IOm99x*Zt%l&qXgDno~4h&crwl4&hkGfju2BOoRiu6~nS^ag&!Fz`K47dDj)eMu9(|JIY0EZC&YK8AdE^%9=%IRvXwjUYoZR0(9ck^K2wzKPFiCD6~Ttst8uS(uY8T9I&+@^BV5Y+1fIG>q%<eapGSw~ACTP5C&7NjdD@WgIIjo0kD6<H3RLVKy}TQr9}c%NsE7C{?LW&<poZ#m%Yf899;oz+~=pd(G~Moe(?-gsI+HIFUrRaqJgsWOuD(+gFkv%t@3JYFc*u*5%s?g<j7WN{fibKW6NI#o^O6}<t@{=s^(?!o!Kh71lJYY9j9RhHdRP{Q_G&oy^%{#YL6vMtR*)A_E_WU}KC$6-_+GU3HLT{kKAg`oCYNBeSGUbgerM4M_%4-c-&C&oE;q2l}qU#8j{T5jaqSBP}&CrM2m>3eCe1?sQZ3A>T}>xA_2F`e8BZt3DO)SgUp#7T-;=%hScCNoGagg#qIw$EG@?4DWu$@+fV0_>d#rQP;X+#*-G>9mwU&ShygWR{_ymxNs_{}gjKsLRgn|4x!1Uc}$`8oK=*y4%*VCH`7uDZJ$|vEf8$#^Eu<E`)(qau~uf4D|if@s9^t{{Ne@@|Ta{8cB~vj8|oefrnx3>S~q`z3WIWgM`6yqplWhp3Of$@%*MW!(~N8V{tc(WsD^=3AvogYHk#yw_m2A@~7aWeEO6>X97p)=X&zTha4Se6h9d6xn*fW7fwmB(~LIs1c#Dt-)Za2=^~4N(GSg{H@KF51?eoMq+LE+XCV+3Z<#59w)TM%zE}PepvFrx;(%G0#X*xEwUk_Lku0q}dx*hu*bZvSW=!Phj*RU{w9a}1^~FiPQqW&z-U;1dm|u7c!6AH4{kZZIN@FzZ7{U(tvu3{$A7KNCA)NK4|BjCTZ(%gEgJob%r0{Ji>RdQJrs}+2k`9DEp%%>dm)QY&{~gREpM;^YnfWUfB$KURE*LwNNYK-Udxc_7j$H7<s9*fWuAXUWm8<agoFo<7i^tVDfU>+><p*>2;HHdn#R_zb`ETcH!<tEHfNPS7IiPlR`kqn$y<%lGB@iIkNL%K*cho*0d>)MH%&$ZM65EWgHsP)XtXl8k9$~6FsBQt`9T)^tvk->LDQukO=4sk`@#tp6Q44YTX2~2DPQqfHXONL*tOsR-2;eo$0q4fQLx&8>`_h#R2tA@3@e#Rn-BdgPsT#unmlkn*1GM1DF%&*JGEh#acXB22YESCz1h6zCLS=t2OIEe_KFVc6xq9}LlfLZ9Wt>nu)@B@yraH|y&#%L2#s;A4W{lBP%HVHq6(NjP-W7+m8qH8pRv1IzbxF(MgRkb6*@q!g+LDvNCpnpxqaHG(l;-U?N5Xv9aW2J@IWYvgdtSRKkIIQ*exJ-NGi<n(3r(Pn3YejX(PpRE><fMICY{H0|G&Liird}P%oj*z#hmWCSGDrC8wkqTuWRw8+eX(FA$g$sD76;af~v%YqlXhJeBN$VPG73fsE@veFj##@4~8T4)W&D)iesM~W~-?vF>pdri+i&7)H7dG+CX26z>B4tAHqs4N`Bljo;omv*?A6w#~4`{jNzrpvC(M8imU!rHo!MzVBPb6TXZw|X}*~yr5D$eMh3kw0lY<2x|!cR4+9+syLHOHMu9ai?u~F!%oocx5`@ISHNkc(YaDAXvi`0=Iz;~&6`;CQ4L%xaMT)^!N7Lb6%|jny2<1a`1L;#?+#cv}zbK$151gBLK~+k{VLNkgpSG3wZG{aAYYGYC%XbAp)|!-5HCw$IzcH47S8-CLY>y}SwC$9Xf3v~^pR0WHNlHK_w~R_c0&T$9NnGQ<F~grH^9g0pyY>>~7F!E~vXw^k!b6fdhy^T#^F2{H^st%FZ%<Cg#1`ggI_dLOW_RO<zZg}}*(qM~Kky(WPhE}t?P62mQ<jsd00zs&r~WYe=&<X~W?rb<^q6jFRa^$p=`5bW&h6MrsFE?93N)}TfXPNEVjv>uCv8|Mv?xs#$Pp}0Pj0c8c>@_w0u2R$@^_Ju%yH<g!B9^boE(6f%}qD>@gM+m>h*^jpdzCRr+&K19F<DYb&5w1{TBvfN}g528$*2hJ#v>3v}(S&-Fd+Vinr&M%|4V(RJth|SaGsEzg0$q^+pU*E}uq(@cHx=(<Om92D6)-mK^saB_CY{yRd7mzXCJrIvWB`QFMbcL$|JMf`jr?U&}b!%b%8e__hPmN>M((d6K{=2H|1h;DI^c(t|_NrIA_C{Tyt6B`1xfuU>QFV!LX90EXg-<WxeBP(81_72?-V&`4l=ldu20{~Y0n+WP}q2a#58kcqRTrqVb4ZjvYwD?-w@RfI|q$Sq%j>sFZN5-~-j2TL8mRVj){HDl{xzQN5Wv?!&CUv<@ssZjI6bSIkjP%zwpdSd9k(!ov%IFACV7h*<`tmQ{hXHtexZ!T$i+piBNA8Oih;3@UTBo0Wfo_N~AhgKegZEif1S?Cznq?DeI7Yp`t_g}FMk@A40JcgTXkO>((1<>5XbtB`E`~lGFG5*mpnTfh3eVxdHCIAy|hWdyN|GDV3+ua?Yxf1?GD6Av^onqh6Oou(?uX#d>2^CTk7N5d&CrvB*w3(s5OX9{Jgo>&b%v(T$84$|V>Vfmx`>3%456DV{l+@YlhfB71<6%tL*H?9|^P#OWf)g8LKi2AAd<v_jAIt?pl4*YxyJ3BFPfI<M>faW4;0po35hH-+od>gLI|x^w(a%CNTbCv{{#N*s&ZFD%a)@M~Bj&+`{tu^lmY$OKlF=UjctsEv+dPQuVi`hxzZD%qdK6Sit4Gy)e|Sj8t8#I0u2O4-r9p1ZiX2F4@_!j&cZ(Z3=M2%?VW|>>@w)urEJGsK&*4>e3F*bs$PE(*UH-0HU}~xscbiwPau(wB8M1~2wS}QDv`bxrvH;S;g~k8|_u1U#wfyi}uWkU_u05I!NK~V^eD<l41ID(BMLv<whYqK*RY$td9wLMvoYw1gkRo^x2u+yJ1h6Y5i@Ma0B<X<|@OzxI6^V1ZM@!l2swUd0qY4WP3t%{OAQI7@K)mF(sj=u!$x@=`UuL?lG_iXsx^J;ZbCqy`nRyc&Z(Hx*4;_=)(NJ$-Zel{KN9|IDZ#W_8dwg%F6$!*rhp{K=@p_(1|GDN~)>*Sk)ZGeeE*+SwX{FS^cNZ<P*>AORoF+c?SMt{}T2SO`4MOpaNm!9gVKb6@F5?VTYa#*B$3I=mKML9fY8_9MulU%RyfUSaLYoXFvtu%6k%J8e#S_>ppV>)ikXc0{KUE8lTKk3%<eMdk`MVT@-N3#FoG3i~ZpQ6sUDO2a(>a&cSD}tti<&T?3gdVC8{$~Pm|8Bv=_`q@?rwDCDc-<wTtK}0rR(B%s~*j|{OAuuNhYUd{$f5@8O0Ry*Var5L)Y%g)^Nof^H{WGg`mONC+BHI+6=l)?hw{;IS;Ms3pKbl`T34`QP|Q<W#fE477G^^5et2yoewr{TnA*zs30>~{vN`%40(#!_3?tA|1(zjKLhfuym9v%h#4op(focHXrV}@u&>#L>de#f5vJ;xcde1T&PbbHW?5iPb4=L=py5eajQ$UL{0kb`aUMJ0Ab*9C=o7;K^NPvLo}l#h!a^c~PQ|#KZM|-2L|PGzsSAM^-hpnnX2-;C$z@qgRQ~A2rX~}mCt@zq@{PH1T#9o)S%2^;US)g3cJQ5op_O?-6(#A<EOSbx0B5obym-KlI=32;-OUfPY5<#Lm=j}L&D0Q@1$y?t<XmR-3Jxj9q1aLNC{)zk6=jdsT;jgU9`sIEigdhaV{e3uuY_iE<;^=tHr3<Wp<5DF$MrJ1eLafqNO4;?BRZTO8=Dk<qnI>;-Z`E<lhTwJL9R|G$L*BOhFB6#RXPg}uwN)QDi6K~pR4;k<gr^&D>gD7TVr5Ry`Vj~ortvd+kfb!E!lzAIlW`AKJ5ytEn~A=-xd5m@uo>h#pjG#R=C=BU;oe`EgS-we@umM`HL_Kvk+CX!7k-pq{@r>8RDb&%9v19R1+SOv1P>pwH$%f;dqfhM+(GFdl7fg@~dM$c8N%S>irfnM7C<o*+d)4*{0<u)*&a;3*UY6A|II;Xq~HM6gmNKr(9#0qv3bew1y&G75;-0OHt!_d_!#^yhE4?2O2qpQw{z{4ZY(Q@mgx9DR#8C+A8GWd9EHO>@B=3&xY6wX&HB~m(AF+EqQ{G^p}V|FErRqx^}<#503%Ntig>$|2c!~Sc?}4qAbk=_cM^ZIT7tJ<GdJp8!x$XvbAZGtQ4gJr#u6R$6LIV>iL?3p*sr7$~-O-wOIXSy>hs9x(nrJ0*Zq3cmqHpI6$MSJ3)U-9o2B~963-G5CQ`#*tdvENVXmB<5B5@E<)#p^{%!nbZAqX{oz)(uRVSKYfC0zSx)c*!~c<zcsc!VHE^`x%2$@*;<`t^^zHRyH`y(+A0RNYK8BgDPst-o>d)yH|CN4WoKq`*4JfGg|4Po|y&z@AT-@Wm?_z$z30)m!4OzU)m)SSd2pbdlmX9PXeKkeW@5tG1N%szR3uAY-oo6vJ=9&Aa%pA*kLbxkeSKr^YNbh)WhyeI<Q8(yp+LMi0G_6hONk1f#5bq2a%dI`U5#WVhH;N3e`$olMw*Hzbj^7ockvG>1fIqmtA=*i3F@2zFv6tk=36<9wTdP}cUF-;%!=STIX!1Gm7HZKl9)X$c9m8wC-eIu%{YZffv|hqMw|76a_$R6-O60_OXWwkcVN|1uQ+YpMORGR<0CSDvc4FH@<!cmc+MN_fJ;!%VOIr?Tg8f+mP(7CG#uS9?K`3_eot@28X4d^)Q|!(U{3>Zrzw8@qC7db!qZ+pXS%A%Ypl)!UKmoKszF~U!G6UGY7<<#ZsZW+>@UQhk<J5nT^*wtQ*&&tA(0B8}<LObRnEQYiisUnn8@|}dwA9WceSD`hk*`g&N0GXjZRU8bZ#&b6m$poP-?T)zBx5E?%m-ZM36Md_@1I=hDl$M(gNB={TRqSiiC+v9ZCa~MIwMU<n2oH2YT;Ko5{G_Fotz4-38sVE^JMcIorP+C>P5XM_uIoL0#*?XrL!ig4MQbt{eU>h?_;`|4hhuu8dNQ&H`pDmyJf~99Oj<nRI*qEzhB3BVI{Xlo%ADAS;P?1Zd1unM>KA&&&^_OhxKP?>4ENO!=Nz$2?sQd2EpFE!4TGvWg$?IAaR`zeUCRwA}dwJ4hJ_%Y`F6KQ`7Nw><VSRBN)U#I{ZYqJr)mbSdRt<5Y*E0sYp>&4I+WrS6wXuCuDfm@go}c!UhKM>%ar2b!QQk!Y;Oha_ON}ADvTZAyq9~Tw+&2ftHn15dP)IOT=`@pBwqJdC$m!{0bjPyM%rWC}!IWBsH})RjXzB5HNQ7`T`}1P7}8Et#Y_`3>}b9#4cKCdMGtKuZ@Ydde|;RxVjVA$8p1Uq$8dI>%L;;FH&QGt{6b^e1QtMkh0TB5XYQFdAgZl{*1gqT~DgN_@JKX26(h0dQpYmZJB0N0z3^#+*q_WLG8@PDlQL}AqJCCCF#a89gSIevzPeEo#KYJVe!6{yEoa89<)j;V1%#uEB=C)&XvxQ(<Ve=oAyK=0i2nuev9mtGbx^Dc9wIhXlx<u+R%KaimLQ}_)J`$LiB~!-M~yAx<4e^Lz9_OhvS?C3qCa`9$&75TIZCVX5LHdL#nxO17cN}_N~bKn<yiXoiJng!|x1b60S~GzFA@efk!V*b;Ix)U`0O{cugZXyO>5{TdM~b5BH{>Om9O8zA^To{tixRu+AMHDAqKLKA;MHH`5HuZCG7BgLfm*&!0?~0%ZRPUyD)%cM|5TOE~H|wi=<KgbVo0DD-Iv(-NZ@s!q^ituofXch$$sRfnhK>&`+a#dPv>AMogep6-B;&q~RAI&@3?LJWhB3J0x5?_&^`Pvm|5?FUomK{}fGI6gBA`kM+vE;k$0r4&it-8YF^M$KxBkF(!me$?R8Ct{)-0B=OSnce}5skT{BAa35rSB;ALK!qYZWJSzAD%s^uC5^-^P*v)DH58(xxKRb?_B+3f%(MPjfh4=;_O{3C^yL0Law5v&S>#|x4HkTuAsBmXSXcYy2%p~Nr*UTO=hcAL09-UP9eb7j9g1vA{fm?sX$5II8z;x7fKvqOTxhI@)}DrQ@Wz2UJkbBnBCHr`x$Xx@e<1N1<TBoHhs?aRLg6QZKC;$p^E!1X_ZhsevCxJv?QwYA=tcz{^y;fj3xQ?`)+SC@<HkkDdh4iK-V#y2F{&>)HxHtm&B4;9f2(dP<0yy~7>(Em)0I7R6<#(L*Jz6*8)VQF+a&t%#bHIN<bYxKJfy0Y#p>@u@upl|GT2oO1JIXqtyuZ9sY8-iOghqjXP+<I5*WxRC!ig2qh4^jc4&a<3!hB?4h0JsPXS0RuyHA9^XkPuJnNlRgw4(p_nLgIU4yF!wlJoQ%`Z%RvE|RHCt=Ml?=AXJEB2~N2R#G0kj6(JB?j$<K}^^Es7tk+{1geh3|4eQ;?8`~^y&?A_9geLOCx$V5&j;DqLGgx4u;+Xhl*YMvf!Vh9_6;?E$;FMe8IP>C>CZZ74+n5rdSZmI?AXV#%5zm^HDVjD`J$#8&M;1)gmTcF1&C0B!n4q(ktHM1Ex(813JY@quykaoSckyopEn45Kp=eoX-|_ocn`+Lt(hGKZ05ijm>;oj&G(miOLb2DPHsmgQB%yRaMZ4h9``A_1gKwwtKR_gn{?c9%1b(XX<~#O-YbxYGf8SOf4su!Wt-M`i$R^*5Y5i_<|_D7-J|ggTM7m3o~u%%0&Y8w||_TT1vjl68WhIgvk3&z!DckXLSKb`j?f_I-JJo7L`Zk-9G%aN}c4Wx&Wu^K^(Z@&#m&)Nx(zGIyV`k4R^=e792YN^QwIrBNoYcqNV^hjw!$LYKrUUmg(<oxPh}sTGHsY^D9PRo1GojU!1PUQeCWq0!&(8P5&t+AztPjYqF)0Wy^l1G14Xw^vI_icKjyvwX`mTYMI{Lr`<=e%Ay}D>9!@!O-J}138d8`HLGWKvD*+=AjosmKnVGiq-r-0lYlmkM-Ed6@*1t{1CG|?0u&q*Y){AlhqRRrL45Mzdk6S-?0B!*AidE<2=<)z4Oa#ugI!66Y)gv261kKe0<kE-{9fV_@!ZRsZnf6x;>$h94|Ra^(w)|imFz>6)Uq!8eyc$Zo%dhz;Z2oIiFP}NJpX^ZH^ZMoiJm8iU)>67e*im#Z+zbb8TofgSN@1DtmaSg1I}+xGDdqzzUA~a2I&g1>gx7n8z>smvGX3z*Mlj^vsZ&Q-Y-c#zV3z!l&53MaT#iw((yT=K|!Ucilq#Tn7{@hM2o2_+t+{Wt;&%p8d>b1pDco-o8{RQyz=3v8jd$vjPmx1*zp1D9E?+xMgw|7o?a*(<~AlZb+%nAel%Xjksn;hHpPOClC+y$v@E3=nF;;370ZmiU@Y&n5EnZ#A-Uq`p)y!lR_HC|uY}^l0W@XN7|<Y2$3e)b75N`Bp$ixSA9|+&w05dv3j&WZzWjdB{O9WK1&_kHWVIt;s}|$DOgQ%4F<JMZ+$43qs0AfvIx35WWUjS}Hi_$Z5#wSmzJO?EhH;eh_x}mjZl<vh<)-SC&*rG;nOqwkbaEBlTiAkip}nfC2iq6jYL?&JjV}fKbr9hXXn`-<URF`vDQ#XQXyv^v1UTQ&{xOyJ*4sKo7&{<*Y=<{>PEUN6`Q6TnyQV35&5W~C0}U%Wk{j==V8W7zy@L-0HP?=XKT6sLX)@Wlzia?aJ`*&HYRaO|=s0p9#KK=rTNY`4om$e*3pbT-v2cyT62qHJDG&x}bNsqAk;ee^X^JNn*kZ&!CtW!Viqm`OkAVC;g4^9)3a-2JDDkdscd2Wiw!k!MSvXEftb+&kFfK)2e@#}6I^c8j>$2eZGJ(VEn?6g)tC30CtLN36-nF>Mq8=Z^Zgo3Br@<Mu{b-i+&}_&mFUX)LywyZYFue}BYz57|%Dff4y>lwVfN?voFk%@`ZfQkG1gkU<3Sr?tMQ7Q0a4t6x!r``#W(0>Lx=wtMuh<uEDo!kOqK8&<Rl5a^o%=pC59n#MWiJONorV9+`bj#uDD){vaE9F>6y#B%a(ayQ`8eFN#-<xAfW}@t`Zku^c=^Ag%4zqV!Cf=jr9)j5)%|fR6e{y^9RVL8fEYGIFa7Dv{}2%YM@X5rZg$5uo6CEct=AIvmyqJS$yjvhPmj;-?-ta~zl78EEO1!M2;j8Ebb$mVb!OzVU--rFUSFn?DNxes{r(TAjO4~>2k*1M`qzy9`Z)`Io8@vDcHd%1zO+3LEK)&ab$=eZD<p`Yd2Z@|Zdc#Lc})V;QxmQP2fX8<0#D+shnB0iSG{hZ0f6?3hCjNBzS!tlun|uYqQ+}>B-6+I7^yK?8;Z^?Q6q6z*6WDwIG$w_RHI2CouuKkKvUvt)`Xu5Y7a!ocdRa}y2<&wT{1BIqQCv0kga)dz`6rE_aO<SrkMj%-p9cYQr8Wg6c^cPo3VWhk6?xVH_(v|N05H+ON~#1%n|&~{<COZ`sXE{e6be$;}Vv<8{3{2a=?qRyTTQIX4bToby4MuPEk6Y`<p#Q*nyn>Q&?m|VY0WAoep^}GqQyrYSv1KH4?eS50SzoV(a~+<ilWe2|V$fGFgp%s7Ncd!Tsf!6Z$1rX&Sr^Q-CfWXTQBX_qPx!C}P*SwNY|5ewcA+nRmb~2@Natp+_%XiQL4=t_=x|71&g5l%)h7;c6B>yN*${AAS8p&nDP&w}rBt7`zHnYE_v1DA`N5u=mKNl8vao;2vc}-zAn21f5H0P649KR~ss@8G0DNf?fvLJet<7DV5re6%pi`B8RBKEvd0WfYi1BztUw?2yF(#2G$g~F*9ndT~@C!chT)pTgoIWdLdU$Kl9AK*B_(CwS^_q8h@@IBlD(D2_PF#hII~b;95kA=*Iuk4uS?=JtTrxjY(Sq<F)BM$;NN>nz+FW5&J=X$%PEFCO7+qRyD*SQlQF96^rW8OPdq{F5sE^NM{a~zHfV{$&P*g-h29fe89@H)bZ~ek|MF6(?tL6+GNvHGYS+|=vN9+E{C7>C_O0SE7JZyX3#3*Zu)20-r!DdTXECAc5L%@72FLI4$@j#=HJ0Cj$z;*sH_1-Wf5gXL%i4qLO$QNGA8QekrpO9;3K8#;`X`i>SyM@kqEFrkWUGjo)yyE7&)ZE`18;{%&0&hw2LZLKmkjDa{N&929S+D01d_z2;+5ZYX>NuzT$-URHaRt1SfBG@ws~$S=<M)%bN{3h(y^YU(w3jXe;BChy^wQ=tBCaD}ptk(7r2`#J4%HEM-PX6#&SNY~$v!c-0SLE_eL8Xbpkd=sXHdT-iLLNOv{W(sd=pLyLKV926v$7+s?yyu<X`U+Rc6VCDmUcK)BF@~FB8996FCFp*Hqi<<7O*u&w3slOb{Xj5^!U@s&r?+W%*HO;gAzIBkqUs>P8xlrC^k}`@9#ZR@WccphdbqDdul3F&FS7J|fP0a0dJL8Q8TXb_O&}uxbyD<R%>4bMCO3dFkhXx-bp#l}kek@mY&m`6Ufn56#$bx6g){mdjga%B8hI(fM8N<>`X%4x6cL#+2Q^it08IR0K?+5&UqHRo@zxMjfPlC({{gO0%L7lSL6IgGEZD!iuekW;6Xmn;qFBlwO4x7q;6(l4l=m<K`J+mkeN~6wU(Yr5!)J*883TErz%_d2U8SSh_$*QfbovQsw(R9Tf0WQj)0&<uL_V0GSG5)#%qt`AwCSD|l6B12ClV%}}QFhSmCzbFr%%Om)LgPK+E+YQY16}KjQ&et}&h7_T!8m`?flzJ}T&RwyG4+X~_UFx?+Fy#yNUay?%rX;Y9<h+-8}ciGD>6Loj^9Y7L;S9~iIQ@t5Dqw+%*U(V?%*(}W`<>S!Tua6x6aP*8!i)f{x79pqj$J=NhR^ts+4@sLxp`*M?ZCvcH-!@92{Wn*IsnvS0OHxJHH{#5GqS@7&kdVD0X~U?Q8CPxYxN8h}*72B&ClcR!?#aU<H}U){~O?VX_{c-J}`%w4`!H+-u>1MDYQJd%2`=1&?1*&g+)ocmc0hmu#}_*IolrRCt0ons)S`te;NRm>fo+{B3iKUJwS^kZwpky|)gjyd!13F<vN>O~;1tQ|mM<Dah@kn4Y^C5^9K?N-s_gD_AEgpbP)T{9cL5GVvnyCx}~ix2BA!kTwY)xbS2=9fRq-aDF*Yb-qkDR<p{hjT3R#eMzvKpnb`cm*#!?8lGFcqHN8R<}uKx9}togQXbGF$PH?y48rgfhE9m&<^Y44`_`m7Z?8xcfcoo%Ef_br4wx~O*Ct54I|GG!D17Jyx3N!9eF#VrZ=Xm6?5L%v108#fV}HWo2X{Co<ciLpthUu^N!=lIs8va~@AzD4vGxj>F}H`d2H{kHe60O{2KA|TV28u^hZp)h)a8!(W_hNhmd`vDEvgU#XXo-pEWY+bPR>#K<NA`UqHJ;4?;Ax<Tuv~pPZC7pb6zpTTOE4Wa!ruw)uZW(m{p*UnQn^nFT@L+uspF*4k#V2@q2<8H+z4^Wghz%&+UskY%_qb_4Ap*Khf1jTGdsr>TIlWg6#)ctY4j0f9Cnu+3b&oI&fxuP|=K_x{vCIKf@aq;r(Y|N0qT17o(%*rze=A1*V|M{Qo1(B;aObf{jV=Aw25~#Ia@?5q}unf@o938bHE5O0X32{^y#X!7cC^nnm%d=A4gAt@2yOR;0q5by1QqT7*Z#t0+IyxT_*LqihSfxoQ!e^tYUf`Y|ec+zS)w$4T|lifRF=2Xq$53Qn)!m$7(qV<Jhj-fzh5AP#=hUN562cp!F+%TpBXQHa^*H?D~#3qW*Y`t;*@gD~dUh_`7R9Nag3UW+>NL))N8q}-(_A%xwb(W1A7V%mg|D@}4x>bY5dpoq|HwT>!wR48eb?QG8Ur29t^=vQdbO9MokS3P5nprkt!N&%mGKPy2mnxI#gm>myrU$9}CxivPtuR5nTG7FCx{TF;3>YX{ODG~eaY*WKfj^?HINrJ-%XPmOyY{k8k1C4z8dKODYK0|H~P{q(d|20_w0@wL04GG4n`5R3$SS)YYS>mJpCp4o4F~~kkl+ysMi>`M9)V1KvD`gU!8iCr|0&mLi8Eu7s#{WBmYz^vQ!+)NP*TA<l&YRDGuA+iuNU!&s*1|qQU$*7mt#iA#)?920%g&~0S#kDLh#{!xMxS!vpkdE|MJJ1)w?B`fRUy$xsne6fi<pbb!kZYM3>%3?CQ*OviL5Mt09D+f^`2kFX3}i%j>k$~K1#?odm3S{duhPO<S$iwmay3nP9{JYP69!^srte^lX;qGq68)`9-)O54^U;}j3TSGo5Hz$qx}3fvdwDYEZk5vxxuRBEX8zHG6FZ>4k1mwX3N+R4t;w37AHvW<;J%#M=$%5r+a`brsX}X)kC-<({)I|nWqR~ky(AfHya^Hb$$rWy(=u|AcwDTxG^8+$Z-eGThq^UHxmTYVlRfZAH^~1EwIu<Xp6$xBXccLHQMSn5CPA@Wv0%YM87irdw5`YS71=qfk3kJ#IrTm&kf*ZJTlDGAYOBFq4nRoz{In}8#b+ZWuXM`ZwfACoC@^nmf1eP;fAPT@~e1~Q|Z^0bBd%Zs*AXyheM*ZXUcd0ZS^uydwHeBvY5OSIrR+IpnQvZnApSv2$;YfdWF{EQEI~|DnbXVxuMGhMIsu%8b^CLFn;*5)?859$6!YKXoiRLl5Fs{)9FlDd&8|rdHzO?E+*h9#{`j>agygblT7_aBj%4EcaIv<MVCeP1KWRh@BMi!wDP6d=^dPHAYQ&Oy%MbVkENk+SQPbl8TOfQd_V-yEI&GKUa4VdL`4fF&`T=E7dOh|zro<}5j7zx(#ecF%M|-a!SjciN}N%Au}6k_G+-uhqx!fU1mGY-fD-sx0?oPv5TxE4jo~bvP5OIwWZm8{lt$K|DJVFIvKEV-#2fc5R))R)sk1yxeKViwbRhCR<EhM%F-1FkZorw4>&UI71mxRC2w=0rJRqAsJPGQqS0N7reYg_Mr3a?!|EM#dyN>Ek?jhG5eEFulQk>wrKqfgD#U4HCEggVRmMchS{B-D00_Z?N#TpkV2Gz71W1mZWpqia_O%XN!B$C`dKx^3L7IfdBmAP5pQJL_*R5oCt@d>>-u7%c^ax)6U3?VWzv|%t_TDi>pqRBS$oMdihI%R{tgAjfr9D*pj%Ley%EW@HY&0@$9KUkNWWLbKR$R~S`G^yv6Wk4r@#%CGyW*m^S9VqCUKSR5-bC2}%mSVRv;2Wbro?w<)aa6KKrMjn-5<~)^kT`Fg^v8u0Juw`7pA<BjH3bu1QgXPr0h(&e+k!Vp=U~DSW(d?A_=nPBA)~xv*T;xzYGh?oFtTrhPW5#-Xr<1%SEG>hlvlWP8&Il8L0{HBbef>mDQidi(BD6~p|yf#{L^X15tq1;i0h+?ChKYFZnuQmEL5cgS1<_~R7V&@A`}yFfFcU6wzgKdc3bba4yU+%jUG0okX%5WzUrpH@1<XY5F?K~svx$vZ;Yu`v|3Jb4CI6}Y{<8?*B?7iWkbO9_hpqAQk{e=j=THHk%|%@u)(g{dFq1x>Q7&(>p=OSVknBq^OtO?T1(<lo^UPY+(?NAEMK{@q5po~YdTVUtLA6ovJQFRb4_Lh74rfqS<^ZGi{XO7eH@ebq6F4Q@P#XW8OiajEHktMJ$j57Bj~UMkon9G6p=Fh%2hHS8Cs+>;I+8ZT@?y<cQlE|;aC<5@bUz)sq#0c?1orrX8en81~jm01R~u`2tM@^j~F3yh;wDVV4fKER{uxgKD#Hr)GDr>3O;8>5IY#I{?NrC6w|aw;(T_`iKr(!{gpi)J<)PWR<Ewy%k45wXj1clB0cdGX7|im5e^+&pI>7;-UJ(s)_UV{W5$yesabs>i4jddY6TekvZrTkTdTzl`@(;n<E{e|{O7g~MIkoYjZO+CTWg~kw`pI*DBx=;k%jUE?*=JSAR!L=&m!7JzHVM1WfrL6KPFz}KUmMLssE7I)2;a$#)=T1q#?j@n8#C{q?p*hE}kB!>&QqAMdD}#rFR(CL{*u-ewl9~`lYQdAOCF|(;^U2SZ@-NT^Z>f8vA=h?NF{9K8N9T++n<z=L{e4Rt&7lfg5^W%WFbrg+xXtvDT~lF*{gVM4>;gfgR|J*rnddJH5*(?;<K)o#GK}ZE_dip8i3rE5&(-st(@skO{65jp<)tLBkh|5ClM)t29$wJ_G_1fF2lbzP2Iq<w8N-7B7qnlck;b>vPvg+<LBdq$b%<n#XA8C(kB$J$J(WZ5Kp1-PI%NeFpCrz$8ns?SxNPJYrbVaOv14W@g_kl1XA!+u24th1X_8F6qP<g116DL`1^sW_^|0R7^9%*f_N(-?~B}YzACk(iHvtT$1jVr8QdH(9$FriRmdT=DI1e!uD?C!mmnVOp~=vt}+@&q7=tYlN0w%8%5!emjkDB=H6G+ZAwWPial-Nfy7at=cI(}(ICKze7&XE^sWHI-+n%xke97>8<rqkf+D7`rzJhf0}*^nZ+J2`wv)`9^}YDpuKUz5s+v{b{sbk<^gTKNvBpZ>g9g>--{a~uhpvPp_BNIo1O)NMOqGb5&q#FmshG$xpekxO#XBjBWC=x~;#?;#SD|X~DkaHpNm3B~xyK-WYe#(#lJ(jC;R~Wgrmy51spH@&gfp1PqJ8sgZg;wtUJOtLP57lcOQ2<;(W_(^V^1~-Nq{Vx@W%S{FW}yz+v<A~u{gOvq3v3JY1^P6=k))z&xCwz3e9*Ma?W|*@|o_2T24j<SJC7+#oJp~5Dj67K9w}l&vP3T9l}91$M6!^6?zd2BV;{~v=1c;%JoG)FOv5RVlSOWJo$HUg{@DXJc0`8UaUxMfC|H=m6OOxP!{psgE3sKPU%YH;B&0m0i)pF2{)8c34utz5Xs*WaXl$#*{+Q47-HP5*Y}V!J3aHkO&2eK&yd$1+oaIx>POs(KQD=@L5k&~6gGl`+#EoN4^~1}oKIAx7|akJ0~E@LJ=YZ^<3{CbCU4`f$})cvXLDTMPD#lQY{I;Kpf!t%^;utfS1p$T_0HnuyLq88M~u~0D=&jiir^>ek}$I;y&qoI3|SE-7sW}$X!;>X<wckPo_m#U<||6Bg_R3b`(>rRc7`WER%hi+I|w1yG*O$6DvA_Cl4Vc4O<FpP*udJX$vEzQg8r^^-DVPb-Lnn<?nA&dl&={p^1WU7H@HGVu){vtk?K>K@0NNp>Vf+mmoJa{l~^b05p0=KtZb<kDok}S)?|~f95YxTmgpP{nL`LCh;yCvsg=)(z^{4*83Xu2$TOGDfTVgOFw@%WztiWJFMDcpz=wEnO$NRyp6~%5IO8^b?n=_@RN1WF>sn7n#za^3ATyM^KY_dWenhvXgdyLKTJQ<29aU)SoI1o)6mLM>iPA5MksLi{b1mb?)<Uc|Fkjw%d-kat`8RysV;DL`ne4}5EiO4*`P)tPm(aa{g>#dVzOH{=FFDvF5!g0I;)@S>V6;5o%cc>1WB&!O_{6NKv?a&>e_Uv|FvLQN==|J57uDu%cy1AtOY!-aL;L%BVT<Q+oHON7yBz`s9l_h=z^ATqe2BED+up(;6<ZOBmAZYY&D9aw2L1$ZWxFvRmv6L<kINsmnl7YU%zz_bHqvC0+qtn?j^Y`Yc7RaXU&bBwk&ovYW0zv;DZM9yz8Wp}ZBJV=12;G*uqv{<Octb8LoW00SM(U96s6t)aGwCux%Unz$1)m&9_tMtF2zh*Xl-*UfGb~bAA7C54v?d|w$9+u#3yS+Bs(o}3j<D;m@#o8P~fzma8#8d7Z&frt#s)hqacopN9g-vrau)mR?*-!Wp)jli#y?nR^M+Ckf$5%vYu{DfrStQin}QbMxs1e(2g+JaMj|u_bc)q;sh5ZE<2spJ<nng)j}U5IBD!kk%qd`=S#2$^CyaR6=3y_-L2l|qI{9v)<}3aZTz{gL73E85Cwj&u}%3mlJGR_9Y2$92(eScoMaMdl=fc!oLoilqT+QBOF4<7(SF9~=@KJmV|1^z&`R5j_+T;FoF#0^Cr7{>m$7}rruap!7BM?HpF9~y(&!2aEVv0Ds2N?#4^Z{M=^0%4?C$*<U{D|iSiDAcptOtb<>^SDH_p&ub+E;(`z2#P49B~Czp(RTRqi`=j!&H6Cqz+4Qfo~HX9cs<>nITY`xl?AV=9_9KuDna*c}=y84BLliwJbiJ^B&+Ug1<d93^YfqtB2y1CVZD(~``_x8*T7<WnXM$cG3&1h+yk`O0P(>F}En7<%m{GVYSxqplmKrr%n9!Y)}Fggi=AsCUytl&txErJM-gZ!%`UzU1?eu*)?;i{N(uA3GECjE5FkBN{CVBQ`rmEgXR>(VIxFXk@IW+MlG&oPVKHXnFPQko^+_82?ZgGV~PG>ltrSSVtv^U(AlKBnHUnfGe&c?x;77c6Oh1J#T|MjQJ561-|W*>0r&f$?;wXYTBpZJ^&nTvWIa8mLmN^Hs!^z_UVKW2HXY2Y7KH-&#_j5qpBVx`JMYEXJwbriHrx!bZScn9PTcts_o2}Zqfs{Yo<|j{Z?Eq_*FbPgfBC2a`qYl$87T^H*BnFym++mjQDZdYNrf`C^%^D?ld{@Nbtx?0i7WH?G%LlQ_cbJ;Ii%&X1z~6h?1uk20ZZGK@*!eK3uUwk^Hw3WsVcK)6eXO%Cx>N4Y;S8o>2C!P|hk7I0^oA74Ntw!(y@cs^d)y$U_j69x^ODGPY7VHmJ}^ZmB1xxw|=1gmndl-#1R?fA0vR<pa8;1jL{<{gEtejRH1hc0Dq&^gVWjJ;Yc`(Co#yP0LB_eZK?qPyy<3%Una1&SxOWMMpn$33~AaLVX!Zod@+l;#q8Rlxrr4c@;h%z5nbb35z!0{J^OFi1OqH-SQ331dE4Eec8P8CliyXJV*$=F$?IjDY3fQOJ&C^>jVo$q7@M}GA8NIjO@}<lYgQ$V@-aV$BEu_=iyaLC`Bg7%`07gjyqaxOxSw|%$S1xL50Sg7cvuJ$mJ}%4&((ks`Ty03K?A~16;18&yWBbNlmB?mf|AG8*JSA7|H9l5tZKh;IDt-s+hk%^n1SRdz|ee3|N{Ll_kFqs(Rok=Ry*57|~*RWh1_lsZ}lv-YisG!W>$-lXs;Flxz01uFVGtGT{MB3VRfe+^c!)42qBZ^Z>|jy}Fdmv&G1pjZ)<Hqik$W1ZnS1&2<PH<aNCVaaP`x*Wo&xgND_iFaVy+`-Rr07ul*(ZCKv;a_jPvn63{QPCECjXE5;jt<1|3;j{^t>J2&KnW@Kp-)YYL=BY%RE9!^eAV@oIP3pBx+4+G<+oUBhztb9-`-u;i)w}qZ2QZdLv9;q36%)6ZDH^TMU!MQ3NCx(GBt!@x8Ds-P0JP`cL?i#5ed=lsMhA}a)=@d#T~hJ(ymoK4(s4=9ptzd2M?<NxAR>rn@gbsy4|O3dM){mdsVvnYbaxUWQOy#fV3EK_{6j8+1tR(Vq7z$B4z{QvZHIjp&;@G9VpBN20I-%2`abWSMTmJv;lL)$e^(IY#{@(|#-SBErX*coPz?#xW{Ff&+!-PM4-tr^EF#k;A%i)JmU14qda`{mC-KPcsWd<X7P4t;Y4o^_b=+QtTT_Ih*OJ5B6zLawFz!xSreh3p-C|hXiGlZs9;c+i%^^~Jn<iKAw<%C0oX{auyPK@y#Eic0<ts!q=IpWvTyS(HjSYlcu6^A5<|tgY?<zW!_Nx}d11uauN;FsNyQkqZ&aB(2NO3LK_FOw16B-Qd*n6_Z)|V5v>8z-wle63Su5D5?sJuY*PMy5Tyf_vx?*{vhM%|TNiLBJdO7jHqNpoZoAd_=q=1*f>BHbbV=aVXW4G=G`G4kY76t-k~9UFeGP<F+?YtFnhS>6L(;ecVSOC{c=Lb>^)*{dbA+KCC&bV1X1>FUM?*fiX>STJUc;DT|tR<fE-gr;X~$-6uuEzR)nNwyrTNtxVZ?z4tT3QO+Xm(l8w50BQP-DE5}5Aa3?4Nth8lvr(!m$X6+gFqUXIM+Cd9c7%gJ=q{`A2gxcYv6g$sI87>%TL6O$jz(*RJgBq$JbvdX0$k>IB-??8oUMy)Wk2@pA1hz2%&#(sgIMN<CMMG4<DiylCcoXbN*`tqSCo6<#OL;^<Wn;z(@^oe|X@W>&WdIz?7#IZRJ^U@L%DJDE`@D$YMq(8M*>zhq-c4_%sxILj#VrZU7Rn&M+Ea7J-uQTsy-1UDq|A(*J24-Lf2P)GEe`D7PG?ZW?#B{4tH*i?2F`@Ypz0_o#cYX*>jB=sE75xy8?ewMSSFN#>HMj|wK%w(6|Ux)t`kUswh}Z^E*KB;WYGSYY}o)6Ah>s<Mx}$#&-kV^@+i7#;=}O-U~Uf|>~<LnH8%Ftv-RN%mc)kcZ$7z6U?b{gA@R0|IafvUvbP!<IFiV>MUTd%xz#{EV5|u8}C26Okc`VlF;wnoF+yZ_4=kB(U89^8;P<iY`e8c&$G*U5VDxw3H4d-ZUirCgM9ze+iGScSr$~{&$+@2B3L-evf5k09!DI%ZWA$W3SniR8F~^7~@k}OC4QEF~6Q_F{J66brq8B+jQ)tOF8tE_?k=9c&<a+Iw3(j>UWY-C$e*^vb{^~aVzS)EM5i|5aETA>VQHX8KLi3;gg*qn_~i&NfUy}b{Z?v)bm^|jJOLPKwOa7oG^|Eqq4fJkvQ!bO-G*_(5S3|cjHlr{0jw3gfv}TnX0ObpKOUEIu0qi=N`Skdy0~@7C%Al*?uH#3a`G=ybWUj^l)Oob{;nZ0t8(oCgM~qkdwCqERyT0bN!li3ccc=B5_doCG*P-s2ywZS;2$C(@GHxu)h#g1hD$e$owYSx8jU#xLti>Jb&Qnqxs@vi^i8Ehn)4YX_TyHdFpyU(U-zk^b1%Wt8N_ZzrX&+zeEHUAY;X^dYnUYr!pj=sVMg1qfzS>sEDLnqzPo-Gk|SRS&HVjVWkI|W`*!2vAhkTyCg5?XD0bS7MxM@L=RSR{sM8WAK&TQoA;t%08emIZPE$)mY?`LN(e83A{~yNrkP@7Twm<~o2Jx9!|bj{(3&{x6<mUr#Kp%l=l8rMJL?1!$2Ir%en=h*gTIt#dk<}jN6vVIZ=gR_9RjfZH2fTaP5-T_T7D6S1z7~p#AO0mdjGFI7r=2mB4r?F;;;PdQzU#*P*qc1N2aoI_88#%9fkMzw*7>&fV@2wrXB&VgUuMtm58ecf|-trCfjHD40s8A*c6RaZ>_wmgCGdYHQ1{c2?8{&luuPgw=~qrq8$KL7pTKK(tUt!R?VNB+|ZX_rjian8gwbWY5uL4w;ZdFvpl%0{5}Co833z`MS!W|wjLNG1kB;GzM{jMHlFFWuRfd%3M9DU-HRlKD*#oL#hy|sSiSRjEdIk=NVx~IdIH30#xR%P`AAJZue%4TJzG0O4blU;gvm^p537sk@bG<EnEixJuUyF=GKq5h#c@>vQL&W&JTLvsG0NQln;V|M{(j~{(@caLzt(<&F5D0t4;qzLN9MrPZVUD0_lpI|)Kn0{^b?E=DG{L{Ccqtr{dvExr{wgdnM8KpnWh%>HUD`u6aFIcu7O-7z1sv@TxvZaorr&J9&R!I<gQ+TgSl$Z3)96mxF$3W@a5#sy+mVX#|LJ=oU(gAwZpZ%y0rGxkhXbmc(wiB(DR2tl8(kD8}+Y()OXJ5>Pku$)b~n5ndHB8m7B;-zENoKxPjhTlh5BT!^BR9y-|D=C)T3~;y+Iw>6^^yf=}s?0&y@@>@|%fDb=X(HONEdzBtjNRibHjQv4!?evAonxFlh0HxpJT21o(%9xj4gif*M_9zteENqTOXtqz<{-@h1WGfg_ge06~fF%W58=~AmTAgNwhU0k*n1>o~dDN%D^B6ii%l;yY1rF9MB;3~m};L#GJ`U%1pkrWZ<Yy=Cw<;e?<+12jw{5|6S8!ndg7`0%ty=efHy%Lch10lEwYR?~ANtkGV8JPK&7OXO;6K5~qtTrP9(Ml3-`+VmfcZL^KX<(DqbQj6pPogDONu@L?b-DO}aA5kM1H|_+*KP8}Wwg^r-(FCwqHtb!^UON2bV@RYvtfy?4lM&dp|2LaZBxhzrP&cV{eygXt)7nK$zOOHX8z!Rv3s$uIak~PJ}978k)}3|{zKu}V?_PxW4$Nl1{S3Z69ZU8n98ueuH&0pZ#PWAgx&X}&zotOA8A>K7(n3NLG8JUHLS8$rJ=cG_l@J#y||AhjV==*p04qwPm#jOtZ19iO3YofDnc{$^hroDix(ii{PkwkT^`i{1g<Z@jQZPzOfxuIdEE^(*%ONe|L|01hW#j9O+-lGrCa0k8_*aHFf(Cs!r+s4xEkEoUh8do*8Gc{fl&#;xnG0TT0el--z9!<b#u4(Gwiop;Fm!$q#+dR6KQh+xH>HemwWLGr5!FMz}=g4Rs4dn$0><P+l1@~{@}=s1Le~T_(T~Y>15wE_T)#9*z`@V|01L!Zx|?>qi6~+j@XGE`Y=Vl3{J+L`osS;jD#_HYmS+&Zc9r&qa(diw@<(Y-<!nte&Y)$sKACAeV^1TaY|JMi9UO$%UQ3$0*=@`q_2>?mK;qaHQjC>p`;|xUF6Ht;by?8A7DHZ#!sG33S$HBe9>T3^Rf>Bv&k%npG1H!TVEGhouVQhTOZK?YVt%mbwqg4?R%jCau2s~UbYqcI`5o%{i6T`6HHsHmOj2O5|G7koV5}`&MV&kSa%+%DloH_QLTIN@kVS6U9JBefh+TUTYnak@PK~(QyzqfGm;gsImFDu<=C^K-_qG&b;X-8l{HK?89Wz9AkO<=c1JV>kVKgHOHl^x2Qdr2WG{A#($*RalYH@IdD)UcH0x>X73_jB9IYrXE!2Vj6|xVQ#sxIQqW@faEh1LQZRbY&?dsJh_V?rTQ%4&3keWS0CFts-`6pxIydAcm5eTK1^9h9rMP&nH644H+p<2@(4E^3fa@G&|C(JZ~*Nh%HpK|gpU3TH(<-<nnmtMTw2J@(~W9CSbnTUG`Qx!b7FDa9^Y|ka3ym4Mh5orQ{?ampAa(+&*gLU;QNK)<3s@lCa&_El5LSuFeJQIy7+H9vv@o#tBPFFbmu>I%^1#;0Z8A|(eZ*$P&v6|M;eQMg*e7lElg31<!9~mL|Zc8e<HAMW;5IORGTOvWS9fiRFpQZm2UP@yD*PSMMPp~~vPgs6E^Wa-=Tl(Rwm?10U8%5#e`n19TOPSvT>UVbMftqH3Mt@{+YgGDwQc&f|T>ABWeS)m7!_z#V<n_<uI;Zv8jo7rJCIgR2EMgnqyVl9dJ^#Q4?KmzD_Xt=r9;q_k?p&9Y;97pH{rs?d2k#%oh4?n+^|I(U+Gly;uaSWia-_id9_oui8+oNGyyS6UNT*r$-FcaAJb&Wyu)f?4X+iOn2I9b6pc=6AITQ7hv-w(g#>|y1I=j0y&l4pz5-(7s|7ni$6Fn#@ivaXcO0WKn3<?@w{cO#CkGgK$m)Rl=-auT~ika)}R@z`1>V#C!8IYR~ci=`@Ft8!S{rQZcA<bbvy&uvQT1@9oq%<(sidyg<pKkfrlHB{rH29|r(?Tdev}v``i<)^}kot4He3ERbdD4v@F%roO#VU9j1KL4918%!5h`O15^jA*G`gog~k2=kqhEr}V2>|+s>BHo)c!(O$xS2bzS!^h^$85@%!*FnBpXAfaoa8bDuD8^QQwt3`h$cV6&fh^sWsXMwfFvE1x#FBcMveZC>1l*SGIhQ12WIEicrs>R1Cil-hgExK@^evFZ2wr-Fa9Ew8CH6IHH_cMW{NPCG#nUc%81Ki4AUHmt!@PDy#L=Rtshs+=QpJGiyef1N&6h|2Ug@@_MQK6P@8nKXUm~M|D7$$v~G91a*6ey_zbeLGWSYPuGxQJggEoZoNV=*DG;C&mUNLE?}96czs{)soFB(ba6~|i{W;p<B1-^+NJBZ3!lgPDM$!1bz=#G(h7NQm8{LCPLEWvv9%<;=WO99h>4F;V)keFr$Mc}PG~{wg_xKFqPsX-U2nvT_ap0D|V(w#`bz{^Ver^G{iIYmnK}kfEr*UPMhM8U+>%Uo0Z5g6Jf1*r8iy+_0ir_SybS`Xf92zLQo}8GXdV7e_mfKr}^$7tiXh9H#=yVWR<*W?N&+oT8D>4<V0a_!<1-b+7$|0~izO|fZYP>%sHT^OPluKU}#k#eL-N)HyfZWgba)%9}WzA48f!{qOqB7*W{_O^a8ulePUM8*OUm;gjqBGg|O}goB{5kAI?#+$7qtdiopUxZ208c;#qLx^ZJw0}QfCCWjnGx^VbS~pjVh5H-QW=R;XaVHQhqhBym>>IbX@Sn~3O<tG<Thm3L;CjTimVA)QgdB+>AeBd`yrD>hL#^OY8_}Ezy!oZ!XNA3$wN!e$=_YPUy=37r{ZKZ@iy)zQtSfj`4u4@E0x=?x&!-wAe@X@`U3_Q_-r)05&;lgeLQMs_l9LmPcglY*{9Lz;6^3c%ZZGstTFGKNkSHx(?|*;<85JuLadCW1|@)Ht%*hPZ=vUSrzH>Y1g`LgT$ZlpEdTMHmq)$e`HVwD`~(aujYzbx!gno!ej3P-?A^st$@nD!!e<rA2|2SJp<uLU^ngC2TVLLj=m`LU25oneF(N~R{q;Dh+FMlq9;oeQMK`*s6?KKtY!jMRRB45lM2XA_7*;9^mhR19Ll1-tb0SVV=ZW`+YZDPgEX27{&0Gcyck>E~mrm)J^!=TGVWx0;wksnDoHsEe<bepVs{l??&<mCo`N;wc!j}I*{gMBV<^JDGHkdxx`EdGWi!%)<a#B2bVC)lZVH`toQKX4b59}SDxViFxN5_!%H`hyaL}OJp58h=@?Mh3cca?R+jj5G_eSdmYSj|{3Zl>Wme?N+gBzL1V7l{zkg^@wNeW%eXp{FOt@gnu^YDbPP*xFV|N6TyPdq9C*`Kc_KzCCn#fDasE<K4QSIU>FXJ_NSfBo~Cr;h6%QbtTVyON#Rcf^nyFLx+e{DyaR|%vpWC2^JQT3hd<mI<4i&{ZB)da@`@wZQ1#MHT6x*iOf=|1w_U!WUew!wD^|2L$g}o4MCdl#JveYb4tx*<pYQbECg}DjV}}k8&f!A*JOnp-aVUK$)2d@<7iF%p#(6<AK?V%tdO?}7VsDwiumN{JrU!gT%<jM%z|xF#N5<gk4IxI#PDP6&g#4c`h92!;50G{)&k$byKx!iTR2A(`?`dJHS1uMyoI_ayUau`Y}tHr-zmZ%N77~|PD)qc_PmTzw6%}Lk-oU%z^9v*C<!t>GVrg~op-B@RfeH+T}JEf&JZvWQsv^9a#<HAmtOs~J3RkZ#t3B{S5&NNhpBNa+-Y-uj068c6wqlnx=ATAbNy#J2ds319^87>Cph+&N=~}g*Wqb~n-*TMk~?%0M3vpui7jV4LZLG7YP@zDbFr0DX!C`t<q2JX-9Op{i#09Pq|w_qB2CVByyv*a0m=f5y_FeFLoK3{&;?1oK#x{h)<|p}oW~e01PO4yeqpMY&?>IHfD<oOFCDVvSZCt)Yxat&u@o&o%}VrjDA3UeR!TcbqM)A>DT9r<ZtU)rdy_+n`ZN&<Nn*&+a-lbO!M$5voD2WEyNDRt5TKx)dq-neVuQ}78LQ&40lPXyCEA^FP^AdeLpxr~bRR^Sd^E->V~B#ReO;@*$LfOZ>cmVXPVd|knN>Cg$dFz}^(#!KnQ|w8ZFTNM$i^*1*n7>z{1+N`RraPRD%zvk{r$Y2kH+Va<~#gXIj>Y0wde2Uv%@fwi;x$f#xXNif)n$yy)LZR$-re@c@v536x!m~K17jiMf7?ibrPh_l~7BA7!-^#*d;Yzc}#z-8PNkGRv*M=$LyIQD1`7{e6zd<kmiERC>9(Pnqchv&Ug5qL+c~^uJQ!4$AHGZlmk`n7P-Tb%CBPE7n~JO*3xZkFDQ>dcJctrI#fYgtIAf%`x#5~z&H~b@*Uw7)A1u#Wow5pCx+(~SJZZ)+{mHCz_Wjgl(z`xAK7~0{UHu^kM_#eVxf9Wl_bd+^LZ)F$q-&K=c)BbF&#3RPnpN4qm$cZ)Eu;E7@>wO8JHGmUANS&`5gS`*jQaD{Yf;xcxxmYxVDpdBM?bQid84e3MDP$a#DYKZ3@-WU`cxA-s(^qzWn&JEHoloe#3U}kPa(1$=JX#$v;J>P^OYqNLNcQ`27QYiniAG`L{J$plYPWf_y|Kklf7i+5kv&Suj6@EiXdb?=Av+?Nu<^_CCzaTqN|p@?tU008YZ$=JSeUn6s1LdqvE{>kT<&V3NFg(0r=uZNKCc9nGmkqn&2W+uE+k&qoCh0i|>tmJd5yIOY$#z6W~(8+$&5k3jZ21_Kgh?J53=ymG|Dvt0u4tZz?;iXOPlWMCw4B+@ihy9~~P-nZf04p3{A$K3m@?iz?nq4+$wo;@yEAEJ2=m2w{qXnnG9`VLeX#Cpq5znL^wq&H_yCBBoJ(o2NtK>4{u?+hv_7Wf?wuy;Ug*z-U>fNGF5lC0ezHb|PK&2;SM1B6T#UDJUoCM?Y@E5p`z48Ea-6Kr!x414EAD5~!ENs*k0f{4|zG6{ch-io&<;^W<Rx+@0lDjoGbtloASjhIY(UcW3m=@J<x^B>o#cNe@3ktI`<KB%!9oD!3R52(FrE}4?Pp83L&Fh{#=yBgaYdfZ~lQrX+P7zeq$j%-x|@m<hO;AW8#<lY6w;HzxZ+zE2jYLUCL5vJn4<@^*|Nh>As>UQt8@jCuATWsht#JXo<A%CWbiv(-3MO`vjn^<aaHVYAa@HbYsL>xSe;f<Z-Xsu#rY#EB=!_1^qyT|V~ZhF18YWKLxvXTUMH6FnxWtWX}%^W78+DeTFDVdMN+Afl~e^dXQ@~&u^(o1*Bw<RroFxklYYg`X{G8B@J)AxpUub$^(#@~qg)PFmT<?p@Jf#jz!K$9aTI;}_u9=4tX5?HJ2`rOo?LwgZ~mf$p)4;^PUk0{|T2g5Cq)rLdoTj}mU8IsE2jrEaT=3As4K1W38c%tm;F7wh(?WoUSv!IeNXiSJLLRW~fiRHxm(Bx!Pv$nncaG+9xfu(j~#PQRhAQ6Iss51n-+k`<sj0%^p&^^Jc!&8+L^40f0Ia#V0jv2ha-z3ltqi1$&6&`i?f!J?9c&;3WC;l*NQ#b&CoNbL7Jo5oc)N8N1P+<`lp96+r5%hs+PWXW?x$NMo^Ik!!%0sU=oj8w8$J=4896Aw5If7U;fMWC|S0kMA!xhiioJ@u#Ku7p>VG1=)_qjxD$92|<bdtL?5-_--=M!`L4ax#sZ+YLBsMD08mKjXWF~htGRszu{!1OLNG}c!No;U{(0mP!~KJqnrE+*gJRMbq>jT^6-Lq!$mvyT_?KyyO@=G=?_u4eY#`g<*NKN{z)DmM*|=NF-Sz^6Ut1l~M`>LN1N7(rbo<dM6<{A!WPPT!T+HA7yTDj@r9-syd-{lS)unBY@TKn5h(+z-*grt<N@kNrCr%F3befV#F;)82~^nLZ<<1?6PNS4T}^yRtk&a;q$!)I-3}=150zY}zK5=F%0zs&{OgP2XEO0$VKrtxc6-UdS!iC}sJbGm}xWz{q%>I#|D^>{>e#fdI=;%B(}!h0<flUVZ&t5ID}>H{{G&De(JV{jVmWY9NgBR85x<j79+I46+w(rO$$vL?inj@!fj4IV|8Ts9<h%(5crr?izx_tN;_)vPrk^!|$oE_xqq=N&^eC1X(NlAKS{kZFL7Y6zdP?C=MCs@RC&TqC8iXVUqp2$63bIqxA5*Jsvs<@do?*xiFqupFRFbTxqKMgY)BECdj2x>-W{z^&OssrS%6E;5cWeJDL9O!s@M9UTds%h($^+c}Zh8-AV%^Jm!vUH6zls1uuE;ZMY!(DyE>d!VtB!1TX4-wwgM>!x<Y?phVa$lm4skUE!{fjyl5GiMZzhX)0GG7*|-NtKD+xFLVX0-Ot$d3Io<oP(Nh<kAcbB^G@HR3L`X{b|>HeNJ5`r#8Gb<P=z&#DzB{X&fFbQ?ZLJs<U?ss#Ge6_)=9Ios;YcjVhclAL`o|>{FCW15plPU?%#^4G*KXsRBKt$Z0UZIoGj<Ba}r)M7EX9&eX8%H-<w1IcnG-s?143$Ap@+9?M}7hcS+~Rl**SubRdQ3U?nMR^GWT{_8`S*BX%O|mMO2_&@KYpDX+Qzbmc(Yx1)=*oD6dP<|(7jz)C`hTr4=8v*@B7&CO+@FrDMCuw1rl3;y$oIsqiczOyx9)JCI$4JdI`<!zcktN4yVz4SA0I4%4OeIsDGohYM`bdd>hOgfyo-u*x#9?b!bDALZIUP~p5pJ$pKzMpD9aAW#?ItlhZhDn+PPN-QEQ*VtxIr<8r@n8u^N9rdfW_}Z$JZRVGU&&N$BT;-A(FR#NZIuKi6sHd2db?tL8m&>5cz!3BayQ(>@RLb&+eENOyRQgm^6S;R&dA$9;EDWuRq9$FCQTSF<d;46xcpDDY7vmwGuUIufh?W?wKH&|m*86hc~#Ww;MJ{r`~O+PVC-@}Bm9EC+Bq)u#4nFhfAmkbZ9GCki~fG>dt@03hvt7d#B{^=6xiiR*yB?I?rNzMA0{<rk<lju_KCdf4JoB>1K_!#IDu(jSHha1c!BGh;@?&9Fts2Ci6(EYq3HOrrXIu!DdQ|0`SCDWV*t-lFm}X#5qRKp<h}2Y7At$=W{e&{95;ecmR{?%`N+1N7H2qbZPycw&69^H2m&3cjI?Sf&XuF!e}LEn!qD3dW|71_pJ2k)#}8p<)F<Q(<l}g@)s?027PiF`rQW3B_j?+6m66Ve+l2+{*6rDooj)~cWq+@KQ$jI)>O1Y^Iv?>H=;wPC4*U17d}x*l@))6rblUVfmBkx$s2YwewjP*E)Ml3m^eQ69W2Em^WRjKgsmIaRPTbeb{W-H#hF6b~q%{cST~D-xqi14dDd2g@+P?Lx6J~=(<M<1osz`EOsyCg-Hieerwzj-<{*jCH*_IB{_EG|61!hM#2kU7cv;KO*{!C%1GD)KGA%_#ny={$fT_ds0^^0?-|0^eTYm$_j(tap3v&38#c88|_zdfA7rC|vL%>Xwka6O&uH(*?2H#-9S>Aw)8)NrZEUdi`9ymEl;x|2-7K6u_Sbt&@Ll!F8xdLLyrEn@lG7iI6KDm>2@{!Z6n{^SXi+BuHZ!Tx7oEg5gZJR|blTe8nd;hr4->fs<_(yJ~%1DcSFZS~M?5Xv05jj?bEfAdSLU_8@2FEdIwmiIeoRqMcm?kDU!Y|Cxma*lMUz=7rky;w9|ZLNw>d2M+oqYGRoQMm1}?7Qg>Fqm@Uhq-0G^e+tM`<SxaySS-O2`&;rbi%2wEd6c`s2m;BAY_DEC*#bNddZKxxJ%Sl(qG1)-Wvm`P$eL0qls6<$b_)Kt<O+T`fDT6{SKC;GTG(@3WC$u`f<HjWzAQRzJMEmxgm)pFi>vL(GYc$e?nIa)Vw|#uk?3NjNH;GxLh*8vv71)oK4wGj`Fsez9{_4B6|>(X<Q}G5pjh|hvs@sH{YOkR37)#)P1ojv=6aY2^OR6oU91gzi(7Qf4S`unD911g6|bDN@lc-3Lzo)ki!#Q^I`C0q=)#yg#_32bQ#w}rp&8oB5@My_kyxG<(}81rUEOA8@a`m#Uinb{DX%RWr0(zW2{6)6DrVLHzGQ9?YrmK-)FCi(}VOLJx<}6Wnz0cA9Zi2!WCAc--z0JR>nsLad|<S4yL8RB@%V2U{M5yF_-FGlb!E{lQ=~4sK0KI-FRF`((MC>OMnhxet7zAfwkR3aesw@J5b070H8-gy2gMo)kLv}bB7Ql#fP{bSwo$i2pS>ner+U#^wH<<^hmGOg6vHRpqqwl!=ju4y#Efq+Nt=uAY8!l*p~~osL&l=1T_{3Gw3QTeWzdzxcr+g$lCTF=L{X|wuc0GtpOWg6&vPbby8jO%Ot!DGJ3jJPKw0<kh!Z(`i%V%=RxZ9LQw#dW7pvMSWc7<Q$mM)?>tI7pv@HR$#AL<+YRZL!}}o+c_U<k>#>wX)|NL(LC2a;<HI)wY#lkQx!nE9^vnq0{Kdk=GV6Z{6Cq{7eYZHT9;yTSMuWct@BRSe3wzH89Xkluigk1A-~{?d8_*o=3|+_AV%o^&KQ`UV$Ch7D#n%#Eg@QP2>kX{WqBT;cS_n^W1Bh=G`{JrD)@0X}9+rvuf}`#+gNdtTYSetvPsR-k{_4mmk!M3i=)`Cm5cw<g5j~RgFc$Tj7@@NrK3OO6Vtg3%!uCv19;Rxw(6<3I>acwRb*2LtZZ9_0>F-VA#a|-YLc3w7;K_)rt?)}b5=sF0Wo<RG*#J|m`+;q|oU52f)<WTU<LH4Yoptxl2y+hjtsqj&cr%wdaKC2Nv@R|kazdVdZ9M*-Y$U@ew=cbz0+5oI;F98F`7}145M}usN2%Y=t4_uqPQu}@$W}_`S4LSmuC%>kumH!15vSN@N%{3US{RX9Db`4E^`g?t-+#cFeJn)0^@xGd7P{JgV+T&gd)s23I)lrObR5g26<a_LHm)WBVXNSNJ^a=kpUpI9FCU9(2g?yT%D!A>q;H6Twm%y|LIZ%yRMFil(2;&jrtx40!jJ)%{6+&)%;9}7*;N?yH?%fI=OE-QhsT}PGsE;<j(=XM&SU~Hxayx!7aoeZVTqFKpIq5&$})MGGNcWVH*f&N?LUAW*pTpxnVAX06N6Rsqr067>784ZI4+AJ>|EVnaoU=>h)Lq@n#B32q9s^3qU7Kfk=#8J%vrzSFds>@)7h6thcfngWsAkBvi3Zf6#!}t8U5VnMU9>w0h>P(xfTgET`U|YE!{ZplueBDrPKhh!n-nr=WBW@zW&0EFKRF7$%`A_mxQ@;cZsYl>(FN#p2kp*ECl59J|ykeRj|`p8QpWIaVdlg)++6jH`9Xa<G1US_Rg5RO<7G?yAnLiNeih1$@uKz2)1akU*uof7s2cnwHV?Gfv&F$8czb+Rjy9d_{HW8b#E)ZqafAfR)Me>0FIG$;2tTEn)Wfo(;v^|g;z^JNDvz}fg0cSll#TySRi7I#kG<SWp$24>3vnZ5?Wn_yQ0_tc|Fa5*n#wp%T~ayplTs;nOqr$7zrKvs5hBwA|OZ`wxCFi6v*)laY)!#Fe}P>;}Ci$nqUgh5NL?-n=$2jftl-VzN1lYP*m0erT5hL)-rTSVx*fK(pdCo=}Ms1AH&ttbwC9xT%*SRD1j$AiB}OFLG*{%lExK|RPO}h=|Jh)ZG>EI4!)}wdJz*5mNo=DK6R@rJl@$|Z#DjbGCA%54G&bn_5C2GUg!J&$uNn`?VEgn6rUk9xroF`&1@PUW5&g5nFn7kvPafRFP~2N9(jlOdX<svWAsma06X1?u;K#E(U+PxZJoS({YuTySm}jFq^qVJZ{O%#u6oM~RRHlqffr_Fv7X;}zmcJ9sJS4%*8`MTedaS^ra4J5an<W%KGsqrGR&R;7sxI%0!hpb>^p@)-ipg(1&aY6SeU^VBA@RAH<ApnRDOX0iv;em_aK8M4zVAQ2DygNVXFk#_umTB2eK2(4v>fuXS7i`f}L55ShqlCtfSri@JURtj~zD&iBcX;)*?#QcEg!DgDj-Njc-zs!9#xh7N^NHaR`}LzQ7gs(TpMTmb#F(Jc@i1=>%e?Ni-OjOQl;@3>i^rdcNFD&j-7C+?;J;Hb2peyf;?Y*a)ez3H<etZbe7=tSl|e5e%<KLUkLA+%{7Mz>Z9F-Bg%_iw%Q}f%SV4C^!avqKJ~`GiQaQ`pe5NDk<h^_>?V{WzoWzLF}_M_E08$vqi0C<-NtBuZ}=@iY))2F>=N9c7_H5_XV1X$>(Aei@O=F;a^=Fh`5yZV;}fFNIP$SFYm$hgsnQ4L0ucJmc&kX>s4~sbDS}Ry`rEK4r7mHA{DX}lq`4bzafwYH2g7VwRJ*(N;l;%SXBc)+N`9QLr!_mHA?fXb@aEYl<CX`b|VlZ4jPQV%glDa2j?Gb1*S125(Th`HTCgf<aoH3kwcGSk|bt#bw+|4N+`tVkqDvwgHazH8T0Hrg+tLB%+vmC+7`T(>A6!aqWoN-Or20()P#<m#Ujg4oYdB^cqZ7~1LilU_%c_A7l}%VVFXsxVbQFXg)&_cr>IfsKNeWtShL$sBdIzmGo=s^Qaw6v<l+xHumm$D7~l&^%2nN$5?o4Pa&5ursb1y5Ps3m?zfrxh8}F==BTvS^;Zz9N)+@5A5|(+LU}7!SxO|~2{dfe<nV;qYx&U56F)CYDNAi$phuYpM_?*s+dm2o>{PmC)L{1lgX31rF{)En-$h_?oQimHpQviTYaFkknuOA~U9a1L=t!LPW6gv3C9q=LYOjR)foP__bPZ7Eb{A}Dh?Ky*3AtjQd`P-GiLMo<kY!|U--LBza8P=->hcr6@2}}$1{dGr{yWCQTt2MTkc~K(A(_~a$y?8t&R^_#c???QTr1pG}8~xYwpktt%(Q6O@`lqSwIVN6?X87+?Scn4g_G^Tv9+iwJdG#4LoT(2?;U4r6Q*hGdZmr)z=zYG(@|k6h+7{@-1LB_ImcO4YM#3B8W|V+6WO0sXZ3b+>L>#2JBoo>%C0ba*Va!o_6VLZEh-)NWH0MUB3XN#+nq2lg8_?x~B<sw{StzeO*3oP0MZGjLDRN#@2Ve`!LBlFmf&r@yumMp*;uRM%69_>h-MvnDR1H82NdPGN2Bj#J&AsXBvaYW(2?hlucaWYx<S$(x4n}!=vMf9w3#)%?3#<ciyDW0*S2;rAL9j%eL%yUaoPG+?kQr*n0wv&1w?e9_jzO1_CyF&+Xz-zH6=PONuGfJ9qdeIH%tf=-D?HD=0}mRyQ*`Luet%%%q~DH~8r!?1_Rg_rKjm%*DMk-$nZaqB>I=6O@BqiX6QdiYDZMnCwTx(}ZaHBglK~M&ZJuJx;e~pAtR^O3oQSz7*rug-#YiN=GoMU!;G8tPx7_E3&YF2EQU1g!<vIB%gKyQ#L0;7(gHseZ(&B3EjP@<5YoM=?Tgl%$iSYPFmI99b|6?Rk>?sa+>7MeOYOZ%~StHmvO0r)j^nUTwIW^3%fpA)}Ss}^}!ONOvWzIjj2@InjxJ;U5syY9VU>%EA+)7ptEla5NJG@&W4n9>E)jXjd8_XRH{tEin!Eljy_uWa+(EY!`_$&Z9Y%sFDiQ9)LzSDwiJtpk%Xz#fs_o91mNydtXQUXzL#HRq{7h#O&!gtd^`8EYhn9*&=WUw1eZCL`G--m)lp>l%0&4>tbRuv(Q`}G_Co&03qAmvNKf?0?kfKS=__(6?K!<i<K6-y(OAQ00TWxmEC`Upk_uYXPL8YWhQ%OUCl>9V=8pf*U$lGl}abnjOw*#Xl5X?)f~`}ZD=xmA5n$&Ly^&aenviuaLkiBSk810?xhg=4Y{S=%oI%5L#I&f!S&OrvG^zU>*>*|V~KrIw9F^&2+dw1?l7IYOHOPNm`f^WC2<wI&T_3qhyb|JB{!O=RtMdv4IY-T=%!cnds+?Vq;uOc$1QKzW^l3qv-er-*aDh@v#ct<20PEh4|{$VLBvtlOj|Vj+O`i(CpxRS7lUl>zlAiQdltBOyWXn!JloU@}V@O<pwzBq{+GC|we8KI&`$(_=MVziKOu5D6V6h6S~U{VZaJ0Yuoc@rwUV%2RA=9fh0L)~OV~osMG)-%TOs#>h<F=g5F?B=6Q+;CwLKPerY{EN(s9=)sf{XGI8-ZQ}hMN(nnUA?!>k5to;(>aX%-P8>xcCO<P8Q9cgF^_g$+t*=A%pKj}dIf8K`f34}&a1Gx`%~&On%{9#P)1#PMp>$!@bDzO3D>KfD#%8sCYpnA{Xk$48osejV2MzWHi}>s(F>yGLlD^n}QSPv38YW;mW6D|L^-cNS_&f&<=`a_t|8jkdHpL{7@uS^1+%&b5YkV3%IQpP_=hjU^(WmMf=1{7&$sz2wo*$PkScKPmOS(SS$ZkT%NhRC~DEvNllQ+~2@2~)4%O}Z{K~-uF1h>TZcLwS+2HiJ&k!Y<8Pk<LJQOfuja3V<1j4r~g*|G2<deqPODu4}}KS;SX<SQV#=h?v(u_NB8I{?!TrBX`yvvrkyDqkc{Uz`gR!g*t#B;_aq3n$*O=-~v8dj0|F^><%pl$rQf-3WQS8^z0W+kRTjfLfV>OkNpQC5^rv$f9Rtm;lcLT|c_O5F+oVHmiifaGJ_^+nDKMT@Vq3O0LyOgt(o*Bx`H;g}`8}#z-ozR53V1*S9mH=OB?){A*E`I>=dmPHAWMuP`LOwJZ0UC3_nPgV)-$*U`Cq9YxVSNx6X8pDiTj-0lLfBVMZ#G-Lj|Q8Neta->K>?&{bG&XNxjG8)l~y4EgT*Hv~ruUvANgsr6p2{m=-4(AmqBwcD{1fR^D3Dn?WRT1gfwnEa=FU1lI29bg(fmhMqr+T=Ty<HK-ggGZ5fdYhSXy0xtZl<#VH@AK_?&eP}5_d0B6cm7h$mq}En<Jn!Lj5$jaV(StzRSPv<et!_-&!d_<_5y$sx5a(W?^9IJEP(10V~3cBW_=E<`FI)U?mGKY6i0oeeJZI^<f10lPS=<*pi;ENuhYz9n?-6C`1B|p;3?etRo*^N4=e7r4xv=pJc1Q&T30i4NcdwvS;sVEI|+MiX{r;AZ{lmOT~9p!u1@HIvK2<Ox(~Ced_CxqnR!qN#A8qlg(Vh`>>iJ>9o{8^m#qs*HXHX*b#*Qr##Sl6cd4w2_DT^Nbc2Yl&onr5jpSHQry!*jXoA%1W~DWyK;!3AbBmyul+Ck<XU#OysSY!@FTQ=(woeorJ5caC_`tydp)sKCRi@KfFg7Qpr}vsxmKB0VSyX^`1!7>$C2xO<)|2lWJYY&gW^LRH!T!*MXvcqmW6K~Mr1ASf&60T36EA8T(GI^U}3~fTqjOp2r{g<GSx3ED~h!p_W}-B+kXY$(G_ZVyG81?(D*@LH55WlK<{t;3FII?M*W4fgpdWQ*CjJ8=p4bRB{tXgrV8D+taz<xGwp^`u{l$Bl|w12G6#LNCy|wZnAOBQ0Yuc^6@LZ)mn(V#83Bxcp!%1==r8PDerFZi6uI#sv&7vFmCq=FLon99gz}%O&hfKyiNk`;)+#QgK7kuRb0hL4W7d8~7|4>`G5A{0&tkR+_0B}KAJPxJq}NelvEX}3ABi?Me;z#|GBd&ZNvMDCbCzclB5<qAe4QQ8h^|Shy#>IOD~-ZK>Yhlyli&6mPh>CKskZRU7vP8uRCpLbK?gwR%p5H((by!vY+MKP=psz(AMwxv8+p{7aU{FPww$L!<Un<5=T3S%T5*<$BZtW)rYY)d`#{7&ksP?=`WvZF2&)h;#l~*tkX{K-i59o$tI=EwT>+^xd{pj}TF}9Q7jzte{_&Do1Sp;~5hvGam%w3Z>Ps{juOAd_<hfRrdFg6tGP2znG--Ka(krfo$W^mKf8Rvg!p?kuJs3Wa1my&Qc4|#7?TE#xSjTv%_i`IQsGEI-fCV2SX<aso5lL%{`y0yi%a>!O{sSSmi?6#EY8Sv{$`okEdcz1~v2mGHkzPWJBjz%1pu|+sB-P#~CFIB!3v&LM%1N&k9#baST<{(7veVbgU`TV83LLJb0H4!o!z2rQwYOw+^|$)$2646H_yu*=qMN{k#o+>fjx6GgCp~wb^)N-sCVl-k62=UqF_t5qF*G-sbj()~;@FDo0?8QR<^i9pZo_+Ey*YMwCnA{jshv$J1=QJua!pL<zT+Jz45l|MD*Ath+S9Ogi_?=en#h+UC3B&f{66C2R*KYuiySFl=+%}08pm9+e500wyT662ood(sd_JlHrt&)A_vwYtL)hI#5AY#I85S=@esJw0*_9QA@C|yN%E|_}sgcYN$J%*-chF&ZQl!afWU8ftA;zzN1{@VuB7-GGLvE*oW52GjpsP8|50VuOn?2uRqhxlp5-O(@#}OW>ZPe|=YJU{~d9CF=ICBu&W3qj(P&Zlm^-3Bs@328q^Ois;3$16N>!=Rx#w;SBO7uAF<<-l0k+k7p0L>a01;S%w?AI3)`iWj{^Od5h1LxNnx5AkeQr=BZJyY&ov_^}q^vzzJH}OQ39~mE6i6jS8@alM*hAh;2OoKDHoSc`{4?HBAsG@~$S3%|sE>2CGuj}iU3siaii=`pnXJ$?L(9yR#-pbKIoqMyU-sZO63gwDp3I~QIglc*5_kIxJC7`JP)~tF-6hEO2YXp%{7Bl7_Qhf4vYum_jkF?^RSesI%sz2a^LHHYxRuV300hK!3r7WdD?7bQj5}Qlfz^?bVY#FfnWCVfefLKWSsfjwvV#D$9r@b%`=~EQvh8jS^zYa~Ci4yfJbY5Z@cT*6TIe1dZex)9r-zP9cI@{m-<(Ne&B+4K@@~ZCaSzjs)f&b1h+E|N70B68H;%9j??3T4V-~hP-ZIFp}o5J!}8`5%L{_wB<Q~yy=EmCd0+(bdHV94ZtwDGiYgt1mQjE=HJq9hy*U_2X-9bm4ojjsUV%3sQwA7|YWE3P-rUNm=M)D$|ffmej;K~0u`Un+MX>a)xjmTQoZTDxD&4!KYmkgoQs|Cw>A^1dal;Qw9*krzkU7tTtjo5K@EA#ljWF(t}hF~v57{x4zGuAn&;)lZW|gupOWW|=LOmt9^<)K3~Ehgd5k$!nJXA*_#qGAuLVf5><?w5-(!X0hY&WJ}I@UoVVXYtqkiHo37$^Sa?;Qg+?Ws_0SQ%Zta8W{fJE9LW=0f10SX?*O3OV#*LHQ;UYni?9obBRF_?ox@b*rWnD*{YYbN$lzs)Z4oqIo~#&yMR+NXe{l^JFo%w=398w<GHQRfK3_TzAa={<dty*txYtX3+|T_ff|vWAG9j7FtJScC{uwaydqg<xkbk=`A{>>y+h5+x!S8Y5s&L~|%<>kpBs7U=_#UQd9bE>;Us<~PCf2YZ&675+hrm{dAnyrt>aSDJFK+Mb(`;untk8AoAw0H+CXaq-t`EwIEHfNuT|QI@KH7{|wxhIj3(9M0uQV1eWm=*E%w3*EbKHE}@H|!r8lgv;zJ-Aa6BX_<?`qC#KS*K*Y+~mjbZfCvB?t_?#yNyQphG_#=!tG*oqohxY5(3*yop?85~2eKe_I*utBy7y^7Iu=vB-0NPH~9pd&bu(#w$VIas$7JRg3`VsPy=^{xGA`Y_`AadEaZ!K|-BPEW<kIn2CJ+EB}q^9=f;gIKBg85Bq-g;x^vVZ*Ot`*_hB|h{Yo$Oxo4Y5qj7K^{Z~c?H!h-L9|QxPEZNVnVr+XN@F7B%(eazeMxhnsj}`GGbsW;4*G>0E|=7SG?$tFK9G{;qz-0FGZ9c+r89zBG_aDcZKX|J=TiBWkDcOS%E|{%hUf@_A+Yu~5BgMm;*{cAPx9r0;vX4&KryJZ#O!CLJO)c!v|le&(!#pwwNOesPuw&2pGrN1m#;b~od;|QF-hDM5YgLKUhyF4Sd5~o3!N^EGb617_zKl5pUDNBsKlEu@=U8-B(|w<SAJ%~HX?%o-*VdVx_F^6obXaws9rerid*Lda`5D@vctT!laWgg%~0qovrOSl9mkxAv1qZ2qainIrf*5mq+soA#`G=4tk7$q=YOs58)iq>ds!{31msO{4&^Qw*R&}phA}Tk`FXf32I|2A`ZIj*Tgik9%Xe?y_d0|Yb@G%kL^4U`HO}$WVrwVv;*P98yKcwD14%;V$ku#tpQbjfRolzFFrNvgMmFfwCUG&k4JyChPZD)g|EQDtX?p*Q!VaDBQ0HYeIIfXzzT`Iy<F&B*fuLj}4yCCB8Xb!{quR?S#@PYmC<XvVAu0kw?kl89Ij>JgwMglG1q>dNrNb4?PU%KCNX<2!7^uQ-5&g;aWLYJ2iiUCe?gurzxT=brFZ{^A_aW7Gk&Vh%AV3lRU?^Lnv&_~nhU&X5b`ah(QZq3pGdYe1gMbzA%G`saJL~;?(5$zq;y$A99o^R_b0`+ulobXc{QyfHOulEGMmuT&(ob0eNRlQ|l@tM)7a%OHkMrBkT@?=WP|<JCjBG?sywg_JwQW9=Iy2>J&U7sAN44%|2DZ;mBh0GyIa~fWv;<W}whH}-;p@;dfF!%>IGSeJM2x;A-JQcBD_~Xb7{d-v@xGG6Rs`x5X5P&3lwiYVLUggh-Vc>Gsswpm8^V2#KxM+*g+<lEboYNN^-$fR#PXS|Y;KPT@SzH1da<1q`gUKTCNSEx8tve+XniPcG$eb*gcojLe}XRnR<us6QznL!eoGwZ`|MV3v0pva?Yc?+$$K1yo^TFri|fO3Lq6vS=*9Qh<c!84VoT_nkRf)1;-5yHzP{B*e7le&>8Co;X302s%!r5xp)c}aHh!hWCwg%}PoW(2U@n->)2f_4L>wOw<xKGvRGi_m=ein*-0e8yji8;MHvX~^<}=x?in$O1M}2JRg_RV*Z+`i^^Mt<+Zmf_5;qQkagfAQN%kYE9x!$T8Pl@!^CLhtI8p}K*7wCyN?1&inRW#{`Z3K@W-iFU?CJV=lS>O`b^v8%)yf>ZRDEgBU?+2G~(15uoEX+?D-QUZU5>x!_lN2|O%+^pU<E}%(8w<C=IcX#!f(%{r7xZIE;q!|{{I=b#U$&230;J1H%W&hG+;S}`EUQ}SGdM+*y5X^=?1N?upLEL&6O`3g!S7;-8%lR7Oh|F|J0A0`QZgoa@V@|0)13Tk7SlKc$8h9+WoaX_KBAg<xpp3%mu&Q=CEZMyiuoPEir4!)USc9QKQ)3<=vh@=RSIL{z}Z>MA1d^fOAAxJx|n#IWyAg^tJnjAPabL55le&3cl&Mz;cb&hecsTc%cjV6w1jwiV~cAOm!*^+dh<<$)O{Xbj^df;T*`img~yaINFW!c-v!t)C<Kw0%xc*DCq=2CmtQ>kx%1%mr{zl4+?-Am+`0Lu&$a=71$1|1$Wb~{sRZ~Xl4LsUs#uD<+bH%qkXdnba(SXf|7|nvKF6U9O>&?rKz(!c4gJl!nHzE<(G3-(6z3Ns+pn-#uPyU|-T5&kci@n~rro7c*i9W}X!XwjchB}Jfr(-l^S_#4&Jr~3*7wH6ciHVS21PB<vsH%0nAbOvI`~CTU7pKZqA*)%ExTjdzR2%-c2(an;MBWm7(ud4^ivRk8!(DvTnK}3Uj}>odAcgoCJ+xv2NvUr%!{dP??~blZ!|rswS=a%zK0B=({{32ZCD})gyA<<L2FB3VlOK^kYK#y<Z|x}^YFmi_14MGGC)NRTZgLD4Fy~@?2E5AZ6ky#YfH|afqBRLS({8DAsW7enkYroKpymV8RniVl38XJ(A6GFt;~Ww|6;TseiZhDf;<4-3jvR4_H~kqclcl0ko#zWcod;ROl<txU=7?w>3=2<$(uEwV~Q&DwxY-ad=|M<N2Ue*+p7cV)>orXcHn`j0X_?*m^XU!&@vRhd*L%Rz;~=mNwS{ql;yEoa9CZDU=BB9D}K=O{JS>_CB6-g440>duNa_{M*CZd#`vPIod%?xh;|00l7}({yMap)<$4Fkm0>DXYcVvVX>70yUeq{u*o#-)#SO~Rj+ehb1O!2hldpQkT-OfEzMB0m-rUxQ*ObdJ#;fd)-zsJh=O%P5@W=*!PreFtE=^O2c;dkb#R3^LxA3p;luZuYzA)rq?*i2>MDXrj$kD}Bovg9*fOnVKZf?$d;-k6l7k{M!kR1~>u2P=sby#~cPMEmm1v7~w+cV~;_)JINh4|y+i$p{BIoxde>F`|}yr-u2%fC*j2~1eG;^2qY+e8Um3k{F}>-i^SShXoOnd(kbVWcW<<}uYqaby-HBu*w-yggqjoLTN?ECeY~g$|;)*c<W&JX}m+)E;TWYyOBki(>HXUBGL|EucB}NSgi&`TfXcGEvucx1U=l8b}0+99gOw%#BNx|7TEE_Kzrzn1+9gcdZa~tOG=k{Vr@E*rU<l3tiBHVSsSk7wX%CKq;_<w|57An((W|=;U{ZCMA=zB~HTJ+$Nl*9R>N(Dk66Y0NJc2AX2Xy_Rw-YtAkf=b|2SfChZAo)9_Gy{yAIq9}6}k9*msf$Bp%fyhiZmrptiVapPUWw+)YJoMocW0uTvfNHX84E7f+%(|*QQSRDa(?C`K0-T3m6>qfM9m0$7to|?V>h^(?60d?t&h|))WmQq(sC_Se=8=aTkJ3A`^&#G~RJV~`-RHcAPcJp~skdWF7$7+2u?A>vtF@#kT!f)tJlw#O7;XndyOVmvct?SmJ(Zkn-xj2*1eh3j>@p<Swaoa<y*o)@&(|=18@U*Xlo-Po-+xzL^1<LTbK4(*U?dsJ`%wL>uy(#jg9Dm*{AC`S(r8fR4H4uT=ft6*scd!;q`#;33I@b%1=6m$I)6VoZJQ%>rFy1S`xs)hJU%^1JRXDKv>PNR4QFH2DRuqom?HObSj^Rc~*$>wRE0YzqAt>7Yxz9~tO<YQ##(oRXMTYg+UT!D-s-yY!6KxGa*=FGSNGu<LaY5x8v@G_?BK&Ut{{(qI>WTm+@rz#VratisVk?r+`}dG&Z;(OUR?ow<@y+4qx(!9sXt|5iJ3X>fGn!?}@Hw3G;qh(%YUYWFu9t4foT$>0;L?)<o=}vIUq-lgQt)G<Qb0*!6(&lL%}K&V=9zBGO^FV8oMT)a*e={2Qwe?`sxv6+^5gTZ=-apNNG|<5$s)zl_jdeyBv?HB<MC12M6h0_qJYVeX>71kjhlU1!R$3v*L;A_@Rb2o1{6=j=JVM<@rmqvC(0pmOVCN$4%%M9cR1NS-nH^LC~}Kd!zl`L5%qk60zc;JMt*WZCKtSFF@>Bxj|ioi?;X<YCZ;;0$dL<thTYZU7D6Myffkj1!Zk_D!#>17P@M=b`9;Qz`%P%9l(7WBV;uHHKZUseh?DDr>e)=4cx&5fph*#!lY96{ZY$`x>kSk<RDf{@OP206G?{)ssBxiYFbfYXyfmN5^+rY$`Ja~MzTlvlXFv75(SRJtY(-p{#D-znLlcH%NRlvbTtS0Q5De#BM8?MRG$uGYkqA7?+{^@la(*HZQU_mv($W-H1!kcO*Ir9(s-X1tE?W@NQ?LOhr_v0maZ*_J%r9UCi47h8&G+++-?q2kHQr+(2~@uRiKw&{?z`d?|CalVP)1FIz=fP&z*$y9y1y>j5CdXU1Ao|&`_%xa^bsxM5s8wGJfBFk^mb5dkyEeq4DjvP&TfP{hdoyVom!oNe;#WI+gXiK`W$$5WK|m(?bI~0zED@qA-%NR5K^6`vyL2JJ^^#ZYp8E^8(_%L9+exm87NrzB)%Y@;Sn6Q<Qd$zA2_hy54Rc3FV)xGnxgxBc{xlL%AL5?@!R@PvufGJp-6a;{PY&J+|0NzJUi`kF`7Z_Gwa?$Yuvgk;o?L{bsmO?4z%g|o!K=C$ZkV1OZ(&kCNmOXMozVBB5~;Q35%8498Nkq!4*1>@82$9AMuemBn=ec)V%~JNU*jEyQHi<-YdOF&*FoD3=ZO1tfe{^QYmnDyA9`LZxKopG{85J7=)sW5b$TaWuACNHKq|G-DJ?7euVt6E&w9q5(n=XMI9tsum0O2x1aI&;S>LnK&9|{FMmDkG3-4VK1U(F^^%vRRcks!1;@!OJK)ZZ2Rz3GMB-w#Qu?kSmB<4~JMD+}tqiPF_3C^9W0Wt|P2f39UeODTH*sYHSVInnyF<@qb}VpkIpjI5=(>kFcwP)!$s7wRk%>vEof<qz?cECJEU|=>O0^kz4Ib|!3MSIbjd5Ca+#ndWW)5smbA+cpXnh4x*gsitbB$?2<}EF_S&liQrD`2_@v4JeDuEa+Zl#o=FUqDbu(LNC!v*LwgDq_Qwt{F!nq_l)OQ=stobQwH6k)G-ySv^Yu2I=vqUvnJ>l^b=ebRgcM??n`a`_ViA01pmd3Qfs<u1a_wi-zye37|H(CrAYUs2Z4koW&&dbTDzU?V)gav_2JgQwbsq7U?84w0A{`15_OKprg2Kl}?2$HkX~(@rx7)W_(ApHk#mA`rGt1b@1{USFIEwpXZMNVlkZK1<VcxDcGx7|#g21KlqX3l$9xL8ekN;rsaeokU?kBc4R#7)KCR|EgeppgBn@_x~SJfx?I}Ua>MN0VhZ4*lx=$I3zu=L)ikIc2M-MuoBq=J4B)rl^_jvZcmf)wSdy@zYJqQ$dAO{$fcNiw^%6d2@!`!ZjFBGEq<bENU{PYso-`JTM|-_zmc+@oH2>vtr;X6l_lk_rfkNeQItm7OSU|G0>$$YCe1OdnwbGQ08CI8!oOUjBsul`=Z72<ya!kU8|CT9y*nOvz%S-yJBRGak1<?`hjGypSa1bq0tgg8w=`(7G<n`RQbZrBv5XiJ=;=-7-|z8q-`OY^*+mZ@zpC8M@fWg~Z~!^RN&aNhNBEMNLrw_Pw!PRCyvojqDgCE1w*FoCo2h261UZ3ti^p5k&aKVQg7&bx2?d+iQ>z@^xFv>=AvuJSUmNjc5i=sxP(_--k$RoptWU2YyrNp|_15KJ-cZyhmwqSc9l#B1TTBIuY`8R!tOwtfdUf{jh#BWwq#E2}*gJ+TqpK^1^stmC@AM5I)fE}AMS)TM;9h_cI988l(Dk^dHab1AiGnkW45-5CpIxC$=#UsyFodZy;i>6F>N@g`04{f-lL~a5%gh|e-wU-_^+cjbOO!9yHd81oN02uF=}Ghw8fn(Nrf!>0Fc{}^J4!$vKU!M$WjDTJW;}MGeUH|}!4)1Wl)R5Q&S?E<=#-&7WakfV*1FH=QWNWk;Z5?`f{IZgT)>N1Tg3U(>5?JhFw`r&4G-d!`L#vU1ipX%K?&WBWb-5v?#J?lBK-?Y5gV$KOiobNa!OSLhkca}pvaxrAyB9k`Zm#aS&N1;ug+y}cU9<4*b3A$1zMAAI|&Ql`#Q<>mavB?k~f5{@n@pSWjaIt0soNCCWSb?K#W4CglUUh8!^!RM}#<_Zp+QGKkecyF!RLvj*MWp+-aHdYKWP_6qM`OI%vSAAt|+O{{9BDQJ-jU(vgmOGTA-s#s%soB~Fcn8IDj~d(fHdNnR(|<w$tq$o_8`+BV0vl(mZ4sM}cK;1y@~h})t$3`NKJCq0vEo1{)92zkne^||~gEU+KlJofO0y9$!}D9{9Z{~P&mC#L0be6BX)kRucE&n$R2a&<tE&u?11%6@ZiL6&cM1O6*B4v%a^EIzco5g+SEklxRJs3T+!%!YMyRQ%++&G@OI4(ciVImd4F+zUyFOeT>FeEz<!Xf~8qH6C@E4Tt}0Db4B}p@jXeI8D70q9R2Ln;Y5MmYr^V;{3g0*0{)0Cof`CCd-TyVb5tufEtPrOKK!v(@qPv73RD^bw2=_(*uKLx5qy@^{qcS2RjY!YAG8|sKYVhTm2!rPG6<%4p7u+UA)&`RM)L^_K?qUyqct40gAc5RP$aBVCCmT{I|-(XNO2?Yc|9aME(~B0_$N*@{H-tRw=WB%G72b?jKjs)bcxeevwpz7Q;N8LYs<lu0diCaC(VL7F$C&ZQ_Owdt;=I1zAS>z364gwK$%H=IGTkAx6P=@y=sn_px3JJZLXq?GDvtsoKzSZ<;6<)ou!`|AqGrb<S)#F1q@Fy@Q+#QcJ7>ga;bJ%BRgcIb+k8VvqYnrC3W-^?d$Ni)KjX(&^tA!-J8<FfzaRv$<YLY`EOcrRB_Ul9fiBJTU~mdU?s#kvDx0M7_8#rl7hi8=$*==J+ORbm@MKOz_R+VpdjGkAC!>w}j4dR$YU^m>cP=d`Y$jlG5ZtqQb$#^Fm05nr1~*S>{5s*X4$J7TC!#L-rQk6Wxv2{d>iLTsxnMJj;VA)><H^A|Fheaeo{y;;q#fpk9oL-b}XGAlz{<wvMdEkti%S#MIk3(<}MX?>(}}0*cJ+#fHx6C`Gq*!yW}<gr1T#oY@{n2<$zf4#`WB^cnU`+Xs6eRN@;X;#C@Ode|}2sHjK5=C*D7gMki%?EJF`{8L<30kPo#TFSblD~jdS;*xGMq0&pcZd})dqH@pKZeOMcxmJS5Lz(@~w_G*IW61FEzvVyh&;!^^M!P&<sIkcAIK}yv3|_ijAML4fB6Y<hJ0{}LH>kr3T9Bf2wS5r+^UTfUHfv=%7V@1d7)eLZR6JEN(Ak+pf%+G4i;CYn5A9z36Y0(Z#U8C~ijoH*NAQ)cI;Q~9v`W0$ZloTZsd{CT)|Svq+`w?Y-THq){y7MFH_3SB(tC7j<oh*Uh4f4Fq>_hbs1uY<1|vW3e887WV#c8^0mQjrdyq$=dvb&E9M7Qe37P0E6GDH+4#XG<^Pz^n8({w|po1|K_94r;h<=s#sNuG65m3vGyLr68P_y|XMn2d5C$T{9DB1qTekrAwe%Mo!zH=GX_YodQCcHR|YN4F4hi?-8ZR@q6dad0usI6#qeFRK4|JnSv7vg9_p(3V0Y;AYNIXD=1;YAq_OfB&Kk_aeBP(8j(8ErO*@cBnmR&Wcu{1?0@z-=2nUdcxPB$apvi1g<>;2QFp<!x;JqG0<k4I!7zbp!0I<V5Dl%&IY5xt%y~DQ!NGKF(;GCJ7`RuN>z&Vo2P{-<EcCcm#(Xo^e$ed>o&@(zt<icR#O+oClPdGWfm93s{V}>EG8pqlFbR(d<baqC0q%mB+Dy@68hfRTK0&s!KDeHrL6c?3vjYW!7{|U|#<(uDmvz<pB#wB~S^omgA8*+QTT!cxYY!Yo|5{x>TI;C`>_HB*~u$WElqYL#&+e1emONhw=*7sw=?Rk?jrQyvtv8BTK}cz}JIQD2E(e+V@9$<x%p->y{_zM$2L)XwJ!o3V_Lo4EFxEXTc_@datU6ax~eF*qOcO`&Cqi2B_f-G_YctUYv@K@DS*~SMw;|j7m!9{YgU^>VtWcX9?Z#+Z;8!6p6SR<o&_!2vf&=(Vzm`rF{DPf209v_}kOIg|AwO<^>V2N3t2-r-EnAZYv+Y91{nZgb{1MoWex|gJ~PFK_EtRrKkFHyCciLAG2-<cx7usTiQ3Jk^KWrithtK<fnobo73Q6yAT7D;|{?p`msLfXwRz@yT{tuS$34}I0s=@L2{MRkAC~RCZvYS!jJauq9Ye5dBlq2%;P2F1`X`33_k|lKzo>A3U8*rirF0fL>@ekkFiVJ(KARRDJAsyiD9ITc9`;$#f&bu9f!4p<_oQ}p&?mr)|7&bLdSHSdrs`8v31z6M`<x%A(zfQ<v1XWu{-NT#!~<UN|!N5fa_6UU9+IgvS5u@*@N2Z=g*kW@(Sjv@XB$IF~^v9$6t3J%V9I~QpD3Tp<%henS31K5Ns++l`_$T$S#ikGPSlJGm;Mu{-@P?+@*VBE!O645vEhoyX!GPCat?1$&)C=%ex7bBT0I<Ik)*h@%@p+0Z@!76$(-HOVT7Gti<V!qA$ji*p<7oqm<r|Bflu=gT4Jj`)?s=L3R-PGtF7smU+ht5L<O{is26)T^g3OyOkaaBu;@~77xTdPT=9GJwMA>v%=T#6FLRVDKDG2kP$`^flB+cb2@xcV$Q*BjQbt9AclxB1BzWQPq#COIsj8KF19?!&(g4qK$lRc(P0_hT4nY*GWE5ysf_j1#=2EgT6EwXQ6vy<y@zlB6Y1^b<}7kDYnZ;z8Ug;2C8CUPnt08UDah11_ybVQ!P+9*+lX@(7p{|GYDHzJ#8a6n;Q(@c-M5e1JX|X(W*Zlyv0TAUqb$a(o^Yd3a(kt6>a^wOpE6QcT7P{)8hTzq;spBbq`$jMZq^{QeoZ8u%dG8|kJF|9IRE(ty2uBX_N@=Wr@@aoF{mA`#9md-EU~250qCZnJ8p#|JXiQWKS3nZ4%OgcFXlDiLssR$<Csn?3y(fV*LsCa63#S@-3`fOI9KiQm}ZqU;aIxAraT^^O^5~q#%Oi@fq1RjfPu$A^I$~ug0g6LM|(T98NIewB@IPt2G@JP@DJd;hBe(3AZ85lsL0l|1VdLMBEujAlaZ!0m(y;Ci<*HlN)2D(Mi5J~%AQ;$l^gX*5@M|vcQEYh6c$7OjIOj5!;vna!;+cj%-PvPM$It#w3mm(i#n%bB;|FS0F9MSjk9#O*YwVtbbV+B+`}kxeuKE0?t7h*v=))LAfaS$Q4C(vfs=i~I%Ee}Kow|G2kA_Lu}&}eL&?$Zc#aP*bc}u2@%5s~Ap>l|D`EpfhJqhV(v~&I6(&r;&G|LR-R46%tY_vCUw;c|6mxqbG7hshFq_13aPbJ@Q1lv+4=iCuRLG|SWlMz^(rW{(bS)Lgi>I!LQOF$v0mia?K4sXbp`je+8Lt2wvYDvjc+N2Iq<PgmcCU^S+Z7X;7w4MvgtGJnnP_*B<$QYi2Oe?6!$WgVd5zo*na#V=t<EWT>g8x0Vy4>fR0D`kE(GE3*T;R~PBC6Izly-4UvQ9CdPuKU^J3sQz*$DK_c_%9<IAFz7n?vn)Wa$sInA1g{3eqiD=+z$o7pO6NZf@w>B|QAq})>pk5I5NW<P3zH)Z9;d~MQo4DZuFOr*%yWd(n5Epqt=d#gZTUfo&y*?Ic!-R0WU(2K;HcNYMv%6Qqy65kqj-*Y5JE6*R}J^A>}wJo*rP@cFX8^9HC>0%+;w2&9R-`n;FwzPqZKN#B4ZSkd}lT_3mEP++n#|WNFk_i>%&%SpZD|TfCW?5=KocXia$;*Zu-9=OK#zuJ%nsNsda=asA=2m{=h7BZEk$M0Ds4WwY(-7GTQe5gjay*!-n{BA!6*4aW1_=vw-i<a+g7nQHIPqe!++S5Pu{9<YB8opH_G#>Il39VbrGqy?))v~pZgJOdAS+CWL^fm25;jz(YAC1`JsU)|@Jin{g_g|y-ru|H2qd&ki|sQ7<-HwIkWL!1?fvqJb5?tOJePW{nFekwGuB>@qBl6zEMpNSe2I|Tl?1bYGO3V{47qL}*7rD|QR&V(<;fTrn>2n^O;TrT1Z4N)?xo*M!?vys$<~#>Zl|~MoaX{Q$)!AsHSubtYR?6DyTroMhMr##=4a&>#3-YJHc+~6T5!#qB4W7no$$LFBQEh1fD<CD61+2I;QKd+5(N0|aK_}9E1=99bPHi~frB#&rz6uCU%bnu1ywAC6*IitA7yw^R99rK|1l*%U08CMo<4?XVz{3|sfnvF5>z?9zd|{cIqrZtlHwoC>^127W)Nm!F#~F6psL1+Dpf1${2?MB?z5Ejv8yA`NuEP{Xm|6SaI6R8h4fb$d_&^CUx#1kBG4sForA?9Q>6%XqGx{@KoceNMt9=vN=L_4>);qeq4w}rYsD3am+~>W=#|cr&OIU~a$iVB@-<pe>|DC2Om9VxLWUB`$@Cwdf5k5V7K(BTU+zHIXK0sY-2(7?yz$uxHSv5MXnU<&W?kf6+$(nC$QC16Gyq8+x8dlFrX&DeTeVHGyVU<A$Jxe^MndpeWSZ47Ed(Atd+e%TtKr~h`2BM(FZVo}fSNVDdl;Qga)79v_O1yMNySDzWeP(U-oCMauIZeMNwsA(y$kG%oPQCnZU#&E;R|z|@$;Lu61xg&u2rc(<7IV~9-GS0J_=_-ndh1+{l^d0N0C?mmru;X_B*`Wpe^=U6)gR*k!Ecz&tcYTBM6+9?Fn9hC~xi{ZJAu*ggW_eK0|bzU)Z?5(&TFcavE^&uVB}@Z?j5!&y2)XZGEQsDS84fI6JiLZ5Ik(3kHe#N4VM73Lf>8a>$$Hd=%UM6!+G8C(5-dWQb%=UnUciEJOc#uNQ>Le;Xs*x7!kPpWO>7W;NUV*goVvV#f*Ytp8#{LV15$hRw(6rpsE|JgNUlDk!pgvr<V?2;*y#_5$u@=IyIrf>1pl@|XGy&4Y&u1CdM|(nZ_?7Z)An*5~7+;2S2JG;4T!nRmW^cCg&M-~onzUIF~3n6-u6u@0URU^V&N!6>}09x?crx_15PjwGGcC}yF^(ZY$KOY!dO-RJ$n7ffDl1pKRa!*zUdKe;Mi{J5$CNBrMXuUZeU16cfZ8mzN1m*ZPDWM7A|e&)_2m!k@u@J`mI$x0js<Vl!(e{ojeVGl`kl-ZuWX9-fAwC#8410x(V&PvS9m<B+?rsc<VszzC?Tb-B<F^N18oxW%OI?dN8@1SX4-+4#ZCQ4}x@E%mt`>N44?%Fk&^9ld!r|B97c;p?ZFgfA|Y7aGcBKk<b(Wb}D_mRcg4rlKk;P23o{BoN<nRs=J3x)IAZMYnx$TGT6vdKrO6m^6#+%cLFlg0wXhB+Ej)f9_lQ>6_qb8q?B%N$#D6#PFjf%`1e&@X-~q8b$h?9+fD!_;|^RJHYp-f>f9MASCSi)E-7FL->ij=Y`={Q=~ag)@-VId_C|Fz7S_$O$ZMi-1obQJ*>4Y?PpK!Mn-`zWfz9sAX@%v<5~VK{P&m)j(0u0E=k`L%HEi?kOvuR;kYvc~hk`3a@8E5;7+Z<$}>>9oa-W9(ah?WEQpzC7)rjopaSAY0Fscz!>3#)x;^bQj?!HmBepx93Uj16L6XR80v{%U|aejee*1LR78kR3R=BTwh$|am|VAS%E!JoJvB?2U_Rc29fAo-)(wCa9jkK@M42}ehYk(M0H!IRf2xMY>j1Y11Y=N1%0IFjQAIbxxG<xWm!`fx55;vz?&<3u-xZ+NZhhnybS1U!y4r)?L4V-MW92=t;?YRSxuyQlW_v$SxgVFjTb#bi2wPug_|s6IKNAn6ugPj-W8)VVJ&H}g9%@C#y~vS06AApZ-nD)=ljMT@`g@kkbC%NE@f&8iQ<bYZ54fli5r-RqOczZ|!QjahU9B)p6eFS~dk+sh5t^dsW&0qYpH<1wka=T)5ptVLfD@J5NllzczthJqgRh@MY?di$|CH}2*BM<N5Bo5$xmID&(JzFA{{-nopfvM3=E9{zaA^(f%H1~oBf*;Zalh`bg8fTq**z>xZV3oA8Bgu71Y`b~Liiqs`JbbOi*QvaET^@np%UZSdKBV>i;>5>Eb6i%I{23W75cUt?74TQ17BG5k?-a{v8Au@Tyw2L<>u%t=Y5Rsh%UaCdf)XjR^J2)5X+6~ClESdLwVy}0MkF!9tJ<Q%IAqa#rd!FcN{|M(oFwPL1f+bYGsJqp0M?jLstRQa1{T_gi~g<MDw>isOgX{cMi`I<T=7qm3kJUQ*8W6K^LcRMgU*Sz5+B#)*)ZB@FKlLX3{G+v$le!a2P;zPn{Pl6FzI>YJ{;1NL$lrB>}0$=yD^#@7E8%Cb323#Y@1(Fn8C@_~3vm@l{^x2XeTo#!t@#`W6egrCBC0vggI*Mj+L!1ne_h<d#vyRp_Y?Sn%Gxmn3*vG${`ClZag(*=WB~S4MI>BkP5G{x0xZrFoH8$sAlR`{U5#0T7F@fhTj^c@<aWGSof)ctkpdEj(8;n;)6$r#Kz))ok_bm9Io0(u8b?*0bZ6R>Qg%r+|ilr{O6ieQQ9IFQf)NMMt`jhiwT)ZY*6-n*zjc2S?KL?QF3$^`oqUFH-gX+F4_7rF@WarBB5@`}zgw8O`hrvb)unIWI==U*)K3_(E%V9PEs@u*Bi1^$Se$oR4^tAlO*AEB!7`108HA!kp`4na$y%PrTUX?hxAi+^J~jXy6=}OD2;b5&gD~rdux<`?2!gzs2WazFQHcdcWzYmlulQ(XxIsg*Rnw0?BB$?jK$AD>FE<r}pQyr%M&@{^dmEat1ulUvy#<n#`M~BVYl+Ke-0#3S?X24~WK|r%Hs&7qs9}d2cL|)}l0u=U7_z!_BZ}9{y7^ilSk2>2|bgD==vtJ|Yzo;<554HZ?XqGb|v_5S$>LYvc5~JjwL5F&0bRcLl-&;9=maJDRsSSEzgY55mT7qoo*^8rkU|0>)W!ORmZ0IJZ+}s#2yFDxM=a;cNO3E4Putcshh_KNJKay>sY<DI)~*g-e35Rf^Y+oQ6l8IEgY#j#NLc`^&uv<P=QeSgcezpzBq}4s)R8DHPq*oF}oA_wzi`c&3kI$wXRb5nNApKtYTv+K6Po=ei)`-h-T*vxr_9N0}!gpS~~~6%;;?5nVT4D5%~$$oVRvH8}i$>gKjCO{Z;Z!r=#~1ak{(R*)#iI@tXeI^y(r<<T)9nn++@)l)>wID95uugBaH4~@a&o!_s!jS@2qT9qhe`=oQ9s<LjTywP-*omcE%?ZA8^1BJ@}xe+OhQ9YyhpyYSCj{f_(6C(XFvWqML&h6wezPbCPO@f<v9wIS}W5KrRA^5`K$lIOs60#U;vp9hpnQ)%k-)g!r8EZ)h!Bpn&ji|^Fnu6hWl%rSBb)GXrxN~%Jf~grZX9>2Vw$OhQv9z->8A4vh%<65e4Xnudnb>*P(jynZV*amV(^b8_%A>qLUDDd}d@l~cXbNhzc(y{;oZEg01F(Zs7+AYLYfrQ5rf=$8f`>o}d$2i`&MzKUOBlkyDYz{N@h}>+NpnWA-3blExS$7|JYutjT8RLiuhvU$pU+ic?>^N;aCE>0=(9Z8=t}+wE|_j*{nzZV`ZAIu=5tM#=HFP&Zl1G$cP0yd^Xf$Bk2u{z?2tAJ(PFKNh8}RnL8y+z>i>W~7&86C^4yQVc(|Ajb_q@ga<xSqoGbWdqM{199eQ5qksKjL6bbHKIT4&7LkkR_&6v`V?il`>krNw?)WjX`1oG8+isi`xG_Sug^1uPZp~W{SIcqZN_S?d{dfLT&ggq|YJV|3Mx6Xwj?}<$nbUJ`!@l>bbe%R%EaVx;VWUA%-Q(p|SMG*6sL!+0{t!SPpq~|XY6ZCc;jt}alHXj~;**y6^9^@kk;ml0qId!nbCzHVcbbIMMcbptcMGKVU6!Bie)mfOKfu%RFIOIMGRYeTXFwP5*e3|43x`N5bqp`Fe+7U3zJ*^JZCX>%rPmR?AJbh?%2V-*g!sMj-uy#GH8UQ&R-!}N+7<|vh?RqkcCK5N+hF6J}k$G8o8!e0jk)P*iS8y!JwM^&Q4uhtD&UNf<pz<d`KBN!VF(-!c?7^y{Y;Hx68MT9Lk9BgURZkGYT7lzL+vJe$NQ#$Ofaq6@ajF^-6r!d6Op9E?zb~)p>O5~zPJW`oK})nG_&xBsyKx(a?^|ti0U+t6l%9KoZ%ryj9z=sB$TA!48A4&G_GzuS{@*0sYWJPp%7+a-j<*p*`B)%rNybMX{}@qq>Ls8U{yhir7)>B1uS-2O<ui6do#EyZR&S~^Rw_M5Ky$avZ#XGSi(F`^w+h*0YaaYBVtVw$cwICgVDemn5^`>kPyiilhof{)Lc0WC8>knt%mOxMPJrcG<VqpBn%$Jm>zy1F_**2IcJoDKF-9>Ji}+&V_(@Z3N;Bq64-*@|9C%v`e`S6%EnQsAaU}iECaYMY5-&y`(7SWIVHZ1zTWT#;y7=vvFkf<90u@@nRh$w#5AjC!>QWjP%0Uuew%_&MmDM@I{rhl%hAG)E#^Jp<K0rvSIMwR>GeMVc#G@PmqA&%buil--OQ#B2;bH~?L={_^d=sFZ%Rc1eIkA1wS~4ne8RNjD3hRNH<4pv{yJ>`cU`YBKTY!N1ZN(inK`B6~E-I5oVd+^5;aXgrp)}QvJdk}C8?fuA@&-4wnEl{w--v8+Pp`?bNZ!nK@h@$2TCV;B5SRs<<|dc)9;)}~Dmrnj_td96G7t1F>v7XN|L9Q_68R8@zo)7`Sljf%;_*QqIB{Ebz^CQF4x0WZ6KoQbiTsDj`1SNDY}L$NqMgQXk6zQOq(^E7WQSU5pb|p2W{MkDr!Z`a+7G4yB8DBijnA1)S~SYB-6%IFJT-asxixsHP=?h*<&xu*H;4ji@N}Y$tv{)?^YbqksaMJu#h~8rA`!4V+rm>FOnQTx;~zaLt}Q3k!>Wy=QvP=vKqc1|siPBbE%$4~a<T?4`)%qoGaBg|SoT3Lyb*c%PX)?%!gZp>-WG&0qYu$DQdr#%6MiWpuV0ot_}(Z}{q=7l2|v8@Fb})9FQBKtXEy+aU|(xzgCE+Hd@UXbr$6PkRN+WyszfaM|IV)gqyJCxy~;#!Z3nyX)a0W6AX*|IZfE)4#<N?!DEf^2=D!zq>+^~c^Iv;)8Otn-$7l<@8b7NN!6Q}BG;~SgQB#rYt6dlHNyvQu@<BfRl(sTnh_XL!xingVP2ogfMAJ3py7lA}Kz{b3uO}@Es{kv7{iBkIh3;bh1Fb*ghPrE*#b?GvD%X=shK&w*N|cm-zM4$r(te7;e;Sf#*0v{ex6zF0SWEY>cXziEIPSb?e%4ki)8#uZ%eO=@+#c2h_&(y2x)DD7MVdIcc`&e1QW237CF(!?uoR85h4TmaNbbE8m~UMkIo~CEvZQ83#@JqsM!O29)``&iYvWVqCe1Jf<l7^WO2DP0-0UczdmTN&8(P*F5bedAHd0#<j#euJlPy}TI&`$^2N<J57Q$Ga*>_VW0DMs;6U_c7LX3D~p5}0nf!smToc(Uj-;~&L{bt{FjgI(4@ngg(Xq1}sOJyP+hgG4QaZ)GcPcD(Ep{nb{%c`o6=<Ennq&|eec$^7OtDkBf<-+ForgbBn<8~@&v0(p+3m7&{)aM|OGmK%ZtyS(a*aOdCw5YJHwgf048eW|?4M6VN)Ap;vTAl2+$7!drC%?LMiD%v+X2$&9H`3CzrYhWHGbpGc65$k!v=5HES+)&a+4bzRKBiRpV=C{q)6qC7tgfA&?Jgbr?Zq^lrgi4+X&Cu$biwSPtlO?ZTKWGk*#!Ia;Br#ZYsgH*m&Du4EcgOvNxs@v;>O*hY-KcRd~URN;wm&`Z8+g=4Vw)j0Eq)1Tz}N^a21`3&-ddsoQUE#2V+ZNErX%BKqsQa{t4jaFL~c)ibav|(2`yE-^KSqW4^+Y!wHbm`8XxXipo4heONUV!z=U?u3Bpj%t4bzoYEVyHnip=a#6Dolc|PQGI`*(q`z5l7^R!cy8&0xuzcsol9UI{nfEA`Tl_JIb(pg0_F)GBY?ru;n&t;X2dAhgI|2D%^6yMiX!;p42Ida)n&BzM2dEi1fV>YOM@@;9V*+!Xuaw>veCfM%^7=Y3cyiom!T55{<m_II!-~HH@!4PUKvb_~-i2K1_^TbBt7y=VYlFp|G4Om6;64+|Uj;*s_w~+Ug@TGUH@NneN7#s0Uko|}B+~wDWl^;S?l|{zJH#S95zWqP+tB|*hIK=4xMsmx?sl5<lF%FY6Rzb9WPgE6E7?QizQJCPYmt4R1&?D)=>$(c#H?qMF|l*I-1N{r2S)*5CPAtac)j;{PX&-qo<2r|+W`fCl|zdvDenZf&!Jm*Z8uuo_KA2Oh)WiY{Q_ICPv=&K<8_3dr3R}Jjb2I-FZiQ7V$S4b5g)zU^b8X1u=&;xmYzjaxY(vA#$hLUt|VlVwY?ub3IZj7xSWOezzu&M*qId32O?*_cvQG++|P;cBrUxWhC9Z8D<$o|1$VJ94Qyg)DVEn_><USnOv-2poB!0^tCvMch41u_U-THfJ)jIntN;9_>X+PzGz2%_>YA8=B0eL+_r-OoZ{?XG&`oi6I*Zv405fZKGK6<3AUEFe{7zz7_xo{!>xfZ90OBdJc&)2I#J6AQ`FlK@m$Wj8a%g=9(Ff!DSRGsOH;rAnE~L!9J%<AXr@E$~SGguRbZ6y7g+wKq+i(p4^fUb%wYUkeKVms!5))4y>H>|&24TZk#&zdptbT2F3I<vZJ>fbRe{)g0&19##klmfDLHV}64slxx9<7Y-{Z)F_9a!I{c+vW`5VK6wb+eQMiyvD0KG`HOROB^NShQiVAQDk;L3=BplHxoy>N_A{Fhe6n#ZUDCDwiexc3flF75&$&sbWbdUO;2mncy0?q4}nuL-h7O^r-5ZfT&q=W7DJGfJY9Uqh$(DRUm*<oYGwNz7DoI@s3AFYSpG^K10E(lDS*aJHf4y?Y){HU3Y__W^Qj$lkYQ%^sf<1#sG<{buViWF&PrgI8od@HJd!YZwm>gKNu)pmRIs6_o%SVA5s9-3|Q>$l$LmI2)`lqXJf0L)Bjd|0ZBI5iBUD8G@6Ty&vnL5_3}E3G1uz+S@{~xKJUPUYc(-|IypU&T68fqk&T`&VItJ%CD<*!Zu)Sx`uVJQMWLP&Q4zvTEWG`#066+_Ne<STKf+bq&Div4#uqz_fEsC~1$P;A%QI8@(&;nxSv<BF<Aep`>kf?c9&;iHPIxgoW?J{=jtkW(Hqs1*1HAp*d)DMZTKG(%kAYLTstDos=Vxa&WYfn-ACmuFZPjOYoA5osI3HFEj30#<eDrA;-~0ZMV>07j3K`Ds7_wBR@&pg{v_~Q<As=4g-<Kn%QWK+y%(uOWj(y5Js*|++#Dc!3iJ{6}NGzyVs?Nb3_;8BjKn!1wg0*YpI8C^TZE%T*qwvV{YD*LMyk-?tzmsu7<1*%#pD3$CLOU@>@OE#G+*78fM)hM=l<lt*;)Y4H!K~SITqahvt?0+_D1MlvD|kRcH-q!rl8Dl$d%hO*SG&=hFT)2ET4Xv1dYlC311h*pT|SRrEr<)8&*V!rET&#;$5@^hX@S@K!Q#BfC}41hKYW<rrZO6oEVYLtnr$#n609h8;@}1W^X$(Yq%B-=IiMA4y=8@I))64>4LZz)fC3j%0x3q*25t?<DY?6vzxTKW*BdmK)eUz&O{IBaUiA<2RMT1u!Hz50Fee);jLqI`vf4JVe@9t$(&_b;3PMeu?(M+6z9?<mPCyE?ccjnRw=8LG61N^5@5bq}q!$lC-wNY{x-#87k>*4SD}@&XjK1R1@@W*JJIS8R4x;g<`a-k<Gx8C(3BbzD7vK^RLTD+k^gsUVymG6VrsKqSIo<L66Al=*bSV@t`0R_YT4w=sT}pIN0CX;?d^-X$uEGcytZYBlI>@_&A&Q!geS%HYPlihKkWp|$N6d};*g_5Lxl)xbajlFxg71{YMVwQ64;+b0EW+ovD-CuYoW0RHHf!H-s`NDptP#P{BaVRI#PXXsxex>Oo`-h{hCpVqRnP+Il^I-rIcM@B)xRpD?<@rEmKI3i8UC4Xe18J%-+>T+P<u%7JQzNC_1h<mn?-esQkl_ZKbGUA_GlD-9nbg_V{maq{EK{)w#eA6dyS%6?ER|`pkfFq@|mjZs0rp7&3&qQ<v-M&%F%9iCfeO2{K3=JLJz{@>x99@v2V@srF~zMogG|YLWwj&vh1ut`N&gF1^8hr`ZOIyy<qy0GGbfjgzaWmj{zmd0`D_mTFwT54ZM@R*w~m5J&+69(mnrD)Qu`y^y8j|409CF^ykEwy7ZBMDTmLRMpvRUf?h2_na!SqfDkFZ%(+VyEsjknvM&3FS@OVz*!9lR892-RScTRd*qdJ?pbW<3`rF?wH7EY<BAyN{E7jJVSIduOS->&WVN)HBSD!GVM+Owc9GYsN;JU#;lQ@J-r94v*MW!ikkN_dclx9J*=x2lG>)K2#ZcF<E1VL|s_U8azedSQ4(srWg4X5O)ERjNXe@p4SQ823PaV^;33s1)XpGnVIU^kqk)!MXZ0(~A;Qf6~A@HP3$b=OK7pW|QuRyvg^k(LaqyZKr;UO-^&1qzN}w9<5?_mU-Y)<W9N9yk>e%aqx0D_5XHRK9YdXqJ1!o2I&MecSEaDN03O`HQ<2%}JJ@vT`QvaIabK<x%<qR$+;?U%mOcEMeCH)nXYd(pLzEDhJXK5>rpGJ6G3I`O|c2IJM5$pxs_g%<{Aq_XL-}e3Wplx&L?^@qiX$&WQ}@hCBsXYW+n-b@+zY^PyS(LpMMu0?tysS~H<lJ(${r`e4<}L=cn|)Kk<|pzz0oil=wtl=4hkh6&nNn?LMVZ;1_F7zNx}<(W`XMUQW)Apl(XaB&s&yvX|{{D?^MDT`s;vTRImlEF``uA0MuGo8_Cw8oT$8BjT~4ri*7Y83uXo2vY|h(-krvX{tnb?+GuyKQp8oiMrFJ&bF0g~7(h|JnewOcK>xN(!`T%yu!va+^3p#qo*5n?~?f)1>!M_t!s5@W7O~8wK>CQ1%`P1U`y|ESr5QW1(gBF**70p#yd!+<Bl<CuVL%8;PY-FI|SsCFchqP?E!ou5IHYE{HBGLt39qMgHi~AYk-)Vn}PfVyS0*)I+nP=Ko(*FNt2b(5;$=mYEKCz;f4)JrpXFoi4^r^?B;5l%njG5;j%-SWSOI1n&UncpUukoiYM|&aQR0V5x<8W2R8s&TNmLKDuGiE{G4F|6H<8Gti}VSFXODn-;amrcTN1_`D*t+5T#*3fIhL5X21L4#51V_TNGEsJiT5iq1)J6}VYFvRNU6rp1-H)4r7RA<s<<q>ifV&HHAjyj^b62dfCe+}v!)4YGgKA$jFG7J}R#%~eE{1k`XYW9hGfW(zV2Lr!5A`hlo>n&0uH#Q<~G(_Laa4&_~zKy4u$*+`J`bTHF!-gatpVlPo1Qj$|hq|GaO&~jiX#+<1UGmNbl`e=w4P$M>8_eSnO==;i~14ybq9~8@7U&#?3?cKOe<{yN-Y5y;gv>*4w<Bo9?_<<fofjG*(kzs0?QFN093tui~>HaX{B0sOpKoU^>tbbsBu}Wy8uDFoWkGSt+ekzf;0>uPoe?pn^KM9}f^*3VOnsNy1UlI>?w<V#MTN^{pso-CWptAkg7R=(|DDnz0;Z_B>rCw9zJ6iM?mH<nD2ilcwO-`)EvulS7tFE-~n{w~|rzAB9@nu2{9pHTnA1W{ckPst&k^=Vnha|aR*uaRiClf`X{`fU2OAMJcaetL8rIYIwPV}!e=9k`WS)pF&pHVzGaaH)i{APjvPOp3fAMXvA^LT{h0;Fjur$95jrTu6~nwDbL)tS(S=c_bytJY+y2$oj1E_#UgX`%)sT%r=GfV5Kw1Eu)oeUInhoE>kjkWajZe4hXfoiZB`Bpq^D&SWT!ES2j?gLJuluH^ln$1wk$3trQ~?uX<NFI{_16~B~il%UeEi#Nxj<)CpzRLZ6hE$^$;)&`{^jWQb>w}fZ4=$BVY>ae=%%lrGgNUZ=><Q5exIrHTycbN01|3yd~b=8o?o!P74xw@f5f}%EL;1g3KW0haJp?yuLgM$maDwR^i9|A0}z8fODJkQSiu@VH-+{`J3o`9rW<Tc^*;od)F4~nRkVl`8_reGA`Y)iIkX|q%L@uABEG0cGin2|6%oV^{Jlb@GI;0}sFqPBqGJS_A>hX-L-@_yY23M4O3Wk#%1XJhaeGO;ar(I>X6Mvmd>9SE)JKjuwA=Y|6)XqI~l0rT15Ws@7Y&gtmW5=?0u-g%)J2OVhL{+KGIVUZgaWq9(N$R=Pr_l{n~Hs$I#Mnz`MN==%*!G}XX;a@6YCm1s2rc5$dC>_5d`ZqVE+AByTwG42NktU@%h_gHU=dsftS!V8UleZOjR!Cod`&d-JXglJ)&T`mzbt&7={|<mPhW~cO@9|#RjXg!>d<_{<aN~|6TqHa*XqrB`7b^tN&c~TTeB0-SFJ(g4Om3hFba1!!miESYKnpW^9dIWCT^POJu+SEo8u{PXpB(h8SsQ}dW0B(%G<}ErRf+9$VYTn>f9j`0e)*~M-Id@{0~Rhn^hFe92drR>ZAk4x-<CWJy&uM6iiddDMb0k5xAlgJZt^*?Y!)R85zK+n6mK=0vZDFW_&)0~oSjNYBPt@ku|!9VBasynfB`?8PPMCU?RSZYiqkjA&%Nm{c<ftvsJ&drc%V@oXooS8o1v0+&yltj(1~IwUnGxIg9TFx$OoHJJdOiH-y5P}TU!+8(x9%S;*teU==FyPHrV-z&lrp>v~wHcKc&e0jlKxR_-u#4#{UTOVO-luXV&67lc63MU?<|VX3M}soj*b{&8%k?;DR2Mal~8fthp%rN&HpXLaLK;&E!2=xin0m97RG~-SS;gYnVq)@%;#O`2CZ@4po!@?1MBR&b*Qhe!Naj&6v7+p+UjBhS`Pz@Nb7aKUUCf8c@Z;(&qvvB^;!=7wSvO{ry>I`ebr+5mRMmEZp<%FJqB)`v46cD%+zE$fZ|%QJ7r^Wp4BH0OrZ0oi9Ts3676@%+Q@HbE!a!Mo%XAd>SfGH(V3E<9ueYb15)jSRNOOpSRmx2+~CH$yaX|8cs&4oH<q}W=*gO;D6yf@TbO&uq#;?h!SS^N3xiOD9{Kwlc_V~t@gp$kct~>rk%buWTwx>Lu_Wsg(EbfBN71I(G;*!@*i-$!*9_xHAFXG7TPyH>C|Eb>Mw7uJaa<$-h_50Urh3rR%W%G6~%QnmO1YIF#nJTS4cd+VIF{b5r4(n4IO38;uqBfk=0|7Cn!dYLs2i&^=$R6uj92hQI#@gaTyrX?~b$Fl9*$hew?y13}}d(yb#27hC80hlLROUYlrK}p`RjAkW7%>oM^WR?$t8u;#0X+CI&C}Nin7K^U2A2s^9qYA*{DOJ4nhG|H(c_21fUHq@Efh#tpT5%7<l0KR^i{3Gb@m{7ur9N~|77S$`?VF%)vx9E{%3)A}-Y;MF@PG!Y>@E<NW-I1h4M9`1<1hM$sIvwL-HtTN^->>Qu3f3zeodPihMMyiohF?0jD=Gk%e8cEX5xR)+j2{+gG&ptYg?WVZYWKSES?V`EyBychI^|?jcdjAvV9>9nZ)M`F?a{zQ3MKN(`3XKy(an(`^VIn~Jf>uoJXpP0`Ee)yIH@fZ*Jpo4FlUq8tj%AsW^10~yjw$~%*^`!0$6bL^pxk1B^H2bup{J)KQ{)=*O0>w$2Gzkr(3GdFv<xy>OTK_(qfBcmBYOOPTO&+q;ZpuJ6>^GeE@rrrhzm=r(5@h-!}{LuqV5XhUb+qtU5>z2MSm$JH&x^6bNrK;K5oWp;6d1<dulLXG!COoK;ods3^+o^!@r*pNJEKh5)Xj!@*cDbRvTu6Ld+4KU#5w+1<2-KN}lOuM2osR#DaXe;|8Z1N4hArO~R8y2mt|1D#BXI#SVLG^};Xz#al-^wv2cPk$j+6{xZ6&%GB+cy%XW4ud;wGk%Y@dEkcT*BNw8N#AO%y%jOM`MIy_wdRDOQtwURDQ@d%->XhEYO#=#uiP@h4@0f$F*lLC|UKsls9rBNTLo$GvpT(XBi*m(omka||n7SuE*`YTCa4hTeRStaLSgJw5fAn(kVOO?nnLxADFWO)8=2yEoh4kU}Z*btQ4VS!3yv5}9j+Nz7vnfYzlootJbQJ5B-dNj^*aXK~p=Z0a?u(4Yab2U7x&0}Dd{h9_YwLH!a<zwZ#q1NrI^luJH^<IA(0jC}XX&R02j8dqs|9)>++>4zCmo&SqUB@UZ}WEJmDIMG?<6YNe9IBQk+lZq6~G^ncHN;Bje7scF5mX!;Q%C*AnQtaQa#D7#KA*$#Z<CQl;q+nHI2W_wK=IeG(w;#jpc+$^Q*~AIAAB*kzbdeBCnd-<N-^MboIMf2|!O4``G|x>+ffT!sS<YB&wVU3T8_yhj}C+P#GT{phf?*-&L<W^A4IfKMGRL&tg@~U;U~!y&2{$2F#B}0s7-IsONH@ytrHuq$){jS)uP3L~4#FvKdJ{v<Dq6>C-EjY>q(vc^y*0Fd3a@hAT@m=6%%>JbN&}hFvu&B&7J0GWni?-zCc`KW4=}P1IFU?Q*<hj=bH2CmCo>^hfH4-E_IYQ5K3=n7qCQtoPfdO@T$|ZZMBovTE@K;)=Nq<XEKrnr12qxG|i>*Q+2*LnVQ!Zd+b=ZhS$<5;Ro8Vws@sTM4O%zmjr;O9G8_p3ZQ10{$h(Tdb_b4l(pDg9xy%tkPvYR?1#Y5}JA*XNub_BZ(8?*w&cy&XG;opbB_k_pa=}=23pIh;DsygOC|sbg-ZJqdnUo`zj-Pru(^!8#tYr!^*bTz9ycE*RrV#UBln@g_;kIe6A{}yE~Z|g@jgYFP=W*G2lxC&k6RD7A@!)OT6D8XI%i=4x)J;q{70K6oey9j~(K2@t>eLq0z1Q0=SUuqj|>(#k_SWVJZOwT?PT-0z?akI4nrwAPa`(#VCV`B!KA!%mg6T9B^|OE2f(X;i++;D|>sZSpf3TQ05CDxShs8RrYt{+^WI}o`Rbtf@F$U9s?T`DJxXDxw~}kG2Soe#C5T&RRKoN@DkUR*qD5Yhk3=r>CwTZn<OYGg%CGAo@pkcM>w2gey%y0crTCW%wZ5RW&B@aTQ(U8)Q?EX0Cczx)^mvHM+bn%9#OxDNE8<CWZx&3B;_7E8`jl7q`l>KQdseb_Vu3M?+^-t_?PqLPT?ItddT~QQgeg@Pjzg9-s8N=UKftfI1R3E9d`4!F%R2o&Gs>niid3*aQFa59t6&5t(02vb^PHA8&qO=qij7>^b%wMUaM~WPes};0I!d?BQj(flQ@|v#d6-@9DaxnaIHyfi$*AiL$dJ6z(;)+`NjSF`{XoFODA4PWmrLsuO6MhPzK_ZZ{l?Do{~fk#8k+ax|#dR<hx!}4I4fB!L=^l<i@QlK<$GhiB%vbN6Yg7)Fe<`?plRY7zm>msQ%W~opQu2kdxS{ZX1Y(c7>0<7k)e}KwU`V%bPSD9W0_lExT&Sp(r>p-qzG$*XSEXl$NnuDhkly2J`58VjuUH%1FN7ZfQhX&`nR?YuGD73b?LwL@Y;^`eXMU$~eg4!@ZUXWsbH%r-pgWaLi1Ji@WQEhkIOeQpa};e0Yw?@98x<1&*D$GxMVaeaN!U0)Eu$muG@l>N2Irr%W}Yf8@ewp9eimx!XW`A1Q=}>EVh7i!qN~?r0^;uO$fPR2o#-q*|&39&lz*p<&~ka8I-Ky+l{JYog#=W%|kZ2%)3BoRyOhc(G4uJaFjNXx-*=icuhGI0a4Y8%N&%`y2HeacV+?bm?hb_iVl0<5YgLBwb#=eL)<u3~I6f?)fRf5fYWJXOQ?-O)`b???ldEn`b_4SxC+;B4H#ALyVQCo)Z^F>Vo0HqF%J&m7qEv+F-JQ3zY{?zyKgRP-+S)Y14i6`<)VJDdH^qyWE{;=hs^kt!}dz{v&R_mql+=a3Nr3Jl*c>{bJQzCc;V*CjhF;i&1c^JFIz)+AWSKt?MdZEUPC)d(+Tmvl%8_Ru&9bs?O~#U1xafZl<_=MNhzp>}Vm;94O~QVK9$P)olvJ)%j(HKpbP?T!y5sd$Xx(7Mrm2s=$?S<%Y(CR!>g1EX{&`TRI;{l(K??)C7RDZZY@rteT}(#EAQ@JvR`d@wskr1uFeLC-(?}R(b9e+k35*1a18y&Gtsu|Lwm<K81+u);^fK!+KM;i0gzJ+>|J-(GSDOH&fYsTUFrIMM{hF5<6&zM8lSVbv@>D%#C?h3}+_!mXi(%bzA0`9SE|oC6qLwsfaPj=aqm)wUj|hWBh!qO+Is-S!$@c=5xLTNl9A5iq^?Nj#R{~e{l#}<6yV0$ZwoThTTSc{oKi&EFh{j%B}K6*AG$7!8YmHa|u6t%<a0jD*Sf<3uEpH>p~Ydd?3*t1A8Gq?)I|yKJ3<iAOH<_SmS^7<O40X=dcamZpQSZipZ%r9JU*TYoixiiZMW|KU5-=>2TD6ZOEWAffqFG?=;m!dEkC6jkV+;=E#GZ9=F4ETK`BOY~&EODhK5DhNHYiCmA2NYYInvd$dm)wPx)Sg4;!B45}tgP2}&ox?SBv>zO9pQwZ(%u?sr1xJT!4Bs+UcE2K!}nCD>_(biZk3_?WIY|}h{>!du?ekEr{es`{%(e3AuKRI?E4ruJy*RT^yi1Vu9YEqfn=>jVo@H%yF-TBn$uJh9!J=7$(t#7xYU2q3PKSFeiHkg7)%jBqtA|FgVm`21UlK;=P=*23(xwdw)C8fWZFS;}J9^Sb+Z&(zDExCk9L<e`d`Ym^%6c#x1^?+`gGnC{NU=X-Wumya)L!S>S5SArl?PYlFJ(Gi2y|J}vtd*PTTjgy_J|mlvT9#$zCd}A&!9M{kutOnkiSH6D^waFLRa7YI3D?3&FVjxOS-hYzxiO<npvw5B0+q_m9h~IXEwb9WYe0Zo0?RU%CN<%bRu==I#etb3E~gz<X1$XIZwz0XuDA<yr@)t!71{tRYbF(^9Fh|yDr?Hypn;*0;7q@rC09ksE(G5v&;oFB!4|(Pl$CIY8P#ne+P(&w2Yzd{qHo#m`MljHDHO;#mb#}Y3=VV49jhB2$X6&s?_Ch^8)K0^8Qa(e@S3q{sl#wn7C(@yTtZn^)3T3*%ln7z<&XUskWVA8LI37gXD<pvX=YkDS9}`>i5oOP9jl`zH{w{OHK0$ij2I{Mx4<w?9DWQEdx~XRN&LqSiP-{WmJSC#n+QrTyh-;Y`;HMHh_`ZXiCS%R2a6jQCB9dgul9u}G4S$(TN-WR45jD^5K(JeWxB5M@j$ETCaK?wwUhSC@kM?WO=uIkJJfYo&XY@CwD`Nq>hTu0uI&&`qxZ~L!%CSM+!CMP@@1N&H9>K#szET#@8TrDDvOJKflq{J-)9G)OJ6i{Eei8EaxU((E6fNiupic)G?>{007BMwM=oXUe;fsP9zjLwQ_ay)GsSzr(<0dDHh)O$H?KC5?CY@%Iu_iI8Eys;Lx~Aq+yKP(0{A6>4M)}0M)mNf;A?`7pL!droM@E%ZIpxe3B&Z*T^IBJC8|yjNl`O%n9DpP>ZN*w1`+?ih7%E%SSw_|XMp8X9>iL0G-SPE#i&KL7oxQtvw~9U_7mA@jw881Y#BKHo43~6uB9gy5kQpKA4b_YPIPIRkax1jx=7NNJM@*IDyqsI)GnqRqz3-|Q@{EJo<N0R>nTX4=c&4(o#gV~_B~aGwmuM**YoC{I=rCH%;GpOqK?GhdTg$77JaT`tifLg?r7H3u&v95cQz{1aNSGd14rXmFXQ@9wwVc2CbM8vo`)%>c`~)F8CjXF0mZkZ2Vlf-RW3~*^~tnf>2|F|sLY&yYzWALeMj`;bGJk)*{qynZPCT>3RV4|s=VD5R%Otw;>fR2G+d$koCTj$U`!f;d_yvtlr&N2e-3l(X8wIpbOB-_csoOyK*Nq&$j--3S2D$`F&}p<|2k+wlB~fy)$w((P98NYkRGGqRq21W>qsBQCJ6E*yzfsEoZ?o_kQ(>Qa1}6W*D_FUT9hM&`+GRmh4kvZMNtjC&BRZ#-|U{D@HEUM0KuKQePiK%8#&~zJFHk!ELO|qgyzPrP-YtMRk5ox;0OPivp-S-!~^`#2Oy&6s@3=@#^0T@HFcWz`7Q{JkUaa_KhfE-`R%^jW_tEW&4^Wi*Sf25H1edrR&E}|9ujB11%EW+n%rv!B4h3V-B>zMsjHeqT$&o264>nn$`u|@zuYUkp!7GsFcr+Bc{RSud{apnM?U`AI3?y_S^EKqKa$11L~tPLLE1XzcQc}pFjR6FuD|CC^7dTJR%I35CEN*Cg4wWej7VE_Ffw-C2(C%LP=xVxOlInez$(nZW>|M8|G2OA>m)B?K4GPt8w8+=v!U6iTj!OhMOlnG(rhA!oLx`7mJ{)*rVi=y&_h(*OZ<W6s7raV_ZG@&1KHtllu)u)1A>0cD}Z8%o5`1glV?7rh)|AY;4lSNxyvX#qnM)Sg(h~fWv`vA4u-&+De$AS%xOja>Ur0};shWx?P`$6hbn5(V^xqT9Ulab*~kDEZb`)=sYJG-o3&7r)kZ*h{Vy|}tFp>T%F8;VM+qlbEZjA0`TTexdU9hv(6W>%;dP(3@N8gqa&O=3(e7CIRbVusC#a(lafYwwo_=8bUpc^XEBdrm{$i6^7jB{z@?&Jn1PuY>X!eT(NuBvjxSbEBc?>G7&sZ~W?1An~`4~Sua6qY5E&U@4hZQxR*;2k44Hn(Y^z1utDif=U1~K*Cx%ITYEW}bymYl#{|IT%F=W6-cXt=lro&%ZuQTRe8DG`)Z9>;z#pV=g|%E)uk$zuoHiGsC%Yf7b0)k=#+UgRKZKpsnc@&)Rs6nAv)sr7G>y|_-Y+h60L5Bn}blJ=?sG|rw?hGQ`-K%KPJ@q7%yIDa>K`k?LX%76%lD>l;}Jk{(n@#hYwb^jo|s|9ix3iZLV7zhafwSkIz8rg_3Sh(h*b5S3(DqDSiQDGH}7aDnG$)&6f9kj{k99~f8|5}e*Euf+H2FL)!k{{i2baSw>Tg$e`@}f>dnd4o>Q^tNRwOryYm7mP2zS~0LDx?>tWnVka4BHuMF%hLs0C0+}a#864UR&!?fZ;23xwUF^(Juwvzz~=^{lQiJ95ol_5A0dP$4Gr_VxWy~6fY1D<V4h^R?9{U-;P&tj=Bcj%7TZ6{6Ja*Qo`*ex(Gc9=DX+y_~wzW0>;Y%2V<Jl$9kpt3U-Vyh4_!0v_1SSuw6VE;wm5(|DrMjGVu?pPr$UXW})T9^nt~>Y&U#vEquaE5%!sYeg=uc7R7;KeO1hn1!o=>seR%hN_ixO<@euSBz&@<vSbqbcFd~e>>ctlXQ^I-@0CdTJ{&)2F%7l@2Gw~>f8ew`A^T~omTrp^XQ?>MmUD`VplgoFvYBLaD`n816=gal8*|rPz6Kf~|G&8afr@-Yfr9m6E%%(2_<1O=;^7PpUXnMP-*!e4bca&EyX5c+)6kYQwb_bnp4n$owr!tAdk&D`f$rzkIC|DfSl2f=#X7B(qRe)Dscf85WMP_@;j-KbStrQQyP^-uOE-=@=~+J#kuJ+GIY(9#6mqRApw0u%jlt)zM(*1p{v6WuXk-6lOyZ#b9!5B@=Vduk@;EAwaw$OC8k1c80ux!R@_JqM85!w$L54|ty5TG<6+_u0mHXTFc>2~IB=9(!L9)jY8irOen2z%nlrd(q<di*~@Cq#tqWu&VuGf5sU3K%|$*pv<62_Ktb2Q1hWQO%B_#IoecugG<(8tN~d07d86FaDYaWy}K`}#7byr2AIvFn-k*b8iZ-UMg?R8V~gc;snDrnd-KMF52&rjW|CgRYsOomVX9Y9%!;MXJ^Zg9CN$9Ef8AwlOshzVoZeti(XSXup1RVFXq^n(k4{-po`nT3O)OlgJU4w8@2rdIH*M$6}JLZjHi!LU^(rTqgH&cf^m^+(!qbu#P5uW@sr_hPr95i8W)z({eWRcTkII+N3zYunhydAp{oRVS09&SiIsm)pO}@pDv~$_qQ9E->wQRH$8BqxU7DcigM@u0AMnZ*O^_VB4+uxw&xmZ8&7cPEPDexB;co~3bZwWH_EllgCM0wLR{cQ!$s4Q`NP_8wOxzi9{bK^d%lMX@lB5tmK|f4>}Cc_z`PfVRLZ!`pQJd;Ey3{so_f_}yphs589axN6)!CK5qNX~N|$+N+~mGsWF=w|(>|t~qOTIpuptkpXXdzD0`j(^m)aHQqc{z&<x>|~V`ym2ti7+-%3!N3dX=jg4oVJn@|p~qyV{ji%$dgL>QDEk;lpNX|8;7xJ2IZT@wMX|hJG}*Zi%?4)5l_Tors)?Z^Jrr`Sx`$)hv6=Em9K;e2`Q(NZ7Hoynw42?oPHF+QPSa?Ep7<##`v(J1Ch*;KhP$lO&@G)CanzX4%6`?J3Fgu9PWlG(E#gUB+h^oUKW>*&+-;3TPj#s?0&lHw=wymYBL2p)HhuCj<L*;aAh<HP-oD48pv!`ryS=)P6XhD+`{)OL+1825lDO0p!auC-7K;cj%2U_f%+b$M@7OrH(-#zFC+t7TGq5lFbpn&itT0@p}1K<|kA6S=5;O9aGNv-IyDq;Jwy5z`_kq9!687m~UKdleGC82P9$3Q}htL@j%u&1bb)|l9X*M%dZ8A^qiJ!brM<-D(fObPy(u!zr}{4K0|WOC*>hlHf`l+Vzf*r-2JF5Ac-~`5^&I7ch=#-N*}o>3ECmj4!v3Mo;_|)E^SG@OJrK@ih=O@T?BUW&5AiL65kPq^$R?nWgeU0^*%FFvi6(d4sE?W(Q|Q#2ft+|;Wk5!#{XEzdMJLCye^Aj`27A61EI_+#tDY5=i*H+<!_Fv2ZUE*lcTfRS#xCOcW-ijiiF&F9W+@u`a*TrMW74Z2`gOUv#@k{)|>#zaG{a%WuXS0>fVvoFk3Folq+PzI+F&b`6@{)Ka+f*$NiHfm$oZnr8SHmM4LzR*fSPdAt?J@<UoGycg(}#d2zkga#fF>3;t6ANMBuN_GeQHr(`9*D^#J$!(XA6yVyE)4oy857cK&|frIY;?m6qxW&A&#;E0asp&ef7p|olJK;i|!SwG}x@S`i8{E*W;)tH5v;P!fiTyTJsngVY~ENlKtyLmNi=i|;q_jAmDK0KRe76TeM>pj;vJhr38Jj-Mb-Yq+(i>BSPxsLd??~dBAlN*4XL`}DxLMw{&QOb2vJp(UPZ~X!=T&1^LJQ;$CGI}_gkw45WQ(1J3I$raU?C>7P#QqsJ0WEQ<+EY`h#1tCLz$Od?jw(NKX-wD6cK||W3B~*7)?jw=hAY)^+mY>6&YpT*W`i~tF-DbHApVF#W?7~$*NxLkH1TTU`Uo4~oR)ZWa<i;ymHVjeVQFszHXRWo6p2lb%w$#Wj|3tdok`BfWswy^P6-Nb^w@dP_%K(EzQ4XVae71mmM-^>K8r>%b{;dzV<1xwb+3ZOf@LUd^&Zbsn|n$=kGedBaK4;XfC1<!`R=It7;%K%`i=o+TH|RMm6NM4$ICsZ<;<xrDxL-k%Tjb$`29;xeymp>q6XV}-*Zn-WqJ#CAw{Fn#ve58G|_#A`hbMU3IsJC(L0V#fTz}VtBha5?T1WkcoL+<%zEt-y`|s9XC{H^I(`|RmTH}wDaZ+KGqd{Yj54ihW`lux%V#zYji9>wahu^<!h<yp(?((Fews;8<%Ys`A+3&9LBYJMnJU02m8aRa+6%grWFUr7&@C_Mm_Fa+xCk)~3t_sS$8bcH<)Qrt-%5e{|Em86pAC<R(t1nZ7tspjfWJPFPAVm@X++&xa<N2LFHztgnzNBmlw81W3%q$}ooH%2;oT>jO^-r0`I!QTUi%R8C~$eOg3+rpxRYGsd&@7W5YxNO@`EX)=>TpP3rSz$cG*VDUb%u|-Eq9k^YV)57Jft`)*=4yXEHy^m%E$u@>Ps!xvH!%xa%55t7R`0NY}ht?oT;M5P@?GJIe6d1u{$LX_fQ^3_j}#3TQ8V)&jvo=ttPi1@62&b9#J6l!7S@CsRJgk}VbZeEP2X5Szrt!8l$Mm{Rmi-cq^Db^028d>W+A%EsFq2Oi>v4B!jB4XnA-^tXDmorU!!v!c@-j<c00mL=+7*Bo9>C(rTdvj^9!O(3k)4(N4oY!}a9*va)aew|?^#Ttf4+ccZCF4j_(FIA=T`X<>rYYMHnipZ~pk!ykyAVMbWQ2h4E@MvkAYx0(KbXKOPZZrUn#A@AxJdF`u4ee63jpw7Mr$FQ^!9qZmH-}zW*>w4__*!%r%X+Q2pLsIYG}rBs`K7@z=p=P8TGWMNIGI?w>VZ{ek&qdOMt{h(`>Ih3#}X5&8zP&99vKB=EKxKCL;>|0u3qC1%c$9waFa<j9QxTYqbEoEQak_4=)}}0xke#YK0s|`aBegj>OP|chrw~CVFh5034mY04&%>1FNgEqHedYtrer9%SRqD~N;^vHfRYP4@*lOat&n<vRJmsq@PM+<I*lwkui{j>2ew>bN=Szr@O#*Oso7L{6j+FQOy$qY^y;84N-4U5)Z&@sq<c4ofg3RIAP=ahEk<GxIKKDbZVl{1pM-d3vGXeFI}Je0=oZIAl@!p0*>UMrqh<!3rmW;tq)^y0U<}rVSO6zF1+89Q4ZaNs#eWZv|2mPGH}gWMg5gJbnz>kE08&xskER_;?H=M^cac-^h6D-Q4GhWpzfSCKMgX9AcBLzohr>I69EY91{+%MSL?XQHt^3{djOy8H>i;dJp#waJ=;|qZzxuC8Is&R7-0i8?qAFA8nHJ`iP;?se2Qk$hG}|O9STS6G5Xz-MF1z{vrh><@4*6BMT=Olpc@Ae_nS1Ch?Cmy=rUf8MwdHP}1{yq>K5nviobg9iUTm50@^yo)x_m%%D`#P-;JP4}lU{8LcbDq=EyeDInts{IiTNUNQ)%@)=j|i1WrxUxGU<?olvUX-HW8n<KUbhU@k3_?QB2N<ylF!r4a_m}=O0W;{nyS7gQq8!Sj}TM3q1Nu=aDlYO^g$(0~1zQ@j@(qPT!pxLzhIx$NXus@9;|IzKOCxl>_6(iUA1_83&J-X`#caJlRY>0U}p|-N8+T9>kjLdJu37YdT@?YDH^i2I=V5jPi{D)tEUTO8u#mRqK>rbdW00ho#YL=o_KKnAbZ^_%`I$1eeMds6EKL*Uo!N;+u(XuH*4g$vFEeLn8!dn)J5GF}1wf_7t2CZ7+(7R>V797ip9}_ABnX?5GQw_wuPP`;j9uM!1b&bdv$eO^RCTMMx~o`*W4WFTu81@|B*Cm-hB%+hln8IvrVFKg{HzTb#UJeI=A*Vo6<n$Unr9Gr)qUUaba}j-?-H3JkE#BOd0C_8mr)I!<Z7$w$8+w<YKnNqkg*`@RqCc1Gfe!`fv`JyRz&Y7VAhkDs!UikDg%L8AdqfO+<<Jjadh9ZGv|x(PAh5Wo2?qVz7;tLJA5_?^2G=TwbD#bptAF*S<z_Vy5)R#KhzNi8d+xyT&v40{$U$B)&^f*~%b;u`CJgFwwm6DisQ7&LR%Owfst3H<4~&ig2*@VI-cSy2VPc~G%khhlGDY#%<;Qwy{F2ZjL1D)LuIAmO9zo~4$)^-<+eCuJt88%$RY63tW{uybFOuBa8tR$d}V&I3fw8)+JhI;%;2Qj|(;AH|*_RZWoNId#&_xNOg}8biKcxqBDmcU`;If|fZ0{+C;0a(SSRI}ONm8LQx;q(q4xi&}@Gz>}#u_99t4U%GZ*B}C+N1BYr($(}*}0&Fy8cUMxR`{10#kC3{_i%2!xmea(~FU%J7ft*6a_MUrp?W+X}HA8E{#HK!69K|_srLvXe2F4fS0(K0@8E^;Z^9wT+5JLR-4rx$%-?aSB-EQ^uAfDTN+1}s4cC9l}K`>Z3Q`B@TnYsyv_3BuhrbyWU?BA|+93rUmv``PW$z8^4{d-@!J&*&~y}`1a?F$JJyjTnAb)G9PtHptcz<vF=t)pohB{iW>nrxcD&)e2%m;%qU0QmoPMB#}WGMdDL*ZWy}!rs0kh(<T0D5dq~4%<Cf1z~mfqLVo>$@%CJ@%{ND63$C5={9rUxrX<Xzk7WylGGH%5QidW6VzeSj=H4yx;&nckkqa`=6%Dw-n>Ynf(aOQa-m`rxZRS>@Xze)!gASf9goe-*HuKtqBeQ2ug@0Ae|IIw(PVp9$7EGxQybn!u`0Wq!TY>z!;?+J^hG*251?5H#af25+kQ=MCid;o3E4A)E!yK5abU^ni&lXLS;3tWd_MDdjq>6$CjBz5r>S%^*76Ws87Kn9^Qk4`>vcV=;=$*vA9^pL&nny8(e)ho7{lww=mhav54<PBBO637a(OsF`Yv?JDgN}+X>BQ3*p*vj@k;srY}H)4L{pRTDC*nhC&Hy6AK&-gOV5~Zh?c+Ny8B{c@O^qh_jQmCis}0!@4^A=jyxr7AU8r&E2SbJB@A^sY8Le(+LuN0t$szG%HN^_esaf{ufR-?Re@1s<btZa0!8<oSkJ^vnmI8zqTbGhGrDLpeyTcAHNzihNC%8s71QqW3o%BcF2SU2#5}V3#j@7X{TYG_aA2F6{&BC*e^VvB*&;Q-$+}~Op1{hD!8_y2Zgiio?J%TJ!+dkTm>{H>LKX@dV3#&C8WS@Ge{t1|Z@`j6R5ZE?N8_W@QY&DM3g0p-!ncd^)!GYw--ELV#GY9cC-FJ}P=ej!B?X3^)<_+Vs~j0uy-aVZJ}O8py<}COLtfs65?^M4mq1KpZT;}4xj+?)?AE)c#dBV;Vuwu8rX{wQf+nRSh`oinFjkq!-^T?_^=JNeGBcfOq*>34$m3DfXdsc<W?O}6Wa{S%c*=@BgaQo2^fO{3=t}`BB6+ig&@ca|>~a$lod6~lJn_dKI%!Y{wFc2WjCa{ZIN0l>k_<m*gpKIO*|VAmE+`S6elU1_T+YTrYqeIB`_;*0cgj|PKvJWM1rsTi+4Y5ziOv0N{a-i{3xfgl_Yg(i<T4t%X3LNncP8;N`$;gTsm{tLKNG&6iu2e5U5uL)X-03CA0EiK9-=NO1sYUd>fs!&Uy9++AUB-H(*@rxqVCu5b1wdl$|Bj=Ia`THS@gVOh`TNvW8`Lcvk2A|SH<k<H#%~D4#Kz`+Q^;DB|)Qj`8yV2XH*#b6zN)X_!;SEk!1*=>B)9jhRKW>-@$!qsi^Fgp}L<-7QjBRVff9b1?D&`YQ;$*QnJf?3jrr?mJ0=fmX*{u`<30%((D73`oa1TE2(n0vm!092>F}X@d0Cadkjl>F`66XJBrcWSz|7ATXABP>`vEQp5x@$h9LgLIWdlNqg6X)eq##6#jp^lH!wVhO_7g}Ziy_Wz+{J4P?-t=9MIAbt9*9f=86d2sHnQ)^{&?h(IoqG^ru5z^n#=TJU!inQ<D%{n8Nk-Q^TM!oor5D<HR5~u_p|}BL==4FiufG!I<q@o4M>?KTz71Dsmw9!b{GLk|ezZ%0kXLus!o0G7M{AC9UD7L@?{s&(!?q-&dO((BAUDx>owE|3OYPFw8DvPG47YBhRAn%{#OHjDMKjP8;ukxBF;PW|yh&ETTiH{c^XTG~(5f1{feb5f$Qh@ZPg}z#yM+f6#PHO7`Gfu@rr#mxrxZ!UUY54qXYCwq9XFNg$0DOrTfmDFm%khgg^||F^e^h4Bhsv&W0T4RNRDhDT+IUHI<nb)emJDC<!FZ#7G2_;ShJ<&&+`LtJ-sPxHwxd%a(x!Ds$^YOIq&C@K-Ngo7<S>GeKY2yP<}-Kb{|pQxzu+-s0*<U#y{2;?P%{0Fz@mQTP9#q(!KhCX`XR#6SyA$o;R^R=8z>pL5Nl}!KS%RBHIPTk`U=JM-MtdRT5n*h=8L68t#O>`OM_}e`~3;RtbG0`^NWu6I>v)CQ6(sL2_<SL{E;Eg3%Q$`#lBHU9sqUlGm_iZc&9yj`76_qIGDd9X3*6o8LFOk;?j|f^SfB-uHSSn@g97Y=`nI)HxN#1@98yxKe^N@VPj9~G%zYSPi^p^FkB?Jd9aR0}PlC44NS1yTWMv((vp5}sPg)tkB7GnGBcu^VaML+=+C8SEfextdpOEM8J$sEdj=PtQ$+B6`Mx9py<COHaMc>-%I<Ca@v5b}1xw|UFS1FJAJ6|JxUVd8tyxQ&;Fp6)fD5uWl{Th7?`cMRx2@devnD{>!duYBSqLEM1bx!uXP;Tlw~^p(o@o1YWvU-mB;U~D*q#E?W9GEfANz4x|LeS04P4P@cS^Bx=Uh~+wo!%4b34w&!3{JiyblwPfoX7=92p9>MQ_pbKxdj03yw>=zt7hVb2Abq0>s($F}_cFh}Ha9FtFeAhA36bDiS4*vGmJ{&xiyVDdeafjvit-r-6E|u}EIV6NNY(P5SqM677Rh^J5uX!NJzCpQCFo~nvKD#8tF&U__oe~(ut1=zvcPf5$<y9Z&3LVii+3*3UeH3cLk;2^dE78`hZbyOuaz<9Kf10X*L_+fE0!~c$4F|LsI9x0gCc}PHl~R(W(h9>LSh0^TJy<Faewsp*Y#;pP;$mtA|CCh_cQY*tD3+cuP<=ACGD0!&qvwOZHj>xivu&-NQJsRw?;Z7IKl0LpBqw+DHSZ(WB6B3EMR5TJ5Mz-=)x2)p*HkrLi%PJznrYW*oF~N#g(4TuAfS7o{2`dEfGgGg>tj{J?s>|{GtHVrTBZOfzICXvr?e4+S`j32}G>=UjJmg;F~^YE&NS}@8K~@jBAyJ#7&$A!W360*#v&HT(HIm5M>oAMS)KCHQ2(uyGPI;wcnE_sXrNdipXeE!_lc91DT&u3%H<el8M6G^+{>;!BLMq@_}5x&8t7m7&BH=>x@!wp7zN^CGJJ(>PNTuPlLz*#M?VMf8*(u!!Eom%)LUBxspsY0*+k#j+o}1B+<ZS3&6gdb8<U7dzs-#VxWy>=eHm-9Px6~;ecyHyI`+{T?OytC65Z^HFJ*IjhD{%@-7r{{(nFi=(85`UyJ(ArA`i+aj~?K0n+-{cdr^6(#%6T#B)}h_lzhw!#kZO^a9|?)>^jh(<dNPuA0_PC8Seh6m4b*U(*G@1|h?>mN9qjH25?Ouq)qP>nUkkfnOkX6L<V$x&Ng<s80@XSM*$pj9waUaBQN6dTg!74zl2<LA)cBzAvJVcN4H|dB?L!v3r7z6s&4!bQS{Pz4|j-c5HgEnI00(XZ(|toH_m~y<h0=i+0ol97hv|Apy5)-NbB1dfCI(ufcHQuw+fdM6o`H&Plds2m@(VnP^oxds2JleUI|-0I{G$CKK=)Z=`V??%kvc4x=Ruxg4u|Pz1;{paZSsR00zakS8RRv#hDoX#w6)ZfLPp2K1s<uDqh9eS&oS-u*$HJ%D*vSI!7VmaWXAQo{a0?MHW=Wetf)n*?qjon75K7@3D?*{UK#-q9%o^D#3x8o0w^I0*01sep)4`z-p!<F#rr0-KFK(EtYgA}ixTmdc?#btXlYB8z+)>%qCbssQWTX?svbCUmp&oi6WDGB3r8+I$G+H%E^XpB>)Reu1mnpKfxwt?ki(ZwSwDre#_<IX#n|d#E;za-Ctd+=8of;Y3(|WBKv@OET*h%nhC^NJ6ys1zZ)s6%SAq_~^(^db1F2hij7(U_2-Uyi{j+?P_l<a=pv59z8Z&@1n6Xg4zy>^d{tg(D&F57AM~rcdE6+m8B^HilO6u`TuPpyq#p6L_wWB_zCPWCjMvVoWs6}337p-#SBy%O$p!$&SBOPB3ur}9Dw3MC2I<jYv!isVLQ`%<r6L#T{P`;3a%^HX&6u=i&{NjkTbnOAg%Q0Pp3_)LbrqLOFmpKE|)&lfF3<cF~h!!;`|m(B{|Z-r&!70lOMpdejPVMtu?Xa$=oP2teQ^5j!gu>^@*oz`WG*Ua{V?!lv-iDS0|)kJ~9Q^t1w}32<eAkWiWus+Szk9ZyK%-md=c5k9{RPWBxhB1Aiov4O42^n~=lo0_M&3{NQg0%^*1@0r0tB;4gQFy`BeQ<uA71((deC4I`ve_|NbvqZ<*O50mRaHAJx3lsO|Z3vgfdgwt~q5a>`^W?lu7r>uqcyd%D}fZ$M?R974zttePovhPC%C_8eN#r|qC+Rr+(MrDFkVU-D}j(QJTtilG#K#dRO@!n)b(Ona=GgezY=B+nY&9@2e6Nfu|B~<u6iQ#|3+t2%Q)``(*bcOh$B&CUUZY5leVPI4=n#t*tircy68`%Z^+YGH@8KS59i6wRpPc9N!MV%%@nyie%3q1gNH+d)xt7Dkn(Skjmt6o1Gi_<IIj{2M7`0P7Fj#(AGgWjSLXhtH`99W;o6&~!UJfhRg!hq#<aciS35d_OSJlAuL3~|@EAbPDA|7Ugp8lbC^#JGKiyY!d;{Bk#ar#!Fm`n?ScGfpe=xKpGqICHC~-e#i+So&qDb`NF;U#ho(apQ$WYIi3axW|irrVxe1wJx0#4U|%Ks=J7s{q%_?kJ^7>Y}<f1G<R+MbqM%#@r7VosU+Wm?N;~PMK3@D@=PPe`6VS5V_c+u^p2C48rRNSeChn6$W6s*vTNOaUnTuUF2*Xvh_)HF#6%Xp6$HxDScpOFzUEPf7m>M8;u>F{`zJAhGE;QUz0-EcJTn}VJ{v2|6}j2g%i;y?dT{&6MW_Ix8YO>rddZgXFTDwS(ZYNU-P-L&stjM|bJCLnHZCzOn@cMs7YmpUj#_@8Q=`U*AG>#V3}oWO<^%LjXXlYl96$07w`>&-kk27lgdc--M_^pG4~ZoD1X`vWHn$O!?jPt9QbeFl!qv|*mE3*aHmi31{ImY3S5h-$FsA~w-hT#!LV=X1QG?P7Emitk-q8EIAulqA{vJb@b{BE7iBO+^dpT20Q9P5+Y`;)qi%FCmY*SZ29sP#bqoGKpUJeM}kh9-82fjmOOZ|9=`P3_Z5uCoC9vIG#nD25gp>!1guY<z@gY_}%cjr|>7bWuapddZDdrfbbKegwmZo?mB=0_Le5Y(;5lhYFJqC<n@%>8Y=?qXiP{^Cm>WsDBShrY03nGa`QNaI}3h^#41U%P`=O^AP%Og86qWSu~bk{8WiSfE%*8e<^E!4ulmJzB*R(e%H`jhZE;XOAa_n!L3w@8fmcl&!O`=W^iTx^4DX#ZF!DY5UOV)pX9K=QPzkCDjRcv<WYUy(`%AJI5pLzkJ>UlLGctrwsvbhS>6AECXa~OOs*lt|rH{TBEJQe7rB#Bb{J*e@;!uwXME<HVoKmG$&Y5L&8cbhzY*5*%n#zfz4?e8{hLle+UIAiV9vzumbL?l{V9!lXBu?`{~++oF~{8QAj<QL)KL%0(O4|Gu#K(DF$@Eij__pA(?2Ewi5$5q)W}Ys-z0T-;h$pxqOHuB0wBA4IN0RsG2B<yjMiDua-J=>QN`1E^E!$ZFQ4SkHMcX9>U#C6pVEACod}=wK7F=<Q@R8)JqaTU8u&Uw+MCIS_NlVx@gFMQiWV#p3mV#>Yyg4X8qEgWS2{VEt(Gzd~#`E^VYBC;a_#Gy-^=t0vl5(plq9r_}ik7*7>@25P{cR`vbj$-A-CZ_twWTmLJ}gPpKPaXq_)oZ#N$Te7{?S6SX5jS;x!jy~D<m`H%0Ia~5WM>(DNPX~PgE1_{DL2dvit8I{$R0BYa8H+|C@6(f|P>EpFOZI7t-LxrGFd|x7$(Ex^&p9D5kl$|qXvtJDRf3L4R7f=1>D+W(Q+3xr)0aCwxz^Yr9LBr(B{w%e^W>tcb@#O+LNp2!RbEiJ`%^lmHgJE08N}y3&J==Fe1><9pWVok|HuOlHe+oRwOY||4p23aAx9wI%W)tOH_=JyRXj35{Cqpley8u$D-N!@0F{{Ksb1M@0i1=cqE<5SAlw<W~c1lWU+W^Lk8C1;?UA<jqTJ4q+!(=nykI|*+;?GBPGeQr%-*DzkeJF(xbeq+lhpn~YK;^kyopYS=@7P#oUz~p%lx4uve?W=m?MZ93xi$Gw{!a$Ct0*OFDt7(Y7K4^g0lYYCe%TYQJr^1r?`(SN)gO!D<&%e}T=Dd6$Wam~vLtV_FkJrygIhLFmNFK#Y6mUei}|ZPD+HqAybT7;8_zsHjs5WJ5<Yap%Z(x_Z!u@KiUSzvN(`!GIw#&9ZR2Ft%uE~X+`%lP`nK>D&CbvYQLyAiJ8^KsvNh#!b&3KEW6$0uhDvj0oCbk?Xm*Z58+NKY0kk#)un4io8W5an0WwKRndjFj=++F|!_l7`bFQ7^WQjez)MC%Z?TQ4EXNd{PnkJD*cvr@bTt~-(mM_!PE!M6bqhyvrG#tHKZu>tmezFOZ>8brhUv~T{Q7wbb6K4a0aIDZ!eFRUx2|Q`_&1v4gXysa(*Cx?kGIANw&kSL}<7s%2McA1A;+TPcPFMrC#i$WGVr*jC`F(QUY&N?^H7_(}n0adUp|f=;FqQd8gQp+=x&bLrjp+jku|rsC>UxEX7@vd+PY9LE1b%OFtCw7i6oIg=#}7E#-sqC{c<li(mkuNkAhY=HGZY!;97=ct$h|PNNhikJGAo320dVD>BS~}2I}>J!&9khg0gy*}>#mLWQ5?|On|1=+xj8`S|5i%nuEY4N%{OgQkN>O2EdDN?Bew%#glP59&a>};n&(6`?0{b`_^4Bn^WWHjtjx*%<k+PLKd=U!=V~&O$x95NsE1H-V7xBt_`~26{JjHSI+uH@+UJi|Dn!Ki9bh#lb>LMwAwJjz-3;5w#;ci&GWlhFWyrox8nmW@gq4MA<u@=_b6Wi-)_F%(dkYIAj3=o|Lr11)DQK^V9kuLEKH`yO4<@pushpLb1(Jl7oPW&(Ffn(#x2b!hR6Yzd>BrX3pK4;1HYx3+9xu(v&tL;1BRUMk(m_g{Df?A~VrFjo`c<N|ezB6oIH7`<x$x1DxF7{9KWh7TIcF@9&3F_uXxXk#Ab#^?gr^UF5@YrBa|j)Q0Ij?iG8^0n@f6U^Bq}Ja8anD)G9rJgB(~CWv8Y)JZ&R&vfg;s(Pda&I2==@ZKmqd6F$s$ZM6jnmiiTnpLBsgl<7&L(L1jrp)Vx$urFDvPs>B6@&D5WM@juoh4a|t4uX}q4TV_;WZ&*%_CvEHu*ffho--v3aR@@K%V<LNW*+_lUY|#E5rR^gcB3TT#KHc3v<itQ?L{kay$_uH5kFFbO13qj9+1Jx)YVE^V3_=fSJi8KS)ryRdxCN`c=LS`?s%tr@m7x#LrTuJYScD~j&jaa$?Y&l`CTywq(=y=UTyO!bLyrYcaXyJ0G}-PAC1{S`nO@NPPtV$TVHr>%vCl^QL12E9o<6BKr{=)^h(79V#VkeI7)ets6Y+?uJB-|ybvWG4H|Ns5K*G8uHIjo_`vyQXjBDS%k(+RsL!Vuvf(OLWxL1#mm};DK(TWA<si9kiM2jCbf)ZJfS}?H)GINBz{N1(HP(l>iE_P}k<tQIW-5dl2H<0nPFHq)3OgBqWx~90n1>yQ@6z}NBiyH8)rJI19KX_0%%sF0|9-kk2g1zn^(+u)=hBA(wGW;>{cWv7mx;_Y|mb-%pJkgf)BQEhbAt>A}YwvAOS}gR^4$3}H?}l%%6d`W7NWO{w_RRE$!xHl8dM($aTlB^NoXn;y>Ef}PV_?S3>`(e^8p`PpZbKc1&?lH@H19@FppN)ux}L_GnFniD71eN|{b$``|B}T$Ye9Ihm+MY4hE*mZ+O{UH4B)IN<P9avN`${=Mo#>E#c*-3fgg@0*lh0SIFEczIpz9&?q=zhNyk(#h@`@cTU9LFdVR`d)UJo)Pelm6Nj6@}Y<*yPT{a6BRZvhPYT}$$gh(NFd|0ns`FHeduJHJ#s}P`X=zAVyZ&<n%QUxb>SX>wLjNuMYX#UAs8|VW6xH&5A83>91CQ^&yU!o5m&B)%axCpr6g*%-MR(O~0ZxpKtdfov>Vt(q!MGOZYqd3!;Io|n$_<)>RU;)~90s!rekT!Z;4CD#|=!R6f&qSv8q^X;;x<@x=ZD?x;x<TD9^3wm@n!AS|>K5Wa$wSu$+DQuuR7GuC4~EuIOyzdoPT=ZTXvt!o`ma~9AflXz4=~osYfU&RTc9y)>NOs2wos>cqbhc#6_K^}k}m&!MfpRzZxI<$(7!kT%(vV?^hkJB+470wBDKC2@sc?^&$#$HT8Yoc5~3@CFuGsLSJ6EHUO{xkF0~uJq6FtWZ8rPw)c?ii6gHoMpEg4InDWvw*AN^;#Y|g1IDfXQZuhtS;oSg1Zrqxh_@cO0YNgcDdj<qj@6zvb)+N<&QUnP3AxN=E&k+qvZzZMKPMI;VQD34jS8k0X%eMf-0a$g`+gI?5Lu(pn$kW|u`Z<wr`Wbhl7y<)5G5cFMp$nc0+46zGi7!WE_{WV1{0A0tu1OI6t87&#=KPRZr+5cf{zs476Fie!@asRL$ZSl%GP8+726Ap0x&y*fk>iXHiG8!XB6$m{XWd!BGao21yBhT3ofuN3n+T;|FH-BPm0-e3#{?){^4h74%{BD=n^5BbQ@ik0&t~d_Jc3v^UjE?<&*(V)kUKSACCMSl^w^t16=P>Fsnq5Dh5xnj^tgGzDtGer4uIkg<cv;(BqQqV^>pq(EgJ@ieJ+OJ8zaRxso})T-ci5yVA~Jy1(7JLO|VoYUljj1`s0>4AV-=r+4(;YAk=GGAMP*?N3$~$f`RMAf_VUwI4s|%Sotm>VIlu+aF=B_XNx<51y(`g<rvt08Sc7>p?6ia>e~}T#yL31UYPO2K{$>i3w<f@rCF~8>QNh5U;oPM?6QnAMqRT4yb5Xi1_lDFd@_Y~vFs5(md(SLv~zFUBSHWFkE_fyNFZ@dRQRJ;<5$^87~hg}tN+`QksvqNsQ6d)5JtA(_1Z{%{FI#VrDspjr<}kD=QB9oHVn|*cE`5@9-36zFbixyGQ=@7O<9Y(=%M~CO9Gp}KZx2|_wQMw7=@YGx7y&~WL9?BvgX*;rO7Bb^g{Z-i$M#*RA!neQ3(m;3MoxRAikBx#C`*0(}*yMfu%#-1t^tLIERoLSl^kk{91~(OLSx?RhGHbVlcB%s$d1pS~=g9iI6jnQ86_w=G<g(*@*nTgB9h-?JfJD;zaB!?p8^pP0sfWl+q@!_r`pEy*^R{`6fGlc?PfoxQ;)`Acr=sY*IuJs8{4JJafL)^WF-n4%gu-u!@nxY2@pxNS}qXf<`3`_Qa?toNlJ?B6NPLnMb{(BbyJB=jcZt^8B;OlR^bCLx8|jN{{jXt<5Q$zJq$jBHc{D)gt^23|I6m*9n7dH0H!4h<*%(RQX*$PTGnh>tESQvd2a(`$uP>=gk0E4}y?4?Tk~)jHaVSuXAdTJh@ry8tq%4&Tu$0iC=TG3yDibGn!ACa0NW_uPk8-mAPXui%NG>n1>dbu%eu`gaAR=a6_m@3Bs<}uN74$i|=cryJah%11*DRW}EnQ18DI&X8Q!qxe#42^{r3I@2W9__-$;|FzGj>B_Ox~k_c7a7|x&cV}icGlJ!SChn<F6t>|G6S!g~_Fbl|>!Wq5t8H!&Sr7H(8FzqftAc7nsAf_FK){kCDU`FtmqFBK4Xe@s+hOV*yXf(PJj4GTcid|a3+sTqZhm8&wR;RMaTah1k$>~@q+An`tlQ5Ws=={$@3R4~qQQE0!S1hcNkWXAhAd4Ru{x2Zp3kulNiN=}ie@Wg@?S_8;*^j(R=7&8NT0P*Z)9irM32YM1{%591i(cmRm#lds%R2*JsZ9uXtGFN*B^dFkm#mA1!6kp~78Dh^A5RL|JlP)3D0RM5e_~1qGU&F@6On3Us5XflXMk4!m5ObT(lm=C%7_HP0R^SjTM7iAopB&Hh@o#?gp3lCos5@I++GVHy+KSI5QQ>JtZXb=4hzCcc;!iHa&$+)$5eW<PMFoxODBH$C~+hA@%?ux+MY_fwf{LH3T@volh6j@rX+k@PVfI;wEHTzh0J84fm8@L{wZpC>5Ua}gcxzYJ{%+kXa3ghG;e-rRXuNKOh3ZQFO0<i>s2(ZqSWsgBP{tHzal_UrZ9{S0)p*fH9~?Gl6t+Nt{SlA8bJ^3i*!trVnD(nA)@mO2Dz8UGpJSIz^xy?GyM21zo<kJmGXj#p&cpjdRri>kXRhbCcUi>&QaHOgzw#XH~zARfiP;~Xh7FYB7GftV91{@)nHpCU}ye&GJj<nJIUC}A8H-)`OvN~#CfjZ2C<}Le9S|L{~7NXg}Im`MXIQ{6(_ZFu=H#|B(CwEG2T^GoZgd6flUYd5EHOfPz*2W5c!$fF^)gjA-cZ*#RW#M`$V}jWQ~fy2(x0(aWvLZQVhVj0iNLxG>v4?W)#S0q;4kHT^On^Ld|Vo!TmP4{Ae|PkPIGt{m1?&-ib;RV(PME>>4xKkDk!7<?x^`dJib)DClfvhBlLqQ;%%LY0XY4%0*2)lCBAFFVd_ALo#3NicxR6<^lWN?Nrn%EuC>|8}$?Qkrta4XiCIh`D&%)@!CO(Zwe;eF&&TW`)ujtV5`iI+xMGeI3if1M8=#6@_#6rxzF4S@vxw`BvgK$W~0VjR;wcp8w-~AZg^k42p_<=LGqU_N>xx+Kzv%?=qr~nto@DKZ!;qixS_Xh+p-%Wg^)@D&Y?U9tqW#jNH!F=T_>!=5`lH?z1g!N*8PZIu1N_Rp8=M0^%b{>Pv~&vdEn8{9+AMZbV*u|9wL&u?o(3X>1Hx7CztB7!kt7d0Szl}=Dzw-WRcp~0LiQ}J^Q97g7;6Ei$bCn(fsCxY75X3K4plY$PonT>F@CXE3K3gpd46kM3L$z+1yVh`h_g=9R}8bpo8Z5BoNHHr82h?17e?CJD-JNr~MU$lhqumbawsoeRpLrLMw5Wyb}%))<3MIiJTv00nHVPaUCvTc^PuCqW1`w(&z%(2+`D1y`NFhqB@@yUL8hsUd|$YKJ1(k4GD(?+ao@|fTLFni8zalFT;ptg>BE=i38^isu#ma35hp<B(kaHz+q6Re5`CGjh+03-0|Y(1nD_AS1&a{=!WqpO%~C1{29jL7fQs~qIBmCj=dZTm4YaPR6&L%)^tE8Laok13BHR(9Fk7qADb>vfXASQZ-H>23H5JU^-6i-TEQVbwwxj=|7pv<l8-ydZ4sn_mRRR8hvKp&a>BBYYUX0FSh3IsAmm{;D4+i)8qO#0DhV3E7xtnVe7#I)yFvn-?io?$FL@@njRT<=(DZEQbr`oMj81c(0U!4^^}%H^Jece*lt`V!obtaxn-QxOZ>|kkgC*8}K%WpRr=}Caq^P}9*=`VbR`@EcAeaJAeP>po6r5p)Q7l+RvY|#$wEorAN|OvzuR3fnC(){%=e^Ym{Vmxcqs4*0cK8NT>x|)_V_W>mrrNRz>WwQ+3-115OTXG8XTYucavIvQ(bEu?^oyvj0yo~KAh-z-P(u_h^<4z{iWX#iUVRV5IIQ=$>RhM8*DUicuJEqhdK_qIKqO9s?d`iXEVBrmQ=0G;P|ZR-h#NHz%~UB!?o43?l~J`0ykYP^8rnty)}xps;K`UWu!EbRIbI+@tpjyz^TU{J_DTiq`o>q@1vYht#$dv-7R#Mo3@L1HJFY;X&?AJwkbQeA!-AmS(yuf_jT4Tsj9da7cOr|*SBpO!CDOiA!$BN7&A=t>GOu$tE`27x*nBpGS-Hfom!Z~QcdW|YjCh>@mtf_6_z=-6$M-2D+oCt4Sj&Y8*_I;5xO%P?!FALbeX9r2VVK0V-PYbMMrfwDYyo@kGJr&4)(|3^DS)LNw0yRLxm$_yA=E37RD4Yyd~GGth4aMuWZ0te$rOxhPk%UJhQlgDFAfMuPNZD~{;=^1JOp-o5v@zp%T_JqzY|OB%~EGB(O$dkd;Z~GdnwRIhlE`w!JiO77dA=oU=x|MFo~TfKhcD%$VYeA$hc!h_?O1NyzDe2?2w1Zf#fLiR%X=<t?*@rT*?#BnmfSsO&8eG>~(9BhuSQBRlC(rqPBrj44r>wt2a?~XM;NTZ7ej4_ZYv@V;E7aNa6&?hw2F<jaTq>*p-?{id{7~T;ZWHQ^7K8^J^o<5j@-b@+8*gk*^dPf(Yy9RjZpM1PV)&a|s-U#Lk{=ExGtfoX+8i&@#$<{89Xa{sAQ->NE5EfNFCkXe1yldi}=DbPp~u!!FFlBXHmfxUVA7Pi|Zk1w<bpKB6<<8rSmRVR3XvIK*#+Q%<sSS+~Wz!%q?=3*pd&LN%{wri}U!eHlI84QIIEj(GTjpKAmsU{r{UvmW-=n325=14E-z$k^s%4!_8AX0K3#-J@r(vZfb@@QDZ7&-Zc;33Qw&)+7Uzp%4xK+QuWBz0+Ci@Getv$8q)e`4a3fUti6F+Iq^wB|PqFh~251xX$K4U)x2=+kq@1Z}-95tflVSwo*&u*k+#Ap^*lJ<L6M4Zhg?Y<LDU4w&J_9dKX%dc0!>ze;rlYmb1@kGskC=rC_ZJK=O*kDN5@JcTcuBf%m<69e82->cvAPc_b5q=#h^GHSUA@m=wevtP`*Eq%&m>6qS?35`XPc=Fq(^n1|%GKn0|*;dnH-3t%2dJ59w>VkxMQ0*=to9*`zX^lJ<UXmxOgdgzmKO=jE$wPu44ndwu25e%@6(>kvZFPnn7z>%9jm%P(8gx{r%wAFqh-wU3|6DG#ACbO*E4pEn>gI;7OSD@pQ?Xqe7iWJ@&fY!GYo6W?xbzUHA7e<ItCltHgbWcJrJLij}k~18+G<;-h3?WhX@mh69SOf@3BO3y1LrXU*9Ce&mQ7id%cEdzKO}t|41z1<#^?>-gtgpT|+P|+cRiKy=hh*8w390=Bx~2Trx>vA5Znp;1qeJNqna^$W)1~(Qm|-+~7-;o@aoVqlj;IEp2Nz7?q5ONjrL@F6Q{SZ<Iv>rlWMQ3Cbw0R;<Z~ALPz~N;4Ly*=9}z*unlwEp+)O5h%>mrNvEBfB2Bp`?O9q*=wrl$5gMo6fg3T6#>7)fb=~XcQD~#{HpLZR@Y&8=?!i?&KZwG!&nGi=J@KWo(?}osXc8V0InZ=Sq2D)wwzg@0>jf66KMMm8p#oKwwl;EPYRoFhZTREd??&(jpc7fHI&)=-&Jgebi6T16Tv)@00Th$ry-f==OCgfV{egOq`IdnwkKJBay@ssJs^pnjr#tV45(m6Bu3}k^<<@Pv=!TYw6isXov6DcLK_Fm={10bjh;6G5Q@mQn9XZPLD;d%RbII33r@N}GdhPxjCJt%X~1dCs7AwanzY<H(<cM!wR!v@Cz!2N{3jCM=CvQjjKLWXs&q1t0E%OJSpxFTx;iCL$vD&CK~e%Uh_p#<O-{r@J6>O<_=dnK$Cw53-A0Ek9XcBnc4tXK!V=gc*FUqnAX+SfT^X?+p~RMz2VXc^C`c!qqy_$9`b0e%Ei;qC4e$Ng2r!j*E~IrtSE0)27~w9iq?!BH(5)djk@TC)=Qkl@BKP)zN_msdR(8E9R0r$$;&b9C53&|@=g2CS#qtk;#;T9P)R+y3tywD42sHg`+5ssNzpxj>uWr8Kn)WMZ(du07yyMpg1b&7DEUH8t*tu^Y-8x<sc_fBaZ@h&|cuZ)Jo(A(ah*4c*3pD>lHdLlc(1OKF8RJ@~0GNU0fy4GeS6PxZE=wL$7*(SmmWRiYe-c5MNwJ$3$$a|gxXM_flbumc&8upTXzb(e#T8~|4%1_E&?nV*9cKKF(~X&1!G{1H)$l{D&T_<29&CUe%ok%LIZnM&F0kMvzc`WrBxJsm&e=k<d@i~;T}rEn7B>4KR@q73qrdFK_J7@JvG;M_c}_q^?<q=~A>9N`lhe~*!6>YjiUKJtDuC4b!EbPL(-R9In83w+gQu~y&J6|LN%rB`?gsAHx|`>JcQXT3|neme~3hDA~jP{+O-_BZ+VnZi-P)fFPDIS0|6I%-)l9*t);P=CtVa8CmAGk^ignE5G!TZs%2pyA+k4r<O2|Fi1C;rj+w4Mlb0{`T_L4pRr$CIS4wNhHLAXMiZ2(9D6ZgT|AMH$v8|{NLmYA>O2QRwbTpjj%oH%{w-!oQ5Fv#@|An(zCQRxfy;A`8E)1M)san=)}YnDt$iqdpM)7$ntY#Sg=KQ3xS0$g6FLrTtP4F3&fmhUoOWjqEPw;?!#J^F3?Or!{Ebvd+@JBm%FJ}NR0b^u)KG&EeR-7m;gr0RI<d0<Y%JVVw6qM#0?qy()@wO3w%&Vb<XrYjS3F#%$TL&i)Mbz6nAby!lYVEKMova9d>2pT}HJo!_#0Fj0gFcP-fzy>dE=_^ZO!XZ1SMN!SpwS1U#aqD4c=?@ot<I1!*bH8dr5>wZ?K}Pe+BwhLgFpn5%C%G&+BU6w4bmU^m`4*WAZ(Kdcr8iB7^KJ>o^E^=P%v>p7c$Wu;Fq0b<C(qxB2*qY8Vo^!$2fQ0@%US$Wsh0S?}9oMp%}z#*rMML6?p?i0}Rn}UCiGaynYe8UFezT1O|1#Jkk7Dn>f8sN#<zn+~}Yl(9mYeng@AI28`bl#r`oEBUuM|yb#uui(g0?{lMenyJqy9jXDv31FdO%aNCinpJN!d|N@aqknKN=t|mM+}m*r74o(_#(MzlxIl>?7UD?^`~WYqgNQ(W{KI*omV0Endn~kSk`+bPTJ)3#8bBHK-PE~oK5+;?hVwLzEh>aaWg7RSOmQg?YC+dTrmeh^ip(>SdvNul^L@@)@*T2CO1}VRxjxe7Vt*3+m}p%g$0vElzT@>F9}Z1w~+{url=!6mRL$C`txrbfgoTykZrYfVGc$azYMGXiTIy)BfyKq09AJ)H}7mp(pB~BM;Xtc+Nj$VJ{Uy)#Ui`+0?b+dNLJ~xJ@f#GGNB1(*3VHAXJ24V!KefW$CBU&-+4=y5WE5;zk#NK2<?5K*M9VG0VW6jFT}CYvu8zS$ndMQfvx+&2(1uoL-Cs$c=$ljNT6mtC7tz*dOa^~9bZamiM_rJOl5(P$JNkfI%5ZN;K>Z1+VR^nUTjp2<A_>Oxx>DBdt+i|K7Z<JGF*uctp%z5Sq5(%#L>Q%CQ48VINZ1xK;xR#7WI&u5-`+$@pu<h(^Sav=1*;x5WJr)+YeT}J<KF8hfs|lRBN*+n0fp^1R<HP_1{D_9eBl&`8|rFsw6V`O5Tb7s+MK;MeNBTIX|wtHU}}kg#*ITudI-{#$sJn&);0@kPs2b>SNU0(M$Jkl;lsJoGGlE+?^LVyE;MvL3XL4m5<rX=d3^XU!ZLlTd{W|mk5dh3x+q^ZK?*u4f}<`#+HgK{ne=7#c&&qQu<(XO@I?<&5VM(d7)qM6?n3T5q$vRY8gRXMe(AV8Z3u4(_SvSvy6>)EWd(tw`}B$4gwGEMI3$bW=a;6@)QaFWGJ4LUZu<$#7H0(Xgt3`Lq?3J$Ck@8u`}S3*$Z>XD`z^1?XRIj)O;>-cRFqLaF)Tya)Rs-sQN;Z&~~f=y#jM{1i~EF{J-f&WE>?A=-@3QwdgdXdM=fTpFZkzzV?IPVQ}JID)})sOFNp3@XoR*x4eUzNY)hwv!qoYF(=eoF*}GiSTuXX9D?xz(T=O~PyN#ZDu_7oG~#Hy&1w5GHYAdf#F$o9#*DQzq02o9Z2?rdaVs35G_oFBVW$Uc&R12A)slneFV`Q*o=VP}mH+|0f<kO;Nj*HSj5@Bt1A0PkqkIDEvM#>`B4#S51$1c?>O@;Mf<bkJ#Y)yhw}61WZFlqXSD9W}dm?DPM*1QV67kXwhX{X~JxYQQ^wB>F%p@<~h;_RiLkZGjp%l=1x%G^Hybk#i%Q)HTAd%c#(kXa=WQL`$M5W9|HG-E(=9*+R=f4x@Rzw3*4hQNjPOOG%A8lP?uEI3?^Die^tCv(#Gn?ZXJZ=~}u?YVh5A?OZ(cT1L1$~$Rqe@ksHg2poajYCa*NzLgq4N^<&s<Bn{7GXPKNidyMfENQ)ls^s9fxtQeo&W(S>#c^9DR-8D1Fz5A0ELks@jf#=N6rN7&3VXrQ`o|Tq5INR^M_8t1T3@jB7zMY!EFPF|!z}5>g3ym)s@h`U8{NzP~v5u@xR6DxpPyzk&GQi&_;m>2MpN$b)-A_iKRZ{&vZkjhy8z3=oN8pl=L*Mw~p9qbEHke(QDP3UfJ{okfkpyt_j%&oT1mvyZ$&xI}fMa)`J3*u?vl>JiF{$J5TR(N%tPEwAy9WxDs4k9r99h$R(uCgRi7?7;IuS*goaj3uzUn~9imvKlYr8AaE8Jm>+5O;3e+<7EPvT~#du6m{_lgbyO7lPvms{n3I4aLQD8HL41r>yyKfV%!;Z@1D|=hR&sooC=$wW_sNejsb_1N{&(YHw*<{$iN+A=2$i)-`+JZrxn$t?#ExHuwY=XsW$J2)jNb1EW=fh6=HiYBRMB2=~H(uAje=cuGPHNr_n%-5n7X!6z?#vr~EqO_p|EJ>5E5ZQ5di&|6q)+xaCJ2mTO`g)wr++KEY#^h6qHHdtSVw=B^Rj?4<i0<5ngC@m}sPIL5Tg5HxgQrczzF_>GF*XE}X7kDExvud8uM+1aUH6$*1UQ~wG{&jc1;%(hU&>+j2!g-Ab15*E8%F7s4Vh(COA=^hiXQ<BLe8Mm{`P?qdFO3Pfy4=W3%_4z~%MT0BIrE>$q-XZVa$P5&q(uwl4({BsPuSYpqyaAIRH~L4hg>v<PM>c$?($f?@36kaS=iUH?6wsw76K#ixHWMnX77{9*7q+~aQ~=Bb0`gwc#CtLrQ!tK)k|-CC9B$Y!i(C3_ufYsZShG?4bnp=K`)B`yYUfv$zMXwUdJA1lTTI?XQ~s{Wdp7&LJFc6${+C^wlS0YCQ4>NGUhwr*4VWAh5xR;BRy1vTC`D9oOX4lT?8N2vuw(Mj6AkRmm`_qzte#r0Pwf^vnaY)FN8}umLJ({t$B?p(z{D;i@;Q8)`Wkrs&xsL>UY&A#MnE{}FXfw{0W-{YQy#CdZn_OyNC&B6Y}-cRcdl6>AFG;l)V~_T5gwIW{lXlO@}C!mx5Mc9^fwUL{fU(2xyU_rxi_M1cA)4pXU-X-ya%&&$^tSd?NujdNWFt8&TNf`6|~shmU5@4%Tx;yU5|X?x0M__R;@|}YPUV#DG>_EA|Wi%yEN9~FVTSQ{fk`fL?dIkT|}o=ToemJsuraS%ALsfec>V5_4u=5Ejfx)(ihNX(oKy_3yHQGITPdvF5AMZauvYfq1c;U7FSqvnc84?fjN^;-53kPI`cvl9+-&XYxwkb(R**-6F(%)9UEX<jiyc65U^MjDa;&W#l2D`!q_UdcOJ#`+=fQ#l_P`Qb<2d<qE1_0J`M#mikxa(R$a5X4Az=W_Ay+>R|wGdM6K^A8V*h+pf{ax*^rcOmUKrLV@ZE|8T?PFZ<}lWIV`diJ$ND?g4lae_e+pLhV&`cboY((yWR+T(KSth&N-B7hC##=9yYQs!0k0V4HwlXoc)}S6Bs`M3jrl*f@4Eboj=+(H?oZW-$3(EmiM4<KHxa&XmK>$t{ItJ-%_2{djr4nyoA%KiFu7yW~nWsDM3MA`vlTXxW5VkHVMm`cZ0vR0wMmE9*%9)@5SDxambk+1)cTcOmd2jI&(2=^@f-g*qMto7?{J4Vw42(3<yLX6x<t%HK@2C7h-cNVj7n_IVZ4(Wki#UxlUNFZpKTVB*49$X;?$Ofs`!qRG*$I=Hy+dN6r?_Fv0VPmVeK&X+x7$PnxCguy0j0lQIOA?!cl`y4q4U77mGEmTN@_&v&n#A0DvjHsNXxvJv<r91pmaaTL3Pof(-ZyBrZ2g;5bb5KO+V(X;Xe)$*ZjFPp(ZO??x(&kjSLfv+UpXJkU$bQ<yE0GA)-GDCz5!iqug=jhDROLp<9cTtqX(<uPB3ROYiB<aqTm&nX-=$jB+xvwm}{s&u7NpxC%M>)khL*iwU6^Bz6s4B6l{#a%}@utdq{Lpt?kTS8UI%pw2tCEF7TjlJVg*jVmq#K43pb``TbQ2s+Iq{Ct1V4SuUaXTAZDEiDoS3@n4`d>?V|+gr5aR`LEE5~14>X%MDt%FI+I|lm)PC(}Q-B$BVA8tR=<O~=u&w}U+AMM3M(Rs{{`t_1Jz&h$+6~bQu^S=oK;9g-dGj+NoTRht%JE?$_a$5uo-~@aBJWKQ8*=$f-QR<5r0zMl4YN}cS9&j^R$z{cZE?D0GSuGU4`mN(UUt~_!28eS<Kg!YYu#;t^l{?0D&`U6LXqozo?JX=kZsG7hu%Wpgo3w;9w%(pzmZVT7Gn?aL<Q39Z9mjchpw4E{WzMmjF<C`CuuTG*Y3_c<bCm#JTE9wW-%Z8QlgeS7>*Cg{##%WA@&f1Mr^>DgCy244{VbpwpSq$z*$G~&XA!^CG<Bjn$5$iJzcTgc0$E<$MB#^cOc^uN$vwtwr3J@@`;~ccxg?4geLOtWIeZ9I4;w5dEgq&c;ibub9F6R0`Tc{q$zJYt9AD<+aGicR7ylan`1H+<3s99Vb+qr{&wOHl(7h7B_)@f43W3v(M0Z#2J8a(+=Cou^a3De?q}-&@aaW{W*fUnfWQTN-RoT(os>18_j|8hbr!-$-3G%tTvSvm<ihNBq)6!Ygip}p9nwf`H>y63LE&b30IJN5ftq(Am)aO7%->Fxm<Y#~UqZDWxX^!euf&ts;~i7Da+V(VzKs)cY8x|dTe@R31L^Qn8MJ*nBPq4LkwX_Yr7FIrbA12roF-IhXwwiN-v&9F7s@E0@>*-)lWfLxPo%0~vwr3ol!p8c;=W8SxQjz@tFIGF$m13P>?=sx;dcRI(5?V}$h=(iGs0mhkPnW~**=44P*vxfi{0Nia+N140O=4$F!YKDXIf(<PDPg$l82xOkczcq4at-8HrRO!l99?D=@cy6$5b2-Dbc}z9{tMry9Z^bCsRJPsWgSO;M(2;hxZC$=0G6?>>3P&ULgYkX#sD31*vO#_bkV%(CZY9Ffq(0GG;SQoBgcgcW~i^xT8%GU_$JnL^F6r3j-C91(V%Uf|Cc=c#E|p2|K7p=(g5KH1&DWDY?p|O1-jkobg2}yk-<|tz48$F`_xp0vV$X_aZ(Y^FRw_B*`UAv)uUT5u=+IZnhhFe2JltkxxeuZl=-O1W|5^g&!GuI0&3al@<5xt|(tjz_S0&WBLInM<}n2yBbsJWYIK2a%il{y(E!Fb91hRYh$17t>JXZo3evrMyAh(#TRKTJSNGa8%5iTg@r_@g;eI2kW@ofMd06^^99~&r1Ol{JTi<SU#F?cE3c1BK$iosBWsGlk9Q45hnPN`cDQrpfgWlM;u6y37D7b^R9Aqk8_9$A3W%vCURtZ(k9;wXtksLCPdD;}!>%o+qNvBLCIaE>jh@9&d{(gzeYQTWrGn~-qxG!j5!tFtSh~V#8gGbZeTdDn=NU#KjRo>xp7nU?T5xnarkC6CO)*jn_#}cBynItK`qHx!V7-JU{vNo;J?kkK62lT7U^C)4A#k>s;UCgKi{`f9kmkQFRZrBMN4{}CcT)<8UwMcVZk>|#G#mJnFzjmj8E|VR3|sOk3m$skfWs=IKh2SwZkKsh&VbR_h*CO7Y9k*O5-kdHunl4SLqd4QodQ~%P9jP`QAs%30$*K_8q%mAoz@s?4K0yK&qE2dLF70xbRdwFWh^SA9K<(S7PL@zvCfM%&1RrA^8yMQOZvnHD<y%ROVfOqB_mK&Jbt?Q)(pgl<l?2(AFw5~h;4oa0EJ5OwKFg_^eTJf6GR!|w>)<y880K|QPv7o97h3xb%wwSQL&i@*=z~w*y^PiOW%J`4ivyWm;_f_-bK6abEg(dr^4-4rF+0e!dU)AV@yf~!y|3|0-!j199ZC>kGg}d(m1~1(=1E1={{_EH#~yAZWmDQYgnY&Y%84eYo&4o^V7;d+Rj(pdkPBu-Axh>!cqW7!p?!}^=YLZ?nO3ldd}bh+N&Faj?xIS%YmTDN!S&am|uvRIwqlaB)xI*<+XfRDb5Fx#=LM1t_oZVajm-p0IsYQReTZyjI$G7_t^g<SS2-B{wi?Hl&%+cp6^B;m<F<6PyjZG%{ZtF?PVwjfd}b>px^homl%n{52C-=GE*XL!riCX-<ie!YBtozYiVDDYwo4928Fv(SY29!^LgTGjL%I%(KCTi`EB={tF&E{^iV3xS79mu*umE$hR`k$Trc@ERDBBOW#44u?Haog!1VW(*_<OAk;5t(SAzWwhqTB&8Idu$Pk+JoFQ(cAEV{vW23{vWg!lN#xh*U&6y_$o!gfAM80MO;8<ydk9{mt55xN7?_yJw*8I?<xVc^}ymHjOcn^`ndIo1a)c#G6nUVF~Yx4<V)r5HtSCwAzuY2xuIB6x75i^PgHl7(LpnLnl-(ZC_L9{$M7>J38n9Va^luakeOdh35zdIk7rKHDu3M2!9$--Q&=G>kB%i2hdTua`(Yi;pv9_K0T;S%q?Ui3fwPyrv3}ttTE7u}2#<$hglN@+67n5k6T0>2sm$984O}tOcIn@5WO<{wpnorN^y|vc3ueS>x*1xe2CF`wSg3)fhhJ*Qnv#oGz);rFvIovK=CJ1^n6#LT2KjEOT)MM*}jPT^9F1=e<E`&{ile(74N*C$a`od&<J&tV&Gsuqu(<Viuw*l@mk?ycq?X)gRcM;F|3=cHKdVWour~@6Eq6Nk3<F(*!K^$(Vl%k7V#ihnCP7$W4<Yw)7GNLi&5+t*U9s>!*^^0nbN3%vWsT^(-xwqeF^s7o@sQqi=QafjWs@XzqjHenjDHQ5fc8L%z0>)Vl+TPv>84VZ%Hy+b#DHH6&XI(h;k>IV+n%xl^M|QvO&?dM?c3YwC3%SR8YDbG+tl9h-$>(m`~0Fll}%MNl0AiQX}OHDpVR%=9?e_~%kOuvfeR*Wns-%6nfRZ5(Q-WCJjQd6k7$&mJ2h9Y$eeE$TKPok;~u;8Wf{(>P7)IoZ*e9loyA7UB`-$%8>?uXV>WY6Mea(XvBPN^Jt(yqqY{`nYm*lKKN@-YdtBAX>pFE(1^`f<9nh`%K3tK$_a^6EEYodB22Kl*;kGvTkC?<oAWygxwSc!D!H$AJ$ze^$0Y+#$6`uCMJ=gypP6~k;peiC+o0ubiBQA?7&m;@&@jm<9s&1MkOVsX7j5H|LQNvU>ksNy=-xe<nud?dhCmlpeS5TMN+aWG{a;<*-vbKBML8W6=Y|}couqF`p0(nhE;0DcyK4=u|X$f#)A^j;Yn*eu`qVwm-ha}TFx`p<RXX2-LU<b7c}1+jdc&nAGnxse(5_81HL3nTl8gZpV4RgB3jhfB9`+#ogb|&kGdHWi11^L$K?n~RooQ18J>3~oK~G!hFO<5;kghZo!$}|tF=RcyVwlSZDjY2Mk`t#7<%^}j7hrDgJX{C_fTdHU3E*=H!WNF5j89)z`vB8;SI+lcbgOtyyk@ApM>z5dOgQlg<(e8R^cyB@ugyCf8%`A7oRduEc5`_69Q%>h!J~?zi+jq=@9l2I%=GNuQXqBok^*#8VC7E>KFAexFThrUyL8L{@!VDAztIkIdMd3Rp@s($lzs1C4X7Au}k)EA4h)}I<(^(tz(G~Va{jf(AYo?`2H8TB^M+p@mwe~Mu?SRcaj7Xp-P9J#=WX6Ugb>8bRv!t<aBvyn8bCmBp^FK8RqM+brcOMMT7Vuqbc~io_Fn<W))%5&577sMYOw@PJ76&EK;;1QhBANk8~$^Iw30w2TnEr3lv-zeX28Qcl6=o5imY2;_e6&ur~7&?-SL9PAPVf)K%GqJ@TZ2_g1<X!bcsUW^bb-hSPjYI<g#^GU$Q|MRtFSnDmJFLF3p<04vqfL@EAQi0$315dW+3WHTc><=)}`Fh=n8P@sfPA};!IhY%1j3Z*pdYGAGHh7Q->?DG?o<+JF`jlwD|7@hA1`2$EN4o}5JtP~yVYlTixz_Rn(L@cEtwMjc>Gc|E0=WtPW6#USoWioFdZnB{n5DM?oJq$nFNTZ4^P=(%c*UNZG$Q=;;GYsnXrxCARghj`84*UaFVFq(cRa&~gU?OrC8IG9IJ_VWpu*a-vk@e;XcuYmIBiWaey|0f=A^XYZTRhHTT$;8jkia1!dSELk*9Pp=M99N`bC!(VN)9Ip{z5)1;J&9Vl?YIu-nmO@CAr2pgxx2=Z|!m~i?&Yc;_f=xOxL>F%f)Ni)20mKcB02PDR%IxgBDEfE-TL1<=p4&cJOBH4{Yus0Aa8%#nqCw&58Y3NB(}tx3`P9kF!KyUIJY+JHu_MSASAcME!?Dg@Z2*2ojf`I8xYwq^!5nKS+hkG2pf+`+Pi~|HdYW)|^f&I-mi~c|H`;BYo;7jykN{%e=gzylEj}5ECp^k46>wNB*$059<USa~DPZLQaNO)X+d;hsrZF(Ma9Z8UXAAnvBK0&S-hhXIfb3iDy6_J(<=quIpzR!k}eF#vTP|j-bjaP|(9DW0X4|zKiM8(aIO`2}NT$ucc<>?wrrDg0AyjH?cbN^o*-Ri}=DHl48l9*Vz;0Rid?aN7wO4zYT#33to}m(kW2PE$UFlst*r7nNi3e;?of06IHS~k|T@JF(TEJMSuQH2rb?NcV=e%Q}h5J_<F8ZXxEv=3#QL7#F5?oE0A^i1>1+_<XH75HVTk(Of8q@Vz83QL6P=~&4i#02bB*6Q@IJ*?GCZjyVa{Zhw#)Nfw{Nu4tz9=u>@!I@Y!SlTQwWiI`!D;Gk2Q%&1;gPrYLMH-3`$4r+(2v?w#dJ1OuK<$ftfsEBc@~Yc+O+tb+oC^e#~xi;9RW5hw_E_1o=;ndgmB*zGB814nfK6$xa!W{_(wAC>AnllKj8nyT8tT}a;yki*2jOH<mhwOQ0{6cDK+kg&|p63yQybyIE>SK^h}0`{ST|8fujT=qtmdZel$gBEm4X9y`q7cU-HX?SuH!w`Ce(j*YFa2hxp<T1yG$zd$|a?cSFA#9nB8j4@k2q^i)0n3XqjuKtZoU8l%XdP4#(&PGm?avo5qv?{Ik;aQ!%xDi$#=&p7@`L#~FrtoY0?$Q(3ZI=mDycpFZLht!GxcwTh|u<ljl$tuv!n0t%&wSoydwF4&gwXeu!9smBR%6`@={sl-UY8zcpR+RJb%49P~&?My9s<fy{I1ocMD&8m(Kq^t2DjxZ`o0QfQx8GaE9f-b$GAgPTQf3Nt?LEcPb&_wfZbFQdN1k!U(D#%6HeAtVG=!&dHJ(iQtwo27L)}2j;SJ$hYEEEJ~pe<0#k*``plAK}fd+CdjlgA7u3PomU#Vy;;puS{{=Tk#fS|J@wkVq%j+{EAPRo`w{{MdbrW5r77f9e4ZuahWV!@DJ0SEG5{`Ki@1v>N^F8!3yz#yalrW}l4S9|N<BNUm5ONLtq$ng7d6puEBoOEsevt`5+{X$o2@S9pE+sNm0jg_nK`readJ!Kml;eO2J%%w&W6qt^Ll1HJ$VoBNnr`d6Ox=PqHFDl*u}Z2{dW966?Et`#IdR;ALi<Z0z!1WZ6&lSapdc|yHAXS&aky;6wRWw6Xm90_951tG9!nshYreyPpZ8xO&}5q=yln<c+Phka4+3Fn<6^%Zy3pT5(n_$aVitFOsktRAB445g4X@)l>rk)8eIQ8R8bP^33C^+Bvwr@%wa$#kkS9-rI-|z_7O?RfISZ^@lKSeQNg2%t&R-godV!k@FP<lZ>dG|Q9`qt?W0qIF%gs~cW&GQ5X&(Vb)5LkWy*KjLu$Fl`o+$(Mhz4ivMF4+enXj~3#Nrd5$z)BYXsN%V;{LoG)!)91HA_N6C2+j*~K@gO@FddU(OR`(bbXgVIl@l<$F-Cj6qD7vgecO1=K^>R*{T@N*qIOAc}3r+DcqB2uY1MBiYWx2^CPBoVNto4x%;QX}O<!&*;V7esXuII0hlD5B~F~XmN-?5TEo$iF1=UUZR9`eG*ZpRK`vqaIZTba&vlN_Yn|0^H*lHPkh&j$Gf**zd1<J%FMz^)*p#CuVdh$Ki^Z~*QyK-ksexErMA-XUeo6(W6s(v(o$N1$kQ<|9G@s^Sh5|t+1amO>N(|!bl%!>)m5;{8h$b<4J$zg{{?zsGO9L?{#6o(5~6(>i&s^-$%4hA-S0ihnvr%cDa@T#)G1~XZ?Rc-EOI#JJi%1JZyPP&BZwAxC!=PiysPC!zC?`+rGq2jPhlHSEnrFwvo%E6!!IQ*J=>rak#3w)MacPSUJ`I}UQ@}+j^&}x!KH3@LaDR;ba$9S>JmoQj7RX9cLCDu4_7%&ot5W_*xSGhPURrK^ly998oIM{{*pgwwtjY!WO=^5#2HxYE3C<UAn5+jXSVh$*a$jmqbN4sp<*6?)$wvCwX#j$#1^GlDB@D0w^;FLg#wwTu{(OJXKr+E6qtRI7JO9jcg0nXveAyR7ztH~RGi84TmQygM>hS_RY}?Hiyj{0Z~aa~@u+gwn`d)Cl%ZIz33M{c!Jphtt=dJHOzm>yw3cU#4>mIOp1$V{WlBj2N}&806*22Z^VrU^N2`dPhJo4uMWevD%_nAp-tHbI3jBB3aOS{m@7lh~%URWV<*wD6>dku)z~X<#a9a5$n+Om(QamIyDo<4f7Uk*ggj;t<5wVFm$*Ag;NLg6TrGKx`UPODAsVboJG=wFu;0ydLf9V$2u6N%a^_kJtYvRO1c!VllNJ!k}r_*%1+>jw26uoh)asjHVW%O<Hko!SFx$5Uat;e`UPR7lj5izLBizc`z|H_A46kWkdG*LZgm3G*a1L(T2_44~?tk^JE@soXYu$g<HY>jBGh^F%|#W*x6R(hjy0qF<9a)O_&(#?(NQ<?mUF+x74Ozt8484$|(>kdD;B~hp{A+nB~D;P=KFbG(-<@Y0q-wX!Py%53La_(}7MD46$?o<BGxE9>%%8eiMJ7xeKc=oHvA*KyUa9@DGr`4>MyIp@7b1dde{q>`ledxxl(+yIkWKv*sRdC49y@~PoIbs9Vt!loB`$X{6seej>0cbI#zVvpB!-nWRW4);qs4`vw8RG}m`vd=eI6El>W$M{Wvv6Ns?B=<gmqP`76bknfB*>$>Pga#b{%h3n(CCc~-7G-HZDCG`>sW93vDJLo+r<ju)YIhuuhK+)+^HRXS(+?Lz-|W#Sfv|{*)Of*5bFgV?M&j86Ovu}@kS}X<inpxv*Lh1{-tVJWuV>Xchtn!==O4qF((?Wxl$h_Zzr>=KF5}{QKl?g@AJctOj%OR9J4CK_){8{vO12-y=sCO!^NPnZ>O&)t7H9kiq%S+3Z<|`{`P11Cmx6kU&14{v%gTkRBlR8W80~5a7GdaN$9TNUpK0Oat|z=<e;qfSay>O0}D<1{{(~NgqEfsGWse<G?3Lyg>^~8l5S0jTjC87_a$i0Y;YWT*se6*eB^tX&aCX*FL?U3=vzIl$jctMAi;kU%{J*84{t<$Tl~M4y9uk&AT$zhoE|4AXoC6g*8e$4M3>va7Z<nPr8_q10thM6+d8c42<i3fA=__4F5b4yhC`;(TzsPu^e%Dsm_rk~@}V(tF;-;L%~1CW)c)-!@F`~qOlIe=TTKB)1tZCqnB%%(go=`GG#_J+?$UHe#76C`XIdj`I{N`wI0fhUO{K{IlXG-8S&E?5$z52a&W<8d;YmwPvpO0<vl)#?a;{R`dpdr20@pl@I{nJ(`g%eLEzvRVw_$`nU&i5-I<NHqxLm61u(W^H0L9pDK)#ha4i;*e8|t_=bS7}~?`}}X^U0CAa?tI?WM0BqzfC~RJa}P25Uz}Q-KgnBRyvS`W`Tv-Lg{*YG9^FCBH|t=ssF^4!@?UKT8d7Wo4Gq&<KD1@#0wzLT)-#Jb<hr~h<q?Jed_+qNpl^&!cej_as%~($!fJ_syIy~u?y_!6Y8gH<b4iqOQ}5nq2q3yal_Silyw-5w32StU>@!_o|o;FluQuQ8S!x$*B~+k&a*9T5+M1}jl%&&+n;});ZT}r%#jpnBlqJ70Jqr)B%!)+K1J_Nj}O=4+o{~23*f~XB_p(xRbWOd(RPsWt@1d#26~yN<E48*()3u0U7?{ra+@Q_gNUJvU2`jd@y078%u-C432GYo;)!@Mky|0<#xx*N_b|ry&DI&EksJ?kwn9p{1cq=WhIaj$+^YG4Z!AGa9G@Wvq?-9FSKJ|cnV?&Fw@~Vm+K7IBI4|<;3AIUkiVfr!DbrWFETJ(?L!Ql#J5U$qeEVPPu#70CN6E7|`1-id0;j-C)EG#KHx%5y3ChkF@Lk<SS$E_9jaL>OQ^~`86(R<ZWBZPf20*8ZfaT1TBJu~?YZy9fs&f#$rWz`>`Y9}O8?fn^+JoXCyKN;GT*RksUr(~zFr;2Te#js@7f4kFW)sk%%TYy-E;s-R20gO!c`%N*z`HZT1`mo1k3x(ZP|COn<v-Y=KY|IL4F9@CctU43hU4TRBtKTn@fcpNLf+A`n`L7FvIcH`q^my5j5Z-Ng;ww&;&B}c>Ka#&LyRF<L5+b%;G3kYnlR8H;3e2qAHECR41xq?SoZW=cuR`MNlUqqp0Z-@D{0_8bw)2Es4lbYA<k(EP<a+xvcZp}(D!)0e&B+p@JQNP#{;D3nljbJf_>`N!CBj8Rgy`oP*2yQM7kV2ow%EXvJx|le$Ix;Ln>#+XX<H^r_mI$+8PODhpFIXf4<Yi7xf`&I;XQZZJy??6|uIDvA|gqe@9yLO;-hWX3(2I@(caRTKNce+^Im~Dhxms4an`}x3F4NU5l}3`(!u-bgmJgg6`6B;Ugp$Z~Z;+38~(&L|?3|L%HF?S!G_bTCZ&UJ-tv9Z!xY*1xeB}=e56~-$6P0`y39C94J1(f^Xkg)9P-I&>C0&1+NYl*9!fzvnMJs;>+RR*ljfw|3@$*9HX<qqVILQJYfwUM=l%%$v3?R25X;R5E-Armr2fwHgFw26rJP_(};W?B+*0ftE2f>Rm}P=0m!+wfA738Ix;DdibVnns%08~+Q~{zjp!epc6lAJi!CQn*|{lA9tF_v(>}b1uXW-~9f`viBzU@h5-`G#lI8R=0J|R@U7z|U;ax8b0U~?|csNPGB3grk|Ae+HPFA7e>~IKMvWh^$4(Y`%Yl(P43*~yhC4%y^<prH6wz*3s&<Y<e0Eu{PVMp5qYL%ks6VlKzQ=$~lsX!WSkO7kd-FT3jD&~=ZV)-k%g8pd6m6PqreeRCNg=xMQd5L(9b5Za|V~TE6yDj_wmO7dKV}8Tji<bY_6v}f;FP@ZWV+*tIIi}{}E^({xfC@A(XRd2Ms~50flNbcoOl~;G+_}Ite%agtTJ4u|JwkhOM8kKMoSUU+R{t2WH<&In?MGv~Ys_toJif!*dU)Ga7P`u;k4|%FF1;?HIsTzwfGtQJD3T`gp|o%+?<7=XJ-am(_>I>t1FYwJXeR_H07S|3GM@!cH}ZGGW2truOm;L>{%(&<b_6ycq#dJHI7J*_<kWk{Z3jWffed(;AS+*9J&Z$hkdfeh2FC5&DKHhi2aEc{&IAWIbMnriSrQNJIlS?si2D`4No}UbHV)F%Za)L#eyW(MuDBJGr2-v@g%Oy?_@Z5ab|9st4$Mxxu-zJ;&}G<HKUE0}Q|RjtrU{0uK_Fx$cLA`IKO=C=nWCpC$?vyD0!F@{`V|K<Xd^yk-$G2<l?{XBMO@2&(^K{Il*@_|pFvqDgA})EF@_C4?;`Nl6t+&`ei<1Ffb<-G!taN2g#jT4V#V5nxI4=40A84^q$Q{878eq$%47PoK8vZX-TQPOt@NI$sN%D;cmoMU^=fv*r&MkLHj_v3f`jX;V2YF?dZQ!v0rc2H@Klu;x2~e`-Er`psqfLO6F@d+H6pVa$eksObVPD!|1{M_x%eERL77=?S79=*MhMc_E<tSRmyxa$3OBib+1zA3;GfOCPnjuIgd+rf><|7cXjw)7G;XC8@UzDOAL>PVUGVUh=q&eVe4<BuIbaOt>StLhH*)l2G`NDFI<Axl57Q&u@$rbSd?atumKV*+^$Bs|r%=|PITZK_y$H{vN`6Jls?g)U%@UA77)LCi@C=igv$8}tuTQApYPGrgORTj}6@da-^}O<NsTu_gy}v3H`p`acU*QIDOODr2=-*N1v*5y>Qfb5x4!cxVel`3O-X7m5B+X$qUqd%Y8*tI#my!6zP?4jjQrMQvtj|Gpfy;vUg%A>kdN(KO6!#K&M~)GiQ3E^qp%L5!K`juqX4nuTfO>k7loWUD-_Q(M64dLEZx%#Q8;E@{AU1}$df&@IW98Bn&KRv`-*&#i(4GK|a6)d6FsOQ3bOdj=OIamgHkg?(M)D-;m6fN{8(Zv#kGu=jkdvlLw4C>190YWNM#g(QGA(u)1Mrc+CczPTkT5sd<>aqLNEEl2ehP==)*pWwxw_KLR+?V2Q*{)Vi>UuH%O&;XM)EkK1^_<{^xN1a8=A`5`)c0Be!GMo;R#o4NBO{&2->d+f-Ul*fj3R+g@}-3qC78YL@)@rJ@Kcr%#U_R45!|dcE;UQ2YCS%R7GMpNKYoQGq%hlM8SL|%_Xl*-hzjs<!}{AK`z9|JfxwJJbMeqZ+9s-3^sR5c2)~L!+N}^#~-WicnkX^uPFQCi8d+vKk#+{f@32tJW;{)it>^G*v)E61#<U}wQK#_zBI$mD5{bucVPQwXaxt0hC&|$hwbSiDw=eXE6SnV=Z2$Y@K;#*mWQatCN(hIPyHq?IRgp)Z)D78v3$sVT=UE-OlxiTn<m2@5q}{5{!->IVyUd2^WW`%NEMNfu%tOI4a{9Akm#cvPo{za)BFz!emr5Sdtest-pI&9v9%B%c4ZMJ#Ta!spF~>VPk&{V)xTmj#1YUBLQV|M;CA{LP_y9Zo(cE^L9*Qv1bRN~5O&$DnRu2x&%o>UVc`O);J-BuqK$wf1`!bLo^Oo78#PWA64zMeY^P!_LxSigQ;s^`z`mT&d(rEkEf~MlfG!d!(U5S<q>R^~=ACc{HcbBswc;adRmReL<)DpL3dCj-qBU8Y=G4`1j6oXReS5FMowsQ}GR?lpR(7`X;$ng|bTqntsYd}kHAX_yd+wJy9VI#Vt*!3OK;vYIteIL7Q4QZwt)05<H3n(7K%QA9xDhkb@^odC4v;mwW{LHWBFe<vy<ItIs~GmHnG7;8a*gq=fJxWW>o=vh5>QD-TMGBo(<P7SHODc$ZZhE}GMF+v82O?s58vZ>7$jv=wbv8Tu?m3y+Lw2Hw0Dk>SV~RruW&oxaKx2<m?HZc8j|<eG?TQdf3VJs>jUmCDO0lPQN+6olWUWf`TfLrWjS0xF_j~jVbMV_42Kcmci0Yipx6Y!wij?6!ZG&^Rc|`$)DywWw8BY4T4wn&(9o-7fG2b<AYEfgO6Ts}F`FBJaP+<uY^5f~qLo*@s0fUUSW_mox^y;AXg*xAA=PpoUiPreqADEBjcz=SY<Idb5d<HESSXK%4|6Cz%dnZMCu{GRtt&-Fu3o8rB;WK+AyjD7U0f{nMhj9-WZ#C{tBF#09u3b8G@uucyP*W7wmk+`&a9!Pvd747`s>ng;3rMG)VZK}WrU6LTgE-oSXo6zNUM$uF6m6;WI7s!HGAf5cK*95t>$Q|2P+Y-%7RuEk8U_9udh2Bv?k7ZaAh(=i4=XLly!|Q(ed~cAw3E9v@yKFZ+IZtg=n7N1ur8!AJ(sN&FhX1t!HqZV*fx2Hk3ZS^PB7;2qKI`@T&^-Uvr|`=YrCPvcVNSSRQ=GeFpRf{M`%9D#M8NDXxmY@7mD|)-(^yxwIgrxrKCT#Zk&2?vogq_dj$2oZeOS0iLe$@)oAQA-H7;z5mw1V-?!k?#DOBk@;bE`iCKjeQr$>Bn5E!K&H@ZW_OAO)Ko_eaa;^ydeBo)vJjny3)}M_C?%EhdPI~wWlltk*mXm+WkHYd_s9jOA~@f}5aXIL%gMwR=O%i=*<T8Qt-H<F$Va(<$~fZRXgDDD$x~(dH&vqc7d|>%2-WX~QTss`<92{^W+s+cPgT&N!T7g#2`bkA0mH^2eI!8WkLP}8{39o~QH?#K^>ZK=5dexKaDDwd`04-ai@g&4)U>~dpm(1!1*9H^-0_;o9NR`P1PY`->c`1NE1p}8e`Lwvt63jB`Tv2cORff2Sj}zp#8gZ?NS`D&l6pW(VTJAho*C!vbgtd5fZ{TG^td5${^76=tTeP$f_jz*_ovJI%90K~jhcN5d@Emc|CB5IpUmpO<RV3uRKDG>(q*h-a>;~+3jLUJ5;$Y_&E`{oD)!HF;)sulbGc~&%L)tLh<mh$x4i;M0=si{ErZH%oe!i?y&>Oo=a@i_&WfOpc8{i1)PEyaHeGwGks(LGoaAQb#qI$cmjjjWzBKmHv5IDo_3Rb4&*Qwk-snw`=2BT;{db83^)0v@d{Sb=TY3f0s%nRjcK3JT6}v#IIujn<TmdlkPrI8rySB$m2&Sgv4ke;2g|_Aei%mhSPl&%+j}y>8&>agHu#mn4aPGGqcUdeP)yvDg37o?h7IP@ihTnO8phBUrgU9lw*2i;FfNKAk3vT=%n(tWS9<JWt7qN0YKX{n%!STcxqj~oj#t|J|3eV~A?!(_#3?A=t^G$WAEyQ_$epK-p=yCd;1@j3<3<$4|<f`y-a5bC6P5AivwbzIt6hhYKZE!cZ_+V>0ru`DpIlXufCK?Uo8hpiYIGtD?FTaswOZO1Ki|7C=`-=^)t1ZX2P@N0z-%%D~Ii>nnOreHQRw-d8@hB}X9w-VR>W4zYELBpx1`-pU{~Ec^{r}Z#GQchb0_AW1%h{4zvwFywB@w9n{^=p{MKaWOBLh%VIY4eYEsqMBS;D5HF(-r*Pem{(jS8M&&}i&__R|H18MEZ;#<b3JW)I`-(mIo$`elHv2g`q<GUIy8sRblqxB1Q&S8VIPqU%<(|AdYcO3LvYO@q=VFCoVQ7aIIQA{S(nupFP(PQc`ZEv5l)swQqWRC+!P;_M@GlU$)Av_s_7yE4$7g?)0Iy#Y_q2*^GsA8l;a201ZcWZ`ATC)5iqI03?($|YQ6d0hWyaR;alf`blEmd&Ue<=V4;)FK>^U|Gcb`8BD&t5H4ywjyjIQ%uuXEke-bu}ou8Tm4LnFyMe%9avl;Fdor6VYUlLLy+J;fKw=~MLhj#ntC~}M$3OJ@M#a@PQ;RSnpK{HKyOXTgyK?fWicpt#}OGzbzIGDoLNYM+GpE7GziM&%ioI;9ycDr+mMLT`EEe0+%5;|AVLXnJzGY@Mos~PX8*F|DLi{27U_=HKv9)Ee3Bn1e<te?^XdU&z6bo(;WR!?R^+y%_L+VU7r-xa1%{!zC%#J`+TOXD!M)L$3}b9y3!jS4dn#_6E)fRM5K^TDVd4{2v;wK|gTvEoiy=xP2IdbmdnNZ-uG4Pe@l)`y0aIp4u9OCG*l!gJTq%09u+@SZX;i&Wia3ToeoW>ufs<VIK<8#!WQq=J{UNF-5)8?QMCWAA<V5O7aAo=wVF0}}O8dRTbWf*dbq1e~<C)@SkD3r~8Ov0CY;J3qfaNjQ7x#k+B87aQeQLQ|XnjHqNBwM+5cpU{b(~6ys^;r=ToU`SS-uxYp4~wFXpH^KdKX|<uwY>>M7$gP$N|Vk!AT^6mmnn~J~N2nZTgzSV4$%e)Q%Pt{xiI3aV1s5?`U40FuK-yb<_KGjtQ#YF)!EcdffnS=p4*l^*y~6`A#ZG9U^3XenBsvZ_%f-fzsNJ1t;pW5xOVF_NHfVztd=3Tz{NG-yZiJt96;73$7~$pG!PJb-umN2!=S^0r0iJEu7>ZM~3ENQL|tqf`>AbP_EUK4mElEq<M@Sck4aPDHpx+L1-!Z%#?x_gRmB%b`EitD7A+2r{Su3MHhylprasmPVFfGSBy2640!P`jsK7Li{(9OOnK7I2J^?wp%vdIj*_Tmg6o2WnAwt^^`tHjnX6G5Vqvk#Y2ynyXO@K<2?gfLsxXvd510SL?>F&Nqp6ggB0}RX7s$CxsXR#INTZ)MfG(OlycG20Y1>|FiN6WJCRDg3&_vYpK-jADLnwN~(&&Xkqbjo49sben)9S+*eqAkc6i{<=pJZ#9ca(=s6Z2NddM~8~>Bj0f)05=h2Z5`^Tt`%Cb00f*MQIz7N+WNT6C}dPktR;%zQi>>`*p?}iN@zJbhyo?VE9waE$oi&8VPdgMDNlqStgg5gE`HxDYIEr>sUf2<9v`Z>F$iG**!~E)yNGqsL@BvrR>P$d!n}vri5<d^k4bXmXrk#o}@i+swxfhw$vFd98kruju%5wN3b9GN1NPm$5gp9{sS;DEjKL_*apOtH~9Nj<f2R^bpC}My3wAIAT6|`p4fW)JrCDhecKTN>Tbpgi8I&VSj_^zd8YTXE76aUoil^%3txFq8`?uHyrIHKi37^#SsTuP`t$KG^B(0^1_BUU%GLZp8#!m}ipqiIRwaeq?N1Mr2Vrv1cdcsv5Mxx5eJ1a#YqAYO?QoB)wTjo^t_ODx<lX#JjzSdR`{%~Fp)}Ncf-93;IEJ|sJKg$E`V_Z@IXg*S;=0VYnUw+)&|C8FT@3g0-*Is=;W4j*MW+{1ONl@K+pr-=aIX@I>Qx~d$6mDJQS$w-FxNV0ntO8=Nt<E?GrSA!scNV&{R3>wm&R}LVpv|*iHWCQoDd-u{fqZL1Jw$ujwPQZ>vzil8S!0St#aqts%N952XMolTEcjZ4*8vU1*by!`=OA(j|;$w9&@GBS|L;&7cAaPQixPFZc4*cSB_H`39dW1aq_79m`+5%m20~a?%4jqcShFv_o2{Y`kup#;HrZ)&io9~B2wJRBxKJQYj2I7KDkRw+5zGC`C-x)8F4kw(w^x>?MlPUQ&jNwiiLw(O7o$rYxnsGR)4=2*jOz^-8^S(8HT^@`ZcOu12H0!$K-me$Y&y*ak>Qf-E_o)M(1Aq40SDja1?pb3CIBqV6_FzIg<LIYlBoPJ_?^x&mlH;>wo`bT%nULQAebUGUtp-ZnD=ca$BQVm69n+xY(ceB+ou@8FiA4#Ol3DUZzo8{8*aPy<}?;O>lVCtX(&=W`I-sh9^bDBBWKQs0YF!)>ZdjotO0um7kK~oMB7Ko4EN9B#mo4rT>&_I(qXqp7$B5eQGM{dX*?=t>THe7#s>zD`X&u8lVfqN?Iv!`P+tXpjKd5<@yvTs%No79A=-08lM=HlKe~tswoRdZmt%S;1>CJ1cqKb@MxSA8bE$(di>eF)PxJ)@w0Ai8R}wS@v~nd9c}2Id`?LZ@()Z%<hh69_Ae1hwG_+doiQXf2WIc|nt~$YB0-8(pwzd_#eQ6tK93qlfYD+JCpY!#;Hmbr!vQ1K-Rzs>7nv!=4vZGu-ndYTquLhqje=OQf>#f{x|Wzi32coqX>ruUedSGnu*`}5PGeSB|IrvwzW2Y(Ps~Cfj7Ie}e>}%txpuWfV37hqQ}Z`|7%OMCMTD7V5f<{m_=c%r>H}+#KbQP``ZL?|Bs?CCCVZYuzUI`OQ2q9nVF6bR?u6^xs$b2K1%5ib69ITyw|on4Wmk|6tAc-*f~x`)KnOjKS;j{<0ORI2Ilz<Vc?}C<*x1dr!C%T9el)m`>dOhO6u$%@PN6)>*q9KM96B2rEdQ4Ry{cCuVfC}1FrEJ=YeU=u#`CzBkcuSNj4bV1I!qvq_8WEua>>T-gNlhIgltVMB5zGkJi@7vo_z+eU9RZEe<Jxlrv3|8+zT8raL#eT7GS)_=o_im+C<<-ats|-=FOd+I7|<edFQQr`)m8Mvwc7Pu@M`B_uS#q@>1L$Pog(8b%-XJ0)pU|Ptcz8(N5>8HqYo#X1wov+foD}tP8KBxxOt$js1(Yr8uC#9&qA^{D6}RH*?y?A6|D4g;KTx(vXnyFkq_F@*W<pG>pv0NbRF&M0G}qc;<!a+qC#V@CFPAQM=c|c&qR_*Twb{S$du@Juf;GfkalQiPIkq#1M`sp5I`tm`krHBpqXirnOlmfPWO9m*Id)0~ZgD%dw$O9o8bRynZ9VIbhSqlNujA+<#U;Bx#I#z_K)n14+M%Mz3P*Ub)XGGfeP1h$bdV-VI1ji);hA8xZr$t~*Zq;x}-_TxaE~PZw``n6u5i^uFyu7=Zr-_AqY^BwGg%d4V6U2@IOv7z05Wcnbm80l@}%<kd@TT~}F0gvOh64?^idZMkHOnS}Q;u7B&3M8QY+{Md2{TowUR+-M~54L=!pT!BT|3Kd1lc#uE{LqtEALf)TnkEu$7{mSAH{%9VCv3S_I>4G|37L$>#MJqgg7Lgl0uIxvpjsr0oiM>8NuVb1NVvA+}z3ZvX`Ypd4?*4*&gXevNCgR;uWno34(<a9L{$2QGQNr(*JvMOBt$M-Irgp+wI#CP*O$A9=xq3@mouKQEcwwO;3F=Gl{2zLh;m@IS;>JFY3?i|r?(uju+vKzxM@)X+`B}#1>8N4&jSPX%?UuL`KkHdc+B)eLGmI?|phRm3t9l7fsrQ$foJidam>$CGO@(DMng}r~;9j4SU9U=@QeOHf4tF#;0eF*}q-lj$Km~QK%Q!9XQ;f-bE<b5XlAwF)Km0>>?qxX+<Oh$)AddLwLNX*eT!P#$DF;@}W7}jObi7No4;@xDrql9m>^&-vc?L^A3Lh1ipdWNHBlvAoTuNxCZExrM?jR^sf%YYR3rFUyAXdL(z?xk_LaQ(gvyF|-!p~8pQ+uEg&aV;!bn%01-xL;T!UXkP1()eS6pb|qYlo`jfM497EccS@=M+yDbWiu-PQLaKl7#3is=u5$`8wzzf)4{!XW4N;?yyGS7Ejb=jnO|wgWx}Xm0pC1*3vddZrdTu_D>1vWOwk0voo!<H6SExEz<d_(Z6rVEn1EmQb37eBL~|Gy5W;z<T6#@1&wXd<9dVo+91ne!Ad;y8%Ssbv%=iOMs^1sy34t6-$VQpqRdf9+~KsC_vN%f(E~`H|NZ15tSO}29``LcpjpL%Xy#j15EBoXPXrO&ajhQV6?n%;LppN8@}>?CyS#^Gf?K%bC{_AN4Ien1If>u8eGK(+CkKCFcd{mbkzg7s-*<BV-(+j00|jxt5lPC20@~Kc{+Dswe{70uTw!}m;HgCqP};sma+hWWfDtdnPYKkPEU*<DgQv&Fj)WzZ^qV!z`}q3O&adqFlY$S%q$xmN6KxKwQ|Q=fvHsi9OG)Nqij0ftNt;vylxXA6+DSO2&PRRQLUHf|lUMH&%bfIE(}osu$12utu>1<=q_SJGRS-|@SVlvkXTx<?PuCq?5i+}%8D{yT_+nV02#$5kBS3Ugfn8vdtxL#?2o=3+{+XCtdu&o+@S&}q<0m}uN;+F=MP5E<R;$RF46`Gp{wkWoQ>5Ib)Yx&m=tnD}4r^vUTW*eJ0<E3#0ixA^sHIdK*r9wMjjm^Ii+}w{3$Yo-ScsD<MwOtCwVmd@rdpzGc_l>*R<yi{)ArV@#!B~(aG@}NXWh{dEULhNOJCcgs}!_xUuXM|F>##h-K&v<xkkSmuOb$I+x86CgGq4cz%vPIILk3W!>G}3iQ11yBm$hpkEH0C$*ISP$zMMvVa%MT#I*zKH|RBLtn5v-9WTNzQyZn8gqvFEi~2zp<F=UPbiPG)7f%%M<hsjk?_cqFP>rV@Qy3E|_*PeI_oiS!-nom9g?;WxWlm+#s}8G#mR;m^mq_#kqI@WnS~9!?%dDpfP@wo<te6zJ^46y(T9$?m^VVipCIHSCrK}RpYLSK$ZqN&akv3m7;yk4c&tBv@B_UzLE^&a`Bw0-AS&xs(8nPZ^=fe_`*i5@a3<s&;ASRBTxY_xd$e*+m4)l4_nc9xz*pr{?d8S<|39ojt@(g2avQ<tNY;oz)Lks;mRilj^UWl?Sh%t;L{_}|woF(d0#H^v53m%(gq`B`9Xs0rB0J=GabA=ZsyhF~A^|{!db}oRADo44ZGPxns=(Jh|bToe~<Rp)l%NZ>HAgF_3?&$CsK_QfnM^jFd)eyPpFo@DzWS^48rJaP0Kr@los8!vKH)8NU1&l>wX?^i-sgN+N9Zs&W#`AR_E$N8a!<}rz%-B67c0kzMGH*Y~p~l*<tVvW0JskbP&G<q*(KPrT>SVsVO3=<*G{H(D1W`2UjJlIYJ*x}by(0Y_tEp)boRUD5-1*Ijli<5l@Nib%me?LmPNs_6!hgGI$V~ecw&QvWyT0carlSS`no!c<TC-p&?%-1op9be^P>c)wb6u$B6`>pA=%GU)J4VD2l#f%SGzkA^VQn$-JMQNKPP~3gwM=-iX4TPYj)Db2zHHxIvcy?;3{DB6h+@Lm^mq;6oiefEG^KgMZGMB1har($nC0z)&{^K(SZLL3D78bH-Kp8K?6b%p#uqGu@a3je?vG<rEoow9;cpXDriqQb&}lctt=r0Vo0HwQ48v9-n5oym&U3>&W>?o$H+T94@ah3`^t$@%<42}0Vi-T>>4ji@&>pG?1eKa4v%2efS7%8-^MJGdefzGW#w2A~#_(>%ZV9wN4>iY)hW!d6ff309;J#^LtH}OQdf?^Dl(Db&3V~}iIrblJzA}DCHH(JW6O%_X2jP4H=8OKcrkl1Ov-@=@##ZsS<I8~s=&5@u?4Ujsdy0|@WU%1(V&BQ3D5pskBrDG-2!yT-V`TdEdYGsy)&RJB4S@q~@HrACoU3>@6&v3L5A<cLfv)2BPQ|u`!lnm)7M&I-HV>GYEWB7Zul$#_;)DE7p}Lv286Cp>VgzdFcBlxiWw+Jvriy^g$mme&QTG!Yrw}w^_}u0#)<MK|)G>!Tv5_AJ5Hx5+zx*Xug&?Y+mr~(B&+O5OFRow%BGglDj4gB=k&$o_!yBNGJ=@T&AU5SG2^DC9z9(mzLmf|iv8-qc`W|dD_ZewA6>eZB?=)d|m6DUx2n5Jyr!Ts;4vj*V$?=2Jx%++w7fs)Swk458g9X$_p^&ZTj1(2!*2haHF~Vgt5XYncuLIII*mwgt)Dw##Gu)lSN85PxylGXXs9UFY9KDSfx;ybg1b(g{*s`gu_c((1XCb3YHFZ`)yjx!GOzt*M%R>tq0Ft@ev$|^*aC<*tjOC95Yv(>??P~r93=GOaVL#DPFvj!@#bne74wqz?6;mySnBl<UA%;pbD8RI`IvZ4tp)Xuc*8ovJ5+18+;{K5#FMOgvwD6Ga7WORJL+B=g!Et^u^-`@7<_#dflKb6mft8?CxdY1L=F>M2`5Y`I#hPhaRL1By&?bo}(YDk|LT+m#%cO~j#P!zV#6!BhNJ&1fVJtCAr`Svc5<rC?0ij`)l@Y``=FO3h4m~*)4eyu3ahDiiuZOhJshDZXx{`EAfCZwCiw*%Vo(#q6CGe{;{r|$vKjYx9Yz8y|<b=T4^UMf=utfg|&9R;qu~PaX*ZffkT%ZuMJu59~Tiy+y@%8DxWsI~u{p<9FZT4XHJi;9!w0FQqzE3!%qf~$3G6w{trVQU^9Y-m9)?#l-i$8K>qpN~7@IONYLCdqgNDma*uMQw^CS+0d?}#rRj-a_l1uUHYS`ren&+7e^Bl&J);9IOEC67FSSwgg9$8<E>ak-Qi5kjRzw7`(oEQ$$R1{r28(B;vP(nxhxh7oHTlS;5$(+DL2GK`eHD}E`8DG41ytc_?tJE*wp6>0|1W2%<|#_2~aq@2UsM&{}693!(Os3!y>NXqpj*BfuZb;@h#g@;9EjKNDmCyLT}XF;?D<6j4PXaAtlt7$~(T8`JayRLOLp#S@i0~vcD6fmqHhl0>Gg{g&Mh&LTETOgg<y!G0y7EQP@NGus^o5r=*IF<+d-{-Ks0#?&#RR^Q0|3LFYtBVu*6E)5~9r2-fZZU`6C^yC$eF7<$%8{DZ-bi)DC#rveA@~<A{G8f)fZzHj?}9>vDdjE!OzV{m20bnTSJEW#XOf@GC+XW%C#;wcJL|zx;Nb%BkgZ_P^@rwf(@m+lTNB6+!GdhHgI1?uM5iRj{++#e_)hkU4I$F9VBL@zgXCgg^W@l?byOuE{R*6ZuAbSIG%1g^U4ZlrA_BM>CY(*pBrdk!tq2LTbUNyx{5@7w;k2GN7s;<*?#j7T>Gw?NGM(#Rg`xSsgaZEeDzipd4P&h$Bv}g#+3gaeoP|yf9;i!LA*irh0E#VlDS~xv+@PBMQSaCr?*Sd8y3hd2&Q34{MBK9QI`*}PUBjY7pvi2Zr2f(|lO9&5M_${&Hm<qp?ERh}{8RBe6(<<t@sJ9XRr$uFZ*ukyNDry}122rcn2nUa4!s4imAzYNhc)Qx|H@v61!5yI1jnSG!ra?fMJ?L+yW}4k_x^bUW=}Ae`Iblu{~B;0N+BF3R~{Urv;4!Q2VTQD#Mzi)Jb9MZGFk8h`yZVj=0}S>QvIolK53Hec3T_%;eg;BTzT443*vj8)!ZI+0Uo2C`W@$O*2gzDiphvyL9RwEn4d+V)gTm8Ks#7{`$fl7E$Kv<9hS(vi;4-1+j+cA?+`Ih$}HM9#a7MnSuljf9dM4?RxX)ts#C4sHPU|`{0GG%!%9FR=_|xNzqK^L=E>s9PVzy(NZnBm`q(fj$Du{Jlzu(+IIfnsxaTA5<NgIAnB!gb$Wie8VK<&feI)7VbvHl19>lrWS(cCb)9!5-MAkye%v}YHs%yakI-eVFtV~$FMFPXQ&=o!+qSwFw;xVlFHIlctg+juEoonbY8hZhxsqfJwps$*`oGk+X#TJcGhp4vyipu$<NvnHIY@+L(vXi?EK`>e4m8#!emGEA70fw`kk`I7=l)ZewmKbpPw>|P9ipbz+H?CwP!2|9D!`KYiC9Ohjbp)U0((9W*(vep4?EYfBi@XM_EYRyU1LF<#Bv1rVea1466A0+f$ECx9936P}J^Sbts&%*WmHHf<p{2gX|6@)FuXM#aGsh95pXw`|T|lzs$ghw}asQPjTD%<~?yR9OLU%+bvwM_9^RE(P$_R<G&mJO!rH*-r9f`<A)PA1#dvO$gB9T~c+;w8IF4+od&){gn0KMffV(a$QC5mZe=}`l$w-v;BYjBo+AQ2ka{H+R16sIKFH)s}c<bV-uL5eaUbjlH(+5fzJB|REdrjn6>nzC8^`*O3lC=ObjD~(9^f|%-J6n>MjlJ*AOrCZcGGX&xU%uQx(7&`K@1&gC8l@XDDBDX-QC;gEolsd0A_iV3}o6!SDDLJVSV|ZqXu#unK4cF=jhBbPo5$d5)Bx+i0%e5qbHQ0@0ajIT32F>SW=u1b8o%U7$uyj(V?Jqkb2G<~VR)7Eg9oyzPCzZV%7KHGBToLt|Zk*#5^j-kF<rM@Vw9XYjJhwtO4w0bV2tc|S>w$kvk|=A*q(BvFrD7}Yn6v--R;LY=F|{fI(7tl4Ys@FhLm3%P5SAKG$A{ZKh>5TLbf>T{nAG~T$Cic@{BV=bQwCI?*n6O<0!L9Mk0}r9(?bbim>*-CI=cjdR?GZS{yvXaP4Dq<Xdpa2Vhm;kkijp`>z_FVmMn75XC3l)F^b<wu==si2v+SMTvA-c>}w1ov!JW8sNttf#nXSsh7$>HQVO;-jO^V;99;M2HI{Ab;W4rm!#V8RtKNtXCkOUWno|WS#`;hsC!uXwM%VxXruKT~#}W{vkija+uJ>1dSphYx#HGumH9^CR^x4+kW)2to+bx2Em!L|tkswALfyU9rqbgec*-|xrP>=uJun^ZIWirw9^QD~lNrD}u6_a{BeF+xvps+bu>pW;Z0I}s5EkDR01li|nKLUb9xqT_bw5B+0kQ#uR{XGIhGp|EcGZ2}B<mZb`X{$;6v2_73XkVW=WE*w11!RCpjQJ+a0SL4HV<O$D2IX3e=7;JY9!yAtfWNClC6swnoK-Z1vHL!k7}yapK;EJY%Q_qzlb3`toY_L8qMuOrRZDV1T}kPhP|n_p)UCSIN)0fS5?Ml(z>{{Z-wD}DcFpjaJ5C`<lb1Dl5CRAk;|h|qG0eO8??Bv`$h+&Uz#IU6lh6x{0Ibs=uHd^w(Z2=IA#SO)FROsJL4;QfgVS^{Gx8&1-0z86Q1}i>!^ZL&DSUjCGzgKUNw0X!wa*^TiL<9xKjjEm)ywONkueHI$a~+W&yO5}%<}1`$D@VgUvIM?xTGMQB{7plC?Rj1mb98{{xJR2-S-rPTjLffE@L|wjwly~IiB5y!Xb;F)?Og2vo{O>uHl|W`J{DT9p82siP}Xk>)o7Nk@JD<w8EwJ5s5+V^XDSKFFg_mi&gQ&?p;A47|qK|-2wYn+`R3a4h{%!e(S2t6r6M+%?`J)8UPPTl^9odCm!1@2=T$_-E3j_-$71A%8}hao4us6uF;Uz05cRJEXHDOM^_Q99yQpU*ZVj2a?O_Vax(L+O9%CkPS%Jah!DysW#rjfmxOh|1owy2751u=&-m+(v0s(;h&MwCSahJ~bQDa$2837T@BRQs7P=!B31q9{o|u(=LS+}NCxAn<Nf7}3?QJ8d5DkF^frpVJB+wUDwy!@1@zWfCmEQ@?EeP(@8z`%DYYH%a4+eaV=+zq$VyH7FaxMMn?_6I}OEhKp>;uU^wP}D4Hv61#a5en|#V!Ox6xRd4TknI~#UEQkiLefanR9X$W_5PVrPb@wH2l=8w{!W5(r;FjaPFGQ`)PPu<`i40ELNb*m(O$*ABXhw(`!s-QS3I;_k64@Tz&GYmI;1+K%i47{ry;cUH>`WUVAAVUS!hcC}P*KmntNS6@GUgVriQTjGqI9IC_Z|FB||fr|hU%`b@eX+3g2OUPP*r7ze=K*Z-i>Z65F{!udg*aAYGy35_i7hZz#<@S5FsQ>up5h3u(R808~oyDD@q6*7Awk5Hd*xze~|!SZdA@fPPsQG^B&kyT-tOx>v&%KiS7JjI$Zi82}CQx;GlUzB?C)Kfl?C;?4@!}BY(&!GJkNN*tEN9Q5Jn6YcuXjs?V5KW!Q;P?EX&RmdVl^tsVLiR*<%|9`b0E3aeq-C^xncy3Ug_*0{kSD^85V|oznxSxb)0*mG(QCc?xT6IedA4l$uSEJ8xAlT60@Ew@IdZS*-xB3a7*uK4zKE5*kuCpz>Z97vTWNayOe?rh^xjBj+12l<=V!F$*~*){z;WO{;_a=zgkn(#{?JU(KGK{7)ua}v@LR#5uVy-TWx8;s5Q=;08P3yyZ_3t(`SPT|T9CbAx3|<u@AE2i1#MQsg}5i4u2yCZ-kd7>M^~*#e3!|caTE=yiyzAC3)o@h?t4>ek<7J%*_(g=Ilu;I;)b;<J=|jB-VNJNpULw1Bb3ON^9x(Wla1ElZNOLN<kkPLDca${h5}_RoGG6DnHU}s&o3wS9pjXim7cRAbJpZ$x1_T?7$t&})je$<9>VLXafCg5zPl{JT3IyWmp0->^3yV^;e#S{y<r#T2?xch!mj6KaWxj`3+bNYYYW!{(dI#d{Kedn9ABJ@<ID7KZ1%<@Wev<hi$g1C5-g8ctb^}8Y4)vuKtWc<*HB@R7h<aG5touL*dT{;U(z;9qNZzG2w!VzI?%MmQPACkcd6FOU1@fE^HSUN;oYj%-f=gc`CKVtSb!={v{1~#;8y2Vcp^lPw5nGFOIoD$ZZF*qRW5WvU>0y`4CwR8vyDr}M3gJjV9+Z7k?*Q&V!b0EXO}N_KG%KtRxirj5+&q70pw`$ZBbSPz1W#(r$9So18N<E2<c?{6r?GdH62q9g{<J*eZ}{H&n_-ip_wCnp{`cqTtYpWicmbRL9}#?0*fwA9AQBB$X8?6^vUIy9cbQc3Cz$|2ld(0Xm)n^ddDNfYK?<?o;N|2mSsp91|Y`ZmvA;fh?N^8--kE#am2f&1Ss$5V1SKrUg?VK(lvFDJRO_mK^G<uAN?v;aI^W1ZOWbI>YmYNEos<N;7wYSP3W>?k?mK(`0|<nyDzs1qmU`m5K;&9bM6_@<<;eTB04qW>Q7T)TJSBZ_)RjTt??DJjwWaHuew6}0E=yfd^DyFU-0+^z8C|4Mz3;1m?$oE4PP&E62P;@3Mz8f8ZBUBeL)(;@W!P_xgVz8i|T_gRZnfNvr@@1?V`qIyo$9kh#*%P>ayl0x_cUYTnYplpcInlBI!?f556I-K>FA^za8{N9vyJxT;uM(9uUKpPbzM`OGjPRbhlev!NuY)1bNnEtz;-9=%ZO0$Zz$C8?mtgP<f;=#|{{PqFg(2*&Fe9(&)KO)?Wd+%}Y#K@``w;nrV{m^P7|gSePyVW}qyw7H5Wr{{i~_pF8CDjicE~0;+K?xu1<H(B^W#{|T_epIz~LCsg;jfxH(jf~lh+_HVhUQ+z|e?Gchdn%WfPk$u1%v=1AlpY?^C4QojZ)yZR_&u>!$)0v_#^C_zfBi80(+w|#B#X_umix0mx86Rl|eYjb&;&<UlLZ8(RpyN8%lWhL5D}L8i#^plhWQ5ePJGRljnL)`9OL;4tQ_F|=ICuiB*GxA!VbNN*3Kw5NY{a^IzD1>|?pWDNVN-!e@k{V_*E_u~d}50LqxJ<SG3y1Tqx4YlCtJqN6osYN(qxxJ^T<^4vi(c$E^uua=n^O6dIg8ZM6QAh_1ii~F3De}w;_?7mRSq*9PrJoiHXwfGw{&^3-E3^+KXH_W||S?oc~eP6}LecwIa2qla$4>dQ!FYj*9o~Z(*4u<RlGfD@kZa-JJAcNd{V5ks<QdGhoVxaP-yUI@14^TuSTHa|TO$1+bQ<2`dy~KFoOQ7PMVH^uZJYOiqXt#>~S#wJNshrfiuvfxS)wX@Prj@m{Ou3|#ola*^jXt!f#NUGR+K)#j&cFW7_l+D6>FYc(bN^-2!j3_!<Kwod(2@@7F2A35;~aLf%kgi2PH?RZP$XjPbl1um!2g6{`);jkTY3O&E$c{6oT{jMY64iTGd8^uJAbH06g7&=ScJ%gm}#h?I?AGpulxHxa7*%sU+u-ns#$HmF*OINpAtP3=0yoOhxX=(Ou=#34@iWT^fbWWL+3#C@L)3cw*JD0QxBLHHl9z<(++bd4Z=<d2dy98asrJDC4zY)RI>5SG*A<-cZKj2d}CO~OPfZrJ(y%Z`P;)pA6D~(~Z9Dl@35(R5A-rdHyv}~yDE!h$wCq{~vL`lKv181>aGBA#!6r1JJXKUtHsSr12<}C)zYXB*NdI-8@|3a6wpH{n9VYmCN2Lg_FpoQ^3&?ErP(A+oBonH=6Nnq58dKvZ<+_*>$Sgbw(Q52s?P#a-ac@jmuan7O$uOrImJSyrGZe60>zAF{QA&>Usc0#t6KXDG?z3yZ%;Xu@TZ2Wwe6?EYx?OJU&Y|Ot*Z4ui#Af@7oh)8BG%&HRk?_v}lS7wi?f^xR-##vsKfYDF{^kS43hP6Kp_)iveHarSaM0G2ezYnTQNxWv>9dn+fXzwRF|D*FqXqTr@=qH+3aaXBM>fv&X!cRqp7ACo>+GB>`>TGKgFj>L%cA3>B!+Pt}JxXp1;EjwiIiog%-&-keJ7;8)@taJqCu{>g$D%tN!S^Qe`Nm7`QOJB}6^x~@d5=15@1oAlU)OvVr0y)xC=bx8@jS;v^(QW-yc!bHkd}6U8!1n<_@EI+(E&kiDPfGd{(C8p%!~-T0L6xG-NN`DjSbwE!QC83U-Q5Xu&nsWnHbPQOE?WIl&sbZqqPd73M3hM<ypu1LFI;RZPnEKAK;~lXK`oW-i@t^HOfZ&FeT8%KcL?eClkFOp$)yY^pr~*E<-cfKi2Sujr(^lozuqi@PB%GY5SgDJ>L}qlPB3)1q4Jt!+_0wZmCj@S8SF-(gKg)UAw5^CJKCvH^Gm+dr;O{H}{!DIu!VJw;}){K?S5bo8%2*j#=zu(bLRs-c5)-HeJ@(#A<?mh-RI!Ytw<f@HMj2r75v)3HKXviMRlPF^cuGPuxjNBBn<~rV{CAHEz5@#1PQ$&7Rk(xCuvlh354Qn2T@bRAzTQazXbL4^LA%odAC#o*dJvfkDZ0B@!$GYrQrw_-YRoVYFxgPpmwIUQ7WO;8<t64zmLeu9!+Z4op46wZ12k!vCZmIt!X+Mjrd@-25NkfyX7Z|3$wJ5k>Nz!6K7~c#5Mz<d0)~e01kOh5mtgka;^@jApP_igpbhlTQs@1H;ABi8ZU7>}M!+XnS~Uc-z-^M;E`yJr!v-tv71rZ9z5&NO<VYh`N-VMX3v%_T*3SSOP!fBAA^#_QFgsh8mhiz-!{qJ-0sN8U!^3E8TZMs2!X77xF?gj!<UkD22>^$fLR^gaj2S%(Uz8|Dfxhb~D%+5@OKtEJhea5(>q%^o{V`MRjh3p|W5Dqla(nql1n@nn_kHE<P+33zHU5ChrxwfZ$?M$`y|+0xm*I_Hx|5PMsszCX>+sP)L4+7r~$1V=$l&I*9Bw{o61IKGB-x(Nw>(dTV|$sYP-3(toL~ojsvdv`qv}`0^S5Q-ItH+9XV3{3uE}fGp)oM&|=rj2v8qe_63s@hd*w+JH`eDO8aye*<3kdK(r1Y=1s=f#q2=1b1>9h~xZ0fnZJs4S3CU)(=U%t!cg3X{U~y?8UtxQ@cqr?w`?aj8RFg2J&{;YOcChSZTA^eLt0Q*Vu3}B5@bD8IA}<a#1NipGmoW-=>yc?$yBle0N4{-5ntqm`fKs4%Y!vR_T~|Rj8*h^o_cA>6i!2hlN@Z06zM;J^1G!L|Cpic}8e2oZS7Je^J-6X}5Ryp!wDL9Ba3rwOWztq6ik^UqiNZuPi*IRLCA<LlDM-8v?J+c{3Wy)QFMo%o<ca0r92)i*2qkG4?P1{~z#aYQ-|js_OF@5b-6+vjx-gkM}4<dP|%a1df{J$lJ+ZwVw%jG`|mH(l`r}-uO>WTmh)sGKz47-V|MD<Q4wm%M0Y<o+%er(TtzzkjzDGQ`WVsiiLAYm_h#{Hy@t!r~qKr+CYY($J7nhXIO8-Mb~zEMb&36emZIG)uz}qXe3(4ag!8X8~X8)wIYotOT{*J`d;yhSvv@KMvG$HP3iT-PDfZ}B?pWDz2i<OQ|)Qe9BDm{@z@z>H8^cB*;0+;<jIr`cJsJ$exPvr!$|1v)RcBc&Q!lq3`$1%r!cPBrWho!=Z_OrHpk#{5Tkc9MRrsz;5AUT5D4t5z1QRDu~T0SHl~p#PFhydMI#IMn^LZ`eYaJ;h1b<4=hcJPZx&vI*1*-HdalS$BRCTc!3H%bYEtO(uK$!8@=QQ;wTv{M{h-YTw7p*HxSmir<j~KNZ!2+MbxL<jI}JMWqh81pq?iBZLxJLN1ntR@QZA7gn=6odqSg*vjtv%;JgFGnfKUcpys*#-m^{5#Mnzj%WRg^RA>2o}vhyipImV_Fx9fm(YFISTpfp|lMrx93YR$VlvO9>W@j+C`ddlq9-Z<pDogkVC<*J8;Kr`@(fw3clOdo3MgB^mBQzZ+gOq@J3|EIWUB2YkGl#s`D-5km*_kPSl$mba>T<aheOJ4+Miy(NnR%-!-9i2Qr-T3wIQ9OopTB~F=mpy`Tu6jE)M~gXcJS@Wc=q3Vs$7AcsR}9tfZPa`1TndZn)K&s;Y)%x5@${t4tvoErYGs`OkwmTp=f=Iqrc^hXzu&gHu<^w5PH5^oLH?mlMG>>n950(+Kmn0WN0#2zF}t1_BO1`7yFZq=T$s}+yor_m>u*O^t>0U$Hz~Lhi(;p%%E3yxg@%2HJ#yngS51ifBPE>Q)_)Qo4AHK_g<&3)*)FJaQ}vFf%I*gBT+?~6t$>R6;dHt-oY9eeOZ8etF>Du~!JKBgkb_?l<yT7R?`hLA>JZ;M>mAyF=3!eKr*qT39QI@Q)Vn+K?T8vburYK$1~^fC^U_|AvX-uoRZBwU&z4%E#P$rWDmO;{pKhT6203ABg~yu1%>~|f+$5`D4h(6A0*qUBnFEVWc?dE#xz40`{HK{QCq;#_;I;Hr303{<A3LSC2eF9JbRQ!pmo@WpDn$ZpMl9!kI-ykxa^J<yHwOa9LrSB;tbJw1jf1qY_NQr;ZOH5wW$ngEWry(c{2OxWI>RWS7u(n|4$=D4_skaSN-r7J(am`-(<!p7l*1%={uE=5SM1umY~Y?<;MejqKwoqcudnX|cbVo;Zj5vo_`9G_y*@eK@=l11MDoYR_j3cD#Jlz5W|C0T4y5|BD%QNkR#$k%+eB77DDxzY8hR$(nZ%^t>UNmywTt%hFNzxtiONk#-%%vn+x~N$4}60d2neXS6o2rr-{(H;V@CY-%vsHi|C0q6Ws)&|u?)5c4yv-4UmQ#s?+SIEl4$2oK7OP)ykAhh1bdI?CaGxz#w6hALFq1kbfvrXo8x%sGgjC^U=+hS236GP*cuahnxTfsm#Z6>JCr*vVJdYN35BilgY=_(B8KBv*U1HnChsL+lO*-fY;-uyCFanaH*oVdkw1R=L}4%POP2i{Bj0u}tj>fGio2&eH;`Jl*Lf#EMp@YfVjdvEWKrFf{SDTlCdN2AumtU~KZ?tTAEpr#Xk*?Im+yUeo~h<B_X~k)#T4oY*k6c>#Q2u?l^24pCnA%UB5u}l;wU2;IKzC;SpgYTrtGJ~K)glKX*d+&b1LBAA(@Nd%7!14?h_8{{)LJGL?tu{hLT`s!{4NRJF#iqKFr811db)9TR=fShNPjSGU~bif?8INW2%?8@U-ukIRek|s3-1-VZ6$5lqOsni)gK47G{ueMBKY>2U2K<&Q4xT8|7r>-QmwkSrm$^HMCT-aeyneRZB!E;QK>Mnh;}N1}<!tC(`;fTw!g8B{klOZJ~Igg*hnDWB_(NO;cTCn$i_!OE(roae~3k{?WO1=>-OG<#5FHAUjEhrwk&|$xC_7AvNu{l4n4l$EGbQ?xf7-DU&efFz4lCR?nR&(U>CC)obbdk_rIL2<DayGYXHz-^Tt3>EX$e8k?*iS!V5UP^@bhp;CPsnB;B~7gfnD<s_(xjhEb;P*9r#A>*-sfl+Y2yNy%Q$FM{Q>div=v{71+n8Ug91${n%&swrSv+G!(n#p5ZVu(B1Oai8=${*(rQ$3wa9DQ64Dg_Tu93b6VMNJQ63d!pE)R0m1=o|d`EO_C@VPR}7?AblWRa&IWBPIR`zdUN+oxhU#9!{%P?E(unU-X$i!1a(g^HGTzC|YYFb&c{-_^~<Y!e#bh3klxT3ENK^EU@TvS?uHEGV%2n6qeDAMpwp!zNi67?=2f`HL>N}41(Km6vq<57RRzooFlQ;WbPaK9I$E48WRfCr3n_YnP%1V(CgDIRyo8~)vRa$gqD6>6dH;fS7o01fPugsY%eL*#%%BSK%*hgN0`T5fq@6k@?|VhcZ6+9A9Lbdbx}YR&8QAZ6|>ptJyD&~(2yW(IXSa1m$_+<<S#`=A{Ab6iG=+y0I!J7sl3d(lCNPrQ9f8%jZi($>WG*=Un>W`px}%`AkZ9@!<Co|z6<bE=R7-lk;%D**oABYFJi|{hj!L6L-Ir_J8y0)LeKoM!k`=>`B*3!)b~{`aRmPw@rje70OlRZ!GiGuv9uRbFkIyNQ+%Oq#r~C9V|muX1(K^YXG@G|m}2-)s-MSuSG$XbZbdbl#(^fJ@KXG%nFr;oY8rt0&T|>gTRGH^oI4lfMJRAP5>m|;vHSAR8tJ#U#0R3Ahe&H=U4__P`F;)DS@mi}V)~01mCK_#xgQA&b#y=hBv!Opda*W@*z3d*(hRZ(f|khgB|G&hj8^s0y}JOp?k>_A;A^sp{+zXLYE{K9C1_kRXNbtF80?q`+Tu{D%HX~Ooskor{3>`|2-13Bu)$^WHG0~Da&UqD3KbT%!SS)A!T|*ynR^YqgMpE9_kOyFn_CnZ(jnsEQ7e=TDJg%&*l>>U=~Sh?img1((9tfO3BE8sIX52)fC+ztmO=j&X4V4{1|**`as}Xm!sli|9F64ac1#@^j5h`W?EK3ir5VY3;!as?^np|UJUQI}XPV?5UzgbPlIa*2X9}I^1?TZ|uAE7-!KbM|B#&}{K{z3?zEcQVVb4k(pZLdo@UQf1L1>6WX<Ni+E5DvgH`?khY}asUNveMrpenH}5bSUzXyh?h)w>Eyo?+rJ*mOu<9!8FR=BuY7<iI-|idvCT9=`Q|+i~N@3N2Q-$P8XxVKA$J)suu&>5jZ2L|M#!^jHsxD}35$DIypZtDpe3)OdzaE^&NPqP$OSSK&}VhRTy*C(PTr)FX@d7cBp1gJ&zLQF7UJheR#@tERw(dOFIh{8&&;Rw5OQ`24kT!jjTh(X=3pKKnLyCvpx^XyYw`af0{#TIi>_A5#(w{)$Q2Eq^JQUW_<hmCA00tB|G_a+Z{T)RUdICL)5oT7BjNP@_yU)Y0>{(>wmBHtI%as&$L`afq}il{YcYMUjug{0ATA9i@5WSw<8gdG1UYs%BD<s}lbhIC?kP<ZL1#u~_UcAy5f?Q1kKjN@V|)Y>Iwd$Z$2h^psqp%pz0XWMgwy)Jl?O=|>r-BDsD4|9o1I(%#7ka9~|*1F{`?<^0j^6_(Sc%OzpMIDE*xtoBGwsKv64S14qKCEEvo>1j|U6rCxc4GmV%XJlX+^{weZqQn7LLg#~ftNtn#b9NbIA^pHBYODxKbcwd_4<IN`v9L+vR}<dDSUqs2Zcgh1zxI8R&n97<K#UL^D+)Fs$C*k9$Gu7H>{O#f_=C21&_}pK58Q-S^B#$>7(+{~r7n12k!kt*EEhVg?(DZu87))0!c0IMC<oUVNqG{S{sN+lejM|!!5NVdq;F2pRoxf|MM+ZKfrFI{LXEpGXd+tk!Rm0DPh}KfX!N@Ts=N2I*Wj4~aZxUu12)N6xSnA%lt#q&APaJ;9k{!xy1u;K@z9besRhHb#E-2R=D$Hmx7$WQY~ogx(n0YbFt~U{k6-37a3Daw4hsPsdN1(U1^1h-bJ=w$^P+!Fr}5K;9qQyLK|vnngDU{WeN`Nui`pagIaosxIdPr88|Ku&LdY}<$bd?g|0{09#Ho-HUagN;CPF^qF}mVdtZ}JdZZ|DbWnWeNs!}E*0$=@<Gz$2w_Sih2Q8ix1cKYlH_>EUEUgFGE?ZAh%$0T^sD&|RReXQ*7h?zau0WAe9pND=cvAXc%8y|4g1m{g52`zR=t-e;a)4X{fWA}k*S4MIAqJ9-G1WUDRD$ybqp|7<8rf8)X*~SFH*PR>k4u0LH;U{9~Z<68r^wD@8;ut@1@>qtO>4+6BqXv_ZDTsM$3dCe#A7AG~6c?8XglS7w48IliQGO4~8k2qqiC@>#d{rqCBb$*3(3OD8dXZB#o-0U;|42ep12LHrU(L{<-oKU{pTa@wt^_4XAjnxk&sD8+f|>yT*B%g*$7+v`A2UmL6}?|2_JPACosJNB0kSHDE>bDK3>x$4yI1KAbCB?Kz8U+gZ_qxsbK-Ebojm(WN9%wKY2&(;cV`3<5DCj%_&ZEO0F)=%q26Ja6@0bse8uK5RhB;~MzPRWZ<+{gL-2;QwqUQrN(O;J9b-hl1!qi;y@63b5@k;Bzay|TuUUIMr&E>NDJ_Z1SGSBJufuwA$Z!ZeJ7G$tqFcsulV^IH@eFGA)0sOB;5P}0dt;*#&KJY~=<&=RocBwRJUL22JXo%L1;Hu##n27Dg%V9yBdfl^12;AnA92Yx<vT`hOD49s9Oy?A$Pp%!?vqaP$3A)|YOX@uovsue<{aG+^7YCQ+Q}5-+e*b_`ij})4JPjP`gV|6o~tU<y#u&dMCSzcQMToHswc9MDa;lq<c}LLtj0ti=<{C7(yfd7^m!v(OXi3TCbTEhrc0l={Z#WehopQoaK^kjTcY<dD{~z^xfg(@)_=^((G18*V-!0hxikF3p$>lOr^}R*Kk*s0dda()BAy+W_Em3-MhqMM%>Uac%$W*SVyevASK$CAWO59^gz7J#OKdTQ5v>^6s#;`MD`@nUdeoC8bqL%`Pd9U*TH~fMcS9UEi8ts;SdL3hV--E~Qh=};K*%g;a_O{t!(@KMZU~Qa^EDgpG*zJ%y%K}xXk1!yOM2}=oA9Os2F&B2fw(Ye?Pew6*Ok(GAWSK4{HTyE-j`2Az$p0BvLc_ru{C@8#%m%9r*eG?=i>d72unuU9btW+uNvTPoyW?K)l3jf5EvXBed@K6L$V>k;F38dR~6Uy{k9_pehXA4>S5c2E)P8lTh0A{o!}Hcg7DWsDlE{+iLqKLV@76Kfi3Z%V8Ux4YsqEOd8p`yc_ZEEj1xpg@q0AxyUXso>_k+4h9lsEHBgBZu3Cn6XHZYD!nEx{liWLtM&m|hykijjf#E<f4xZ~@oMmA~aGZA5Z9q;yC(3wn`GQXv1e&x0M2CRUdULX)QhajZDP^Y6@sgJvh7IwZ_o4mR$b<q3AE;kJhxRvCxo?B0SEK?x@+@@Ta_R~*avDN>+@^(IJEnG`{`FtWVFXH#2jZsVG83aS`3sm+x-bfA@rr%l3w3r85vSTm(3DR2MS`zF19T&i;R$>f(Q&rXZ81z$llG@vGOh0O%5V8MQ?}ANF_@eX5gZV8+b?G8)RwICn$rnXv`eiPSji&GZ7ZvV@&1b5(y@zZp)uxZo0m>g&vWME0M**~IEvawLTkf}N@$e$c{iK`@%B5ZFu($zksVlq*tn8KxU3uMvA5eg%P<^NsWWMyuR=m}kU4W@K(yj9%}N8qL$tf-%Tsna-}8zEzPCpv+1f3Uc@_m8f{Q`}q14c4VFNMT#4WM!1ceXZ{v1WIf%+xdr)p&Kx?7TUH#;%HMhWYuP3QQEa-Rj1s^@#MXh+T|Z@Gf)Vk-o^Xh_BJ!SB<RTVl+*uo^bkFm+p~K0h|^QUj?@8ME`PtJ~f~)cW_ifE+dodOZ}@vM9b8WFM5zhXC`99ie-sB)=wtAo#e<42wM$IJt)Qu<+}2Gd8UUsasxRVDd6jF&(Jpi0^4lUd4JoJr_R3u5#Jft0$y})nEw>7dUwPd)qThBm<P)y{hS9qB^YwPGrim7%yecn*NlrY&ayIRjbn3$IuU|ejN_sL{O=aMO5bK&Iq1taLCz(pAEP@Y`g*M_(p(0nJ@jfV>1$t_6H+Gys?$fHe`2R46cJG%ltz#;3KYB{A=Tx<3BwuHkhLexrO(V<oUI&v%!m0Y;yV<Q}b>=K<JzlfeU=&t2SIexjsf}pK|dQ@;6wgg`<5b00U?I{6u0wO}U&XJ~h>P?JQFb0<E7fJl4oAIT62sHG|kb?acXF?$%PBwO7>DU@LS#rx0#n*K#bh(CKU$he63`JhHQX5vlnD85Ue7VOTI0CyC`Y15*hh$7DbYS9Z@8%-;YV>DwuKV3St8h22zqMinfVUriS}r3r6~z|qhsuS=L1yO*;2w{g36J&0!*m2u&U=u@A=6F7($Y=uEp7m%x({?TRPu6Rhak%dBdbbu|f$wQ%ECok!`1H>T`?Lh3Lg$>bGKdj<*i1<ZkVg$d2R?*8yDXqCgER1#EJA#Y2No-Cl$K-E|N~Ou2wklRIdG!RL6?zmV^3lj6ex`^g%(r}N?((k7%1T}0rJZ_aQ%KU`GylF&Kl}xAzd9Z6SeKGh_wuA&;HoJIi1dC1KqiO$^mK5va6Cej&zUvrjOumiOB__kG&($6H#`5jj_ZE$)Sp>Z;`m+vArC*35hsOS3a)9ng07K|A*cj3D<xB@=Kyufn6J#}Z_W^o9ngg3&ZO4P_m+7zRZ)y=S6i3QM0N@b$@xs8BoQm!3Kh(U#UPDx#4Xds8HpRqoo8`op+K&TxSa#(6&-#K+{iN1;Ce!cl{OPf!IkcY*rN}`6-%W8!R04k-*x-gJQ*G5U7&~h8iAL~WXFW7Cn!^7>XivG8T`H&&GJr<uDD7e?9dwHLT;!Q&~c4#i!^T#*xvv>CyZcl_%&P`dg6~cm^0&u7o~QQp>>>c_?#E;a7BubdE2%%B0W@@`jMToY>|7${w+qp@(FzZkv<S$vd?SY0hdcif^L9ZiLJ%PYic44YiL2FC4H}tP)eXuwq3vK#_6vOC*J*JEJ+WL)KIlJ-^niL@~c$y3obv(b(8!sP6mWcw%IKFYbtHHva(zs?vn(FA$1db?@2`T><!w|Cx3=W)f4@fzsF7RnB9$nSakwx{QF1FpE_>-SG69fES4os=W^)8BRJKUzSMcdGI6b=QgIK*v<Hb|T0~GZ%w|=0|JI#Ii6@hT4KuX=J;1qWz*NDlbF-jqbEAmvgY=2bxxKj*uUxj|^B3A|qYl<4i3^NM)rtSFKQb(tyveRO`QBYmNq-0#Q!ISiElSChRx=a09RE8ys}*16wi{E<8h_|FDTNh;z~ky|LC%v#h|GcZc)uO$_3)BWEo!m<;t&xj>L8{+5$d9MfLC@KSC?zlj&X>DtJ^5)e!M6EzX7%3m(4-?EfB2eUWYT^kO2PJ<@!UiBxGXR3+Krme~6KS=`MM9Dtbx>5FWZ;$sDB{q*E^V7p$yTk`zR!QDHj5eZ7Gf@!voODn-2uO?%yzKj6KP9%00q!pPIMTNj`%K2f|A*B*|6>f7@mUTe`$(3uJdXO+1Yok(5Dw0@NC9OnWqz09GTpZ=RNNXE(&*}`yrjW88=dLRC3p!xpgSPWxEx;h?VyR6RWt>*m0<%?cwyA2kWE80)Lbez^qrLa0R00i8X%X6gd&6$)0>UOkbot}luLxo*oX*=*2A7d%X?(2|tws8Ih1Gl_7YT|JnlRcfj?+fu&JRalJg`HvVxO~&xN?O!_UXY_vsq1uSX%FB<g>$S>N3p$WLi;f|Wtj#m9wb2H8sUqr6dvfDr}hC{)^q0c!B9X6z2X2P7{bn1W;g{Gf@-0uW{qJg=-%CT>S#$7{1x$*p50WwJNt5ZXXZ-kf6wZEVIoX-5?kYOd`f2LWschT9oU0CTdv_^+X?C-C|}rv5(a-=fXzx_n*#E(rc#Wj8CIHZ!P9dqIgr-_l{UxYpmyP&FKstmMC#gNwai`wBz00Qi@WZ0v}I3(2UQVADBU}ZWO8!Jt^++DK5$?0%H*r)^i8OuB;+bTqt0U76xr8>)W?!0ctFAKA{rx1FZ<^Ry7P0lAMV!ACGNPfWDFv63X^ci)=*tQFzBL}d-_1E1D#W`DtdmnSX3ED!~GGZ{bmigAkPdd9@`Vc3r)Q%Q|A-o|BxwQSFv@9K3Kd-<tBE9(L|OJWNOM=KBEx<fwb`iq1fKT!`8^+zu*=_a2x(bH(u@Qo+q@HF(nC{G!4hR<P(U9tz^x25-jl|ufeQ0^WCwUeCW8DjUWlN1k)n9vW*KTA|*Njl$<C+ZVUAX5NuIg9T#N%V=;HvnT8a3lyeLFY01>G>y%Nh?&k+^EFj57Bn}_Q@@XVts@%S;&8Kx~F*{4>){%}Zbz7SFv6L%LI@kpf7K~gcV4=XU#^oJNJq;-f=F2-HSaE7mF&j|aw^<_ot;{C63l^(#5;hD<O}_T8t-qqzf160#I9|<J|LDy^G<A|)XVDF(pl9$tM9=ZJNoQ6-Mp&WFfM05pRns&CW=4~;+wVry?7sUF(5Qjmb_6!v!GXj*K)+ZwU2I%{0rs-EGuf84Le7gsTGzEU`x0a#iwoHTUmOos1wyDpo3)$|5IY%KHq@aG6v$2StZL`TyE%oMwRRZva)mRVu7m!y>vmHIE#7ini)So0Rd&9(*yg5I7BS}m+)w}1O!<9u7>$7$;OUro>0ck$;csh?BIRm=-ngypm|aC7p!~(3SnX|x>I>{Zg3n>10d>&0zsF;3Im+|2ix@{<T^PBOi2QVVC2<KVbcx)|y!KdIc<XAWNqO`&%)~Dhe2n&t2yV!OzOq!rr-?NzLh4~9`<Yb}=};_<lDw+`X}XI<`Bst8++;q4_s8*0pCs_fDd6PcydLEkZIPOb)1WHqiM#_MPB@2<GF2+J%3_5P@VG6&9+x7p;_@+!TPBuO%9UU|(UQ1Xsy!_rgvwE{Z@9@G!qF_fSPNLBga?!4($-0J8#YD2muH*Z^?MC&;Zq2x<(A=qh;}Kq7O%oajgVqbS}f(67r)L>9j)SHZ6flBmMEk`5iHK^sW!RxMXg%vL?^ccB1s7oft`2~Sgk|s1CEMy=oJgW(vu#v?*Q*2g`324=vY;q_)ivO0r+ZUlWre;=Lyh+{Mj)wSZq=e``7ij(6*m*Zq^k%15;!7+()63-4V;VH}{&jX?2dxg3o5@B66|oN_Mi4ZNdp}hovPV?`I7=w%aAJBDS)h#1TOq4C}Sqq#aIMlU_f{f<nFk1g7L$(Z$<l@!QsBTi01>D#}Kx(nO44Wu(oKb|CfJQyBl$F6MUiTC{kon0R6<*nP|Kz6s6UL&p7z%dT-<r6LLdN?*hEKDL=w4ajduH}30Md|WssEsf2N{Ss8z=rZMgp-qf(HC^oZh-1#-trJ*mDENQaOYvT{UoiR3*K{BPMN0bU(l~k7+M4b?Q8AnxOAHrd1YB+tbHfHy(MYFOW&jYd1jaMB{JFTKyPiTM%_~}(F7@Be)%C+b?Lu7y&QAg|%_mM^j;Agx+PG-d`;H27Nz@T8Si=D&*^UjT=H3+FhvRK_ls+FZ%K(@yg-hmE^+HS^ugt-}`^P{h@0FWFuFYxG@L3yo6Kc;?r~PD}x|D?IJE2;uQ;z$v%30Mk!=@rNtzyi|WObPEokYIRGs-1kWI(Ck7P-+2)x2_12f%YN*{l{AKzov}uOZy<QqtCGPI6~Qf|GuyX24Cnm}x`?i9P3b1VREXaBx)(hg+t6lCqhI2$Y}i0J)*14=ndhV-!xn5w=km&#=&3eP5J~8({EzkOJzXFK}BfuP<@NYn*UFku_E0`g0a89YALMhWFLXT&R}dUp~@o*8s@KN|QGE>RDBbxPME9q-c5x*Y<+=YF4k*QbB#3P@yh(PD*ZgX%lz4IlBhr3+pgidLz@u5A(zk0+H=#@LY!&B)L{ax#fC_z1q^$hj+8WP?A5K#uQ#CFB9N#Bj+@-ew4Si*;Lv&a-SiG);Qpmo2Boe6Tf;c8JNT4$4=VhsTfV5Y^H@jmCnITK9IO^r>ZgvpKcX#S=btJo{c)rX{;T$=xyW4IeOqOrNR~6ho9CeNWeriztR#{db5_u%F2L9Uz0I5L0Y#PrTQ~thQL3)saoogj`TrYB0`WtkB!=;95_yWal)2A1}ra2Q|pQ8hC=ZGM}C6IQV41F{~7QCQ72p5FdmCf@A<gG-?yt}dk9uevjq$(S}I{Ytg<`k@y=Y&({Swcmfz+Y6HoZnQHjnaPi;<{37};)+_`K~bfmvy)Rj>d1SlA`j&i&kgxLHL_7~#95F?Xzn`r5@ZoZG=)sb&bxQzADx1D|YdWB>S)6<!n)j$t3a{S>v?he#90>jm}LL%$uwYf+GZ&-zT)oVT=t(OmN-vMMKA}K)0wTKHqRa49`C9~)DHRm`nNN2gFW9k`2z*%M5koGB$Hvd^T8|?a?l79c)|B9$dVS!d{gmzWe3E)JuNunkn^ZsY^k|RSV$IrP+tC7)1HXBKWr}o`sSEogBC<9#0$=J3*%_L7rD@*dGN|0H2y^<Qf={(b*;zr*g;aAl@W!g_P>5b;u+OVf*0s?$P5-cqa?7D(hebtw>rOIRqySu60=Kz?H4{94V&w~W(W)@MGXKfHq;CO9jnP&*%YgKyJEb2DB8-S$%fMccYvKop+BLCTX)L?Fz&QB)ZaY98=i|9)ZQn*@lOAp_-#8@4h0{1c1dR*@sr=ARUe1JizkO!`?K&I-`jVll^PQt3-J*G=Vm-z$CbLDDyLm0`*7d2+10K*>u$M`0zFVrvOr};~TMLb}cpt}OXknR`Rn$fZL_mw^uglK{Za+WG?q>;Ag3-Xol%1m<7U}5ve2xh~Dp35NxEo>@}OF(^*Y`()gt8hvI4R#ADaVY^Y6B<X1Q7L5hoz9QrA|UdGVcl0NdZ6s*Xctr15YMO_6Lf_f63*-eg5waVS`<Xew64$K^l)^Cn3BIVa%v4p^EEBK_gKq7qVa!5jsR^ooYYp^BPm8wloOfNofWvA%cnBdBe>nG7La#|@X@ZOxtsh-ONMR0P_=HfOgTC00e=<^>H<6~sWBd=drTuj_dQ<AMcdCfpt*!{0!fY6pe#Q#5Dn)Bpw}vY9m*R}8xxA)GZXmqZ}G0GT-~$-Rx?gYV03qDDwQE_>$bPy<5aA(6%#)o?6L(Rs)_i8sBrk^6Iplq>Enc%7&6VADwyF3_5=4z{omP!IAiyX;Im?pCC)obs>^>^o<~ion57GjiVD<e<MtHV^H8|#-*$-VV^I(t0@d84vrCe*AZJO}e<Xqhkht~;<a+QH7E*n47!@%TZZ^e-gcB;3r5pL-_E`8GkY?kV9L$4G_di<t<%-<?%ii4IaZZOCBC?nN=ko76g5XGGf)2yM_UL-B2wGP$tp==voIpv>c%}JqTA8w6g%kud9KKbCI3^&oKDpM(a|Lu8e#)NdWcTMi;?oWeH8yXep&30u0g4TOgr}%NR_*{QZaFk4s)K)gkP1gpDMxNp<n;^UP>k^tYz!rYPUh0VWf1nozUjLR<U7rE8}%cBNQcYrw|w;fdU9-kKQ#)Xh;{<KcyCmvKl^ntHIzy<u5Qy`^|{n>cF_YL<2|v87gd{AQ3f*Y(+vM%c|J?N*jOaT6#YLYOzU6jnPGhE*-U8^jRva{t;=!PzHzGgDsrV@$s9B!16pylNAkiJ>n!hGdKoGu`pH%*n7oxdfXs72g+W@*)3Hc1f=ty1&2VeMo#7m8j{_Z)koHM(BK=T<G5Y(dZaj?U_eb&j`WCJ3Q8tPG%ZBx<$b|awttrFI=zlwa#him<eI5Kbb(qwaA6_3(2{e33B}3-CZ}!D{FtG;d#WjqZj^d!w_|Qf-Rj<cid6cTCe8U%K+Kk>E?zW6VPmP6<eNvWphseFit<?ukI!ZBz9oa}1It6oClw8!74M{9MlhEzfZGFc6Ap6+?fol5h91R|N8`?w5sC8?H|Fj~78B=iI{bjo+SFq1OZY|veD=@ds(G`)e%u$S2tK~_aGVoI_sgphof1Cjur)K76G(SUU^4@Qe2f>L&m9bZ~l@V3>tm77R>pzJKp(=uiSoVb4)D?t*riEWTTIV0t@rP4gJXC+zC|4!#-Vo4=;4^s4>e0LO33_iE13Sxhk<JG;d5<arU_Ywml&bA0?lNo2*V!qwYse-s`lcuey{=3rRly&Xp2fkaV}joai$?ylHj_(ti5k6U8Jfsa^OSUl!-&uzJ}BTug6NJfKjgKLO2rCsymSarpMiG}vgLn$J%|qy=YPdE!hhOpZt~kxwcN$O-pH0lWW$#Wm4lVN^JDKN$sD6VEmg{`Z+vRwk|*wWgfKNJh%UYxAFIoa@4P~jS+}lr?t-y^-f<ha_SVy<c<}s;23x`@mNJx4W|VcZm-K%ZX$7&lhN$V^$3xgOkKJd*KY$Zc{RHukceiw+dv_$?KYay=GV3dbWW|GM_lG^`WL@Cg9L1_*GqemVnQ7CQP87MVC0^W5L~cH5GZR_;>KYk{J|(mpFq$0tg*2@Egc9tZWq9HSvBWQpNC{C*DJLY1mLs3GWmUyvr~QTh{$uUfc@|DD0hS1mJ9E~VN7HO!P$!%>-D+1JiA3%yc|;TH=@awboOd8b3#PKUJhDVtpxsPVkjY+evXgd|FKYj_;N((!Gid^H-=Fv77<zYbS?#$B%WbBnOe#w;>xH`lJoH{>H=*}_Fl#+|bP^ywPhIEvJDQJ!@qsogE8D)aD;-e3GM#{D!83o`d#y*r7phYXcZa<B16u3A?}42NPK4Atl%!bNic~5tzBs^IabId@*!03HN8X-Zbf(v2b6&xpYA-(v#2h5vj*ZXlfBAHxyshGmFp_~Nc40<)jtnc>L&?abN$t1B)_4XdgX=<U_~J;F?}@fOjkCHvEJ=bB8xF<W1JaYfOAyG${j%jx?LEGhmb_oj0dPrLQDm=(Ps2H90Nz^Mj4s^bdHG7hFJl7PPCuQxIj|(~b)(I>N4=gAkqpD8u%~oe%e28~sf=_=qa<Z!PfVR6^>{haGzw@Odd(Y?|4y7Y;DSxTF$<R*BCivaT63AK;}W<e+<u<G?i2`&`|Z$~D7yb|4<DCl3DXt2qAAG|w5((3s7vwbaZWYQvIt)NiW!?kR{2sd1dxsd8tU#i4MzMiSm~p0t(?Tw%Q<y)B`xTx_jmosLns+|QP&DRohr!wCK9Za(yRgk!r)RI1+e3O@az7MD)=+1?6g~?aWMQf^Wvj~mrxkxx9JMHfIEFCCH8pQinm<bp$o3$Qap|7b~!mg#zMpgor`n)s|Ee~z5cN7c99oS8^V;W*5puCI+hKR`{%0*a2-OA7J5rf(H0AUQD~IO`GX5?Hu|$-(Dn$Wx4i3I5Y4{fU?ih5VN?5hlHKd3De~P@F2y<WBQzltG>l{HJdB!-#F7p36s1xUVcTxx65aWG|NV8Zj#ppV&GVqvD&K;<#<spUbQhxNX10V{z3Xqk^zptcuXh4;H@{Neb|&19-T#JQ1i8*V+ZR-j8`|_qb04H?lrgc*l&z@=nSe<H%l{w<E=)}}4l9OUdLHTN8E!u5$kMK6{=sLHx9RR!fM#OJf8A|K4sBifGf&Szd}hhHHa)G(%CcS1!c_l1{0T4~_LJ=x&p65W=Ziihz=rZ4!}asNcL%MGLTZ|Oey^6ULsBI-R;uS@fK=@>Op$;HMG3PD5PhB++39ohfPA^jM;7<0ZXR&*)CW0Qt%$BtnoOAaY>EQnj;K6>{?HW0xicJ^v~)&6yeo8#?y^IK5;XD&MixzEs^5#zkq)%3udYR`aUxfyxB-EzQJ*ki_I0F(wnM4KfkUi8W1hV(q?xsqBlVa#3A(7G6dCVY0~IM+>l6Dqkay$XMT@W8r#&ywI#iC1qTMH&USNJ?6lvoo{FxT#b=>Zz)t5RWcqYj9J=q#`=f7v$Y~KwWT`F3dKml_YB3Qy^%fXOG=K$ny7=829Db3=b;9SS4QXSDe4tpZ!BeU`4pu#sVV^E$Y;BUR=nDxBL_DPbH27fx2!ka4OIG||7&8+<4cWFA|+)1FYg}MS7gqM+GM_A1a7SH>@kn^rUP9IQYPY&<5;LTPQp;J@n=d7-!lbKTGNF@vprC3Ep2MQh;2ey!t$aWz^iX13R+3}{-xnsQPq>ea_WlE6_Mt~gc+<~e^dRkpa^FZcS{(;t&AE(5%`#w9hK;0ipH_>~@MPCV+yp}-`TWDRh#3DcEJR=D>-&d<l+G602Jp;(p^XpbgcLfZNN6O>3j)pxEdEsAG&oX{R;g-Jm)FxUhqdKT;!e5A05M(y)l+EV4HfU>X;WcU*Q!3OnxI;=S-ik2`PcakygfKkc$_#&@Gk9LofLjW8u!XbiN45Gp1W5WmBMUwzG}ow6EY1Fqi-gemRJ_|$J{+U9L#OULT>v17WlTCAQuT1Cc2kPOl*ckE$HxUzZvi~LCIxE#40-6D#+Oh=g>QCz8F`GD4X^GG5CM}MzxJEfa;VbVx)t~z(FSt^0t34Cx#)Qnu7(1-iRj#q_)zY+-XIF~a#OqOgZ{|lXxc7e;ZeQQEK4Rah4i`Ea=LNQc%|}a&lY@!1I%-^E!VOX#3uBwAQbT-SSPU_jIygCq8#Ol30pEqZx>6gV#h~KJ`%+{&Y3Qe*|+e?feJxNOVI~$vc?7d(u7(5)YjssNig^7p2pzzSS2WkIj6yXqEIL)vfsBPnSuuGK|Ij&7eY@$o6~jbmY9(wlvcgT2H+-x{a)0#r>shA4HU_Gx98t5as*Y?HOm797zR*YS)`6U1>!q!Tk^g%IFqGRO&$x^H*PUnO;!vQ#U-UPePLY6wVFBi_Kp_)(<>Q&1&pL!<C7LwezdaKOWJ<!_-lHk66i8U*uZGdj^lS{-=$7h-K|A=KVULSkunv^mPk?Kb~&;9_LtgfG6#!D`du}={T|SBjc?<b{QOtrfh@5v!|AcUQa5hU6kDfqg4$)r?U!wGH^>0tDt~jQf$x!Z0@8h+@{*5)qh+uFQJ`;1C@DOUg>!jzpV&Uy0D8!~^IJfh?&uxabJU;W*g)Pwfo-60B2EvK@t^W2e-qJf#lT9}q%Vuc6~?rE$vq{H<b<1r2+sm_@8!6hDrW+%FYr!DK^lP4W0=iOq=uO@;V?;0uzEXqiMyNzx14ku{T;d>ain^`JHlEvs==9Vf-4p&z2tCJPGps1ToTJJo`7a1avC&yR*NIH*RU>ua%9<y8$u%U<N)ywlaUY)*0dQRAg}L0$y<N4NE#SO7LNSFsoqK&k5_2=_Lt@967?MJ>Y5=ZNH}#1T#8q~JC>&krmS~DliZ-LbH$P}okVPy3w97M+f-T0(;y5G7K;ft92bC|6V>cY>&K0fGpo2*Tqie4%sqU7V2-;>*jgPA(2T8Y7?bVVb0v%)UI#v*WDi_8OZxj6*|z+5GI_G4H@jTXp`wXo8@-Dq`{-!_QU;Hqd`bh@I%_-=o#Qg*Oda?8d#$;o1#jy21PEZ=ZxbS_)#&-|u*ei3h>BXed^=0O;F?8rc7XeTzyi~^Xpv3&GQ-NJ@aBB$;}9*b^5_d%$s0>=2-kSPB`1>V`eDUCyeb@hZT!yh88#)~3XKGK_a34dHyPkb_UH_s1p)4adPCTT=(LUoNW`VMN?G~I-a41W%TsRC0J(GGPt?8YY8k}Wv9<EOn+vXk9!eao*7&c(v4@MT0TALUsDImiId)_WK3q>efC#@AlV@AK+z#6glXawsu#x&(B~*{Z0Z&WaXDxjHN_`z9C*fGLz+bypyZ9@miJob!_S22;l(22}+Hbw)6%iu;%a$#;U}IWt;?^+A=3eVpnCP3VmN-dMoW|{07>d#&cImdMP@@a)uw}3{@A+HH22NL`Fs)-@JPuQ*%Q_w$wb1a$n%Hm%x3NM7|0;<TU@b&|nmslt_Fw&to|7N=9GbLy%Mq@FqB9g1cvF~f^4ztu0tvf*cNM*)4VN|^O+9g%?p4WX>s&tZ0O?}dPr05v1Y{vZ@km6g0Q$&^<7dvK<vVOh5kPKt!#>8B_jSnYc8mdw-tN>o?y}3;3A00!Ch?%x7OL7LP5f6LD=p=P)b?B$?y!>}++~d|(?RT40c)9RO}8gsh<GE^ORt-<{9zmNie^|(EKxRaYl|;>AskeGF)~$NvGKhPhtq2GRW0O~Srv&49SXCxFYJSDe$`JK=J~r`Sa>)K@o1GmVB$11DtfYSNs7!7-dlboT@o<GFZX9B>p3~`&cnP+nFo^E;BzFKXmvzQQIg!7ZL=Axf?@{_;ck_nByA%RqAK_<V^hv2vmmxLIQ=^mR?x7}P+2IX64J6M@`NmSLZKP^g)Ub{8!j9k#&eIg?&>kng{=Bu`zb{2k(x0^^E_)ctM^~WKE^xPSC0`4JXvs5he9gc>7&Q}q?HnAZ{fwcI}`0YcR3e-l`RzJD-CZMCARuU)g}s**!XeUyDd6>ZZf`SzHG^Vl51-~)LGPx_xc1LEhAOLO19sMa7clw{^9O$-{>2#zQS&2w`IC`;ddDME6mH4m-MsVAgZU&Dbs2Fip`$f`s+(qz=6fPb9*}6d#4{|9`hyrQZfGC1ofg94Ynm&w%{(yTRV@aVBG<dGMVfCVAT@`)1MPEdI%qw1vDv%uJfi=$1AA(Y_uoo1<11ZUh1C72p+J4Q1!MaX4*3M@XG#m;1xh@-8#e_FW<|ZB+m`;Hji3zXAY--*VR47BDxAzDJcd;eug`dXAKF`a0Y8)__?AF3&!}z`=<dn-v}@}C?*K?ge@r$n?)HoLED@NoYO|sWcwvR{LNo7PK?q4`mx5My5P)e1kQ8+kmI4`go{a){Xd;qOX)Rssk>END#i;)Xp!xrGrmQ?DZDONuECK^1_@e{BLKEyXsgDyAx3lgae}p%^0ti>?JqK>)2M)H(pmC<02r#dprHqP-@h%t9)LE=^)s<GMyOBE1_r4#i5VgUjS&6_i||AIfeU#{xyB?GIA@G=eBagTfe`bCi@o`D)jDUyDp)pA|IHG92GX7Hk}#70C)D>=u}K;lDyhGVwif5D>8e2{Dr2JFskHVr;K$1@pPkpNaBAc*m4qXes0;W?pw?R+g4n4Tyq%(}B1pT+UxqItVj*A{B(S7iZgJ%`bTf~TTP#%)gMPve{rcSJU*W-89ACO{sI&+H`I@m`)lpEXHVU%d#CfZeuPwysZ0aVRb&;o@Rs(?UzYWnRFK*ydeeFX2Zj0@)Z|*Ea>uOIs-TZ=3&JtJYs`WebWU=^P5Jqkb%Jdb)NvfWNvUAT@YnghzlWnnr2cWOfvzSuK5Qym%CtP#ZBH<`u*3U*(Ze4lr5rz3#Aoid%KURyK+`&bY?$43*J*M4P%{(Lw|2|$4f*q8z3XGu+7HYGc%QJ@6blU(C32&yfW6oJjYX@AwbLudfV~@mJ)ne(rRMb6zb$ZR-{;T(<<p!+tKO@1e=r3nLE`1gl3ZCqi#;B=?F*B-t?j7R7rIvxh^SEH8oQ?rSfkkg4+)CKD?&1C83=VGb6IhfNjs+bdBupZ{$7aZImwut*G&H<cZxe^>;EIk%$=P%0+oc&_NmTYd3ld=@EOwWeZig98#H7?08nA4CtRa+hYIfz@gH;#N_Y!-V7FC>}=$Xu7$6n<JR@ZiF`i@_=KUZPzct~DB{5{1^HLmF2(Yb!wkuViAi>jx@0St{sOxz~WVv7Q4#l$bERoOl#Pkrf^q+k5WGBti_vH)IhN+3<b1&lo+lY+P@h03@46+e?DQm3Wz<D&yY2gQ_dkN5-u6V482FrW7{MZ*Cbq1Z)GN<7mx8zK0#+|MU1@B^~^WfF4mXr`f!R>+@u|1Rf|B(Wy&>fxrQp7GpICdNjE<uQKuuV>p1=Vc@G8jv@6;Zv<U5D6G%jO_|8*-eI?36o`%{WZf9b)z{}6A=Y}=*`n%FLwn)bbCV=FAV<*(sGzgnBqOa-;}H_v&2J3y~yTadNG7B5MCaVh_0QlW3Otlvj(m*lJF8*JkMXd4~!N02u{m0LNV1|A@#ttZKQ$Lx?T!{xo_XS0q3l$<a@+vuUmUK3?qm*!9uE5j}tA2Fe8yOWjVCxcKJ6V0we0Rb#D*e=M#wd`A6!p2kVRzh)=uRPMJBToyfOHD<S6CRoEn7Ho43ZINS`shReyW1<umt$N&m(*{{VP5caGb;bBOcg>1JL!ZkWI48e6x7j36PvOrEXZ1o;Q)<xNsXAWLzPasCT8{GS+DPuFjb<QOmgHq3h*^-}$91ZJR=rnN8!N&od!j&G?pjEoj3Is>A0J{w3dRqek3S*E6apzU0ztm-PGv@7CNsoom3ZQJ_=vw!fi=2}5To=u8=DWI0W4`qYb_i<xM~Mk(>43(f9S+5VM(Um$Sjc9Q2JnWRn(|=R+AX{p{H&uMJf=UG8!_>!1G{ObGXt$DO#chMh>QI+uK)q57XrmZaw1AIf%EODiB$aE$|$%L)G;X*dk64>01WPryU#LSlEPB+i*I3$>c;&1wVhI5&UjR1w!*EY8PN$XG$@f4dVIoPZK;>pdnG2$!xGtng|o7nzuft~I~%dlG9}CTU*BAh!ZL}m2<GeS?X^<0cgzFGaFm#Z?-*v3B)S`mVgq@fd2`l3_@0r_?9E>vQY_E2U?%wg9qq5}O61iy|2+Z%s`Ox{2LrxC_Q$4}BPK($N*T{MT9sz1FZ^Z97ZQen-a@>!S6Bq#?z0afZNqj{tqidl?X~v^?gR5+=U1Q_2!w{|x7ej{&ORz#^T*FQ0Q1UTA|_G;61dug$d-IG*<DAPt}5ez6+~&o0vLrU6H!PA%b$dKRbbjH5_IyUUZk;D)?UWMb{4iF#*buCC0*`=`X=?jPWH4cy+PMMXT{;NiKh>@KYM9bgfx{UP!x8x*Nc8PpN0MUYqOP;U?)-klP68;z95BC35p<Avcb=EWpFOzPvU*(VLxx(ZDXn%FgyFWZll2)$huRJB2aW&K{kBNj`OHMEC)%Mcbqd4nt>x`R73wtR<n(jhx~h5d7Xvu-*+5)=}M7+o3AtWyK)@~=N=Ll>S*22b;nnVIvr)NU8teEghk_7U?HObqIC${Gp!~~pM{IOk-lM|?(x13tYW|DDyZH(&?seeoB=M!PDupbn5%43Ke`n#**@^0v2R?2_0DJbVFROZ>co`WjX06UG*fZuNpkJ|(fpi$@-nipDwVo1a??eBVe4lt6&CO!9&zNidZj>6PZuMx7QR<mp&^);S0-()hi!kFo^w0bhrs?T8pCG<5>u|O3rk|4I>`=QqJbWk2<Azj>CzGN9dDe8LfW@JC!&Q}DNY#E5t~38DUB$ecne6Q25A<hPNYzyA0Nt)BDyc!o|5E^#{>?{Il=~!9A1a2KAT}0<{fHX5xRoR*enPS76PkZ%xa2Uq$TJkE5mfWzN)1X-@##9+@zi->C1*}4jq`q(}@TD$c9k~2#tk;%XC=U^Gh_*7<~go8nj72vfC76C9A;}?ZkY>>d1U128YJU#x2{VmS<uIyn3hGIa$Z4ZSB-gK{AuVcJD+W_Z(;62xyz$<oER%Wl}-f6h7etA#d=$sHYCMxXVyXYng5q1O;*|C-y5rBKm>2!rZOkSrWro-|LyJTUU0h9!eqS+hba`OkqAE@Ha{!ed}3lTf1i7#_rgb5KZJ@FHQ|R?S4Ghi1dsFwuA=}d0{MS%=}lT!ZohLyFI{W1|PvrGn$aP!LLC}uaYDwkzNeLi(@x%vaRcrM84pI<qhLKNBoH-q(XDXIF{p)=d%#7v$L6a<s1-<RTRZ=OkSttK6y4+k74Wz^}tpm&Bn@$NX@#Sw-42X{EUUuJ7Ky6?|yBM^DFu)ssHhsI`;*s0u{$EnHEH1ZIgcushGZ&s7sS8ffLY6qYT|yPt@%|Q}Ixt@znN+D#6Uhj^%o2sbYd&j5Ir=$u%NBwfX!jN#i#y4m$ztHB7EC+$A-gv8FbBk%|~?`FjU%LRL6szBEthCFRsuBX((YnQbB>>-%hU{@Ipw^=^i`^=BO%@Vp+<7TP$-B=$xv01<2kE{;oI=XeGqx4s`Nc}{RQ^dv+&e&B`ivow*{j^ebYTlJ7AkuV5y>+}`O#8b*jF>tvd8{;+Wtbm+3a*RaP3%IZEjr&kqCcRBYah{?V%WhW57BB+PhxE<KQv@sO&Ic2}*4-%EcMQ%j2e!B)(uX!UtlCf3=v7c&mhog@69K`FQM6GhA)%zDE2SPe>Ppfg?Qdc7`UHAZrt5MI6Z+{5{T)LSCKMuJ^#G}`qg9fnNaDehdfLHuS&2?D&vZd$2da!bp<M7wbkag&N_1P;3rAUo_fNS?U<LBGQV;VO5+YbFrQf<LE_|Q&ltb^UwR;6e3^)uy6;AUKNqN-iPS8yLi9FQKr30ba39MQ%M9ymNl~dT!R#BXVA(oV7hSae-`a*ALL75hX!f&<DT9k}iyrhLQU_;(9UvaxxpZ$RK?;40maj<gRO(^%Jq7H-y*6_VJ9*T8tbx$r!5owdQn(W1hqWfoxs$AHZxK_`7g34tFj-*HBWeHW+b@|!?U0aFoVj^qTl^(S2u&=JJ#rN0%j@LQnj3M2p=tCl#;r^MOh*foR1e-0bm&ZqF0Jw2PdSOc@>;N-ZWBbH}6XP2<Pd;bD7jCaCN})WtuPa#aFq?#4pPN&4sGb1faHgFyy#fY<^Sw0HA~s0>B4L{>O_!ZgohlV}LsLfp7~*oMlAW~+Pp!n~dGD$B=#;BMH8b6N`s|7jDX(>2dv;a1aDhQ?q4dJS<dp-SM71?9X$1;h_Hh`eJqBTW3W7HY_2xWogo^_5MiK0pw!W3cq>)rK`jAeD^#PKZVIc+f#e??{v>~u4B`I&q$0`c--mkQ_X8a<l%!+_sGqG1*i-WlbezROub=I^Yin7X6tn@7)043&mfsebdPcnV^{4t(;22>kT!-D0{Su}5R9fp@FOCeV`D4|2`S76O}ZkV;rk6SAk%zZ5N>8N$DNCX&K;`^c1ZNP}($yQgTC&M->Y^Fv!r`n4bZkDybS!XE<Xt+x_Id7{B8Q;32*Ec27#l~n9qFt@7VtK0iTybE7y=uQ(2kAz8gl$^4%yIP>JQcCA3;?OHkW-htUUCR8;bH~@lS6;a!IkOi_uYBUYs?9GjWZ;Bqw7>_19`ndS?5nUYGoTSOId&UH0uo8*H{^HQI?Hd7j4urZI$u2m)I!kt-DI;u)IUb63&?6{Kc~M3;4iOzvM_k;tK%VmtlBJP-E{laPo!BQ&PK`zFYj9#(%<-oaKCqk&)X{{hW|$imIXIP;k6^OJFkFsNqjV7#F=DmM8C80caDFyKOAt(e!Cxbo3M95&(;#9WU9b^7yy@ZUak1?8#v|T57>Z<X_Ne_VmKt;3Lge=!#+3ACWgmf0K|lJvx{Z?`#iGQEB^tbz8>>YwFH|r>Mw1WhEG5;2to{E_VB`$q=dtBpV{SBye}=3hm}3zQ?_p9N1Yo`o>XZ@CLlKfQ<MvI=J^#Cy8JL`y#LZfz8zzjJlFf9eE8pV0jU%Hi#yAvV+rz1|an;3fTifhg?x_n`>}>)+uATcuP(wue~LLtaxzmI9UD`2CnYTFn(%cg6)_-I%w#wW+$m_B~FQ$UuM_msa%SV`F4a>{QDJ0*T7uW4!dGz4hUdfowSz0;2g5dD_7P`C&nzo8i`2r>wJxDRT*}T?RwlyuW^#kCHW?Jl4#3zJ`KB?a3#&ITIz4~`OUu*X}<RKtDn$!O~5aH4KgHKG|EJ#9Id!O7o<Q}cv6!7B&_H+YE@!@1JO!;Bg%79H)T<J;yaK|;~Pv&vJxGM9RXM4PzV(A05PAex3yo)v=*?}tW|7vuB*a4pAg05b=_Q;h&`)d0TdaVPQ^HA-aDSakV-aT(|2G$vU4u@>oe<*q!l_rrq0&sK>pErKbdiai1pO768sns2??yv{BkyO6<9$0=HbOJbVlH4v+xWHw~zQq_gEP}A6X!#%aTlqo1a3pcK9RGm44*ArX@wUn6-ISoxXs8iz73;L-g5tq+g@DgkK$lCx;x#!B@}PS<6_o{MdS64dxar*(jH1DM<DQ!D@&vyj6CHHvltv+_tolo|QY#P!A0{iL|F;_wh8z*)}QhjTuyca(Q`Pcog=gig`*+Hx&~1b6K;RTs?FmQ_*><lYI}U`~io4eRA}VMfTyzn23Ju2ukbOeyXq0Pu?<8_ZpcxBD2g)Cj8g1vVx!Ez7+m<#OEJcjxDPjfasDla<yeyg}ZxWV`bVPdIiZBCWG2rWpb+>fjYCIiAG@#F^@sYH!1A+JzJ1$Uq{;CML5E~U)~Il%w9N8C}waorJj{>mJjtbZ-Vl=0WJsUa)S5P6)$YJJ=W^S{3yI5s7;v()*XUIOeJhM{?MO|ZMc1le0Befj%p=wRaQPZ6;rv*#y(#AY3Iy8Ro&_~Vg7Z|mB=ZCU)p)uj934RC(^8_7ChP4hsnGO3*oySHs@j)uq{Pq-^52SrKXwfPr$dbWhuuie6Z_gI1@@YY?)2Et7(UGaj?;$0gOIwT%|(gESi$n*_X6i=ANqo;3K-&@96x6`B9juLSN{CFGZyIW6h#j7nBYpFtkneCdC#mT+9(5tg?r<qB3VD{RSNF<D@Z?J1f7!Nw{#A7xg3>f=c>=ki+lyoz&5yk8FMj?i{7;)wpSdRdT-3utl;_X)JZ^|7B<mIRw-9(_Y3Vw<jIWX4sK6QWBXEYX+!zPfD9R#}sKLbRoew8>77ICbD6hH%{uD`;LOZGHaQC@$BCR*h*DrF3j~4D1nN4P8H62b%ZfL;o%v;w%FL?Ib1SR6*W6TK0TkRm1vC*_xYFQ%7egI*;BX4=}icOv~>wtKTgv|#+4Bc6)H-WG2K!$v$q&bVhKt3lU0fO1ZN|`nICSISUdcFfg?79kCv5bLNO-55&6wa6QN_mq!gDvjjD^W?w2s1%6hs#SRAKZOs7YoDHkuu`8DX|1g=<BdKp#>k3z<<BKS7oB}Z44-wHek?~a}vb0o8_s%_$++NVs-JtEkSHEWE9+z1UjZ{`0r9MuBR)T?LXGcANEVQyq%<8aH15`T)e!GhjXghXI%{%`K0l36Yn@qh&QRKRkNx?fw;R7GM2ts@6umQ!i6<5KDUE}PK$tI$)BY;{a8X>`5>^HVtHxe<S(?-5s`3u>Jcwo9;yViy9wW9wTd&d!_3@O$ueB}=-c5Xz=q0OXHlO~_TMn?nJ&e<j8ST?QOLCnZr80*XtoEmZLmOBeI);m;z7crlF$pL=m)pvWTVaV16TXb0@JPJeKAx&Oh~@z6BSAx~6ysZKE@^W-iYPX2N4-Io?(Q9v16$%$m#+}^Hh79FPP%KDP|`w1Zxr@qf1R@6J)^T`1{N!2dG-{0S8>QGI947B*iBA#ZM7YU^KN}fL*9D0gi2;$(P&@CNHKZB%daNb<BpL87?Q3y8}^#ijkjC!?m?B$>bN}BRNJ0@8FgJKtY4meLxlwXL)`3O$FEc-d3ExCmr`UR6lvJ2bVjWajX2cppK<ivwt47>V7z64x(A{Rdt@3H4W#?F2>#)I(6C>2y0+>jx`((oPr<z|$e4~i&0IcW&PH!<K8645z<G7=&<UeBk=y>40s=b+-SK>@ce*l_n!woUw(FgldXK*1b3tFpt9_TEMouy{Eh+7xZ^_+tMV>+(Coa5{cm-dg&ZFVu#xPiSktS?&%OudBQKkSiMLgOfZk_=2JUXaYPPVB1x;9<Vqs;oA+5;xj$#!nCSjC11kB&R5U~5_s08<ZQ?GT5bkgrT6aE7TtCNaa5Ri&^2X@HhBOm6p4$R`B;1*E#Z6J3UD$1wF^xS8y&9FO9owkLD0_1)D+Eu%^03f{$4|zj%q0=|L;ViM6dWaW}t`DC22&!m}8?u?8HhtQmXr!2_o&+u$paR4f3c82mgKd+R5?(aTr@r4pOEux9)lfCwzmhIJyKzY~*g@CVFu_0+y9lk8GNpbBdS)RTglHUQ(l8kwgD9?tQ7RClGr)5QTTbIGV9Q;>7>@EbS;hSTzZdzL+~UhF|;0=wt#8$Q{Pf22@Y3K!xfTiPp|jam^LPI^q_*p!&g#Poll%X1T3LFw~a7{OJ!aansc6KHdxt!DDct1Il{sL4yq4Dr?GpahPcS?hO%gdQOB_)tM%x4*Tn&yuVpq@!MSrj9<S$dvYOyPh~=%&UR2&ceVf)h0;_UE#jZCzBzMdhZdXR*9fTswVCj@d`wP_o5+peCKlWNfjchczp0L^Sn?B6OZ-|M<Ja4Qe<a=9j?k*WkDs*Ud9~-IDD+Lr(J0F#W_6ZmUba5UF`Q1gD4yx;_sswCvU#O?LZlBa;qDy)TKwmVy#K*UV)kRpsc1ZDqE=Ao*Aep~L!f`cKPgRAHg6teaR-car7+>+*ea{_TtrOounbp8k+m8%!f;XB>j@CCGhJnxz)wT$CuQCX>;2K(mv&01-Fxp{W>23{pkVj<PNPsnMY)oo-3Cx)Hqxy5`haTUdUiRK19k461MTd=I&0TqpG(pQGUYbow9BW$t<?*zO*fo^MCt-yj;Rnyt)gK$-c<nrP*j`^_myTmF(@t~`-}z63K3rl0sfoXJ;q7e`6#g){h%DU7_NOh$(D~DL{qY7M+PMGu0FFm?U?m^Hp-^}-nKN{hN8xGbi!XCI%i@}n);i>f*9I7*az1zur9P^tec*i;>-D?B)f~Lq&}m^B=M9|VLfluBjPw10z2h1*r3W7?%$@!9_G{<-5}<}sE>nvG5EXq6{HKsJQG`2#$x39<aY%)05fby0`SU{FZ0|Bf!!u+3)ms`IX?*+9;&mc`w174Smp-}HP2mgVZ;Q3=Pp|80RYKz)bhb0kGo2>E<PeXZ%~FgafX!ypT1Kr96}IE^?09fHGvakTc;dU*e#-{;e*fmzqJ=w$)R7+Ejii6){+;%HE9X4b)%lJMrrE+Kce;9Nqk5%*S^-}scS1t#CAs7Sq^Eo>GX-nNm?%;fJELuEH>7ntg(ti*)NM@%g{At?Th9Wv*N<q6T!x7slf@RN>9(?V^&IQmAV>QG#?aLS~*<K-0M)f>Eqt6gy|PP$NX4b5m(cjCE*c-P+lT~j2n?8|FvCw7PZ+1q`cm&C1Oml3m`pC7cC`o;L}wesUSNy;%`Uvt%pVut$QU0=2kBR3Y%!r(?j$#+IaVEk=7OKM4kPnn(}#QWK<s-kaf=n?Hbz^IL2wpfIeGpAw>x#f_wYrqy(xd5dG&+Q%)whQivcbTTi<SXxHzD+Dsc~?fkZW$S90QfKT*^DO7dJid)*-J@)67>kzwyT1s1ahLfp(?#4yRcY^swJY$F};O5-$wqG=e{X2k!)Pz=*ni&ci0351iQ=R~3y<E3n!-{V!|5^8Q^NM5Q&(LZ!AvzHl$8hi0vHoiy432=pzBZjpMQV#D3L4e6&U?_95Q9-dPIbi{Eq;v*T_8M}=9y6U8DgkneE$`G#5o^kbD4ndaMVK~HVE_zbl>>2cEQHW$}I1$^V06AYi4jM$T@wDUx_j2+|6b)I`Bn#2dT{7o5*+;G;Q~kwdd3C)rv}jF5TPDB)BFVVqfTei^<<J8V24TijwTiL78WdT495@wf6JEABga>g#Cg$+m@o9L_vTnW|rmm=53uzmdPQ*A$RvfBOqk6hf`CnR5jKIP<TC>y9r4<9Suk(9Uq6hWq%#m+J6e$9b>@65w_<@+?*If?R6^&b{l<vPk+!0<Gpr}QW>jFC`tV*?{NiY)_>+d#MzI_g_cXO;ZmYKAPax_)bzmh-Cd}?9@M!n!`h%Y*?gMwk$DF+72EZ-MP|vB9sB1qm&oPDv=k^n>Qf=gOGl}z!1`sA$wrw*KJc8C#_;LnZ4y>g2ecWcX#qe_{Gdw|Taf;06mZL9y1-&lTUkb{1nyX2+-w2HN%nQhro3&>$Lv+mV)g58AXp&(Ty;(2R5df^Id0R~%q+7#@`fWwX(I%Hgr{wHHaL)g)Dg~BlH3p-R9{c;47P!FJk6PQ2z;f}ghspg)@<a-JB=I*o43wM`*lSgid^1!Ta}91$3YG%MuRF6^G*-ylV^7(c_v$7UKcuRf{ct8^#<^ygxA~8TMx@2pDBBR_iD#7{hGnT)88Av*D(C^PjDNuu${qDUcu%|mSOp~Y&{OQCv2uJk0clgjm)B;J7xh_|0iywPU<s^sEcjck7C!(fQLx|MmgVW%er!R9{0*-nt|JGmFQfp@N0HtygbQPE^ZZ7%aJPvktfpRKC8N~H~I9{L`eN88)a@pxI=@3>cTBQw^lz<Tm>TJxa|V|kN5_El8h2?hoG2|rhDsr*mnteUXx-mNECj7(_vM=$Fal?<pd{pZO0%)U2Hfb6d)@dhV5_biXTXy1)~l*{Ou{yvfIm_QM(4HS9PZgGoI(iaX{<fY(1RIZ~=>+JRt9QgoKW!0Q^S}X1jnfd+iTwH~-1WI$%9Ktn1`h`b|Ri8Zj|}RnIoU=M6uo;M#7Q;D~A94zwUEJ8aoJNL6G${Q%Vy)BYEK?|ZV_D!^RIdv-*OVD~r>BegM_|KL{E@g>cPI%Q%)#fW^Qt6N&1qIwVq@<x9NRdD{^u)$U2G@y%kQ34Z)&JF{I#!r9RbAGsDTq=#&eCtJsKi%;P0TB}Ye0QsK*y5~t(Z4gP*a*EFMZyG8REwk)$D_{Q(yM)Nv7*N>z4BaddSDvfmS|Un&K_EC)+fC(HK0euK)o|cdk9I{>$!Y%YVp#VT=Qn>dzG3=Vw{g=Md&*!97(j)2bgkM4hTh{K;fg+PYb&{4dFEAu#_k}zjTLBDu?Mt@jWK0u=)Pd&$fQxIoXqaaxw7E;8V$o2}s9Z?mUgxoox|nhCbm=1r_M-6<zfzRnlWtSM>Ow0qHXptT?n4M+vk7^A<TCTQZ=@SXd@3pjp)S6$qtuq$ovcT#2{W%WJwy^tn@K0kA8Ll=ZO5#@tMX(;iauOKUjC2kk3clIc0;>bwV3OO+-BNRRnlMES&6+wLLkRJ^=?XCcE_1%bhdXs}`<_euTh&PUM&%!}IJd2lGBC2D~`kh*w&8!k%4=}T;8?V|M2i~Nt$#TbWb+>K|h3lC5Ov2HFqe|UI1{ELe##}si+OR+YICtIF&?b4U-Y;<6SnQ+B7_0_VW)0}O5GA4`7I>A7ShlREv-b-c${qU4$T=ZmL8H80&c#p<2K<hU!VV~g!T>n?(tF10Wv4cXl!WZK6lI<iQqiRrl;5`z38Cx^Ck%Uh##I|?NTgWmt3hsw)pk?Wua>gE!JhhU+Ruz}*k&;1fky7t`lZ`R%Iyz^aqthKQuM4VXT_B?zGtW_I4R%`7U21NfA^;HzsV@e`FEh0zb+f1GagmGIprF%`cq;dGPpv;mbci4cbj3Bn9Jj>q3`YaU#QaTDB2-sIV+Mi#0yqMr;i;#ULb@(yyO-%6TvINRgN<9N>AS|E^Rd&Lqf5$NlKoRThP(e@mPvE7OmAJ)<xRmDsOy%#vWLDcb|PHXgcw9F9u#gjcJ70S&YipT57E7p08RK5dI9jmWl2Pl8=|M#FTDT=rf@z1fI3gS)ZmYT$lRb~5+yTP-g9gXKAeAv2ux$(H`q%L*vM9TYtE!-r4vVJ0&$@-WJxF#6`zbM^$d!*r(;8U)P?W(il>E_@>ELnB{MRZy76+gh`_HmysfbUO3kD$JBhB4h}oR6Eoor4{-W*BP*kHrM&P4yvCU032}fjVWy#h(NH?AXz#oI|tMZWEvyj%<h4kQd6CFCBk7`g;lF|Qyn`n<ivg`!qPXC9>3ipLJI>`&`pc_-px=^=x0lplG{e2l^m#ELVTxCg8Ee97?DKsr<vNe%ARY6JMoxITy?c3=k-g3A%oD}h*oc3Zjy)pR(5u$i;%{iWP&I{Ce>oXgy+*|vIVfco_SYQxytUF3^Ve8zg^p|^t{z!z2kBA?+xR|y$HYDrIm~YTeQNno3$1{yrAbF$OBH2MjpPI`K-4Jn4YyDiFm#Y|P=z%jt874ZE9&&A=uC3a+-LzM+#<_|#HKC~#L%%;x6;{tk1;#WuLkMYUJ_tE|*$7OZznyMhIMtWM5UG6QSvaAt;5_6N;yiu2n%6L{)k?_2hjCov#R~7(?7*@Dn>q2UHHkiIQr5I^4~KxmSQ4{}9nInO76zUcTAqh2)761;-u`^t@$P|dDpR85`|LzQftK|o%{hoP3CeCdUDvUQ^O=UDV_&y_xJysh1B&Dh-_-kQK3R?yc{oQ^ULsjgOh?q#JRou9=-So@dd<&3T?a4|aMfGd8QciKAR@PbCmi_QO2UaD0l4u-tEaC8Pf;C?AEb?f^dQFq6SerKn^Pz$Fxs*_!gyU+WO!G)wEMV&#duhk8IT94={qYL#ZI2Lmqrah)YY{$gP^gFQg-^zWXua?d-kJJ@xM@w{x@qu*)P;V;o*0p{dBG3I~xzrB$*qo;Ifm<Fqji<n@GY($R5i+CmQn}EUPn>!E1YJ*wq>WQNh|&m@IWcFk+*|Y_UY>D+GP+5N8)lH(&BBX?U*g>!jXr0HO&<UQ^*q#eiBLb}PYI2Ndq(yPu>5Gb;Xb3{2(8!U@f!BQBn0W_{zBm}m3pqZ_HC0#F@q^tZEQ$Z{US=f(%DYHVRO?o_O4xS1kqSDH)5Uq~R00CnS>*+;UBFg*bu>~RB3{vr?+)H`13f76z7xV)z=6B<iF2EpPVENCAE?*Re_a<lBTw4718t+uYre8Ld0nl=5B{swQ58hJJg6Nr%P?A`?QfQ3HHx&&a6A$U&yj3&^3z*((F7WHsi8>@*g0rH(}Ym?qm>c`T-6pUib&=%odGUyFv{(y>o7$+)7NvW@IH4`kv3|=f$7NPcq#gTgcppO&TOBfqd#c}V=!4rj~yye^XNc`6T95oVL<)hi6zTb?l)cjm+sHL*q;Wv0A<`59dXVW%^H>EbH{c0~;f7q7+m60dUJ_4?#;<;{pX*a9~*C6x3RYJRBn$Z3uKLZN{O5ycOW`H##&YU0p3bA(TKcU*L-p(6I9>UL)UYXWW!d}omkk0JmQcXj#z*SO-Wvic?pEoULS-C@<Ob$Hw$EB0ZBrW(zJZM~VH(=p;ez_e!Xj#?@Op3(?OXy=uk$8R4D407qlt-XWAA~d8&f&5Z)^fr+Fj+|}Rly!L*-rH_V1yicg^c@k=2>Iwq2VAwEUt|y_q8{7$;*izU}gtcNUFz@0IZx<V)jx_EyJXi{HmNKHFh*+pFOCqE><F3@7zp3d-&DbQ;Hz>N6LT^?X!wBKResggfi4NvJzifhem4cAR0{acem^iMLWjqyn{wtB8QhdCjJ?DIC@v6L%s?h%Zx>wB0H}M=UC!Q0?D=voD@n;9CKW>eqe`twkDK1c5_0^bj?y%-&4eg7g+@xwlK2X=@R}FhpV^FO8ZI4*BHOc?J!d(v29PCd?a7WDId^)=$#{7(lu%Qq;QF_V@kcQm%p4(IVR4?nu~7*r<NE?R;3RUti7N9rP{k>qia^?zufD<Nmz5=F~>(%TXCan<HmU^WJD5Y`Gfz*9QWZSj^nakL_{b|a7k)M2!?9g#vuk3@74v6H$U8-6)+<E=?}}n-E@w!XE4<xZS_8LGAl7jXOR7X7mg8L;Y>zmfW%gSlzHoFoxr)FGCWkIeFIq;eO3O?rg3lICY&6=q!0!;LJ*iOhp=(ZIxU?*wX!Q5i-iB1F#yaN>5hfo*~lUcuO@k%{F4fi1~p7XenVMrc6aZcIHH85vMfwk+O(;MIV}M~u|j29@7Lhyr+24ZpA5w2tS#WH1$z%XL?by?gtCJ)@p1a*#htnzVVARZ-MY$TETC3?ldbSZO5w&Qi6yW-k>=rw*n#Zqvp`==oUb|+=TA+x{occ+GPW2y$R?&+=fvmPPAN2rvWLd`5p}@ZOO%rD)Y;*-+e-@)hVe48o%5TnIGOZh1Yvzc5`uwnf#ZKG)<RDa&H7`l!Oc3@HRiV?qB-r}l%UEo8gvJ=w6FP<h|DJ`TOT>`m5|w-Qz}N3r)g?Aa)2xh?}?QpPE5+}$P_+dcu<Y!+6OY<;cA)#dQmDwV8-MX1x6$1OtB6W(eb)Xe65p-gJm>;7hj&d2FC^sVun&Q&sUca!fE5^TfU;F06qd+_G9yKR9Ez+DdtL~@_??+8(f-?($pbwljbghK2*$d>et3_(rrFjPF-SJ>Xu$`yYxLcgl$<`7N42(=0Jhy^zcbYq0bOd?G+cRB7_l3i7RjWrnC)oT$hY&;at<F&w!aiQB%K*f4&Kx@!hqJX{o{|2eiHK2`vV;&P!O_9=kDfl*0W}a1(a6AM$}(rvCrSkVf)xBhP$&JK`FUs^=tEem80L2e6Du(LLn)N)2#7yego(ME0x`WREcl!XL`>4SA`Ne!@Hn@EhMs!2lf66?ey7*G4n-P2$_S)>YO`*cls+t`o_vKrQh2W(}2OnsgGUD<?N`@LtZT-#!K|%qihRsXD=15c_3`BRCr3LKg1hja!Ws8^(&Hbxq-WHRI9zWs)B`fch;|+XKJ%WmGiX{Ts)jqx>e-O7c)o;dFrU^2pR>#A`Eyxy9I5x*c!rrj%~Dk&yPK-2I^r<od$WYL}0Y1=b}Y%sJeWE9Q-O83yI`VWEm8Dx9V1EliBW_Cg#X)jVi7Z}EEZ?*BDWnVxBP8nP#3K>BcSSIKT=$^*>uyf%|a(dl$cdfG~M5<R$_hF6%kqZbkLTz<|e2Vu*9$<=q#T7mihXL+qelKtDqN21QiY?a@tpx_ctPdvM;hM2kOTO=p-B&#76&@R{bqXE%A|AbL#syPCyLY{ZX$+=ByW5I(VTz?#`G5OC!>S6CwgSPvpdBBC10LyzGGGi7h%)8+r$lMfw5zuGskj~E9Aa;B&rD54+Brg>z9K#ZIlUjz7zJBYF_}1j`LL5aHWG{iVWwtoT^)6*VLHti>a&W(766E7((ayp<&>oJ%al4V`2hfoJX)P+<j6Rw!yZwCgP-#%P<_>Rvh$+Fv(DD#+BgS*?V4r7{LW3P<PBIXM9<e8lxg&=dkAhnQdyQ6?oWlL7I0E_!Dtc>V#WJ^Bxj)ahtmazX3k!j?6Fq-6u3LoYJ<cf#J+aZCBx>z)ik^4vCw@kO67oao-^e(X1iE|H<N}@`h2-GhnL&c%gD|#T8{Q(DhDzJu0Cd^Knc-cNt_1WHR<TfoE-kVs(F3Y#jLe>(T{g7Bp7H$_|NkejU|MdIS9~XzKWh{$^CEqDcYvZN*t90AI2hL~&RmN7=~d-WpFZ}B;bEI~8n{*9lQ*`gBi#sSTLTuISvN>Ciuwc_2M12=p)39y?S1ix1Bn+7@~~LYU5bJTJlMu?0eQZ;(sStN$QdV&&97$`W0M6$h(jJ$3czKptw5);n7V3YAKen>vxfDcrv#<*NQ$;W0gesl=O(-@&TTQJ`Od2yJ&aUs`^f9UDUg}%1krp!SyHh;IjS~3Q)+o}H`#KWKl&r}6E!aV=JX`kS`><7=ah7`TWAATaeqr=>h7UHKe1C8u$dH4AOorr*#4WiB04Y7Va_Armkr25hK>v~JcmEL7(%2p^39|w#puc9+8iKfbU`dr&~Kv6E~|)=yEb}*-l5IXaJiqjjkMeJ1^n2qQ)@zG=r)$@x~=`@-Mqs@=`JfQ<&iXpr9lO?k0re_jrpj4GYrKrl&XENqmQTsoE`;!^jlTYY@%B~))hDw;(`#h)0C=j`JUpixC>Zka=l60Kq~^+rt<gT^waHx`@`r-97pio%z3K2+(8)j;L+@bh7L1bx;oO0UuE@sBp<5{k1%M8GWWwb5l|~@-ju)mejF6zq;Xrj?>#5Cs@C-=7LnezM4uM%Eut=CJ$_q?28jFXD{j{fAf8#g(jx@*zf8ay#?aHy?|tg0159|sK<0@L*H;i;|GssTerrq6ZCbsOYA*h5i9lB8$TCuEj4z@qaKjKCX`TP?j#uThgKn<jFx~Md2?evAWKC?={O<RwK~$Lb9s+n9=-+&Pt*sUN{N9(hmdhdg$!iDyUld*(`<Rj{V(+ato6nhaQHD_i7_K}F*MCQazF42E;9Ej@n=f$w@nDz&i}iN5Lj<0{b{B>mGOugn11d!jmCeTYfKJq1fJU^J&D4#p4w_VVNIrIQ7=ZStfK`Ge|EKFy7`A0swmI6EdA+-acVyu-pa5m`{f=FYp8=Om9?m3&Lox8Um@GJwd;-3`x;6>e)`dJ3UKcpSqG05;CNTDSdWE+KhSvW)RKY<>2*%`wsusort<AZyhmvKsXyv3n>GqyeJe@tN5Ir(0U_}`xmK3tYY1_%AR!5(RKPIncdonYE0$XqB)zvrvTfr))ptxINIXI}d>wV&v#o`UkX`92*KPElYNMxWE#jw=A38!8XJet3y-rJ)A9tLTt>-FX*h>RxO9i;2>#|pWkagVFvC8E16jXZPe8mIVlVd|e^g{0O($jrcDC1dnS<6|3+*&rDH)nyQko3r;c-7)+D3`HGTHk04K_-CXQ+oW+FvdeLoUVLlm6w{MnuPGKvAGqCOsd8XX=i0!3{?rO^&_$}RjQ)OGP6`Ix9ZQ>3`%Hth7<<yNiek-+Qr0k97%JJmOH4ZQnLdxlgF9-LzOs-%c`Qst@mCt;Q($u_bJ{r+TcQBOoUBIAta6PPPWusEVVFx=3hfwDz+uYAd!u@xcM>V>FEQegx~`VoQl)4>NaFC7PnZb9(1m!L6C^~B7b8^O{ftQ~7@Bz$JpCY_B<I}gU!NT9c#P!RFfM2JuMSW&6G(sYx1aX>G;x#xj0D>~<?DL7Q^OU7b=l2t0m<mxL3N(T!8bc-rg65+i<{A6QAgDqTnixfFyNX+q=r(0zsV6ke}zu5)ec)t7TWlQMk!}*aF>wy?f1i!#?VeBE*n#m3%m1K(Uh6)F9xE$x~>QpQcgOo%-k^&iVn`N%%PSBz=vqnyr@AEosp6S5#W98&o*1qQHXJ*JDUAfVMhjX^ZGVgOXj}d*PNFwUaaamff0{`F>4wfdZ~Abm1AMsn6hNH(%QeH6*VAK%)U`rP+M?}xn4ymzc-IykZM{(q~`|9;2<*IQ@wW*+3D7eX!cp^3`OR6*GS<fufE6{46Z>9eSHPBa87T#=N(z7v>a%9huDZP>n0C;+RjbT)<AgbU5P<<zqm{d(Q8uZcG~wA&Si<Ybtc0)EL!4=Fd9!?Tue!#^`r_}fR<wT7pW*J0Drb;L&&Zrr^WJhHmgNfE!=R5BFf5Kx10zdwz3L$Y;pYai5FOS8k?A|Zy}PoXGl<VCa~?%eFc)RM@2_FEs~;Mjk1KfyX(KPLj_LOH#)j(Pw7Fc#ftfj%1fRzES6IslzV*w{6_cD!nDZzbB=V~7)-fmL{VmnZMDNxLJB*GS+hRoQ*j5DlMoINK}0D!Wi|H?<R@e8YLqa<xBOl2DXd>qqG67+4qFaHojzK~@g!?K#l+k2bO%witg~<Tb4-_5z4w+`%Q6rJl^Sj-(;d6H6)H#naT@XyP2JkpsMO{F0n9_u{(_QsCFk)AtDBOoO&**{$|@iF7XZbCNR%19d3Jx@`!7~IPm|r0<p^}iD>`ll#{8R6IDW9RAdGjWW6EeOyog-wSURSjHP?(kNVIC=1^|>I%cTD};&I+`s%i1^$8zBU7)OTlm^pOM1F)^CY7USb$s6u)uYOs6uM0^`Z4UY#<ocFBXuFBLug_PNIU*!A-rTGZA&kaCkBjVVO;+H)YAPtPD)*o1Zrw4PB$Pr|C+T#2#t584VSbR>!+gV83(~5y=oT=szRnZgRm>*38`Un2RGXLPV$I(*%j@OP`iC5_v)1?xi0V~e>RcA^NNQ!-Lnm+d*FYjNbs}tn(ya{HxD+fy2hEFG2H55H=l*tdR^$*cuoz}R@`eTEoc1~30ogldez;#2!ATHGb8QT<(82tvyXrvwO&x0cdinP?UYyDD2PEhI(O0QxCstxuYzx;xA}%DZoeh?FlklyTJV$I$#H;SegEl%$<SlJ#Zcn%%PVd8`xkvf~xbyipp6~4Ygu-7N!A~qEb;JYp-Q0y12mGpBQbjm=YHJ>I3gB?D+w^ijP3xW0aD1>tSXH{FEgu;X2``Q|ZV*pggx*w@w54DX#YkO5`iLcO3?L;3C7@w&J!&Qtdb;I&Un*#)8Tnw!nuveRoH(WM8Q8u1l88#pW($pM<p6o0Qu5VAma9HvTqmTIL$4nLXO{~w@XExymUZ>v8iB`9&FjccTOEjA;>zln{6?WEAKOTz^KW@maeLZ|Jib~a9T|^{w+lA=f=E$w9_T2^zzo}6ao%W~{16aFi90=hO7S#9z9ccWvCqj=B%=vFypfY>pP+fSyf6r1!D#7l{g7pTJeqJwW{eSU%~P&yW@m0_K7O#jzV8!MgQ#gW&8*<g#@KrbyM-UVzhr_;6rqjUzifLzT@F!nC*@!8+!3oH>=>wTBC*lBAhywJNhn7PomUfeO>ArsBw6+;1*3q*?*jx{Dse&Uti>6_W0&!WwmXf~uEOpcY?W;m<NyKV#%$(gIS)pcFB8TP#|O6K4QC)-ydMcE&3f1?X{G)EK@Zb3ksXmwAWuJv@r0sYko}y`LvQ9JoBYWLEtDt85qc22+BB^Qvd|6{T!=!cv3+iNlK%`weZ~p>UT&6U)?R8h9<j^iHkEVzZeQ#Miu_oZ>kFChWKgPi`lym}U;5!<rCu?3ibydM0LfI4Y(K$7KUg&U45vu1YP@RuPflju;c-G~x1CV<4icZGqp<4c3~e|E_cnh59y?k6xK{;8^cL8p6I${1n*<J0GMCm<ykaoC?aTG&HKn3lKu+F#x?eZ72!`V8o_!!_ic67J<H~@>G1HdVCAh%L@``^L7Ahzr4VCYImmpm7_*^X@^0_9VS{YU-;UorG$YbDY6H<hdHjcZ>&`<}BP9_o)2I_X8=qixwb!-YIUs(P7t3sII2awm8`mAofn|!aEsm^BNKs$=jH4C0M=9`J&$~g}n9X=+=>>?$76hWvt!(5a%`k!<XKx7T-RF7PqmXZR(@_j=GJpj{eL>VQHSX=08Xga*?u285Lulz{!KZw@#R_RT~!L#B>C$_r{7lQULIf|@TTgyjY4@8QHMH^KjxQy@{e&Ut5T0-&jGb*==@{H6!;EXTX6heP!9n#{->ShUyG`?a!!=8drrbs|D{kt=uBhzM8lLidjB&~%q89vak-;Y%Pw^Cj_blO8mwKw<a{nU(58y%=57d9#fcm=E`mck8aCfPx>>_%&N;ho|bO&r9TA_&a(jCxk?3C@#<tw8X6jtWm<dv_`3;U_N9R(*g^WN_HDaOzHW9;bupt3KyA6q1f$iTc1_6H155>DR>oDQwn4pVIKi7jx=~@@=_4mVTASxmk1LYaxR<o~yl7)G^3&Q%2I=uk3)raS5>>n+3L!*2iAgOf1YyvnpC}$~JkpDKlge)he-nURNS+31$eBOBYs=eQNCNWsP|1@o3_oY@zsX=5A^Z%KI#G=!Sm1P^{Ck1U34o=f+@z=!`;r1z{(!{Y_7L`sJhkPM5)RH<qwI(mHfZpoOC^5$IfF*I6nVsuY3-)dE+15|IR}vyB1EqUp0!tPO{jC}%Wfz9i~xqG_Dvm+lnPS><ytZ6A^LfzAwTaT;+;thp!-?es>-+$t%9Hk^EahNWS;M|r-<=!p20sjqc`M0`|}JK(#8gPgfchj1>h2IgVX$>S$QpqX1L_DQ>UjvlIbMPe+_Fi=ZehPJTA8kgVW&E;pgT$Hza@Jg!|wn0=Md`aod@u)&_U?zq23)SdlLDeB}Pfi=xhFMSJm8BI0Ip^#&G*wb7`Me^!sEMMV3&L=zi6)sEroDN5=j_J%K<`XnYX15s86;yGFDxtrYEb(|O#k(LG93RG5Qy=k`fIHQvj)z<eD7QNT5e8R-rWWo?OWFTK<DSNSPA_VGyvxPe7ys4>DdF!G8nq{4DITD-Ke{;;%%H;1PXd&r}%HOVT4J>b^3_%u#4Z8oicjh>N*qFT$<>uD8w(4fOcHh*sK#z{k0X&IrUf@Jggx9@~xYUtBzB{j(uf5laj>4XPOT8tdk28d~Ixu#AB{qtdPz#dA*xnEPO>ogeSgrXBv}hmsy*lgDffV+51~yaC;(d^I<g~u#l7f8GbS7m|(VSC#pKdZYIZU|3tCgmXW?=TQxrDti8$|O>`39k#fJK$o+FET})3fV_!-lqXM}U(FD5-7A#1$$Eo?VK4SqFxwDyH(_`%vJ0*v7Q~bE=MahwS1CK2lSXvnhSK?rkTONhBAh_93zCh*soYO-IGHEx@CvM!d_&u*tQiN80zzON8y7f&Aw*V)f5}YEYHpMDr4}oA~z04v#xe3IMuKW<{L<OU|(j@Khi<|4W^Diga;YtZ*#f_#|QT79Z{hge;^DfNvvLW54*q0c2VyN_eHCu?TKSSeK-_G(w^xnvf3SLq4eOF?WGrac%O@!)~TMILGRn+)c@6Bb3d(#-KSDj_sqntO3Rsc!NFyq}xRI1a)S5CoXmc!M+?+yUT5Q$FVyS3y_2M+0^))-C@B{94sr$xk>VI)&`O*Rhdq~w-Sa1f>$qA(pQd*Ff14ykBpcS6IYWkvKOpR1Q;M%@duhC&4zCfm*~)|L8a%{R<fGvnOVuvFSRTzf}0lzzP9YVI#-%{d2g<9+jV2-k(<uP&>kY97~5t3M&OxO0nfC6H`hy(2#Q{rrf%X7?4wXCd(DHAR1pPX94f8d@jGd?%#feR~9&6;?-$A{}J<Wafo_X$58I4p(aIoWnvOsJ_m_+=EN}rh_;aC|}39b=o3%TQp#_iIpQu-|6WTtevfk%kn2G4R2(alC+DVgAXc88D(09@6^QBITrzesD+%Sm9u_8JV^hSm0J8RYdx~1N`c9?yoqEI87D)#u<Xr2I2mntXQy~Lz2rd6iBfTGJ*j=AnsZ3NGbE_~+;8`&9AUw96o(WEEZ{i~@&8j34+^7*v&;tBipS`4>&|evjIo^F**9KP`Suz}xrNZjWqIv6?{amfgkQxBzq}Dk4o&9p$E*L;Cg?kA1b^Jz-}U*N;SsL~nORFNDFNGjoAK#G#=_?l^S_8GuiRwlp}&)OV@F(0+-es@e}tWPopptIhVyZvhLK=OW?@21g4jkXK>rY^HxsZ?6?ZtIY6PuJW+}bFTb+Y*YY*@t<|N`ioRqh6!8vX}sU+PJrCn_CeA%B&ft>3GrcB^<J^eL(#7xa7f2G-9on#A7UU{teUwru{s|KRHv+TX9PAIL$iL?K3){)h+@6@r;XB-<#A>QKOWc-dtFA^eNnT|RI*1m5t$j2{}D}6tzz9!R}Hvt9j;^*6nrJ`T$<Anw(9%VYW(tUuwE9AfPjniGVMV0t^{pT6T{UF%37y}W#J1miVyj?u30okd8@1I3bBqz`1EpSk$ytdG{ju)oeW|t?x)iHIy<bDGH#8&SAdkJ3uKjBHdC|{&*OlQ`21oXH^RU!JOiWR8Q^wTA9(;O)+9WxSp{IpOtr()R=s~LhxhxAd6rL+Ohf{F1NKYq1QNAZA_(hdQ!(p;1=)?x+;`4`x^S|yhD1hS#_i*ke&8i$|FX8V{ao=LZ!?(9OHak7t>eq!1D<<y1AZNT1()x*4Cv}VTM3AzCFI`l5p$vRA_fc@U47+onBggCDrs+FoE`#+B0yw)_V6%xo+q>j9BArthF%PmjdgAtgec7U97?kZM^Tu1%2Cz@BYa3o4eC)g<$HD78^;>S^iz3IJKILaP4u}H~WYt>osg(iN&6h!gW%be>Rk$p=#=EtCLJ|JV~@4-(sVl7P>KijKh5pkg25C4zOg<A$&xIjl*b?k})NngdrtR;kYj1SblMBobZx-t`Zq8~_u+RA|+u33VDuD%nYTigcU)QS9FpzU2>4Lbr`5qbf84C3cw>)M5&kqkp6;Bu$8p*1we@gIKp7oL~bVVCgRgyx3ir{VNpkk91Ye|ov4`na!Ny$wjQyx>HXX}^b)B8HxAY~7eCGliq$f#yE<yIXn(@Vc38WR^#42^R&Y2>Vlq+eC#WPP01YiYOgB+7Y1CvzFjGj?6B@7M8~m5Hx&`<HiG&14m0U5)X4^u~Vc~+bRH*-p{VQnWL9!HT;JuJcihvzp;w2Ny^W-i$Rop+l4{mAsWRjw~xaIIeAANjB{uf6_|Xq_f>`1zRWl)q86FRT`$k`gyp2EM7Apl+b7AU`(#?ac(kQtbBrqUrhy7MB|!^xsdiQO_<xL?5Pb1`L)f*M_5{+0aD99TE{4O3A9#l*^G8o<z~IhFVrB^|L@QMjyn}*$)h?1}GuI@0O^Oe#=|f@jhbYxb6&1d;G~y#`B&<KErA&Ubfz`L%BiWI|dGkpQkZrvxhM$?gR<40Q(YawFkJhgw|0&`P&!&YSdFS{P6jCbg9t;bS>{8v92M$ifb^o~%dl2QM7R?b)$`7e3@Yt$JJj}%jJhY4G)^B8&!l$uptvTw6YDrQeT=b+UdywZe#S$2u<VK8fFD$%{J7x00^TPX&cmF+tFd4Hlc;Xu*;e_6DSZ>wKu}s2MgnFI5WgSubK`#+pA>m}$d)C9_PZ?v*mjMW=MOd0pZ##0JP%*`3(Z(=ZB0zj=3U2ldr^9fm($f>l$A>&t@8&$Jq1OekxR>ddTtU89_#c-ur?Qh9j-tTuauW^!ZG#WkIj3)PvG}2H*b&5Ep?p|3BF6D*wN%rMe%uaM9&mw9q$l%tu6e6*{l7y5r3h=xp&?sk!z{q8z1F7{gS6(?D+y9UZ}Zr*8lv4WD;I~}ncdjZ;_TFk3IRPu{dy<1c0|H>P=!YC1TP5<Iumd?LeTyqgVf87sFSs2q<{`?s|IfPRBhjB+0rU3+L-))eBYiL$8LQqnh$uANLeUrSw4jjxk*JoNe=e6Sm_?fa634TnI9gQmg4+g9^*qe$~;1zPL3U&$b=#b;YB~bGE?OYxo-Uz9=aE*f&*V25P5Pw^_W&o&!A!{mTrY*I@6{mJ?YA~zp}U?ZngHZRzrdvZa_?n6%R|#baFF8x2l^Iy5p}0==q8dMfP>eA?q=Kvmp5I{K^hmN2`fX@u@*EoB|{1GPZ_mFCA$$W$X0g7~U~$k#^&g;|RrEo_?Ky`YR((N+d>zEUD`Je660;Z^G5x57TYc^rR;34&a}>R-BZE>dSx>Mi}Qr_`P~{MBu~hs=gs16LTtg>1EYv!S&{vD{`d^DGNafq(||pwkU6=JvRlH+<Nay7uguD$)I44k`7ezD_W!&1sI7>iidNdD4OO(G8o;Q<D(k2C!gF+v2CEGzW9uQw+wrZR#CfO?SUs_7a=+25@0r(U8swvEYs#<T%Ny-Utf{-!9G5wt-b<Kcw4?|;7k5m4v?+d8ud`;;VhD&gMwGTDR;j*0SW}Q?^>(TU7nui8@|sPjykTcqIdDA4mN3A5E)S{%GP}e0BI_)8lHaMAUWEL20WaHRGYK+E0y?kqoZ_<_g+JxlzvbSp)tDxcCDlT5u86`Jf8}iZoRh%=rUWY9C@Q)RlVB~%yV-GZ?i#it%<`r{koBK*1V@fO?hCj1@K)TAyKM*d!EEScC6*gI+r1#`!EO*fXm6Q#ynNrZ!{koi(;-+y2?&x>3+4eH#5P@75dbD{plkmU{Oc<rXs!QY}RVO;{hfVta@nik)vv-%qZdcfp;`nGUK~9B3cD8HTJyvq`_-6e{87id87h{%o(X0Xd$U0n`x3m!Ir>erss)rs{au->Yv-d<X0eujz7T5cxAJDXjeq{e?^7+CKLr+9!MQoc%0$c4zN8q?Eda5VA~Z`;f1!)l9j975cwisSmGnbARnT94c11*5io71TXmeHe#n-MZff1!=qL*^e|Qjq7Z<-Da~J+OjPMbc8F0L)>EUiUGvC3LWqwbgRBWWgF#i3Q)odo(a@oLbc@*d8m}`dC(E`~mD`i_~N^H_XUwp~>2fc05;RDCZ+2EdM!p4vCVsx{A@vDv?o-kQMZB4o!u_YFl#HxY-n+iO<y}{F>h>#K3+l7a!jfs(>F+3=_-W7pA5@Lnktovb$y%EMS&4n~^VNBLyNF+TloJpjYfXZtMa|;;Fkb}kn*gbs(huh&;%xLFT{`BFf%oAID_0qdi{`uyx0jCo&(ZV5M&$e}i8@M3^V+hsdktJfP{iMo3AY+aK{<BG}Ib&;dbBH$*{bjcg1k7VhXQ1%-(C;k{VY4#@dEtI4hn<|Fb_><Ne9PD>#JN0jM%aBRyL7|8?80lEY-2lMlDZQVFG3+dQU+C;da>YfMl5lUiNy0{ck9+asaX|L2$T)UFe7%Q0ckwCq_S2b6MT_2ur}1RQ-okf(O&HA?-}|lCO`~f|2a(#BnRP&E(SG0)Qb;v?3a18`uTxEPCpyTkQiM~VG`hRsMkhMu`TZmdXVnz`-=iM*)jYAVg^QfYwVhS<iH3Eyx`0QA=p8@R<zVARer-hw03Uwn<{&B?RWKS3eR>+ewEOig>~yl1MX^i%3>CXO?7$-d<aw9VeeO$P#4K2b*cs9t2R)Ul~@USez^D8c|RBVGWzS3x5DfQ)aAxn6ItkLwuccJE*WiJ(gv8BpcI4Sbi#Z`{K2@D6>y1^SH|8SeA_+~X)jD{Xq0@R6_$CbJ(H7Y^t0N(ppL_jr<kcvMZio6*yhUpMn(yV1SwjxPoo@aj>*<oQF#Vn`!^zFaaleyxX3tIs<RD0^V<N8wQF^IhDt_TqE(5I`^dzT(Gumz;c2|*l^2qN)>Q8MwH{sH@1uU>{|cHnS-l6|n559k-cfuR{<9<n`FBxb=@+1AuJ09bvV8@#m8jPlEJK=uCLJTwPT2)FH$PV5>drU~M{cn4XF%kX6ZC}M8nA%bKEWVeMvbz`d!=z#R7mh(?beYPwn@4?L*Zi~USOW&?iu}o+A;5e5S8s6JA+5RmR4Xe-lzhEC=kgHj)PB4%T^HV&=+PAIM%tV3}2-nmqI=N!D1+%HXIno0H*szHeG5Xe9kLyK+XUokRx&s0R29(P1^qeV1#ZuF}gc1lE}(RwS+GP=P7+%%;&5c;jL&nGKB#}w|#C$@$O?44{mttSSK^IoNu^Hv_Imn*|B$S(d#5bZovuXKa%wbyvCL{?He=Rm@Ww{l`FibJ4GRJi#lsibiVrvi7U5FX14fqX+r*n@v7?Xb(eecG&8l4R{qVU8Vk>>ZluiJgFo7%aQq0~Ch(%tz}HF5g2BY~Jo!Nh>7XaNNPX#57dtgwHzd^uC9p)7(@7$#B?3A==RK&2QmH0(!Fc4buR1VAw-N_1eqC~Tw1lYnRK7slhvdex9C+?ME|!j>9WLwgtd2YYr^*IwmvLh%MepbuMBq4~V(~+lFI^}m{4^u3Xk)XNk<_hu%kAnFNawEQE3s8)BdI<pi^YD=BkZ)4#YbWy5;(5ZFV8L`|MYZNAPCGLkGemNZFYv%Q-@V$<)CLoIE`}<hC=lvLuDYLXk@uI(cliy#~VgLxZb_~z!8_!PJ1a&`3bPZUZZdWBDE0N2o*nyZ%l0SC-wA?Q;{KUfZ~N?_aCBKD)<Z~+}HqT)6aUK0qLgLiY_8B`BP<5&t<9|PypTY%F|Y{tDZ_P3aVoC0L*nAs^!{Uw0fwQb$ikM=P8@lRo5eO!71LuGC5F_;Mbz^7%He{AOSp=fiS3xoJW1tF>xLM?{TGzEPAB1VfU?*_Sl>5x!P7~o12f0>8V0ISS-=c_s?|1_I)!Fw&f(%wZyyV1bjX%{S#8aIPDAs(Ti2840bH{*m6*?M&^@`h3$f5xQcP7BL9e-*9(*)E`T)@m_pq)k=>sWRXe+cjZ}6zrmWOW+8MZ&IO%WZJo!O!lfw1(g>za|^&M8v$@f`&ew!fm5AFYlEfU(*t=3DO!hkXuckmKqZZEND(@>WL7>UZ$`ozWP*>a)OaDgpfn@WvXQmUl@a8^?h5{4nshWtMiRkJlHcM`8)DxaVkgiybi5&a%R{uWZov$UZKkn)8ew5voKs*4?hp{z+j+9-vH>*=ivs|XU&%T!&PRwRwPE7RsBS&WbO7+iPw#?>l$)5E{j8Nhk31Fys&aF(GC_9Prq$1=4GdU%55=tuqVAdiiFqp#ZCn2vo5ouC%N*KfsKwNxS!@Gda+>?~0IBWk=;c~BIA6h(jGW4{ssTg+QPZX&X_*G|byd_SnS7p1|kg}KX!ZnAw&3ll|Dt#)Uct=C?_qJ1ISAXXFqPsBsQPwq|`XE`!1$FKGijU&HJq8&w$os_szT8wr)TrVkZ7#Onv$=$gvakGl`Cz=WQ{PGUsSGS$8@m0=Nc#lO?+dfem3q?J^p4g`ko!;TC7cvNKbqn1(=E`1^Lwr-0QIkk7dkgZLhNKG3RR8%$7P`3|hU2c=%hSF>t-hIF!;>*_%e6y`@5x{xoPN$hma;u^|7WY`3J@&L=SUWEc!-f(S`>iqgn<fBu-}zo_K+}RYkq#*r3@L3D1BoaK$UI`e2Yd5PU@BGKI4aifUO~@=o}VY0A1GA?IpQ@`^aZ|N<O%7oZ6%eFjKN0d=!Awz{$>LIvsvrUiOp4mwx0N%tK}~OJH&!;VW4QIix#TD@4p3J?!kj2zu^_sXDel4z`=c2^VFaD@ZH}7Nzl-36Bk@k>Grva<X|FJc0ae95Y>SP-TfFCjZ`b`c#1L*g{fWIm@K$1@i|@eq_kxlug^X(acQ{g(!LCaUcG=?cFl66VCB|7I`F88!$hFU?vYWaegMuaWC6FZFQm?!**)q1H!YqbSJk4n@^5$pyjQzxpP=TlIgTZPkz&`3;=SfU<ZraHM-mmFd&Zv^1!)zW<L@zSOXCCcwm2uEg2q2F)Kb(zL-ODr)D=c;*T~+ZUa6M^~aZ!^j7ql6JQf-C*ciP(u9-~V{8Q}%_z4D`k8Zv(NOCrJdB9YHEWYeFos{a(j!G29wM1=4!hoc5D=2p=u?_3eYrHO-^a1msVQ-dX!AOaAHu4Eeo_w7=}AG&>N7qX&gLuf$_fYdoalE;RR6wX_Xa!CNRDlsX+lPl^6kTz{Y7*Aq(EAa3E(uM&YZ2uG+_ljj^nIj|Bd%2k^l_1q!KIB+W$_cL1tE$LN}$5fg!~}*VQ3%lEVn`^TMwY*)G3&AOfP(z`^JiWz55U0rdK|?9W@ci}XBH?eI4?f7HWhhcO#S5r0=sdvKEEYN<!ij_n;Gv4j($*5nnjtzRZh^B@2@+2K*c`WMeqjv8&`^seM(P5S?5{I_obgzJ+W{OQmZ299(82)@|X#GMl`SY~z;8;pIgSj0o4m{d5-%de-a8dlG~S@(w_Ev@E?wa4i^5-Nnt>2oG8yQ*+Dhb~$g&U4ZxM{G=I#ok410L{5d1sV=_mN;tGjEC_er=AE-gJ$!N(o4w{y?crIv1}bz#n0uXKXIOm<E-=~kYHf(XX~H==+41)uZvWTiC*W84Wytx=t3$7jG4`cRWtCD%1~^V={>-9)@KPWMSbJk%6vdT7RqEf?-gqpLlyyM$;wgp_~L0d*zfGkolepvcf$`mQ<WLJi}=l#7>NzBF>pX8Z`-UncSXpH{ujxz3bJ8z`$I{z_!i7h|B%#Rido{_Z3{?DcD1t9RdD-3iE)+$(%C`)4)4WcL6+sW`hogomE#|{<qCp0ma{)nC;mM(h`XctVFIpp3;qYpa%PGf|GcE{-g29nld^CbInEN?HR69=ulfP}c$zgb!uR;?x=rdNQ3I{OFFAw^N>KsrdZy8KBwNc<)xMBgmI$gm<a{~qaw#CGX*Ejw_v|S>`LR5XW8boas0mWBYMexKM0w{R*fxcA8q=nU1j}|i56)s5&npcO)xIDu*8K3@e(|*G2f=H@Zy3hQJ6OKoN;30M0{-GPQ;g-3cxsz(m1kF+o^7f!v{&gV=%^6;#_LE&OTPf)LBJ6@8l%S@^UTMnL^zdb=$$<c*~D$OZk*SN<%j*tex+&-m^<O%e7*`~`ukgtYKmmH>%w@)(<{U&2$0+8NQ6CC3OjawUB)_-7V^um!^n~pjnsg3AH>8Jaf@N+a5yJRcaVx>c>agwD1_{uGSw??{ND_cr@41O^+<>)Z*T~g4g12Wt)2lm7F!?_2d#1gm2w6om=x^lLK6j(?Ca%wFZGD$h!ug*c|+i>$VtPQaJYbKC{Wu9Ft|mMKgw=w%42#qOJzSwHe%C`KBvr-_fIS6&tT>nvD{ltdeb>$pn2ioJF5k)Pm5rMmn+jjyPB|8YLO|LXj<Q|;0_|I5TX@%4HfD9y2g#W(U1c3G_8o?$LZ)~K7a!tz|Q>gf7sI#^dB7h=Ob%&;F<LA7&L!lU?KENuMT8b3TR)pT>8=l%2aAu>w{`<#nu!{H(%y!+cM67;d+;WwV6jSiYf3jk4K3Bg0dp^hqoLwBle+GKeOMEop*x5IaDxV%(6kM5Vh2ckk=i7rgXVa>u6B#c`R>8f5p8azLE_(S*G=c^97iI?{K%Fi`H2_8pY|T$U^RsK6}mQ!V3Wxub!WYq|L@qx$m*-{Uk$(O2ro;IN(cz<@f?!)P5?3hf~cq#}>!EEe^W!bZ==Df!^}8N8xub_MgxhjPX-Lp!dY&SQ__L^UGl8A4MR>!&}#JLhwQ;YPt*)DX;C3jdgANTc-db9Wt@_=Zphd68yk?&4T;x+B*>oA+^4aXs1%3P{c76Z%=1=;Ib7t9QU*7BLCwkVv*y?ri$sSXRKOsZlIp_C@7AhGAybSgdt3uw}0DWfW>NZ#(FgZ{~dROz2MAOZv#WX$nl@El4ojqhi0<IVf6Y*9Z*1dM!^C#*wqM2pw=fejCY^$@JLip@6s#=P5(!Uenl~BE7;q^H?TPv3gCD;Q+I4tlIJrmM$tH{vr^l>sdhVjD~j#i$0)^`OCx(;QUTRJ#Y8gQl#~9pS`>11txG3zy;L;qs@cDOn)&V+Y7H?>ui##Pp?7G*>PG{XtwS_T3huDdY_}SrRu5AClDEfgESW7jM1}nF{Mk5pB`5NT_QMbkdZlH3_n5bt{2WO@qiMu!u1~W{H_mj|OeOtl@suJt1LJ7z9x(M?na<TOK-+Yl>8_5)&mB;DWomEsMrB>P0>dhldJ5_W)FX=3hfwI2VC^jQGc}EwtK506h>r$>VsQ%Z*f7aUE||gQRVIzZQG$Udm>&&%NVGVwj)Y$HBztIbgixQjEKC#+{ulwDBFgb@bIHFuo|Wx^=R(*spo#!H8p0G}-^$RqQ)0XMn>7S;7;a)8nKAcF{KPhglu7;E+ZQ3K_y<nY8(dJUqLK!u(4kq3nq<2eY}E)i-igWw!A*jBK#7fgvc8M{l6+od3F*PkNk`DA0-kxz(WlGq%x7MO2PFE7q7_KQ&W5jyXlL&tf#;jRHK{o4b!ntMm)g?i@vGa?n}Z?22^E*z`^Ta$7iDm&g?_?u=|$OuAm&QPF6%Nye#8#qixQroPR(%z(Egur9_6GT+>E-x&SC$XRKM0l=p!9f-)Ln|XMM^0kox8x078fIQ}>EQ!3x*Kj1_x)F~MN8f|28}P6ZG7N!4^*hC1(_R|hHQ>YmbhvAkZD)%l4{v{Jp@Tnxal`?2_WvMttSp*vVv8Q<2BEj6-IO_|ya47j$lPL^GBi+h2}V-+LTnd?q}@MyehE9ZiP8LNz)C$@4+sdyZD_n>AX#sLIP>};zQYF69kW5o~r195xG{(kTNF&YlL_?`{<(bJE=($nMrUsM6&Cs2di>U(2(OD?qJ&8M8hf3E%DmaQ32=0&bwb+e35gNR`zS9Kuj^9khYXyBtTM9UB4-+Vwv9I2fkPdnI=D?5t81YTo{#SPmV+}(Mn`g=e2VbP@rX8+kr@aF7}gF=q9o=GcRvX)FV^A*fbq&VyhUMl^KnC;_`WrXM17YQns2v&M&#O)N>&)%`}XlE8?SDvPS19U)Q{y$Tc%sxq(pLJvjx)y0g_drGaUP5p?v9Y$tF%74|*O%t}BhNfwb^kUEbjMu0dxgS#7TWL>+mi&X>kjM^!3!+hyzNSf!lZQDJe~VDt_&#<i&gc~QvrL~&e&wmy^Ag7wM?%`mxFV#6@PDyXbnTb0wNbRze8ThttWCqyq`i!%cqrcZdfbjS@M~+Nfl!wFadyBZ=}(+#^%3Hi?sAX00)5w68obD=fp8C`0vkgAiM|;sn1_$8;95-#mVaLL9z&lcHhW`9wcja6y%h@!nOO;A|gt8XYiaB@#log)~m4R4js;tC0R8la>3FkASsh<y9|q2@8sOh#O=`>hyE%IfF(I~R*{#>aZ4GATFxz{<a~ZV&Sy68Wl;l3EFCmk(K&!roDg)2W~Wfmn7kPtb!#al@+=ba&YC3Rk$<p^qLCkTNx}jV`Wk0q^I1DMyj=XZ3I&0kpZHnr-gKuZQokbU{Y<`_sR?o(MxR=lle1=kPqrv#El*&CVUpfR>o<kg)`>joG%s&z=$-RHuUjtCX7MtQv`U&w!1mY2##T7ZbxYxDp91O)>j(C^NsV`$?kt|5(d$|3`gOHh{i4fu^bA|<r}QkgOUsnLIx*4M4B_jWtK@5v>C#_h)_@5l<hmxcDj1+f&%3X0Jyw_wdSN9^S;}hsw*mCWJu5(<OZ=0fqsEL{!|&Oi`PwiD7-GBrVj{EC)MDefADpp1NK;zcu7v=cdNB_jJdwykQY#v>Q67X2guJjQ>kd3(V$8telYg*5jL3v=Q36+TqjH|hg)`<W)W-aGhAp=wBND<&vOh#QfFDS9l#2u0RB-OlM7JRhSu6S-QC<+wB^3h?{g1p_J4D}QItH-9K62atGZmh~g_GchWL1r_MWg5E`jK7TxMSeksk|DSB6%gx)xo*2XK3VB_;E5&IAEosf_B%6BsJ05H#u-yH{<NAjjK65IvsU)E8fXMMNfNZc%xNTuoCQd<wR2XNqoPfg{@^e^+l+|o7q&0W7kZe-Np+oy1kzNizlp<gtw{Oz8V7A2{R9%EOL9AJNBcoJ;X_{l(Ib2R4@++Ood5V-5^OUkAYFj3ap>C0iRhIg}1y6P@#UCwb#4bADV`y+g7&(U!ypOE?XUjTS}gC>JGe0VNV6j{{SAR^i5r^D5tf4N}|N$WbG)Obg&HMnVH6koftXXprwdnOZJQ;htkNpI^@ZHyP=%L>JIFptB^~5@!_yMv7Cu6xq|ouf*^5N6!L<>d@)I%kYNT&?;=miC4Gudz+_o(x8=;Kjy}-yfLpvEM`oDQfQnaBY|7Vd;2R4ygw!2al7vs)=>!(P%Q}fi)B$6o9oKYd8MSmFy=4-U!hM!8J`X4$^?Q+vKy08x4?f{6pmSi-#M><5{ZhKv+l9RT-A3@|W${Ue)=hiQ<F&>Cy?5c#Cr&<v75r7=z_x^5O;<JQ=TQNp|M2^amz5pfp_^|FA#-CU6uZ0W{faW>cX=}ny5usIE-kgR_9lYkGZ!z&2^7N=NzwmDjlD1U``D08ye|p!Pd&O7;Aphaf;Gf<7oH>xW%o1zLQGpKRTJocX+=e7)e~xk3kgWp_0V+6CpO?J#<e4u;qh1O4kF0R&QlzyD`QQuV$Af|vXOiYeFxvU0y}_~3bF*E<ELvBwyW~ylXn+!_ga0398yFiMW*erWB#-W`!V{p!kZRP0ZXhc2ts*YfAuOprG^aQ`(ct8|ArKX2Sg^OywSR9yXv~kfgKmVo69gcIcCRh4Lu>Cl37EbO77+3lYsP47ULZ|Mx^S4ef(dXOyyowpJpqj&X%}w0Ls9}_U#c^$djG4Mz-&Tguj@JFvFoZ2&ms+=<itU@x$pu(m$?94cU~BK1ZSV>AWep{fK-uDPe$;3zPLCsy=L<ZU&dbN{Z<z{oG^3r2)PZz$ft>ltI*!6h@9wPpxK%COv0XzRd9z#VLE>3dYR@=+jx_%(HqdNYqM?3D4J+!2--t;&Vi=yFSLg)w2ZgB4FjxG-g0VbCdWVhH?Zn#@|J|TrO{@frr`U-4?zoYV|CLZ~VW$i%Q~&)p?BsYw`vZy5D8*5;Aa{RdXAyB1H7YziV%B{JK<hM+`6F<Bz93N#ho?3!D)C#*M<~!~_(LF?OXZ&F2o8uqRyoi!a{;1E7HP)jtIn*jo|dn&Rri9_=-wCh>Juk+*LVJSfc%b64%NpH4M^<`}EPN)AuD#d7rmku(96<}&O%svWA8*%3!pgr@NTSJB})ex+r}O*Hvv!{2GU1ODZFak{!_f{kn{$7g_D=l~FV{@(d$+Eq7EDdcN8(y>Z}5L$LfU=zNgfP<>T8=`JhMClxsCPm*5WzMuDYUPe*r}=OOb22?cYTfY7rM^q`#o~&^{pAj44T+&5BMJbN*(+Y<$++3E9HDUsi;pO>Wvx?apX{Pi6S`Bs>YTb-td2qJ6DLo&*rbg14q2s5m!|(O<9ysCgOmH|D?wI_Z@}}bLnA2T6fWYc%Oh=ljs+@9)$v0YPCkG-zzksEK#=iK*-7;Um?#x)nHL@#&h$@}hii92G@E0TVk#xWF(Fpu@ORuF`4GsesO1dV*uYp^CXe0e>Xqt2z{Cql)fl8oK@^UpXj!K(qYqooMb%K@22BQQinQtzEy^A{zuSvL1^`x%2m~uB3q#<vmE;*FJa+9@K{AJuQ_Q}T2Q27s2Jabd8eOdLz-Tn@OatEno7w+sL+R0q@>qRvwvTa}Us1Cx!bBDQTKU!yN&)nbn8TB9ghf-{zf0#&NLDyX$4g49_b{q#bwOgtK_Ecmd>4n{&&qwNiRu^xJZ4{5Hvn>2fIr%5kAR<ioU~`ZYhfx1AEwzn0{sU|cKAh?bs#T++@RiZbMtGT=w555o(6`_a-<8T>hjmycTT$D8jU%|7-a&OP%>1TJu!A76{#;|;mZFV(W=2r^;RaAx;H25oC0R@BM2|c&zMpd%?zb4mH#ofAbPs)^D<i1)Ru(Md34ktpvwu9AEf;?Dx(#~2w40A>@j$AQ`8_{fO|KYE(QO+wo9cBxWdvCe1G>SbFZJ&`DW)+nGG;eyaO}>DrlJ9Jx(CD^ZRDZyWa5(?ObwWWDLNf?osnSq0d``b&HpbsaWZiqH9P=*$7YX${Mm0HRx#6=B2o$8?-PzNW`|_CQ93VZFidJun=F5_ir>hBf*Q5$75{w=SYQx!*wi}-B8yfvqT(xUz_T$lkIDqSaLrO@7e>olO>IX^99T+UaHVB*aET#14CNX2R>|{fuQRpCjYXy#ct8E$wN|`MmN9+sSj&y4Yi;o&xl;#&yqRWZu?K@8AIVjTJ-=}sB>-|)HuHlM+8f=QfJ1~EF&6&>9l-C!>V1)T~?k`WD}W7dc&eY%p+vU)-ngRFyt!AR8B0dzJ}^*?iviutKO0y6fhk*(G$7e0wjxunBex=;Tg~mwA8j)nI3Qp+LWxIIPs;9*v>e0`4#~0FnotVR_ood{%FAL_{Ub8p>nkQkjCvL@k-NWqHNv<LwPS+&(nOELO28r3-4{|TWgS4^70=epg>eN3<k<h($AHcZHC{h7q~S+INegLKNQy2@SuOduKGs>d$IyEKu_B9qUE&-abN-(;>26$y=?c%Ogt`@7@*9MF&DfrR%FchQ6fN0u_SOU$#8yPB@5`;D@CT0(fnLAKb_<Awh*-?0<IeTC3R|Ta%#&k0k5ZZD!djA@Bz_UkWN}mJZKTGnjZv27mGqkXF+SQLczeJs%D|-Af(p-UdN#gz-8=hwu}A}J;ZsO#3d|P^#c|?hw^jbN+jEUU%-7}DePNpTk?E;b&xilS0Ez+8}>}+8V*T%3suSF1T#sEC7C+jm{Z)>sl)d)5<0<$Z#@Z^SgQZwJP!~8z_RT3vzLT(d_Qph8>PPO7(d>`4=n*Zpp-)(?$6U(300)Mg^<RcA(sj-OoX1yxB9Nspbguqr{&jIDdANSB2wWu0to}B*Ed1xc!qM>rLizp(O6Kg`!wo{B&4cyn~Y6#R-AGUKjv{Hs|N!1DBKDPbvEWTD4*ae%1<yfp<TMX1^H`&CL@P$zCedLC?4r<crCBS2vPd5D);N-HfJnZ!Ys^9(>EGExvq+o$Q+~+Rs0mZ#lYenut{nyyn!GWS)u#0`5{Ul8*gXF6oo3rP#NuOZUuEIYn)nMPbDb~(}VT)Ibi_rc)!T?)H2=pVfa+SF2wtYV|VaZ^^jN$Jz2^|KN+7)5<wfh@3P{!`8fM?gGk*ri<){#g<XM5+SUeB=qnU63`{D_p!p_016EKGe2N*%_pDR=8L$(n=--FJZT{QcQ8R}VP!M`v>@AKLmroY4PH;a91dTaY;)7Y%8}YmnwQJwB)qat!A)<cnn0M!~qX$|@01{j33F6!ituR~w=C58B`wh`k;eKrFRGBDB%YZ|fw)aF2J@M$>2a>c~khB@Ix<|>`W%2Yb7^X|_TAE&-5Z#z^GR9VKhAe|k54;lE`AD6)b&jompJ3{pAgWzvFB{ciyqM@rMU*R1j2HibqnQd#dh+l`x22RJsctaNwGC&RLABHKwZUsJMksZm>=>3B{n*5&J}u(oQxXk2$C<+#Kwf2F^qgCLv)()SD4xiqKyesa)u6w&!8(IJgw@B}r|f8HyQLI7NEA^kW-FyIM@hR0I9uv-i(h9!M{RQRtxZ><x_2T&Mq+AG?eLd6-oZ??X3bdPYbAHl2~KAYl%)gH=q_ZA26nw~8mC3^#kW=tq9*IG8=q@Pv8fa`?BAAL$f43H?nU};^rpXhZ+}^dGtDPT&n@r3fy2S}v38`3C}~gV-F8Wyw$oBm0|$!@X*v6^FB1f1VYx|H!_W{h$T28|;ndH;eAlTtp11h7ry8_Vnc*RXX~36ypezEP&vJ*P51yduR(IO|NreE{rw6!;TQPj8>t7p|<V{I<Q4n!#z=Ct>B>C{1BHFY=n+BI%51i-I-z5Y=RJYeQJRRivq-CTFR0AIr$Me1-+RhM(z$kftsGkR3f&BAfkz`oL)Cytg?*01_E;~}|Y+BOK0~Dj|5t@C6iQAFcIQfk+SOQw!1DbvxiS&|)&0tqz{S}kAb#R|Qulu(v6pJJmBPO_Jj}hrJ6(TxY+-B&>k9^emUr5j@VN$PEZQy6p+$1XI_In8DKt9Os*$gd^+sI`0!<;&wGJOFd(o<uQHKvA9fVnDfUSoiX`naOlxUXl*ei+mM274eHySypk`SU8f=O~RWKf}c<!8M}l>zcD*rBYoFB&3Cu7dC^1#A{V|(Ez-(2OSKF!%{ZSL1_HCh$>?g6iiF7ZHW?C^mizq85@vP*VOVCiW(cMd6Y|Q9glVE#_91!su^{u@1{T*Yn1BsMG!EK9tyRnd@}@bBXPPDnO*y%yzCa`g@ovQc{|GLS5sc9E=C?lO0LL~Ai1%iUBgJ5E<%wHUK=I<z^VeWih~p;Kg~SaRBw=yZmL3^nHlnqqE1w2Y4EfJ(C{c(>{X!d43M{uzrWV!xo}0xn{~z4k2X)|#ArmrZp^#;#e}%YLQDc$-Hc1)N|u=m+R!gEp5&>dM7Pop3bZ%?Qga(CWoSt^5-Ohb-|6#a`KPU%GuKUgtZBeYH{p(3kSBk9qB}t*W!7`~BTLEfV;&E5*J6n=fij}X;EH?gx<FC?z?OIPGj3tbG%bAvKfO0M8iTaV%d^obi)teeubGns>HOym9Z%y^CISKV^d#7jOsJK=<{@p;On;esSaaU52jFTgb4e><gDGP#9u*{GMmla;qxUx^?cnmY7c$Jj$sEn$_=y2FS)AH>Xw@dt^}h`t<*%`X`1R)8u%_372g*3dTd(rcnSAsP$a`hEF`>gI&>7V;AaDufWOd*b)N~)l8kHVb&^3!$4Ak~W*@s~6bk1OoTHB&{Rcho=xr8DjiI0562fa%Pl~L^i0VA1V(eF6rL5syP^aa1CI4{13Hw!zTbMg$pDyMkjDy-OB_%WAyG?Xck-ln`X(7Gq2eceMQi$Q|5n^A14&$AF!fo2t=u2QkX&eB|08r|_nOA`Lf)KBaJQFDzz7>9DV%B`zlGEVpB^Spvb_e;=j71v8VX!tX{O;<OPb49y9tIe)gJcF7KKlb-H4xErZ@Ic;MM28q~iBU_(Xi&_w&99Koo`sT=eTn_-y+RV67hckY#QP*Wi891b8TlI5eL#-p8+?U>E9RPXXX&9h5tqp6WNANpYy6urLFh%k;96)afKsFH{BvrnBEn*Ii$ZmDl8@V_6uKlDirUN%B@8x6IKI$1d71bNFlG-F%Z}<;)&<}<+HqJY3y)1&g$z@%izMDtvh@{OQxF&5TNlcdHw|-0a-=Sn#j~Ycn^7I>hcAvoOxG9lKy};Z1YIEB+W=Z2cr&M3+7drwiukWV)hvUq?-HGnA~l5)B7NH{P;A4F^|7g~pRG*r3jsj)mGE@6RO<m=m!Wn20j>|(0Gkjz(u4nSqw`C>pl%1nOeH5{s^-sap)=+*HD6SHfD6y4?_@WvrRlrk;^i<1gHFDhmone7RVnw&=?%`=L6{VVzNHphaK=5@4ezv0ye@oe)H3PbuZ#D;6rkUmb#`Lxr$!S&$$RL!!p4mN<e1^>eq?rS%T_;;SG+)O2Lm=XAy}a?c==2GXD<T=`R7*##8ScqWzS~_u%LC0-NHjrZR0G%qI~=CmZ>eVp%u7{-V4aZu{cm8wW7dz-RdI&a{<#RCp`w@JF}sZb)3KI#twn0m?ho(g11G?Z1_|7+n3I!AVYUFGRk^jQJQQgs^!tEH!m@KZ<6p(u8!7OXkhq{3G<ca*wGTjoBzcKIV89Ta0T<AhP=CWKa7CO<NtGg^1r474m3{zC5JhCbiK7I`jWbZ^`jkjz18OsA>JwvE~!;5I*`pn={!ef$X8AWVa|x3Ml;Ee$<iRM3s8Xo`|*bD3z&hN>_MFQpWhykk(vXwgo-83n!X^q<EoVWsZs&vacT|&bm<Bkm6B_>;2!2oSQu<G#rIQ_HsiC+l@nn@nqBNe^%l(maxGv4M#x9%&4je5j1VM030Ztggw;sjAs{x2miDJnR1rb9lvsM4q!VVfzvPb1TxI6*aipf>F5r$&{5v(?!#K15IkFejN?OMS!kAtSXFX^)v&(iXt<4yO?)qlg7R`g7Sd-?+aZxp4WFl9NSjRwlkMVbBvyUs1YjbsN-Bp$YkjY%l9zjpTGKT3k?*JAiR(+@__0T|!zI2gY?}^EI-Yk?VZNeF>;5<x`zhE9&k@WW6xM2_lTZdfj4&>fH4>~1ms`P*7;ulUCo;@Kh(9=3@cMxibrT_d{On#bDVLOP#GqmOLPE{!)^rVAUKgtdl1uJPNi_>lUOwXp&{^ogXTB`wFI+)$7v<x2*7Gn9i+zY!hjbq`+QqqAE?;E8NTjPL;LNkG$@1u??PjZ8hMWNM~QS`6We8c1kqp?g>(N9Q(g=l`)nS{N^*raH>))I^m!n~5Bn>5y=7_PiH<;MFF4>7oDir=DH;cdH4OQuO;!&inzifdt%a~2G?R|!7Obt1;#3WW8{-kEyGH@}|N^4MVg2f*zd39qiBrYm<nN9B9}rgte60ZeX<oxZzmxp}=eStB+rB+JFxM6|$@1Dhsj(x$C?8?O52bf96unX>6|{~|yQBhcX`ER_y4x1z<+xx6Zwi~7l%b2v`@?&~P>Buw2k-4d71Pkle*xGyk0v7seFFW|F3tG5);WoJqawxMfS=H)G9cmO?X!|(@rh51wgf6RP5i_~wC36x*7oAN+Y)Wnn_BxMBILY`_t95ABARd?<QfYtQFPgX^$bua1XkiGE=*_=z0VT~L^59+x=p}HDApsljcrl~V;J&N8PcYOtKS3v<k-b<>hX~3WKO4ipn8z+vdI^KMQ7E=ie_*=_#+@{$oX~Z`zzT?(WK9`!KK}Jh}>#)>A@mVd+mA$)T{(rpWvqAc5&xAk<St5bTnUMWUa)ouj7jTKjDDO1V2J|Tr^x;|t*N340SCR*abx<F&OZrD7C3uX^EC7nj<Scxqi-jA8CDg}WMjACuI)@TmyZwLZ6gbH`_|fLday1wqDbCWWyoUBn(QBisNA9@m6+g7dDle2p(1pJY8npnr%UxIp%-2IqYivRn^1>i1c<ukDhL1>nP0Yp_{KOs}tKG172>94$3UYS(-6s#Sj7(`QPzhCCuxXVyAx}2N7@T&v4~FTtah4GF#0D#sRX-jzgrpw)TE0T!4$x$mJa~ituxUK%@EoW_e@oogMqe*=R+~!gMC0qESi+bXUv+-2Yt_M8vEJ4W7XB~N#{%7B?r<eFqB=id$@+v2A!YIX5C))zPRU<im(Z5C<=+TUjrrtrsn0V90zCGcvrJulq;qFg4gMGVq~M<Fje*phAX^=-)35`eke}u%>V+ZS@7RGwjKA3#?Hq#@)5m*5k}zGg0r7Lp;wwz3cw&IV3=h`@&)KlURc?@CCyeGbmv@>Y{E~USDfNc@!@j8YO`Ap0cV+yF7W-4S^ZOtr&{y7`7Xy&&OXBC>84Mgd5W^3wj}Z|5mw{byO>bzY&<2mgVzwZh7Xffbhd4Wyoi}vJ*jsTy%;WqI&_V>*vPWGZ#kF4?+-5t12<7GCH)0H~H5ifCk_6sI){SR$ngi>!s&&_y?I^uk9HO_skDkc6i+OC=jFQgbZ2iZ?rNXuB5fEh_)JIYoc7R!XRaN9s%u+~=GGskfa{BW-!ZVWTgj@rlEPD~9!DDgk*%S&omx-CQgtqDZIZYeL5ADPmCj0N-hf@SmYWL8^ViZ0F!tI-L#(w3l-S0B4k76C1cC1E)3;nb7*3L3X2N{PqB@>gimoy3#FX2t8KPf1#8Yq&hE-t8x!h3`s+5Wk3QFg6g0An+Xr%;e@6y|}*D=+7FA_l#u0D!h|4}OWm6kbL^0YFf{{m98M<+aLDC$#yBF_M8+Obh%_STcNu=1*gTnnfTkOu3l0L%92{Eu^c0t{Okl>~*I^Bk>mDPD(1w+wC0aH6shAs-gW!tbpG!&?YT+!4;XF-^fD998K4(^P(uy^Q_!V`sup>rE%8OAoyeC<;TAeiEbJwNx0)Uomb(ZR-}lAvg6$Iv*%(tw~yLq%>KPw&Yk$(y;mQI!L_d6NQAvem_J%5>|Ze?y9@E^Mo|t@7cYw28+!X*@h}q)n4KWbcS|P<s%kk!^kBa5n!Svagra4>BWUvA?dXR2@RZmYVBqQS$WvO>+N;{weBcLmaWUFt6|#!6s#GNH_-eH99eMTx`^g*LEfjV%BC%{j&l<<1UM6;=H7qxp+O}_yCmWTkAEqE)A2}`UK=VS=Mo|{G=x>9x_P9M8--{_YRlt~!uOToz4sc%1)`Z@U5i=aBd%J6Xl{1JD#|IcO#^%+Rn?qwQrE(W=#b}kabK~Ma!AJQfpqYUi5W7VSnTE}QNbeC@Y4#ya4&iisXoXln+y?MMW^YybV?;D-BUfQi&*pNEOMt1?k2=p_DA_BEAEVGFT*Mv>DImSvoe|_k`2WlNg&6har;|Hoq0~=(p~x2p%E$-taK5bG)+2wyrh)yYYoUR(BieM}D@I3ZIVdTGd3IWz2qBTwdmA#rFp!xtnQ4P{A+alL>BFN+3>`p!ozZD|r1e9XDQAjVCbQWLi;pxPM{N8K?07gUJ|KbYe&rnYauSm?LtaWQi8Xw&J`JGECit!E;}+;qMDAULT>2WOve!!R5)hPV3XS?_Tx!`UXY@YBTnjJ_$F8lb$IEY7A^nAc)!7>hpBbvdTb)`gy>TTFD>jkcl0nESuuZdAZH%NH%tCu+K%VK#hkW&B-OUrbY|>$@n7EwG)?V8=@2$53Yu0)PtV0;Sgbgx*Z*iBLJ}$~)VB;3oaJqT&*X?KSO1PkeQPBZ9y|m~Np&1Okz5DJ3aauy+L0gSK*pw~h43Jp6`x^(reTk7UwZbON+C3)yOCO+3M~wymZ>u~)bHezReVES<{oicrdeqk+Jbn`GBCAq?O|iY9eZ3RM!_47bh)34H=Sy!4X=doH2GYvL`wsL@y_Rkf!mq`~g*HkPOELqA+ln40V~f*9u1)f?wP=Dbr!r$~T`0M=_tw^2#>DQ~mv6G4tp2*hD1;UkraTflJ>X@AmBk8Wqeo?k6@pR^q>sOkEbdQ`gRcHk_U1@Lk+B~Du=LF{+&*F<oNXmlRS50PoV=&LNsFQ?67@jH)(CR070tT<D%od4#E4}wJ1U%5(AxL<x~JC;L&#w<;yb~x5DJ^ImlH=ivk{O*!Rr?xNX3CcJ=XJnF{t5m3f61G@(*XyCy%_%&l$Kr@ovQnDP_#x_YdcM92$Eev0(uHva82_&}>q%eJvZ5#v*E8P^2v`*2bDwYjtfPu>E|td84W~3>=I4VAWzS#0vUT7He9%$3t{1cw7i{+#+k5SPsz@Z#gXQoM|Ey5;WR@{qE<V7`6|Hu%luJBd{lA7BDfS4ZA9_n7j@oV?ID)CBH`H20~wqq^YdBYr|t5u*s`C|2TaRKWtvfya1josJ)$RKJGQG)MAdtf{b&`SO(_rrcCcE*?F4*WZ99zNf4FtQ5)zKM9iyuM#-TS6{VgW%aexVag&R*npl1t%+08J2AK<Q*5Y6S@AHH@-F-MFa@4D?2-m)_amEo|)x<Q2z42%1eY|m=L&T4=Ys5dTP9LI&uZe|L_DvTpeeSadWv=1&rWr%BtYS}Q55rSSK`WN16~N>&$0Yx!LkES#TEVH+D_~sy)_<ve{ycI01KYB3Zu6G_#1x-NZF<9bfndon<b)*h%MyR`HI-*_v&SLiFo!?q#;*aI3uRqKc_BG?bhyy2{eEgK>9#12zW^~<wZM5sGiI0%Ulq+Ox3KYg96J!%qR3MP^&nc$IHo0%`o(KQCtNwAd<u*Nn|e6>7Tfu3n{V+}CMdZd>6Nw$bBMt+@{W;6W(K)TlC~b@$M@{&PW5I((CP{ZE*F(FN=ohmBo{K(^6i<m^kWYG|4y`lQTmjG6`*W4ea@6XZYc}*fw)9V{i9uPCAdu9JUENR%H_iQi|>RStn1xjoZ*jh|MqrWeHIdr;8!V}Jn*j)`fd39+?{P2UV~&!yAM$Hznev$m?M9d<r6kGxp3R(sk#I2!#)mUfs8HTB34dir~*tj20|6*+exe5tR|DEZ{Q4?;^BeP?se!}_tT%<Nw1yAGN=o6sVxzBGy=(>a%S@;J89fG^SGEo_!+_4VS7GhB>f8V50>@js@mTDCfS}In4JA!{#B+S8b!SJL4>5P<PrmI&*d-2X_6Ht_wp!%<f;h~?3{hCDiPEH@T%-43ZFQm<=CQNG*Pmqd>qO&yp7$S=KntHLvb!`t|)ROP&nRKu~`XYv7O;AAdiAQjF0D;by0m4A<y`$qMK(gM|r^)6j@6?`EE1v)?4kDh#5Us72w8p4M7$dNAU(O(vm!S+UzOmb=Mk?BViEzB~kim><<ng_|j#(AJ_9fheSg<P@}ys%Oh@tXKW-&LV$S^vgXpAbp!}2oQyl;m@I?tWarp#=)Q}jI3k7DP!xoMKY67@Y?!vcMc7V^pT__P95<JWpH`=k00*Tnf?J_Q&b7>rwF3CtwN~^9r(O68aHO4Nj;GEH34%h=x-koB&XUI^ODqN;hV^0DKugHgKQk-eM2PzR-7WLkBoYsOz|M*f^+~JN&l^o%wB_@T3LnIeLIvyxe4M0oUlH`u-1o;|{lpsHRwIWyp{i*n0L^5Dj%J(C9-Q!5BQ`KVI?{XU%+P9lg$_6MC^HUis0>7thktpN_aoeNSY(IuDFIf-2e1p2%x-fsQCc{e65R=GjL{XOzw-*)FnY~#k^Q#vd>hh<JZ4k>RwsjGrG|8PPa$|Vvf=T(m0M}zkzhM{iC^8PbYhG&zq+J_Nx6267Lnh#*@&ykm1lJFG2guj3mgAa_TBX@d~DpzL*?boxgZ(6G-^u2KL`w(TJbgFWD;d%gCdA{#K5N8?>EfxJy`Arv0qBs)RUxzD+*v6a*iXL_S}OpWxT1S?j4qfQTO*~ECMI$J@p`=^He2Y%eDvgO!g_8eIIJT>$aYVvJ|-A-^tS8b3?mAD~tX}C4d2pg1V_f*X!e7i05e=?9ik>{Io*T<9uY~88v3Q$lLhN0>d8_JOg1-QXb83BmFHpPl@yi!M$M9oqIYI)umPj0($qbcn&ChQtJUHEE4R)T7XMccoa22&;-AM!h*XtMqYjY#BZcE+1$|qUk)eQGdKJ^R$!98>oko);J9vz+u$-E(Cc#TCxx$T>xXzeQ<^iq5_XMIPoT`YN2A33GA%ikDdIU}`UrDFikwF|Q|H%`Qmy26Sih%zcVO?15Ak4-I(OKiDdZ5iW;d92FDLKnfEG*#2l^RezISk}_g(F1Faddnm$z2ql+Al}A_=JaB|}?xE3|nHtw+k5K$7syb+wKxLKxD6c#mg><DNkN*HH3V`{e=B|H%(F3Y|=Y`W(3A=gF%ahc}dJfyI^m--?~5GPGif0fm%iSQ0f%Y)iuHn<`Ouh)A&~sv&vR?a|Q1mi$>Nfwwn*z+C%nPgA1!I%VRrV<GW-27lN@#szPcLl5gu4Zp-{QW^arbr5nP(2M<<!T`MUoV^ARF_QDwWl-xBLvT=sn>bJ<K-#|uyp^iwLK?sbw6qNPb4+BjwtnmMqej<Dk13T2V@Kw|8Wm#ZKvm818_;(Nk#<~3q~R?q7|!i!NhOH&VF^x|T}BHqsU*IFlGobAN=eAZuRQ)pRu8CRAev!}7+f!Y+w;fSz;j1AvvI@tC)f9HjnXd}^8Kzo6qjpz@DPt4G73w{(luFlh-Vl>THHl*^v1wgXX{c+e3qYBo?B}*QMGjdaVe3Ap;I;89M_LbPn9$7(@_(obPsF4ycy^{QO-o{n6EKBY@Bj}r{OFl<7W{1WqB;t<w8w-^b0obt4Bk~*nOA=F5{m)@E<p5BKHxClQ_=oLh_phscwd39v0k`59B?OAiFs<GN{2zB#plELM>25;=E}yRNcLDUoTQv#T8ZX3Z5?(Q^l(b4RlveW^x3r2u?gHw^%BwzwIb{qoQHZ!XBG-s`-)`m+vdIZhqEeLqfV{1%kMaO-&R>YUtmIy%HVz(6f;oqv#Q;h4dnKIxuV2?V-R_2e$zfxU;XU)uC7qTam%7Zuw!APdrd`^O7`{c;L*Z9wamI_*&S`+!DwEHn_l<`}Ikq4c@{IIsx?SpHOvEmW%4&4|R_1E?Q=JtWK^r_CTSi^2&CH6ijwe)qq+b>5TFU%wh&Q>`aGHb8OIMxCkB5tv4-!Z2*e;?u*Xi5C44RRt7>^FEXCg`z`o-ki}mC{{4f9#WRhPtU|$<P-WaBm#&(%aR73n)CfyvduV3M8R&}f5*B?<gE4x{rs?AA-FS9Xepg!4E0AYmi$S40lE!cU4_ay5z#(LBet4U#Yz9Dozq^&<cr)CsOsV|rz;5du%g~!z#s{CaS%yjgtNw&!4UPD*?ldIYrTO^zQ5*GdiyOL$=;Yrk`$K6>gR}y>35)L5GSe`>HM>?CujMQQD@~<QSuD4uBmiz@UiluSxUE6kw2s4Y$>I_AkiKfFoc=mC`Eo_>fTBA_a8!Kn2N!wDznT51>@o<HNT0qDE8B2&0_<glJ~b#$nlUYG(IM9wo!zR$<~M%I>3ZCi7rMKP{208*IuVT4mUX1&24xCR9EkF2HQeJTpX+@_;b9$u-|ghc9=E;ZYu@_Fb7ryIMl~MpUt2c6FIB=v8v*_)QdZ~7Q3s{RS9*DN7YNeKxBM9u@8tc*O{7=!b5XU;9o?LwBC)!z4+wOQZ%rW{od#?IehIlimm&>~pgq~Ul6|DXA_N(<9J_R>#qpWoH8kJ;dv+1jLWzfADfrCKx-cEG5F0!-1Y+zV#kER)8?TVuz1Ompa&Kg`7Js+wBz_?{Nu#cxuDK^2GO3Io@LD>2n&ya;aTVeW2j-ar4ZgIc2toO=ZJ*DNEDkOCDSl<UY?n`kTFY7?g9W_)Ik>&0)NpOR5j!m<DeP*T%<}xMw|6{APXX{=Fu|#j{f0_yVscSY$U}y2fE+VX=c|f8dGpt?ZrRDrcW37be^QVr%P2D36b>U3;TXNj@d&^lfc|}9;Z>zke>zFM;Rf~h<i*ZZHsVpGpwb6Uq2(#Rd&me()2SX6+=L?v-hT=>tE>IA1XOO;QoC~@B&vOV7YWkz;!!f)-M5cr)SFG^dDBxI2U@jmP1v6zIY=*oO&J9m)O9@5r5EwbRnfe+$A8no@d(A!8w6Ut06VY@|KY1e4#HZbSl1*blMsp9XbKKnBQ1n|K`dB1lnSpmHdXiY);M)vt{|V7B33~cd0d&oOYm+79@kJwfsIlo6X=-&3%~7FWxp)Rwoi*8KZIST)03`RF*qGLYc@(3LgPMt;N+4#vQI#W=Fs64tlq5&&%8jZrFCLQi1>{%QspB=+`nzXCpVr3Su`}&8i^Zzc#2GO9u7$?3a5Av>tO1|$>%y3E=$SrXaLw9-TH(n5h;epejm1XG8zynLlS!P_a}>EumFRk{?`hD!4Vqk+^a!o1X+aElj1W-XmXVeyPXUWqgEx6!h$g}z*RhLG+AadbWj%xUnPPx*}ekAD|q}?sZnp#nU3+mp=tay|K&T&M5#v@myV|?4~?#w`H+Lb^u0eW&KX|7`OCpOX8Pn8G<zCTifW<cvwAcJa-Emv1wBXECob%@FWFtgCUqUr>(Ig#K{dl<dzP_XJan^shBJUC#iT3)zbb=%IAFeLNZ^Vi7rdT{l*;m&iua_gM0nn4ORq&L3YMd>WE|Em`!8cyi+~<L&*htl14<+5<aV->GADk&M|~VUplZlh=Tu892kgoIQgf_-lvRDGTcUH0*Wq3WnmcALMU`ewF*n;QTx#dzE%Io&<qCYu=eE}?y&v>CkHb<R%vN*oAlSu+z#~Jey}J8ej&fg3I4&l|B5(9f#k+#88D%R<=SXA+#YxG@ldgjVH$A+=AG_B4s3`GI_ez(`zdVq7@cwVx&qrBH+LIIq{qVmvz*TE2Sg?9#_jDZ_5XmkFa(@j1R<euxe5aTayMQ&vYfP)13MW}eRFAIH7O##ibfkQ^$GZP!B*f(ulCf;2LfuSIm{I^EX1Y0Vsd_G03Py8Q!48pq0dH)hS#$I9&ejvT{HA<DUF&R;->yCy19>vYVoZn~FmJYdmEUqgbvub%WVr2KPTtlWUTq|7jX;yqmZ<pUR}Vs+@A(^Lh_A*|hfykw(Fb_;BWCkymz?UMcA=0@C5V!1VeRUfS|w;ePPA<H&(rR)g69vpoU(J<6a;TY9=dC;hs-`LmECDZAL0`RgTDj<;>%m;CfUp*|2fvU&Pj2zbK%lTb9C_Dde_Gk7gZ!dN7-5?;EZm~ly_R(XB>c_EUi2Xrh+4>_FekM_~;vnz~Cry@e(ot>~2uQ-yo^R>0`i-UTdd(AB)&d3#BX#D|yKExbdp<o#4VUfMgD`J(TLa#osZlm3W{gsSQ5-{jlFd_LpkYnp3fOJjHq&6{-shW?FgVSItlS-ax-ZuZz~(BBA7Ln(u<(=iE4}QzK=GhZX41Ga(6ufW5r~4LcG=ueg~U7ccT>+q1{6QrS1$XW8X;C1eEwpW-^oUvhly);^8iqfqsI$+unPg&C7K44+KAq&LfOCOY;~h?XvOdr36U`a`|Vjt63_EO^~AooPLs@@zllMMu*_I!-hmgss7Rh{!9eSAQJyf<yXgyS)S?G%dW7#<Dh^g$qz>Updu}go^9fxv=xI?Y>7djUqka`?L7d7T}S&oTuy@oOKG~A5%>!l$RzwKPbTM0aAStl0PY&a3vj<CpPNIN@wORZcqOYP3YOQ#qjgprz9j-j!-)>Hq_Y)yo-E8fIr3bBy~H<@W$eJJnsm<Wa?RF)&wEsa}IDcsY-)J`GJ5bwL$7wg94$tA0sMZU>ZF`<>>V<&dxkomlM(RKCcsNRNhAuZ8;&w{5EVE3{O10?5;I#VL?wz=`Svs^?J}+ft{QL^_$fEAjw&fjW3D+=le7v2mypPvjsZjxxXcY@8`y?6C^_N?(4)zvnhdwU3bF=0sYb_pKG|F*?HPXl7w+EyMt?ILUu+7%7(!T9t2KGi)8B6c+v_F2~fTgHIj;vQX;fVBowj>fH&pfJLJ<N3VMeFk__aNXO&fLJZ@)RFqn8~&1=V<NPW|OJ&W{XlT+=qP=l*ggIYY}xD=!;s4caw5abTwz1ttH%9ij9aVji<%tzD79{VjV`-dHZJHFGTacecI?^)QBs3lk-vQl~>EkV&CG+lcNh3;?26L>S&S9yMVFPAjGMTSpEVp;NP3uHa_9Tvk;Wl)x?a)zBfN3G4WQ1|_IxnC|tj|Us%o*8h0V<!PK5!iA+%Di}Zf_l*IL|+kuZ%iuD_&)%1=8<JdYh@$TgVEljA>}FlFu1zbnu6)4hUZZq@g>7(|49S*M0bCybPoH}1<GB4(fy%<UW9Q<dp{dtJ4&i|@m`uEL%3XNiic9URe-4kXqIZoS~TEl>tV>dL*zjHk3iz=SSx`vM5L&FBYNEJ#(B_F%B_@>v~JICMTX9Qm4dBz<E3A{LL|y6|CWN~3ahJx<Je)$u@#suz!QuW#b_iZ)p_Y#q8FIe9`w)4?xVz-zC-fpf%HARR4b5FUJ=sP!@KQ75}@utij7#?XTMxT;F4~+NuJx*!o0SjEFX98LfSZD|Ef`!>;<0c4l%zT5)=xhBb)c`S4E@L(y;TbE!UP{o};+ImDoQ9#_v*0{{(`C8~ateOiOeHjJ2A#US93g+^v%j3DUI&w@>Ij&8>1RdVd!ZgYpdAGYr(w&<J(y=4GKxtr^=c&x5sZhg*v{qcp;#KdGF0u=zcfPw2}gtPH)QB~3ybxjuq21NuUBrYnqAa~?WL*T9AM-Rv?Mgg!_~*PAMl9OGz2)<R8lC_#TQMzvHKb!-8Qf)kHM4GDlkwO{lbx?_V{`w9NHh=I_{Rg2;FVe<zG-BB)6v2cS`L*H|ey~yWqJc0jUEy{d9(+4G!@%DaTVLBTwm374V%rDdIbGLtWc)aUm-RUL@)aCD?4C5B=xiX7lD^D;+9S|F@Y84=!X8UsY;<)*ibiB8bKNvG9rx32v(V0KnoPu6#S-vx>>W?msnDMD`W47uzZ#HWl8@Nrd)lH#E*<ZIc0~0=oX9Kk4#;_F=d0VI*PZ!Telx&@oa8kaoU0Hh&z3p*Vn!H#ls|U9xWj*NHbUFJ;V>fb>*~#&Qs>^!7y40Z1VnzUH@H02W(kfSlW<mz&O?E1#6}<JS#Gs<5X@vC8mKIZ6VQ(?90EjMH3<g~`aHnDEDUL}HDagOGk{!+|1o7QFsYrL^bvTIl)^QQN@G)!Y(6*s+NPP5^+k|)mA(!5@zUgG@4ArSrS7!Aq_{}E@{uRcqDB~*4*N!X?7(-f3;IZn}wvwoFc=>D`MWNDbA_7BseVrnt{aH*XokqVzC1^Q_DEeMRt~J{H!xrN-$wv4$RwP7kyrkJL!$T^t$#A$0PN?Zq1dq_f&*3y*xeredLvKP=22m!YGO`YklYhz)85OZCTI%PU4x~!KaI3{%1G9dr1Ed!rurkr3+)Ch8<!ThHLrAcp28Y~mgx?T?#BY(Uy4A^`7mQ(}&95|`O*#>n^&)BVeubDDmTpY+9mDV>>*K(-eGYe<vur-V+pjT!fl0JB=wRoNQ<D;p?_P1);T!%yUxl}V1&p-d(a+@{WLn+%@xZwEy$0kwXgQ>gSRoYSKq8s@SNsuZIiPO@vpSWrN4M<1PYtVM7$8woKo;^c+CFwM#Xyhx9sLE!ux0!P4)b_}$InsT?jDxpjYWK4^DNx)Xw&f9<&oWw+w8Ko88hRGA~1t;G4AB;pav%HQ2{a2!xSVPl{#g3%m$GtI>m^ogG!xrfVA8{hNGOe+!<@xA~;q5hC5t>cb^69#RJVsOH@t)C)*uGoqqhe@FCNas^7F)|GK(Ax4I9z`Gb>|BY=1mH5G?3Nw-Gj;%I=ycSYB4s`aeVqxLRb(<^edR<2d*vmjpkr&1&XAcDKCM+%({`p|=VrAGWepx`ePe3s%er}csXA;xb@B{-b!ETj~gy-;8f4l|Pu?WP{$g@oJy9pquxhl41^;%n7MdoImK1jcj<rK+DT&<I{M?g3Y8+<>eY?<~#za=ED|rvCWCXR9mo{B~8%=4)7in;y8eoQP!;;L%PsgrZ_|c05~lU6?M8E>dMNA|KD0(+4?6wjQauP*zd+K;QL1Cu=8E)zTA*D|K-YAnSIhMeKixJ@Dd7vy*I(zk;N`t<P<pB}8NlFnz-@=5U-Wyh#3-_uMA`G@}~>-Y232uK9o#wZhmzSq$h^k^EwM^^tRdiq=ix+26;K2>@1RgzL0q0MalEH{zQk1X4Fr31WbrNc#T9_yhuyTK1!L$OHOT!F58~0^`7QJF_S-$L$px<E&n6`Iwh(9dNkBslK=4N2;DvBO3bwc|w%No==P!fD`)$%rbn|JR}s4iAdmHXI;L)+^4uxuCi%I*@cK43kwOR?t}uYw^a=_``3$ASclGA38J0;U&mo4u>&eMM4gu*C?wv!7QZU6e|IB0*Eg5RwVeG!i5>v)&7zub(>fRg(3eo}c#SED=rBCn4*AB$O3QsO+nz1X7LeY|)nxT~3FJxQV3x{gJUy#Y7jJ!}*m}y?qzjZNwq6sRo?;*~h_V3h)UwZJk#cH3jq4+6kPh$t7ZC3}mEbl8PDX|sbZb%<04Hqyn`I$Ld&(kQS6r!c_CduB9s9}P3sq-zVb+qqjbf|dH;x}!>QH|>X(E5?HO0%*%#FdvA<*JTz;H5oM7#OU;F5*S(wZaVwnAE$z@jVDc=uV*9Ix*QQ{7GD*@*RVC3wu}p#(EB>i^K4Bo%^j{=*V0{(Ym*T`H4<qRCO-0`|K&9*6xiSro$G-h4bzcA5T-3W!eHWC|{!m=D1woCRZ&_;g=k#=;XiSNh55%s#7?ZuAto;Odr#8n(j|$5}{666QK0(8YUUwEi7)5cX)aZT;q~>4Uo0Pl>SX*nw;n!*~c8R7GkI#>gzq5~CczV_mSm7;Y+Qgn)G)cQEuHsd?iM?H^CbG?$^^545=l^DVhv=a*?*4TM(5ky9b5=AA_B>n<9PZCf-8zr3d|&_Dlnu>%>GGxV8cjhLY1Y=V2xu(XzE(Oz51wJCg_OvD~fNom`j@dmCm)eTUs0=?Gpm^#A1=MHOO+PtsBe2NpF6qErRrLgZo&!<S2%XJt}07B%mM?X|#QZr?-lqy|7qs8V-O|8)guP_70%_!*2q*0bxxGQ%=1mbxM(9kZV3JvmaqT0HWW3Z?#>j(G5zz#k=B~Kk1Ojl_C!(hvZgry!Z6g-rAmMPVg#+mo&r$aaE)XYczu-2=#(5$XiZLa6v)M?FK2?^LSVpPev2H58!NDsqA&{;({VARe8hw9X#<o+Fzu>o0Pc>|z{UNpC=lrp0fxc;;^zvm_846YJ=1rMFQnPFU`K94@eV!doKa^)(f<qm`olDK8{z)xl-!h{G$#&~JGdtWFZgFLIb?&pm<<Dd@sRyA(zaIIJh5CJ<yVGl<=vLK!j0e0ZYs>|w}<MW&Stj(%ZIP=cdKbq`$Bct*j>pv(8dLm8JP}^y51>HoS%y>*Nl7W*{Hr*t|*_j&62r!py_Ml%}MqOhDe<1qMSMVVD<TA=41&9H6*|%egv8fj1kV_W~@U^2_MI|`{3X#G0u!f|3V^aOb=5-b=$~ub^?_#G*%RZGwzP4@i`M#~g)LCsh$tq{Dzq0uxX^dWje7YFmGis)gepW1)&ts?YcedTc@lqZ&<tzpUCp+lt&5%#Zy6U0P$?<jq^Im7>^Eyh>dzg{|$_Z-*SJaMu!17WI{r0fp``~!n1-9!~uerXOLEF}+SQakL_%B@zZ%P%++@J&Bpv#f4hp4l%L#9~6eW~79el(%zOXS{1<KCE#Ff_q)`o=cROACd;kIil%V7JDU<XL-H`935chq51kzb;oXW|Hlv766#V>1zf|`cf5A=KNxyIhp<<tVTV|{Cr`7W{Aw{Td<p_8qdmJqprnl>hbvEz0s#y2#iVxnrXh`_;->RO<t6hy*dy^YKwF+qV533mT(B0M(m(xKpzB2=Ku=6u8vFM95>GZjCckOl-%*iwbT(U4e-3=*aflXy(2oQNZo=M%|64zFk-$9yy5KBppYYftdrD%=XQGZdtCz?8Qs^_=asxpPuXD@KlISSRFoJ}veVg1i^??&LtbvN!ACg%z3`29*Ij{{QH0q_39|BuFWG}lxWY~-l9nfyqQu0zq?~`6Nq8RWR0Uy1`2IG3wW~6#v2Qgf#1g?FT$)jHeBp-J$=V+NQ?uC(>U!*JLtAgRlsjro3S8~21#o+}y%tuHdhuvH`?`g|+3h#I)w`|)HrLjAN0cF=O9!_=KMwA$W51lneY=lAOKiYOuCiZ%Fn0r^%@P7;?P#JP@KM9-WVyH@7debAnwc7CzcsGYezy6Ydg=1L$Wj7hy(CEwR3r=p?>OvGHq<aX8&BqeH@rIKbDef$!l~&u8(1*nqD^f~xHsy+(6o9H2g)^iFjm66%KR+|g{r>tU_l9<1o}aO`1yvjSZe(>t7l_ZLG#K|taTMfe}j;`2+8Xw%(53ET~PoPs>sWx|CKVa-C)Y?ZiX{fK8qr>Tv*>a%?xlK&ORAkY4PlykMe&{er4RHrsb59?cA{6+Zg4cn&R!kVnCk#RLTcMx`8LhJkvZd*j^(CdD5b})cQysSmMUH?1z?ATpioDvUu_plm^JKl}6o>duNqGsm1$F$u@1jG;3z8A!lGR`fs0>t%$fqf@jLek2*qn8wBRPc#$#S>Wkk7OEo)vKy)#>(Gt@p>(p4jfo~ra4{E$rL>8S%4>VboWU;zW<ym}!%QM{-8VcFRwV@d=!M!XgyFJP4A*U*AsU*AIy3M~SMZ(*F<#s=%qnKmj5n}Xj*22De;uA$kn}+$dif|xy&V62k5{S}P-3>lW=}%gXG>xVhjMe-qg3C_=z$2ASw%$QwlQMAbd%`&@2oi(Lr6NYTiY;ITry*{{;HaW7HHk_U(JbBVsB0Kqig~f_FoGakU597o1)*nz{CH9~Q}r%fvo?gNH5vr^Q%5Zd&(0y9tN`kTUt8`~;ymjMIOaG{PEz~$i$!eKahx6={w#MSOJc``a(0)gf)uL4cK~wcQK+DAsteFM@rP(N)#Op6MF=oJl0V;QK;2io=MoD0?U7?=)@S7RAcZy|Ri`3%`|5q0R!2UmIo)ohO!m;Z7jaImum1_fB2-QeWX<FJUg=5kh>He=e2mdSrdF4B(Ks7Nz$1l<JcWoRwk&5MxQxKAX-BIsjMBFhR0?vstmvGPeAHZls$(b8&Rn^{i;m~j9w~l1<PpdQYNKKkWM3g;j_1!qxQD&$Yu*wQS?$Z>4ta;$-yFUbpDKA8oXAGTaY|yp1XR7s{<YI09z9W$`f^&RE{UH4Bnd%(bnx#rJL4#zkS>93TR|6jQSIs43hW}ozJUbln3lIocR&t|I3+|QyN}ao69Bg-Ki^XM;G|yNiP?W-ilZOnpQPu^LJc}OQ_xCE(fjr9nNRvN>QF+z5<P&*ox~9Jc=!;x?x8NDExrxV5=Z3mVQ=&@8+HSO`=0lS6M1sMBaIL;yng&Onx)~+$7B_@EC=f7=+EvC=IliC!ZpYX-dT-o(5>~i(3z7mj%Y30HCI?=)%~T(AOy@IK9Q5DD~AZ=<Fq?9L%Rh+Y3}edV)U-rECoG68nTmwzjbpN9WDfi)9d16fkgf~Yz=U{T3XZ94*blH37t^=a4AH7fb`^RmN5d1`CL-LMKy<SHH8)t(ty7zw;?l4^WX-7pQJ!b@!sS3k*oGn>guRs%gY2?Lol~HI=7o`_~xzEmXfi&p#OEahbR$mh$}DUS^L#1riBw+qOy@NU27Uq=|on9I5pJYZK7YyQmJ)giRmz8YT#Z)u5&}j(yOsnN=(-+^HFrCLTCmv*7-bRIQXXZ;hJV|@2+~AU0pnd6?3DkRs!7uJKr#Jp?2STco8|Jrb2|Tb~gTjG;Za?g}drdM+BQyI#fvrbSs@0(@|n6*fGStpptC<fiadfI^dVhu*D$q6_(CV^p^Axz}(U-4>PtN%zjU#hi9;Vaz1Y&Ue`r0T4x!#Cwcd1y)mzfQ}5v<_&d6<xXi+(-{m;!V+p+=&*E(R#DkTzLbMM&c(YXU6*3=$>S3*XlZ}~rfPP=Je~k=KvEWSOtu@MsZ;1B<&sT9hV$ysbe1jV0b4=D%{a|dQ?&95_0_3#N!uLGjyw+sTvw9#z>wWCT+5E%0<x=JjHP907Esl+xGzj4BK=j1nNnP=%oW&GNDbA{s7hVwthqKABz+T}lu?Of8b3-<6bzg(`#I}P3re@t-yism`JT1Sq1|z&Ia5|g%MK1j*CR<i+XO++MOQ-ToEt_T>IH@^3_xvI$ma9eCpm5IEJ(0FKV9TVgkR8Psx{zO1r^<TxVm%<2hZK>GQaWdOMqoqfW_Rj9Mw=2hS3f-!%kn&g559|b%_Q2(cwiS;0;nj`eU(KFLZXv0_cARcP3s^hC-r;=&h_1jY8yxciA)61OQD2saL*_7?NJW;Ais2nA~x!pL3}G7!bDg!A3y$_)1CAzYD$gdm61-@QYb%Ncl{_3bXPtE;CnjZFhVD>dZRv1`V()f&_b)Lh^d{};2$%PFQ8c`h`TaS<0pnv(anEC#QO_+ml)05bgW3eV(pPlohCulOBxaVB@e4my;$Ci>n1%P2gI=X>IuoxD=*ArQIhcFcTP^2pxFgGr|K=3WI3)!LKZr|*ux<8;pBYaTitF(GEo(bo7VTmY=TfPdJcS`4Z@UoZ+rxTy$g!a_PL6^VNwsfb+rg1?nLzX{Ty&S(CkJ%SN-f~W<yfy*Yuu{d&M&$x_^V?Nqm{m&*t7V*NFJ<`3-k{(uZ=v<*$Borb=Q?58&$wRz)1?oAp?^0m_l0|E-~~rzm@f`^DSADFKHb?2JxOM1X8GQ1SClmA*^XNP)^ueRw}bH+wAJ4YXv%Gw>)hn9}_o%(Y>OtFRu<MLF7zpV;io7oHG<CGw#TntC6rI4y<`%U^eHqYD4i9lbSD3`t$~<p-IF`d7!mj71u5y&z7YhV*M4<fq4|(X-6fX{G507G5{?oY^+tvC0fwck>0_rH#iZZ}mph^Vk6tdwE0L-e8gW^uxOrv@uYRUTJ*E`S5(RZLa4|T(RJ&(19Knrfk)L=LX*XKOZ>L!LX4*$j#8@78AS_nV)8IjI(NqT9BIjlC3P+7fK*0Lo3sXMNteUOC(hy6ZzqJz^p5><4HtMbCO^PGydp`Xn+Q+<P|_|GN7IL-t^U>kSIAy{|wpbiRMqK(T(lx$QtRx)ltJE&^d{BMc7)v5p*Pu*&ko*YWSLSGpVe8#DgDP7FP*PTXV;>wsn4Yt-+pu`QV_p7{Kb}NJ(G7?dz`6Vg&pwymGSJE!Tv{LXK7Gkwmwu;?c2l%8q!rBD>ZosE?^`uCKDsE4~OJqc7^2VB^fEQF8htvCIa3$0AblU8i>!=YP*^0IAZNXw9(kS7da-R2*vyJ5)RFIg@^Wje?PZWkG^Im};lxUhCR31W=wu6^q}`kM)$dQK-r;<Bx2iRB$IC40l;9ej&cxKu*{(&_rUI+;tUN1o5INQc-*bz2nRw`l=co5u_QbfQ`hMv<_uoBO(VjEV|CzPJrdu+WaKz#T3s}uN!W3a>Gu4o+=Fxjv@VO$=TbNQtqQuqgaShTnYjV`2<$__vMJSZ95O%1Rg5b^(JBIT7H=mWSg4`I*`--cNKi6AOr&ox~=dyKfWVsmoXjlfDx7HFbD~4>`kQ3b~iLhL7?H)ImHIOA2y)B0$ih&ASdITsteer-L3wSQy`GULZ$SSD6jFpo}vh`yT8QH5d03QN5|r|Pcm8Ao=-I6EiF!FfZz`cTk+(P+vyc}jJJ=ZiS^hufHSaTZ8!#bAkv-P!zF88HAx31s_lpmcS?e<nREw<q#qOpD{)bMbRCd<c)J@kNn8(wcSNkyVUIH#VINZjnonQgUkuLlnCgJg(vNzrTR#Nma`_3ueV)%S4H8t4NgoC_VE5RvCC8zqM0YeMtd3jfNOtZI`9?qydzaj4AQ(x`3i7FT{i>GJrwXYTMAg*l<975_Z3K`GV127QvSaBO(%YWE#)OaB>8-kcG*Y#x*0s$k+GK!1Mzx?R;e{+b@u!kLCCM60S|(---JxuLeiyAdoCfL7)he$e5w+Dwn;~f!i^wxEHigu*r;waNf&Y%(ywh9o<<%FqpY&MuE~}*=9`)Ea4;Z$Nr1V%meE%3Io{lpTJ5xXe$>#cJ7Pg`*i2DG_Erd${+H2>Zu(veaCzEsPxDxd9E=GkwJq<##$ZMij?X<peiaW=}jTxpN&AEYVWJ&uq@CkvrW?Sl1Hyr&G4kADyzKv4ih9UBcpt%JEP6<+ODj?O^O!)v!|B&i+V3Iov2~OFUFy&ruk)i**X~ybLpn*#&B_!4kKbY-bBFOt)Q{=Y=G82fWtL)DHrrHEPMog{D?{xo51Xng-GXNm10`WLIluo+`%ae?kxg1CZcEZM=xu~4u0&GAzorCMbQw@fw$ZkVQ*(5d|I>EpWM(Z(j%KoR$RH&C(zMrKk&&0nb0kG0nb0L+3K!q)8JLr|4v5$__n;c;=;n^fI$GORm)6VkbrSez_kI;-<x2uu#lnqaPNFZZMA4=wF8+}XVvXuZZqP?JNq)3eiC}9Z@huyWYqAYV)(VYJcPv7tNd{ab82<EM&ZqljWxQTg1AUyAI)n7x^LU(39@~b(EtVXcrLsp`ros)Y5$Hk%mF^Cx3Pnok!Rxs(HA!t!R3HdOs0<E0jQRC8#S+CkM_kiBQL-mE0fT~t6zb2_atJ6x4HDB5R$W_GHp*~d%5pyyoSr8Bq2(V+I)>0iurye-@V)xo~Tv}?(l2*&|@yXml)d_;wykFzQw>bW&h-SIS7yIPZUzA)pZo;bwz0;9<B_7~<uc^*Tz@q{aaukS%7`dnHQ`q%JrPx<bgX_l^kOhaZW}LKxYm0Y#k?@*StpoDH-sF*$>vHJ|z}7=yX8}a0lUfbdlu(Fiw;uCY68*EC?j3iL1Q-y_lx7@?q~F($0SYEti@#V0*`1_=l?C77j368?OyEf`Cdnu<u_2X*7)JlK2cJZz7oTW<7L-M4UGL8f(|ShEs_hnUH|nT`^^oB9wmCHcVDaODCnWbooLP=TSUFYCc!PwRzp5Fa6hFsPs5L!cyQ?%r14#q5!~(t%4(Vj)KIO01yoKJic7F2^w)j|KTEo!kh3FN`7TZU3Jb?plHtr_|(sxl6?PC`Cuu1RM@!Q9WqC)O`6hCk+nocu5YNwoj(o@C8aE1;wUJ_%+eVxrW`UQqz+07LYCp@;!xsA<&{b=vI(j{DXqgFq78)*(#JJK?4sOzC64}>SB{O&bra(agJvq~1xc-UHn1Nasc(h((Q#B~MZ1WD4G(LlUl%U(WHucho2favAeq4$8m7?1G0K1G>hx4P4XTdrl+C)TSBoXBR<&7#o>w8ksq`M_?Ak|py29|zrg;uzMxcuE>#Z1;0UPH$5j8DAwdWb!%{Z{(M~goc0vV_Fa5A_qi>(PPoas;}z=6T)H(t-U*_E8fKO=OrlP7S4E~^Sc;q#X!*zHstDN!rj~kYeGV=6n6VwCKe#l8r7!}tSTyD5S+=NyH7!3j&-sbKoVknCr$^;$U>2Y;!qOzLj9C{j#U)kr~)QZ^&_UHM0dQ1cXcW!r`+LzEe2L0H-|++s>B^jhKKKgvT987<K0$-R)LCBt&&zHGj;T|UQX-%APnk2{~MZ9Xsn40S@#T-IYxa_-HX`9PFv-YP+4f}ytpMy`vU!jb?%a+>b$f@kgWeCbY?BLNPFLD2`{i^Itzw(Y|MCh)xj^t#JC&lHnrA_|4|AvYP3~Ga>c)IFyhWO$3MS8su57&KV*7jeWpr=`x}dW(-Xi0ekXp)(kOVmT$rz4FiIY7;o9|49WB*KL^YuN<*zt*h~pG|=M|L}nOqOHrUN>JX{W$RIkTk$6X~iHh(5`4h0RcnqDgqouh@d@QGZkU)hBB~8WoY8L_{lcSTOv}7Iv@nu%!usE1Vtn{}-cJoBLxv|MtN^P|<rPAn`XUKFS-o61<0(V+^R6xZ1w6Ng`VDwrJg_jV{Wd&Gjty$j^TjN_>hVr|$p$U_|ANP-GmEX7_%1g6>Y${FV<TI~ZLel&x9usi0E=!80V{cl~5|R%@hVT?tgX9>j<)A5g_!+NUoL6V|;WSQR{NSi0X6m!ch?uGRm`4kT{$Xb&nKY|D$@gMx(V6B*_FR=dJF994$reNiFsg>b(KcUs>w%QazZm98^R(gTUJwLX%CFwW|$YuNt}-c;{ik%jLL;-&n3v<!knXnuyM8?ry2u!W6&)>2sT{x?*blT3V<Km99=;H-z`IMRzIv{ZYt)4o$(6Tgeg=dnI)22mFZuM7Tp)ev0)&Ji?O@LtPP1HGN-r3{EV2JaI}nozxVuVNqeC1c=T&*{01p1bvZjxDTmgN`e3n3o^RI^OiD{%W}q=%>eHlrv0d%9bVI*v&yRVmA>Eza9ECSP+29F+*$m#SNH<W$g-eWwSpV#-lVuT3)G(;hq8Z1c(T!GM{l-B{PZH+Hn0jqH5fsBrg@?&=czE-K9(mBum3Y-27*mOo@>vbR!fY@!pHHxfA4b{~NMvKao@&0;8p_3zRCAQxyGM<5>VWJq?|wOP}%ydy}2|Hs6vXl9_OcHC<^X;%n7yN<7c+nfxMton;X4!`dn$@X@H9ZTmz>o65lcFX8pRVRTP9b}n$EZLt4@3!k4LT~p1byd6fga6}&f>e!eDdi;H?X?PkoMel$`N&PR?^NwAjwxt9TFgYi&x6|St)fEPx_Zv=p2BH^Zz%Kr=RZLvFH9~6M8tJ2NYqzI*t)>^FD6o<&s8oWaThdFijBH;d^~Awh`aI=OW!jr7;;q}0q<64QI1qrNqRghLeUb;z!xhaYOr+P}L|%N&Hqg+8xCjRGpq_}8?87sJ>ZwN8E{RPegDo&EJnJ4K7TBi!pC|Z-#AQ<?*C|IgNp)~TBpKZ@k<RZY9G*|nyi?MYxuVM89Lk5fv79^x8sWf_uVx5n0PKVeufL|mu^U}R=>-_8vJoU~b+OGZ3Pz%Xd=}k~T-~M>ds<%9DXx~LJ(%Iic_fVjPx*?I^a1wt6YFAS-2-j)ABh4>M=GuqW$*TciW)eh3^VP1a~jht$~HI%xeZmfzWMTOn{#(kM0#&jcLay=u#i<;@&DGg-u&W`Cq)@F;1ukkzQ8E--Ub|&Y8};iP<wN%KR=tO+bl3nlBhiao-Re?x;f^jFodIn2GL7{z2I>2lBZGDp#!RB-42564T;(_YWxcD+#HE_hWdqoUImL&&||$R19jH$Vjzq*R}-ehE~SSOUE$POjU%qb0A6F1QUryHwGO#^F|=r7N@~NUDP+rhlO01Shht;=Iy(HZei#gjuGxJ-*KY4Y5PZC!u3qzZvye1qXIU8oz-Iei{%7{rZZG5A;sV`Og&~7z>!<T0k_6)!SLW2AF%!$zhzzadFTIJTI`ufVXA{_bO^919hu|c9#u!$_O;C9OOA1eGwRZ~#1MWl&7ScUeNeSFgpZrymJ*AEeSgeY`0I3!uIlWZ66TfDZ^2}cX9l6v8%p1i<a}_2Ih6@^5wToQf?{_Fme}>T6##cpW7sqQM(i*(IY||T2?X(CR*Bw~8@i24bTW+?NmVc&NofTcoHAc6*UUz_Jb@rh%D~Tk~slzrPN=E5I&42boe!G_oyZmZERD^B*DjCNG!6+1TXKL1)6tn0!Rp64uB*>fwK+UicfKO}3Poy%lZ4l^qE-6JZOzthcU=<*={-Vm-kc7(Z@Mqz3S3#wE1nR_pcLtM=N_g<m$%@`O-*isH7Imox!u^R}8S`EdJbj7mQXjDiC_)h*-6^5-clpT36cn+?F>R9np_WGmJ>GL*X?booz+As{D^)fgco5fL7wG~e-iZR-SD@D>`B9IZj~_>1vQNQL79Pd;MF_UrvWRxE>~~;)Sn7SMPBl><O7;H!^v{?X43&*&@{74roSQd0KzJ-GBM77@<;QxgBj6#YP=#*(RAvuse86xXE(BWvnP-c3v{9UI2b7-{c_3s|YbxYN9R$_qO>v;El-ts83HTXPrfD2XS<P+!3*o<Tu_^xA+u5lQbwBkkCv|(CIE${trkTRc3TCL6u#a+f<V0Y%c#s?f+^(L(^L%Q>?SjXL46dtD5Y|&*g|~1zI~EXyE!;kC3eXV43MB-bn>8Z1CU{>ix_vadKXk>Lt{oVv^Zv$|r1wjaD(WFzbvIDYKdhI*6TJ>?k2W@ueL(sTcNvFt$u{v<3|e-xt5E#DKgJnk`lYS3VZ6&0D?P@(Pd=D0*`b`YI<JnbrDd<<2^#&T8FD5+sjr+|EuKD9uLwX=IciZ;+l0j?(JWB_?T}^t&{5@juk7i_Z=14SEbRA3J)Fs$NTrgQ-1Y*cvelz~%jJ>kY|V}l0~0~r51XJ;xVB=8@uSz)>R`tdmgb@aX#@-}_JMCp9Dpv9?t}O?%eoyU3urRRK%Edp68M{rE>(cNp(NNK%apQ%XSs|~F7@67V{jlX+%i&4HkFvpcQA4p0M3t+XTxL(!WR+0`660th@nUp!g%~neH4hSyRotadU4IhTPlZak56tgcm)t>3nSTG=;@c&WgFGlvtSuo_dCIE?{PJSwe(YXaM?bO49ottOCC2dT`3sw?sM@}QARF>FkJVC`=(_!ael@2Ymtcl&0D%pDwINbV0}EV<*PkIc1&8+ly{KQFmVyZo#NLUdm@HjALIi@xb4yIO`ojz%u-@F@IEouGBfVSPlbBC7QBTZf$s|Hq=p1N_Sl}~VtyF7Tt5FJ^7=ce;`VzL-bk$u(nPPKtp4EpXnJl0g@xrl@BaXMjj;jlVSH@aZbMsZ$ou5r{t1r^M~|kd#mn3wrl?Ys1fs7!WleoS1<-dM{&g3tzKAy1&CjPhiLsEqG7f4ODfS8v2MU_8j|)X>sQqH=nW)(X&eLNip>wY*j<0U8tgEWDzp*{#BUV`&G5EcOuZ3x!?v9*Fvk)b|@?8Vs+kpDAc{S9@p>x}lCHF98RTlMav5N%}^>kSfN*J31nxPyFB8@hO{5|MIw&AZ1$+%Q-s^lA%?i)l9jg8s0w_F@Nfif9Z+12GQuXbZ`cc^8uz#p^Sy>?^QfPf_a!O?+!mS!iqWR)>}I?U=?*$5LX5d?E&jw6ktqFTnO#Ls#1O)BNe#VP$-$`Z9_xR7L=ok_~~Bh0efnNQ}=qGq;SIJbpkNJ*|f>?gzIuvSrmgrhFoID+*_8QSi?J2u|Se*r$<ejh7a%BYFP$Gm&V8)5A6xw70c!X>Rba4yo$I|k6A{?+h_KYx~!t>*tF4??`odJadQOrnEKg*xsjWA{tmmUo~>>0H7&A=NL(m)3zgq(9C;rt8)ljh7^wu$NZH7CfF#aY@A6=`Vj>Dtk=y*{XPPEOu9v0cI(~3qO`>ruM?)_M9NH1JC6$|AGJNTO=c(6tr@B<es$Yiw1R_xb|A53;p_GG6-<B6s&p~=y_`oymEriiF=Sb%0;L^wx~R*&n0P4d}w8ftAWtAKrAa+5q7U(VzAki=}s_{`%yNM&@)!rh>R@^^F;tci4i+e6$-UhdE%FPT#Qzj86T;ha*Mcgm!U`e)8zRYAClHRJENttg>eX?&>|L_A5m!J)|p+rc1j;tDG6=km$vIA<{KW+TDwkN>(ZV9)2q1H3g}UF+TCmXR-F0|HYR&4cq6$tOKBneVBxQf4Vm3X?P}LB>ig&dJW*tKv~RerL?IFyb^El;9I~>5F36bS6>fKmm(Nl@K#|t@1_`}!9-}cxMh{`_yyC(8@Afr-XTE*awdZr)TM4u_89pqa1N$^$ED|!bG0q&&1}pT@nZQ{&bFbF|t8a$Re#C^Fo&d|XnB_)`00U#N-?nS`@JzDwGDZGoZs%R4($o-0doLi&VMIr|K)SeUdy5qHY)n6S3s~uG=q0jQ(DCT4i3O^Syn@Xu=uV*xTrnYYc!-q=JpczkBF&Xn1$TE*<)NRb2&F3cj_qO)2ku2{LxKH{oCElxb^hWV7*3q%`9>oK#v$2L)mp|Syka}p^Y9+6bA~#<2)I7-$VaSgvE~yl1fU2PkFVR<-Eyhz)(f|=Wd_u^3KLgRwq&Jmask=OPEpYx7Ma=1cI_g?R|?2Q+sm?PXg1Nm-JW>?59oV$N^fX6ROW+5(`nSZ#28|!)Hm|=UZ6kqvxBy%^)?-s25(R8p?HtuTveC+K0^Kt%HuJIKh_~<sSS9L3Ig7tt(WW1Pi2SY`Sj=476{YDG(w|a+55Tq16)73{)5yFRxQD2aL{SfZj91Vg!qPD(f9_%SSAUZIgF;Kr*REsKj|L13ebQ6Sz<1A9nFx?=;RVGS=}$=Jn&yNhF(t(Q^t<g&-;5oos5d#KZ$<1;}^#DtfVua>hyl3J@Ur&GJnuA7#2E2&hW=+Pg6Pb3tQ_0Y^Hm*kg#Ua<s%8y;s8EF#B61LcfY~BRYFX})i0&?JN2rlTPC}2|0?XOeh7}nQBq8+3ui-)Ymfex5QDc7|06Vbn{3Z(-PNx<jvf>?v0>m<M<d?@&6nP3ZxYF&LiABoke`HQp7PQ(qB=<g%CyX~laH?v*he1CWAHq3;hNZFITYJWKRcGOgVDe`=Se!sh~e8;`UoIAjx6)0HCz9ZaOkcac+t7~{ztI^cWz0w3M-d0$*h12g;e{YXlT-*OF{}^{v0Ee`&5cn0rG4%p+2ROI32*p{;4|b7n91|-%%4LhepbPUMU(LSvj;QhtG<>_))KxFw(HWqrWe_Pb@WjH7)H0%A6)u-{>54Dn@c*7OAB!sTNi@m5Ds=JuO__m0-YHegp4z4ExkO#uqqS?Z*pgb5hQC5c=5qSkeipS!GBAgr>t>%9+%U4Y3K_xM;3Lp8?p}It!hS{|@F$Z3?xh7-Ptv=KxXGWTM&wA1>?gN+=D)7l=WJ-+ClE9Q=Bi?s$peO0(y9pwEU${T6OC=y7gCbWnYsQ7Au&f9#2EY9Bh}UU2ae=VUd;R(><V0IFn-o=%1#?Zzt=0PVzB;Q6q6%A+WH)H5rx!nM9{3};akdqI{cA<2uHZZqJhPhOtKE`|fsUv$zfV?gEjbzFvw>v%x&lUbI65yg`}-gQmnmklv6Gk*%*k7|D;?dTj#K}iN&zcX>K_q(F!ah;mU;zze#EN4xp#WEL9ucrikYR%B#d@f57O5##Zb{Ki5Nn9P{zv#KkYHpKgbnGWyptkKfsj1~GXFReA;cv%%q?ILgp%!<H)oIwM=a1l;2Z+Pro?@$#<$)k<<A|%`giM_vDd=8_xBlxElyRFV72%a35z!+-#W#jB(|>(SsUf!!XS4D$#$ob%`FzVM`vE=%-`mfi$|CF`dLlc!&VIY#k4&D>Y<V;bmE5g~s^T&kZZOK1i7?K#&Sw-Zi|l-l%Bqv_^J(sAXPi|>T%hJ_<imEP-?DjFiz+}S5Kdj*@=S;FMz1s6{ofcf%f30M?b_}L+HAoVM!iaw`T*t=%4OrVbksQHbJMGFbVtk}i~t%oAbi?0PYx4`1Dcnod8}$wMK$^T_--;V#OM`vBD|O`8r`Yqa;-u7lA9PS_kzXvcGvDmE28qAdZYTS)jhFdk!fr88M($p*(GOL=P20cKAM`Aiy>26T}4nZ+SOrj!z$?_9UXu`@(HmPjV0a5!CO^iNF5=lMJ1kfn==%f_J5kR7DZ;AvAP>+zTAC`U9kGDZuXJ>^lA&8jE6xYY0=b97ST7i*O><@p4v*IKT`t*T_f;;iwH|v5G&=FA$acksL$F`VW%wML66&mYXFCkDe8q4$x#wBsa_mn;Q*>6Dxb;(Lm{Ri#rD?XZIxoVKRJp=EY_rYTbdw}FTk#c6mgT8&eOQJ+wQ(zXKaRv*|}z3-UCp+E`!hde9bQo7g1F~B^Z<C!4Z>Mu#bt#zgmNRXPEYjQU2_$-~#m49SpJ2w=LE6J<%v|KuaB!zbY@KDM#Gp?B9$w(h20c{x+8Xb9CjqJo&W82H~^ovLF38r#%wEhxj8kB2r!!Ak+@=d`e;2WbVP|xqGbaejzx}^ZE#=n1Nkb9-6g%InS}3w>=k+4rwJb?-o11Gu{SvZrd=Y64BJ=qimo&ZVex2_Bm8uVTF{Vn-ZiCkaEs2SfJ?&q5bJABi6gg<ya3Mv3(&nhXWC+3;8oGWm+Y;%PEgC(SLc}>B-mN#~Vw~0igE?;J`DMZ0o(Iz8=q~#(p&#k9_V4+tDElg2i1AHc`uknG+%Vx1$j<*P1aWS*s)`kcsli1$I$q8GTc}ec+Rw5H`j=MC{W@-_As~Y<5&^xS8r%57yh#-1&#_L7$KJJT1zb+LZ*`+*R*GyKUsZJxVFzP8H&t;e)2cw@Ry$Wb0|9F%%IQK)EoI5*DG(1{H0;$O*w>;eUN_U_UnGDzxTd^*uSEcnLih+Nj>u8d;3TwM#k?8h>xH_9-vOx9k4#qn{f*VZ9pwLnPJ+@oH2dCfO6pUV86vm9E|#w!GO6*II(!@p~ot@$j38ILguK!s!(y`2D0IXQIZ_u|FeKLxJB_HVD#zX3Sq|de)Hx8mmeoDuzYeyq<d4^<&#~+%k%G3hop%fi`uKR1z>7ZsC9z5m6+3iDgr?sICVwE|f_DAXGH4abI2+QZ6E|%zzaX;X)?v_GTHtiQLs-0l5{b4jK}u=b~k)Y#}x8xO*_AL1q{A{taRJ>}WWXxjixUaqBGFM_WFP&TKeio?x9|-QQ_3v&U{i4rbA90^<C0{cPe&y+N4xRRgLfJd=@=Xb!ThC3ZH4k1Z40z4Lw-@KHmP#S7fWutCY|hd0iuCJ=8w=70?*q~)m&!dd%eHNkfxZ_4DFo|4%bZUA>jQd#m-wz5be9XV`ZHZ6sryutT-gW>H5EHBay(^<q&Ob$3e;aBA-V{+eSO@z;gUj6(UVH$V1C=l^j*5pkp=2YqtmO}RehpLkps!LSx-}O8EOd7-Q@_k9&m2(W><fg>Re1cQ?dm>3t-;bIo&ms4Mnr1iUumzsC&FvW`(yY)RNog>3hWxY+x<NCdj9;4SgYxtiyjKbCP!+%gII1g|#RG<eutIaL#C43{pbTkNJ+1a9YbdiX8Apw+h^FU&i<-(FB`a^V1c^-f#@qe`WJGaZ{$Wkeq!G}J%rzw5n~ak^`3q{0C3@HiT2AnF%%mZD#8+rQYmpn@Nt@lniKG#ZsUo@7$5P%XF*<xi+<TFnTqq@{tQpdOg>2p=M!RzNgzibtOjy>sI|WVsZ8e>YFSHh;$xxganFA>eB=g;!KiIsMi&uOj9p$SHQ4B88mKW`fhj(E$m%|ztHd(9h|C+}(KFsLJm(`xJU!WxpL-_zXzY9P2u=*%Og)B^u!3fy#A-P(EJn}SF;jUUzVY-AO0NH~`6sOmfpXJ6u?Qe!ri`Lsd9`KxHoB3eq@vZTr+S!fG+x^}u8QC6v((nCjQmWwp!x{jbBc0KR{9`XYvSLp+3xS3@JU<5L?sCbhG-v%Kahx^w0#X3zvvn3Z*8&lO=mUK`=anhlh{Z>x7#}=xZ?faJg){?D-6k+WP}jAWaT1U`&e`u6;>SJSVCz99DX^UH858=tJnmpxzmN;%rXwOk3&wi_><nav!ac?iHO=Gv*gs5`R*N+UOy4gDgVCo?jx?<|f@q=d-yPG!T%7@Oui~acCxtOmq5)WKfaT(dnS&WL2(+6XP{8Ob`Y>YR3&Izjv{s3WuE+<P#+Rtk(0mHFcv^fcgpvr_8w?TztVdIXzmnfGA5cC1*N%c%Z=dth(qf<grEj@F%fjbItqOIYF-1!xjFtNz*O5xqi-MynT?StBM*GjhlgcA3X*7!ThQlh~O45HwB(aoy=BBhni&aYUl=%Rx-p2m-oKWE~$C$rkxc`wSum!^B!vJyr2-@*xzdUo6jnnTH(2svDw$1k#P10dFfdIph5s@cn9RgqJE|#t+0umn0k~=%cpcSPSLc`VvaUeaS#+@Q&Mu1SM0yCvkCU&Qed%Zo*IC#U#l*^s1QTnWVGK6~=MZn8oZ|PSsRJao<v;AB~Cy2i0j6q5ZHD2)(+u0fw6uK7VAM+Mvt9Jnt!C9^oF*!KJU2EOu&@(|gcPTY;ZsgZeq1;%X$S!N|PU?~RigR5zmx&8FUv0WpI}%V6R+p0-dg4jYH&B1Y06VuRZ?|=)0H_Td=4dL4x{wS=^oIQuyw1BIp}<F2V=W~h)+C7<PybcS5`&i%Ka2Y=&?&Fv=N#{WM-WPcqpc^`2_oRV7Vid9|C|eXtL7%I1&%ZyBQ$k}5#OJD=L7E=D%kL8pdFuGZ@HY4ST=YLlo6N20ZYmTV3#L0LeEsd`aGGp(|dFF$e#;eM@Z|ZmxZ6aDB;b-VELiqRn@$S;$eD=SUOh+=$xpe4vcLCNnA9z-(Lm#E?(KjFonQOq_<+$i4S?UUG7VWerC6+oS7S%`{I};=<c(Mu=pttm;{^iT1_tZc7$nYHE0SYJcGP0Fp!;z6RHRZ76WxWtPEBCo<Om2i&bVD3mT0ShHs;iHYORQ>z$m7KyQU^h(q<2ra1~Jg2#L9Nsv7T|AUw6%I^tD(dbY(JFa9rZ9z`f2elr+_oU}SL!^G-P%YRl{LQguOvTQ^kU(JulibdkC~VH}1We!jxraZlr!2xh3c=39?DnGcFFZewx4j1E3^!mnX1Q(}AbZ6Z9z&T~;RN>N0*dSYeQ#)X`sorv3W-Wka^xig!LNhaNpVB2w0MmLmXDh6-TBn907OJp?r?O}S7r|B`KeaE#hwuZm?ASu6W)Qds5Lw$o6Cd5)BbTC%^Hwe%1f%5hAyLd9wCRLw;~~gN?S_eFBSv|DMsTrm5$i*ed?@1T`JUs=QTw)xZ{pJ?`u;<k^7*V)sCRHNnh|pR&G(|>JRg*{9pMUJ0k&!-}RG6mpWWVn#Bd@OZrnL#^2}xb^P>2{{6j7i^p(G>E&J97U;5uc#&fR^fj*wR{BAZrfM|AU=Ai2Y--!}R;r+;y5U7%8kh8~-x5JDVD&%jrm<v2<d$A?&rWL8sXt#J?w`aG0BSc2cl1^Ws%Jko_ULKAUmnSw876F#*ULIXZ|emLl3v!3l;W6-BN~|4wc-0<tKU+lbJCm{i5~KMO+55!`UvG<;TXeAkqp^s25sgf>BKlZSa=uB1f1eX+gDJHxtMYhIhBe`4|iKyUF{X6_Kai-ua=IcJNk)<_%rC4B<M%+^Nmep^IU()RzZ?vlG@Ry_x}N5Ae((G0ZJQqDX{ZCjc`=7r$tdJD7{GE^5~y^LkKks=1aVvl7wCskY4i+Eu!=WByD1Jg>iSw{uRwxJ^j0zeBO2sTj^WrY9nAfJe7l>>>*bBrur;kI-W2X<{`;N(-7n|Y_OR`l9KDm;v>Dv=7U~KKSu<xRQN-yt{~+HXMeFDc4k}K*f{)pIW&#g1W|2B*$5kcmHMtt^vhDSY+Y0eCci*_>bXhg`rX=!DiO$vaP9uH=xbICzRE#$JOBdF2ZFaqQ^vrUX_R<3<uiL-;Mx8;$~V!p+;>C4x|i_BW$3vIEEsx+Wy0wlLTI?4=^s~^u@xRU_&C3015TX!IcjY{DI7{6HcPUJ=mv`ZpT`KJ7ezCTiA@f$i}H;g{wV9TPks*q?p|ZrY2M=n?oBWzxBTw9UinL?8JivmtQ44!E!3B%p&fnuFE;7-lo$=SzE{*5HPvp?3Mm79>zD+68J)d7BzZ=_{nfXPw^xe8*~M(E0^+2n6}WYY1N^?ybH9eg^rZ6?A@?3xFdxNVPvpach$JLpJ(yS$v*>G86t3HXAI}-9!%ApN|Dk@!H=J*|!5RY%RHs-ru_P({c9tMQc_vPMqJ;%rl8v3iDNDI18_-b%p4nh^Fiax?_nbp<Dv!bvPcFp_vc|&2v`}j&vMdL&A5uiNKcf!oa;NXlJMg0L*0f(Fo<!#jo)ueDa+Qtyz4S`;-iU3b(V^vtmY{jJazU~%fN?xQ9V-86MbK{XUi{QtIa5<;S_XR3JmC75H2La;W)J$k0@*{uWi8qPPdEHLgkG-MZ|60<oqb}oV**Tp_8O{?Vlh6rY+qqL|5DBLMq;~FVlY$c?v`{$x~)<J0vnl8+wAEsUB^cDDU6}llO7A+`Qqe^%Casn1y=tv;MD`?DhsZBH+!tZ5|b6Pyw#j+Lnf8kS699Oc48uo982VhSL)S*?%ILTe-ga`G=qSj0y=n=1G1kc7I_K{j*)D|1b+wY30D(&9U(G3>i1h1KsXrkj_qUe(ak23oMZVZ;pg6sIgKZ3TG8cCA#uRjF_~Fh)P3nl6A_xZ!t3hH%&a%4z<5FadRATDTMvi;tP^p9)HRdEOll+^?IUfOgP*_|okrk}1k6Bgid4jwTx{@hljztpN)bA3Z|yJ-`9(<vw6$VM<!w4Kl!V*Rv-n#n`IB|uKYE|a47I?)F_{6?Hg^$BuDZr9raoxkVZpzDr0$G{*sp^T5*DsgaMp4JV{6_i?mzgda_S(R^=`fZ;mg+AK+yg!n;VU@C}2J-V4`tXvZV}Gbf<v5C931yfC9K_id0QeCuiT%W}T+diytx*u)LUMILNM5WOkJ$zfF3x+~Te@m}DE?>!?^Er~E4w(cuR_I30$i_K#RE>40R(-f$vzb>1ew>5><%O|z8q4Bia*5QNp0|6YA=>M&+r_S{d@cP1ur`nMdYfVkuqf?6=NTMOuBWk+e#T+`OH{#3EU$|^4yp^HweTbvdNX^74g%&t8STuWn9kA#w+wbQO8c5SZ0@fk!5FVM699P`2O%}8C{F2c%=M#n%SI|b|+nkau7cPqw|bveBc<rOO+2RO1<N43ane=pzGSy%CJ!ve2|FUaKT-KGr<IW|yU%s01H-a>Ze*NihuG-31^*3oVx%t64tFWbUBVIpekdUnD#=Q2dj-z_^F6rnbp!H+#XARN!JIRepd_VLCI@fzq8!lM}VbCC$d^&;jBWcxKuD)itDS*7RTTd155kJb{8U1zJF5R|+*kFyI>+7_Y|N}(+9CY7A13J2xppqw~D4HZG%KUuu(3e5s#|87$ZR}155N+S*7A*`eUPiML64mQ*`QWOdCz1lw>K`Z5NMo3HFWsUAdU%xOT0G4`??Qj;pNus>hfMQDl6F8RT^eF9#>k;ewiOtp<9<Bq;jWp|=I*NplPW&=-Jp)FBPPHHjR6Q=I{(r2j1P}12HZnyr=HDZ^mavivit~T*iV}T*J}(btGXa_2ASc1V{BMb8Uddc~J`BYCUvL1XE8Sel*4B3KGEWxZ7F3?%wSy}~m`PImEMeJ9uyy%&L1Y2^eYdsrf*+v?bc?7eG$g-z>5WsQ*@~mYstneEARRR~I6;McsmHwvIt86?z|%?Ejo8w$cQ+8&5U|5SjdY<lIsrJ3Ro=S@?b0dMsKPzVN0E8z?Ajp<!m6t4eSl@68)@vu*xgI%-001+KQ|ahWy3~ATET750U*;53*^{LkX}04aQYwt7TC5qA~5Qgc$%O8f^71vtNxqc6Q4`*Wk2?2#z600iGMSGOD@)0vI~CuE)9T|CT)^y!cN$IACfmnO6T=ZE}@}Ikaxy#RfHh3Q?tPrG{_qPDCliz{ETztD$8iX3#HhUwad3vb*&o{vkjE&i!oGOX}TO5x&A!^J0DK8iA}A&2jt7^7V1AA@bHdOuc3>z-wnEI8gYnXp5ZcblebeW-%P^7EQt0H1SB%i7|0>nv78mITy@hfy6)2T*pj7rEc45^u}}<RraQNPoYAH9Dvn?7{KaA-9<IJ4xsiqTsOm1I6<cxZ=Y=-MVmV1{?jW|5t2EaW<MP-zu}M!Y^q7hoJ_g3GL3LwQ%d!|98!jQi48g6jBF)pjH;~z@s97b6hwM)(R{mCeaA**9)NMC_`fC&kBIE_y#ssW@u(1-bf9Omd+`Xl3p3F-lXLs>mbdRF^*1}h>7~1dqI8siRn;O`;`BE!-lG!x0f6**L)2V60bJ@Y^FW8dGIxxiQRCk5Eekn4pkS7r!i)uR0rTYbklBCaMRVJSvH;0jOCUU5Hg7R}X_rc6X@J|&+sF%9)q9d^#vI<BV*yBa!HV2+{_P~_GHP$(*;m118>$^+X_UOpRfW&W?83#-!iA@Yd*C=-}T=zq<#Jx5P<AL>)3HEUqsLiHi&iKLk!3KE_N)F2_Gz95PB;v~8fkRG$$UYrWK%eAwy2d?jN(+8KA4}}#ObHvPP}v_;JJ#Aq(rgg!>3~?xb->9WMFVEzrbG;i%f<=_=0u50vvt<lHcfbZ{1gd1K%6E(&C)$|U*V~6vLy5)p~6HOqJ)csdiq;3%W#4@MSA?t)j_U9pPK&Jl}I>}YOq7n6JVK>1jK(&rPZt~vsIhPKbh}a9WeE314sLmi!kayZ>k(I#$QLaWp8&!H8+C(B=A9YeDOesx?|NYg8%(bUJB4L1Bo`J2Sr8tMR!T=M&V6^Rvmmk{<(b6qrG8vDK(OtF4$~LnV0Zb4-XN4BiTQh_T+6NJtoLF1i8XKMPVg$I<%9Wo^<EYU#)xfasvOGE;0H9oOh9sNs3#9fE>^Rd`lrCW94m79O%<b=*n7%aiRw|W5rt5dWImo00e$VP(gPw*Y1;wh!Wyhk^>vz#F#OMc6RDH+Q8*&D7!904(`$MdHfOy-@p65JJb>)Q>poIx^xyDe$p|6t%rnkZiD2-7S$%|*=QcokKc;LU07PZW|Mm}PF+YBWQ<rw(Qx#-s-zBM8p0J>h>&V;qnFQlVwj+1waJf;WFx!Gf7<h&C4&YE(g!P8Iy}u3pEq!8z>qU0!s1quTM<#hxqWuAm%?E_x@GbZR8UIl9`Sz`{j!(e1NF3I1JQogW_Y?`F@sLQRqb2elFbhH|LmToYVsM_@CnViyiKNT96Ifn8CfM4ky#0M9tRV4A+@V@a1<2#=W-288qy;rR?3{Fk!lG>@l<kA<+ff|z2;BMml&(PuY0kt+jet+BkTY+RC#I3)aVbG!@tb*FXv@0Ay3WzN@LwH|L#_Y?>OE&RUJx2!8d@3_vG}cSA`ath-8*Tz+yy_imaC>d*|oomVbgDsD)b@FvuqGW|D@e>sY6%nAtfB82}=WQP^+^bb-vEd)cSp1UqBnCo<lSWLcl?R5DT4+EcM3!*&KL-rWK8pYGs#xsqvbRDSdb>u8Z~r62>-J6?kKT-K=Ai=Fi0*&?4Z_E;~V(<n$0DWc;Y&f`=Tj<@J7hJ;<5RlCtEw!w_-_4*q!F5b84Dx%WMc-}Cn(u|jSfJtqJ?b-R&M#e8dfKHscRs8tXc;AyLxBm8}#l(J;C|CN~6W~2v9#8dk5CAWiu*UaE$UYwA>}*!nro8Q(wlzxsIB*_rfjz&D2)7<3S-*v!NH=%nVHY_4{Kkv;1sk1Ok6{zYkZsQz=G=KWds*(H6Rhs~&u)F)P!47}oum@fR;n_@In-IQ5ST4R^&mPHvsb18aJ7Um|FZIu6LGT^;k4?>EZ26ZUBu639u9@iioweltQ#P3>Xx%UtU+rqBCb-r94N)*7g?ADP>O+DxR)70H0)3VFC2A_)K*u4)1BS=x&x%=L8ZIc>FNM*q<tSB7wcnR>kb^_@@S^2!e<B_sThOWj<)irNfY)g+{?JjD54IOWi+_F3*_`C3AwUj05G6(LAUtm?iNTtV+}caJs!7vC8z*afHwxN{MWG6kp_+l7Ql1M;=DbN_?RO>jNVxB9bi)o_!8U>!|WCMnrG#{PJ-!}shBQbHiH9KZnuzl_s&wh;-n$X*KX;UD}!z_vEGi6G1&W0m27<<>fez9*r1L%>Sl+lbR=o~{p9(*pCRLt1-0;AZte$og~076$Y&3+5PWDU2Cy3AD9V;tkU9$h`)dBv<<ewdF{{1u@8a28x?L-v{Y&41+eLzKlVcDY6*m~Mb|`)FR4*t>C!q9ogAxiYJWyK>T%5gfr(&;yowsR>4g$uf=ZD3SjfFl^E`uqql6%4<6a4cCtx$VCFh$KojM$*URI2*VL|M+;AE%b?(#FK(`v;ni##$xv-!9!oVg-_2l1$Nof0$Gk@RT2Y!BfGnr{)ba+OYVwYGkdxM#vqVviqovMR0X;_mWa!BM$;DaIUf2S#wd0rnts*tNdkE&~0kDhJzC}d3`==^3L9`^lPEB53Qx1HQfi2E1;_{uCDRUmr24Q(2h-<+596^3n5Uk>G5_edWS=vw<llwKh1^<P@2lNBC9OFDLg~|Ljv&F+>T@L>@&qPDnDAF^B7w_nEIR0xC;%!a+$1<s>u7wWA?aJyssd?`eAKvu!|avc-sytjR@U|a-j8d)HA2wlz0T0^RZ9f2xZ(|4hbP*Sdt!IWi8kZ#XE`bswG+%2N(0g#oI;S5iin7tE+_50C&X*{y~GHh9=+Q<vl$i7Ze4B1lNC0u);yX`^5d@)lN9SEEEu}BJ8%RGY)3R&&W;o)0kSb*H^^An^PPQLEL7tx~$L@^&*$SNQHJaD5w(WB4D05vJCk%_?bqEmoGTXDrUCPr%_8eYmGZLA!JMNvVM0!U~^CTsr5SHhsK8tdOhjY!EG&jf3f-(nzT<VZXSIZ&?5n?Fo+KAhNEx@vhCx*Ar4s4|A$Dm3qfH2(<fb&E9-G7K)NK3N2IsYaBz0S{$^I&eMf1k>SWTwm!}?a9yF~>2CU6*bir4A&zGk)?zz}!!040Vl6&XBuvI;kmiJoE{KAdcN*)erwdN;oCu?MkBUWJo$s6AvaGzcLFbgk?|1xr+`fewHS{!iI+O~Kz_^Wb28Wlt~S_KPlnsXduP2-6^R6%~8V<LoAV*^XXi0wD=8I%E`NgOo2{S9L=eXd8Kv1so-S8Dc}BHQD-T|jR^@G4d(ghn85so|#b!ZE2>&AP9&-!`p8Z9x(kg?4Sbe?51X*F*32VpWWK34GwcQ=1Sy;v%>5?$0E3xNE6fdWe2taR&mH4Nb$=-hQ6mc^fBI-9BA&R{MGVRc2&Fu+|c<07x@}i8o?Kw0{j7y%-F!T1>K1JgMW5Ob^P38rX1o0FwRjb7`J&r-1?+;2tWt9e0<!<;<^a^mW$vG=X0>!xL1jjNt>xJCLX@9PPps)TwG1YOq{H_WBb1KBpde3EI_j9VHsp8JO?_eX@NT0NPbEz$?J*T*O6Ko5R^`vv+X70VLjHxwo}^lgb9*UW>vYzj<8CnwPVi%d_pBTQUiJh7c{@7HQL;3o;!Tt-Q-eeUN|k<7%02?|6VLbIMK&7edCMDw)LA^kHHoS^NmI(IZTX6eoO8f3RU3Wt1)zH2I}KcaC5@tLHjtMLWwhm-EgpOdi%f{j>7AX?c^aGLzonY#aBirAS^;Vi*}m!)@ylzktO<#sFc6aGo`4GV+!HiovHwheZj=Vf6!%j!%AK?K6FxmN0t2Tm7k7WrxagEI{P)ho4wzV}LI1H5?TiCfG?A^1VAL$K*Il5mi;-k?H*KQH&T&>4d+-MUbIV*3q|;LwgPXh9rHFfVL_SMT@Y^zHAO#zw2@_N2eSoT!g~zFQPl0axlbWO&1B={Anxa_&S5B5)fqbaa>&Zd*suYx7?K=K;9sCS&5_xw5U?hXkWv`*FjxPs7=F7>pg#I0Z@rgJ@$QjWYZ-xyO9Yar+6_W#<dT`8<402Q3qQY6f`dHp$AXVN19RLhtfGB&pcR7yhf%NAT(_yet#Fj%u8lHw5S%izlzS=GUcqR_{el=2l;FJWXom}Z?h8cBw=&)L0q*rH>2QF|6_#n+G;3HS@{c1`I1Q2Zv4-yresF}QDrJI(hAP@1XlpV4e5$$-h&$YvH>$!=WtI>$OKIo0LhKx*U+o{uJhJ$LFY$K35B(NR?&K-%tfV=kx+$5crek?N~69g^55pIWB@(U%`K6My5oXCycFp2pp5`5wUq<5B{hdFVa;@)&rG99f~c#x&OEz8JeM2Q_vv??B@<5!5oi`5(X43qhZI>&HOES|uj$g`;!#)@ZWhr7T9(-Px&_BVBUMV^Povqy)8;i{E9lczgxE=alAb7-pvF3sgu^qRJM^5o%$0bq{Y}|Xr!j2_h<)%lmD+FjIkO^L{Fg=-1rAEbIW}i8cibg2Hz~S6t3iWfFCVR;7@QbP%?H)A)n76cd4}ArzfBDy{D41*9*t@{?!OLx9*VJ8>?`^XvnH*?C?;%>z@H!XGW<Dg_@5K}`l-sut}02@z?JIZ!AJ<R34&aXg)$F3_wte;I&J;AYh`v)>Xd4XT$i!Ci`mvZfRQ9V>CikPh(PxJ@2(OOSRNqkdHyLxhNPE*P5csbaD%NJ){zJ_XL4nWGo{^NY>w&@wtwX2G|K|_$6`Hz8}L7$Ymhah{zT=4#kLb7bk@ceo`vzx$8dqGTbcAZm@LStWr(atM>37>NzMTA<OSGGO$HeD>cSScH?#AIQ(`X489q@&F)=%0Rz^CX>X+ey+;?$EBE2cU97FsT8?t@he|Ic7V<bmhk5?fN%<i5f26`a=erK27rjXs_T`K2zddBEUY0=|#=Yl9$7-A9Pr*3BQqg3(O?8{I+Y#U=`DGTx4#9@L62z6nEnw})YRIjzWNfAQ}PH2xuSh~nEN&_0LyCPACIy-5pKa8zo54xuKu9$ochWjPO?_GTP7HKx$DGR59@}>EefcHj=DCG;;CxmmqkcaP!0bCGHVaqPEmc^0-QEPZ~{W=5KikpPRh8L0#;-;$DR(vvo9|^7AGOC;m4h=!-iNW);PM{tFEygBsj^`RxX3*=qIrcz&$V7zCgnme10`?ky1o4;*)6<fZJDhiKj7hT=EY^B<?GaPXK>TZF8#Dx1Jt#9NJ_7z>nB6B`8FC1ta=BoY3sNJa--8evsETbJZ6P%nTI+Di7#D|<eg!?J-UlTi24~w4YoeQ6vF|3Tvz7AT6%AfD=yyAV2bpHqN_INMLp=9d>}Mi+?iUs46EY=*p<7YzK9L;qT<&3X4I`<mO%|)4S1ROZ+Xurfp#I7_w&_xAO$x;w9jrs2Tx1>#2n|K%a@2H$&+2Xc+hI!#?MPFtv~fmE0ck&Dr7X<vGxM^yqIZp7J}6GIdjX}lYBq%Q#FvkAU+Wl_=Z^{x0zhhG@9ChVvQtVNOh6#PecRUdgX6uT-C>z-%f0(ZJ$Kc8-!yyA7tukE8qOZ7HRpA%q%>V=^Ukj{^4t-w9`wL@OQZ*xDaCWEMk+g5@kv1Dsnl+Q>}5F<Yc8lIfBleDPHTUTOzj5>bZK=YZj1=;vLzb<S>#L(v@z2mA*6WR>Fjk%CZ#Kz(6xN<{$B(tCA26%0rZUGb%G^WUh*yz70{VBqZcT{Ki9r~$0?h-w;AJVh0t~5n+GaRs{)4ot_X6@L(s4n8~Em2*Z0FiLbjm}sZ7e2Bx1Y}SU>HJnYE171zfkHn}rIw$`CnHx_-~fanW_&_>`9-GIsW6Z!3hTKj!wOdk*8bh)#e&{$1UNXW=Ib+qtlC;|yLnb#T9?Z{@D#amhBzk)pGDW#S!KnDXA3rzF%c@1O9Fg;)`2No}yo`VgQ|SPs|44IgbTjoHpo9_yQ6g&54CiKE~{^cdZ>f_$ah`HkP55-nk+A<N~#CzmhhJ3`!`Xg)TWg}lvIk`Z<e${K`E)tb9olTL*@$~%PSx*mc|Iw0hi#b}$10Z<I3E4QjfgtQ}Ul&RC5Qs%XB131f_7eBYT7h7?Vle0z{=I<pDNb%l*-5PF5>`nOIms|kAjdf?DYkW@Nw=LiH|Deg(KYYGnxo;0AFZ!FH&aYqP1;5g}CW1sg`vxZ$S4)CLO%I}(XJdc*QHewk+&1uVgBl{^u&0vW+=Ud^PZlc$*2Ho+BAhoN)hjdEehLI2WBwKyVZFO!Fg?d5(1W;9xo|1fY9fyU)c)QI@<$OOy<@QsXl|eMtFK7t9E~iv*TS1Lth(L8flA|2FQkMwNPvaWzpn-p_DNc6-VykJEm+E5;vz2vmcxQ<6y7{-K9e|@o{I!<q%jf2k{xQUiCwkumEKr7tH_a>N1*4;bAy{e$2cNwM-O9`k9dY7_%N;_F|$MO)09hXS$wNt`R;PLT_a<N1Xcn5ACK(=gZo`s{*J9J0Fs#n246^Bhx=~aBAI+K97xqP^-NDqy$4d%T)ON*X{H}kwMK6v2^bjcX$=55@Y4wQxi<;SvjVD#isCcOe8dfwH@1puWr-(wo%6L{A%wN-3iT(e+Llyi9~*j2A?XjBjLhtLr%UPn;?%^nBk@RVQNY&6P@Q`BIMre3WUOI=8tZ73bS~i1zov{?J-kcLeFxk>q=K~Q;dj(`4A;Ly0f_(BrMeVWzSOfWC>|H<e5?B$%OL}FM;UMaXz8bGf^GB{PdJG}oum-k{ljBpB-$}$9<b*N(Hi>CcIT8OP|i{NF7%s+Ae1662%!*LRn4@N<LMs3tO+<+TL)&b_-*y$H&%zgHh#}rH6!hgU%l0gJyP$hp1w|MMd8vot7?<7RM8{o@lD8NY(iRDGMzlW--cQ_2^7lg!7;yse~ipQX^CWLh!m2`xdT!LCmUq84Si&GiUWexJ_D5<GEaZ(9fMOg4m6HxFz*X-;J$2(5)X+jEl@shs}YJ_iUP~l?>je>ur#|rGZaa6zZ?!T9$R5KDDev^i0$7r`ZXA+yc&awpc8@7d3-NQTw`zyE(gpJ^W+DD?aTqEu;11=*esl8lG8vQ)DI&*!Svley#FWr?8)9DqEb$4H>QU!#F~5fl@S?^rA>eKmk_k)wFdy`cFWCuu4=vR5F3XRPQ4E#6uIU4z!#XML77E3%KO_zH++kjG0~v1QFf93bQ=os?5@3?S9d=cpZ@~fo(g;5&T0NH5s~eSEhFt3@axaiAlLKLZiL}6?a<Td4UI0@-E&G{E2A7`X}!zRt{r|O+9L$0r>QJeqO>x`-*Y&8oXUg{1vyQ1d+R}E94+&@k@aM}`qwJfKfp@6>oFG6^{A+ez|B<z%jmIj3U4ty%Qq7x;D{uJ;DAW8<e>#D=q!@-s*fzotAr!~59Z#1u>i(JXNtoJ-k*Oa>w(Es=(g`yE)7>@j4GSOA6%nDo<nJE2@*XlgL3>QPqOZS2}2%<Q4KqhQPQ5Hu(H-`Z^ia5r6A`{rpsaKUsWR416|TSJofdFWS{yJ)eUI3#CC`Dv*$On67Wf?2KWpF7~c@p6RNY4n|x#PCvj6KOl+;9U%oIFw?+Yk$BC228cQ~$wzD5?ZaKb+|FDmF^^v}qcp97Gl?fM`uPTDaTgLN+6Y=1miy&k4F;I5zA`~g7fNa8oPtKqCg2T;wu2xM%DpMaR0*T;SOly+TUf5WWFu}boE`cvO(IP#Q>YGz+ErfYnMfBw#o|{0<su98A!x9vEXN}8Nn@iwcSu(+{0Mw%({_VTs{kcHX=oBkBQld=HvUjR+OFE870ALl?5Q8KK-^_sLa5tOr`3aF8u1DrU3`PB0aw6_t)wTZs%OJ?NvSqoG6bfPWq+-e}?C=rXd0+BE>xF{%NHThEq%|KxKa@jM_t?*8%$A+u#=x1cDs+0wdm21?Pv<|e%GSE?YOxbKP|%Ek>gN_(iC@;c;k2Wa5M+Pm;gV>ViYh>g1C18FlRD|&$jf#iOzf)Extm5biA^BX<i*73E^w!(8d#}IexIWhW)}MB4%8Kk;WJyW5BjTh+qkRQ?a1QbTXKVlUTwO9cPO_?y91*EEh-wV*R~?yTg3Xh^WgpWOj9Jt5_iqkbV-)=Gd02)-gz(L2{X_svM<*Ee_9{(;(#>5STK_${b#ee%wsO}>EcK6<u@i&nvS*%BXccQUfeJX#SG5*9-QEZn|leyT?=WAo~>dB3hnJS4ZoU?wg?AE&EHeKYhNREK8!km7EpXL*{;%+LMy;Kj-rP&9O`*pIlrwAw$pHj;|y90Kcz7nRF8ks23;;>4Swb8WC9l&Vq5u_<OY|9B>|zTakmWUPqc)v8NO>0CSs}@gHoYc7nn7~kNxuPUHqw?)wy@kK@Rd08anb39*r~ru*m_>q#FEeFp6e#5qmrm<H~c9-Vj&*W$Ve;jNusX;!%N&icQ}40lKtdq0}vvdKdrfv_44k?YxyXSpv;RinqVP(Y;tJA@3*8IZr|ydnM<CWPL6bED*8U+~C{X86j=7h1L#LmpA-;VDn+jYFrmd5TMSHmu8zVRt$zEl5$Vth>{tR%rRue!zjA5UR(n*fo~f3MTY}-*PxW_mZ=MHhU6k)g0?W9z|MTsy60q&KJckgqERF4IKtK#@qm6WknE2De*aSU;U)dMgP`FktY>Jlh-Z>W^*q%%LQKhxC|O#UPfh}8`3wt=+19eKufg3rQc492J1*0+5(Cmd*6ihqQlA<t>zs6+?7650ZVIeU`SqBJH-GFIZiK>#6ppb%_eBU7_RtyzVfy5q3X1q|P)(m;!**BH085awds@mBclHXd@#){;YRo;m50xpn)Qvk-_u+Pjb$_vS$9^*W@?V-+daXl4<FgIiNG<=VI8aaaqvn!mRiyymd~b<gxyf!`{89x4Xs8>fgt1>F%>%)wLMcouIxPx6xI&il(;q-`B<v)z>+oZ;9*w}P6THGIj#2EY{855&dHm9>{1^m&tN}CMpL4pyV~iz4vx-xxoivS(zpfz*G7(>(8_LmUB5BXPmCR#l234moC0XS;J+w&q19XoVq|npZc6KcB1S-|?B5kZM4?f0eqpzJt)qOKsk2X=6{OwJxw`{A^Aeq;{^vszIeTu1SJD%6hlo6?|O%4o)Xi)UgyVq-T0=X~FU_&<t)^jDaZW*P{>%U*;1&EDnr!*@w2x!c^O|#LG+x&uK;mtt;6fL`aHRQzh>X9)|gqxMtQNc)fKw>7L+Eo{a`_qD@5GpH$R2lI9oF^fa_EH~PA`EuN!#=V;fBow=gbe&pDj!?9ZUs3cbr?<ZR7Iqh8nTdj2GEzp`)y($n8AU&ZB1ND#Lc<ZNJ@ugQ781~$&~wdY6n&A8Fc!Tb;qM+$DFRtm7APKsf9_TKA~faU>^;7?<L9Wn}bNwzo!_^tPK~FHJv{_SSJbzKdEnKj+)GN8toVi<ov_H-a0y%l_s?*&FiML#_(V^8E{7}CG{ZLZn0iT)}<2lkle@TOiDZxO?TM2uFug0oxOH+%jgcWpe)_JPdxZU_>j1rxDNpo4<nP<G4#z`!%e`4ICPe=bBCy9dbCML^F%&dUswaJ(jXfu7BiF0kJ*Dh^HV`xBIrJ>DT+xT{rqM-q!85V2|P;Ya`p169P`1S?d4j2F|cgL-IDi1=y-Rg|BM@&2mPD*Q`459?sRB$$Y_fPF;Py{<gg1`%7^l39)40Sti}EJF$Zl-clL*8AS`F)&|tEJGoq$uB|sDe(Q)wcrVadaw}6`1Lr200zfQGa#8~PN?RW%NH9)unFYzOUw&AaK=)>o3j`l~pWz$HMq?Y|bzO+EAJ%LS26bg24WCE{EZcY*tE;;kkk~Foj=5(4NrDZ=4)Q2oB#JNsM@}3=El7UCb=L!`^2s<^JSU&}F&^q?mNP==X<9(Z+S3k`BZ?|eWSpA6amJ2sDwdA62IiD`i=(FtIy?OuD1Z&r3DfG-c%@<^mlKBLOH&lmCH7#0KW5LTndH*u>1wU3uFpb1h)qmC6^y44Lt<=!o$cnCYwY-#-YDY>C#;x&v2(u8(70wTb9X$U2rRo>(N_pc;)R(7*cdK^$3lyDDs<mAW!To~#F*mn@sRLv~JXp#9y-j$AIeS(``Q2_CtmEvu>caENgUq|Zd&P&-oWWm&aSV}ioOaz4k!!*FFE2=n38^0$_V&P~2CuSH*FtYh)0S@-HesG~N!kRl=TqR3P?<q}*hBNpD2;lh?oR>TkA~=-?QO@JfHX^WKN(#jqtUJXo<(odfd|i2Zo_5aNg<-ZR?8WGxLNeI_a??&-SgzvKdstN$NG(6C^*tk0%1`5_lP9&XEcsbq1STTF(i?u!9};ue1dhd_M{z>U`KB!PG%S(ijinl?+aTmF_L6=`{{fJ05vGfA_-cjqIzqz_8F8zwudx#bd^HA>8YgEq{!@&2F?VQ+<|DwSsWAbT4*}FJy(`-CnBNbYnw4FoPe70CvYi3=nnWpl*aZ5n3a=nymfy7thZR7TC%g%Fsp$`g-ef+_;?I+;h6$r>i?}I*O&w=>@^rUTnA_d>kuX+))i8ceqiHrWw&2bx-U-;az}u?;B^QnK0T^xs`b%5zQ@bVEj~Wn?c(_6w|;eMAk*Fj0inf!Hh6PZ<4<np*ssH{5CHuGbMo}M)J<<Dq=(&QlTjFd7bR~bjpkV8`C-kAj@hZOYbv>RD>9i_{hc7Bma)4E(_fRbF$4?)jn=-i0_1>9c8d|s`dU$+oOgcn43E-Vl!zAKpOf7=vsGMl<udu3Oz#B;bdg1RS(M$Yc&kPuG_{xm)gdT~nslU2;USsA(aQ-DPQ#X)elF8Bq{idO+9oUfORZ2*$MUF(acE@oloHx@P`pmi4&)>n^c#C4710s<TY|&%K%ukaRqY(aRcq%j>2)@1vkf*FMYM%>QowzG-=|#)42*@W9y8(kXPL$Dkxem{E+Y8!kLUbi1VlR^hVH%QhrHy42&4}5z?mayhE)o1SueJWcRNDvo$-IDotO+-^WDM~;R5@i3iT(m5!CrY$BCn>-t0^Dh&k`AMnj!X6U)ZXuQ($m`m*;a&}}y;`jAi}30LNo;>TZHZfPj&Xz|D&`>6~G<2fGswPX%7vQ&f0N6Z>VRGXT`gZlpYnjvLnj9Si6pR*s4q-7e0q{@6JWsjtg9o_2o>qE7Cm=IyBci$l8J$nNfJaJ+KyOLl&lzqtr;$iJenk3?atPsO^*;CmJF6@R6dTi@-w#0%Ns)4v^ks`n-f5h`+#?WahKcZ@D8#5=Q3<&62jOfNzvX>MIz>-3A=4P$v8|r%i@X~roA=9B($sSn$!?eRHqn!mKfeJhqSVG5Z#7CfrA6VOFRP-5$jK;~`2bb=Rn7V_jVH-LEe;T(w8J_<fbk_P1%r!$*9nYmU0bF^O<gbvVG1Kq3N6VUxd89Eufe6z{Wu!C(&@k(&(N~(1*H>y8Onkl8>^NXi4#C`g+a!AK1#;|-hl69V6x!|Ym+y15W=8Dn_jq}NM~4vTIwsT`sqK1Pr|f{e{QG&;g)2!ZkC%bxSKRE&uy0f!yq*aV6g^b8UhA>{uf)`7o4EP3Yh%)um=g2ipys3u8*>i^vK4$Q3G+0n?qBX3C&MA)DX}siMSg1EXH=4-{mdx-L40Q3Su+{N;>nd3;K*TkHYa*9FbuB_=X#dVIG_&mG8E*WsX)FY=%Ea>B<C{Z5j%0BkpHH!VSH+zjN@CgSHUjLB_}L<@e{D6Y;z!g)hubpq>8?9=%=8U3LZ<c6y;0`)$JK0zT^Q$zeRGt=opvOp1y~(lKBlOQHTir8kEk#Thq*S6Eq|e^Rhr2m#f9#cJ!Mm4YwIoJu(oPY6#sVkF~*pyQYq1k@lx`i*e){Qn4i)D9b~E?uH;WA`)^&(4}1$Wq^c=v|LTA86Tmr$=l2eEnOCMgZ2Ss#j%|Ou}Vb{20v6X!_f1~<e7;ENRa_{!s$DvkHGpEe^N*_f-T$uM|noP^tL4ZcEQ8qA;7X+B{AoQ=EB!c6zRcX&)+m+vcXKaLH*~GkWc!(0vD{J%ZWy%r_@6dK)w?P>jw8(5B#hrfv^bT@Y&iYwMZ!V)|x1q>2&UTInss?a2>_0#aYz+6)x7RzSAf~7uZ5lx!#|Z`hP|SzCqa^p3~UoyQou%UwJe3tbAB{uDmIKx2!XqS;i#VAw2n+Km!`l3svOvaGZDPi$VAT3N5%KI^=2^wXG0xugzFCa+RA3m-op#G38N}L;1w>L+eFw0S>a`q)B~bl7^&UbjG!xW-^c>e`TrI#_NpMaD~Ej^&Muzy_<DAs$tOVJr{VmKfBxFb7S6yI(tT0sGt9L+ASwAXz7M|BY(Wn=mN~E-Vnv?6r)6EnNN}ura+-Cmea+=wDYO+UE{}EjG7DOt$bJSE_SF$&&{$XL?PRmGER=V&Spr#YeVzi=W&;8&O8Z$k9N64Pfae1e`C-Ubi-ej-}E%V29qDfhy4;8Ak=wC=!V+!h1%?LxM!)5)goe_wgIF8(!Zv?EZkw8H+&8Cx0Gn|<456wH4F*c3aoL{M`tt!$FWl__VOXF)SnJl(@f?}e&~tCSIrlJ|06&F1TW&e5<NvYcS@jl^6f-<ras#`S<gwmunFP<A6#|Z04WA=n2yBZhHVn$;zjw<1_|E6q;}_-t|{*xj4GTh3e)>>5R`H#*<DdT@vE|Z)RC$MCWgXJ{BzxaEtTkixjV2e+3Cweo*hfJyDgo?|L+tBvD6qhm)6W630<g*1mM5RweiXZG3Smm-GgH6r@z4){zjlOmcLykoj!h|NhqWoy!Es7Y8kwGLm(v%%_!G~gIpN5+*vZb-hUreW&y^?LW6yaDf+EEuRk*=Q?DvKIPV|UNRtS&uev8py`<;@mdXQ#3x1}$YzkM$-$!cV!4skw9?pcvz}oQWELbHLMaNzHd3cxAxei#ZwZY%=zp+de{!+hOVegR$1qT}}iMd;js-izq^V8Nu48%6HILf!3i3KQfUs!6vAocj!wal})M~r5U!{S=#dcX{Knf*(CXOm}FC#b?;+cLbM;Ad(rRcGf8LB@KDIOfQqLWi^9w?f@Ml=NCaps;|X?Hyd~xnwb!{_sDt<aH0{+d{*Ufa_h<IRD`_n&{Eovcxl4uK=#B+~H<<byhJA1^_a_=v{nB^-%}KK-(d8u7t6{P%~N<Iy|K*&e4Fn{ni@@b8G5##O-LF1YPt+bWz86LoTF@01YAt0G#SzA9HC=$0V$c+l{l3$Q0e1HczywC4j@Ci>!AZNT|Q`Y$<uZyXCN%TC;`itExJJwg}gf^v42&M=zM}W{H66payYTATF#PKXms8=<96Fy<Nue+K0d|;0D^iRxD3Bv^Mw?$M-+$wv^~Vk`YY!TgN}03Ic9w*~*`qJ-0NW@k5_POpUxXwCL;}t-^CBO>Q`_3+_pTwk>7jQR%A&SS8X<br8^ssXE_Jak}dt-^=#ev;V$s;$XcF%67ZKdm73npri>Wp@g%3-9tO>T|Y@@)(0?=OE~(E4YUoI4@-ENix{GgT<#h|^n&|;r)U}E&__tDR5hk2D3bWQ%F#$6wCh!6ol+KkTg)Dgk<iigB`ufq0tIQL$Huf|F&%3UZ!LGC3bmIXuI1Twy~48E!`Q0(L6yHXbjfK0B!rqASEM+rOS`aLBfS$Gnvt2RV={Ba6D6D-*z_;4Ct@W_hH=&3eYodw_^G8%LH0{7BhcM^)Fp6w!x-@AEFohYa0~vP-Z8Y<(D1CIfxH}xEFx-_>+nE?f8}PYhgya)XKS<YZ_<o@)&p}Go2C{C3yLU9A=AP{3p*+Wc+Dvd;Eqzkl5cBX))`H1oYhok@}K+#ws6mRM`IWAN+q=)JCs-Wu};K6hDZ<8?ZY!)1tA<=1Ql6Ev<6=~;He|jLO@DRs~y)K=_fr7zmc->9>aaJJ#>m6Q`+r~#8L1AB*hGgLTUQyc#TS$$IP0egYMCEtv+ula<TL6jQdjXV_3ry0fycBRrk6kn#P98>mBbc4>2i}DOG1n8|oCFSyx{zo-xOTtK0N>@aI1~<A`D!&r5zIH-0}lE;tO$M${P_KrVw<G7MYEp>HE};IY)0!qZ)v(YSHOgyJ6p%jZ_5edJZPX?CyBaf)X8VL&w`5<2Y-2rhX7_JzUk<&lJ#DeF@R!g?IMPz?J^m}<eFXa`R4)bO!0BPztgmH4#T0om0@n0ERX=ky<Q*hP->9V@HA0=r1jbove9J{iCBYN=un?yN+XV75mfXVe<D+f;sVAO}AQ!_tk4jl8mv1XCh!w>@yH3vS*Nm3G`^PSpbowp?y`6AgHFzZSxOE!b(r%1~Z?g)py|L_+{mDcXIbXawO}ACC;$sTt^W>r~+GivIg3`R+zI`@Tw|v|!BjwzZ5qz<hHIDy(!y?3j|6G`gsd+vRU&ORS~wPEWg|7jOpu7=E`{a+I4^@h1^KD4JnyS?(S|O$kdrO`H?_>5V(jkR%CzC~J!Nn14mQ6Ogu$xX*W2E0-9fdhrwq=x%1MdRwSP?nFuCB(qy^W-TP~fR2P2%8%68aW8_YnuxvS{-w{^w|CB!ubiM4vl`p<%#VVuzb5djElxEaAdhN}et1|mt;+jblh+z=GO7JfM2;Y`O}+`kFqU*(vmtDn4G~W?1lM!5{j%9#ab`74^lR4`+B*(JS%Ji%eCrxQs3$=<@g~T>-a_#Q*nr#XgDI?{WAc8<7=KrDUZ0-Pb~Q~HOP8U@pWbDPV}>Xs5Ilc1ks8dq4rPx-AzR>#wEpme)kJwzGM%Ce%y2+?6FnX_$b@*b$Tj!wb(%h$Nb;=ie976cHX_Q)e0E-@)i7Ft-hje7yPf!YPWQ9YTO<S&mEqw8P85r+WA2KKm$Gi*nt{W9Gwsl*Qqxo1^q;q_)S6?6oFs-^LRW4S8?0H>J&T_AnTZ>S3(yxUgL2!O$N__4b4G_~8?AVlnC1}-c%n+Tk7Z^x*|ckj%712k5RSs+m6Cl34axf2*A)x{M~VRc<0L@Hd(f;;AcDLQ{-oZSM;E^cW*S4P8L=n=w8jK===XD8Dj>N?k){}=XCaw=;%4tXZ^D%ZUEk7Pm=GqejgII@K(xnM{Qr_C%vEbS#NU*Zi9BtsR)f`PwjtFB`sR#fnzF*XpvgY~V^Wt!vD>e*d?Q!Y?6EWFws#cfR;cBKao>0_F=mQ%>f(qGA^y#Vj6E`D5R<OSWI!VHVC}+U<n!EWMWh#JmG;u35C4+2BVNwp6#Wxr03zsU8E8L?s=fewrgKs!I}3xa(C8^|@!cf5%D=FQh(Ke&%gzt9{s*Sp_`Gj3<Gq8j6u^<nYVzlF3z7&1DMCK2))M70v)h~{Z0h-lZTMs?3}l^YT70vp{KyaaGY`gFQ3%2WieoVe?$Z`e@|(L*IRyH2I+m-I<qk}sBznVZPFY=9r>{h1+PH9Dew{Q^HYz`jvk}Y%69GG<3Eisusnk?(n}=nLdy@v|=t&dch&R^?ndQRD7P!XSG0=%_is(j?RWphzSf)~z@TIa$Ckkb{m(V+TY)^OjJIm^okCac#?XA<36R0S5_u)VXP2=7tz|BFDff5}AjK?o}svSf10gh%afH>V94W@Nb;gNjbR>I87Tv2eQ=6sYzdV<TJ#eR1DMos&1Ido-ul!L_8)SD;>C_JZPHwWFhZu~ZVXA{h-yv~0T@c&D|*;k1ur@iv3@(vy&c3^Ky$+@SyNK}|rRQ&;#l^EsIT7(N4u|M7|12$H+6H+lXaz-5zRt*!cnF5DY!^#AWlv;(Y>GqU)P{IU^lYai1b@as?7M@MsKHEiaUR{=;tAhSqya0C?Dlmt9A`L;W(+MD3(RrjDezx}}2sNXk^q%DU^fvZ~)B`6yiO#*Bz-1-8Oy>~Ng`v3%2YTF$c;Tb+=c3iO#LWvOY}l3m0uix;81L-`%VD~`8Pm98fPJ#`P!-*4h<c}UVr}Z}rEp~ZgU9!SHELVb$S~#+srz2(y6&O&hD@JnFS~uSL*%R638=O&mqxOcH&D<lIAqKXg>AeS0cjdQ#bWT9n<1P+z?gn`n%y1`9S()y%zox2`o8qXrgQ4|m>V9ub!_)^onKz-xQAgFC#z%oRr?gVG9~r=+Bq!|i3qVO-5ZdZ(Lfe;7h6M_y4^*NNn8H0=$pDr-&jFJu&k&Q6Y+W;$#6BaaNp5i7uOD-Lhnxe*zK?8`;RaLrMF6JrDGu^dp2z-OZwJF&+L}|6ulD*cdXmL4LKOjoR6oqBDXDLe?)`3KE*lo-ry5f_ap{#XFNhC8=D6^v;)kOB~d@5U_`S+5id0CIZQqB{dcfb)C#D8?&A+zdXR|GENtK0<*GQjrMkNzU6E(S7)HoqWW1HW3_|{fWl2{_-Tnl2*B;z*SY?Y$Un3+=X7iL@%CVByCtSYx7wmL{(SlnhRwRQ?6plEBdVS7<pBni1J3`=+rUQb0X!456u@-skIpQ%<t`I2Sg$QOyI!Tk{;aMmk*>bFU_nh&717}`0e5~VHK7`hMSIif&gBuP@1lKUk(T<6RRw?+{RfWSRddBSBe_vN6ZL_X5fQ0??F8j``mPTzNs+p--UAE{C6lKh>1wI~iYS4Ds773&+@N_w=Df+W6%|Zv7%QoSfn@d9O*-cp2K!0`zs4IXtV4#}$7_PpDO~=J77or3)DZUqj=su3eRx~G<wf)rjxN6+r5#cXqJq&IN>x=8kVBSl$(l;|pmN^YzNKp5~WX)F@YcH%)RVOB*iQmpGDOz<|5f-m6!%vrp(Fk<{rUgAz)ISVk<C;j4jQojYSYV;_G$5%;_~q6?#R|Wv;%P`t*l|RkZ+|;vG_d(X%A*CkIdbl12|ldIMgUg|{B{JuB`|SoEhnIJGWDwK(yO+SKq$vQL4~S!q*Q&OVh!{pCdX+7y0mbiYbu!v*~0plGOqkTg@)7)@ygzHJ0q<d$L>iAUyIY;8ES(#QlD&~OTUxu!3xTlF~>tXc6b&XaWC~l4gVmP2=CXkM6?kTGl5e-L9|n@hmvmV33<1yQfqy5FHocy$U7+1CW<DPX1wt%lBjK<?FKo6?qc-VNE;lh3swhJNJ0vG_w=SPt`r5EArKZiP&EFW{VMk1oaDObbglk*<{nM7y#%Mnb0gI%R3M1&a+icku|`O)asl=NEh}uckyipz;%u9gR;XTYgUH0$!X9T2qE<7I(kVu`?yD9~b65UnpRNl)RmI}SS{cI9|4Ci_lekbfHj4}a<Vb#nI>e2}@GFeowkIM)#`H!^-Ngj0mBrVVx*JP4tS7Fk_NHcF?b9t03V#}qyPCeR3LYx?Al%)eTp(|Uh0GX*nf-%?5i=SDScQNYnL<e7^wRv`my@n&a?ZHa>8LCL|9vO1>(uGZC28cjh1$!L42gLkljYhMF`aml!^rc3uM)jg_rgLHj-1KV{}7IIRa$nP`RA1Zt@<6_+B4e;+n#!K8FEUcL+|Lr3rWEthXZKF=ErmPl*%G^z_zWxk@khs%B=AwVlm7m_Efsu6ltxt<Dd;5CTQlgVBGCg*H^{#gf?BVYT3Nn7n*GnhMjmhfaC^SSXd8Ll;VIA=w#U-a0x}S-|UM}L|l()!~;b4>sdC0HL(%}b&vDWlCrr`a}cqhB~kBa=#9Nb_Qj_gsp>*y{}f@+67eiut@gI5$(QMz7Y)X$BL%JPK*X0PDyto~(CqisD*O&w^0$i%#`!F35}3$O0f-DiZ!Dk9<l(wbY^Q3eT7CKKG6@AE*-3#yHi}PHr#<vRhsIPPG#mFq?$RvZy_kPh#y{0tD{2KA(60~sMSskl!4#3y1_jCbhC{_6i*bO@af+Ykc8$fpqS-B;5rN2rv|O!Rew0vZ3N1|)yMRe)`K*sM$^woTI6)nMIhHhb?Q@rlPetYoU+AWF92`?DNR{-D86vjFkG^J!FE|mHZ7bL^iVD~Zrn!^UkT3ZMmwtcs^qT}P3AQ>HP&DsEqAdqVp5+$INs82S)qF)7{D&JY7~F>2;q-DhA;fbf4+#C#^`Pm6@qm=xCDSYtL{vEVGwM&2l?cM#TTS5zmu4;zt4jSLJpBU9Vrb7Yz~iinOZZjA9Z>$@?3UDet@j`NH`!`sgN%|AaIwgdWVFmv-|p^}2XkV85+7f8_1E-B*+77B!`N)|Up>Iuede^#3@lGY>R@ElE18UcVmYfWjsa>CVhu{T)`Pt>OJ)*n?$0kABMN%mMTwvIL&+92vKc7D?7^r#pe0|{pgB7~KN!_$I()A2ccVaHDuEKa4+zX_--elU?DKnKR4|<dovAb*xI++x$ffbXbUmXJ=MGsdo}*XQ&i5j~CU3PT&c^|j`dnjb;{nh?cms|^DP~T}HJs*{k&!2nBEk%g>p2j}*kx0OoA~Pbej)1TTb7uCiPBb_{43Rgd2Ab5qF-o^vZkN1ftOK~{tT?FjO$Ta-&Hu&iTlUJ!B<5S25pEKg6V4ZtJ2yl>o2Y_s08&Ah)Xw?20wS;T)D4I2@=-@YjXEPLa<I|tq&KGu;J0GyZsh*;2U_-zI};KQq!T{bLZfg1F)@M1?<fjZgQ98UUo`<99g7CRaVBPmcI+c)hZ|b_o%(xGnUVVf|31%02<39f(B^x`fd}XxbExV9Lrk&Rs0)d?chX%GY7-hM95ZUwlGB=hR9Y+HRH-zLH;s%^GG1GbVo8k3ju&wUH?$Rx6lx+_sBzVq#D=36v>7i-DC*5)${Ju8CrgIrX9QFe(=YlF&y0x-ayN$zC8)5FREPNe)ZQFrl3T&#J4(JX_oobLvhYVnO{lGBK&wRv93_UbBr9(P0And`y!Qw$U+_8LjAV54!S2Uxcvidc4Ju70C9hHoY&rT{}_3%m(iTk^mIK<ol@#C&|%IkE~ZzhOMzo~%_&gdVIOMdy#J|+6g7W|?3kSGV``02$pmtft!C-92vhVNE-QAG*2#NlNI=oD1tCsmMO!>nQvBRs(#0MRE*O433rHkBQ8IXOuDI>Y3p1^!N${rg=<ezN9&0eVlx;?Qm;mJjb!S|d%>ZL(;_oXjw;T8AlZ~P|9EFDNQJ2-IxPUK<9R7$-QT!EXaPtnZdKF?Z9p`gbkC-97FPF0UI4&?%)&6gk1=)~8<Wp3m5L=QXCNW00+5B%w&yvu$eQ?c@mZL1k8$(UfiW#9kj+;B~;jYZXy@6Y>nwl#G;A<fDUGJW-q+Fm4hbkBH++gm^Q>_{|_N`D=d2FX{Jd$K^qYiBx_2#4Fga09_?20G7|DhPiPyX_w*U}C^(4;E-3Y+-@ZCZ>H+Q0YSIhS0ygFj(M($S!89*PYPU_M1Y8o5zgq;U9nPd6j^08KC(2+}?{VbO;Nrb20X*DE#k5nZd)gYr~v*0H`VyYei;aFj?w%4f_gd@2(|7HND0%$50<a+VZ?{?X}_lJ3LE6I9?pnl(OZsD^%SgqV<Lw3zIBaGa9Y$=x`BdR)(z?(%TmxD9i@j7BGMwl#KL!1*7nVZLc(YE^K3Ju+4rL649i%@hs{$Z4oK|JOq>eXOiF&rtw*kr%JtmTo^86o8x5bOsl@v_q7nzNNZGrm*=0y`uqe{8b(!7-yD(g&qQiMFndv($+&Zs(!4stN~OAQc|C^`~||()HB)St=@1ThxnE>4jmET>U#Hf7>mi?vM21uKVQN-*Y?hQ77L4PghixZ3{?yywn)GX)b;A|4tw}vKl!^kWSRC-_ulwLF+&hMJ1`wo$>H!@6mcXzMER|l6u;%4gU&k<jNI=83Y>d%8149D3{0yq-kj%}SIt1|uc@Vm9GEFLuelJ@*x-+{+ObFhU*F1+l(pjnPS)5HmoK_-P<N82CT=HYjc2)jhV-z#Q<Hn&l*Xcwnf_>H@cgpj%VDstWv9m4*Z*0GT#eP*myLJ7A@qSUwJq3FUUD8kFs(JZ*?PxrV8)*xVcqz=UNkj-Kkz3_Z}TjK^>)3>zPP$VJoe?d9{w6cAvmWIwUB)ek?NmWZ#EfaCR_3;PO@h?+GIW=2|93#CnMDGJ=v~-EjEKi#;>hqht+x7I++8*I@rLlOA}zVf<q4`8lyqe^&aPk;Rr$}+ro=Nm^N(|>dsf3_<4m2TC6Lh&<yXE+tN9>tMT_(kY+-KKdN-nJw+sMsD7_gbzxi!pjvG3M*D4T{8syvW~&>2vxO2a=GL@unUeOO;bycV4#PTBA^9}W)D{T#z6&|iqA|KVtrysB<(<@ty59UNNtP>r{a0x~4X4^|(oWA}0h8o{d<aSBG-}~=p4pEK8lO)FQntD-_Wc;f+k!Vp4Frh!jSas!LK~ebZ^h(52@?CRS;frny^rBf>39U_XvWPH7ZB*^Jm6$PgNPkK<&|?`mddT8|5YVWkhBH#8J-6?Uf@taeSVZ^kdqIgeIn0Hn--WUM0JmvLYcuAJpC!bD>=DXvl!JG$Ojn7M&!{Yo%K6v)ufMkjyQz7&^R(S30!`PM_ZdeeFcof*?>mPl7UnaS^?`C$Wy3FuQPH900IuznzorK<-GU3{(e_?TzS_R0<d1Ar!;JL%@w;D=gEH(ilAV5h;m}DAg!c_5L&&Z+%qiZ^+$7&v(3Ya@%_Y3IvDKN@NM;_30&7fGUdB*4Wp;r)6;@UWCNl2dE)w@e)ir@wxU1wh!22R?7}1)_~sRU2UJ5|*Z#rXfyL@9;WOH#%7<FDBnacK+FSQSx2_ggB2Up4ghmG6<(we(f;D%ya{awOm##GvLiWJ5b&(`Bi&|AYM%Go7dyR#ZzEH|;2u+<_0ALv1uIbfz(d;@PaH5`;pCz_99AMzpOMf9?<ay1*n78r;hZqr6YE2K)09oHE#{{n52*me&m2pl)ezo-gerSjM;G~+N{`s@sz!Ied(-ndY$b-vWH61F7x0!_ij_^li#V`j2b38)8%rwbi{$yeWAYCSOoWEmV&T^8HfVzUgYcL~#GrO4dZKb{~!ky@?-ff&2MZtf~;&1i5@8q&uqHL6Ij`1{0Z;zG4tfZT2)Uz(vNDk<!(Iw-o#f8LKz^h5+ZI5#HKI+$HZ+(w_#^NlT*H>*mTYhk~8i+BIA23g_#Yy#Moref2;Ry#qGP=Us+0#%PPtjf26S*uU49L-}@YnHegeW@v7ze6nkO}MWWF9Ll<_%t?DKxsn5Rq?%uqSHM6$4v)LGn_->(7^u3y>57v&|%J2U5HklD#_Jj2YkmJmXMJbMdIpVBw$N>HKqOo{--1+S*z;yM-r@1^X{;9yO%n=%KRYvAR@&_<h$_K*5l@K2tkUme(4O2YZjgcg$;ffEnU>^NM$7Fd#B=PY-COcV)-iGm4NqitQv1n=DH2_>gt#seG4U8nXhn%5KM-WO!E^N1P#|Dh|rcwW-{}A0y`vG3;%2Xq1oUo0>-YnJRbJCXXzY3jnioWW@zI!ya1$L<O8sqtlWYd=KjYFTxe>!*x+zVA*4N$sxxL;tI7N1G8n`wdjSoK^7!{0e@hau}$T=bz<wjN<8(@a6j~@N1l;No4x<v7bGC$q$+;O*Q8{VGj;gFlMw1>;1XS$apz1-VY8RTa`_*4YbQw_EK)3<Rs9wW=a*<naQZ{uKl&K2!G5S`Kl@Z75Y5w(dWpR;A!YH)|HNAoNwxxxF_YWp&7N(Nq2-fq>3$9MX4lhO$;gi)6qwiuuWN^-btzcrz4HWy*i_I(sh<#g;{ShsRa;7|H*e_n5<pO7%W5y@=3|ioeD7%5`OhG*eK{!BMxG)hJd3Uk$~9kLJy-ph0lJ3$we~?Fz$_Sbt%u6xYd9@b^)?1tuCE!kCNx9{xo#3f*@Jg7C<bzoADI-Kbo@n@<c1kJy&}^<P5j)J3Q{`B`@#KaGEwCE=|!*aC{}I#YZ88lyJ$KJkJ~vixYLkNg4%Iu>(Z7+AYZzrdhwrj+jwyE1-gY0ME#QwSJ{f5#=3JB-2bjyR*9_1HkbH)WtMtq=CUUV<F7Bn590>0e@33s#o-9r^O{Rv(pUJ>=GaX3&wG#4v$}2gMQd+s>P_ixLwHC*<|4^<Y!d==(f>+;=IhMxXREq5h>^<~4<<4?;hABzEaL+&F4O=llSnRIWTNomZ$+wR{_0eYE~1EZR#NXxrN>M#@=%>Cnrd!dZPlu{bV}xbEA0A208N~yeAeeOMH+3+s7|HFW^rYY7uS+_O}XnjW*JRRBd;VWgAE!v1`w>;$1wy=7zYLTslCa6g>K-xin65q>bNpgISh+<+MbrTShjpEDjsb9Ryg%%B+;XN-2sX}(eRQDbcE2K(wl}H-oZWvcTx40-ZWFfSM<Syku#`Y&xe!zmc{T{r<1(lpUuiV188k$(W-@ZfidNuNL(d^CF8NWLmZ%d{T5cY0Ni@;M%C|l>QNunBz-*zEzl-Cc&-ipMXW7dI3UPl4EYF$+{Y@s(g>{4<}&P<xt}rAk_<#f1$A;k1>y%A09PAMH<cxiLCu@e_?U}9J^ndRO)*ozpU|m0@${sPP0N<h(c{(kXM0&~;kq4@@VmWh!~>i7)ql?Qzk;T6pCZ<-HsLo?BNO14c0jiXgYRRzDt25J9G1md_a#5WEjtX8qW52?y<$hqsW~6YZh;dcU;I?neC;47`86<TZu$YN*C$$LBj4UUDvW<Sxh6zZ@EZr(1jmtS^aw2Azrk|$7&6gsfU4*`A}&4}%<e&#hx3PpkoWX$_}q8-epWLf>Bg%knDFu<k{37WiMjnP;<)k$>FyK!SH@|xB^IZeI*mzbRx%-O?0a&uFB>AFZxe}`W2a<bteNMW2S|*Z38*4#REGEhtrI@s1-I5ZE<+fhJLJxPxP$RJyptuNTpy{~o+)qfPoy-Qf$G~->H;G3rpDsUYxk%J)pXseobSf0AzHQ_H3Cx}W%zymJ|Gfk=3!#E@z|mImj7vlLhvnDjE|epHDCX>4!!ujxN3xrEO&d_4o50Zp7i=2u~dX%j)`z6smmB5eKWUeHe+)wjPA@Q(}!LWKs$mAk%i%ROViFG*OlQ950n>bx;Fl&l{8SzA2KFRH)6$7`5emQb(J|W%vv%X0%D-_2WGl!PwXT6YvcsJ<1`GlEiFpY+ZYRW>JFAm*d-CU_)|Hh<(-4p;6^L^b7Uy48kyF8j1;~3p9KC|g13id7s|Vt_A_LSSMxW}?$u6CE20x1V3u3@2%LPSp4E+F=>)jN>0B;>tIfq<Pnw9!e9a$qvSFF*5HDL?x&n21=X8y}nP1G!PM>8TYMWSO3~cVJz1HwJ7UJXJ5=c>-Al1_^CZw0u)u(QTn-89`L#>(BB<;BnUH(9SVJa4dhaP5*JrYX2p<Xc6L_<{oPcN=luW3t!U(<c!$r}CWA3}v%I>=E~2b=9yp*OBsF)XVLv48wGm6>P6IgW*M5=q`g1q}SZzrMWaD^lx?+Hqd)S^FtkmxxRumqG!-(&!o&5xXPzPY=RUlbaEcgD*!%4}+tue;6?sx6b%0WBbHGHvuW+L>FRDBJ7X+RYDBK$x>dp-B6-F{#a!^oqxh-@wU4I5WpX{<Dui~)k|U*@(LtkZhmjW4$aXxxy6UCQWGs_o4;pce9IBa6U+);PQ;9wviFBbW44DMO)o`_+K<SR02}$wY(T^tAZs@;0KR4O$Rn<5=Z*M=Ib~gUDG8J~*Dsj5&Cl=E7dY6%P|~n^uos_{J|nJN?hocVTw-I9uL4x==@4!otBB~ID6R3e34or|Q}1OCzHAk;By1Pg&ENZ}ksPoMneIOZ_=BDTdqfZ@I0ti25y?j#@V9~%r+a9pB?M3W*7|FY1_0+WtOmL9oF7k6k@hSf=cbL%m&8Ku$u!G>RK+cGRL0JrLdh<PWc9k-%}b#n$(zumdNySwF_P`ETgTqwo&fv-q^1D{N`9RW(3p$R=S^}QGX}QdO}_{=-*!MDRB1~?r+wBRP?Jo_a!=rIu;dJf4tM|)&}okp@rY%ZPjfMZa8_Y^oKxr0;z#q0&Sqj)TV^A87S@+q3&T&7`6`sF$*$)_(&tPKI5#@mvmP_2&9o&F!K^H9D9`WdHt%yAQLXtLHXl__{gx5uNK?H+Z95lbYTA9V<i53-5?0l-AEWdEgBq{utlJ9b2zqrT9)m5$8md!PcA;$Rwq#Q;M3x#-gz2QB^7>UqBbnL?7^ImIp9mkOTmF&(S8VA#qY){gU6_1<>p@OG-BIBDX1ZUtgfRL)z;ozOQS$$dJ|4EXKIij4<9^9})O_Yx5Z<U;l@|DPAzleuQjyP;mQsLmMxS-v;v}cvG_EE2M>KV{7MHP2DW~(yX1U`TB|MazgP=LTHOa@8b7{W0OYykBU+69?1-JG8IL=PB`&Ck}bj^z@m2V$Z%(6_-35g@uOO)pP*CI4Yq8Fr#WR!RaKP7y`&Eoyz!I{;prTVzL+6<I$Hbj}m3_e}D-4YH`P9$&i3<!<=lzC#ZuwS_xiel!>-K^0?hd2*wWRakwgNgbpomIwb81+zxPAHHi4S%JcH8acgWO(h=JtW)Y5&8ZGzFR~kG{7Iw&%=M;a~R=GTxEC*0!=)e@ToHZNkyzPBRtFpi<t?%mjj9$zRX6}ZkGjWb@;AD&V%^gCDV;|>>FZcuk`>sD)-df?4!p3$b2B3RKi_}oPg<*g^GL&h-jSXjNV#tj2?IK?E2yK>gvH(1u-HzvYXB{Nze?BUZlJs7+83g-CH#0ZHe4-7^?rbU`aC&0>pRmu_s!XXWdp+i?532NiRAQXsJ3L@U@Z6|G&QR-3jvJ?o(`gxg%+GCmuIOarBW?QxGsuddU^QSqwiJ8T~)vJb<Z{mN{PP9LNw-V*Wn&6Q*kMK+;B`U5NAw8l9LYX3enAX^@rJSuA!^n(`r>6|!$y17?DYpG`>;key78@CB_;!URvD=iXS{E!ohTILM4BAI?_C;G#jHl;jY5RwDy-WR(ac`}!%3X+Ck)k%c>AJapoLvDKhTCU76V98li6)1i>7Jf6;xW5!IrN5Bu+4ssHZ^(JYbE=NZPn3Fx$v@f3Qb^YewV?=UP2298q<80(^L(O6^J8hx@0fjgcvFx|oQf1yMTGUZ&#@Bnf%4{EAj)7{Yf)QD$?wDnzq;_fQY)v+vZn1krPC2%wvkFYNDz6YgB>0A|Gv~xW_)hWNEn;PRh^|NMh8O|UZQX}?Cfwx+RbSU(-5g<%n)}e<FJ1EJjwZ)MpjuHDbd1WQ6KU*+4i8+kC8NccG!4kg3bPjbv9^)X5sY`6sxjYty0tS;K4{i>4I9l_1}=*l5tBQi(0F~9of4#e@35C9Urehgf^8sJBW{?e0knTBWq((SMYrQ_2-tRMbE#^czV?W^_725ljN{4KDLVMlZ-6FJ?xb$JjUb62=nL)3CRzp9)RLBxpGnHnbLwJWOAJGe)O&~4pAqm*mO8oKvGjW?IFJ_X92?hdQ!AZYIM)L_J;w^Pc;LOxRVG(VF_na6l^GAyG1<CQi=8s4-ra%U6are-;tV=c@JeI;_kQ6f>H;f;ePN>M-M!nVt`|kqSYL`L6F~mno$FjQSd@Byg&=_>5`oL7qdMBSOv{%MI#b)G$tAMma4o@G=;Oie%=ga#AfJ+`t{Mp$!)K^d@VM1c_gS`b*ErNR$wfx$-jD5VVojWqkiG?nIESopFwYih=XrW#)E67HHfN$u7ZJanaxhVCP7s?&>}i`s$qIIf)$y-#iR5N|#2&BJYT2M|SYkE__EL(HBSkU=ydrys?s8Gf_qk3Z=Id-gxX{aiGLo7t`Qyg0n8ujakp{eyHdA}u+9(ZZ!6Gm@w&Bt1H1>mdrl<ohiGs5_{Kg0?GMQ=9H(%iXhvn8#2T=4STdy&$3<Fa>C%)zhu=eTmPK)2#I-OOB0NC2V{f`gFBKMoi=VU&KLK8)DARpptS=Y@m>2(wBhX9gUdi{2sqhVs3SwQB<1ZtQvCOKsSkTlj664DI9I!`2YWYd(FiXO<ncmOQ{iZL`naWJGc)C;PhUf=uLFxZSM)`xRG!R;Q1{*wLT#I$c@=?-6hjNE7|Oe>@Y&L4#VQPZmV+43@zBI5}4l5oCe>%E=Ku}W!~nwiZaMooW2>Dg~qil!)t^HR?SeNqj0sLdbEE1kmE?{So9>d15O3)7t<s(2s2E-&Z&jN(J-&z=)UsqvrSN5uNgNd$!_-Y4Y(%dlDNxy|d|Tq@+XRoJf|x5--y{DNRCj83nE2*9g6s}v`EV6@_2kR=&AIu$~2=HBD*GCE8VmljGC?Bt-QRdzcGyS!5_uq1&#lqGPGwUx)$&O&Pfo}G_GFm`zIAZLkmrTDpACsm~@DAd{u)Pc6JAN_C*yJ65vE7$%*1bj~(>|DDtv}%bX`YLxiGEQG;JajfHc4Jp9LXkmvWTB!YkfdO%z1nc!df(Zm${S?6d=N(<A+6*m0d0c5-x#Hcfu|Inh3UQfwF^6*Se~SmD;wgwg(*KsewN990b1a@;I}==HXw_0#%hhQ{coFzV!9zP+YRvcW#xU^&dA2rZ3XH%p_q+2PX_7ngz!ojH4uY+zqC<OQ#di=3{fDq>)MzZp2P?|raPE?+RFb!$Mx-K!_Caf8phfDtdLoBp;?s^-5k;LX`vE+0M9=-6R)|Bp@L2H+l~2FKi=FEDH12B%63Fs1F>c47#lEN_IErnFBLds4*XqKAA(Xi4JMIp5t+Qar^y<-EhAbARN@#@YbM7@P|FeqSzN1|sFL6@xBZeo|HBLg`E^DndOm^IC(DG7)3`EP>Cx3qJdoQ8&RU`zR8o%xDyk%m@s}t*!0E9r^UzXYS&cs94X)p8+CD5|p`Z>vxS6(V21Q*&JC%9a2T&7bscxhpIQIfx-C8P#Arv_Zpr|hTUXWpLoUBhg9>@l(&rS_!N{onfSK@@fvd`o7bs2IFhaeQ3q*J~us)0w!3Vh3MOO^FZQO*eZ<yu%JyqBLkIyvNE3_3`gTA;_s4Rp|D?mrx=f4IoP{g^6f2?qK2Uf?-B6}F#A2MpLaVU@GoT_fI}ve*SpWdTS#009Xz3`giQoQl+{3lQYyB>e(9Vv$oh+ATk|4TO8!rw{4rP{h}yzK`-PTBA&=<J0i-s(Sa?W5-SUVdZn3P<Fly7le40fObwTEGwGOv3f2GoRoj6YtR|)LUl!y>L5Y$erxxe_YB*F$3r9{?P0%jwXiu-Ra)^d6h=h<B!CCqr#bgNFSrbM^P|<B7ea-id{VeOz*x+s9%-Qj201G>07mh-X$fc;W3NJw030zj*R4ADjgGS#L?*gK_;#wlLjeSO2lw9M_DN(DRa$0M4#UY8axU$}1jgdHkVc)#JvB6Oh4bqf6E$$F{N$V*)|v{VZ8UDk>q$|_@<Qi%Dec~#F4mYIO&lNNxLAIgj5md+XJw17Jh)-ZkSyiw(9tjj6dMJMvs0j5BXF7{Q3mX*Y-4*p|2id<Sc1EE$cR%XQcWzKs`e{~qNJTmh{>#qNh}ZV2Vl7dXxZH+*N8>1wGC5O%{)>ALUp%jqJfssORpJFjjrA*P<pWs<j)Q?R!!o3gGWzpI`)o$c*&}LXZeI}DAjvCBd=#T`~y`E^*CCoZ3cjEkec?w@mf_<yyqFSjB$&>O<_LbE+$TIjK4UZmV#b64wJ^zh&9q^v#5K$*h>{vw0san%_X=^15bSQ>p^&WQaAdR)(q8}i73o@u%gF+LAQUs1{i-cp9gIBIyCPD&V+XmEv`Px7MJ9iE%pLOreu&-(d)o0ew|ys<2lZg{z*fpHro%V^$g1!@*lqPb`4R$vZJQ747YUuxiI=$<5Br#=phfGe;9#GGm|f+KP0AO`Lii{H1|0;ehq9wYB1u1U4Nd{ssjo{3d1p$a!&E&QG**;nUsP=>%9Wq5UOZL(bq6u=M!`Iz-;$j-nTnAi)nHX8QDGO#AeB#<dLpgwpRqh=0%n7N1Zf>E^o48(T4vM@l%LP^P+cesNaUTS!eHA5gV6$Sv4}WA~8|BrS&SXNO&1S$zpCiCJb{GY{3KeCg)nHkxIINozf&d_Lr?&Q2wH}eIG*AJA2~nb5L^>IC0`2LwG`YW8nm?|NY}I4{V=w;=u=K$XOc(!Pr{f#nDCbn$RofS#1UfxMdlTU-khGua;$s12RweGQEz_Fnx+JF^IH8In=TXwEaJ^B_0TD#*vUY6=}>N>E&{H?nx}^`vh+>1x3s{T}lv}&X3_5uEZI^N~21(z*#cXBKVBIVx*->sSFGC!dv8k;ntyfrNSG0w^j*{1jH-&Se4z8)y8IG(?+jo&d+!kLl$J=IvTC@b)Syoc>mcvh0E^Dn1MNGsW=IeDLi$BY;bZEWla7OO5hq}GgT0(%Bp!R-<Z1A#NE_!>&<p7r0tZ2=*mC~0+r)XwZDEtp;nbh71nuRcy!Rb`-T8KJrKX(@Zb}1q^C%EJO_tERpbUVUGkOre(kWzGdxe-K8{?xI2h1O+ns>VZw+%VS^#g326NRfk+C4lJE^LmY1BMtPGShQi1bO>g-fUxpX_ck{mikE+@4$`q5xmjLWCwnz%cB4j$$Z$qZY`r&{FK*LKfgMwpSr&@mEifCebXT=u)<NEh$-J2aMq&>d})VGn4ki@N#;t3iBCHHTAX6FA8w#9H@#q#O=weW$(}>rzWNZybu8~Stk0{ik=@z4;eLVi=r1GkdQpp8^Alv#%zng+b*>?`IUu@%P%cwPa(|wpPxM)@nl8_YVpIDCFAm!xvurj{=)K4&-f$rX7!{S2Vw#*5O!TxS$x#3OtwRM{7JggihKa?N@9_pVi16bUq1zhK9C6ZCvR$@FpwU8vd!}dT}hu0<TWOpk)DjqU-9)<_BIQlY?OkSdP!293JamgMfCLzS6nr!-t!y_1Tz>2Gqj*sw&2ci!i~e}I<Y;A4U{%Llg1EhHFzbZ8ESmhDtTmh@lBxdyavxHTt*m@L@7_G8+<L{1hc5%8e0jfGfO-wK5Sbe19r-mBp+RiTSuHG@{NfI@PZ>4_FKH38xxwG2PR-SG^E!?!3bs80fN>5m?8>t0(*hu!sYr~pS&37t;!$6HWp8-3FrH6bL?hsRX;NYnj6R9e267Bh-4~(dRfRHiRowFSw8icfi!a-^IQgzBFgzkb5-w%6<p1khQH@d5s(??Axz<BhGWHdNu+P2iu&-r)SiAbassQ|)u^)Je97ZK`}coFbko>4FZY@?IV>wb7X<#VD)^60X&4DDj9L0GlZXyQl<MCBwH@|u#AR;7qYjIBY;~Vg@EOXu81}}@MLJFHM~(`dFG#>E^Fhz$9iaEezxU}ZKj}^?lnmbCxLW+-Jv<r=w@c<@P((gh<EWLHc)iV=b|)fkL(?bm?h)Lqww)thLS{iqOcqU&$%s^B^I|+_DY?DEvc+I?P!uE?|EYvXaAlYt(I3&?e7G?%C+4cRPDCdr)GwGUng>YHPE{xK++QZTwgm$Ydg4d*J*}Ym>B09IvnP8te_-G(*k0Tonb|8uvE4w<SMXSx6_FC11Nh~XT8Lc(f|V^){mb&6hvdjF{cdv~<?6k?2Fw7v<C~*6S*R1O-J}hPQeg34`>KaS=BtRWkS007U?vJ5&sIb1G8K#M=IaN-a4<mXbcx0QkI?R+qM+qy4yh7pwr1@x*>;}lHkd?~(Q^D+fEIeWp6QFg!^|^)CzaUCRM_QIt?zB*%_H9Gt;479lMY$eG)fc$i}vdCAmnxnV-FtS?5yAc9b{j4MA%$uW2LdD(-!Midq4o-0OTwaHaByS*5~h-J7&oM&xVdCar-mne{T$j*=ZU@j2KI<(osje$|>u5HR=lYnZdjp$_OV2FPn=49GkA39Unrab{>NVg1oH7^7*XX;V%zx3#bizqPkUscRF6YxJ$vZ;%Ss|+)=(S!uQ{!z41+%V@&^V_uu63Jg9*P5a`Dr0XNj!iJ)%tC}IkKI+nEBijkA|Gxom&FR`>>u@Z6a7!xA%R3Pa8k^Bd%nf9Zj3(hKnV!FB%&m_q(PD{ZZ2T-OpsEXk5FF8;(<77p(o~OP7%>c<3Ur5^hKw$EM8T2z3#Gse@&Dj8<yu+!+744(3Jzf99Wu%hi;uC_DE&JVOiTxKpxMim3ggaL*#n-^QLka+$PJ9<PBM-MH!kM6&FaT2P@N_Y=ry^zsqYgEHDfbv-g;~2TCEJC9&x>14<+aQxcw*k+D#2yLFnniW)8X2fk36?!85-n9Q%i%=k`dh<DbC2~#I)J-5|QOIkzekgT}5Wu*8JnU)wgE_`X9U76%wb#<yZ}*@hf+#;J*CHDQtaHzahAg{vbv1?+_2c)j%AuvZUF_u{F7r^qYzDxEQ&ZXgJOTt37Qep9qCrhE?1eYeF)-SO*H)R1(M!OZnRCdL~WFSk!>qRX?a3*5ZB{ptT86EGhbdB_>!&W7$;*Yu50m_qkD8aHhRFq2Yi3CsO?o;uG`q(b4TQRA1osSiu7!jG_wI6ZZMQCOMQ)Zv|(;4!k1|yRO3l?H^7`dBX?F0`m_3S?kBdzu`0;(|0%vveuQEI;MXn(zmrx$Bt9pz{Aw8d$!*{73{jr$WNw#z!q!kA?KNoLtB57fm9W-|D75wh=s^A9A&P0tb7mw?xauPV^8Kic5sv_iDtUg_{}cX4U-O+-@iqk)=F{gLmJ2-L==Id{nJ-6$wMv8-Yt{lSd;|sR%Zy6N^wX(85RkvMQU1z9FdK{(Eo`KZEY4jh#eaHgo@c42EtM%y^cAG^|S$;0wrF?An7+~$|V5mjqq)7FD@gg!X~2p|K-5?Jg2^a>d(Od=senLSFe`yB+JSUVd`Epojt_6)ES{`WdkXQLJ$3)Z|s^iTig!icR0kv`^>qDUoZa<`7-AX6m?LaAKRo8_{eWS6CGg<WznQh*vMb1IU5Trg@EDb16cp>kkZvp+7_MwUd06GI&m_61g?cUg|yxk^$4S!ptmv+pS)PHHLmk;oovs0?vpd}VM;N`$}=K-tu4mBlAf+3+r4lV9}}7~02o(P3@f!(g~pX{4f)Om)uSFMWB!=IrXr@4*ZyD>4R^aUtcyO)X)Vl(u&l3b>`P@gwe$EU58AbnV2lOAtmj&ZjFRn@s@W+jNjf<_O_sdVVXI%gnVOLx)AG^wC>`1?*vVIZBtfXS`9*dF_}9)>Gn!U-_8f;vAp7|TspT9C;F9%@y=&K!Bcz^ydQ{#T{EzUnF$a<o^=Q(9m?%;YUm^^5!iD|c@^q6SDHDzJFhnm_DwjwafJa$IHsToMrLo6^`(^@r2Eeu6qa<!@;#~R@Eki>r*RF0Rj!%vWm)TG}f`>g0bY!yK$5ocZf*B@?JX#*#KZJHDXz-4Swdi{!8DUwEH>Tj(p5tXXP)FWWVnXL=ip)8dvGnyAK`lk40dn}JKX^|5CNQ5S7ByA5!x6Ri8%E@g1_(OGBRp7_c0+Kh&3a84G8?LA0HE`SiO@wwo7tR8X0$W-ZLgkPR%GNzhMh>wHmAMoSMzg8iYQ12^PDWC83S*`RRzn%Xa@KYta&F)*eAL=Q(Hp`3Wq9=1xsO{^mmAtD2;@l9e!*vhqvYdMY!GT>^K51?1P6PKk`W=S$Qx;t`Y-uky~%Q!coB&lyk8ef8*2K5$^H5qH&-5d}j8rR7LxtM}CR|ubhnXYyA*1K2jBW(du%>J#xz_vlJDeS)_^+E4gAlHBBLZ+0Nf%%=DV(sRWH{2YFD5V9~>k(F6W2uItbfu>In!G7HdAs*FWHakt`_&jaR7P>Z>{Ft7P+Nh|n=I2_==UMI}eXFj`R!g@7#I%;8;-8SYIHp!?po*(svx^qWSK{uROadhG^d+6Zeqe4fyj!a+rzf_JN;xXn2q4SN=ulH4K$1pqsl_Ufrk{CcB5?dMy77XhQtITXzp3TN$B~*SSNP`dMj{e?x^4#s>_@pu}kKBI|zm1Z)7NUV5dCNzfwZy%Z)gj-$GIr9g@cO@1AjIT&4<)!f9;_;=36<Nz(wS!yCyiR2XZHO6j^j+@8?zuqk28>2Hw{WlLuw&u`~LJ%(?4!|;us*Gyd&fWTR!V+x}as>p*c~l3vcWj)9kJ_C?%&y6|K1YzSo`mUF_IbPEOVIc|!ej$l%ZPoTwN&?PXzNbi=-sAxA?9>JE}omx_RJtSw2mN{<%GPQiZtInJ_%q#de(CjsNsgCqip9>l^xL(y0q@%WiUPM>)JsKR9_1Lirdw*Q;#$VAN_I4I}5^Dmxm%zNARgrKXN%#9f^qXmF_820hllp2FN#|1bLj|4X)v^(oeIsPGuymOr=Hn)}NU>1t)Jwj+&1G};=Wep)v>u3G}exZQiGRzzBvcI6Z=oK(r%3;#bt=ZtV=ufV$vb#txgHjO9J3^C=(4#5^(6$W+h2rq{66-18S~1K)g;UFnmKtbQuE`kOA=%2D!tg885P#H>rIu5zmdx{6kine&tHGKyM?jCV^c^?_Tqt{Vbr_SZ#s7<4_CyvB!9{pnZ`ls?;XbW6%ri=&VH5d|myZ>Uis{4p36(0rm8MK{r6TSfB^>FbYVwdCZ+}6bJ=$q)hQ$4tdDU6I&v2;#V;^5LGG@oq5pIi37G>#3nZNy$`u~#36M^&opQicE;DZMl{2xYzRPYceVkrB*nCoj6Yx2NjxOF}La_FU8oL$AjysiH`70x596ljcji^Ky~rvBQB6e!}tm;eF6QibSXoz790TK;se#%Fwl%OKIrwscR%WJz0|pp(R?Xy7m$#9e7ZGz<yw&?;Z-Wpcr)5ygKCrSK`;bjv2AnDF=gK9-cIq0OaS+8gjCc!B}Fxf65;g${Y3LZTG@82<^tpI+<6+rFp|!N1N>E6gAB*)mLXXBL@C)*kv^nqvAl1JJ-}5VJl8EvL-5#>JLc|CD#mgc@|9+)6B!zsa$TnhIYf-0;Rlby>dn-4v#*)^Mctl^FskRTwyaad4@TKiCV3MH@kGR7JE0B6J|*jz!Xd>{}4d_T&0BxB3hyTQZ0J&M+eZrTfM8zX-R5gv)mk&eZ#?C0`;m)#kZAsqv3%0IonK62cID7@x<P{&OWng8vBX_mCC0>-ud*1cB6;VXH)?Odv2sr|Ek_lmCQlcwZ+z@Hml9V7@5Oh5CPwz)mkp({so-S9jA&%G99<1=FL3RTwFj>`j}!lyHQUKA36wC38*1{Us;3ur}8*ex`}KvTtZo^K&Exjg?w3t$Ce4caZyDXe=~u^Du`Q6d`E9jf-+m^&jjEGCcWIa}y+0nf>@Edw+Va1#ozv@*pmNiD(tr2j_}E3JRaG-NMEFpzmV=L~-@e`U2@<R!{lEVPye<C_`qt!XG{BCR-y;K>8Wc=}j$K2(79fbnOzkAzxc3GtXw)H!FpUFdIGAv;2IP1C^R6Wbm-VgN+4Q-8H}Q9kNfElE$95;4<*h*0Q5E6$&GwNSH??S)f}$_onq=Np4^$s@#Y;fiL6wGHC3}^_LcAmUO8Gdx&3h%9fcAMvfS^H?IHZ@WR%KPDW{D&LC5Zx27f=Ouh?E(A|0=1kbsTF+`#U6pBvMy4T^Y)F$kfGZ@$4Pe*ufRB=tR7fo75L|N?>=Fbb6oq(0zt}>74n-@Ga{MZECZn2k7=7d`X-V-q{0Gi{n5;X=RWd$3Za{m>c9aU~^lXP(8;;Y*2VEggby?rRVNv%P9cttiD{jA7&#tcH4DeF>gzMpN!fQyaTBg?I?8WiH&_M5^89-8JPRrIFcbPo;#(hx`si)>|u$n8}B=7W(&ZOnKM!N|Ha?tcy+nL0|3T>^kKJQc~M571u<3<CQyE;AAzgY*fXz^1pyg^-XQwkwE8G(N?|lF+C@FjtRm-GJMT8Q0Sfy(Nsq%=6)=`iC||Qs9OX%hVuxtA+O!WDF)*u2+`8!wGkPJzRHr^H)slA}Edar_t$&hUF9t4-OVjB8H*3Kr2@v<LZW6VtDb;8D~3<C^b4}f*W{Pu1noJZ6<3ZDG(8MPT1Re7ix4FHW2px7J~zvp|M8;pOiJPC6-0R!ZQPVp--qivr3ZVL-dUUONSG->Hh_i136WY2W@i$zM1RK&Q{cj=!ldemPO;?#g0}k70!2|W#2uV*6zBFlj>#Pc$`9A5Ky$+jK#;0c|>{A`GdV2+8Cs#K_!*@zEt{|i|k{$Mi7nMDV$G8_z5QL-gmYPBzH_%*@ZjeP1VMI-n?>XGNhnVk3QjYR?ZFw*9Uj~zY#g0Fp?Yg@xQ*=0%P+1RnZgD9b->DiKqLE^!a`BshIU9JKUu8!|EtSNaVx_A_*$}U`}xj2TpB0C+r!_t(|_k4W_X3W91T^3#qXgc!s7rk{7wmpn#?a)G^pAU;1PoK}wcl9tSF9yN16x;I0AvR)MhJ<<B-pAe^t;bL@ME=TGaKlIZZ4wgu7#qONOLXw&vN>_?ZVyvhPjf@=5=JBGu^H+=a!R1{W1kpKuX&`yMlh!7yws^B>i3oB=KG?j-~k1i?jy|g(^R`Ww^qTn`YN|;_S@3YFBx(HUlQI=(~wI)sz7bWwEPbIp+Om-nWy~Bz4x()8gjP+78e_@GD<4H(D!7p9b{?YS<na#jsHRS)TNdQ<qKmw4lWqJ<-5hj?5EM?|IzDhQ1Btsp5l>FNA8Cet>%m$vvFO1}nL|`7BFBDRcd*@w-7ZBw(srEp<43zme#1F*@5sp7CTAyF)Vq7@JQlH6e9<Ko?6dgj;4G#t1)cnW<(G7D~#FIw8d354oE`!x&P4I0GfYa*N@VQ5p9#a2vHWlwAK!DziFTS;z*YUv|i+jvJG(I}hSjR<o-3zp7#WCj)pQJ*K=GQuMPFS&<K(%0FNhjLmVNndKAAa^AzU>5#O6VNX(u@xDqqYGqx%;j8EG-tHRt_vl#O`sn&{?$N;bKSiC~|!sfV-?2@^ZH3U*yuG-bAb59+FpEu=71#@gP=$e}%4Td;RMow9xkok5E`gRK=zA9qwlpn&umOh<5<a<ngX3uNhq7=it$DSY!Vz*V*DaaV%WQpvinv_DJ2DS**Y9VI2`JCQ4C6|EO)ID;Qe=%AT_(X=z~Mv>iGsWE3KpX1%Ccy?Ld_h804R2u_qSqXJ6R6~Rn>{b2RUkr5+8`Rm^CZ5qri#`u}8ow((!cF;<0mt7iH#Dos)Z?#6Bq+!UiLJ(S?A4~A1pe@=H#UnlywZGPv@TQ1aG?VCw`P%4<ruZU^IIZ)lI07)1<_(QTz?sS^Fl{B5e3Xz(qbdblwxIoEO;#;<b=m=a0uV+S5KTKC&dVFMa`qP(^^Lk{fa>x`L-+}(5BZN~&jqSwA!TJ&wyn;&#*TMkV)<JQLQ7VFb=vTz>=@N(tA2ystP8#+Rb`3-e4w@jQ(4Le)+s0+3m0`U3;w1>9qpN|8(sLSV{4-}Ueir3<{-7n+UpZc*$ZJ7atTnU*m%!-&hI65ph?5aN;f>CF6He#uNlj`M>Wce3hO>@J`KnT6m1+6A|)xvHtM!<e?hA9eqK4@UHkA5e%QzjXx|(=DPZ1bW~2&v%$ocw|1D|>I(m02wWcKeEx6M{RYj<x;))P=K|NA--k3Jl>i!Tvf>#EPMTdddm|vPp%8*SierApl`V78)a>fV>tgrA7NMwgI?Gt+j(aEQdb2x>jqTFEh$$8Biv&V=)HD#s&61Yw$ID`&9!xjYgx322_Z8x;YbST}Bk7x`&O2vU(QR$Y40Ke11Uqm3fR8aK2SXz~Np)|TtGTXL48~bZez+}~LY*~@#;oo;|i15-UeOvHYAPkoPG?vJ~-#(dRzp4%}jEK>VaEeCi=RM-Q`%CCJ28D`e4bfEmA^3-Hvi!jiA2?c-sH0TmgC}b!{W!kDkC!Ja*LusdbM+hVU+xA_X$?KT`l8ow6PNZBxxj7EU?K`b!m6~c?lcGw0ju|Qj}M23wmy}<j~2na&246Ux~Y-$m<wVx#pZ8=FO$ey6;bl=JL)HX79lhSp@R>ju)BdG7sTw?E6B!P4qf7h-rH1!8Hnkx2Xk`cYe7krjNuQ}hmO1@B2!wsPY1T|Oa_lxpmf6rp=<?+r5;B?@G*I)-a`V(iL<1WQtbuH7H_isE%&H<Wt$Etb7*u8X+=3}A$`>2y=9k=5+Jx%O2u8_e2g^J3D$+YE4K8$xe!a)$@#G*bRB~r1wt9|uaOSrK-7EWsLp*Sr0MWKa&X`%t!edf;WK^kDnz!U6<7qif7L;5mwgHOLU$qPp+Mm#M+IRq*AA_|pqG!asAelgt){%3-%)G0^+$SJz(s=A_V0FGVH|&Hs5q%FG^V%G5nf|@T{_%)<5CZf`D?`|$Hw!I@p{KiNEWHdGZ~cw;!^ilrhkHCU47RgBNV=7M=kmZOBU{onC7JrRWPv`5efv40@6|LZB`5Ht-667A*0kOZuQX*n&NJ2={1x1g8B1&=NU~lFCCMEYjU~`7y=4<zC*TZ9lknLak>2W>rnIRwVweWlX%*z5MSRk!?=>0Sl~JaT8UofB=S+*f!<+pGsgtk4&bpG7H2#tzv=8-z1NCJ!Y^Qp6@P%NpR1l`?&1KT3b<#)r}PY12<A%2wGbo(z*kp|#yt4x0sF4`7O;f>pDlJ{UT-u>t+&_oGufdgkE<q2&O3F3JmYWbfEbd;SnA<YOOF=p5FnN$(y_#QuLsgLG51+LtKk{7Q^Gy^lpPXHHF$ec{IOjm>cpd}9;S=CBRPUgz|SdVv>HM7-nmWWH-r<@XuESmxR%bXE@fShO{OoP0CS-zqIH`AA(jJQphpoo@i>xd#?uD0bTHpVIw10^PIae>C2?B#yuL{*EcZRYCIDlg%UJl%XxU|@&j()Oe&b_Hzq$ynTSq5>B5wBUAhtjoM{$=M&kCRiu!Jf;B>*6ZIO(1hWy#ZyA?8<2j<oOecV91ID5a+6&|-o-G4=d^ealF=8>|ePP-`h*%^t``f({v&Cb9%e(HGlqc@R)JCz|OIu4Wi8RRHuFRiANlPxR5H-5jJNVcAD&5rSHxnIfnZBH*yq0PzTh4B>0EVVzC;gQ<KAD&UOuX_J5Xow}hePX^a4xpe#`Nq%~`>2Zs=1&3-pn*&Zin6I^BF#2F-K$0;h9$PTKNom+RBs^|dA82JoO|S(=UN#3)khKojRJC?f%TP9GcU|*my#1W`x3L!>gpc}Dj#D55aLiV=3sS_20`55oZlz&<!yi3kD24jo<0RW*sEX3ftr|LMn($(4ejF_(-RX?0N|@WeRM;s%g(&)6?Z>DJ2csaZ5%JcFmRvZ&NAah-x<1d<w_5VR-AwP=4$AL#l+>%(eGZ|bh{0ti$6dMg(o=aRcgaXes2naEM2l!(?|Zbl#<YS^i3dreHe=y&BhLY_E3eUx)yB{-jTqp@6o1wJV-U{_lxXwdQQ@vV%PF}!>-<cYh#{N7Yt$EvCbLHSbLjwXNVpr0LT}-tx^`a157mb110i@dRf2jYg`HN_?G1hb8A^-xH;$N0L&p*Z79r3Y2JSrq5VVxdws_pyraBbUe|ZQ!A&I8)s{lUlL5`SlCjGT$b2c`33~cz9Pvqnklq}8VI2tths|EZR`CBAgW>c!_vb?4O>liSCiyL<gE`~uyMA!j)VQwrD6p*Q%Prh(@c2r>CpR?P5V}K{6S?of%i{;hczHSB&TTEwJ95r&?-%zj?;+%`8OTJvoYaLk8_>mVGgvvx>XU`@G?0^<-)`Yhe+{`}mbu#Jh;gn?a@Yuq4w_Ecn>o~S~5~0eLmdo8lO7@2Fa5?^{#Z7+gWOl2i<!WH{yrhkNE&g5az<MwhBvn3@3AabG<-+8LFF!p+d;Ot|vz4F#M|-+-6Ob88rP2L<H3DdnN5*V`l4f?G{2WPc7=6~No?RVq(OAj>Tv5m`LiK&z*XZfLKxaR<nb#W<rR`*Dcb+ST#wu6C85lP}{cnq*pS2O8QedTr!@W-_)fN<-xMCp1H?E68Td@5hu(@HaRp1gu$-kc|V7zABh8ZgtGC;9^b8*JXey#jvvAZb}E)_QX(|Z&8%mR(hSPcFs^@V5gX?#k8P_MR}<YF!KyQPhBWDYB8KFa8YufX9XUH>NSZ7l)MU9{=BK>90%Y4=FaUbVdDTN2Xz3Yaj21Z}pp-fQ>!p0juVCc!nlo%paBQ{l=4hX>6a9{?#Ah|m9oERKvbF?6-v&YD~yR~bIzJtLG#q>vGv>XNLZQp^xX-G1oX<A(j0lXyAzl$Uk6Sr+fsjOR?Bb$_RTrgH4uxOt|)#l3M7_}^w8eR4{WlC>P2jV}NKVsyA#OqvTrnrXDqv->!f^|x(Z1&7knOY7@)r@$ChEqcpBf~s+<FF1q3l_@|^`!nFG>u0sQdLiS*_@Q7E+Bj)khrMCAlQ|0t({S1MeeiD?gn_Q9zg+EYmm7}D>Mri{3fe8m9((L7X`81g@x+CuWOddp$-a}_WfS-<5l2bIO203DSiJZ;<`M3tUicNGaItZqm#E+rdEQZc#rI%P({|=na(?8h$%Stg*ajMQd|Ti1Qc+zKbk-=qR>hy$;hS(OwYPEi6z=+qt@(BCXONH)R!Wkd97n?MZLI}PM!{HpAALb7{#Ubh9Rb#FwlOeDu1YDOjds(E<9R4(SDVe_@d=6N%KG0T-Rz$Y%xYntg>eLBEk$^vRqb%AtBQX^L)HB|wCHZfT|vXF{jJWB-mUk=p#C)GpT-tur@fextY>gW_-6)31H;7BIo65hcc~Q(YO%_6?}e#>6av~?&yqVU5L<%pSXqE|uYbM3_6^!hMfKOxn4w}gfcGoq<?>dYhhWW~X~uk&LCjrI+oo}ZDF$wd$Pxbf(|&nXv;E*>6<@NQ4pFXSGeH1g!h_*3N9E4#tMWB#@2vTW3aO&NqCM2hyN^!-jQ({|z@~6NYP5|Z1?&KziE;KmxS&IV@lDP&zKi7gYVY9!<Zm$p)8e35NRz(-(fFDYPDWlV3R+c?C0<lS)3U;ftH>_IXhU@KduR%5d!a!-9S9S<m4p^l{-SRFEIbIEW)uaVBnfEwBcX9=z#9a$twdfRP7M3k>46=*W&;<aGGPs07W*=K>%O(DCQd#?!s^!sWlb5F&GG+KdJ)7C1QI$LhVMZT=kbEXnPRODvR+~&*%3C+ul_EOGtcp+RgDXMIMLi0kyor~SZ3i&rkub`g+P+Ui;<D%^728yx7z&seXf)WYN{L7q`76c-bE%#E=r;@N<D_=3}HXTguNi#h`ZGZ{<$YPMEGd1lH+OR#r}kuEcE(QD(g3)0aQbSZ9F$@FLLNk%T7>r(#>hIt^YG4rk6*M;9k~J^mMRW-4!cBeHl=E6R|D*HGQ)dX={kHX4o+%Tl0_-f5_D-0yImETZvctt%pb=^_@~Q_2otqXP*h0pj)?OQdlhYi<d_iFPct+ZSrnJ=TvS`)4IVpOvOT9I@b2sFt8Fu4ABHEzT4EC2*jWFs{DSlMxYwp6yAljPLEw3@zGo}){I`cwX13S9n0p|IHgADtxfDx9<RGN#Cb0}q--^8qU9Hry-B4-|AVfKv|4+AAcF26c6H2@A3K)PvY0W^E^g3NqGh)y-i%aT+9;{DKqdEvgaHvB3YRS9{#f8e(7b$2^>(8JkO4-Y44Jwl>F04X3=8&Zbf6U6<j+Ov$2^vxX3D$<0kf5^-iC_0wPRkbHu2~ZGwzpJgYuc9M#~&~Ez+Hp#F3NB*^gnnW+JkAk1`DYc9^Xpiu|qYnvre|qN17+oiue3-kbn>DsZ}gA)8_w=gMB_$Yb8|ul~NkB*_!=O)lcdl5HK1RCJ#=ThV}{zSYIi@_^1OP?Z>ff(0SilZ{(rlkS*IynMSddE73uwFaRm9WsRp?(oUW$X!%aV#72sUT;mGT}zv~h@07c6tJd1MH-o5<2>R$DlX7%6aT;H@aian3F7PF9mz>g_ieO?w+7wl&e4{uBuK?_Slt0cr--FREWN^8=~aB5!`)uKLyqoL5kd-~#&!o^=6NqX{W0ghea=rc+mA18)YlAwO{@~mEGqD($X*MP6{Qv?G)JNa8*2qmMfPE>7;p&k?FseBSB8?aVFN^ub;R)3W(^pbnsxtII&w1)&hosW(tBotTJ?{0pN^$3#SU+3+Pb`rQOIdRY2m0~tNi>TY}SEOEHJ&Fi~#&VamyTcg#d2_7-CY^JQ&K>wFF`8P!Elu1I)R#0b(}9`u7O1rE>}1%>9u1P13C~-&|SB2s=4|S>0=*wfKLPs%0hHlp5cHWk56Dh<?$9{u*ZtC0_AR`F3{#arMc+jHugl*Gfr^YG6YU2PAlIM9{ca5JwOJTz{ME>DowKUKf*b!!X8M`>F=@H!&|q<Lb9Y3-w1JktF7kxr1f+B?Jy9k<PEN|M<rU!$3!r(C)1K%|(&kSIygO?`g`)bGzWLhzT<A>NdsWQ^?w;0Azoe$!fcgC~(4O(>@MKDznJpir+u(*>sX|WsmbV(YrtUjqyh>pFMlZp2BSoa1yz`O%v|?(gxys5M^?9RFI5OD>_m>1sukTS@T3#`gq~}geQBoGy3(<o6Et&kuyv7rj~G0V|YfnF)XV&=p`#H(N%&AxNl1Ziq1_!wA(1D*jVgWijw2VT3hI=vZR>q69gGz!FBKVGqGv%;V~Mlr`aUwlQ3M3s<r11SsA)23SVqnq%;0Z{mD39<>7z&WZF6MdsbUoTWs%WAqs7W42`Gv6+QvcD#iApbXv*oeyg(kbTC+kpN+fVeg#FAEaxvw>W=#RYnv<jNT0X&L|zOx&v8=Q9;J;fCfj>mG{<m^5ek*xFy6|<#P2Ld%;~w32fGAbg|5@l@1a)=P7^3%UsamK?P5<H^YB6Dmcu>(4<diL8==c_<rdUk7zTDQo22^Y<v6U=Pwp+KALkIm$njZVuB>7F&R_6T=__SMAZ}s>qFGOy<JgSMgLCD1Y>{DYANMBrd@%`7^*)_OP0)7D_+(tT@sj}x{WlE!b-hNqd%yGRuFt6{93P)!7cwQ2Zt)$Wd$0*bv&gl5V7V8WWdVD?!lDg(83xy#1M24@UxmAtZj?i%^<66X8XRaA=|3ZsW4`Z^Bg9uzblxKp(bASX#8+Gktd^SnwN`u?g>g(pg8p{MKDo7Cf|)PKraqd;D@Z2d8pD*L;IR%6cYTfb19)>!HnXUz7nZO3DJg+uGP7V>$a<<k34Wuy1*aCcf8LvH9#R!!cCeAdBL+Sp3IsDu8Ni@OpVB^L|68f9708QMYtYZPbT^ip8$~@z{>&bZwiwR&u!4aQaWo?jH;hn^sUhO81VH*w08tMnAI+Rt`UTSwSG9uwLq>y+_yd)t(!aVbHyey{<EIAH^6F1b`%<3#LvU&zggvwIUi|Dn4Es2ng!K-i>VnZ~Q@Y)wzg7F&LoZj5uVu5V5G}uu_w{T644hlpi&C)jcaQfxTogLwhhe3=cw^x$@!JW-q}-Q7#3X#{EF{xH7Es=W6;fooP>au|Ir5Qg)nQ9ea)=sto`?(~F><K|xypb7Y_K~f%y^FlCZ|hqbWF^%cc%y51rcX;YKiYP6)nfKvXR2@*a8?+qwe?CkThwu(K8D@Kj;r;RxSE38)Cda?TR}CIplFG-<0w9rq<cQBb1wSqvcuu^TOEZN5IN6fbJbXB1QQK@{MJQw&_7GpF|V8$^in`jzvE38HKK;m<3R;AjTCqXLSIvC50;H=dT1fd!kUtXFBg*t70>AePs>R;}KO({YW%n;j=PIqvy|n-RxX>gz*K5D%6|OK*6Riys!SPrixixjQbDhKn__t&39lf$WK8D!hehN-o=ihLOY2sV=@8OQ|((<=`s{qT!@Mrb+Qj`0pNIRYsDUZzD#O^ky+@tD{7z+IuTpo%;~;0-5@*{wMk$&ma})U{gx#HJRPAe75Eb!idV6_o%X>iX1fKMB2FO&;y{l0kC0@Sh)__|shWh(HSsTNiC)AfPkdeWIYmC)m>5T7E{<=9fGo)TXzO_{X`+n)hl4{;Q}Je9_N2MIJr9~(Zli^>Yn68bCa34+%d(5MX5Jz*H#33{dZY$uQOu{PYz^n-<jZR?S)3t1!(@njoW~<EF01zo3sAQzk&=t5&4vO6F}-H`13>KJZ-jj*eZ)|5GMaGrW|Roi6f`>CJVpup)DOME?Xv7~4}KP)fR91Y8<P-mSetOkU%X@O$7Z-!ZsJSoj?eR`r@1m5DtT+qQE7RX!peHHyB)>PM*78}N(^@iQGdDvt7tkvgd}1gq$@)FmzpU|le<j_w&?0rPG3~Xgt4uSm*zWGYga=bgs-92$O{8ZUw$e2g}UA{Tdc{<&yYHxp_<jSBkn%w4t~NYn!6bGIFl_@*B2*>xE3iZI<cK0D&DAWEPx!o#(~$9A&~Agk5aaQ?5|@1xPc7{5fTv{o0f--=qss$SdaWXdbW4udXtUcKP#O%^N`z{s^p7$B7rbwiGoWWK(o!D&{lk?#Lb~<jkCDWJ1`B%;FL3B=aGn$0zQXbg#6xTXyOTRIvjZ<pDqh}+m7IC>ET~`<3P>EC3hH2li@=9pB+lF<)G;alLsTPqE|SU?~XZkw=0v?f)!{assl5Bss1LilJ}Uw=O;J;tOPrb0p24Hh7;}s@yrZL`jV}i<|n&N<~dFH;*<qi{^vDr5`TQOA;Zeh9H9nxx4V585GWRGSWLm08IjZVJ)HZ#@53hsr?PDD<felxr-c13AyV@}D$m~J%ZSaQ5OXE_ULrmZ1!dTnryBI^q-wNzzsi=8Ji;NJA}(|(Wgm8;M^Fu1z8fX1_N%jW|52jI)c|MhN{|wHI2*@}ZcO>TP)~N2hchCvISgm<AusWFm?nLkO?$TWYrmY0575$7>fJ3zS42{?a+crD9o)U7#&i}-4)hw8=43|D?6`5dtA&F=qu@Cj;GdbrFNNU!9Nh9QSEJbhPa>>mrHbLu7!+D`wTblWT5s!?w~0!CDv6oT(uHBWxiGHnhZ1EbKZ_|q{Cp+rVLFgvEA$}Pf?u(}y#t#!uH#!hlojPbAT70Eu!Lgc62|$-zC}3ttpZL}dAN~vV5j=~an@78t>&k>Vol^*U&<nc8%2*QFl=W&jAw*8_R1Hr)krv#bnt9$RPGC@F`=1pXR@#yig}D~>S4ESoUTG7uV-ORE|7QF2%q!77`ZgmNq1NKjJ9%@X`8Td-22siI94ms*r^J#DbN6KwU)4Fk-nW;U)s8~3Qpm~@Utg-XQ9V`mzOZugd1t#m<0KXYhihP1uyE^B=WzeK3%D<eOoTdN2i}-WI;Ctu)=`gEUDoPj?Nabrb#fzf}V<+k_ycQ%o5CjH`<Ld7HvgrB9hOaRH2K??~St#3BJK#7Ys@jLZKnrBrNp!1od=ziEzlF7Q;L_Q*N18&Q^p_ooES@<)iTp{IK|Nk?lb#R#(4@me~{A*ItA+8^R-{6;n0qhDP?kqUZn+Cno9@XOCm<jydx)zu9}OJ*IVbn%ZN(C1ByzHaVIMb+&#POdrnejk67l_Wk6`-MS4)C~D6Cyk`yZHAyc?RuS3q=A|^^!UZ)ZFcFei*TVGluiU|Q=_cDJR*#S>T&6RXFHgkJ0Aa&dEKw1y$+kxzK2|iQ|F96P@pMEE#Y?)8lr*+06aVI+X*aw0M{y<;#rAu^qle|~5eDVeuiI&)N8m3|IG$Jn%CG~C0Su^8(*}6h!nU9LhgdDd2pI9wEq+^AYnr7Jc(6iN6InTkDMZ3wN?=QF0ebzLN%RcHpS8+}-AA0ZIq(A5mD(QWfRyUe!Oo{qg;Cc#iRlZu)jsYT`%WjqplCuBm`*NSL%3U2YE}QWG|w<!?-hj>3iWCp-yC-(+|>?#JZ@v*QJE$h9jkMzQYS7;R5IQ;AA3F}^K^ZBYCl3Wk|c?)siE-Id>G3!X+f>%8b5X{oqQ`1OE}y(bsfVl`V%K&ccG;B-*}z&`7kW6=(|m3>4}wVv*b&Q#a4?I5fj%Yekc}eWYB(g3=1%n0j`kD;6RDU=*6{H>zD07t|F}2)LCa<t;eJ{dYURakgx}9{>3B0QOB7Az#xN61w_y_A~M@7I$*V|jvT{CN{lqDf!p#_X)Q;vI;`rN<v3~X8=JoYlPA6FcB2+%f=DxPFH_FEiA&F3ea&ddZt7`T(jOEjHkbo+H-1cLdVteTaEiCHdl7~JrqZZIrmgF;2l-n7UE7l|eTm&BpDDLLQ#NtH8;{1|caZ}zjJo*qS$s0)+swnjcWD6mLyGP|gaxr06Sr|jEjV9G8V!ysBQk1pVKKGN&U`*)<8FMLG`!<?9Hqu0eUaE7oVl~o?ePQ~yP89f^m00OcI%@YH-Gql?}h!;iB5BtM1&2Nk&f*1^+jX?oj<g?hG2dF<1Ruwgj>~#Qz5!~5NHI&fB+)FQsd1~I*vzVe`dYK`t%5SuZCo-n_gB!*Ugcx*S?;)KiYCw)ZAjYlIc=D7C0n;jeeN6^gGf|5xTxmD(+#a{l3Z3JKKca6x(tHKU<w*ZbxYcreY#}z3I+T)cd!S+Q8gROjKL$&V#>r7d41X>rb;gRZ=GwMCm?&kbw%`m9%J~ymu&1I=f---d&i{Ur70Fpw=ELeDq4sGONAvwzW<XUx?0ogdNtaaCmME%?bc1iQr@K!j!#u&d@}G>VamPr%RsC(uB<I!i51L7>QEM53uV$U=e#q1DpW*ZOevjvQPW?3XFl9!vvb#ETT8o3>s{R+be;}gW=kbzMAFOQuIg&9zSA$JlekiICr}mQ7I&^oj5cvV0ZUXEfbz?8Aj2AXS3Ig?Cf2K+@#DYiNi{U3`g@#oFVt*yN=F>L*!bjaiZAhBnyK>bckbB-Uvh!L~YsZ>GZK2SQlc(^6_A<lMcLgg{HUl3|@c!?-wZbNT1>E&HF6zKI(Gz*l{>UzxlucmZMRw@_w-28m=9O+}N?Fek|q^;0nKcLdaA0nU%JB5NS`x5TR{*)}Tr~c4atsNY+o9Mc|AmA{(KX2x9E1c@aW<4Wg>Un;%4pDAT~7@xpgSrmdZgx2-80^)eDtp}h}r+XqNd41zQ=>xyk{J1K`FVS}bBw^{{$SaQjmZKfdF4u-hWuQBP%+`R(%!Q8rIUUG!8!6Ft$5qEezIh?3){*n+pnUqhh68wdIf&h02Io8wurWq9`yQ1L9{^tczMQI@m5Bs>@!8(iVxp9Q-Z2?22GP`xij&FXe>&SB8_K9Z6;YXlW%J$nk5NS>^1aVXD->1;L;ZP#N=LPT<q0r1v12ZW@&DlM~IW_5&fda_5-p}htOndH1k0~3`&=0}7$W_Uza-XEAuqcZZ<RwVj34U85lU4xo<jp}FI;Drhc0VUGnwp&ms@*us`O#vQQ4f?UhiS4KDCx;2;2R@6N%)SjumJJKLEoR=KcZb8xyqB43GBHw#r5;ERy#;$!;@QaLVH@#ub)wy92t$cb#J{>sBNC4I)92;!UvKALUTJgr*+}fgF2*Uz%pV7EcXU_g<s(qf*#8zmwQ{rJCgY*k<=RD)XQW_HYs=m>oy!8iWD6Y$43DNM=EqfdoN?(1RSyEov3gNwSxL0JBw;Q5h;FB5^JrBR&6H`YBF!UcQPAQ;Y{}%R$e(;87qHo?~Y-g%H<d7n?i};$3rep@_pA}N<12yuPTc*S6G=RCrw+MmJVxOwR$Nb?H2f@_NCi66>{#3qJJ_TP^N#^;91ado5qM$KjLmNuao4-4wC4}g~1VJ$?8D{!mao3SEXQ4)?%`>>(KBHWmo`>qpwh4z{wkws@8OMSMwnbajGgCOCUCdl`*aVQH5`HLjrg*W9Lgz?~sVc7D39-xC{m|q<2+v+0*EIUH|NoL2sx%3*kK?H=v*e4@8Q$h3U&ctVzI*-*G%H|3!``c`VZDf7vsrBZ2D9bcSukInpb0mtF3?gisq!`{LKb%s1gO7R=r+b=_U?Iw?@vL@dVwpDi&McW`%R@vt490=yklk)1a(NKlFN+vaqrx)Ko#MLZyJT>BbdBC(82<i{u}jn$sxfglu&GQEZKUaI1AnBw!EVe_H4Pb8G3fvf(?AG0uN&!D)z{2SUdOHy2&_PMVmDG~`me%6->lM=cb#2mf6<nQUOKe&)gE@Q?<6uztcnSUn{{{evp4iUOkqwdtw0VnXARRIsnWGbkQ(Zfgd!e|bpiT`V%TeVy%GhjRkW9@GTvI9b8?&pyFM`(SuP^?#&P})_lVL)=j#asw$P7_IRNn$HHwXp19)AVWvv3-ln1$KJz_x&K4xr+W31F_g$mj(q{gSaEy{Y+;D$qgh9*a1nt7t(Jp)peSl)R1ZIS*0eS4qDF&A;RNx=soMWpyaOrk%|zCrJTxKHL42qe0n?zAs$RV@s;id$=c1UJSkrt7qBuSSUJ%8>pG_6o9k~3w2lRHi$B-6m>o%_Tt~iVtF|33&TZ6B;w3{%cJcq56KwVYlS6^<`m0uq;!5nc=X22sbwc2`UZu50gc>v&a}wFWSE>^rb19kO0cov~tqrTXj}CgR2Koy|TbmfZP?FKcw%x>{vh$K8Sh>@k6Drr|2t{EdNE`dQ0pu*pc)LdxHqbBWk(QAftyNwj8^`(*o?oM949oH*QPW=DFr9D#vfW;n3WF>sRf=m}kGnEpRbeuINQPPDykQ9lAT7usvBP8{e<w6YI+-Y%c3t?Dm+AJFz+=nl;?30h#_l0`CAzsOMVH{EVHv32FLEj|faP?=cAV)>gF>)j#N_lRoGHP3dSFtZ)G3pUe2@d2FkuP3>e+Nr;AJN0%%cg~rQh=PFzXytn!+7+(n&bUK`(d{nw{B%`a^2DDJyCrQ7Ms614>wlp1|C68t}&W?LtZfS4$^!CwZuL!v8|Tbg_IBzm<31r0LXx%lZq*xcxkmRIVs7rr}(q>nr)+Zm@f0lU535HSsg5spFo#PE;1&dq-TCi1+7ItW@j}J*Y$SFD>j<Od>S5qs(l%#-sPGEk5|7fJS{rYNt9V_=GzGV+aZY4@+o`*K?~baPS>BP(T_yvwtEnUTq&h1y4LT;C$v&bHE^@kA5zW7v<M(D%R^-TvVntcmGrH8@tLJ{W97mA<4(E8GD(O4S*yJiE{*Q9dqci{h?HEBx{++YKIZ~T&wds6pVo#Cb4`}9fX+oTWOwV+F&$lr%~nfm{h-EEAiJ>(@ucu*=Q=TFg^m6ot&pW#L0w_5NkJVHkfpSFOFd!?P7<n|1Oi7nrgl|AxXx#Wnjw_Ks_wbOa4g^f=Ki9nvrLdpO^QR4-^0@1WXFQgqi6yxj_Vze=J4=3@RAV_nizw3=pEcaO>pT_CJf`)k9rj1N{6|4U>4A;pvEEoQ|(UG246Th#H~dTdIxi;<SS2;xj90b-gs4qsq(xX@$rEZmOndN&wd21@~^Rg(Wh?Ol5o-_)M6?Bl9;rPBy4DlhXT$WiOU2Hp-h(67)FuTE@6)1EX<bobllSVv|~ZPZ&#~+b%bYK(Z%IXOh=eO#ynYtPb7<R90^{ud1VbVSnRDAFh&#HPLeVioIm@9P}te11`hz)YWWH;73m)eRI^#95Q^yoeE^R9IMdeHt1RB&Ty6Pp%~oqqpv}E6X<D>NH=r@3gw4s2{@t=F|<<^Z^P5?REce1GP}Fc@9*@jfjlUR^Mt_l5Z<NK+S*ZEF^tGK;csz?`!Ke$6WeHsa)^*O$Zqsum{$E;-J*?VO!*>32CZKT9?WG%hftUT^tvPQJkU=rSz5KOCaDGB<#7P(VWVD%Y;4Z4i(|t2H0772?RblmP&090j?HrzaZrrj#e~29U9LM@Km>8!%f}Dv{vw%r_!|(aiNffqpsP*73QC(}biK7D^5gKy)x8>q9JPLa3!Ia!iibfJV~>P@xReJ0aF_cA3K(|LO#Xi-gL)JYDT#X59$R(;?|M|U7|<T>nQ8^-Al~Nky{Ml+n8YWH2l1vY8uj42O9tR9`By{Tboq$6@aua1Q}!^;CUBs#Vf-k}1I8CUjHocZ`F6;wq8E#pyZx!6(vPUUUv&&<ms^SAy`!B*``!y1S;gF2A~+VWy}U~s98=o7NwEq{-M9vnyi47r+%ZQv-j&Ge9l<TSH4=LG&HMQnp%ijIC=MjX^>+8<*FRD_9taM%s!MA(%O0F4E+6q558K;-e46)So`3wDB<A9*bxI$2WEgC9Zv}p1YIc>;y6@ErL38zNG<+$GYcuJ*OH71RItsKFr2Nl)h(E^HDj}nNNeX@EGy5ek+PlHh{A*`A&9xn!M3n7xZvCx(o|~nH<)Lu8^Q;x^++RbMTc(Xl!>g;UZ|ph$IC20I%;;9z<`OB5RC?=)k+CEJy<z*K+sn4!K_paiCDv;j{Y%s@1C^Kv4-H}1;Bg`O&Om}FFqg}09+hWQvF%tt48Pj;58z;2n&{2Od@y6p3L?QEJfv4LIdodt0Db+`vq|jt4I^w}`mm}YY~(ZUlBhGCbwtJhdrLuBIJo-)vm9iM=`o&$C3JG$cmxV$iz`+OYn@c(fR5N`5CU!fyg(w!|8f;H;<OxXY&%77=vN-crrzM(tK@DUCSblp7aqO1^$sG?sW_Xiil%`;$lrCP^h(11DGNH%Y_F3=Lb_I(=(z=Tf@f1TK0^MxSpW3;sk8F<-1Zland{*B|Fa`0hUc~8Ktq9V67B_oPas1O#pm#+Z2=cc#o5+g{95i*6LN?AtC?rcwqSj{QS+qOa}<0bP9WwlIJNG6*v$$b%*w5(=)+lWMVo$YBw>g(hp(XxY;s4xAXfZrqopm&UxjE6Dtu>nQhS=?igIZ<K(3MSg!hPtBV@b>N)0Ds)p1yg(S8bVnl_n!en|x1mTs!gm_~y149qK%0V*oJn)AEZ*ue9(s*34+!8J&<%l90SjubP73j8lRxy&VJi;V9{%3EWSix1C~5Y8Ha5|7dmQ`@PYm`9{DU;)`dA5KsjDisY@Ov9OtnJrtC7y+Tfypu<EtK`(@`S7a=Q6orYJo#87e?3EyW=OdcIY|Pu`go!2@lnaNl!P);GMoMypT=w}hxz2<*1xJ41~Tb656(Z_bbZTuj@?(2;s<Z|W)5Z++I2erRRr4C(AKDK`EHVfY^IU$rEzw4@^C0=0W|{1Pljy)QIoBu46eS_z~|y$Spa=@<L)W?l0Y!f5vmJ!w4Wc;h&gjA(`8&;=07F;g_||qcDMQ=PD!hgc$J)Alx`i?fy1aOAm(qv<U1s9OCs)%NQe#XB8h-sq)@5I09%=~?aFS%$Y<}1$&01BwDtOn$Y3eD5PNM=B*#IC{82b>g7n6QpDz=T=^d&E(|bn5zaRw6Y*;X4d;X$=qjaTh^B9?}KSmGB3<pj}B8OS7aeWk<!4O=Va^xNxvy{|~I)duSnn0|PoZZmTz95Nt2(bw#BNtRl#@(OWO?cM6i4%9%<$9gFmMopJL<0J6IuSOBT-Sj*+g1weoj^opmhcimS0)oMdk~q+1F0S-3^)3Gs{pE1J|tzX(&R;{@?lBBOhdPcFj(#a%6qf{s-4@wj|IyiP|27k?&A%T+ik1VEzQYGL3G|c{jR5x@EF*~EVxYU*E;!Wqs=q3igg1kAGm>@SzLFi+0M`AmVu0lJmtN<&CB0<Y>o2@<)QcoG~p?iLJ!Q&@!7Tw&^P<LMTZJiW`3He;3Z+H@G0ou(<F#)WKP5gaU2_L1y6!L97fFJXvpd4?7phj1GGC^Tuo_yr24V9;$4e`XxghHoo)(enhfW5_m~Q3QnmO5;!VeA^Z0G@Nut@mHJxF7xjEoR&>Gj`qOv3H9~DfHb33Z2*$<u3isDV}RXX8Y7;2tmUvfF`IL$uFK}Po&@oC(Dp+vh7$<+)m<p5M+3J@502?RP_%Cm>bWCdq9rp3{4nz&sBn`aY9hm~=UOCn$qWnOo5`*EVny)?@}UBmVnCu_WrU4C^<Pl%JMqXn6Q!pcWq;5`#KKXIx`>2gr;%Z8(CNv*neHA8SXz7FwWlVJyPqcWn*g1(B({K9iSeByDh3!e~iQNt8h^@CWIE8!xMXOWf~rk&@X!tvv8+A$1P7}=(#+be?u)l0!D)a6g^_d+MT--{TLPfXMgY&;Wb4uaTp<+9}R#J>kC4OA+Z7eq$}#P*06;!+WUKu}e$7uMF-ZxL(vd!x*t2hQGC6S4WFKek<&ft?$C>UZF-yVP!4)c6+M8P+Fo)Ucj=E9Xl?%Ot7jkGc!grPPbeHIeT$T!ql~p8c&+YYd%=FRj4~>QLn-w>#OOHTg4oaFeoHFnZoa^Q!ekdYbIFFhVFpCU$_@l(o1I!7P^#>6D^!b_zG1G`@huhc;*Pzdf*h&FC?`Y&5q_F4?(dHem7R<GHED`2MfGvIi^_7F7M-&Q6~^UfwVnWB^@_)<a7wBWJX0f~B&P_Ld6>Y#t}M`kH5=W8UNpE4qO!{V&93kHq_3Rlhj~GjvmN{cunFsC`&>9*qUMQ<$jqFU@(0PjFf_cOz+))G*o#zGB=cF@A-`j+|@OH>8Rv&&t1PIDQzCM2=)^k7BoETUSUqI>w_GH3?i0Sl0NV^-UP*%&@1X@|GAbhBlcpN7AKZ5Bqkf`^4G<n|~IX^0&WRmOjN1%zH(PZg^_K0iaBL*x;XiaTU*4=-7PnXpeP0Pz~Mj_TpQ}loU3^vF*vB?R55>1IOC#zL)v<j{&XC0!{a`f|07ZuA%Mm#m%I8o9Cw1JY|ng(1&rXg>AuZQ^ZD)d{cyVNs5``(OyPZ>|VZ7EO4ZMtF#azqR1E2kT))a6xa_J15?hikT^?g9rCJiYKUajIiw^kLDMEE`18Wd=fTe{L9O&YeYJNE`rbx{C5yG2hXdwHRKS_VS;$Vq<wy}#v8qDT`cSdCes3vEAALpoT-wU}NP^BGJ|doSzz*P>qDMrQ<o~<ljgYWiRRwg~11^b;?&C@wln8;!y=O-H-ci4d^5ynVF@WM9yC=36P2g$jXQHGf+)2f2X2Rik4@wPwu!gdt$B_e;kC~jbhFMHeY65dl!u8mQrLzKK{!}Yh<DV##WPGC80m6NQun8|ufhuZ)+-pk1I3>!YEtgA_=KwI41TCUu^rF)~)EtP<1vAVdiKUVsr92&`IcmFQ6sD#`RezOUNsU_mw6w2-$U8T;JjsHaqcN<z4rH*P&6@p7`^wxy(9qu!CF1D4sanjwBxBR*0|dRap}H9C+m;md>oZj?=f71ER8fAK`M*^}FYR|#scK)@f5x6vPxf}umA1)~=B<S7I`7$c&ldVJ^AjowGD$fLLRIVZ8uDvi=_~8{?Njqh=-qlJ?!!^QuY8*j&phTOf%J#Dyw_pbkeJS9cI)9|P+X}mWk;@7DrF7upqfSjh7<mLbz+XRQM-`VG)OjRx`+)1U{p65YuBHlr04Ibp8v*lHH*}%a#uq(GGwvEv)Wd=p|3kolu2X@#FQaI+d#(aM^{?uGtB@t@-tplWxvItj&2y@+o8e5#nEV(Y1*K~jVG<NP8!93&=B68W;kNWGr|Xp=F#OG@Ztfrs-|u&KUqJkbRpP+KabRR&kQ>3@=|JYfZpbM@L6pYqCDI+vQW`ad$OD3B2G!airNywKg1<v@W&ZjiJ=n6x-{+QA6PCEojm)Ecxz@%i(*qoH}G-JU1cqpM?1SS7-Phe4KE(g-StQUOX;r7m&j%YGlRFoC!_JC2TR1`UTGHueG_k#Pu3c6eWxu*)P=BBvpuxD2X}nM-2wFgEYK1~hpbCDZGO`{&I!)to`5P>2ao$?&K@M0W+krM^OptV1cqhTDbkV|B4_mRF!e~MdSa<;hQBVccra3S$Dy@j>n+QeT23TNR8m2{;yI^T((T`Y7BOz;K%}f=8V>IH(Fb?bxhDHRzJ-%M+2=lQI9{F^Z4|VOBCt2_gl~S?ObgMwYt_{R^5u8d;BIf-2|#nw8V?qWD+%OSlDq&~eh7d6BIgUWYjRO}>M#UGi-#l8HC!df#*@6Rc6%%Y<k~$7ONSOFDXjCM(P|B@0IX8ls$*DJAl3Ci+aLv>+BI-Gj?ZpGfuy=h*8)>QG}R*7{G+luc}S6qw@{vFq?Nw%%Y6A2(N@;I44jH0wS}k>LjNF%b02v4I&g@f1R@zv-$;jAC;8g1I;hK1UDS{GDiqHTH(Xr5ORUsP;55#ACLAKWkY47iAou@7Qg<oYhQo)@31V)4uFrN&>Cke5!+-#cxD6BPPoQ;nNDoJsc5myRB}tm5BpnqEW#6i_4Trzl=0mA3Y4k4|U2cJiQdPu4vy=(S<6~gd&}bf}Sjk{P@uJDg4jgW*BwS$T;DyxVaL<<!)gb3F_ZnEQrYIqT=G6H>*2jwyVg}IY9@o)i${6@u%|)db&SDb`mUH~UxBS44n<E!;XpW34Ww4VTE^z1;W%Z{#NWw}F!SyqUdOVIkdHLZ;7$e3^f;TcJj$Dy*H`FvQ-23H;fyJ0ay}V9&1O1zU+dv&GYWln_^cJpu>2+N+0_Z*oGl9fGW=^)`bG`2SqpZyiGWV~0A;K^Q?AH4_{oH)?A37p*J^KkG%Z)3CtK3rORbrUuBUedPT!*C8Hrz8Bcyk+{%aC(}QQhinhQp2Y?(zZW1L|7%4wL20aea)S7t&%HTf?7Ceb2g?K$~TtqST0jF0#Y19_(-Y)zV-d8Dsyxq&Xd_f#9&xu8tFs582iibJ~ytgHu<EO~#e<Ywe4(51pnn=}~>qq`dN^K5riKpXN_eeftZq&P4I8rL;dNtXa!DpdcER<L>`ssYva-1P5Jy>$&8FVA(pd{TxJRmF!Wl64Eb%yUX`0J>3E_^=zCXy`{Elj>@Sm9M^)#)IBNmFGAIK)bPl;GS{!5!kEuJnMfQBkwr#XgQfw?_um)DphwMQB3=QmH%a!%=S<qz&zprFmlrq_);`ODHd(j7IuTcX7>`)pfG|w8M*Z+Z-xZQeHt*ZDV2Mv@qw`Z85Vc*8sVKks`vd6^1}+f#_}y!oi+}FlxAmV>;Fh#4XUy42t2%JIC}^Rsub5K)z-_DVN_nHd^G0zC#~Ar1^IN2_aigpPek(ibF6+S5-DaulE~#b;vQConDw?37C)C0gZ7TQ09q@iKZ@!ZaQ@VF<qLqt*s^mhMTf1UxEEa5o&$MlGoXwKSlmYhy59UlC+v_1?2_mlR)1k0Fpx|_xof&Jp5lzc@rS<udaRx4kZna3~V?TzsYR3+jH1=?L(l$2{VSeWrob`ss24Wnm4q%?J>j;rjP@Mz2*JON`j#?c8${fl!6c`Xf*C1W&t`LUuc|SuNu(MAgVy<-&iD11MlDyZv?h@?_6Cz##W-|->58@-frVq*VhvG^7qFGwflqljt{`$TF^Oi)ynV6X)?@%xt)Tnl$_m$$YXTaMk*<4c8nMUjK>}s*iZ~=m<405uf{0ljs%qejVJ+Hf#YMZ`sBb;A5mRA>tMbN|u$%U+BnfEZ`@K&_L00~r}4~c;p;R~!<52&ChTVde6S<3*Sxu+_8gSYfZ@qL%aPb;)+?B!YeLr-VMh|4PO?YRhzi`Gt5J$4mu|IHmAzHA;)FqSjhw1*beiWcCk9Yj^5q37A3h+#m&7|;0^xaMMl<;d@{WeiRIRJqLju2D?zZ)4NC*N0rnqOv?khefQC;FK1##Lw2=S)CnSAt=?&TQ=dy4UB8oR8W%E)roY!1Wqb`zHvjunP-6k=$%MutN!SN=l8yKSwkmOeBoHZep+aN$-T76okqvRZ`U(QRW7y;h`ew;+7-K-j}<c>*CYS~FLCbn(AFA}n<S~mYJqaE?ZfnUOsOLF`^bdiF%hPf44VEH)ggJUv;%$)CI@`Ro{_fR)9cVz7UpUr_YrfwndxS(<Qz9wN*6vALj5S2=jy-i`AA}I_lUObeJE2tOWl?1+a3|nD2tHLI?Guw(;^(9$j^WK%Z`A<VnWTSV<nM(e?ID7$G^Hs{}B7!87lW=a=Xb9t;2Gb4Ey#KT=R;5LytH%=G|9<(NcJ*&^|383N!^;XI}S!*RpRx$o}XeH9>2I7fMutZ*!04v&O5X!Qs5YQP|7_*3~E#GMd~?(5B_XhE4kBq(B;Zw<<)s0~FuvKUWLJC*Ehy0X5|2__)5L_-9C5mJj*r+~9K14h>0RQNa%d;j3a4z4VQQ+<Ql6lh~g|z(MN+HEJov;>rSDDyfK4DB)fO$PL5%Fe;YE$<u-a!Dd+R+nmd1Qg_~z1{#ApZG#-zwMiPKoPZ_`eyBiOA(3+R)5YKm9b&6&=9dqPF|hM=d0@^e+BkQivH!pn>@icjX}d>TIaeb^Qd*0PdyDg}&e9PoF)A$$QP+gsd$Y7DUXQSUr--~}oG*bE{*N~`!3=}$Q#6GrhLaDbfH9>vR^}TYQ7RNr?UPI==<tRkgTGG$Q}hZ`Hn26#jYgd9ETZ(8AJs5aC>MFIRfTm)AeTX?1@B6MP&JSw4hD49e(<1$otTemE&+&VOtd?~lr3i8aLpniORN^v)M-o5e4iq$(Z@JYHVm&6XQ?Taspub1_lm3j|Gcs4vnP*sQH;FO9dJZ0!|5++cS)EL_)8AdlV@KEqNm3YN!M{U<G*|=<)6UouWtNurzSA2makfN*UpVzU&DR(gFWasXg=vXDV#sBJU%2YK5Um&ejZ&?WG7NWnT}YCl=eMb=nXYWu`=r^o?Z6doE0Vo{u@<!B&krbGw8oNzI-v<PsTyOh7qB{A#|&$TU|QtP4R#dd5S<(MQOC<eXr#MWcvnh!!F|`r}Lq6ySPN{@jZ@Zq#t}a4ctok$MH@+?=<x1M0)*k^^JaDt>XWQYVO&CjFX`S9doLNE4Tx5HV;f#Vc{Jo_$g2bs7CCwnvTh&c?-!weL3xDAt~5W&rM?W>1nH+on6-RR@SPkj&h1;i}<^m<Z>EVuvXeK-6w*XDrZQyZ69~z4+3c+&*ct@WWJ8kt+AQ&Br{{$@wh-{0Sen^)SfB|<74#z2~uN$PjFKTK=bU^6X-{tINVo<(m~;>&hjCk39`z8u#HlND&-Fy&n&d2IM-cqanU<p*i<Vu6yhJahk1yR^i;5N_|1TvBAE;`9tTSA3UlK#HpR~qfLJQco%23zP7rtXnZl{+Q&?q{HIRHm)epKFb5RpxpW6meE=P<LQq}cCFLo2#wE>;%1!as)FL7u8?S92)UN{jyugSI1FG{!mw2#R#xNvH%S1HGc9P^O+K^89MoYUsBEH5676(!>$`%W}ez|`!)<OyFV$RWqNXwwB`Kn#;`iRT7?r%+o<z`LhLoqSvs-JOKU^TrwchZrfY=8uxc=x!2Oq7s+UOV%d@qd0+578BHGj3~=!Q+7gI6{m&a^sG^k^Ps|_mX1p?mc(zvm6mOTg>+#hJSc>3lNzaTM=;eiUwtRG<l2v{cQHq5Q*ru`h`EC_Dq;!7DdWzn(X?xgXo?*-qCo`AHY>vMOB(FG`>``EHw%&#;BycpIyKCh5AljR2lH0*Q?uJ_OtlO%Dqi?}qfWnG)cEi{6b1#>&CD06Bsi&ijFtAY#9!_DD*y1(J6gu&Hec}W=@sVIh`e&7KV4cD<ZxDZ|Fa@Ls{+RA67Y~yKMdo$$eIR?Mb5tEah6|OL(-qaR?gE>Py2naKIXsxk0sCKtpPNP8Z~ivrGmLF?@uouDh!I?GBY9jh7B&m?gnT&c~zVjjw!*}T`U+3hKEaz5r&@8HXKGKCd4W({OPet?FW`EfS_dzl0*F*a~?TPDO!ge%Bz~SrL7htMkTDVr1bmU{fDYUUMF75&d8>s3?|;MrRk?)%J@M&84@g>lbK?bQl|-;g+%q}3>3c@dbKExgw6bqRW3mSd=Zn{$0|q+CupFqSed){O@JgxVyNQt`gTV{HM6<O+MY*g<?^@a!EjhFrp2pB5-XtMo4P9Xm=nwvNGyoQ{;ofGvkfCPBWO|K>jsk#j`C`2O8#EZ2;Oiosg6O|-iXsvS^G-LAxS)T<fD^JhZ91?W$%X5fi9}otW*M*!m)k5Ccz@%LxPgU*E)7>?pS+DcLv!-O`|E|-tm;mqaqS2En?mWv{Z9&I#?rRlx1vd9+Lr73EhTfuPhokUR{P@^YyF(QuqqV_NMTg^M(4+Vo}5{HzlHdE>1AA5d2przbT0aN$9D5GJ9{_6l_>2mB{Pps`0L!lw*E(Km#^wMo7S~9_Dd)URWHYY{|jTcd-&qpC|>2V}Nqd#;sd(rN41SEtpX{woLqQIVSrdS4=(Ve)&h*>wP+HIC!YYF72<bn7z#Rjo2SzS|A#sLC9Iq7C9D!PHKs63i!wjFEWFj{VQv}1Qzst$)rLerT4&Q67>^$2ygxEOg=A{-RE<9^Iw9iH{^B}R_eR?;^U3gzS&6qPV#S(_yBz<@oCsOcGKA!xsmtSe{cC%_GgZzECqh$4&+=9tfTdLT;_qnOqs~HYf9}ypnibrB6VV1uw$9U*5e5#fv(kJdF*p&WPDNu?or3|LL^qLxI^9~HHGAZgxxKRC~>KwNU*|8!`1XcF8#w?K-Imlru<&8jrwc^^NgiyMd3l9qh=lPVveGXN7xI{9MO~%RM}5j**c(-p<iH;)XUb?3cpc?)J6TRHP2B>h437$kh42cV1tGhYfkkUyZ$uFg3&K!?Gd2Mcph!IjzI|C%vxyY8Th+IQI>y(G0?|yE!m}w>Pv1pAJ0O>rjzQ?<;IT~n&0s)1{uvPvzZT~8MM#>X1eNz3(vQ76QP5rV154BfQn^n6+<HeFbVF!XV6Uyr<RAdQ<=~<$iLoQtO6AuF%_Urn#md2q+lUU1{Wy45i`M%;d_B*dtz&*63PiPKQ6_E_NWydPvbZqZc+#c7~C|G?J0Uo2v=C54{N?MIuFB_xU_Ex@N)AR3GL%_kQep>wrpW;*ySo8$^J-Z+{02^-sjLPEdx%i{y(mrELu0>wY_qTv}Rq1ppRtIe|NouU=LTssbvSNN<4RtX3Jhu$(wvtl7y76RzoCTKQpIdBw?Z0Yg(PH|D!M2=deJSr5lZnCoE?C9evpWgsbd1v}l_f)1|BK-f{a`=P51%emnlN1N-e-sQz=~_^;^zi)C@*Z7hB;6(HU}NLe?wUKoQ{>wD0W>*?u-@_|R)Yw|kh?EKyIni8|9cLe5u>Xwm<h1xK=BH-yN=BbOD0`-Af$9~9^CW)@T6z=sJHqr1nT`jV|1aC?GFX`70&T4$Y6isdxyPz2`e#lNU9sPx4`-UB~sP|f|pFuM#*zbU2Ox3P8?+062g>CH9KgVk10M{3%R~FplO!+Dn<4|w8S7*GqR1WS(D1|kE&1Vmd)uVrYUelnq?x=IFDbGdDD{|pj_d98QSp}#ybTdlRkV*ux)>-Z(bb=_&Mije?-iZMAOZgL%1sVY;X{%T>b7d^X{9R+g9`D3UIm7<%q<St$IBeJ@mawr0mClH(44hoGju;ETK~`HjOS|SacEnCj_JcvU34%AlYz?c|wOe8hi=2u@4~HrbvbSw+-`L>Rj<`f<gik}(G$--S`Y<&I;yEr#dDVI^xby3dI5|OStT3Xd2wh`Mk+fOTI|WBXp-W&}XW(+QRyze(DeB@L|E#5=X)^PpJrlA#apmZXuRP8ZRkv%E_fm3~Kj|-%?c@h?Jt`rQaDd-5uYFyZ8?}B_7EbBBAJEK8K3Px6$YukzZ)yOb*{Vs*jXWQIhumrUl;AS@x`n&zcP~ijd|;&Vi?6M@fUOqpY_tGI9#BDXA$ZGCjr}u=m{X#Nzky@s%@%BCBe3p5@aa!>qhAL6JvmkBkG=TCj)NkRWRVg54RFX?OS*bR=)pbc>-is51Rh5}{VWxEoejZCT~~GPD{O(ySNYP--R)}(AgYAf?rghy70VCsgyF3NeKZ_*CG_S+Fen>}qT4efrT4X}fC&G*FbTt_%tQ!+120)n!C;YI>tKp<QxH^t;=!wk0p!$-&MptmufrA&-0*813^QF@Xa=u!fvp0rS9npo`rOjXu}CD#n|W4T-(yc!)lK64p(mKT9%B+64&ySC_NGI{Fy?Ak5rT`(!nmk2HvuWvNwm(0&F=a`&n~cjW5D&t?m#OUNVT@VPHdUlOs(=9!boWV`v^oIZ)SG4Bsp2nb(T1^vlQPc(e92pA@rCKgKQDe-J&d|+`3I{MqZM0@9Z8Si_Xmj6^OnU6B$VTm3^4CkQ7KiG#zUSE*WP&JUNR`BvsVJY}=aX{u5o2bVN`D4fhkTq!FIzS>%^7_<Y)ZP6r)w&Drb3+i*oYrCb$hsCe@c<(TSCR2V7ZT!j@4b1id?A94W8N`oy<PI(RGgHse`j_gU1Rb%Y@AQSR%l5)&9E8QcwJcpG0gtKlHTIS5sn<>ig^Jhr6o@Ve=PsP4LJ8MQl($)QVB6<-*Lcs9@8>J<yvaXv!%+KT4mb)ou9*KbVCAfGi&8AoVxza<kGrx}pbM}R;%>JsFI3QDx)#HuQ5>85c2PzFIRA*2Slk&M#u#?p2sCc46U3Q}q@+1FLJM8e}FTktyI%k*4lM>(UjpfGP{H?+IQSh|844T1|CN&t~BoiPzxjaUAnmOvwCSgmjACrhvYx#k?nBH2`dlY)lty6fDpDY7zb{VwE69ww`Cq3#0LZ{4s1`gRONPh9X-y9)3V79bqU&X#xI@TJJ3*&K)y|o34k>17d$5y70A??t<<5ia>GYEed6i6t515Onr%q+DfN@elh?=}mEX-RdfXuPALJr+wxD^2jGVW$Jmua~mlJ!^<YK@{j?W|n6OU+7f31*g3igQu)Q2YZ9oCfQOyBX~jS`*lK5Ev5N*<JR)W#c|S9DOfxx_2RqFp+t$0hirbTjz%1)=HW(`!^Pv1FPEHzFZBTElEHeBWpkfazp!c}ivpI{AT4l`zdQ^(QTTFdQkwlksY<x)Ip5IIV#ERttQLdFg%HVZ^*y@56KTQ9{$nrlT9af<<)s4nH0MwnLLXy%J!V|_SX*jlLSm!wGoXrc(IvlqK-1>KSgv0?dbtM~Em|E+UM&j4DAZ-wR3ev$_x7+w`W-?oji{C@LdBlla5#X*nzm6%;;^S{wCJHy4EmhWH(R`JyZ9uZC6)1ZHtpc+4*=e|NukMz`qD4S^I0gk0gb#l0j)yv$Jf%9$9yWmXH#PpW_!M{<_CT?8tpIqf917-MDw=QzBjRJk29xCjP>gTdLu+>+}5&PU``kSY<UOTdoPoz(IWtf_TqzYyw;zdNs}S?)08&Y1)|?kf93%Fxc+YCl?t^@#%=gXLaXT!eK`C(=6<AdMKmzA_q=1Zh00QWBBFG+Cei<)wu8ayvCw0ha0(vXxOmb2NzXTROZMb#`Nl-xFcVyjZ|S?oj`%ok)4|api2+L7yg5VLYhhVLJd9WcxOJI4iH*e~<n6&3Vj^fiQDsh#Zrn4Lj(|iNan=`>q)X!~?!lm?KTW(HG{-f~;A!=|S4la!R~6e#xVVGzYgpzYA-uoonCndoHR;bta!o<NW@Wb!b4K|?rgR2-G=<|z?X|trdkN{2!?(<6kE)6c7Sk+SjxAeF1zXFo#p>-H9$0I%1ZG4GPs7HqG6hv4Jj`O2ktV;XPpAb&>UMa9s_T|T@G{GA1%po$pZUv^viJA}XS@^xKQ$o(R?9XA*nLv@!$`>|TW$xnBG)cFw9O<?)Oad0d&Ti;+)&(>_TMLG@Z`+fH24iyEaHLg%<uZ_zYWVys5}~X@9~1h)AMTL8FZ&2MRpr5d(lWuRnrRdX(UU~IUGCv5Cze&TDKS@q}_}u{$K6qE*&{fPwz%`uopvXw)(oN-$CFrMZ$!k$R`GoaVd42&h#-AV9oO;SATi^M}CQ7lI7nT?fJS`1O5t%lx*$1idnHnOut-ukkJ5%W|IwSZn^12>5cSsrp^Xy8)bWqqFORf-?R4&n>Z)|G-a?~X9Mi^Z`Pdztwn=oAEoOw3XJsuMZ1s2s{jM7d`Puo()31*|0#?T(kO6a(|=UZFm}k>CHc&h2~6}fC$7!Nt#1-$R-$>d-T{eeb$@Q}-YnXma6NC-Q(%P5pM#i^{^Ts$|8OflIc=yG`S!(M$dq{~yw+cijtRF}uq4+@4?&{A{I*7Z2}Mm>wrK|NaYtO+M;h~uKg2^L_h7nUA)kOWRi8a{07W5UGv3@!=y}Ed1x{(x)iwTxcOvrkuQp@Gz+2Oq$t8rn5iDoIUZUh!T_wf&P;cFquWLagBYns>JURRlBKJX9z3qTE`}*+((0TU<yvD)iG4H|2?peaKd?vroYeY4J8Kiq3DX`7e<c3s0QfP%A*+itmQ$aH1VcvsmWOP!5j#G>10}Yn_$F{qeIUU@D97%G{0!L&LP~M<k<>AV^A!i*;lnC!YjyQROGK=V)2bpaDWByK>MN7vLB+hYsuL%xz`(QeA%U+Jqk>-2RbDBnF|8t(uBcDDGay=XZ#ZXt>5Xi&CA|-rn6tV15vgO`{_#+{@SX{&<niT2KV*WB=VN|jJ=8Yx0w8!~{IZdQt<ELbsXwR1I>Y)Pt4&nvguu@(w_M*<(_a^4kjrn^5^naQvIl~X(in{uy$LRvY9&?9f$Y_~$1n<aV7jG`7J$bFAEVibL+vG^cwFgQqB!R^(6Chadu62<hs_d7La+XbhzLPN_bEpg5$R>W5W&kIkN*`ceG3$rs9?F?N%T7dnM{KdV8HL~2S#rO09;+S{P_gFu;+iDysEowJ8MnFCCv|+bT9Y?Y0JAUC<Q~RZL=x}~YoZk~n{*pm2ObkqjdGOI=YbfI_9&AW2W>rO;mQrLeJ8>#4MkhsLx3x0AbuTq3+xlRRD+UXSr_6gdX^aw9AIMk)k}Dt=wf+KsV(Ljw9*jy`|mF04B33nvyn@2TxX%RRM;hrAJY1=)9+Cy04Fy5wSMdiK-YpWTT~@KGlxt5T&wi7+jSO(8j$t6q#QEBybtwjAEPLW&-@5}L29v>BLMIT1e9`gv2oX+CjpCmKc3ofF)1|AO*#~uZlV8J<HhMX_;$9My4AUWYea`_s&$G0MJJKz!eM~Cwc8g7T)(xnYq~dLGw0C>CRdecs?MUI*t}i+D?9VlDlZs)J=Qe`?sICL@y|L-M@D6XVckbse*ri!Q*M}E$|`V_gJ#WNUZ>;+^v@G4Cz<~&8^IxWfsxd9;1i)n*NB6MD^(1Gf-2ee-;B7=y41n|m^zRt!@s88T9C1J6E#Nx4OjN8cu^?V$;6FnUm}3rK~nfSe1y<GY>_0q0j^9-i>;$;SG{wGdW3v$jR#-IbjT9~k?1h7O9C$qoKIv_z|g7ccm-a=^q7uJ8rq0IyNQfW9T*Exi3QTy7p+uxm{+fymMGY?_`p^sK*?9teSxmgydWd0umz}sil%%7EI;NZx_L-`XSk&m`*UB*;mBuDm1q196dbOg3w?eDvg)WIMS$u`A2!b^g1YK};$=zf3V{}iO=cf}#xzg*b7QVWI%kJT{hZ=o)oq~W3qPr4XXB2zi$36&M2<DmUf8^V?x)F}CItNR7$kOJOxW@>MUCAf+w6nAS<4_ZBcPnK=TKR-4cx`q&>YMyF?q~>@w!Fhc=at>*Y_czs$@^lK_kbeyyUwKd~s|0F`k}2>%tgl;Y@~E40BTN&oKH(5X0!{E|&aio<WLyszU6wOdB`dorMY@x2RyXSvPoWYZqzHnIbc;N|WO7ooFET1VHQafG*OWsZAjVa$?rGHSD_YR(b~#g=l3eP_i+JDfP^ZDHE4V-l|f$y<BI9WTH<sjR+ziUxk}LmJM|J8>>oJjzkx*%>ps9SP(J%gE|%W#f{C79%LG?@Ws=YbDeimV1q61{|QGUgaw}eyb0Bv=%aV8zi;)2<_Z7<*_pyJXWUrddvl?#OR<%^46@Ji2|L!3RUWWEU6CXzKroPiORnHyEFHhQGPl`SQbe25cg+eLuO(0yFq8`%^AC0fJe8=w({1GJwAo!mYUE2*O+mUlHz4MvAwui*n3dq{h&gZ8sw0Yk#%(Y{pAi}1TNcIgtFUuhqy8cSTyRig90l+8vk(yA9vuiW2#U+r+V@3B!x;I9<zD8=0*xQEc#e;P-}F2uYFpd)^=R6ks*H(PA>}_M_s*Tf`eYy;X^z1@jHms45COxptF23W$UVyZ0BoO1G>q266U$a%{_OJl{4<1(a=0GR*5i;3O>QQRRm=U@s9NrDQjxTqU6iksVwDITlQV7~!nREbE?hLWy_VU|x2NUWp`&g66{NSHViIzLC%}~wdbg0`*yUAl%5E3G<8xSq7xtC~1}+*!#`gQ2f(1ge3Ih*!<l7fr&Z6GI9)JHvD2W5;TTgboODBDdz;)+v@Jq;_mb!d!X-FKUdm>81E@A4=h18NlyXSx9ydK*!GKMJZUVf2Y0WuNq%2@2Vx-vBe&^;!Gvj$LJRV#gf6+Rn&C(lT<_I`=L6iLG8%FPIM3LrE;9Wao_M?M6#F+0nunZR)#%~ABHdeatE%7UVKh+;e<loaC~JN_0F%O8XKPHn#aO8}eSe?93!j+e0fw!Z`WM?Dd*z&ig{^)$`#bi`nsf2=_p!RORe4+%k?l;{dCmO#^L0_AoJBm6pZ0GdmzX-Y$Acxm5}T>uAQ&T{ESz2Pop9vM6xZfoPG9K(b~s*~KdnSdyBik&}*95V6PjqGFB6`X7-Z(Vm<caDQRCPtG67EVRf%yInOl(>_lynPs%GU7L~ch6w8t>g2}E^6@M6F2+lSEpW=hvz2~@}o(UHdrA=E!1Hb+IDlVjB0~{e6N>xIX<PBk*FS;*^%wkFKxgMGIvBnvY@1{YZ#EV4liADAuBGEp2V6YhPa8o@1-J?1+e<?^jbMupOB%*Nfl~r0|4=DZdw;>{L_9>)+{;L!OUnU=x<b-dAA&EH|FXSU@i3{16@#xwy+jRTpPs+m%H!S;wyqmb;_zRAQFVy9Th5rMJstt_M3s1g{}G9H0x01_))ML2n#1&+vR}qU~M42zLX2A?jGCqK&r>J^8E%DKzz??qhsQ?D2>+ItT^$W+f`DtX>ko2E1<YtvNvH3=sP=ICIgFjmUZRB4UvKjIA%ZP+!?jKbhh;EvxNFx#uM=O1Cs}$nFI{$2o=q8L@tQKFlsCog=m#1e@l-r_h{RUFGX|;)3hvI0j)arPFrzO!a0RCnsMhN2f+_t#X;}86Syx}ZxVgmL0py2hHkQeLWP;8x@6>;(kNRT?CzmMjQr}X`)58ezbAWk$)U0zFTEu+s?yqFf$xM)^9t13q!72Z7g6L+TWDP-7G!1;L`v<S#}YtEKSvc-ZxDyh8Yt+*RR`w&Ir`Vo-}eReR3w97z1FoS9)^YUiO+q2P{+-fvKu<W&16;3HtYXOw7@_y&lk0g8v3{bDCPd-u@P-vpM^kP;{4cla_}OEO;*iDHt-BAl{y}R-@X1Q5>>x&TI^V41(h}-yN?aer?8r(bnNgI%6HJ0S`MS*CTZpY!gEZNIUH<*MbH3Zm_Vq>e-r24_lCkBjB+}WSpH{Ibfk;0Mh`%W7J>C~?s1fPIuHn~y|hn$q;xPpi9@!U`fcCMDBU;i1M`--O2Ntn7bU;j<-MAG-ea}gh$I(Ar;!`Jc-J+N`SgeuTFXv!j0_gypBSKBR)yVsX_Wl7&Qji;J><h@$Sa;CBBz=gY_Xpb_R`GJo1=^?NzV2@5oog6XuF81G(P?eNNibmV%`0^UxiVSoWRwlr2NlC5i^kt_1Gz@-$tmDL?EBno4<(EhZZ#Q{FOpGsntqP)}3{01#v?4X%3k91dKKxOmpXOQ*CCW0d(SeyB9Khz1gB@>-*5q;xcbUY(Bgfp-ui=*J`N4PDr`6<50cWYVV}8^7R1u*+3S&jg2WMq|{VGplmQq;76(yyJq&ofvl5hYmAb|CtlJ98nYQ8i$oTC*>E5(X3sRRaBoJDf6dIcrr1U8rR^RoDH0^y_oZS(jIDjSXx#Td>?XUn2$I%{wf|BR?;p@ch@BhEQmW^lDdUfV>9%1Lmgs>CJCtwl9)WXyxh-ESj=5}1dl;)aOJGK!7J|-_X~S(aUqvg^|0v?eZ#1l9F4|6!L8fR4CdTI{bP=u%R&E!ZDk%lnvK?DbfB8=<mTACU0rBJrax#rTo`dG6$hT`2^%}3^W#%{nbZ|O)xE<>l(?=;*m*wyERjwFrjQk+hmj=W^@=sen-(e<CQ5Z&3n>;ThBq!cX`n+R>#cprS6dqwUY(Jc)loX>cn;#VF+paclB~H-Tx(ardys0`Uw)#izeBCO*Wf}fe*h6CAx!2GcseA%GJ4K^_)tNE5%Ue5a!dtmzM}bb#^F>_#9)2TEdVlKm0Etw9E#wIW3uWtCN;0cvn1kd<3g~`99C;Tpg>ojv&u75BlmhGfs>Y27@^6tSz5LIq^K|3ZIHlUIIV*B5x<q@GiU)W?;C2;(WkKQ3!P}hZxWADIxRM>vsGE9<jZ+fDv~?RoUUujn+DVS=N`rs|w>qYKxy26dnN}~>Gdw;uvDLcKvf4RYo~~ZP&S)YCL5y^o0>rf1HoMyCWBBqV)aA^Ca09ZJay{I9a4RssvToe6h!Urowv;kAL$SaL3^u>dfcp?f>@XVAylbWfEXbOgFi|1Y48fgf==)v2`|(edmj*lnNVpK?<mHd@o!{fOWxAbBcCHO`u(F=F{GRYKE*VrtNpA7j&`fXQQ_QZeJZIyt?pw^D0<RbT#>QaZNJAiCLikOC*;>dezx})j2IFUp+Ds0!elvK2UZkra3;TNkfudX`Ew=oQMP?R_ibVnJ&A9G8Z(u06^z~2*sStuJ2-Nt$xpcAg&MtWi4tY@d3Yctl-|TI9bE!6WOH`0fN_Suqpc2g*i>}#14X9aTRBZOf2OCpZ2_U5NTNH)6B4rtXhY^6|RbsQ);(`Mp#0Ykb>YCuj3{~x#qX*jZzx&rU6EQpc=Qctqz$AnDQP}UwF!9CtkQ<30Jj!k%#{IQX6E&H+{Xqh&QzBL`+^WEs$QWlUP7ZAn;FNme%JNG|`+4^DSb#nnSF$sA@{KLzUpH;xN_txZ*GjccEDRrF_<RLUGk#`GDGZj*3`7#`YW{GYS6i)82-^P%(TQpSI^jCL%OrZN(vR}X+=>T?drHzbsBJRbeE>1Y!=JTM$vN~aWwqQc+5!vF{G{eFVrJdT@|jX+skZUE9+YRW#SnKAHg_~=qp%azVuG~jlsNjO3?~jcz4|Eob}<=b`ImHz>#vo+%gSVgXO;6P%nomU@0EegklLOrXc5B9>fpiTPwcevp=R_t9(W%T#df5}#FHGi4D)fCFWHY4eIoWZ@<iHS;iI*1o^@Y#g*}1+*>ZnN`JKh+BW+XuT%SB+$Z<tFdC+B;kWohT)XfewdAQQ&df>h=AccPaq(iozLdmc!h$5>cYLx}EaI63|ACw{Cff5$Z@v0s&149fk(#!`TmBc(z6}r2h1NSTxx(_m+zr<7Bt253EYY@uTU$$L6d90*@7i8>tLil;SL_BX)>utuqpO;3LzfYzmTDHTV9^6<7>VgZ{8dM!VxW*2AuYEb6<M-c+$BVi80hcvr{>kZga;s5n_*flxo>8~}$*W>p5P}X0(h}Dcgt+T~5dI|36z~chZg}db9JUqV*LU=C-AvUp&7w8W3H!<UmO5$PA}Y*833j>35Fs=fZ0RT|tv^>Qs~|I4WGy3eBd9joWPW5r>o?P<N^imL)vrsDBw~pyCdKzUf!QV~an50M#6A`j_7cm{=$=vXwm3{s5}*YW^qR!d_6-?^1P>8pV|kDDt$>+ZdSLskBQ#r{krfKX<W=}0JXb>(oA?4wu;%4qZx!t^9!8*tX?Vy7jHH=3F03;OHw=T&?Cy!!k)ZXkf9aBpu{&N(uwF!szc44*Nr<Rph~0AHlq#j%eBfKXQoz?VHB^z=bi-zzj|eA=V|t>tll!M9dioAHt>qV5g@8^m$hhMBN~v40eKq+ybkAw9W1ckI!FXRB3s|QNLi))bs-U65mQk32JB&1iU&A{aQ{stxwAbY{r4%uEjLuN0=Xd$?z18Bk^+4xL=HXa(*Zyyp!!*bssiPSX!=%g>oXKm@?06Wz1??U;9zTmi=;0axoHih8cOds!47|9Wo+wmbE^r|{#Y>LJ+oQWY@sp6bJD6m4u%E#LU;nI*2_G(lHF2TNg@vcMu&&qCbOL2<EKEH=2LYSkcGtPx2R?!Vjo?#_|5yIFc6g&^;tz>Qi|B8W&OnIv(E;W^W*~)Dq`*oyH%JqlV#^w&*K3Zp7}@CuZ-9+Fn9kNppF#x_|6Y8Oqja0q1K9yK+XmW@e@ZsfRm98$AREWMto6#}Q-U*r@@uh&CsfRH^2@~FyRV}NE?;0QS=eN($n!lbD2dZq03Yi<$%Z2A26BX>LAWmlDrPKl(GrtLlx?91Ndas*+W-l7<n3fN=02fY&EJzGKGnNJPxZhYu^ZyMHvPnuMn_wz=zOB?v{~SD^YEdOK}=xd2e7-<S;<3JC4~an`pu%dB(5dRBbpLvJg_;~ni*tsv^|OL;W~&xb{pDb<UbPjj3@l4S7?Qqbz^;D&BRpFiH&VRRd9Mrga@UR@3M~Mg2b}m2a^-Z5mE&>RWW<m5w`bZsBi|KAG)u^GQsa%<Gi|?7B|X}-YAxDojH!y=mKM)vy)%a!ek)-6^xb3O^L1LyzF+fjK<C`oT2n+ko@xqsWa6?zz8?+;Y_QJiHTu#d&jJp<Wtjy%&P)%heWVPIKHoLo$q(_HwQrJI$OU7QSlIt6>%}_#I!%ziW!(7k{594AdIC5uR%}k9tXPXqZaDY8+vk|*uxd8d4I*s<L)oEDraah;np0CNd1f7R<rNE9r0T*uoc#VhFBuC7i3H#nS(3D#%;2oFTxl1&+!vR{sGPK7S?W>L2~zmB2M=UPp{3E26M?%D=LH2WrY43X{Lb$;ov#Th;XlxX#?=7=(7b#dH#nJb={s}mTY2bN6{%p%etV<^c=b~f#-V0c!Mn|G5J_)wJ%9HQu~8x(s0tfXhqQ5(PlRQhJPPW1ovCO8v~xo^JumTj>TjF9nAl-<5(F!{OROtxuEiXXtVW4b3rM?4H7s0qK13`?HJS&(Gk~;dmbavpdGhB!-DGIjQ&z4?w6K(GT;WDekGj*tw^UgPQhT3ccp-<NV^f~8}hB6<kOz57eQwsTpsH00ok>nWT;WcGgkoP=vh<>e_rycv$DRW1iMZFK^NP$XvXrm49l^w&YUG&0-I~pG}~qh*f=gkbEoKDF%`Kz!UIswZ{uw4aF?bGb2`3Mm>?1aZ8LL)^#5ylG)88@_QeAaT%L$nT{5OU71^^gL9VA~zOwMjFex^*dl!=%7yNj`Cg}1ZmMBz<32wE+80bO$L=bnngFX>31nb!3KhgH(S4Y=PjFIcPgec`QHt$*ujO154`xzxh9O-k-u$V|YTe*_+KC;)K`CB8GVNmIwuv2(r56}w6fC?*467PzmBChzHk7?(vuGTT<$;45q<J;As%4dy7ugK!?ht$Kv5@68%S|zL226#sJBm@@&2BTX*;mXMl{!L>49<|Yd_pgW=fOhrDVlY7+DnsT(zhI_*m*$PW2dE)I!8l!b1qGm+PqC}faX}DLx+-`Dm%5`$+XHGJLbQNC%cta*+dpCKh{*M-_J=r`O{p>n7bp`Kqeyb#M;KZ`gmd5FK+NbJGp$E;z!+MM^<H37*1kC#y#s+Z9DJNQMh}~Hj1r##Rn0BROQQuGgn(2(-|VpA7TP8dAUWpl2B|5PgkX3sDMU~lE`j`b!Grpz0jRA{l5tIpx(0~Y)~4ufsrmH}>ucw327M1${2#22h;A1GcP1(|jIkmWb&q~q6_u>y$ksd$6>Pf6V-Jo=*E7whL;feas0oC(fKHXr!)xx;GW{iy%m&OqQH`3<0XAFy`e3|Nl8<wvi#Q8{+>Y~rl2%1Ozz;%XAa0`xFo`}fb5rCu02R-|-#=+foK1=}a0c_JWgJOLGqQip=q)iv!+$6|fo&#X_;jH92y_mpH>p6`&+z9aKy)8~PHf}0DuiCc&0f}$803S1mb6|OeboM??WSDQ5G(he`OSIPLF7KoVKqNYV{W<sc<k73=!IY<Ss_^$sfj~~E1`rbgpVnOalN6ZpKS#$nJKQ4Ow7KF@Fmy8jv(IVHD&05(+(oGGn)BL@}0Q(-ax>25-1RMmhN<6F5Rp_NYc)eIMR;7cS&BBO3_VkF`DBA1DJb>kp~u8OkaNV+{?%*PoRCAk#VSY69@mP@3xI{31kum{f`8t)^^lpab3k+3=?niVM1eXG;)PN;E2%tWC3g}mtw_~>R;?9Qe%u?#<@b9*nti)tcK|?ZX9Qb%GHwTi-eMktlNVQbR#*|rfnX;dPlO8Iq|wGEOsgzE_aMW4hEyc2D7CH`u3TUgg1CsK5sI~H=-^Dz;q<G>`YM4-D@pMW%!^l9kFHGn0OYkTD<+SL+Mx5U3Y--iY&5Uf5?Xv;{07^ji8uD`1azHHtZ&MwJ3!A#;)3O6s)9dV0sy4B@f&o+z}{ib~l20t*?($wx9X1(@5GJf6SFmB(~a@Eo+f?yI}3nOY3PHt0!=eO)$`MkJkN(na8}@_aT!O^sI2->jM^S$+vkQCny?VXFVq{wYnhX{8vs#0mSEMPapXaO?t@c9|4)T3ojeuP&E#|x`rb%^<=x~%w_cxnz`u4xQg2hdq8)8nQR2EZg%O?UYJ@~$Hm+*zDeiJ@=f;umou{jW0TWO_jGrNc)^kQ9`>Qib^}nOfk6U}l`eE{Qszo89FY83TTUFL&c;8zj7?pX1JBDsN=)Eh_&EkXLt&0-*1)B`uEO$cx~bf)G4cR#?05=~6V)SZ(V@cor4oRJy4HW$*WnB?%?*8=%o>VCI;F4_=E^ot1{YQPF;0c=)51|JZLJZfP4s$2jiJvo*5r68xuRFtG`|HIITfcv{C15|3f+j!82#E>`@c|ebGVm8Dm!VuE~G1}+{VV!nVDu!-Wj0ErG|&5MpjC#rssqbxqz=H1$u{~76PEI-_|%7+*fy_ChZYW1iP_5)|IAhuPLb6#qRYxVayyDq*c<+=f^I$KBc?Y;&Zq9+bh|5S_8=S4Ftvgx|FSx0L~p|hTwnD$ZBNIC(JqE7tKWN$8*xIMfwcirtHvh<N<+K#grybRdRH*&)`^3X)}@rt+Ss?s(JKCaY9Ro4ULm*OUP!>16P(18)6oSZ}*zB>ti)u`jm02Hk~k7ZU){NYwf=AoQ>ujc+qKac^$L>mcj-C9+WDU8WIQ+E-2!*I((eL?6NT?p?(y!&#BAk8pgVX^xe43x#t7R$0Xoo2vJp9y>Ky2qtjJ*2m|WtUU+l9N~-sNSfYgxjV(a@y^zB90s?hF3{-N-rJAV$GDZUU{N!x6>>fVCHY|I5k{uI(GM9~jfVkya53iEjQM|6GW@)R{L0SejDoynU$$_YzyUFD!=*b@mTzjn-u68u`*4FRaw};qhh#v@+8&vGAwlANn0|gqX{$!4dm<)_sXhLu7Y_z#Y`X`3<dVf#kecBOoe60#8OdiJ3_N_ktf`<`7tTDDXmctyaXjzuXal-o|loaPrDvU|r7Hor?j@Pr7Q@yRgoym=AJsd8pXQo^Tf<Ak}6eAbYmfnjS`>GHrs@-pp+HQ?}A;WSKAaTLcNX`l2;SIR>h-v4pq-lhwi06T61$K1#twG##C|zsml}aHtskW9)mhi2@$Txat?K$^hyv@Vpc?8Baksot*z4YV5NJR1;ob`lkV(&vl!@b^VDz!gTIithzb&uj>>}0ww)_JT&gCtMFkZEYP=OnKa>)FnJ^*)IH62qZH^T=6@qtfyaNDz^;3@fJ?@81W;CK;I!Jc>WrVy@FM_1EJnSjn)?p=R<&4OB_y6b6-aXy(NGYrVp<(N%|=>X{fi`e@tG=C!QdGNhwAcyLNBnhy-p!!{A(*SAg?kv4!)O}p!!#^6C%L9<Lq@Bw*}rekkKdyB8NAC!eKs$nVZl|WauzvJ*!zP}?~#W{9Z`6ZzneZrKEQg2v)5R690<?k5Np7@qQ&2%-u>Rpm<5->Aw^)KZWA7Li&A$vdu_j>f`*n=Zlw`ff-*6tC=+wu>@$-W*j`@FeP<p{drYn%WX);-fmGH~N#%^9_g*D$x}kZYo^ZdLf^o*J+SIpyu;7MOB+F|+a#kZZ%GDNycDSoao&$KR=uBlxd<;$5=q;4AY;wq&2(PlMnC|HP*ji}ZUc(4vCvP|M!akk^hI+yXZOECtN8jX#!`Ls~)0iOSVs`$ol@%TDuJp)sdh&NZ^R;5$eq`TNUbdKE%diMoPlMsKDVGKi0-KPJyCV_T3EQ(M&C{1hnEJ&5d)Jd}^G0)0(>OEg>Jg@q@}#hJU~!wQ0d(_V??<j5QFVkK#r?QNA7x3(^`#$_OHUPZ|qjbiNwZk$c~Hm~9dvlOz1?VI&OOQba`K%S0QAYy?>b-IOQyVhu%4lhETcx*sIQ*gU@j||$8qqDH9OMepGuM-E6`J!fIqi)%rh&Rzk+#a+yJLX$0>0n^5yfU<$L$lW3f*#254y$l$)=SZ4MgHq6v2Vf?U#{5GRecH4apj+)0z$|0a$_^@ymY4r=j-RO$zSyq9JI4SqJaBE6&VbaGF>zoxQVDXV1z^ohbmwfG4B&tV(A3EJGy${#gEEv<-^E9dIPdhfFewnEDrxaj3{^nuQ+~%@Fv>{{Agt><y}(7%`-6wL^gZ|J{SZ1ERn6AMC72}X{`{%HWFbVt*XCax}&~F{FBNG{i=~V&%W>s2#(&V%xty-q90-+S(kh+6_hD(VQOq9TB1Hv{~6S4qALv)5!>m(ku4<e-eC?v#qCoFBWHo5Xi%)z?$UeZV02<?<H4EfIIM=^d)~J`X(HN62fT3|op(mwq8PcG?=jTVXZYdG+u5(jBjPL%|MBRci+0j@#-&WgtyzUs+3)@Us8@AI&q@*jEtRk}On2<{%`{{HyjICRq$#HmNsXp2+M?xef=ty%aJSiNXcg3iZsrggBM0Kf8uzh5K^C^QEPEeqZZRKFbK{^(q%bfO&wcAk$p$Qf{@4p?+dcD+2LN(M%k^RE;`JGyP5O7p*R3~z;jrm-{U8nw(j3xryV40vVKLPFz}13_%tom9!7PlG(#5_YS3kb2^RL5Rl;Qz=<$MI@e1wnp7Hb=z=L}x?wMjlKuEcow5-ORpMR!JQhg9*ocJ+*297t+7JDY#6&Ym1@OzXKK;c(t1dy04<iLJ>7O+z$h_^`Dyej~lTOkV5ln}_`*49BmHXxQedvbt44y9JtiXN_*mq7^9JLI%5QO!#-0%${m3R5Hy%E=nw#I?{@<T3d9I&O`|=4P@i55*qz_e@?P~X=@UINQ6p_C$1r5n`6lOf9Ag<j0V1prwi1xgPU<NdF1z7$(Dd2R@hP$>p!=;-%q04LR9A%FYh;gij#m{AnX;X;)F}iqUp8_hA)kY2|22wV72^>zZZb-nV$Ugw}n1e>=<h8o*Yb8E0eWmsScS&_9s+S+|Na$Hzjn_Mx!bT=Kn>+_=hFW6JXE+5!>{Xhj)I34ee<!z#NL3iSIjf{j+Z=OSzu2)4}b0;9!?buf~6;Bp&#FHdZD4$^s5hXelKo-x>^53HB=IbAjQ*+5MOS0U6bHWWXIvYYMy&d$7*zsSoNKsER-$84Pdwm$%S-A_-DHfg$CCmB&S1q0#U29Tv{$Y~G}8f5lEQD~?kXjiCMWSvj_V)y1#j-Syn`kniMVj0Tgwugweg<@ULBUm42cxk+6fj48a&i?9PIpkx7IJl@l97QEMCW$hPUclXSEorr$rHB>vTp{N{<ZiHYFm15;bR*rScd*}9bY|&#%dIs*KzPRUUmriM?&9w%OS#ZOPRI|YqeVOHio;@IvhOOyp@aSbwocjsOS4DARD>%meTf$Q&;WJMpAwcDXU)D-Nd}Gcz-L(1D0(7A>)i>pjb@}5v+M0m**qsQP(^j9~-uNB*Z`Hov65Wj`{>WpQBp4tVl=y9JugIASJR(cl6YYGpui{1(LsA;JfUF{HE-tXTRj2&`>Rw@gF#)pALx`cBprRyhj4v-Q>IUUaBWgpBezCHF>HK1!hj%Z1_0GHM0F%I_X%1;ofo__%oASRibh_iMOGxOmE6c)*Fz@7kWH@n|R++MQ1MkL?v-T3?wS{!-1_7xQXiJm5`LN=v_Q6TIjzUq7N=X3p&0SWjJA+1U+Nfj8-|;^pf<j#}-((h3%r;)(RNB)&Wl_VPTHi5VP$kFRzT)s~9@<3tgU;Xl0r5A4{zhn-So28)+wnMzGIT}vv1eqV9Qh;_9PjdwbLjMoI&k?Saa9S_E3g}SAhG6|cD&F7;X|VmSvCt3QK#iB79#ZThc;;w|CmVysYRum-GtSHxw7eG5>ACCn80$%VWhfG;1esf6@+V;;m2Uyw3Qi_Pdd6DK)p>-_C*CUQ(g=iu}TaARn8Qb5vMMn^BvDHUoCQ$7H&o5e?x?q#&!w3!n@2cgiKPJ`6Y0K-<-##MCD9LA_}1XwfP&7(-db=ZyEe2-Bpm~FFr!bK!<Y@G#%0!7g;{rY}U0z#ZvYGLbv_3v?(LnD9GxcwgrB}z@^33gW0bAs?1~~%ztr`&ArOj99f61?upYaA?5bzt#o}$vr{|n1wqjFWukSTr^F^gl$fk3wU7~3`k8c>JY|QdJzTc44xy>Msn0$&MNNe0M{<4@CEGf;m7Y^e^qY=P+aB#!_(9^-#Qb>dYu=EXfHm8P@N($YA|Ll<R;PRMPuFwWUQxb`vkF2_g!^tte+gLSoo!52vB3Dqo4Og`nZCsKpA#pe?1+_rBoG)ZSQg`qcM0OhXkxl6h+tezXqXIOf7f%?X1UbaP`9!8qpy~?n5KS#{beOvGV`;-+DpM$tYRTQ$HL8eqr~N@cLk-2mN366zO~gTF9+55*)xKQbh~@1sMkn&j%T_XO-%6D3X||?!*tgZt>#)6uKR?&(vNdaVe!`vb~xoY82o=V-Zzn}0l`%K@oA49<;m}D@MKH!kP?3h(g-7I0tt$w<#{!9{L7dJQm4$pNM$%*#`7S|wG869Mb_MJygv{cAg$<zA_3H=!MoDbN1C`_ol#<2KSwyC`oXBzQAVr)JuJS@oB3)WB3hW4z(C!yDkV8#3Y_m}5QrXD*#tAri9oy*Lz_jd@$3lj)#?Iu*)L8$rq^&%S&Q=xEN}o8=T`qi2*o^sbu(WAOT7m|<#f_B?ah27UwbpKNy3qNOU?&cUnFtd<cFKjAb}KG5HoIn1N1Mp`kUcKO%5Xg#v%sc9V#3$dB!b85r^>=$y1>V|M1*A0(N;(Rgg<Kr2WRYbyF{tIx+&mka0jX6kQLIO}H61oXQuTm;8l1$*6eu7osN6D`ABt2X;Fr6iop`bUK;~w>I@Qt?G1}fABg8A%IR~-FWr`Ia_t+(?E23($L5fD&PvlfeKrC1s3`9lb)=6=ZUn1OZJY{4h~$`Si#*MCSBtD-P|Lj1e)nzeD;MRIZ;98fF}lk$*T?7Jwf9ppYI&l(AW^&`yV>NSOA0S=43rd2e~H|2OI|eMLf8iKw%2A)YN&o-pl#+_n2PtsIfbXb}W+#pN=8?)EG^sAlHfNjE|os4Tq05Ak#i`jP*Q^sJEe21G8?Y9KTGGI+diRS?@+##Pl2A#`WhNJW;y#kLhN9li8SoQEdos$$D{iD}>r50-4Q-b1je;STf%CFcXa@XrWMU@W6i70n^Oy*<g>_xH#w<zq6K`<Fz3U>iz=VSgX{JP#QPFPvM`qhtsVounoRamn>iEb*?^FTg%=p<>hBuT^nHMAap<|cwijP)h{y+AV|F3P~PV~k97Y4zWO66@QESLx_73{HfwaQy@FAbAb;UFw}Xnc;uQ1U1hHv#o+S}L$d06OVPb1+dSiB$Qs}Kk+iJ_<KGtoPVZ6QeC<>jkR9p94{??)D5|5-rA0Jp&(33qxK{-}GG)L0Qm>H377<OT!MV6)_RbTmE+sC(5IRS6rI^skLeio`tKPiIQ1$3PU@TvQiG(g$ntK-Km_>x7M#cP9C3n>m`8fN9wjs-*T^TC^%WW9EuNX0UMb=RxN1sea;6i!C;Tq1jGDe>AVONcfHfK35mmq!XSm%!W;o+z$Mp*+`PYST)nhcazg1;n&t+vQWk!Lo3QRZnA=0V@J+U8dsZI67)zSdnNc*%?#tA(0StNK)ujfV=wjZBP=()1j0r?WXa8M%Liesb~VCChBkIcg>(~0TFV)IQpcW>mCr65g5zAT->Rc597M_5i|55%HXkTVnW;&(|r9+u>m50YziyKq09T41mz%+OzEJz-I|$@(1^<Mp$Voj-CQqdYx+AJF-f6X%+nhV4z`C@?QbTn%9OPHHuLc2bt3fD4xCU|$u-a(6E{vK4&M%V&3-2xGx$UILmc$^){Ol^^yuv0L3@KA+Dm#8&1d?$J^v9rrPB>_QMnO`H%X+*?~)kQF9C#G^-a;TWHZgnwa}qpPJ$1s^$W13I__GI=$3z@qKI1e)Q5D{rUkh^kE@}6^4`*Rps#*L>d#!8IX)gTzUq;dpChZS!k!&3rH{l9@CpvTaKS*8h(5{fe(&Fp=+Xi_P;TX_#9*ccDH2PepDJBj{>wftfnrM(w#(a7B7H7IHOx-r`d=E-lPum75i?Bk3L>1swjceyL4PnQFNx-+I1uaeAzfr+e=*g>^lV*MBfFHXM;wn<bkeL#>?X+9yy{frX20g+mgnE_wHP_y?8RUaBnyIH?r2rtEmRq_gTZ+r{I)b6woI>rqW$6Gfbm?3Ctp-g-v@Or8$t}T<K`60tA;Ny<v~Sr$L0d<2T7~F!*B1~{!E$4s5<v%w6vRN`34M`=lj38X->a_V==}@S-_&0?`?V?XABe4^YdR>2S>RApMO4cLNl~F?_K@%s&pVto@ev*R@^woGXo$+bbO=3jV0iJ-VKk7?&o9YgMi|KP;PkFv(&J$Sq7VwI7TU>VXsPOZT?7Uhbk{=RMS+4%tzb<wLb9T|A;)&TSOr_c6(?MqvXpYv}X;d$Xs~2Dr^v%L1r|2S<PlJTLnGNiTRg{F>tf_OUp7br80u$icrF*kHY<rxIxd_UXT1Jn}~UqQV>+5Y%zFd2#rb>)zVm3-?cUii!Pj#s$&RsPgS>-2b2H_`q-3&sti0r>xU065X*%{-z<+fod>M0nf;q#Tqf?+$zTDs4*D^#AC4+Mwf(q2N-5VKKbuw7tzUVzygYkO4^32cuG*>-xI#|Ch?Dp(9ArNbI6vDqa9pY9WG8R-#X2#IVC-X!Z?x6gPhg8fw}jeP4`0TA{CzfdGvk&+Yw&W%?-v0dnL7_%>cq>)CPD3ofD9ly>$b3PU*)(7FF-;RSU&n<(-G-ti40R(MRB$BgD)}o^l+Vzy-_co3}@rpI#ayjEye3iGEaSUSgezO*&ZnSE0{-HRrnehCH%Gzb0)y8aa(D|aF$1ufSlC`snjo}CggA$3OYGNz;z=9kw5q`#Ft|i4@>@~*;UWetxL90_}T`w=CTQR{OQR>&`;%E=*J>MRnyk769T!&6*PV0QO-(MWQ;P=4LC)SSxhqr@(t#KB=03h8^gw2kv0rQ`rugX{r%G6tk4LU&^~EbS=c>2JzN2crY~xQ$iXWzcA{4lnkxup3@jGK4uewSo#zt#R#cF-N&%?YeIYQc+y4sw(|6jL5|Ilr_i|*Wb^KFG3D>F*H0s2)%=>9Kmq!O88hh1uSEkx07cI9L=2f@eRN&9!2)*#(F<l}gNp){m>@0qUa1B~FV1H6TQgZLfOg!blaIEOzid{jp6L)O^EK*rX1fPl5?u6`Geq$Coc#TDLPQKRNW0oVrw%E<kgMT8{AVzHEG^F^_XJ2m5;8iEAwxuypriMeF{>EenCGE-#<f&#frbsooX95finEmb`y~#?D!>o9+JljTK=AwBuC9pIj)(U6z1ns*g!S6`E3n0#^Fc=a|Q%CQrsM!vw#<N{3)d0QLQTAWX=~BAK^OKyB{PYK16s(QG2_(}@#&ujdp|G?(Q%#UoFM?nL%&H=w5s7)iSgMPQtZ(6)Lp+~K3Ot18$g%Z&vY~wc_D6&IM^9j38bb>eUUlSFpRwO8)5MbrTINyov&6Ger9~1{2H7hHrv#s^8fLn6lmo-ADrMcseqmvf?`4L?+g%}w*fk~YNecg(K$>(e_HsQH17DaI;K}uS9@O~x_2A%^VmV!F<NLT|rh^mYopyP^!)pThqFAb;drJ`mH11rot?A-)o)FKU**8&YyW%RP`j$NBIm9tzx`ka%7#ni@2;W5A!4y7q)B+kWkD(bVI^IWVcED)SndFTq#;DGwDJxOMzdpUhv>~RA-q=R&k{r8%8?Nz;23E<`su^fo_GIy+KY76l%s>+WjBsRkXG~8Ttr?3qmN7aTO-GHv8Z^zO6!4MvI^nLO#GX8^(=r=JQ7d{*rCImYa+OnjWor&Nm$B4-DzG0B+SreVNDiAHuV%*6HZ+tWAKD&nV>x9YCP5B+PukFB=FUO1uJ+U%$X0B6f9x-#WkcD(l(Mkaq*__sO^MP@Z>XQ$u6Z(fySb(C)4<Go0St%`O4*+mHUxkuoutcT?f`cCxRr;Udye3MBuZ7U7(mwB6^w|`ss4*-e!^BGC!2np(O9>u{CB1{_r*e_!^I?pw2_!Yz#h;R`yU_G<=klZ!LT?fHeqt1Aur1nf#fqs3>LngFj2*j`%jFq!|F-25nu%*UhQ+&l2$DZSR0?+_}J65B+kfLhR74=3lNK7-C$L8bKakM?flIq>klA+eYk0#=WYW_Dg-J8ETldw;%7FJ&XXy>obj$WgH#dM?}VHAQyQ?MEHJ!fO$L=)K!0IU;dJyJ!4;P>Gf|)|HSv-J<-Ux&2u#w<h|%w-c+}wzO(AwMsdnV6p$C&UZR$@LO^xO0K#1L3^?pZ}xo7p)0zlHu3LgvN9aXIa2YMGiOsjYT6SZPvOj&BBVSwr7QSdO=MOl_{aPYuwYl!$`4$b%feb5}xHLN632ws0{Rtwn_xu?Ey3z%9n+kkvC6|abAXb+OFGyxdTB7P8SCA%kQjr5d0?Ya}Ekur3_u04pD@)V!L3|iq`;h05~EF?Et>Y>7+V1ij#HeEL}b$tS<E_>E$(?Aqcdh)9LX6PI#Mfd_z$M88aNh&ztkyBMFKuuLgju}Nr)q+^w=m(0kH*sGzlWMp_;D_)yV8>>5PBn}X@uGU)GLiy$ZX+IDEAF-D>LF3~9ctRWCj#^WQ{!2-qb5tjG~xi%iiqKGKPdC?HlXIfWDIO`@`fg1+j})hzsNssm^4L?6qL_G)gUo3OtuKmsocLhZ2UOH7+dAN%j%amYbE#7%WAmg6Facy#L(5;Hr<fb02$&T(9&bH=6u6ZxEB;TJD#~+FxmhT>JfK)EnKh0o^|#RL8745GCO=(u=}I8;0U^>+uQH+i7{k|rKImfT!YY>TTABnBOQq|miTyalLXFGqq6M*(2moqYZv!N=slXS(PepX;5$m(e1t!U-ffP-cqfRIOcK6V`!ZYviVY<)#GW=pG3VuQ1;Tk^3gfj<vq4><*4FEMIyrEhky0tEah{R7S%VW*0c#Pn<Vf3e&R56vkyUZq*U7@&hxtttot84(xP4Se_j9oM){6N}FFdk;uF5zyicr-;lN^B$fdhMf&aQhEi{{T&!*MNlr)u?AbbBO?4k+a3;_br-Wy}&o(nm3YeCvqfWGCX91ATozp(Wd6XYOCK9i1&W1*&3lr}dZi&W_=IggTqz*qwd)dP-xcs!cbyH$-S2v448<y?<OKHy0O)vMph`oXMn!lR%zlBhmejx14Xt0U*M$ZBK!ey_sw^ayaJ(Csre>(G_Nl){Bjj%Gx8w@n;b@fs6W6KCIn_4Mtz(?92<PM9onvMK3ia%oTie9~Fd!6W<|%FDO!5l-7F2+g4VOZXnzoanA=PII^@A5zT`u9juXmD%{5ybr}$6EJX{DWWIcY441vHRIPDb%}mlN>K|%BjLN0&5v>^v<iLY|g3jXnDB?d)V4Nly$7Lm2a__zuzN^W_8xPyP8z@RdpOt>)SdyA!HQ`~-n$bzl9T^f?ez#liCp#Zv^7G1A71N`R^_Lv8PJM>x$s%wr?}~A&ue2l7G=dvjAiW<yb_pkmsn8{N|J@X+mU{p{aiLQkZ(>|`ksF~!|3);Ho=TVXAlFq~^Y$4(Lfvo{ED0Wx9|5xnj-e4cIrx?#@+r5S=gQFT25+CZM3H!Hlkw+FewK)zO|k`&<xAEx2e9G8i=y*;+;Jy*Z0$2t#m|uH$#mphWn1)Ktb68Eo=Kg<2~;R$F*qiNGVP7q@4fp=0wpnoaS=%QYs#S_mu)q%m}73U7<q3eJ69}WXw+K|nd~2`p^e`|7GPDEbDr9|t)Tb~&cG^qwp1|>1DK)Mt6@LdoYCEQxaFOZXDfy{K&$&;Ll5_D0#AHq@F0X7>u4G9U!x0PKC!f0SWe#!oYweKnuY8~bWrnMqw7ZDz7=dpN-k~CI3aSiWOT1H?scO{25IJ_fVUwHtdDUbJD89<+yAWA+>kUN4Br**g$6L08Xv#Kd}0XCX!PWDm?%B<o!*b+nTmtbfAQhlGSey;>kq7>5v;EdW_}kf-ok9_Zk{MGTqj48o+Q~de%g@;73VY*s?Ge??)o7bl_7IN`)>M<M_RE6r~RyBf@+oio06s0Bdoz5N}W!&ools!s2DGpGLlmYx7HW8lgn|vCoCWB5SN)dzEi{djAjW_Gh^Km18%_;&h6*^5XaS{RvuR7uf2%g*5!R7P-p78My8b;Cz=rIv2^~*({XsD$C{MZXx{)Ra2lU@fKVUJeM$Et<KVdTZT4B|nPqWiU&W*`(9g>J6ZW?H-2kxWNYt+#gbg%>p8Ic)7bU@(;1i=MJp$xnhYy6E+c3t5IOB2R_hrLg)b60!bs$x0!9kZ&e}m1;T*fA}h6)yBC$_-lKymHHRRGdZdSW*H06(vEQkOog!#Uxvybx5CvC0!|Nv@Gpu!T@1#NYkjyVp`MmRZ}w>~~?W_0{6jQn?~fD(K+hy5v*gjN;c(FU7Tc_P4ghJ(8y~r=(0y+99YVhPV9bj#0s?6u_uuzrwqEbs;!TG?78VxIdD~_JU-4Y**K=@UrU~w0<&78p0u!vuhsV^4bQ|0t1!28^5L^&xn{XSJ7+CbuKR`_lr%sjsST9|2Cez_Wwn(^WG}KI_}Kil_Hd?9OzvNVnCMF_6qdpdvQJZ(gJ4`p32GDK&UG&_Ef^MHJ}E7r0+x+3ulH8v~`F6vRY38iE&c1J(;4D6ZWeG+cle0-50RK3*+%H+k@p-#NB3D=vt)Oe0x;U(lArjNOmyv^hd6Wfb_}i?Pu|IVjukG^5HU^3DmAF+ocNdMRC%TBZ(nJ%hqPHz(TA3y`UY2uWV@W8mt8*rI4h`I+pIN-;qj7aYCKq@`q)zXII(6C)Eqj{5cvND#Rf*dAb~y-D;0AwAgL659KL4(tpa1(w5&Lef?xucN$jnqf&HX7MX$26(7Kf?G!`btavqOpEH_0Wzvmm^HaWE7&?1p{|!}etq?J7C<}!I?bX6M_vQ^sL^iRAUFDk(TLH2{$w5Q2JL4EXoJPVA{i^uwDlG~LKJ)e_S%RWJZP=$#9x&q=q3=v1BZ|9T-<Nte7V1INz7mPd0*yF^@Jm4jYenyr`UYB#&G=_}vMU(jt6d}{lZpX0^lx!tS2S6*<VhHDDstnp>^HTX(JDJQ?FU_id(X_M!)uusxgpKuV~AZTM@`9~z-sVVB%1K|u(={UHKR$g8Y)F-><krLLiPvzd}akMPO)<-+}#y0;sN_ZIKbgRKs&1qYQH3!3&Y2>U6T}k4JxT#v!sr2^{sgrSRM9QYh#dvcGbTfB1bg(z>){K&pNkkVw^EoI`KdTAhIh_`i_k;?y1Ff16$GcH}Zd(rl<pe$%Lssf>*o4Rov+0=|t`>$KF<NNxx>$-nz->1I*IRD{H?0djM?%Ao|iKc!AWfGx5+JSP_?XHrV7Qrc|cuy~slu%lKD*D&)<Fz#3{{>l{CECWenUt7+;De34)Y851|G)&)FG(*_#=d}QGbwI7Vxz9*^f!0pBqNrHk3<=Mo-w}#k43{@lax8JcGRyn__Z2~5Ikv#vB!K>w(5sA~cpi8l*S{O`_pGNa>5x2VuH7!jf@gHdsh}n_bJY&JZX#P*+5CpoG!6yqrGk3bH9Ms<1O88>3o{avDc?~NY{R{i}D_>UhJKh1@UIhXz{?!Ie34r#XQKE71l>h^!xivA^`fzYw;ZONWdf9mGpQk|liGHW@0XgKS#jh^_FjOYmDG4L4?X*P(^MA7gU>@sbaK`__EqC^*Dt~v;3FkwZEkVtu$PBJf(5@C~s{hV2;{a>KC&X;5ZHk}C|I8!qQc<c7mSlBG-;|Y6@o@5DDX6B&5H6f9!<K*VG5n=fMsvjmt2xx%{<>Y+hiog!DbQ-gNA%{X-<~2j7nqh#Po;-~^Y#FXD<)sn$^6dv4*@(n<CH3NL+ni_PJXp&?49wHpNove(2?jvfGD{1TS*u#@+dRHX-w9K^@-NtaXfF_yb+Oce+&{r{YLIx;j4%j1uID_e(}6oS`Qjo7q*vNVctB0+OFYR_lZ#2HIu3S+>%~y#-;ki81_djIdqwtg^ih^(nS=L?F?0^4o?pkX^J04vdrtk<dec}TU|bSU=<5sapvm~+E3^MM<e3>{ZOc+NLlA*M^>q_K=|hYhu>D>T^LSfB&||#z|&vZ5+Y1ii7S^3zKN@BF^kB7M#|~~S!$x#^?HjCPAV&sSc|Gs_#ZhI@_;Dj`Nt54*^<`~r?PQPWVGj>E9?)ZXN!o9<e$7AzlD0!b7KXT<}C*Nn~JCs7SVul5EUhmY`wn&YBI7RZ07oF*7ykS$ae@A-Q{CO0-n7?lE0`RMOuAdxgMg#Ohz+3|C)}l`D;y}Yj=k9aN7By<X+1NdDSNQGpWlCz-A_R)NO_#Eav%u4MwzGL;q#B6M$lUMSeYo<f1-))&}?)Z<zQLtCO2P%+9+PFrrUo6lPaUxEBGH$+HLdq)9(AG1BmSPo@?BuYq4m_#sH@q|(yhli>}dmQQHDNUy%f2IeTz7lz93{N(h}aZ<SU*bCM6oad|}9U+U{gyL`ecrL`Nv$qzB#5V0G#H3_|KkDH{$~o!x-dnU==va8a)Le00A!yfSuAC`6D7dvx7d;A~h3q3??2)z*vfL)7iw5`s{0?no-I(Xz=wPN2gFc^R8c-zj5&miId71yLhO7L4kt`B|6xsG9Av^>XukI92hn-Ly-c5>*z{}V`?alRcmG!f@baVV=7N4*sZxvq8j>Qn!^f>|un7P$K&03+ZCaT|BPG!4~2brd9n3ytT+_=`MScg&iy62vX0)0<)jKucrU0SJRh<mL!zUiIfHwx<SaCMaidi`m!<DtE%CLT6hb6W?oDF}C~d)#X?nq+XyZGG~uGyo{;`vv;4q`Iy6^v(e#f1&d!?H7JAHMiG$otg?=LG&pgct&xRI{P|hKoivK@b{SY2b1*gUn@XKI_sHCeM$j~mj6{U?#y!jx%&ac#SIJ~a^%bd$lg^Nsj5EUO_!T~+`7DnWBJ$18DLtTmNQLjTs#GlZ)3;v-G!A?;aSoyGo<$|{^<|+<x6oRg2cI?WyZjcL6V}9iM8O$nz65KjFBs|@r(XD4dA;g@u0rfo6V~G#Ik)POv_sq`Mjs(;WgYz^VU#yXCVJI%D|i~g<H_?90@h?&MfrF?OWPJ<v`;>JdHk`@?o3}xr;D9plv&|x(=~yz)$7ByQ$iW5m4TY-a>B}p2hzM^*w{hmd2Wk<_G!d!)2dMYfKuV)Y>iVuOmzwh4uPDq%O9TzB8O}M_RRE-^@0GG=$WMi7RObRRImGllviK<U*De+!2ueHGrd|!i0E0dN`bCcLM?P!_5MS(*`?m6r-;H7qQuXgQsIm8~Ar=uhVY;{we6FF^EvFJ!q6Wf+YYrB1+}qYRJM2A<{1EwP6kKoCBC<uPdetFsyTtTXF6Pb@O>woV~>-yiL~Xc<42aGZ>gXx#2>e!bRko&C0kDSDv}RX#O@%AQ%3q0RPAlz#X<;WQlC3(iUzonX=6_<S`7i3MnTRsVQXg1!>fM^J127d1NNuZNNR3TTnhZ;xH^QI3pG>S5}m&iG$T%L^FNo0;<oqjhKe!L74|7AmufFD;g~2M$z=I-l2g`X*JUYKLaph4D@$5x!Qz@UjD}#5@DXjFR6^7Z!oK$wSZHG_A+osc1qD+_<*KK?0N8e0aRei;j`}6-gn7aBAlQjHmO!sQ7>>3k+xh85N|RBpv2J840Hz$zu4G%pYogKLx=-1^0Z5Rq9>TELUmtUUR2P2s7^&OkKWtbuJM_gpSBO2$`|}6CR7#$)QR)o7L^+a1NAJ6nkI}kVM=S|0#ds7k1vYKeMK!A?ejquM>%Nsw@eB6CG2e7bPV(`sn<cd8hwrJaI|ybs4X)9{%tG$nmUMWZ%q$1+m!3?UOl!FXS+l+0KFKmVOtkJuYLueve*?dM!(!Yeex3Sjx=@CbbdZ*2{;Mv!i<QP*V5reXl7x^X!6GpoKqf{-XCnZ^Xlq!m`%m_`L|;KgJLG+ruM;)t-^25*TfP}i?I;RswOr5pO;De5f+Ln^esgKR<$XzV`S!NHDv=3kBV=o8HhNBY{aS^8qU!)9fOjZHv(<YoM)k1WOExu#%Ocm@;)MAntZ4FPNHT%*$ZpUA=AO>3@cG04?Ra$I7mTvWELyY^$SJ%vm`JcP-I62oyXsU0??*XR4xlP)!MkE$YIylWIRF~R9Fk7n{4>vdxm3)nOFynV@8o+>f&=<7=_>{ORAB<?gbR8KteGIn=<q95rkvk1n~bGt5_k0z?Ld_U~Z0YJ~GMJmYBzmct+M`XwIf6HgZ9I(nH&W<jRov-bO5vncroF!0Q{$y=g!dzBxm@0bB1{1#Pq%H=8;GkSN}MD&!*#y7bf7c$zC>gSE?E@fPcs(Kb(RCRUKOv7L1oAzgvPEu29`;vVe2{$AOtLj{YHxvG$}EpN<_+(n_35JIanrnhX_tVRk_ss(0_yJ+gf6TE=K+tp7ZLgo>%w6)RlxHj=`i)Sh765w`(TY38!Uy&@9hKSDmM^gNU8#F!QOZueLCZ6!eY-QALv9#t-SxmC!?cyQh8lznM%V9og5cq%y?DZ(e3!CK*xbSBK#tOuL2s)`PeY7hzKPW%YjbgV(Ufrq(k$^*v0|n;Dgum!owEE4ga3ECOe3`1=xfY{;P^)*+zX+1VMDvLz(LviW5L2pC2Frw7N0|MU-atb#MmkLnJP8#_|3ywiHa+QRIA+G0)Yjnf2Q&HH@?y?}bxoPl0YcUeqn5wj{HekDbOEef7Edq2KhOZJX_KMzz|{V=b%*~DtT<rxnV)}*WgPLt$pl*o;Q2&9)KdY9kb`Rz2e!?@FU2afhC&KpwCL5n(kxYp3v_KTuqKbu2bu8R&XF2l-sZs(S~_&SSX7!BLzQ!XYO(12LJEk2>-?Rlw*bIQ#A}M(9`x{y*l{+7Q!=dq{%uvzPMnBw2R9LaIHrwSYu|`u;FAhsQ7!LkH<%KYdlM{vwfA2mg*})H2;q6Hopluv)R2@*gO@BCkd(cT+SO|{>0n%K3n6HL1IAGaKR*En5@En0>F@BTYS>ZMlvHw9fijrd?ODeh*fHFJ^m!8E^=n;p7Q;xlkL6X<3**_-R7P5Bl626{E}B{SYm@CC<fW8kI1OH(>feH>s>3VHte>Deo@~ZX0fJC+=)n&^M9jim-FhFbgLlXjFWhgYa<m0|Ka=c>==&x}#E;-WkQJdZF`o?2df|HO?^i4(u%&pvbO%u-0P{U>J-|_Nlvfwm8OW68&bfPt@iieJsLKLW?Mr6#jr_%3vgAKS+_!k}_UN?*d~7Kb!VR*s4Kp~~W5pdiLvBD(R!SZ-(F!@vl5P>zulZ8aLkPUC3np_PB*>ac;(<feb0g@9U3WhrukHW|40y>ydJ4bQ+$4CTgqa-h<=}3FaoIewY)V0836H+69!jU{I65mKf!ftAhQ9G<d9P0AXK<RofOS0i7INjvuz+2D-%^}j?V797amsSfRhY+t&RScRrfvI~0v8F7#`x)Mg9)v`^?_Qq>r@N5B|^*vzP(WRZP?ZepKuLJgNQ0V87j@Ee<@*=D~7vaT(CBrS8_G#RSfUJ;_Pdgo+OdmpfKs#_ZCwEHZt1FME`oF&W>kcDQFAFN&p1$y5C?8hpv@=U*enPt|qml0NU@{@sXJL^s{)IbI^imG=Aht9oDGSjs4)gycqH?<-iS=aC(Rp6LeIV9U3cyup$qgaarVd#k>ZpVM5XVmHTb93V*Ohn32q~y&p!~k*(jkVBcXH3>-;|DOeQ)Bi$%%BjiX_OY5Mfnv=u<;a?l2;^T4gVM2?kS=kpM3yJo1GfFDFPlwG=%Q<?Isp1kxC!tpz#&&S*^;}ac5w5a0yJnrD#PFBSWf|>QV<g(j2WIvv5yaSD!hr1aS|FW=&4sX#&SWeogHhqAPxIk3Gu9w<P5}U`v0g3AGGZ=dStGJ=l4Xe*ZxPIS#isbrSXhuI1!KtJ@GSjh8e$)HalxnPCu#U+jATv$ra+D~h!WxxJ^BVcdwYscc6r3Y+}8-IT>JlhgDC0*EI2pgAuCrp2zCv8{pV{a%dbmDTw%|47_y5}F2Z%g1oA#)6EJ%b8Bn$n1j|)PEjoYdE#S|#12#hX`90%~;vL+)MUC(l03adHjWKO3h-0#uAckS@ieb|NQ67!miqLSo!CFOBQLr%LdKOU2s8$3?dGW(KIwE>smw2V5(3bvYAA}5PD~dSYK6r)=1*>Y&J%fe92=Z8H0h)ks@6rI0{RCHK<!FerShjTa2)VUl(9iD!HE-eCGBxuNrqHiZmk}}c_IH=qK_BtE^VW=uE3JP^SeEbH!c;Pgzou$YRdLzwB0IH9z5cs&{oz-5$B$kh%Hp=`MA_D~OPsq=6EbLfdN?*dm=}poRZ;+QyZHMqG3b#w5123~$&G(C?PN(S{YsO>2taGq;g(0hrnN_#Ogq@1mcUN=Nazhwb(s%7+;K;AYoFe-&s+Rj&rBcWDv(BYj!*EWJ9(Q9H@kyXAjm9{@QeO2`BR&sIGj_`;m9XaO@{mu*w)n;5cNLp74>7;q-1BzA%f>Olz&Q9&FPgIx=ypR|4uE1Ij`N`ra9sORFLDF?4GUQO;SzuvB#q@_XSOAU!ozTDa|5rS`E@`60DFP(tydY!O$&SVo1ZOLn`-J`ppD7Uz2)6E1kH5*+<-$x>|8wrL*5;OIzKWyS}>s=C&T7MeOh~t&R{dkM3)IN4gZsQBG}?5ebwr9lD;{lMI%hnk=6B*@{wdE+yWK>?tAsgLg6zsSB<k>oNG)ro{fi(wXEt_^q4K8Rz<Hi5sP;E*)E)1XfP}J-m;GoXD6YNXlR)+!%0|pbdjJ5h9-Z-*8$q??GRdycAC%p@l$h7z8E`<VtT9BapU4T}L>G02)G=+@`yRy`=)zY?S$&apjSN(8|F}BzYBa{c$p4Va(pSYQ(*5&v1NaRg%t!g9ylia-xV}4w}`fnS-n5MZY5(5wTMmDPo$OA6~2NkfV({+<Q^jxR=OQI;N!qXS=h3o$h*lK6RcOXd2G-IBzbPs;hlt@1SUX>cDwBMyl1zVmrckg=W{Nf|TFWbvusqQl5b5JZ^9%3jy$ixH%PV8?87E7SpI$5-<7~O6?pqDxR@ajyS8vEo3tUmk&2lCXSV{%%g}Def4z9Xjkz7_=>EP6e)m?obC?zg6p_(c9dH_-8{0V1fs!qgmlNdVJqAyX`$>h{)wn=e%KaX-f|ZpIS`xG??v}$zMxW>XesdgdZ<@9LW=OUDGG3=gL*XWRW<N5ex8F|mC2Y!bVgv|m|5&QMZ#hN@=wjP$R2VTyF~%ach#2!#ja%qLEWLv%6#+r`<<N5oaHmohDxmgeWfoO`pPh;g!qD8&ztEWAjDF7c}Njuiv5ItFBxhLNv$`Ri*pWpSHh#`HEz7S4j!q}voWFIL=8}XcGAN6?r<okpzC5MfczcE#8`{u8jJfBpqEsHnb((4{zO%1EZ4^`{iS6TB}AvRphLfT@UiVNGgzL<>hPZ#LC)}ZSOT}cD<h*t;OQT3o9wGiy#BxQ+ST`w5r|g*TFFh$Ox%23#bNFzo~27y^a@M1{+u56biJXI4&o-E{Um=JuW#iTkYuZ`UH*Jhq9;YUsE4Y1uT}bia)B!mck`{B8o;7si$11H9w3NoH7o6$58ntR!?7C-!3Y{5+Xd$gDB7-dLWuE*P{x;sh*oV4hPWCw6OA<c5c9fIWt)RtG1ay^Rhcz=MoP6p!rp&qHe|Wj!}&jpFD>c^4G?aoVJNW5#{A7VMz2+_v_&$snB=zQ+B;dPS4?FNLhLjO=iWh<rRszfv^;us!`NZS{S+zs({NZUCq%>gX}6S5q@Dk=1zgAfyS@C<-d8v^s(%B~55gG`VQAT{X*9FBCl>jfOyF0nbyi@KkiKLEyxb{pQ6*LI+Zq#=6}CVvb4{p0L;SjLd=N<&^^qmSqD5oHX$Wq~frMg62)2NKcEbx;634f2(hLiK=9qd?%q>OkOU-G`-SmNHB|WL>i#fEgO{v-a341!r6IX06G$aAg2O6tlPsN7T-f9NHIop}rL(`esvFJef4##|_zF^PiYeeOV-K5_JcVEfD(Uq-wD_S_GJ@pAdnKpY3-Nh;Oa(Jm!N&X;?)jfn>l9e@m8rGmQIVsr%5!AdF;OYO-sT}RvzXo-xU?(7Mpo$_x&*`U5*}SfmitZB+K5H<)MyPnzVJF@9{7C0s$eaTU7<>bRgC$o&wIW3ed9z=@(t11;D^)=G=9cM`Ki`VU4Yq6;4$oH1k5OUJ(MWq0p|gA8eTx8HsE}<v91v~B)y}csZCA7Q8EYuh8)9)SF4`jLjD${nE!<~})ABf%Fqi!fs~}_V#0Skk2d(|RZu!U@Co+mgi_8{GhR840uIUe6hQF|Q>|WB^Q8KboN%HNV$8l7Akaz-r98dFe?v)_TBq62`T#uw{^dVTea+0VS^-C|UCC6Ozi7hLf#qNw>1#j`Jf??i0Rc#5#?Z~Wwv`Pr%YarwqelP5oxzsGgdP;3Fjx6pVU}1Gi2HP!y-AP`U3eL$)yQvZ3h$p04`EA)_rT3=~g_muXLjiPGk<#5`Aivk3=-SGmcSG2B0-I=4hz-}elp{h@7^;(RFKM@$7+UhN077_vjkhM%VrXWeT|_qzi_h;}hK;_c9$jhc2FPlDYso_*6<_yVz67VA!z%i*5deLwj4h>11TcEdVKkT6NJUY1m*<Rsu{$K%Wt71Urj%vV=}DK9)w<^H>F)u}ys<Ibp8Z6z&Ba_sFm`C$f+kIUa1~4W%hW)vXBjof6z-zC&Ljq70FLJakJafMj*+M7Xx(14RGX^KRdV!ztyxRvfX-!-8}@~G0>f`65&B{JUGl@ZDCLVfu>dTL+RPc!-G;6<UnPke8|kebD|DbJ2`zG`%C~WL-+<%Gy1WoC3hEyBSFOU%mY@|r2h3bKJ!h5)^@^Jrm((0|mK$fTrX5LHm4Ve-p`dZvm-T^z6Nd#L^`#>Of+|txy_HparO>e4Z;H972;r!htfGr`_~2V>pB(Ba>}sYZ!D5PwRQ3`MzO>XO&R#=qrpooqw49Y6;hmjPUyXXR188YUivKy_v`prjXLt#j&ZS@rQ2pjR)XTgN3Ds4j=ca!{`D4UZCD0giKX}>DvMQnrZ|mh#|CLMOdS*mch=PJ;g#PNG--N)K{gbyhdjgUbbBoWBuaLIP{6({XK@hUfwZEGh-OM@S@_bs~SYgqVI|JC2$VNE=S5ZIw-9?e2dgV?^sqLrH1v96X3(zlg*)O!7C^xouPT;ASBW7%YAu~8t&<ts^;P{~NzYER$h?Yj!d8`dVr<}|+s^*J(<<8r?25M<|P$yL52PAh9%2teWbRRThkM7ja>PPAW%lBVi-@uK#C-7{4bny{^_NhM*<8>d^UPGVBaswvE{D2yUM4I%nMP<R;i3(D_+nLXhc~){;qmFqc96pcf5G+cN6V~=o0!S^+q#4Q%G#Yx1e!@=J8ykja!SW3%&4__B<$rJQaU`%xqSUknqJ<}FJPoYQJ4MvMS(}l##AsWH_GJ%)=Xwf2*lFHq{tNvNf7$7uC?=mD+~VH+$6Zd@xe{Mc<uET~y_3tL0Ti8n$cQU)Y-#Sj^SUlWome1MH6-g;*5ljYD4l@M1HWL?$x1F;nJMYkVFicIdHE@&$P|SUSz;RfqY;karC{Y1M?7Q5AmRQrLyxmK9ZT#Gg;&2sj(@syw|Ki6g13^B@)wz2)nl?k->C9s>$J!m##PHLEfrFZVv!|SMa%Q*M%f_0mJ<c`6d$`j0Tm@m2t@-f2vqc#nUnIMPH6({6+p7uG9Ib9!dES{)-EX5bOrPTN|mB!?sanX_E6^s_PjB}hcYn+)k>hA_J;Qdh!7sD(F2r(Q*l|J>4pbpqm_q{($JczwbrdkHBLk{Xng>YrMcNE<kmW}^Z9YzCSijR9zD7`Xnybt;He;Zv#sd@T|wG^?y{K?uwLP#i_kki!tN})2>(|=&3xtu5z)U0AP?=K(FBg-g_lRYME@GPXB!XlI9|*7tyxe1ooSQ!jdf~+t~8e){83|)ZE{d+wc?{<$+OnVW_r&NBO@@|bPfdB8HcB>lK@p!V^u{o6Pc#b<<Wja%(!sW6-m&5m!8;q_CAD5cz0THeN*e5xLd*FKP;f+&7a;|TT&OjPV*VApiOeu*%%zB))b-_cL$z4_4-2H8D!-jdXp=Z(fD`~-Ss<MZ=XY?l0aK1TvnXXGM<lb!>PAOV<c>lNE{+P5(B~VcBlrO0u`20*p4S;)@X*g-|)LMZU<-H3-0i0_PL5h=jZZ^&Fb(*y>A)33<2cJNT#zQj~kY)OhJa!P1b*!SB!e?QooQWrPTkuRC8M0`=v|v0zc1`6}2DA8BzUKIi!}3LC}HJ57<I+lw5e%)mjuMJ`re*dp^K6ozYHn|8og!#QuglT%-d3!`n-;(lNH-^%SgM!AceMn*T`%JpHWUA+=S8<2!UkXUVtA2HOkHBxkAcq`I@=VfIOJ(weOd&ztWrQVr8}WUI$3BV?s7)k%a(Eob4-#jzUv`8Ad=$YI_+a9T=+#0aikOi${QyK;R4iPGW%Wo{9x6T*pw1huD@(P>xs_>zX}pUo55jjV8L->rVRBC5--Q0hIj`yr}agU`7_VyfO;p1Q$z2L#JKB4^n}^}=pZsd<U+9BR)B?%4lS>}H>(`U)NczTFi37AXQO-vJ@X^bJNha9#50LKSUjHA_0vq&D%fICEKp_*1ShfT&f1rx-Qq;*RH=m;VQ5Cr?9Y&D^uHdS5GjA#~QUW|=jlUmcopk{L^q<9@-yd;onJpB1|xT+sN6I~01r_;}h2CL&!r9*oLbX}h){6k-+?9zRiY09twhFmn^5F?~B4RliJzZE-mberyn-Nn6BgaL9TF&FoXu&dsZ5LUtxv?xol!${H;?Y7`_$qSrBw9<lKXUykMxqq>N;96yBIx<!hX$J45poOUunvBkA~D|KmV5Vi7JQ>a$U*rAk1J}8d7sLjiBvAjCjPa{-YCvd`wumD%2Z|;;exv6MlI@d_ubBe&wkTK^EpnL3U_i$Ix7nx|EQb#-jLM2=uESA|UB3(~|g@#*61Z8{F4UG_qvumwNP@IYGBmV@*C6qy<Pxc42cobrzn^QpW3^7D%Q3x}Rux)MEL8XolasdP-Ol4g#W<@~;LVkKtT({PAZm+8q<aFQ86Fds=e3|k^DtcRusw4@4xqrmnN)M~Qk>V5Mp{FM9FIPt-K;DIdP%7z2hpy<SCzrYCnKBYjE0jCxWG#mGi=v-#iJ9lG0#w4&`SEiLf&zPZx`{#Zmjpq7vnYEkcEVZZ30r5P{2TWJn3Qb&X0!n3q@_q46iqg^T#nZ)^9y4lREM)ipvWC)1ofsNi6MM=z3dLN{_{g748?>#u3#HXD3C~wpET}c48_}EfnKiqp6Pb`zAJW}AU$IE)m5-ss;yy)hcyK*EME)p*X^`A5N316=N%wc!i9I_^*^Stf~3JJl@a^t1$_*!6@(X)kzYtDKgFbNaa9HJF2g`J*I~XV`}SDYtTe5=dy9=A#cA@BwaO>ltZ&U_nr&k}nOwat2yE`#14OGS`OSvhI<9bC=gEs}pubu05FgjJOZkA=HVj4*fxyt(YTrkwQ@Cv@?hMGuZE_#6Dr{ZDj0isBn%6|=?u$hS1V1ixPL1WFUsob33R5FLb_bMOtz%3zK-@4~O)-O_o(|qb?vdCK?dc`9+nF!byZ`hH>8qe~yZ43&H*ssCf4C;Pc164vBLNf^ZwEHjl|M20U`Rs3Kf$QBc6w`$v<4PiOKdYKGt6owd&Cx;aMevRXp!63pR#TUtDDJlHTkd&!x`3h&%S&=ol2~)ref{qf%{2S^c|rc1{M(+;?Y7e#>EOw$*3y&n5gCBtK|gBIUPq7L+Xnj|0b)@Lz;r5k6fC(lU;Bt2zd0<(uE2$^JY=^%{+fuGX-8A_EqeEtlfU+@-^x+kI|wxJ@O^U?HSQ8Fap_z6bG4Pb{_sXe$UDcl9Gk+qJC}xn?r<37nhK;x>NrXT0h)T{cROaaU7Ea+nOQ)6T=KqnA_}U7jA(8P+M+zVMzdftmC7@V_U>mtlrIlPuD#i0vGVR9%2Sc5%Ycr+(NXBO!|#NVHt!xC3IjM;^3FaT*93VN>gs%M<sKe>{&mLp<AcqJ^4H&DLh&b-`ACj1${&8Txtrg7Eg+IFz$^q_Ba##K#_lQtyw5?ywVA7a{g5zGQzF{k8px*7bU*>egJo?$nt;HC&GD7XP<8lT3Wy{@At*DK6f=&nBDQ^02;A-METVx!IinH9&P$V(EiOKb+~-}l|)C%=}bHIBJXkrQWc2a^k6YeK$A>w31UFRKP|$J+3jC$nDY;L%k+jISRnkI-<fHUX4Y?OZ}%(EE#8p(soir9a(O-PulpU&mlfpe__zRENR|acj{{wfkd-~2kGw7$nkHxB7Vw`fF21n@72$FkKxJ`xEGOxxmx*6**NsKD1DyO5`g4~*Jm2-j^2k_(>u+3c-<p}{`9xIrMawvZoSi=#(H(}m@$<DziGfZKkbpB2>hp46{Z|*$QDJApBZd;K#9CPD0#e9R8N8R`M82)G5tbz9)n%5&-&|o+gSn``5>K97M=y#r?^0U--dSNr5>>M%Kea^J1~oxiFz2G&QJz<qtlTz$XQuBLWL#WE77WTWUIB(-;fv&|*%}q4U=X=|k}}yk$-lgX<SB}zB!KQ8SBMSXQ(Pb-vBT^kXvfM0<$sY={(?E`C%&cQx9PVx&E@!LVpph&VE`DE{WJu1+$VSaW9JByzj7(6@{@yz>@EJcmm{Rs5gs<0zSwB0DaBf>-)XHa=QnSy4)Zh_JU~R)sIU5NpOQeeG|MOueFZ_XXcUHJ9Q$kf5#KCH5@;!r%GF%TnD6H^Gp%!lmsJJmeDdlXu;L+k!z$b(AI0%HQRa<V=7x7sTcUPsch`H4`kkV-ilgp}8$(`McO=d$mWc181K8!j!iyeiEzK?%A5HI+8>cdgT&>}KiUyz&`FXgeSz9l`=uOII0~bW0UXqr8lHwv)ZNHPLRv!m`x3Vg6mbGOW)a!AX@fuH#AOxOq0eu<{;Hza*Dh(eSx(X*%eI=ob^ys$j>(p`&(&aDNy|=}YleuufPx-g@!AkbeCS}kNd67D&M>yeu{K$&LsQZPCN)IU`(5M)dX#W6gqv6(Auaf=9NaXa`3a-Pj8j`AW1-)ElSg-i4OP~m9H_MKiLURM*3cUK)p|>ozSLlJ-^cnl}?;+J@WbjnGNtXa3i6wClWiU0HCHCczJ(9UcIce?9a-sS9FX{=SAiv&u1nDV~f>^0RRFlFVzt8=ek|R&ip|7()pd+}n%SC5G%E{4AM0Cp4wdyCQnPojxug1eR;eJ*pLVpHa)P&w_xF_%ep`={*14Vg)fC%IsPmhy`m(4PiG-?36)q$|Ii(W6~*E|GM(f>_*wMGEPJsvqHxR@}NIb9RdEFdf{u)A9@M+=VI7Syi{JG_o<$NWtO*PaIghMyhSU;tKoNuv)a6$Zs89oxMlzPcNu^e%0NFTH|RTufMJw`;1Og|)ieP=mt~J^)&xpu*zS{Rsp^Ph@hdLnW##(-O@KyQV>;H}DWU*9?tmry9`{90VDrn0vt&$1C6#+?Ah(2Nt74H6K@PQ`8EK3JEg@wPCt1*Vs7%RpI_P8)XfbLhDciMK?x*epZq++1kC}?y_Xo*JDClI!xZB@GlCOzh0#piwfX2<eXvmT}GBp7PPu=|I}|a>5Z`;&`%T(-SLEr8TgnG34ct1Tg`9js+4w|9P!({-ny+G{~^aEZ~u1^1Z5X2K5?zrWE3kD3$U`*Wm}{R6b9k9a=6>x+v9twez*wsv*e&KbXm4cs3k?sLc_KZH>DrugTL0+pSlL*+U+4~b$dNTT2E@Bc1;BxsBf2);VpP-V7^Q^I=%hWWLeb3FQdHCtr1}-6tgDzqx~PUw|{_xgHt75VLm=E`HgO6^S6JT?*DrmT({UsLZb&Gi!67J28)MbKYqsxaE`-=g5GWF;#)q}!fpyWf0`t}9RGn5B7=qB(*Uj^z5h1nwn9^g7iBd*QlAAok&Xn>!^W+#+{%2VS}CgiGH^#o1ORNdfz-`S)Q-;eqlSW?p+$`V(ix+$CR&N>j1(vN7YIY<Pt^1<T$s8-<86<Q{f=RaAN7+i0FS|K(dfI_@(~aXiYE@gkj=+Clm{U7=q5vJMIvF|PJV?{A1Y2bXkepd3DgPX^|-+XNAg@Q<c`5rfd%wkIfk^2DPqImKbgj9GS3DcPB{hR=A|7Hzk#Eb^8AM|&=^x}Dn7aR%INkd=8hhE9>0@nOlUr{ZfXOM&o;7$KuerYrE~V`3kndg^qO5%3-_P_$PbaHBFr>}du0B-^?SFP-70e%t)^+_B#^R#n!8L-RW&jokIr|YRNXbkA#M_U?*7G<>U)jV7wT2Cb1B6g|D-u5tFjcdQBcmD7a77sgVGk=b5rFDvpH@0IUyI3tv8oB7^^dBBO<T5&ZxyK4vI9E&xNC?59j)N7nOnpWkym%Fg#vmFMC+hHV*gA8idCK6neF;^RaC_7~}^hyV0Hed1wN=GC06|C<NlFHmMOy;Qh5wCo2EazR2olp~Qzla*BWr@ehn+ap6P>vE*aYJ>_&Agg#Ei@`#uLL!DO@lNn1e7QFYT_c<$DsGo-H&E8g&I~F(_#<C5(xB@tLZ)6;FRKAL|IE(>6U)|Yqk=ONkH!HHYvLqK%GL^EfA(W4SVHQHpOi)bxbcWNBm(1dqmvb4hehrsyfP{1SJk;c--o^LMp|}>snu_MH5}(!|XGj&Ii7PQMrTgJH9LWo|T__Xn@?8+sr;Mg$vrozf$>3%&<gb={SIOTBpnMyi2?jf0!?#adADdZ-rYrmZ9D9U^_2i9A7Pxlov8t?JUc{UHB{Q@_JgSrRjy3EYuwDwsQen$P+0Q4u90IOV`vSz7tYjVRA@Kl9b^9t*Gd0xDrLG5EJO1FF=@fFi!ty(XHg7Py&aMJ%8Q6Imi6D9};fS#S&>h4C<TFLH`Vzu4hAK<@6OgbXEL*3<yQ2R>=VykCDdM#vab{XciFO`AE&3c{BWF&_L(+y|uSTOseGVvA7)rDX?7_+_e+s6%5?BBP$2P0LXZ@Dv$LbA3H*?@;lpqkvCORhuH?!kQRH+b4nmgo_D9mM6o`W+IWqp6PWI_5SOw<=}^YSP}zxY&>w8`FH`SK!d*WN%sN(pu84@tnaZ5`DO{jTDbe!E>=QdiRiXQ9*VWF)q03O@PUPNLBA`mNcQgfp#cI-D{+JK5oA?N6`OLAV^|c=t4Y!oyo?SLPzE^1!&^k#oSFw52T{Yqn&5nKp6c0-K}fU|F#pit;<M0PQ&21)NriuU@v5*Q>;1s^}z>VjGwR;&q-Ato5~HtleN~=J#&B#CZXbYVCaoKg)+eB@KZ3iP#LLbA#|5rTW`do`-70xF}I%`P-h$J#SW~h0E8~_+)FQ(4eis--W($am-26Pq6zEW(p+b!yY>P+bB=F@^Z{r-6kkYv~^VjgpbwN%iI|XWJPorjaqHr2OS`t1gJUq>p^HP!0GZpEswk(>cmFj;Y3Td&!o**kz+@xAXVAF^^lP&r0DUOj(}p0|L6N5wrIzZcl~}kZy9=vTE>D<AcO2wA#i(XSClQUe8ucKwe3to6NtuC<J>4g_{eF&r8+f;9$GghVaXcZ?tF{ob1{D_&{uCEKHA~FLs&{k?YA0FHBcz79>fOI<rbb$i!V6`=2MV#uCy4?otDu$4;UxS^yzfLNfO;@;bMOmRv+u}!6o!-A3ppFUV{?q5+NjSF9zM@^$avx)rlt=XfiU8wK7|F9A2V9I#hGpKpBrgr-<({<~etq<%AeoxhZ%0wOir{D^<`lnRm^6%Fi%fI~>!f;6m4P74608yh`MFfP0SRbevkL0fI{zuwS{b#FXx=*svKp1&~m39doNW6915Bt;{efT;*4<;jb{M$v?udd3Owm+M{R&u0V%9h+^%D?AA|RrrO(?Fi+=p5sgRVU1Wsj_-xM_@%=RS<iCW>d`x^CPpQa;@5WV01O>3sTc)GzE;SGh-fqH2Ya5viKl>bcS6aVDdS0jQt@|kMLPhO(bLDUC9A;|GeUH+e+fW3W!e%8}GhoWKHoyH3S$Tg&bl&8<!E5v|pCR~ZTc+pO#aL2QHQAjXs6bWfNqHarLVlJ`IlZ6tXW27TDU(6h6F*CHQMOEKH|}}zW2?awh=-d5YB+Re)4HQ`@lXeVX+1vx7(0kfxtrrs6IEiyRAR(eDJ273A#*smF4<(&ZbLjf4(}2<TNO~UpN6y)%(4uCzdNqo1s`9M0vDOce*<>^4rDQC4^RE}xsGz>dNJI~dUVpvamqvwo_l~wP#a@5TulpwVZF>V0l8~$Wb)EfzN}lDt}_e7-0Mp!7y7VY68yXRkw@kQBXc2=0cgfhMWs_w=Rqf(Z%uIy?goW{hQ^sNBlchOGj3uIcD39NrUMoBQ&w}CW=U^eQkZLs7tOq}nt!>_q;-?+6&DAdA9V|un`nu!nPqV=#8ZCS9^Peim+|zen*ajKF;rdepCy%49hk7Po5zl2evY5}E67Jj{4toFP5Pck@}7P+J0%%jvoESrhuUf8^b|iO#Sg%YDUAas8N7zC`A+(ez<>N(O&+$1T8irn5!y}dJ9ZVG@)DiPp&Wj-?D-iz#1=09Hf9u>{8os9Nf>D!8jfL>!{b^p*&lzX<&WkNkgu)&XiyPZ{;CGA;;s^<_E&A7p@^}&i;4|HX$v?;f~YKnucy%&|G<y26n9dafU2v!NhX-+H(PdUOc}y)Ec@Q7`6Qw_owD?0#g^1#NWJMsj0@Ca*wfG?5BI?FZ{e+8Q?K1i`Khc>8TsnEXgq)-C(SPm#pi`N$Ht5+dMjx|sq0?K%Z#8L*?C-;0n6*py6(6~(EEUKv_lyqSFAVg-kd`~&`4>W$sP7sOyS|HR7_C4ADt-(khepBM5~9NbWvf6b5n35lrwYc0BdLK)_CMAF5e@(%jD(@;7g0~BsiEB{M7~*>G5Wh<=<0sq(@7g0O4%}9b#2^2Ka&Dt)$4%W=~w6%*%Lj`QqqL!|^K^N+3A!kFrdG2_RM~5?^s4vilBg^%0TAOJ;m_m^!1PiX<P{FiBxt2q8YK_(_m3jM>y9Ovmmw=Vb3BxF%Z^x|+zGrrWg=I&gN$$#x`2I0qD`TmObax<zmePxZ;BVJvj)rbS=-T9s~#P64j72%^9Y+>KTY0w<Lu5Kt%g?U?CdCg4D$G~}>e>D}NNwRsXxa^W>e_6$F7tCGWBWEEybV;W5<oi|tsftt;xyNn?>AgFjm^l*u&o}qT&fhAbPuS@Ex-I><np~(<3AXa+06iAq%sxwdvPOo&u_kqI@CS}gZ{(%W*QwGaV4IK?;%UPpKAGQ4BVNHK}K*G=n-<syepy~Xxa4^=6;}O7n&3Tw$EDnTfvy3q$_jmqm19WU#n|}{gi+7xIi>budqe{>ml>fbz#7cN<u9%E8z9$^Nv(q4b8wp`l>=T?287y3)D;dPBYP(D*a@$psie_WOuycQI;4rV5pTKInQIRAV9em6kogRZ#skGm9Yh^|Ek$s)WDY&}hSuDh;QA-xFh(gWY!htEGz-Cjol$bc}1Z8Nz!aQEB#@sMUea`-ZnU5Arql1CS9k{<PrN{dtva6CMxbVAP7F`r+s>a>D-&3h|ddflY(|`8KD4%D&E4~>`A}PS5+XEYjVCK8vz?F;g^~tL8+8n*%yps5e2Dz-MI@yUz?^%sra~I&W;hNHWgpG+>?Q>Kfxq`CH%18L25wL5kS^&x_5b#^sU>68pMhGb5$`k}Fo~Zz4E~No^6_21934x~NTO@M$N0XX9>mg;mEINQCApM$|nxvaITy4p|mZ-hxH6I1Di*-F*Yg^E4>c;E|)Jaigu%Z9%iGSep#mE@jRSPaq`*)RMHs9QF<H-gGo6PS&SR6Rjqg8gk@c)&i>XgDF6Ok<g6>Pe4td+nzu3{Y%rW({>dN9mEr1P9rxnZQjQ;{m>0NwM}s?x!nzGUGti=)`C=O|(PblrM3eJL~0^Wy!*iWRlYd&|hrmsJer;$gfk(=9$z?1I(63Q%>^$Gq?AeZqw*A26^WTfQxJSJ7ely-^3bT}LrgBu)_RBhy5k&N_T>hzxj9zanf&S)^Og!az%comzJTMeRL`B~|#jswF5n43^u$lu&}j@wI-Dco!zaQlpqQR9gB}UW%QMPhuEmUBp9ZG1YN|?0tVj;^@Q#e;aQtnmT82F@w+SNV<{!e8_p)m;<G}%+skf`$Uy7W?Fwu(i>rq&J#ZHpLdKP8ePIp2uOlHwgJ!;IVW1g61Vz42?Q>eufWsK{>A(}ar2~|1K&f8SXAl8Rl8}{-jp%hIiJPu;A534uaLo~II97JT3ZtV83xHm-$OmDAIB7HjwxQgJgj^KM1p8Vsd^RQ;5okSLWg%g|5Sd(2O9q9vh_6KSJu4@2q+sAiGyMhni57K{R;N@=rK5V)xE!cg|J>Y->?p#9o9r{%MDI>ybDoXmIBExC^TpLrnjJY)e1MeM4Y?OaTDf$*O{V#4vxO=f9)O#Y;OEaEz!Poa$&kEMMDB$dZ<byoG#OPp?rIAzs+;sr9jT~pF`=ek{6CJ(AuS(7rZj4l0=>)0DZvjb*jUW>_P4rd=&%}d&#9uX{lM1y5FgFF4wuivL2sN7&TDtM|#%Y8vp5wV-OCN7(&#?@89*K;{&<LXLLjMUm7XJTRxG?4?oRKe3OBm=A-G>i1~bXUrDRczt31!smg?VkQTB}3C-h&OFXWcZ5tk)(x3Vpg7+O4sSqcF^&DJXosj<x85A!L!eMm!R^?TK0d$mpkVRonIJM9u92-M~ShvZ}9+k_IQ2E+0?#IjE;C01e&x3!J>Ez>r=nEU*uj>yzq5fcovOfKZIcXtc&a%e+`Peimh)eS$sm*n&WO0cX{pkIBauqH*-zpYL^tP1%y1RO9K=hM(aK(tbbXQdzE|u7TMoYEAB1BzCsw)jUGMu@nkD>&(cnZqGe;A=&TZM(1>d&wpBZ##@hWVPP?g0}{EJh5n6*nsOY<lB(m>ilco7bI+UVFFbTxZhA^AFfxEIP~OH~6h!9!O-s9nu@F)vpW^JpcId2Y@c2s^F}Cf>C{NxCf;R3k7?MoO`e`9~u|Gi$|^E<nd~s2>-N8sO7BcF6jFxrSS5pxEc#55S4{4kS+)x7z}aY3tWsywxzNl)yZAyBvs|qh>L(PE<}rF$d^2((j{<ZwRZda;?Gw7xcS<;y-k$9rNUpQksvN?d++Tic*w>pB;BeFrih0lm-<FP7Y4sf)I%83_Vx^~O`l)RHwWLh)xjx>7eG36S|<4={z6~1fE!L%sU`EZ!bJbCor77gY;H{`ud;;8n~YRH5!0UvBkT>eN$ysWdk{ts4#K6gw(vep{!Yg~5`eCI%<|(CTqjHxyuZc*tH9nE&hK9B82x6pUf{wuk%mDqRw&Aj{?g2%^zAe!=Kmf{=PhjXnnqTf8mYf1IT){d%j%1FTAs(PdIK3|T{R{LcHw?8HAZ+$det)uHLr++TZUCYU5}l55Zq4b`7pOS#@W~Jq3qfFX%T=hETi3;)iTORYx)KrqS5SXu>rF%G;Vz=-!nX1Yb5S|j%bm@o+7ak2Ni||3eR7TvnaA5$@A}2jPqdp*VCB0U~V|B-<e}myZ$zhdTIoKR2x>|@Aw)e5(VEPw!k=*;;EgHLLVVCfR%646$+~Cp}HsAoWkc=CYUt)5?UrxmN+yP4pO-~e9UzGTHN^SS8e=eRY%9Kw3G_ACQLjrecH`31>yz}p|Mh;bc|Ht_)T<<c$_h)5{6@V$WV%Wv8z2Q@6PWy<CYY1@?*ppJ%<R;JN4z>!4;^^WUe_!W(AEa^Qkb=v#Cv==W_#2r~Z@3&$L?z!0c<k3#GGL-qh5<+4>BQm1c(^j#Q+M5@aJ5u>lH@6-ftcbI+DWyc7<B*)v^2kj!_`$zFF*h-luh!>1YM`p&8#we)Cj<vxJ{sIe0N<tG!2!dIi+s@Fu#s2%yyvDziTbhaBw`(jMm9jn*$WC_=CLgp1P)8AY%GR`y7VU9Qdy#WM|kkceIEvM%$*9?BKws*ZANSW$hK-wVR6b`9)ev^QEBsP?UpBkw*vE(AiX}j7n*F$`y<<9PUm;~d=Z!cvU;ZOLkhtm<&E;hZ=q-%GzQfSVoDjwPb6M+x>XnbnQQbAbdyYnL+wlDx%(7)UtbP-~eXCRB>W$x3(P`PKGc6-ilQ+iQ+w@|Iu7Qit|7a<(gUq#co1tVj?QW`cj376x)4cc70{1S4sS~QV^(<Casfej5=Hev?bZrW-2*PxzxvzrKpIJQ;3EEVJITabBd*3*<CB&ftoJ_?e8@sCgU@+6btU2PS+&sPcT^!yha(|_`N{3Ac#$?}-+!X|2~DlD7oyAVqM1xM<LmuO5%+I`NbRsFHNA3=^W#*(6az{;D;9s=n5ZeSg$lAnKxNHGXR(QG+YGbNp?AiHi5*r^S#i{`?gtn!loPrq%*XsW1lnL15tXk-t|NlD1gC;hi%%%xv?v$l3i^53(dNY$!SdyRoiUpfWeRe}_p+0<q>0i}94_LUJ~<UrAe#6Cj$f(eyoiAUBc&6ux)Ele1QOxX5JEmD!^yQgZU_G-DO9H!O<E4pKX;YLDqWOWoO51Vk6X%?K4?k-=l<ayF0%CT#2gmn00zLaVN$weC1byH1$o)-zUpB_4hyzol@#F7s1s#5t;eSA3bS@@{E;+#hB!-Ylr+JW66yNtUXrqr2y{Rr>tkoaJ{6kchxOA)o(QNn=4m<IyJf1yvpTLClH0?EX?{FaxaiMqcNp7_M3M2oOO<|c67U~|W_8*yewW*YP-E@1?)&EE^Xysm~H^N~6m6ohz=j?)DQ?e8{})Y829>yswTWN^RcpV|tvH9%RRn~RV3WR`Bk7T<Is!|vh&nf^T@B7#j}1jI0ZV&IzZk|(o9A^zw@OV{Ps;WIFD9K4}nNDc5viPBJ#tOPYhv~pT;Zd0-hzMv_M14wYOd*Ja(3GkN|mT@L&j>VFZqVRTN1gi#O8Fw?|3S?H9n&{O4Fty=yks+c@_^L3*=bgLQkRT<QR0Qx3u!an(RHu^D3B}?%^TvYo<c=zly*(f-<&qnEZaV|YOE9os?oyxAxWqu=P=OG=oYVXX4sPx~q6k73fNv%uf_mvI63@;h5mvK1y3JUyfHmKB*2!$q_aOCH2kY$@qo>w^e*IT8_sdMUQRGQKir|G`LLM)!gFW)-U7Tm#8elIE3Yl*|NI$=2ZP2P<+B}$NF!|ELFv^|3xUQV4_ne5zfLNajX19)k7-yE}LQU*74(a5+{l0ZBWeikFfJA!s%X!cIh6MJX0dCFRSDk`UN`UAAYO0i-^n&KR`l%doWqOGto;3;n^@R&K13hXt>u2tB-3nVis1TE<K+!saGwF~;51d^tKxdAVlNXZNbJuaRG(x<u@aE}+Tb-t+h@?Fza|!Y@SMVB|_WeDNBKRIrrcUf<ubMHzqU4I;lJ#Ge!&mkg8ABvDWuSt&wkW~-i12jaP6)gMv;&tDAe94(9gp|bJOsnPSM=m;MJ7f#o_Y}}K(9go1So=<C4$~q`Mlm-C;T!&1vUEPj;V_eEny8+OGf+|J<U)=o5uhw*7k?p^+9}a;z;#~^2PcU&pusU#1w|-`7zE=Ncd%h3j0YC#8LJUMOPx+9A}&c)PZJN(>ag_XSWQ>tzSj_nh;)qoX_*`hj;{!%vRN*sf+ion{dOmevgjMADcF3B~D?gRvYDt{cfCmhYPQxC9@pfw83r6nQ3~_XHep!CayBe{x?7b8Y3Vz@s(dTHNTyZrN!RDIrjUN@x_bzIwE26eeT5yq-xEX7`%A!7jQpDSrqP6Ry0H0<^5I`;PbA9O#hPU_>p}|NSi5J;u2>>HgTj%i_oTUiIYW9fY6u%y%zYaQuR_AQ0N-qqj=-};?HEb{4g!|)*~<h_=c?4sjZ>%L}McgBMnf0SaChh_fgpae0S*_d84k*pEJnOpLa~qyl!>J3^~ce&T_0w`k<CE339iZ&)r~`$})vJKU)g{sr+?Cm&{)8`hdXgSpQ~cz?p;|!TLQK?~$&@QH0UHH~i0O$T@;%&5w?bDPJ9G$eqwUrbsWoI*(SVl1&z>eG%+$LU#s9gyUI)+Ih$bAg$XA>GK0X{!0|yZe<I8NK%QXIoyc;m%Zm$YV<NcVfYoC7o&fMEhl`NlQMn6=@GaKP=x+qTtvF)9y-}?MJk<aGYc$?H1H$L`-B)-c0-(JT^QOTv_6mh`dZ#6Jy#H2DQpnh4NcG&=S~mXk1R7Z@Yb(M?Z2n<F1>2@_@F_<;g2&hethnusOXq+<=>pFc^WkT=)h39EBRr_CT|fx>e_TsZ6zMUjSsU`^s)xwHH<Mm>FN&kXCs`|(b~|0IHjl7_~2)xjr>(^n{wu0s}RwgKj~+oXOVvz&vToCrusv13nh`u8JX_@r|KnY^y4Okgt1?dLHBN0R-Ft|Jk?Q6*;+PIz^JKS6U4u{B<D7281(nkY!7)HOBjG2CSq2P;=t*f1@HZVG6JBdbG=D`ysrynDf4MHd*0Ed`rM{0Ji3?gNf3fW=QPS{;maSVH7=dF+<tT=oCHGgv*ph$-zK?~TP2itH&&z4YdfeW9gSanV~^Uvbp(hnffrF0oZWLtJV_>+?~<tc*(W}2>+89?twbg#G$+V*GE1xgmlJjNqeEEzT4dFQ!>bdx>cldoc`)wW?IP2nUij{&zul9T9Oa1VmrQKs-yvu0EY}=NzDCfWjQV`8L7*n2rV4L<&IppM;9V(AaVs`lM_57&hPyoS2WUp!?Z9TDGSyMzIY9m?g1gp1O(;X}RgjHj^#zbqAnM_)+2?>kcih}RH!=yvS|Tp#rpQ&XhpTob@AGg0qww{6mw#G+I;d42jv0(X_?nC(aaGUPc@RkGB>=(o`!HQPy2sqzD{261RX;c7=MCUxY^%NKGHk^j4doq6aMgca3kw@r3%JGJnmJePDgs_59d(@MXM+%Whx4K^OJD31^5_F>2cL|&kPT%**UYjVn{}M1#tU;->E`j#rVmrxV_bDx2S@i)aQq^;@qzE+6UrzgdQa}R&9xC2g2$>On*zQ{91q#uE>*b$R+DB+*On*+RSdb0@#Qudg|ksjd0qz(wD5cOTp$$g;*6#bLzGpc&G37N@T3d3_NbClBX8P$a(HLIpvs8CGd4;AMyc<GOL!^~L><uW%Tn2iE1H(88u=kBg(~ij?E`xpMwCw4;yG)3C{qjqh=9}Vq0OrreEE}FLjF#_IySWncN#nLku9(|G}ss6DIoTpvnARHX(^ZX9&*m%J0&D`Vqp|dQLhx#4-5;Yp9Q}nZc36{+oiH%S78X%BcdlHHZ`{JcKdk{q`UTq=RAtNd!MszQHg3a0mr9*>Qz4dnay+m_901tZZ;F)b?zs(#K127yDtDB=Kj~d_;;PZ@%tgYi>nqx5E|<zzgs=jCQ?V|fS&r)KBXdtQB5Y--@LDqh5-y<-Zl!d_5=4`S*Atlv@A&SyI3DYxu$=fM>_HiLLy*?Zh!6;OJT%weg#v5*>{jPCje#rUxTOCuByzwTehYd_qctGB$P;G;9V>sR>$Q(>OgYU{hP{fY#jp9AmiswtYmXIH@}q>>^&HOS+A`30{BQ#M9CW%8IUdfy3%6QLE41GijIQ?aLmS(zNb%G9R_8vkDn|iuKeww5TDO;3dy1d&T?*C_^Kq*3mPl|5I=>a89Z}uXsYw|tWTlP$n6?w)34}n)~!jC944C;u;_>+)}Glgu5NVQqM*{IvsEDaY98VPIIBs<e+r9yBztWu)d~~Q8x4VpmkyKBuhV5rvk-y7eF9=8@MFo&jS_w8s$zb&ZrG-!C^E+k_}VhtLX6Fm@^_gx1+<sa#n6|UaAH&-$Tw*d=z7*+hN0d9&c$?E7TC)kW%423JGNKBiN~@*EhR+OImRQD5T4ag#*eQh3WxvJ2#oeK6<y$^Z&?5)Frsb0FW&TM3c)zY0LK*&KO2}&nqbR8DiItH+$&~Uk+{*QF=D0oDMF8;ki3CejQM+E+$Ey{mpQ^(hO<Ux%=cQr4W>;vNHuG;p6^P?y5I7MA_|%%;IZ`%CMgXl&G{vhQ>$iTDtLShs;1<mHDv)aRi`VP263Wk6l@`8nP-`>fapM+;K*1X2aMyilLreeVR?jR*FULft6H$Yp`SJ|dG|{U-}1wBARc|&r-=bEReRmWix;@(5geRd-?{$6<?Xf2ef=5?@cp8d6|iG*<vsy_aL%E*{kHzj4nR8RgEhK3jI4kj+qzIco)V!4y5+m1Y9VgrATrh|6X_LkOn9yF7_apz&~AR~XKx!h*!(W<A4`3y1hlXjQ@Qx-*khCYf6gsq(BG?9XVq-8*+r1n0=+H)#LGVxBCH4~d?_SnhI%b0A52}9I?95?35L!Wq)&Jt)jRpole_>Xw4jJxTSNcJ2{0{np!r{|8HGP2shNdbS*#GFH(K1u*AbSqTJ?rDlHP;_+nsJ;oS5sfHIu^@|72>~0JK0dFJgOC6#DQJL2|aUffDx^S3sNRF4TTgaeacNY3;(c-q3+2p?g12>HlU5Gv5|<7Nl#&+9S0AZXlMJ#ms?{f&}|-x;ort)p71oCBRf@4&>8UGI(RC<}*v9$S1H9onMf}a|$gk<pkzhxS^y7+rfuZJv&}vSpznEvStW1)PvCgfVwAj?B<RTSc(zJ0t@%XVy&p%*AIvImR$hxF`RLr%dl?bX@2Qv9qu$_gm#Ci1|RKYS@bJPE;?v<<%l3~Bn_hh3`I6)A>D$_LVAR-?jqa=Qu^VL<*T5cNRb0@CoQjMz@`<Y=c~fJi>W<^lB91VoHVySX+H`3Mo{#B-3|nDvs?vGHQ=eK3Y0ip!V=iRtyz%hR0BEbzgVTo!g{-Bd@;4#TsMvZ75P~oHZ$wB2-v5>H94g<>an6@xc!H~yH5x^&ogJy;@h1)Cqreewdjg)I+lO#f}N^@yKP3eNR<j#-2CY}{pS67>wsB&MtiPWzX|AO@agh`P|g#D3db;N9UY(1O;;gy=yF(UML!8UurK4h1Y5PEFc$oSjW);)F;rns7cgxNAs*gW%V8*xvmy-Unr{5y9;bJ5P^z4a=B-_|A3bAKhJH5iU4>noyhvLbN`~6TyBwC9L{z^to|+kVK&d5i*qhWoN*l4LXRB$ltYmG!Zttr0)5t`dAa|&9v@^qhKB2R0UjLBPrEVE~XY@$QZHY<t^NWXlI<v*5D?lXgVCj4R#V!C#Q3`BnfCsQoQb#+t?F@fzwm8;yq&gi#ISeHfVC4UQ<@y^Ftg$oFNC)>^XDeK(WMfw~4_*hDG{3%dZ-z>GK=Vyp*=7~vL^#I37!}W7MCJZa%)>mHP(5}3+uf5au$Vs_%gwcLS**?-!spo=Dov$G4jcGC=j&w6RpOfzJ)OCb9{mBSexsK3fJj^VTiH>OIst!>Z*s3Acy9^N<=xQ`Fk8gK0-!$PJ8d(Z5^$btXr_;}YTS5EOAD*!uytoQ{K0Oe1Pg8X5PY5F0<WUuQTBD+YwZxDuRP}DqWaKgb5=LoA!^lHTquEr`9o{BuOmy|_Kbh6p3O5su8f7a(t!%WmEask?S!BOZPc}~p}CNusgE|Y7QNl0(S7Fg;@GOz9_6cn4@hQRyar70<TsjwG8KQEXUrkpMu*@Q1S<&oGl5}6#kj0LC&3d#8>Xq-sdcp$HmNZwK{MIp<LYDhcLWwhG3XpZrK2s<d*hgOLup6;9hKFbvL{{r&25hBfBLlIB|8U>09gx|zeXWL0|(fE4>P9LpsA719Hbc-@l!9Z%IDo-fADV9U(@(UZ*P)?u~$|>LYqbWe--^tO2>-i_iyj5)bf}cSeO_dLdpU;I~|_sMiGk`>iEy+2lZG!oZs%MUEtuwI3pBE`8(F6);uyL0EqnnDIX_qDZIQdfkxU_BHEdM|B(+zNU1JJJbyv-5_pnL?eq*y+BCySbau9WIp|=>oa`%Z_Kq*UgR1e5q5ei}8)b}N6n08=oEd62rhrvc<_XmjjK0b7b-^`Uf+a{IXkr2&;)Nh&MJGXpL6Yn_K`YwW`RIMyLIKMMq+d_u6M~05cW_5XEF4S#Jce9XZSteIloLGTTU*-Vghnu{*(|iwKK#q@F3d-P)Abyxmdn-Qr?cE8jD;6FYp*Z2RBeP!;AQ9aM@u@DImm8maLS{Nau^Vbu^X!}YmXCt@1{(Zf%e_EMpyzV_#eiRGHqn~JZWY$<`e1hlWV^SLok%I&r^%aNm0>0PTl2>O5usl3>gKWB2Gwi3fUT=9v)JB%!FR|I9uO>O1yz}KJ3zr9g0E604)_9^%Gr3KSpuTVFJB2!Soy?s<bSaP<3!#NGMK;9<w##u_jKy6MVzUWOR0D$IFia8+`L!>Q@Hf=W)oJMYP0CUjfW)xCFToz_SH<pQ|1_GoY<MM*LE5WSTgznD3E9{1lL>bZlsHt+0gcRk+z(<efKfHKm_NK6g5H9lhWyT;`87exP|ViK584GXeeW)0=CvGZ1K<s;u=rYBP*48j|$wZv3K_!S<{SZU{#Rd)#UcaijRfu^cpoj3FPJAJ8J@<p+uw?DV$<OSU^S;_CPQeSj^pjY1?2slXl05YqG317op-V`kvjgAT)wx4nURdQ?wMzYld=(o+Gz9E=f}ylV<F@fsnY8obNi9bxI<vVC7`Q5^~l25I-26ou-`igKr+&)S!~2;oZTXyYSij@cc%J`v|6S>$W7i<)fJp!4{i1W>G$_LE7@7v-W!u8BD10A%h(MYVf`hXw8KBCz$+Uw@bbz1r!mxx|#4y6%&A;L-I03=6bhL1wIb^89Ys{piK##7@EROGS!%ecU8_3w%#1R4?t`hKWjy*##J378QzwScP8iV8JhU<JMtu!EjGSQrDRN`+%f$eynakK{HiEiU(q+UNseOkS?FE1$DT{K2BgJia@QSchu7U)czV;<IIx)Nm2n9%n3~}v_4E=zFC4s)aWkt*-dY<i+b-uP16M-nc7}MYS0GOMvOTU4yfW{gvtS0UWdB?caR9n-II~10^V)qZK3|`s>W&}V6Nk&;4p1#u85m#)b03yolj$vncZ{uWJuC1P=`!%wp7ZPOHrdWQQVwq|EBnZ;$&-&$tBT^mDoS?IzI=|8PO#*4=<AL6D4LoOd0t}==J=&g0Y{y;{Z!c-K=!llf9doLFK2g)c`E9$M;Lsu_~P!M)dbbjAc@nx?M!)Irphwp=U)pu+>;z5nB>If$~2b`E#Gs8nbgAO5UGTNw28v|FT5!N_YomzwS4RPwRje`}DWF5ZMiQ(CQAbdI^|6Y+}y3kcFhj(a$wC3b^lFBd)w6naxLMN&e7DGUg!7kc=O6=TT3%yATu6-k+a$9rHKSaB9y447m)%b?sPa2LlP8mYSil`9XAd61^brO6=pBHHY5V2%)?g;6LmM0pTP#2+{sA13Qp5sia0OJ<>F4{oV>E*WW)9?LDcKhe%m4^HDHfD09z7I7VazR*qJKP%^W82;jvqa`p^Dp@!!FEZ**G``8QQ+c&;vW)j<Ro_>$dePS}#kPWz2j)r#Q6NkyiA||kbhVD*NFvPdCZ0`x}ub2v@s@pZ3z7~WD9@C?%SvBczc7NW&H?GLGeHKBW6AEFpO-wOqtrtEWX4sVAns++yI~z;@+T^L$9#7u3M;R>1(@U3P(4sqC`S!gEhQp^$$L2`z_;AG@dz+OV?#*Ov$RHjkOv-kVvGU5~z?UK*b6$MJ8?LzQJK;K+CJRhm(vfMrrAzok@dB#Laox=d0;qY9khXnFRB*@gAYW^{Bk@w(V!EY7jEe0Vz-Gp{3WC@->dHnJKCl8$I%BW1k`Yj9ij|?Bn^bEOb?VtOweq2MO1|=r7tNcv(BtkDMRYh%zf^J0sT<l<S$yPf<Ji9q&o0!OAxCi1v7UB=&#*S~LNV-KhQZaVA^8xYPWO%?U`L2z9}bfcX6%U-l#{f&+}r4F>^2(Z2t~2SDKi!{&QB3iVJ?&_mc!7h=^#|7=QJqcKNj17?gpQ%Id2V<3PCR0f_Pxg@8f-M9QG$duT|9-Ki>W<-{kjB6G^#u0l@KV=I`yzuRdqj+X^~$$+taLX(rN}U2FI*>zob~4n{cU-PPb8dM7RpISh;tHtxg(2HX_8UfH?rLw)M(-6~nXL*HXmv>+Df1!8JQ(N_3ocHtcXEy@RsX+{#yCDoyBaEy$&r~UOV)z!d&g-PJ2yX-BdW%jskeQL;1M1^Ki+{};aqp3VKfx+WVy);HuxnaIPz_&B(3HnDyE`@OVwwxSmLcH{^P?=$CI=wf$DpI|AsxMLU7@{&W!SZE?fsitMwO$*U)7wU&y>8S%uviv=6PkTzaw`&QKSHqtd;-TtHqT<3u%*Ol%NUx%_lR%rWAi=@XV##ffS~##Asy=kqLI;)1(_Xk%l%U`M574}X9V{Aw(ZhAyk_(6$Ei7>x?gv>VNF$I_Ep{1iiqQUbb!HeZHu@CQ_P+RHv*jU$mA;WaQZ>9B?!O2+|i6Ga2$j^f!VO3tcx(|d2TnXQny;C5F7R!uzBhn68k5Foq@KCm7Km8$GeOHF4bj6qDMOT(dlWW8xPY59KoMI)%~?FY5#)h+li%j(QUhqO-Bay$DcpvT*g9y{GQxD3p!XKtFWN%SH7B&T~C~n3>Bu?_XCIUeIb?9ntwz+$?m=u=hna6Z|nW0)&gsXW--`W4}emvmwqt{kKT{d20_p#0u`rCT=&jRd8r%qFoi;n3H|$sB5l--%O-$MS^mZUH0^CDWNtbfiC`=5UK_pG!XH|;IJ09HfV8FKm)-igl|u)ohkGyX@sG0Dxw;khTYNaSs2{liZ(#JAl9PkuhH~%1I8x+NX8Unb79o-4{S^BL8b6>erC`tc7n3rJ$8l+o$;`VmfM2<mr5fPF`mLGAY#MiAOp{Tc@G0gkMtm{~#vL7mOnQfyB72F|k<Qr(%%R8>p7ZYrdU{Jpj42tzfYA_>`ZT*AD(slz$cQAK(HYKXdX+}I%ETE!6Fz}??<IcDzY9pr?l3VYp^F6}`re$mU>#o%^>~6Sio7|Js}MzC`Q&P)3EGBe=(SlJ1~whZNJIK*UXXgiYLS`xfD4%hz4e7bWK=SE=p&b?%eAL>k##v&M<gtCt?2P{kIMr=Bf3l_3!aeNIV>=?o4@Zx<sWX)#<T(Cm=T^ObcMW@c5aRM{zoCMK}}>Chm_m!5kQjGH{V(00{oLXL8kX4n4S9XII9^)y2G4Dju5!h`e0G})zW+^s9gc+zwri4Tz<g47)xM=%W+8>qA6FUPuu|0(0mukm_GUZ$1~tNL=XhBBF?zEnQLJ-jkn;@rLUfIz~V}}a)#U0epL*e4;AbN*lqsNuVViL(fM*6J!hQD=B!Gcx%%1rFCKKSm#MG(`%OpbgrvxViTZyJkcKG718Qh*h4gj(WSERH8)WkJg7>Mi$Hz{R8Eg4;P@7^6onaBP4Ra>a&|o|^AuhpQJeX^aIkNB=_J=@4ikx$fp%wJnliMN!E*-%L>BDUO*S)p1?TZGIhSulsnmz%s<hycS{8>Uf$Aa6Fsj{Q3lyhl{5u+Djkv+NGAEdQG6jBYC2WTq0c5q~&M7BTJ+FqU{C-yUFTV4~U4sWB22GX~hZFK?)5U>oHRL|@6N&)qJF-EB!1MPAaDj~d_D_~kU8@{i5X<z-DCf!Ot$a-q#9Bp`fd}ezu=Kbo|wsj?LfE=8E#lnaDyN;fwIRN(lqE;CWIHFj=elflrE&z9W3XlE@EZ5s2fq__LF^Tu7LKJdk#T#zxp-Q4cg6B{m@d9{ETxlGfl>`oq3DeCgQ$@r0VEdB{vCcWW28g|D?VL#h14J0S%pXj&Ud7k<eUHqfab^x(VU3~#>o$Pqim%tzV;F|PT^&|{&U)}0W}`-g*Z1lZ@xobt)S!f|4QCjA_HdgF!!)(gVvx^SKT_&kyA#LU2F>+WUx^Q&k|D?2zxflbghs9r##BFuFGk``@gs#G$xk0FyEBR<H10Y3n`>$YC#Gu1E<(fzRB{=bUQCfQ48&%4e){Sx+c6qHFF-`9KjrVFYIb3EK^I0Z+AMjkBC8pbXU?oXdQQ(3{@s8qgqV0bY_uh_yeXb)6jdRCPcc1qd#r~fO%G@FXh93ty1Pl+iBICLKiaI;Rs`ha`5Igl@COk%e{~&UzjxCnoiUX97Qc;Ib1n#^C)+VYT|@<>@5bNTWS-5|lhDz6F7e;5dCO#P^?hRra3IiW%wh>rB5IUU@=%R~4ccG^vKzU4G_YzTbMZHT^^ul*<=Dd|ZAN82S}_f_(ar-4Y5QcdP6FIDCta%gRiv>0-WG~=YxSTZZ|V5#znSYg#%8omvh+@Uv+C`pYR3(Q`YmNhb+?~U-!8P)-1}ESER{w2OqzyiU$Mo!u4dt_O~8L@;@9hdtDz@VjlTnpjgvzlCyV(_x5Yye&!XI>1rD>mEAPWptONBA4Xbn!47E81)Sj`6!KbLl&I2Tu=mO3M%ahTnQGvOs)R(sH#;*K@t6`zFx<l~!`Nl@dl60+kMoj8-ZL9EC5h4dUX82ea?B{BuYU?UkM+Nzdv6-ec>%)>BDjbI?gMeI8mymRjGUdQb4|E2AN~$KR%dbEv(Rfa--g!LEHFp}N<f?P=>;biRE^`x62ue%-d9~&p)>?LLnQE}fLmm%jiS?Rz<M5k6eLy!HFz6a`h&)!6*K54^)A!3A73M$jW#qY#(?tDxPe%{t)NQY#IuQ4_WXMDAD7QMBkzlEPiA$u$C7Z%P&Le-aN5|ONLPeltJxWBFc~)gjT}F(sD8~11#M#t>X>nEN)IzZOeyn~&161W;1f`Y1e$|x86UhLX!hmSvY-jpx7cg0wM==lzKM-N%1#7-f^@$nbH%<`C9lG1(0V&pGBzzr+87J+6ktRboK25xkT+$n_Zfs{_)B7B8l64pG(QY<Z-;?n^-}Ep95psFsCOg>lYEkvVn<ZS}W}A-kygPS%1JyfB5dQTrb=aE*Q2ITO#c~tH{l@^aAc9?7e%JZ}pWs1|_N9411GQe!y$a<)+cT7_-+08DVQ9{qYwp!Z_6Yd&VKX6Uu@SlLy$MU|FG^67UgkQ&pR+6%$(N#e(qG2i?giisVGn1WVz+13A{G_CstLfZ-xXBAUP%QMRbrd~&d=^3x^Q&0u(#{lEfz25-iu<8+GZ0K=oi@cR^^1-mGH5TK8jJa1TvG$!*nnAR8LJa6l;NXgp}ZAlQV~xT2+v1&y|CW^@n@2^mt$Hx=_b?7If2wJWBFl)Lj&f57ZcUss+fuK0P_VthMEHWC|x78bYOMLIiycAt`MI0X!46X9bQ{-}V_hO5rpl|Cu9xfDupHZI{y8n~}>uHYBJmLXa9Nn(Ci<QG6Cw#Bm<^oi((EwTwdw(5SiqBxRyEMoB#Kg{Cy{!xISu=a~Rhpbuv=yLvB0cfNfHySg#>pN0d9TPV+*7)XN-Twz;}*9ho@W1@qRwh(6pQ+D~Wd7?$^UwJ|J&50h>EsA%V&W;t8FyL_t#+n`=fj5DIqrKUtPCZg%6Ks)F8<B*^0|4Ld*!%wpfExdc?rubylj@o!!FCQrOR^xRqMYwKnRItXj6x4zs%up$s*>IM&p{4qZI<+()@qO5B^+`E=%R?!QF6jF0b0a-kV#++#JhOOY8Wjtmv^+$To%Dy-_=&Or4SQ;M0ef~%2}~&xGNhKa&5L@#kOlqp5cd)h_3G3k?|Gl>l^i@43Z@g56}O#h>*fZ5U(K~T~n(oos49R*!g}^094WUrX4PSo?MM~f~d@9DPCQgp6l^otL-qEP>vr3oub2_&ZRNdK9oX-OCxQkvj+P@{OjuGQ?mkBQo(SR6d@v=&96A1?#~1`Bc><J&^&b?yOU!Y1_Y6S<N>%CYuda$%}Z8E@QoF9Gc&nHLy%Pa2Rb!y#eAJn>um4}cr^(ytcrHIPH#Cm6x*C;#~W98GlQM+ALNdkr4Amh!Kicn-db8I2)8S7)C@O`2O|ZM0IshXLWzFY=G(&0PM<V<-ZKX&{ia`9AfACw^XLDSgzZOawhIdIi{UX!;*B4qR*>0&N7ciOkCh2@>rIxm0;ZyA<MM$p^A!ouqy5&#ulazdc%T1BxAk?d(%{(@*Z@`Hvp&#KBojP2f3j9_>9G_&x|qv-Ce4hF!=M4s2lH&<)x}Z2#G`+?L4yXuS_+Tv_yD6nP|k;?YA61V39g)`Sq3UGnONUSg(K!K*K<b%*gl#ldoPuM%vv2uco=l8Hxo|!9US$l5(xC=>trsj(KJN~GEe?@DCwQOK=f1hz>rBVl%_D^k95Z{R_vgj5l;)0W=msD(~{{=v}D<wEZ<9XQtp2MLIZ)2GRM~ILMplhSO6ptA+>gYw&eThG)~^PYFkPLXRs_<?4?&0<`Pyhjx8WWXB5cY&F>1h-ij{Kl|U5h$FHweSFAS-fDLGmZlc}zamuX{-Gj@edtN(nT|`%R?jKzC<SwYFzpFgq?+-$nP=~;WA{m@B86hs1L&v*reF>6!?2h9%$;fG*5R)|G1jfWtx*B)%AgoXL5u!ay>gD(1$}(!<kbZhR_n!RQ&=iF`uM4t_@z;P(E{?Yux<cMk^8`jp7l^~-{zJ4p;pS8YK9px_!;fW;e@OT?E_{7*ytAWq?s(h87%sM-lB_#D1Spb9l5q=J^|k&Br|r~PF_y)V#E-kasmuQHyHF_aLj}qqGXHd(A?VV~gDm2_Sf)6Vgg&<_Eim0XgVPQ%Zw<ybtH&sYZrUh+pn?lmUA!(k;~EUG2Ni9__4MwB0D-?-*FO~<e=8J0^#el}mdpY<;TUCT*v+3@hr&_ZC<35SXUJKtn9y_WHJ$GN?Nf`&oC%71O1gXXzoJH0Fd*1gaG|-r6Q!CraK-_;lpqH?*+(TeT4xxfceNc)WJfN;xA^zKP7b62R6`ulzh((72h6j5I~#%Iz-(m5U>T}MM4;C2W9R*Af5<FT6&SYx9WAvuXiv0NZ2$1pXbn}wF+`KSJ%W%1?LRqlUsyrhM6zm&uqr}}(ZE!hky~o6Bv2O5m=11MmE~dl8EH%0n1YaQ^E+EoIF-cfno~<8Q*C6<Kn67LkU{MV5q|sqaIJEpuo-#(V7pFkeZEP+Z{gX?AZ?_iDU2*y(XDH}dH~tif*+0dsZ+pNx_~49&|o(h%Re9N4(N9pQ$n1&>WZU*am{heg?C6Z@Tc2b#+wd%5u__2S-`$?T?bR|bijN3Gi1wn@zyxgFT-<{nk^g>G~4^*FDz>v)XreIQzDa5@ngS`d*j+=GNM0#bsig`ZgEY8u#~aMN=2+dQ}C`RO<gi}V_C1>I;tgnSYG(FV}U?!lAJ`OkFwa6FKTnr)VMm%EtdF^qaZYLStgILzRR>$Za`;3@&rEiiT^WbG+ZN27HlQ?A4wE12L@<J3mMtGKJ?2=<{zPjxQqwc<poaH>8i17f53!Zs!jAn^zPUUW)3E9(;c$DxPY}=y+SB4GA@aXi@bzM0?O7ox@BQejv6~HLot^z`nW9Iw+ofS97|MD?*>Dbt5uZ@<I4A`sAs9|B|ZzFFdh0XDF-FJmTd{p?r9jYf?Qkf<CwQ5Yo9n=Fsd8xk3~U$#+Hi|j}&<NW!!OlN!!jgQbhW?Xw3?3eK0mJGYUg|H-INE=FM)U@iab3hNXCiMx0w^3eg^y@QCIFv8Oo+%{!MAAXKv`w4CK;diOR11vi!q;o>hNVE%5)MbRe8?96`VaEIF1-V=Xc*@I#HEFni|Be$;#57}Euk)AsM0dw!&$SOxNa3}2cfXC_PbjFn6eUTcja`P)V{1pA{6^DOq;;$ZpV>Y07FCR?*$O7SFY0eE$JdQtvD16@^NvHvIkouQ`o)N42z%C}?vO8IxDnyHGC~|EJIW$j`JZOskA1=z_lS$aq)6Q`s&8yvivzZ%^-!PhmU`n)-fpI#R$yC%N8wU)yYtR!op7*<l?{Cii_}$k48LYBBhB33;olyc30rI?_@Pg<4U50-5{tYW<TpjF|#!Y*+Jkr`eQ!NJA4VhDj6l8+`bAQK?mtLw{5rA+bDOWb9<sXb7e4;Kt=m6Khv0G;<4WTYPq4p<Ey9zc^F|c#V&7f`5+2rHbM3RFxsD9D`W+vunE_;rXTqNw*ejxmkXjvFHFkfZVAQUV-+~RYa4P_S7pbB}}5l>O%lCe>kkY~Vtj1H2aqdCX!<>fkkY9fU0-~xqh%91EV0MgQL`vvm)z=cm@@JHy?R9x6hQLd!_B!Y}AEW>ydZxn^HSk%8RPHtt3{d8&){a0MhTHain%}zYFFgw;czjS44N==$Vd)Pq7Kyhq0VRHz6r}YK<JtO@5xi?vV7BR4{bgW}BG?vWC!VYK!+UXvd3ETDBzQSF|vb8L5v&#-h?{|oe><aH*`pmo-s(j79TviI-#X;4SmKNTUA2Gsiwj$;JL<O-)sTQG|m9Crs>&gzmCvbt8xogPLz<ODXY3Pb!jr*fmQG$l;UuBrBd<v#Ji#ilU{5O&}BZx91*vmUvRPq{8JP0NXpwkun8lSBlreLi8_fJ`;Lq5n#or4tx<1EGImf1znm7vo6y>*Pv(;+4BhOtI()Ri2Rv&#JA%8GFjQ}ee7nr_E%H&L57wE&<zpMXjZonH|O&}P3g|6vlqD}I8g^dD!n>?nR^;fRASe}D6P6{5$r3d>c`v{oL?sN%Gq;W^(auE$oP6<qi-5V1&JaPuR|yvMQV9GJU=c{q@^b$m!@@ml0M=MR&iqqo!&m{1Bmx5oi}9_8`Qr0<=W=9^(VzH}BnX`i{Q?lK+GdiTV5xdj{OAERm6Q~7qDXDn7w0^ssN9))~Vn}+C&TkW_}%fJ}FAwu{BnpTqaJVXbi+Hn&y9BHVb%Z>Zgnp&1#!Kc?BkNh+>hjN81_UJn!i(8%Hg=Z~Za5Qf)76zIn%`j8wB-&D9X5eIoD*KlA1wDusVH@)tcg-)o$S@LLMapoWr`3GvtXKT@4}}IJni7xKzEqNRmJXy~%^@wKK#3F&ii?eak{%7DC`s(!g~Zp?TIoY3FXiF5CIW{#(MzT*IeGK8fo3sC=Yyy&X)j|kW!byUo`MuNZa{NZwT-df$yakQw8D09pJgPoyan7B3X3o}%DviMc&q2}{Z_9sC7~Iw5Etkug6W?)+Z$9@Yh!Bmh}j<nYY7m^lT;v?CjDQe>IRTNfdZb=^}6S$%eU$KR3$wroF1wN1wLe3*ZCUMBp1Ww(s?o#z+_o_t!-PxkoNafCoEfG*Hd8eGRV4kPqJU?M9%ZdS1;{!GW@)Gsaw)j?xYW@N#zsDXGtbRwFB-g3%%jvRa&S8BP%0NVW)u-&5ecXcbwO6l6LE2unj3NSLZlBOx<1AslYAaaL$r|?SFQJ2(Im-y$y#wH;+yYhXwUtOSup}{P_SzEwUKNyGwc!7^V|=*?@{NH#O%F?lkFV9VWHpv7+wnDTvk}_tQ_yv0*1|!}+UEIP!Q+3Ty>GYAv|@<nXDt$(-MCgb&?|s@Xs5%JE()?@n9U#95D9pE(z{Eaw%b+5Pq!KKuCPo%)Ip%EVq;wDt=D@mnflMLFl<^Vc#KV<i57s)1;t*P(X%S@)RT!O~ei#B>iAn+;s73_r?2zfvDD>lN^y3!$R(`1z#2CgaZOyfQ}h;sf@wO((Q(s}>JcomQAbMslg&`lR3dqJFZ)T^`alsIt2+_DU-RHM_R-F{ddK4CB0or3Yj_qwqXkV{b}T8$Q^3s$@faeDt8PQIDJ|d~ZvKjvLE}pIgZxd^Z<*o?DI}zJ?<xlNCbI?6?l5kMpqGb<-MaeUSf4uD1n)TmvK=l)d6|=A(YjjbGR<Md-RW)2}98;i}UOy!byCTCduf#llY)4kLPY3#x2VJ1KrMPJ8pu5hr|Rs<<c#@#BUHMRS$nYE_F7pXub{Wqe;K9dGb!H%X29tAs$&c2ropQ6ccnl;Ks#vSVkZ86@waeph;^Xy|K2nhA<@6a*2K%ZKw|xPHhxx0Q$G8Z8CPlHxR9io|<Xf4Jl9d?k7mqel}`A5l&tJea+`b7p4_FcB-F#8kq=IpKp)l3je+Zor0@e8UEK6MzAom;bE*xQD78*XihYz65iy)1Xob#fNpEbeOBw2exnWFlM3C=d|niiQt&>*hM%}mrEsvA6Ax=oF_4FKq5o|iMoH`{(Q(usJ>jFy-Ex8|F{}(Y3x*PBBHb?wA8wOj2Y1nT5QNPHOa5C#O<PrX)4n+^rZoh5{!bWVkxZJQi7#?LCUhuK>>=~73fS**grWNFD+*)oC-EAXX<-IA%XQ(QEPk>s^^&UrQ*Wkeg>4{4t?VSrTjFg)&E2bx`LP8Zt1Bv71@(LQPB%iEp;C}khwx;y7GdV-RbBEM}I(@?cm<k<o)z^fa>}0x$opvs0YuH#npCIz2h7&ES_^UMFrb`c@n^TCdIL{2(k%knWc!KtE<?}lXudAJ5K-4S-B*6cEYgv1a>N6LHVJn{xpvKmqnesPm-0&*%dU}t{;A3sm%(U9FmTx(aBR~3M)OjY)<)2*yVf=LI~1;2uF9!$81S%7P+FR?Z($n26D6qJxu6HGi1)<p&67@$eG^iCT%mx4C$jA^-?^4`lwH{%xZZv>ib`77{y5U>PufWhHVopbapmmRy{0-;mTBSH$QDjP6hB+Iw3yhCO0f@>9AhCZ3It$x!qH6Z5Py@F|P(tMS@7$tOg2JC`TW+?xG5g^#bg=0{qc6p|UrXiL=4}=|0KuX9=HiAS_%Uwi){J!v-RNNw^kN?X2P-z8xylB-~Lx4kQavfr@=K28F0(Wv5Es{zwm~OBUK^a6vE7WX>nNSIS+jK&1?BT<aWA&oz~6pNh=j^NE2o|F2QiK_iS+L<ybGQAhg5w;v8Yu92peKYzX@>gr(1-7i_lb8M3i!pg~`y#V$3Fb5GFE_)W5K*AnPM<Pid(ci5TEoM>{)3;StYeX>k>M+&%B*6n^<V9~GZx9p$5k+&JI2+}aUvE%UaMgt@2sOEe`Aufm5Uknsh!_Q$Mo}>ogro#EQ_Pf_NGe%Hq8_eg{>d-JM*;B&M$@+$i+db%n@SsN5#2c`tc!Cf$YL3kmlQZws+GP!c4o2rEg@`BG-5g60gi*DwUE0_j7jRX%67t6ggq%pgC(t3Z6)STHEph4HSR3lr<?~;zpzjej_{A2CWQ|X@BbM|=6G)Q$E$AiSde<PFl*A9k}uFxW8H-J(A21meBB>4IH*t=3|wl%GBJeCav05~iXDZZ1EV;tw@XCga6U$s&RFPwXtvW@QfkoV0YnkWIKF(ag&HplF&V<P@ER7^mwb`dk_td<1<@1l7h?X2g-gVzI+c={X=?s+y#i?y(X5H}{8NSTL9`wt{6eGn*>c_)ulk*j;&GhOJ&q^1woFSeUaRZi_1b~P$X#6Ry{`3vPDUq@m73IesxRy;-Or4G8m%-9@w}?h_V^vM#8lqaa!>F@wc>I5iQ?_Y%$P-JLGAJ?n!PwCE+w$~^UGh3{#-B1^#QXwmswCoYjmgsvq9Rb2q0SEFj-?+>J=o)a2=`S*8jRJB%|WH{MA(9HaG+%wrQTOzGs^oFHZmDZ-Kr}W&C!9iO=QlT_-ulub|M=18gwlBa;vW=Dzrw{iNoUNk#5InAEowSv4*}aqae}DN(q9LF(mFp&K`2zz$eN)4FpNmcSE=5Sc-{dzhpT>{VmFa#X>Fl>@6ps%{7k2~m+YTyV#GLa5C-OSv(T{g?rSrCZt)5)a8=I6TWc@pu2AF#d&*1)oE6gmlqp<_q;M3{aYRH(kh^otUugi~EfPTwK;_*z2j+p#)rj_3r5}oK|0&Az-PwR;DL0sY!DQXrMZHnUfv*Zx!c2qsv@30LZk`GbC$)|6DS<(`K(@Umi27R2bV2*w7l=p{o+>X@d5FJ@jB^7cW(%MF=yS3xZ$VHaEv94`9wX8rcmjjuo*pYM0g@IgzL`MRjZ<u8?M(UE>R4D<q=nefp!Ew!bk19asErXpv|;@x|?ssiZQAg8a|HzOnacX2IGC*y!5muOGvVTOsE8{#?(L^EE4^8XoP~Kldi4@h*Szo63}p{H9}%DV+dF5;{>tAobWrQ9cW&sXQ;j%F-`%E7t2;H5irncpZ$>-w{F0O|dASYfe%v?6OszM6t(>ivTS8BxQ@q7*(|@dDlMPx@pHOmX-eg%$=!J#~IxJTxNh1c>7H|Aoy~D1+I?wx`h9a(y9fCHUJ<4ZLgupfy@<)!Tjawi8M9;f^Q(QpW;)P(xYU+mf*^Mw)4D*_{B*TJyQ$>wRELMXAC*&2vlVE$H8x>^3TAJ=k@6GdwrUCA(DJ4TEkh^%oOB&Lc^o^{|7fh>sSP@#}94TpP*4{eE4iRIJqs?J$H1!_$HY2H(<f?Mx>wEb>CQ{WOx9L70dPp;@-dSc?mH)lh<#>uZv~U(OyTbn*Z5sFdPg!4A}A1^|Mn%Bnp~uEcc}<R+5O0dx)oZwv6gCL7>^=QuuK+#ojX^5tqfx)>X7$lXJGJQ_W*R(yq2BJ4)8t>l|8Zay|QGT&n-oi?42cm9mWF+=mYyLS$qZm}oIOkQe+*LTK0KHv~A)!6wkNmPI&l+FTT+F_>4EjSIR_+vrX5q@2j!+c<Z}^3V!SvhBQ$QmpC_&3mV84wj}>d0rnxSz8ffG!B!v$_eISGV@4C6PI_+3bzQBQ%PjqDGKc`*>yY}d6%$2`aNnVsFi2pp0{4tseA)ff##{nc6W1d5vYZZ#Wlwi(bZ3Ec?f^2-?-z@Pv^h0kkoX8`22~P%rTTHf@lstn*_~<?VdB?7}DZuAQ>1?r?j3?{bx7`;udgL1r)-{OTcRsy~F}xFSoyAVYdP&kZp4fHCkHea&TEd-93l(`_PTI#0ROCKDd<FrV+rM;9lffi0-;)|G$7C(0Sx6$(!C(Dxz@{iMqKg#%kYBD%Db}jT9IVJ)r-35t!Jk2otZ$!=@pd(gDW+2@&)6?Oy>rnpQhOwR6#c^zY%igT`}tT31iI2%Oam^%BOqdYPJIhxai9KG5}PY_MJnoc`J@)MsT*P|pdxYSsp>?0U-}!yQXDxnBA=)l&ho^c^|G*7!2azbj|Mxy}uvx)c<99KZs-7L~w<q_`MkrXUupewJJ}jeW)^u+By9^tLe_1ekRn?Sv2uoFU7}B@BwET*3GufCg9!5q}FD-?96%2v*{mu>$=6z-3iJmx&e#_n$xaV}MJ}MlkPD(_v(YC>nDGW|GIyx+;FelCduevPGf{x{YgZY%P|=Z|(N}B|DwHDw~GYCGgdgHrf)R?O7ocq}}kU0T}vy3=E>nswFyCd9KZqy>MZVemU$oGw0Q)krphTp2@GX;>5%{Zt$|T(y&K&3}NwBpgEWZmL-J`lb^x>p8X^6O(v?k<~;YSIWi;13z`X$(4E0nr95t*7!=PO>^K9WW()gZHq_3be+BfHmG32Kl#nkMp3>~DS)+klYas0skAsThkrhbXpk@W8?G#A^oLizhT;MMA0X7RGh9CyE89B961xKsln7w<+{QkodR&9(rVN?ObbwW?k1+3-}A0iyL-vd^TORZQTLHy++C}U)BvTd~_C!4F@lkaP#)S4SvR;{Dz=nMzhf#Xe_!(WDn!H39w(iksJWDZZbvxQVQEgDD2iDNz0#CDcdP#C_(-zgo%p|@zdd#eEdAVV=gEzreIReN)h|Kw%WjGY(UgD|LPbOQS%1{puLOV63byKw!&4tISH|L_e#&}n{0l>AdjI~Wbz$t95?$~dO;-fL1E3g9&rOeWavg0dlFL^+P~@%Z9dM9U`liK2RYZ$Lyww|=A$b?Gt8F@v*em8LADPUc3d0#(2F1Nvy_xk=91v2?20L^)F;c`)PXJiyJzZB_wHG7uAqxum$&UCmj^LDYy^WdRnHc9?{*F=?7OCTylGd<l$Rg)Q^Q`{wEB_I^+$zD{w?s`TwD9c^sz9IE^_wD^4_Tdrt6F#>VDfp6r}dkW0YOn`zpj5Xy(*arGUsJ41vti&b_=S8ZnBeZ@JbErk}ztn1}y$bl+QdW?(TDO*s>?z)_hyZ&u7_~YXdtS+YzXZi1W5YHb)bRB+nQcnGsai3U*UV~LylHSDAkf^daYBt66%K);Eoh)08}+^mW+bPFYeXaufe#%`qvCou$fm2HF2<)bd4HGrVus8@pFfo|GA=*-1Ljx(%vhCVmDc2+9NzA<qfwIgx4YcR8eE8bak9pBp`$a`2Vfo>$R*7mEXk7{Ve^l8q+mbNiraJpqsz9D2z`oo4AxlCa6CvpibJFXss4azlQZIaymdA!Pfb42M|c_VAi=?dr#%aqM|<7q2TocQjaY?Ny~4TOx^6zKEcoXC{dSaU+f%KG8r|yU{*FfPfsd2raXj)(qcDFD8ygPyZni=@&1gvk!OtvhS{6_o)Of^U*5BtuX+n^j4%Y4MnDAWM9&?^dE9tfvRT{0h;a25n#Z>(gG;gg|CJ%+H17_>L<smEW{8m}2X~un_>syy{%A2N~HN&4uZ7T-camX7<iL(bN<~kVBNCf!m&NyoV2tU^y?8?#v>k)*Tke6NA3-<7{EZu+_R+tPH(h+wk2m6(U#5u-xopY^{5c}dbrHse8CqLx)rDH;?-@?+K)aN^<kq8qo-n}kkWdm@gq&`Ow>mV@|Yh4g*Vi47Y3+$ZITFNR4GfaxF9Y<H!;iz!l^K+5@$>r^OG2qHccD;c5KxC>Ly1~bfU6}XZWj)C~F+Ln=-2*(4^DPVF#sAW5-TP%$_qF{!)edmAIA-yB96CF6%-dJIbHR1~3vFs=&74+!r#VSX;{n=QXY?@o{*mt$5+UG<)vK<yqD=Gl)F4Bnb7&MJrseU&O-Df@`y+TtXd1`0#D?KU0uN+o4$~tI3{@UmKUH-@LukF&cZjVjE>A>4h>YqeLonaL43I<+win{@9QRv=P^ykVLA6Ss$sDiT|N5WFxTY-%rjg0qM=~#<arQ*}MoXt~LzfTF*RCk}w*STA_p2|`7<BN!2xj{N3s$ppYroV4u}zH$5320!Um?F#%&2)&jK=p2_Zc-<r5a7Z9PX}^13@SPl_QC3)t9Dv!eGW0jE)))PzEPue|fE(KRCT}9iLNfY|cC+OV+NPAv({4S&ufeRv^)Oy^Mw-=lW1P3Rj)@AZ>6H0!iP-{SxuA*8{krn(jJ}x~3f(cYzDwxQI_*_nnON>>ec`^i&>L?sSz)(42B8({xmC*am<(%N+jA4nQ-EaugpO!7mW>$>ZD!R)KXHXr5j>r10O(2IShcdE}L%t+O00cFY=BmqZ5rZm{@cyz^n=EC8KOD8~?+I0>l9^;Xm8!ZQBJvy!luA%^cJ?1`8e#8uu|x5T!O9%h=2b+o>|9AK)UDz0jw{S9E23GW&*85f*-C3as=pL{77Y6p@cpI-*LvgQ`8lA#4Zl^1jPRaP)9Pm6nGUR$d4ZH7EfflEmVehu#4`n5OJGD@)|tN`~dDc^}PUe4RREWtY$E(yC!WV|^t-zeAEQt$@{2#cUeh&lil#av)YpxADvk*f8D8L}ReZj_4;0RAK6)6yH@vp+bY&3PU+0Px0nKIYk^gz0x6Z<0o<?|`)*Vo}7|vxec^F)LirZXPiexyx&hOpVJh#wyk+?(^=6p`Ry07@mb{UwTBY(t$}osFhck@_=L^!}1(jYJV;HfX;{-Jgp90kBh$*I-O~W8tBAST0fM90(m8=gv(kldyDc6D}r*A_p%5ihKz*<bx?0lm?I>+A1~j<%7$@UDW?!$B<vA69TToSKatUxB1^!1(+OYU0eKbJ)|(Q0c$c3tk^N9ALrG=q7=a2)21p&U4A1_Vq@Bqv=cq()<Ajw*`gI|rFlu4|pJmwse7Xz+ZP~uB3S)WAXizmDCwJ?f(bh<)g)H-!Bj}3*PL*<-Z!>lEMFZtbm4l3p#1+WKr<rf$#W|Tnnb|Gh!vle0tkBSh5^7bBhtkIC$&6&)f9Hlkm2=G~)O{2%*g%g$6e?NlslG2XRbQbb47_C&REU3ksJ_@ec*qWq?#bS1M;w=ETHd~l68_R&MBev>%6dxn9^wtqecc^vcw@z@nRd3N{}I-AC_n_lN9Z;6m4nnTh)t?U|3UG><I@mQ+eUxTBN6TQQx+$OdML)>wcUA;H%w~i0B@eFLP}WJ#7E_t|2W`ry~CCd%%m1jjX|0s-`-00?q7icPxA29U#pE)#n&UT!}g%kO_A>$P9Zg~=V{d=P|bVPghkeTZctWy`0MoZ6<lZ>u$)hyg?jH8-DuYS>Q_36ACzU2^#b}Fev<yT?2it}v|wj9M4LDDPyoLy0<3mHOToDgqteP^i=5_>pLtg?w}we*kvSj`&5;Mx0CQT|JZpDq_x%p@u570A>1;on4tk8cyeY&eDiRCJwJi?kK4f0%df9Y@pOM)y=~IB%QFSKgw4F36B?kb)2mJ8nDyaWzM^%O<oaU*56=4j*?f$q7zbaI<myT}1ArX@cmpc!)FMzF{4!Acg88G$bH4fO#T{SnU7ZCGuNPv^8dhofco6w$<&8Wl?djv+ep#RNJFHJDkk+}@+LVz;$dPE(|P?x9-%Ed_+l}}7=t|)w=+c#!A4%0U_007Sz>j19n34)U453<hP3|qW9_!hCIEBu=hfC5x5k>J9(Qh_ZkjXDO!MW2n;n(Kj&j4>Ii0Q=^Q$+4tcWL>e%AZbIrioC*oCA*0HGXSKvg5Phz++DR$z;$DXNM1IL7vj;d+3`_wbj%lcB<SV4=kAgpj5JsaGZfaxI<DxXT(<NIwP`aBh7*?}U&kp4(u&b{D+y3w9qnysVUDx_T!jhX;g1!&*K8R@s_vNI@_4x#cxPf3TEpSFuSzzdqz|X!Not-%M?8s}%QN=%?yQqYR7)wUDNwl32}{o>=*qg+iDeSW%!lO#eu}#0f)^Bs9rBz1Y1&pARH83wt@M_pXpe09H&}pd0Fq%qz8NCQH~rBv)xM&X;~b_`g1NM3bVf6Qp~p?2v$@H$%1OtHD{d~8L<JaR5@<>_%R#G$PB?R^sz`-KZrVXjRX$DW=u=%3%l<EUX9=4;Cx|N>Utdj=iaG$)iRIgDx^Pd|C8=n+X1%C=<*GJNv?3RxzgwB0^Pg1)Bv+Y4eDB&r$%DRD3qhgdM^y~;LI0{b)peY?QXMhEL>pp0CH;k{J4JUf4NIgAD7SGAup2w-yV6EVT#i;P;{5<llmbz0mp%dzZ#}MM%n%S}+sUwT`8Ur0^`;H|%Ks$7Th{EQByS8K^uwEy3Iez)g1Q2v^gqK|)$2RCGBy|?OH5+NydujskNfn@AZ#*groG{!g7n70``$^QCk!#kkuK!5e7GmbZzm}S?Z?J%qMr@1-fC-AU_?L-D_Q;T<?YyUfKaR+a8;MVmmGg^=3^a%3N`Qa)1i`#HN(g#fSmqX3e7@@?xAtD?gEG9a7PO|&Vwh=ZLixAf%`)3tAlA>Vdvs}G!Z}8Wb?GyK?LlukZOiEMCLyr0ybRf5hU-ScX*YXiY8@E=oj7r^&>f*7*sDWXh5HAwfI6*d*LmK8#;me-LC$JbllFok%8E|^azQO;hh*jqY{P03!dG{;SeVc-zku&wX&#Z?lqz-GS4y%T31ia*}@OQmcjHTiXs6VaVhh|6k&D)Xe9{y>%y%=WLc2ZrOI0OxDA|+vqt}&gHOL|Qa-XY&2msem=fDSA;$@?0mYDai&X6x;faU?n0&vxHf#O$_&wM_V+v>jQnvjNBe!nMiHz>9r)HVZnOqg65YyLZMnE5i%3gU;#|V+Yn#2u9J@V!)ewQQRnO8WCxF_dI9kXQ72@0{e4(Oy?43h>M^)y|C7gwT{1^<n!@zZYp%3<N8zG#k_ZgAaYkMc%giw<AZnp==e{afN11oh>8M|G+E^K_QX_l*N+Na=~~$42xbl)$ZtohKWX1~61@l7ru2CF=C=xwI`5i0*dvpX1r~ASD8Y92oRymh1-IKEn*rTN>}tq{3{WAkHiJJgNkwMy-wO#iK&E8k{ux>tpk+id6HIO^>2tpXV@tdI#5>%=ueFw6ISc3h#xBoVWlV(X9Q!-AqHQJr@vc7<07u-x+{F^<9XCfi(G)*X?cri`_!kVh=lcL~-E?126a3l0D)GD#Nk_>!7a(3#ZqOrM=@ze(W2!GGmfL;8}~Rf5y>MoB>3)TOkacJYcjk{|P`(jC02@XA2C5Om*p??ksN7v->uTW?@}9m|tkNilu*#=ZuqD)4JN$vv1rvPKJdA5UMc|TIik%?Iap(4;hsu)AK&uSx7D3t&PEBmpJo&*5Rok8{EP9_?pRf8taIii}C5Eh@TMIXBCL(@#ahU2r~0>M8qWlswu7=(a_@wx?vQyVVe`C3f<kmj-f4o@YP<ioqYE}(v}l!#mMYYf-z)h?}*r#wf=yXP}zjVIE-$r+zA;!gC_|ilxsX-$R?M0{c^P%qQY$0Mf;(vM1SgP`UUHDkFKm<fxD1Bal5IsIuA$!?9_UKFpnV<)3}S*RTv1uHo;QGE{U)jw#cxT01(@?b(J#Ge*lY>TS%;2fa8fI^2#`R37wpq`a0{#T-(oo(Z0?k5a<H#a-=gI7j(EWXIvbF3Cg`aTLGjgT@bn^JGE6}=oR!`cQ+$&!A}mkw{gdw&f%2W{|n`m_&5LOzgU<LDyEtt`i-}ubs=d#QWu(YgWLL59VuQ&%#r+e9<9*Ycq*^(3@1iL0_j;bE<%YB1Z2H^C2RNgcPnZhG-KG&hg<chO=FLnN4-yEgF|mY*<2y~UGHpz>Cx3sF_b)~vw&0C?S|u9pRA=+NQ+Iy2Udiu(@VBzX#WiNJyGu3#k=UcBJVJ}5&3={D_U6*=G9(FRm?u@Dp#QIko#ObgftO~&R?UX$C>Q~_cPcNg@2wZZsNwytNlgm;Iz8R;nhob(P|LLlP_-)@DRr+)OTLsDURJGN*u-ma!(o~D%?{eos*eYqv3yw6K?S8bNO-<Q+ocmB3Y5BbVfb#FY?G<NWVP<1LC(}SVN$9Xra?F>1Ml#6kJAJ?hg~ss^vd!IZTH7(1}R&r^W6G4QwQ^nn{nskkM(0h_LBzVviDE7$VMYo?fW_m~UcDQ&-c$`ALE(Mgup$@Tk=H(HvVpcmPTH{iZFz<mQ%5%Zj6V37L09YT8XG?Yhtllq(@m&u6m!(#0*~$jg<(64<yhN@FSv*8=-ILeON%xz6vkW}c>W|H{>YB01-(w{ycuA$ujSo<~dfj7BkskI4Ez=J9^aH#cS=UE?<`;4MDW^Mn*uoC2RJ*QwjFzAG14RM%4Bav}8uZnF$kf0oTgCQ2x^8O01>fsrXPuu$<l5rT*Tp?xx<8<D)<=ir!;-8g8EYXelQz6EYzuf-)FWvhwW>pB@2h<BfQ>0y*wGfyIOX=vC`uE;gl9(Wt3Z&l93C>;=87~*7{aP;6T^8|^+uTKO#AgrzH#JS-{4)pGS?K5f`k@?bXqIDMc1jUg7X}nL&O+g_u0BT$q!6DFT-c|e1qUuPVHV>7qnBxRaSuEsGULl%aPgeT~|26cE#hF<|@kEk#rC-Yqw|^M2e2xFeIaAK&3ai!}P|uwbkp@AAeucBf61HbXHC~LN)2N99!!3w&D?p-bk$@A>F|p6Bx`EW7)~t1cs>^%Y1R&8vOkM>`-#b_Ty;n>;xfY33pQ3wJL5k=wVdSXZK3mUIPn`nY0TvzpGbM6-=VyAwalr+z5Ub;UI2A%K$fKI1UR4-1WvtTqAJxXB7xW%PeiGuwfYW2wX3|Jret|!UwMN9}YO3|&O=&Z)@yRDzD;EQN`2N^L(Z5~b@g?YXu6VK3El1i88G<~++@4gIWTbIzpb(P79a6={T8JdJ?P>piTAvBqaQ^XqU=M*i9%-_1sBi8a$T9$Y*e-u!t<yH)!r86gIQ-_BDp(ev4@Xxl^|!x~{Lb=yX5~)Gz_!aEAV=Wsxi%omzN0|z)eC^2F|&tAkrI5$yghEb_HZC`K{2bEgnWlYbnxQOwU-6GA|>17rsYb{3)W5VM{JDEQvrIhLsI1_Eo90rYk-9tOlsyCXs`fQ&|i2lBD#kqoLG2tMedze-tUOG`@6(=cdIE}tA*(;^xxlazP#2LgiF7B+Y8DiJLSv@9sSIz|2w_hQ7CQ>DvDa)H_6Yfjp(yow<B0D0u>!VAkDe3)P?<Xb`E_9YsZXf)^GB&ma8Xm;e}8a62d8ZDh(+0Q5xn5+4z$(rN8Td%*kf&c}{#ePD7^(hJJg4)Kq?VCL3;{c!KqUlaWvd_}YaVXu~%*eb60@<x`El&@`1<M5#-i3|y<`BM??)4L*tU!`a42Mm*dAodcj!8D^-^{$;`ir(AP@)M5s0onHLBaEs2dqjKMFL3jU|mU;f#kh$=E*BJsR?G5jSve@E9ohpZ@%Udh!mFxMa6U<2awdLxkhY$E!2Bq6;R<{sK?RK+KMlhmg@SRlZM|`vXd9gWJi-0uWIYp7xUv2JihhzaE_r19VxbS|`e|p^I?X9}^ejC(Ap4e(Yj;Ax!nd%czD^9rVooJ-X+j(g$%42yFmsD6))S;n?U~SXLxYs_&shq%GCUx6*IWFAvWA;otpuiG4<)R6o+f%Es!r*v^RN3|7L}QSL%I3*;>fTuerh9556cr^bPpEjETS;1mTaSBuDXVb;@%(Q@8YB}_klY8pjz<<=-)^Ww57lq40+EKOW*>xt;q(^vp`#QXsWm<j162-EHwu%qZ3L@&#ytaT!eH+STl1Cu5#iXf>+YM&-16GTKyLrXe>8XkN8%B3Z+b6jYHZ!kbsCh?#_cPBa07pohUI)q?Co2ZOD%bjLnvvewVEW8Zqu4zpmPVe78t39b?DADz7c#lZKSJq&n@Owne@fjyj{ek+Mj7#O3!f;7O`#3d)((;zSk9y^@6zcRHyG^sJUtS-XLHa#1SpQr`#wxy$R$2nT8+qQGEeDwi)0b6jC{d!!^|u&ka*UqEZd+>b_6s+b=!Q`YXfdh$&}JFU0uh53MnS*NuOMy0Aj;+syckr!*jK6~O(1`W4tlt&dl3g|KIY-`II;l(XP$xc2?Cx0^@r9m>9?#OkSLd}KfVAzOJ3E1XjfrPM+ZwNC&FQ{NTr@O(G20F`Ssn&jpnt4vuvsQ;3$(m?&gW_&fD$Szo-!%TS*jJU#xSgiYDU+vFOuWtxXvr3=?VjyFdU(PASIQg@*=E!;6YV@#}1vzF{mW1<qWT;?7nVy+j_bX-6#=o>-E0v>TGKjb+>MWTAbvgRRgqWro7AKQgMtG&|j{FU3H#^W#j9A7c@qnbm6aPp`g>-keW0DU*!0+z<FQ{zLkB7B)fU9bibChoCKpN4766eSL&L5z@XDIyOilBNs!a@B=8zAOj{AtmfKQMbsT_Yj&wEhRA^Bwr@+YHUZd*u0#wS&Qm)v{zjCan(vItl(W2oneRY~g|Lq*Af*MBD-SM3ipJPBm;~nb~6iT}L;saM<&dNA*;V>v4wGT)!886gAt0HR6jSCeQWiUwNpJ&1?x;%p}U&{T?bvp9ltCmQG7+Kn(tpx!CxSGbA7JGR>0|WIJ?bpnx4p`eELy>$oK=tvu?DfS2;bTDB=)A4jHgd<QdlN%7&LFDn(ynMe-e)Z(woZ=!(f=c^hHeeZKk_AE;Rp`8|CgkxW7m5J;1N_e|}UhI0A<~TU?Cyfe(8WyNTKCvr05;*43^gklB1k8dH451--%GxdXdZlj+k=={P`r%&-=|Ggf6|(_ejuq*1b{ytGnvnz#>T<3P$!_f%CgY`T|AECcIw6f0<}m^+Hd1@tkV_+cu#N{VW)z9hHr+E9!BI7~2g9%ZjdDmjPSKA10&Kb$UlE$J&LJ_%>!2aP>AchJyMow=fHdX~m8eZ>vrl9$hNW6?kDw|+D&hMwJ(;00NKkds{2>R$LBFOs{aQx$T$?jW=(%-TNe84sJT7@N{)hPh3*71xpsL+*EA4cEp-~z(5%f(gdk+Fs>V<h<42Iw-rI~S{frcg6MX_SSGushR<E?oCQvrM<B)`!msMx!cUlNf?*Tx!Ly>W2NaL8lDkE1IM<gAPBqc!u_MP((Nx(n@7k8|11In(5l&EZ(4Ct#~lEV``Es3`v**z>*mVkk7eoj4bMrF?g<0~xpy415|1zV_4$Ff<-3^=wl>UZo_up9g=b>#h<fTcK|6(EUB!#F7XKwj4fpDMD<DM#D0q1na&LZf($7KY&1mEl&@Iab$I6;=wSM%oD>5yAbeC9jyCkEud080}oThvjOvu`hPN^c~D^I;5*EBDH@z)Oo?xajXKQNj1w64x9`Ld<WU9Po+eDZ-k2M2OS-POUAKBQqPMhZcVS9I9n5o!7O%NBg63BJG<5M8>+d&6DwHU3<xj;b=-X)A4;(A~snk8um!k<;rdb>-J>vwR8ltQ}HQ0%IA*yS_vn8V~{b!|i2QJ3Yn<V$M&^w-1_hjT23ff{x{zT!4%7w#|J?~N4e>{ghbg)@*W84?}Hht{Yr7<;2{P!f$0$0`VL>#AD@PPQ=^<?`h1Vl^6fYp{stuM*{QnZS$`N_7iG#IcNzDiIDVH#nJxRk7>r(HG&r1gaczL<NqU1_qwa{5}1+-m6NT!Xmv<j$dw^Q~*;AVZYOu{23;x>_EAHf^|(CG}Dsa`-gx4Erzw^sfFxZJJO1xBq{@OQsdNy7S}eucM`qP-ERGfR*#z<7$|K#M|y}{?+?SE-00!s{+6|m-|lsX5ME*K91&h8V6{<Ne{9Ehy^b}Bvh=gdr#ipjn?}9xAF%gCBYH4!<by9+G@5$6wa2x_qxT94Otv$*eKRrS7k%b$xf&vo-a#)*!|o~+<78^D%EuokZPT9qz)S#(GkilrLKzzn+Z|4J_!%OgRV)WR%(<n+XQ-XRGXYVJJlT?pKK}t35m^9WuUxhi2&}1n)Tu0yc;qHIRn6K^w!eWND+O$tl?Wuf$ocV!ACMk#I#Ta)j!Y#DdB(Pz1*CsxHoRTTV4tAnRAb9tOcy=Fl5z~xD*c7{{^u9zhoZ^<3~p7brw@HxC*7y>9t{cX+c~T(0|dwBA18np81uSj>hX{@1PH_O5D$P-tjE}<+D2Soiw$bCIb|olya#mUkl0L@qLbDXm8CAcp!*G#654aWW=l0NlKvvwAYdrSbqEWLsz;E6dr`OpVfyNF@F<Lp!YG<l61T^g&iJ<0Er*ea*-rA86nDyv}n$N1d#tY*N}`S3lR9*_U&dBD#VCiUYp|e4awuArv-@oG!kkXO#!GP1ad-(f#%ys%U8LICMiiG<LnNZ4IUt=PR#>VrUJ-e6bZahk;8lXawa>%je>|T=l9v_yWi4@$09yS9B&EkdhBQq1N@QnX?y9pqMZ1OLqG%UC!p-AoOYOnm&tTf;GL6^7Sq(iAUHtgQt9h}G1vOSlSSDvG4?gg&}`^rT#+{h+5<GuGghynrw@3qM!)(<&R2g{RT<&cIDzpH!1;G%bh6mtBXtnkv;K`j^D`=YA22}Qw*6!j<?v5L#_Mn9pQBQqxw}11gK_<NgBBB$O13x|1^UYLltII<fEqTmh9a8E0<jy6>6@p2Wsv2sjrr<)Ja~3pg~5*OoeK=WpgLn16jrh@yvvxCDaeeu?C5@QSh>h84j9U~dr6@gx>N{0`BhBOy3rVr?&iKZnMy-69Dk2wVY$pl$$+mq=T&jB;Sy50RDL0qLq}C_&-VhWm`VUAp<tFpRU+|B^qnl&Kj$>eJbHpYom88;92kD>y^^rE;lM4wg|)8#haVvNupn!YG+jNG%Zdz$6_WD)i+Z;PaO0b^J@SvgXZN{&kgh{qUSRc+EQ&+h<z%P`d#%_wk;Mk-jo_}6m>-P-w7YBZ1ZqatvLQWZR_BwvD0Qc?bf{Yr`-Bz`Q&F*z{BHpEi9dFW!gZj8;hluu%OL8TxM!!^L?nkjciY@*3Zz;poB>Wl>&|qcQw$1u-9-oZZH)Duk?~7I&+dQnm5;h=>Rt;)FX1RPCwB<N9afniaKF!DdR8i44he@!-`DC^1&y#%V{c|ySp48Jx@mChfL0LuOSDQGB^X!|{n68&zN6hW`RT{X4DA-t9?QzO{Uns&BTsh(7J``3+UADJ*NwnNaV=8ejkGemf@mPOI)BWQcuB!k2&B;Y0Cym2rXUG{&4Kxrb!b)*Y1!6yMz@z(wxkiz$s6x_ZVZPSB}9am^E?9YYm2l9$XSgxo!7AgX-r?3_I5CwPVzn*WDTSh>U4J|I0AN$IhChXRpLgrIeG8zl5T4VTX{Xqc#n@C@UvKbiuB%MUF2>3k>R2rkMswV1-^F-oDPiEE*`GQdi+Wi_+9>4jHb_<edcEKa!t89gUm_If_Mp_lSLQyBVgffn$%8({;l!Zt`cm3<^ltTbw8?Vw1}AN>f;;4oy1Pdv-iy><QFNO$4z;J=6VAhkAr%YL^G4^)`bkUAb!z(9pSg}U0b5!%34H{!2EkhdrPmLd&R2Q?K`{uK)i<mL*(!$;gl$~9Zu5yoI0uC=bnXSZjfe@NXcNO30I9|MB<jM*^3ySV=Qo@yg4Riwwgh&{sKxvjs%FP6O!j^K6;{`$r`FcNh=jgQQ5OEE`K1+`*XXv^5&+w?J6`CDXMAZ1$7X378@D{C=R*wiCem^;YY3_YvQ>;-%+>S;1RPvHZWZ5#d_s|wf@!?i_UAiG#^4ZXUiX|ab-AcUTzrA12()LQ0ESm1n{!BMbGO@KNQm0j4}&1n6LACM#R^KHI#n6{|L@6FN6jy=TOWSXmS$-e9n%~m9X>-dj4-;BvgBPA<p`eqGY#*O=UL|{q2dGOL^@SWB&+Ys26g^wHc*89h$T9ySSr1*OLeKtSG<!(wdEn<SfLFFb@?J6+6D(Y=5d+C3aBBHr_bE3$?jp(%{?LJ5I7jHJR4JDJ(<2qj}7G-^C*pfUV)EBEyT_`0DHYYMXv!k`7Zj>7ivNYPD{mMk3>Qn*4CC0*@aTzmUzih<(lBpy_X8+)5wPb4nPx(PEZMLuRh1^vw0#AIl{Lom1M1pLrV)p>}xd(HLdJQZySZAAq-m+w8x+3t7$eOE&1ys;zA4Y-^7g?Mj%{C-?fvK-KJ_fxkn6LyTMH{Xk0GGpbP6SpH?t4QzF_48JDVyaR1@OpZR%h8mxBx^KH!SH5|Qz#J<$dP`xto=>u7lwj*+T9}UMIMx1%3Hjv0jZ&B6@LTCxkVsQLqqi$9TyVOUfJR^*O`1*672rp6L0rDru|$<#r#$&YPgw4%3G;#)UN*n9fh3BXTK<d)-VZ3yObmg<1#5lP$Uq^I<n3`xIC?@#48{}OKsWar<cap5gVI!=Cm%HC{pro`q9v4p%D?VID|p8QX2N3XZ748*^G<VJHXR?5F3J?8;zd1V^8KZ#0j*mRp7)k(3|Z-fv>d$81BMVNHag@#87D+J;BW>Da`wkN)RpvR9My^^L1MCiR@v&)e9kvQ%BeH>BW0|xB#heXdwvA8g>ku!5UH#m3k{um0lE8aE&$rLd)aSq7=(Dlz8roNm-{2q5xfxRLNG5L6=8dbF%cX7acM?v)H)>VoBJ8N0?;0u!DZZFFS3rDrl`~=AXsP2ksB+x(A{YLfx*8KgF9Ytly|n}-tKV3iXg@dJUWV}&tHCt;jq9nV&~%rg~o@gY)Vjz4R6I&AyoyMCM$85x_b5GKj};>Uv}RdTliD|ZFCGT`XByKP}2Kih@Ca$u{i4@o4*#fIn6rMi>T5bf(?%EEfm^MV2v9t>~AC4Tt)8C#|x1&dPiP<k4f7@Qj^BFduO+9pN*ZmCIGyuI+kVdl?H-lbK=*()r$91{D<){?Eguu6B6uo`0VhxiyU7w7U+(%GsALge1^jsN<6GJjFWs(E`0stG^ED#$7>4+;Da{Y)%2m5m#b$y#tLa{JjdJboUR~VVbSylC^B07Pgl8dRg#$Og&DkPGQHreP7qyM4Kx^++DMTgAR6$}&jM7fC}rrSRT*5hZQZXrUUK$lfYQv+6w8&&9@_H@(u;%wAslFY(YFLPQ#lUV0MXatii>Kmrnpk1H$w{QffGaRJL2$;LJtjqC4wIfJO(!)_*@P}-v&`u2|Y~7eAId0Odr{3s{MW2WBU(~lq~iqxm2i$+_m+Fkt3V=7^g9P;I=EaqIiq93U=@c<pxK+ZDBe%JB7o`G0G5q$_Z}!4fH;>wbgNrFz}<^eLEwel9t^E8b3XEaLX|(HZd$p-Ke&>Qiu&itMV7Lh|l{V?b$L{htoq*4R@0&(=)vX)fL?FCp#jIr49P0TrXTiS+aCYTjyGSZ$`{UzfH5t5t-F|UFx4pSZUzg)T!g%mh92LM#3W(4uk?!Az(2wMJb&b{W3|TlkD}A14~)=EE(C`tGaxcDVlaDTTaF33C}K3pKPmB;PAv%s2@OPar!D$+y`!mP42>s+j8g~4+;s`7fP-lg78QAs0-ZFA@dOt|767$&=2X*75{z!=!$g8bzAGE5{d#~g+I>UMWlRHEgGeW;mojenfo(zhzTA>ZX%Lfzn9Orol@C%w9AEKp8_lHM+gurCaM`US^f)c-X?IyxyDG<&cTH}!UA4@wBv@)p`P=w(SC_l0)`GAv8~Nh;4amZqgZ}S6WjcOi18%lD9QW9`h*@S0J5MzVT2~#P@J;y8L+pjGe9UoT(0A~8hd0Iga-T#QDUbvG^z>B8u39_0zwrnZ;nBDT%*<edw9TAT1G%XWE0*R_red}>8B6UGmqjzeC35f1k=9%sY;9iRkKq1)M~UfzLhGI%tsFSBV#7@BFe%SthYDf-l59tS~MZ18kGEo13MQ6<E8UK0M}4>`0;n(TQU1Yh@e1BR)b|UX6AV3NEWtoj@4-Dr$jA;;&9?X6_kQl0Oab}508;uF4Hh4-xCCgNwwj(#_QLkU<^;2<Qpp*LmG4Za(K%|0oSN1HPALxlv#y&P&Y_QW4IV<C+Dw*E+G+FCnsTz`p-Dv@b#p+FtTSBa#Revm%E(8Vq7BnAwo<-GstD=EOpD@*I|*H?nT9nGxIDV8q5UF(cqqH#OJ*^g<sUzrjUP~tjW@gVV?*RdlJQAs_c<lxGAfPLh-L@?P#eEbY26j6+k)yx5iIXT7N1SdwjoKve)`1sL;Q@JR|R*9;N{tD)8rZP4%A<?a}B<4MO9YhnG2vvPSu%aI=)bFrj|aKBapiIb#tlY(V>ursV>20MNGPPGho6gA_@G532+v)0KG&EF)f9CZ;NS2M=hf+bwq=+!s9Q5#}4%(Li@tK>9T!8%EX^1#Xv$w}Aq8r#sS&qg_T0Jz4`wm{w^~5}SD6ycNnqo>W^moyr7C`#WDRNv1P1cupBnh=UaR$|-FsKZ~G>w-CuxVoa5W>}2AdhaK09S7el#9NGM-{j|LKYuL7TFlzm0&X!R#>akckYL7BmT4s{p>PSldz?HA+=d!(5S&=}w8#rS7^*q9QpXiw;pW0_IMsaikWJ#*2{1SBtaYif^<x@-+cb$>b^92Y0K829w8QXcR7$VLFL@lgh65qC?IVxyVqmv3DCkTH-ywP$|C~}+hzGsY+?-}|I_-QeOiTC9v3Dd3O=NB()Ou<fY-*oMcMQT#X&?PWu{E5?Iv$XTGqWZ}#C^R4bj`su;KozE{fPm++eS*s98TbtL<dKBS#Q+&-FJb>KIz|^^d4$+qrx<C=D%_c{rFFjHJ$jfaa@I~6W~v`pPy-gVut=9D+3JJb(inrYU-H>f+N1%_+#ZlPwYQ(>$|b^kbo6s3xeg*d2d~NM<aTS)>o#+oA4vwU%%~*d9S^U08PW@zo5(W3_=0y{Y9Lvyt^Iyj#E^v6d95LDxBy>B<Klf<b?VZi3>SFPj-ncRF1^ppd)JJJl6g<Q^<9w8`F{Z$sW7?kbmGigRA!trNP0>Gqoadi@BU;dKq{Ncn!>{je>r&Tm0GXBg0Uy9t%daM2@(@A&%Hb9O&mbDk_i~RW*A8io@n=dHrn6SUlE4C^m?1WcWEu+%JZm9`MPijDHfAAYeil@BlbctPK=TU__Nh|a<{`^SZ}H8R9cre9G2)B@u=r}JA2Ms%w~kfi9X%GKz_%HXV1IZS;u9*Pc<8fNR~nf0!1S65V9lu%;vuMK(6zh^WDA5WI1?8;OX7%G>Wy5J~}k>fG$2DDy7)ny=l}DVw-Hvy2F|lM#g<nBtK?1if9XsvgAv2#0*L?FaF3KoMcp?BR0!mT<61wgY>l>KJ+Lg*uocLI;f=2^$kylbL_LF{9LA2{&^@~LzR_Q&_(0*uDQ2use@@2<H&Di6DwyIX`V?-%(^OBzA;9Pn_Dz(-^l*`^@o1K4A&sJGFaJV#;$D)Vcs{Ce#9Fcr+)}_XfTbzT<&U?iE-K7OwU@9ja;y$B}k(=&abFW4s!Tg<@k@WFx9CWn7`nBZ-%0!y|6vrpkPjUzOJVQB=&QD;<+`U*lSzAz?o5=S1F?P1b*@3o`{{#ljqBMaw|l@Ro(peyo~gHbSv3{<_qJa%e(4Jboz{V{+p%Mx(I%|67_MXo_u_a<6U$~RzlKTl`^V0sL82Mj6ZpVC0?Xw)dAOTo}B|B1L%|bZ4pFnJtwVG;X?5I_P$EN$(wX-Yw)k^oQM0VN626H!{i;}+jM}>y#l@yL~oGF(9?*4C5V<dxAF~4c_SsXn7xBy^;?t|uj~_z$9(ap3R$+An!}2NEoP0)ryqNEpv#r8p>nR2J;895F4uQx!2YBAQsNU0PUvC&d^sp$M%K*4GW5zPgjQG4?u(uX_W0AOGpGWq2|o~eKHz)u^k{vi78rnCh@fNnGd!%8yPB8hTsVh)O3}e&tZDl9pkfv4sV=*wYZ$|gd0sYq@6Dxk{#c}+;s4en)n*IWcUFM=E!&0_<29^MgnsHg*h{QXUdr|l{ALseOA!)+hI@_=f|*t3PtTxwiM-R3As)34Y7&3Cl4o6wr0)xt4u9d&pN(pH%|m>A$hKfRbEz>r+)?zPB$^+Xf;uS|%#0nmLp`aFRKk}B`cK3)q*HDDD#;~U7U%Uj|96h7i6I;zTyHU{V~ECpSM2s6!oPg%w3@ZeO{TK{iQsKvQguo0w+kqu<hxs>BvzVL%{)ciSw{pz=MJEA`7D5cl3q|o81b7V5#7U!8ey5FE375xzZ2mLM=at=RcNu23p)#ZlwW0UU%v2f?`fX<w)Xs%Bq2t>>+GbHi*q5}mcZHc?XtnI7W~P=x|sLJ7T3i)f<m@Bj&#PvkM3NXLy1895+N$6Bine`-TxS(oO23=?3(eG?87yJT>8V)CMP<ttF;+8Q!^t;3{P_}_bGuYnTGL(><HXvQ^|o&n&dl-70@v9F>@{0m1uo}7JJ-7_@c))Rs~kPVfsbQ^~&^k%}GX}Gg{mT=jylb&Vk8!Z%6#ddJiMC=2LfX#ZzjKQO+6iPZ=4k#F{^g83q!LqlLR&JAXiGBJ`<6-m2y-YRJZrypq(9;6Q9d8!^TTI@zi*y`Yu?w`)Ece(lXIIhU+PC=15?l$yqR21W9&jLb&1Dz4GY^eV(Et+Dw}EK8iN7E+R=6O|$m>n(!piavDY|6xi@3k?l8C9`Tu(z)WJsQJWs2t<8xZUl^A7s4%H+f(&<BtU1h95o|8)8<PLC68%B)^ttdPUPg`wKJlU`OqeiIn92Op4yk%3Bngaow=5i&^)jT*|}@`{^l>s@-zNj-l6;kM{*0r&Nus%(_A70(*gO9A;d7iIh}8dTUS}JlA2|t#6<8|B-<UjpYJvBfunzW^lE|+LXWL76{iL(iHK~4gy$37wEI4b;0_E0P7rdrkk8uB2i;mYMt(=864TPu$bd_oN-8|sGXOHyW-^sG!G%WUWlCxanqQNOMU#RTvvn3_hbtOq)ORigc=(w|M5wvqg%6w}(0;9`%R0rSW5JOF=SIHiJ&3D5L{BIVq-R|;xuk=!ae>CE0A<m3#jH2<)d?ki@-&mzGu*gTl##doL-zVHmrf>6ZWvxp+*12zVAIvNiZc5XCXG8c?$z8nXv1YW2i_DoUP9zarV>B045fEY$%@t-e}}czh7HA2{1;#_GD}fTB0{COyUD0bhOK8p`Ynqm4dGmBRAjcZk|t=qnA17GQu?$ZPIw2wwpVZn@iaq{oFvhnS`-#u`O?+U84z3bkXXoOy(H#9WJ_?4D3(b{B}?RTDO6_PJBUZ)+x;A;Bf&d?$Ldv|JPI?nJebaJ<+YH<a<-G{{pDohkQ>4@<bs^z?7$;#Vr;vn3_^UY#Qko~;4fGchw!r?)_<~I0H*7qb!JBK1vImVn|P1f?0qZMF6q>0?YzWGnA<xGu$)D}nCT!0G_h)>uK;l|m3CJMw?T&n2^<uTl6#B8mJ>1F&+SiLtOFuPaKB-KO%V%!SKvyZzfbOb1gLrV;bM=cGVvhd1BWEALf?yJ0EHkg=j!$RF`y_tTOuKC;IdyLiP7AWyAn;3QeqB`={Yk%24a)?u=86If1GZxosfB*R-=bs4e+NLP9Ov4u@;ZJ?}oX>XmdzoQSBxTHF;<jVxf)JO^&wF0!nSumpKA{4z6-5lU6>$bga%^)F_B(1YiKShR~oL=(E(TZ-K^RdaL^*a^b%ADz<L(x~x&Gm{U=A4kdMpd4rm??}1Aa^UE{#nex2<l)uMJKCt;XTK@wNrGrF08#gs4jK0vit)w@&P*Nd}=F^QFKJaYN{j@QA2MsKNm$y*heQrMt9X%eg7>)lr({knvURg#HqJ@~`Fybd3Sft*A6VIY_0Cq!YT?aDKDfg*6Q!hvpS>%H9*H&m;o^zuoTYfx|B=XYxw|w3c*T*mQ0uVdu6nLh#M@#yLE@}Dhl*~gZMfeI|G_7;<eSZ%g--yL1^^<*o^V7=%3BRPIc8kPB@0)a4ARQ_d%U9YMEUPpN##<vtGEehHuPV<>-AP;6B-Y;p=~n#9Zr7N)ie46HB9p^)JAY8g?q2KLHeH=QET1BrFQXsBtl_`@f5O0XX)j?*$#OQ#f3gh5hqXhh2#4$qjMkZYGCfHLZHexrFn{C{RP+hZ;3|Qy8_5WRH6~-2vw0q;aR3JRBIZ!QN9h?R<7s#B?(pP^y7J|#=v#oN)XFdj%Y^SePqPQ5#+Rg+$uHK!vIe7Ns!UJx<g>C8o)M|0oYsrBI-aHOYGP{wt*m-aV+&@nM%e1EI=y<tcJID(%N#Wy5F_Wep@#fkxa|i=JCE#^LAsL5UsV<H9Gt5~{@?Vdu`8zDbP?NUSO;`%{qi8WQY*!NM6l*rv|drVv>7*z5k)F6D$%stpiOU@TNbiJV}p49)+vg6c~k)CaW+G%%>1W;1D`{QRCT|^1TR{-dDc!}`C9{FXq;bQdJob3FT?t5ls>T}aR9Ivz!MQbxp-TTj;8C{dCUa$2cAL;78H9#3cZF`_50MugaEUytNd=Ma-dm?*)GW?c6f-J&8E-WiCc;e6ryVj0I0qSv&|x5rYPVnHj09Agb*L6P~mvp<|uj@IM6{;@(P@OCG|I*>5=5~As;74t4y>1*~TLkUeR(gwnD0AzLPXjFI8If#6Lf8=DJ0}kcOqV!W66Ar1Ck&DED||bn9vD9F@VEEFe@cxKa9%IPh5Jozp~3{Uw@!+a9))fV-|f*Yg>#de3sj2<hGV15X2O%p0Yo))w&`j|htu*Y+jUWrwuXH1gFU)lZ)Kp>W*~>={P^VQnpXy4(@PEqlt7k`Z<D$5rkU0V!~$rvHNnUQ_B{l`tV4O==v;EYy8~|D4o={2+ilAh(@E88W{&#mB#D%?1l&!I9aioBPuMRh$^I-cpxnBasu2DWfbuZ{pe?j~t<mN9i9ZZ$nRWdnnP{Df91|OvCbJRHOA)H1$+hXay1^jX-Xz$yk6IfslEc!Vi|s*K_r`%g2Y4WMb3VHaPAnRnMX>*4F^>MO4fLaX!?+6Ag->-LNt8yeT?+Lv^Hl9Fquh@ba~F!rg}&S%-zg3P;4`*XOCudr9w8ke7$j#?>!=Jz2@l0<3SSB~Y2yQ3sDyFH_JDa*84CU4-8R@FYiEedu;oqe<W@owncLS}#&!0bGz~JB{kX*=FcOk*1C?G$r`K@HA`~2~6nwXT2JFF<R=;B$qz_Kyf^`=5r&z`9j>rm|H9Vc^EF*ZZxf54!kTK;yI@r80d1|e>JmnD4m8m(80ELbTe)AS#n867&O^=RMVP>LPR+0Ea$?L+g*L4pyII&$tGF0H%v@90l<fI+XaKeSLhwllh8VB<V-h})sUg3JAe>kzsEc%*HB9S2AVv=s%%I9YuH<^Pe)Y53tjfAuYSwM-NBj}ogb$_@3X?(rb$Wb?W^;7Yp6PcYJx!Caq6Qkmh5WH-tD-ZrOqpw3vezT<;hOFYS5lkWNm7MlWMbr_Gct5##rHrNUQ?$j1cGSLVu7~>z~fG%fIEt3>nI=J+8hOBxm_F1o4LedcC#!&)`s|Z8Up_0P7Gy;G0W`M+`M0(>;mgoXWS3aGjI8bj&bKzfEf&+br*5?1wZ-f}np)Z&G=EKq|oqPCD+gBrWZ@)p@(#|KAesVqWAy0(2TwJ&yFALx<J4iz-bI$-Z{WgaVTH^TrDR4f#^aX#(mx++D-%aLx6ajcMV%!BXpF9zJBCCURf(hn_c_0zQ}r6evu70O4jGk^WJgIU+9EnyQcpewNs|7oqD>*|d<kua+#fC*2|h;w`(nu?l-2f9#W?q6IPWgc3JiHlHdC6UvM#Apam$))fymibKB~H`o8Fi4wv~e;J0IJLD_fFy~ZjgsV2|@JF!J7ygTT!S)kUI^9a0`qE3}nn&}@ZUqY0B;N44;l3MaK#9!S7XGiHu=9#L6ZpjF$gGZz7{aH&|K~(((+s3lwlqyYYydD3uk2|%UnOis@G>7Vt0_icdR_%jBFevoNN~GMkhpG;f!uTgiA_~ug*3=zY-DS`sd$aYC6S5g>NiHz{2}UCkqUBO0a_AKHjVB5<Au}1Mvj^yRYC-xafFj9#Em1rX{2M$a!#s+ZH*F+hp88|b9%gk2kYc#?oTg;BM@1qonoAWR0Y1`J&nICE<_)DFoTK}(}i7sg<(*x$?<e!JXAbhgFi;hp#W)Lt;|eYngch=ZpZBj8FR0xkpL*baFzV^-Ixc-&p^iIL;wO@A%ROp7py8@yOL`lbqUYuO)ZhMd|+(0YXu^<Txiyt8H}*b-y3-bwJvjH>mdYYqR3W<DOdo+DBl{0Nm#O2Wezk~yj#;(j_b5T{2t(lWI>3TB2ZgSK0yj(hlt<DDH6nhR8}c6old9N5=#D2p!I`0bCFC#j-xaxCwviMzq?hD5pCd-W=?}V1?|n9KS>g652fldN)Dm^f}aSLy^se1G2sQ1d3*E;v!UhyPg9u@BI>YMIwWagJ^fSB@P0z7j>Iahr8hxA6AUoUUOE-PaA7TagjB5!WK!Dxq*q4s{MqIvTE<w=9g5BTGK^X?`UrW9t2n;-bLy`BlIt8;e`uF-gyUKMo2Sb{pUz=?e@PGVE9B0w#rw87O4ZrhJee$*_xvcB#+LBYdN%vx4TRbXaYmUvMU=qK6Q3${Es<^~5Dj1gR%P<T*l_A=VDtym66fD-*r%WH7x4z1g_;#{Ga>=vuRi6lwv}1fw(2=0(%X-{uCWsvnEnx%v%r~VKV+oLxtn+r$(@nU%+~Y<^oNG9xP66L+=aK`vwZq@bX}A#I)V;t1L;vBlb^gPyO|Bs!WAciZU~Kv*6|+#pccO_{-Q{|h9n#G5iUmtN9%q`1(Bd1_gSn1K#00<FCLOi>s$F`=kEv6%w+`W2emDG?9{5cA;g(BiQLrujx-HpLxwuup+>j-bNyR!hSkYr`6O5JVoD!rI#1=i^WSt1AN9v8--bep%RZC<kuOh)w+<M<0G)8DJM`Z5EKg5>E17@9WZXv`E@Nhn0*<zZzF$HsyT@>&;#6UJU(?6JNoA1erU|`bv%|!n_mnptTBTHxdQi^N7SV8Mcuu3*JX0wH0}k?a@n;Yf*gF+v<C>~j=}EokG_ai^(`3$2JxUi@J>&JAzcKj@C?HajSo=NiKeb(b*>TwXdXsW#M>D>QHX`nPa#+Qpq>79p8|-&A8fUJnb@D=wRBV*dgOp=b^1OZq;Xu5Gm~(z8QtJAe)u4tF&N;Ll#MBircgkh#Yv3+Ab8<9zCD7ZYP|Gi!vO|<s(x6KT#$42ype8iZ;4F`MAD$clB&5Nuxt+R+gM?$s7=unNIpz&tOfKB>4sCpsIj@zAQSzX=YQNqvir;EtCYojv-2KSb58|PS00jM<>DnT`j)D1-LIaGf3zW2Z17fcWj+|ksH|VAdGxT%a{ruq<8l7&r>`gJt+Kkz;@U&3@OHXdaIHbllBjRH7JvLNErfQP^RmK~uC3p_<K$}}FZ}xxK)Oj$Mso*lO&jM3NL5fb=d;W$Ao%+>i1OY^NAaoB8kyrhz@*+m~WrMO)J-A_2W75B}Qz8@1fuKE*^MCx7s?+z!yw~{WDqIJgIidl69PRRozh}29&alB^6VuAtC08v+7IZu+xrv?;tR|u^p(AcOzj*dof5$+dyS%b9gY4)7jG+r#ORb{r^+VKw6ZW{|iFwXyTz32{*Ym7-*Jn&jAUf7tfpK&9Hd?2123;AA6#1Svu4vD7I`{q^*!BVJ)=z<FFgJh>f;(pC06AT&4)9WMT%aw3Y3CnOq{JR_=J4h20_|cS_b&+ZQjlABe@m<MQbogzC~z|CNzxlZz_^$NiH$u)2`yWfw)t_vSf&E$kH5Wacm3LdY+y(Tvo2R}w<r~c3Md#96kmt%6RV9IB2N^saUKrA5FY_)Z2SX?Ey3uO)P&VVyy(^D_<)|aZbS&&tYl~~yN%zlgU9ow5B`Pj4{3|@=(f7FR33p@ZE5Ydiqjb(R*|G*YlyApE#ao^5r{Y1MWiRFaF}~Zg<`1WiMV)1Dio9|Cf5>cqI6=dkofKNHHmxJ+&o*ZDowK<am~UgEP=_#E=PwJ-&GFs@->-Djb1!xh~E!5veM`Qf(`Oxq!o99W{gyT<sGL+VNP6XK)aD}Y2l!NX^|xZntfbH;nU%s=*tO$X3yX+|Fl2CupH}=X6}r7vHY($`jDyDeWMC}f`xUtX0U70eE$YES3jCqlxs==*gqiGgj7g5g-)?R8(|~xTU?9^v=x#FbZxX5L_hcIf^({Xn?l#JLH2&Nv6=|9z2j*5chkCf=L$joOP~`_4MqoA<3>ow=G0Bsu^F)1R5DccROevj4+pva@HMcbW?5t&4#XqHnB-!jru2KsX+Fx!pk|Z5{2&sFH1As)5dgLL1Im%71cPV>iiiT%a!}u4EI*$<+R@yI^G~>3NQ}fR*^iM!0G9~<MyZt|25Wf;^1lB8=#IbkHkM23i5CKi|7%=y*$>N_2r9UEbz~=!a7U(H^Sq2+5<lD^3Ti8+Hf4npEV!6c4t)op?U+=uz(eN;iOO)W60#|rq&7RF!@nLp0hO2=9ML+7-ak`m-UB$Y=e;$&t`48vd_0&kCDKi7J@J<l@M5q&*t7PixKb&*kUB<Rd6CmX^>Z*yB2*&ak)LiWB@7l&nZ};eAFkN`Hh)pSA!V?Z@b|sck9+}ONmzpm%-6{rO$wb@8-gEyig92n)wAiUu#taW46+{<cLqzKabp|qEl>KOj)WSXz=k^(HCSC5z#G*q`?LI9(?f=(YH38NE0+SfKImg=8oK1+8)F|=2h>zkNA~vfu>de5iAYshn(0TcFYj$wcKR`GVYNqbqL6$Ck5X~i(dUtZnP3+qeAa4K>(;Kr#bt^!udkU-L88R4%K>;We*o9(W0(fE6m|2%CYa<(SKg6;cy=Hz<oRZ*8Cd=3wQEs(0VYMmVY?zwQx4hl3B?n9qP?QrZlH?*<>QzH2^3mZbBlJ;Fo?XpC{Z+$=iqlrtTh!QAiUw=ynh<y{)I#*x##@-k&b~+lf1ofky<*WONDpCcR$hn6l6~Z$-(0xPII$~@aWw6f##Ctm}FMK`1R`@)&?eLy8v4VZbQNQ;uyhc|J0*mj)Q1MF&?lP@tbko-}+4BJ4f3(x6I=))1{rmK<`L2@t6Jf>U}pK-!qFZq%a=i94TVkWAv~{t8Z#^%)YrWsQo>d;Lv#4N;Z_j+Y!1yI_y$DG=$-d>AHd{({{%rpu418B}>{QOtQ5;Y1XmY?Nx!NNb%pUT6^U|6wHn#?zs|KqQL)D7oWgtA;1x@qLfDge6<{>W#e|{r2TDO5a~OPMP+)WVFIEAOzh>M!`mymg55Kl>%S!G{%P5Xz5>x9So|uP*K2q727m$?ft6<K5EX~V`o7SVdeW`}D1bG{h}3|cwYfqJ@UVn#hE(T>+f}hRlta`^=0v;|!*`E64A-;AIva&PDmW^ylW~vhs9$^q3+jiaL+jd9pBGBR(?RMLz8M`F+jk{M;CWObwz+KFUY0h#gnL;b!fVWx1PdIUO2BSwc%q@N@S+%HXy-HORmxZn?USxq7OCpjU)No<7=jomg^-DGf~gEA7^}gNw-SQCa}2qVffp7D<(Cap)n?JhQ$fW0yecuWW=+R8cTRFU!yX-Wmnr;hYPa2KQfxgOQgKI|r@%YMTD@NK8G6^M`Wl}`MqAi3u-D$a&t&y*;jKdQuy^SdB+xp@CAIj~1Q$E|Kg)v@<a2wNHods$r`tq<1iPoF+hhJy>Pk@kGYL<b%fpT=9gGSSFve$h>mO5z3s6h<5e0zxepTI)E;f^@mMyahuE+jo0oTaT*B*brUfmUYJeL^#QT9&=Zv+^?%EXEG&o^>;9tH2WceZGykHYk!SJCE$lo@1=?nW;Vqkf1g;mHc_fmdXaT}74%;k!%0L853;AjRAvUILe>B9Hpa42V|bC+^A8GpOR2&{HG>;`nz8cgX*~fN0M3)k4$oXd2VPg(XcjPQ&vpf@%M&L`%^3J&Ddj7xxc0<0&|{sXo)!TiH2@V@|DIY*F>S!8K00wEM>;3ay)XcBy?^MIEtCYt2*<Q&o@J=>Zef5!4E>`DT%y$oW16^czoM-UKRG?)&xod<6Fz{_QSWE|%HRaF^-w^TL4$6E&F;@mbS;J%oWH*o){>TpEFJ=G$!kA%U3TzTPCn(T(lLZB(1own<m}JCAR_)FeHEyN>t254ta-b@oKLu65ldeCh;1aqg|x97|u3j^);pQ=DM7MgE2J2=0v&x-N}zlKXG_Fbi)U5WtvNLpp8>0|l%!An*fJL7G*O%>@Ab5IBnD3J20iPBFb_zLwjv|EL<z71|rCo-nwGD}kH7`ZS*hU4`1FY|W{FI|`cem(mu@W+Gf<S4N_?bP_mHCKu@3M6$%ZJV_+h_4yQVu$SXjOjp`eLVC|$mnhxgbBbuKf1>R%Z2<;Z#1F}xd{&C#o%1|+jBNnqOX+nEhnr~Kd}=5l@9)xb_B@Ej(#f(}okz;Xl+zE4V95sA>4l;MdmdQ=BF>}Vdv^o5{z2D__h6`<^KUh#pZJzC*@Hl*I8t|Th97gtnWx+XAnTAcN95bzmu0Q5nX9PPMFY?{jJml}*|EUsc-%{uWi{()-E(2JMifT8ZNeDxlVG5KM*sX<|Kj1^n|!!%Yk*U|@uV1p`gVI6E_C{j^rKkeFV;6M{gVhKsXRbw0212n-Zcu*4%<6z1(S+b)u*K`Wtbn)BYzdd^=JAee3}lQ7Rlvvnp#3Qc=y`KYuIY%BPxhD-Hkq^L>8ovtl8lA#}~?K&m|JcYWT2dmnQ~3OkWbHpwu%Q-&5HjbHVa#5hFv!Byg|F0(t~fNr5iuT+u;>j$G5lA)g&rhJ^GFLtY=PZ!g&|5_b4}8%43&<`NC1U!jRHOq>yq@*$<9Tt3Vu0Sts%frw9|$o5z=qOypZ%p=9EWDNdoIAvWGPa^cmV1Oov<L#6W5LB>BKy77O-1Lzye&Y_Ckz3lojcv-8<DKf;>F?il2+8dd&c_chUVS`l0)$DAfT!baM#uaVe1^4*orV498s=1Q2Rs5P<r4(HMt)_#G`Au@gS8M%cNN`nlbK`AD3%Em>%CK}pPY`#A_jy-dcCNfLBkj~a&dg{m#;#|x-&K+1AwJz^VHtA{8_9T9x8H9otp#%2iiD(tWRXFh{_dok1&DzYQqG8ch;>5=WdKf`c;}|-St5YoZGFfmIc<40N%FzxH)LWb8^%8`2Wl?MXvGVAbAPUW4sE8xx+9#1CKkf?J<quJpa~ThiIz#kC&9##=i;P9SbM;wb(f4&$k@&rdzxLn~LAp6nio|oA2Z%z*p^Ur3cSgGuo@dAjhU`$Lc6LxhyAktV~~moNd?HK0P;V!U`^ycE?i`P#1~=oNcYu@UdP%CInbjg2`0p8E$e=;=m2{@$i=JdSqE<zdPaM`w%@yU5z+ItYc%3*-qOmhJ}BaD(@h-vsq$ODlT#V^Bj|2WwBw7ZtRFZf0)~?+qy6RJM-9h!V@>hWH4<rC{x%r7Y%2J$8u>9P>)U0q$$<?+7FE4wUVO1&wZk0L`!U^aC3KG_2r7S9sWHv4SH_o?$*j~I3uNO^wi_0qd${@^`kOxoWk}t2kZ5kZh!rocBTjou4@w<j-Z>!pe|Ki{?vzM_9FV1e3s;?P^E?uKh3@BrcVf{>(;<i8Jw{8^w}X&gjX=T0^XC`Gd<9bynD=OI}2C0fQNiI6U*Rks}mP|`9S_@?SrF*T6+PQlim`D)TYChC?{ky_Nk>E*6tVOIHY8sP<5D*xDnbVEDM1Uj(<8KXS9esoV_Z&GpE40J|2nJC+?`Z^NvSYKBw+xV^A(z?Q2w5AEJErg#PLZ(mtg%J(Sfy>Ue2=kcWcLwB}7ob(IbbzOB0ZNZ*7(0~0`<Y9B<iHB%b3#qHTj?BMeB<WSIS$qboK^yQsPG<zZHgHP?Rm>hCPFma<7;oekzv9jLR7y2ZpE@nbfYMlH%Dp0HV?vc^RT`B3Q=eTwPwGpnS=s0N>&e!$~U7#I<e;fMMg3+tPpEF!uk{OV`2AW$^vsMr6PXXlK&a(Q!n7@*7F*KgM7E~WO0<zGde|6sGGoS7E4-TR-X+$6y!S0L&x*EPyaTwXR&3%|tVG))fqs>4y^jeiPP#1FIxeIlY25wK!%mVADs;3(ZfkRe!qaJw+wYbg<D%<*PjVT+Uz<(L;rny1AY&sZD3R$s@H($o6?K>M<E@fK_!(puy^*i&3j@^%}{k&7y^F;L(%~na%1_q?B1cLxd2EG(ZnxQsKPV+1eG8S1pN&(oF37P7P-pESz>{wFN&K>JzDKEylE{wBa?;+m5yNiY~Sovqw5<%Q@V6CmApS+^c>RKG#7TG>vJ-j}qbVVCXe3Ae)2JMxaQ$d?r|IZ5kc4cJ{XQ6cI7|s}Ssk~@&>3-$059-LZ5jMBb)%uk5yQga4T2DAar-C)y-5!ZUhgdXV!9-6%%t2DsJW@{GO*Kek4$Q2&*Hiz@l~5v;A2yjRFav+9zV0z(E=sj}%@{fDwg&7J-US4je5BCgDU!jk-B5F7Lh;{r_KI$T-FIy<d><zMgf$Kl8Ch+BsG#6vt;Q=zqo*}9l=3>=MYdKK+(NxFsTM@H9Ec0Bt(WH?)g-%(K3!+E)LJ=T{NQ|JbUa(ae{EZf4NLoNzb)5)Wn;P$sG0!VV*O}--34ixy?C@a{j`FEKod%0217dCj@P~o^!LMFF@$<!z8X>=ueEe53WGS*T$8wN=&+|N_p%X2hkVB1jhndElU}Y^xASP2aWlueDVNYdGdiTq!^;)_Fs0(y4OZudV^8##8_`$PfaWmh`iw$U`Xrs;RlO(Cp1RbhNSQg(=B%m1-l(sAZ*;xFlr#Zek&Q~%Kax7&N9;qS_P>s>kMO!Z%9$j-h||Ic<Xksh8)>7WL0fo7F?EFQ9D(H{E!I}W!XFIEIdZ~<pPD&D*|2b;@K9$UOLNLmXPJ=(mY_|4g`jyS4|!h=#q92U4Xa=`E#Br-59{TXYDKlLX>UEh3PD<xB3Ao4Q9igM@Q@s=G#4$QGwlD3nPhh|ASb?6&lx)77EqnbMH)PXa|{1dVx%P2E4P28c;*mJk<n9tu8a+1X_h!z03XjzA4F>t3<8gVV~R9Wf)!vIe~-wd)kO5V7l&=X10nYOECVWNX};VJI_#RNSHk1_FjMR)f>oM!norR9UDOYo3B4D_P&*M+eB;F7TV?ZL%81(7YNsSZG$KmYD)}Tq_8_XgX?F&Q%(=-E0OX``C{|O;8AFqSrC({L^)fS9GT`u#3!l%+uXXrKY6?yiSl@-MY?%f-MVQZ=O{{t|9D|bYkZ@Xn$xyOw#X*YPe|bC<9zY9W$E&dSEldX7^f*Lc%B+sP^h@dQi`vU^a5MK6#$=*=Rr4S+hxn8t|Lz#G=2%<s$l8p2f7q8?aLt3xlPY*SSsHA%NL{uJqnNp{M>h?d3#!3%(6TXsqcHFHOFu4BPIdruqkasl|I#7=s^bc=XyrWpER7V}9aRiiP)24oi-C|5S0kCs_-C*y;XaI~Zw+Lk<`uIl&8%Q50%gKU)mgRZsmee)(JD=UTc>zWUNr3Q?XuVAvh6ve8{l&2t<2Te(k)xP8{JHDE$QuJ`ru6Zxpq*rETj%M*RIj~<$>w&2cr4IcbpufL}RD#Pi<x=z+@e?gNL}fLxG|LT47{NkPOuxb$;uy=WpON5o48Z`&r)qAStfIR%POir)qCd&Ck?Iy4*{uy-(hdw+w`1;Qqnj<jFX|5+gNF)N^NenpZB~`HS!vq)4_izMnQUxLC*F?5F&Ygl$F*So!Fdo2}IgnwrCOtN-^rwU|iyDqw*E?i(obhRJxlO;-KLG|7U0YZ*!K#NH&VSXSO_QTxjb`Y4E`&9w$8w+FMGVEh6wZ>_{-q2;?VSevU<9NqEgR~TVHhQF!?nST6m>*?G_V)`hPNHe--RYdd8JJib8Os#P)!4Pi^w=W#X8F&Yj$`8#ygDfWFWqpIqBSgVm&fqQRlUV-(P)Wepi~-eKMN-*1f4@6$MjZ7N?t_CBiV3$}@Te1Y59Xq5<C;R_n-i8uX$XROY8Xcpluw*rJcL2CL!tr??rOAO!n<f%mY)?Q)^;oGI=8<0d#5ZiMqUvhAA$5gJ#1dkp5>U6Q4^N04|eHcrOV^LDvl*A!x;6FcDQ{0FO{Jc%5;yOjxL3h;XE9Ndv(D#vj?<H7r(qE<nmA~i0Y4-y?7@rcEZ4SMBJOUX1__5k5?&9M3Eil3-!8PL~K499Hk9}_n^>oPI;w^Rt7)pXRvw!v3mU?yUZ=H8#5!5tAuZ%#_0a#9f8fpm3(<!#b=IHQcOlJOJI7J^>$muzZT<M1|top$^O4lP4`zUC>5JA_s9FGH%6MJ$k=ij7nU_w4+;@z%!4$1dMCY=Mmt~m<!zH;O30Il?t{GbRzd5<?35zS*nKQbK-w4gO*XkcHc2KockXVpp~M^)S83tV35{ZWBvwlozsN_^bfikri5vt>y9uA#r?({YBY9)btOF1CQkw}q9~+0_J&^R{rYe5Qz=v@pgNaK0Y9}?3zv87=2hRme`l&!aXcllDH9X1+GmEWxAgG*>m$v_p9{Hn@=>e5Zpi@^WTJ{QPZ|r+*K6fT7l@IIS(B7D%kw$6DmbpNgb8nuTQ`svzM<NUnI26GCoG_`12Cq+ynh2^#^MJ%?VYLlrecD)}VbUq&7u*HN{nNm_rl3350wu)_h+XR+(pKrHV+N5z&p^>E$|L0F3sq=6Uh7geZXH(&?GmtBWS|cXS!W4t4=cTZSh_6<)c|yeYtk{Ha=l5xZDsD<i-3N=NY)yg5dX;?CN^GZ!40WjDt&~sGukX(?+^WFx0B*+t82Q?@dP;_|1PJF7T3;zbt`aNULBxU(4`pN=p73DKR!+as~h>;F_9Vi+F&aqMp%_Vg1F)@=i==fty*B>-p<pAj+V`c8F%lfP^2XEd?qe-sueCKyKEcm56VvtB6a~h_q)worCT%0>l$F)12z_e;2s6o`M#9CD{qK%qlVOa^h3=`DL%y$o<|E@F%I2+dXX(k^KQ|f4n}j@dCDd#XGfkZ>)jL)`xR*dW@L1^f@R+$oCJPM+LH%6&J3LR7E9%8xA7w|d#VdZS0g|VRSsdx`V@&154b*fi8Bv4Bl4ODVr0?e!EMfv8yu5#&G%h+MCP1XK;v4_>vlCalur~vJ-k}QF7!(&1s7|IRv#?$9Mr|LJcQic92^KOx$oQod!jP1I53p7no})BIkQ&VfSZ`RjX$dM`q(PFqc9wS0?fr|8zj|p5G=IoeW=E8zM}zksfMTwlL!6_`Oci}4=}X)_;>kMr*VqYgRNbEr}J<1-S182Jc3MAmVt)<=17YUPqVlfarMkAEq75O=Ib{!@e&aZ(q3C2a3gjM25{8f-%YORakAD#-5qcanYK<>K%Yxw#WJRCg3h$hRdb&a(;aAGCD%6b&6x4A*X|=_2GK>IAHt9y33czzB(_fn8n(wH9Lv%^&lM#z@~E#k#gGoUXDTgnh_(=!nz<QkyMO0;R1qfG0Y^d+9;bQirf-zI$L0yp#nJ#b4O8104PtAT)?3n}E1r&x$FUB4_Fu({XnN0#5WhW<8BekHLhpuofR*17C&-O%^@KriG$ZW^JI`z^8ALUGqvSuv4<>#6p7y2eKjNHl0Kb@udlnw7%yWEwyM70@UfKY~+{$2)MI)weifl?m0&4xwF%5A%5zX>ZEazA6HO5q`j7m7~Z72zl@Z=5bnbO08Bn*(;y6>AYY_Rj(@sOfGIfj}!PhlP+ssVr)IZ!x@gf#*KH^)?s{w~F^??S$kXH+*ND$tZT49`!9Rxgp=ZRvm?y6{NXZp@i;G(VzZ5KK&?5jB*5?P>I4D$ke_NWn)`5@uDXu3&baDz2{Wp~PzHPnXd~K}cV!r#r*3OC8GH{Xe;9vRzBPn1*ubgkE$~I3VWx-4?1tSU=PXiwn4E0xp^5aq@i`@2Q+<<ZWYtoq~DSz{ujB*O1o@-o@-~n4LR7ar6&~37B9)N5;zZ>Fgp^{maX6+%}TSHKneqW%)EhRSl8Jz}GC~6c}(0Vb)>8#<Gtf1{lHfB@8(9(qxfrN`!?WZa<>rv*LxKD<TgD$bK1<onc?;I+chiES~12(Dn63zpU}4C$u`nqj`qU#0e+oJ!ix;)C;LF!1W99yj9KL@sQD9=g#5VSM93sT(WsY1e@V8OGKJ&v8{#MJd7?Aui%6mR{~z5%7>BhdO1%&Td0oJPo*B*2Dio_{@f>Kg@}1E7+rWQ6Z+H16gnigw!APT4RGkl;kznI3cYD#ZK9hu-?tBYt?h!@$<lhG(TEvopX-Wr4JR0w2^k2ifM_wH;4=DC`)S76DI0KaAq|J=zvwvTQD;M8V=B7=x-XW%?DWYprQ#$KimL$dTNxdTWf&J|a>n^G*-}JX`07!O665P+#t}D1(0@A>kLKQJ|B1|kNTFdV$;W3SxeDwf;mvR-|53e^7(W@SeHNXeUIHA0A0v=D`VrurkbVK9GFTopP95@nIVN+|jrvF=^l|P0cC+`e=S(YvDac=>Yh#3vrMuV@J@Wc{K+=p%&Y}kCXB31Q(1I&tsf9(+jL!;fh?5Xr$%0b9T^cE1a0|p#Yj+XIE5N{k2^c{s<8u=+T-x1$>t*;fS55Ze`OQw!UGgX}B6)+F?JS#t%_+pcxO@>Mad1q!Cm2WFU5;%3Zh&t}{M_sVs8=a8CUyytxHk}FKFs^2rSCE&F1y*<+8Oqr%`am|7qda2>uv5)QEQAnGnAVDm)%hl-00QHU+WT^^mNsd48JT7J79qXcyr9UV(*dc3^I^0t$y~aM)~20yS;mA7K)ZrVI{CZ7)a#qfUj504p8ekI_IS{^)QP$8a`9D`x+%4>6WijCJf3DT)r2sTH`Zm<;mpWZA!`83F)m&Nkr{FvUE*J7=bM~<Kx+wqh5&|!hurF0^Y-OT%k{%Hsvk&^C8mwiQ43H-$cs^eu|wCuc9q{h(9>juY{`}+>j{(BaR>`62Oga2k>{Iq3pt_N-QO0v=dpBHek(YNysj&zkF$wlxj<+uQWflinWb4Aq2l(b^t1hI+uyzSiWLtj735y7Ldio2z+jqc0dHmkPj#)7JKwpR)V)S!2tHzYsrKWfgrZ8&m%CN{2BX--Pnt~Hx&chE+Y-zLNvkA(|BCag2WTxHepGB)xs<}DofY21?R&T89OH~Qi1On<I|tl@m;L~n%tVvpw~UT1o~pe3M1g|hjFwjWs8R5_)Y#8a3)ROn|CktzX*s=UZ2KFZgH`4O8@o4Kb(~^)Ll|PxJYd?D2!HtH$G|zj?&y2jLOK>yS*_}aRVCS^V1sB$AnQ1Jk?^T8#TIAF^Oou&{QcbApZNR)nh5CINDx~(geyqpiM3zr?#W@ub47>H?`ypI}C`1U=sg~g`}W+g3CT+QmYMeh-g@8PQR4U?Mk%VE<~+&<k$Y?8R!Wnq$nq@HpT4$$?=egMan^8zWA+9FC!WxpNf-$+q%^hJ4LZ@(t<{kDbq@Ii!)&EgFW*o{E^@lvRg8jwfHrS#>N$5`=Kos5F#%0aKawAcdZVQXGC&bw3h^nP4bERe!q_nZ(uQNCl<?(KaQ(h$b+Lvw~(UmM(P1o?wdR=h}-}v>vKtzr)q6Rl+5+5n(|&G-T!7qx5V)1qX{QLZWP4U*>X{>ykj!H<p0Ykj6d#2r%nP})P`QCV^n^wPXM&55^lC|%=DepFY(o$HzvO}O=+P%0ZPp`y+B7;OGU$El%qQY^v3iv*lKv2+5gUZ*q9gxQ>wNL)|gPJ4a8^f0T7M{%@<Xucq0N|ALsvX)U)`Jv^+HvHnE<J&g#@A4m<MDPRY(NQxLwP$=+{)zRH!tLa#xsw#ohJnw*OC?%{{;kj&SKaJ|dE3KxX|^}%=6jCV^x@+~UN*l0mAkN32H5-|4RN)V`g<tFJd3*Q(u><dE1Um`Ymyy0>8nQ>i!_O_%Wl^}y1`v1Q4U5bS*aB`Kie6<z<-ek=LPbGKkC4hKj?D-y*8Gli7zb2CDT7mw%3}T%kukEjebNPW&jsUzH3gEaK6=iT?jP7r+mwH+a2zsUX7+c|AXIh5dZrtK1q^_vPFzls$7Vye^U@7=O{#kuAkTHkYA-LT3Q{wnRh;jM%vM2OxbxOo+IJof9kJYOV*X=3#!%~J-l4B8(0AGfyM2+s>&H{L&y7d<vu=4C*#`m}Lqocvu=|e#pcVO`&fRw5kVXK}B*#8_K*}Be+IR!2zz&@A+lwOY!RCcY1Oa>4w_rP9P`G=Sqn(|ApP2{)iM=R`4!s1K39m>-_<K^m)ryagJ#}0frZJWn<HX9!c=~zpA!_haYR@M#eQK)Xy`f`v$U|^>{y+1yiCRmB-=qA@&ea+Qvm7=mh)Ov$bUij9K@+Izzd_26<DWqo!G>I@*6vr`_v_G$4+E>{&Yat_ozkuGWzLn@c=j%Ef@*nKEoq=&Z4AI#bU~xYM!AZiMdjvo!je^N0GJ(F$XR-@?vk)n(Ke>Y3kAejNrn_KYLnk^`>?DAnuSC$#DZxX#`nCgLu;)F<>M?0puQv7uaG6L5NBJ#CKpdw8<>tdQGQ%MzGX0_5%YV!8(2#8~M&LRxf~TO)GQlUP+2(-~vrE*4FlztND7u{RGOg?lW@CAk%2ZxES^@`J5STC1+W=S3rGF7dz$&;;<hP~^U+ueP6m6?hvZMaPvB8(kX6-ZAqV5j8jkP)7&Vt+jw2oo7DBhHW7|Zs~Ie|583E%T08ixF(Q^tfDwYUiyPa){wZA`GaJU!O15Xgt&Lsp1y8w@P(+!N@0nZri=XzOPtfuKK0hovx4q&V)r9ym~xiJY;f_mvH?El)5?OjxZ#`$>9*N5i*p1f8mJ0kmpy5f%KZ7^1cYR=5~ifYxL8ZhNVw?Wv_f0b(=VA+rN9ID^T3cs1?^@5S`ya@e1wf5S>9e<V#>Dx7OWP03wh6x#v8^Y3Mi9-DTYg}c3DI4R|BrXb(y3<pr50MK{@>f?A}lmZ8Tw3^nHrQi+eKMDyA4JVZIT^QBeDoOLKIDpSGKy73|^d4drPaG1o;1F8@Xv-4C-w+M1wVwWpoS)frER7-@&d*aB{_mHx<&JuUy8VFYnZPhZJl|GGHAg-i!qg%sK)twU;+{hO_UIzJ;&GQs!iaPrBC0Q#`X_Dotd~AygyaJaxq099-Qq`%>@8Isg_&@HAe3{+S6lhM)1co)n!-rw9|H5jJR9tubr=)vdEoO0=s2C`7-mq(sL9YxPG^SlW=Fa|@qnwrMYd-Bj<sgDWOlsi4^UgFV(hD67sTK=pTi9!+zb9$?KkvZh*_EsNN;)jkb|R;Z|D!X;Ju89h$WcTA3($YU_FQ?N8L9sOfS?6YODCCXldDHDbSz0TQ{&^kxvW@1%>R=mT-wV4!xnLKuj-$Y&Y2thE+e>=NutkF7)kltw){9viRwWvfmCMuN^Tk&{YAo7lecm&LR$ec{@Z-3jy}{0pB5IR|^Pg7qwVZk+n&DHo1Ku<5ta@JmKy5?^qBET0|s$XkvO}=MUhwu_ps}bIOK80*l=H(1J@tLn+`NO;MERwRb-L_Thk<00r~(-~G#b`V^>`l|lLQgR7)0dgxk|G@OJ^9U=a*F*IFnu<FLmHoX@1c(Vwo2t>_j<IEFtwo808ohss0t~ELLo}Qpc{TTx_bQ`_)-U)v@I|&|2;?^~zFt$las+eX(*!JWKj1cfXskSOf{ARjn^3gyBMpE4;ez=)+n?NbxJR8+EUyEQcUd<`u6dZeR!)hy+Pqq9O=^f}qgj7p_{w-Gy&5=oRZcW0gqZIB8u#%`E6S#<aGeDg+?A-%$JfhLmSsKwthXXv_(Tsx!dOy+fWR$lIx{I9k69k3Htnn-(nnR*KT6#C9FZg)8L@UPMA`Uv<4>95Vg8K1p3wHCZnTFPjyTe!vySYPySCOYD|721}FHXZ<@5w)s$w4TTG;O3iw;c3h!&JE!rZG33oBIshUr&9q0Ib0px&%vNGOA}|LMSbf!H_Utf4*a_C=rg-!h@S;Osn)IB%!~J>V|&I4^^~ViX(uv8eCUr_KvLf@=9$@t>gFfhu|VH2&u}OU!V4i+?i^M)U=-JCX+{5XX5u=NftJKpszx7=5ZUW-Ei@bJ<lSHhEjPO#GUhP!i&nDUXXwCSwRAt<0N;(8#mKMZN(YvqKyP7-O_qBf-M#AT`cET&Cb_uA3rn#!e-Q~qAB@oGRS3<k$gJ*V<=qY4Iqx&EyMXU*(Ziol|-c|L(1Z-jv~RWBLvs&EmBD8I^vcv4aJxyE>M>ruF}j`92YjyLVd=^Kd2oDOu~Ly>|^zDcCij!^W6aYnh&Opll{+O538I97I<_xga>^N80sAAF8>(H0~{J9u76-Ja~vt&aPoz?#J+OhpV%g{aNtI5E;XcU&G06)+^1|knKx>dmjGmWX$mfwD)uk2D4P9=A_vX}l7VF*B5wu~MA&zMqi;K<nR{VMs)h*47&*69r*W(TEdhd1YC9%N`-*<f6@b2rad_L1Su2uCy68j4-7|!Xa##l=WSyV6-3q>JpC;p)Al>_pePspUIYOxHFC%8&qfuj5A;MhvmQ03fll!VKm1Vo}Z}%pH2p-Fu*uN6F<Wm>#tu$Q3!4gUjmZ{N{i$>^GsI71{_l_mF!brXl1$O;Eou8B-*Gxgw_lPXEWs*ogO1X=FzaKJ^+p>3#%ZKAtt>Nq(vN%JC!E#QNo;m+2*BUz#__qwSA280tUuXS13chIItj*<`;wf<uoBfdAWrAU9fyUX7rg_@GKX~}v<FuGH!%7;GA%VMU>F9^m*lL#HR~?LCdD$RzU^Z5sM=KNO`GZa}sUih@4H5R~cme!{jtqXyNfPg){M1a54+e1Ly1;~98Lty9j*D<aAPY9)3emIebvjphzQwG{=?F+xO(L=uoTo8|G%P$9wB8bOl#nm;3?!gPmo`>UY9kmWit8@qh*6&CGpcEafcHel(MNmS7lvkBcuEzXbBp+!<_|f~&oybm#2rw=W%t83pTo+9LwzynTb_UNv4n;GXxXaFBd(oP&VzbD@AzKV>2zy9E7z}-?PYP%Mi*Sdvi+Oxj+9#2kikqJtZ-gYFn&`r#O(Nm21@}}4MwEKcohYSSK?h;5rOtEMswVxY*5To>1Cs8OYmIRkmE?C0<P&$!^ZsNE?Gzv=Md;T5G{QEr@BEsz9|&d?zi-8-=`XZmeB8dUr6Zim~Z6ixSEoT`Rd{CR!4kSRep>qPRHs_rY|r0WLCPTwrD8(8g1BC{@lSKsM;k(opgxEWEgyMj@%8cp56+^Pe0&doxi^pN-~6L;ViVPBDm%q6aC43;dSLE7<z;PeolvORga)*zizo8we~OXjMaO+6!hV#h_`A6sjpsZx?fL>N>G`oZi@6;`itcBTtN%s@sb9`!y|D*$>ZMP<lgxtOd<hXzpJXzwivlF#Di=tz$EA87jg&J1urfI;%-5es_E1Hg2BRn<vI7(jdho5(6eRX1qGssIn331Qr?z!aS|mopwm1enq3%R9s<UWd=!{^xon=0F0?rRZ?sdRneovI%?ez1EVt~o|J>yTYFpXTqdl({5B1km*J04&96bBDO$(VpQPanR9PkLe=cVl+k%J`VfpV>j3JtUK&Z>k=s#Dujp5b&r((YB^3^k$vu+=iXFp{#)Q0KmBF+Q_Vkc6beFDHGCXT|gS?xdXDL9H5=y39kx<93bOcDYtF<?|uUkl=azaDO0a4-GoXI0XV|mjSlSEY}eSm`@^hQuC`-RmRcDRvp#vMbaid7EK%4UD9jyI6E%*&>=R9p{BFbyJw3q@r(Yk0@Q;qf02=`Wv#M3D(|JCwZ5EXJFijuBBIB>%>T92yDKXad8JZ4F!zTYy8H5O1PJURS8S5uV<k5ad<{rvxYF1(s|=!%rP2THuOO8zm<OhPl4r5)4ON@`nwptL*nJuWkqGOvsUDR<Z!kzyttdgz>ss}d8&KzDN?*VOR+(u6p<lm|IEJ&dp65=~Q+na_+{Wg>ziK7<^_9P#I=W{OT0Zl`{&VGWqwk9CO^TcJp448Om}z2yL=e?smRH0N5<2E<Fi8Nc1$AQ)D4FZ}hU>z7Q8#}$0j6*?Mc*AJq9KL4fBA6(JF$7Hquld~1^aaUqF4snm3$Lg2hlR%%J<Ep7rc_&5}&k{W^LiPXRpe1p&ZHw@&QS^#fHH7&45168FAnBVMNvl=(i*78;UH`wbC5fDX|!D=8l;gu}XCaDBk;vBmuhs(`9Bpl4|*dfnO;Y+lyQ$1!*OAd}(jI&;pCF&+D<Dxl$xG@h7S5&#&$`6Q`jODfN-qS@Zu6Af?hAbkC*>IOh@zlCbtfYtzp(n~&xC=XK5I9FiV1hZt&>H2Qq5zN*XrBjb9Ea<mK8(M;1Aa=lfiRwQpfk1{2FSvogbis=-^BGRLmfKzbN5*6J%ktd=8Xn&K%6Yl#z`ds^~Py9Z>;GkhoiM-KSW!MGrytf7YS)})Y(o!(;>zZfpEOyr%vugas?9CT#Q}G1QA8Pke^KgVRyI5BLlNpld@wU8oU!<R0r^2B0dp0_usrGCP#U#(6>cpJP-iCOb1;qp;r>tXPYW)<=SoQM-5q2<Gb-TDOTS?B(KfVjoDXjYJjhtr*1WVo<mP3S+<4Lt}%D)L#NH@jfUsAI6jx-v&$$ofEn$3Y?_BfG^|8~9L89XMGbj}{%eH12WUOQHI*UeM;@dR?~%(vB#(-kGI`ndBSw)A1G5W3fHo=SnIXdTJXb*kOZ(e5BwMYqUG>C#35l3TJ$?N*qNI;gD@U7d3=K7w3Nm;c)$jPnNI$BrHj%Ia^u6vBZoM1Z#@h@O;Xw=99r$sN$YqL>GF;tubd)SEZx?tcVJHrVQ-#M8E1{LN>mb?TT|J>Uj&e}8z^0w6NzuBg2&hQ@8jz%CFot(Llr(7m}_F5H1L8Q{}GS_s55Nc&(weB$jIK)#C%TsZlvse(>9%||+VS**#_UwO2s3ZzOaStzf_qn8mcx@_|%`kQ3L!&R>Hf5L$AOMN2w@U-Mm(lL>?_A5b~V8ODt72jtKZmA&aF}!q*s<)#tEKCKn)Jjpntl7mA#9se4+G0d#;aYxoJ<_COPP_2l&AZCCVplQ@S?OO$t|PFbTmd}P(H=wCg0F(+)@!8jR-=9{9fo#4V6J!c34I>BG^U!>@o<I_l9+Og)4zx{<Q<O-&$(H5q}gpGYX43hlA=#x3tlp&(L{}wAeP_iEk7u0B5$I352up`N+Vqq!pJKrOYj{EvCMPuYSb(5YXJ-D6@*=ua1|sjALeg5B6wY(QKZak^?_-9RtWEHZm0aa8E&keZ||Cv51n8RhKh+ZXSq^$Nn8aCKQcKjOtg<V6ns?`seo^23YQd1^4-AHh3dlm!RT-d8UzF>wfvUVTyJA>i1II8kSnumzOG99%$h`S#+@*x9xZhKOHDdhg?*8yn5F@|Y!ei>16p0mQlM3Md2mJ`!3?*ycde-=Sy?FH+Z9pn_MWG5txu5`b_d%!K<F`DjR?_h4wi;Jq{;Cz{r@3T??kp^Wq;<o+EKYXeay5J9FPOd&Pag+eqOK#kdU=|5T-u|A~^E1o6oMjKmd%9_3{GYz#`xOXyD<^A7jG7VDI#kj@o~$YK9;n_l}k6H;FD@!fsrZwiwo`+Y8#-VR{}2;>xUYuCi6gy54i!8d-n#Y4k3eR5x_!%9bzD#S1_~P0e)p!=g=)RppMX>XFFLzckv<q*ajWeD%&Oh(QHiLL#E&gg@R+>|fL%S!gGj1QR1zyB*y-k!v8<^=h$O1OE@p>9xQ@0`ZeB)F?MnsHfeYolIedxY^!9*%!jAeruTB3Y8e?<Xl9AjUd<O(qA(U`ZM}vyNVUCF1Sb_aS|#*>vFVHK1!{umPS~u5o6=xSSmrCg^ohNBUJVV5F=>~FBjb_(L)LbI5_c!*Inqb9-`=jQSyy!^oH-hxvthPWVs2dp`43$oKUMyrIb3%5l-W!x;K|aU^HDnzv~lf9+s(A^VC5r8e)A7xzd+Fqesl2#(pGR{WB+6bTm{Cm+|DG94`RA$u1tNWbBm{q6Voe^;4+9+Mz{XyT>u&t7qL(OBroKx#4pAe^SZ=F~ZSIQxUh!brO&!W|3qPh&-FE70#EOR2e|quv?mbbK;fnM{FmINlmjhX#Qgu_Ot^X`1(oz{?fx@+acbQOk2o64GL#R&!5!FWC!=YUELZOSHu@My#Br+8&{kskco9rP+(f#O*e%Je`eZO$_Ou2sO|4LX$56xI;cBLpnZQ2-v*D%OXkI6r&ty=7ZPeuA7P(y`Vqsh$zj^rsAO)JhIf@@s1!LA7^m90GzlRpW}%YVUOLQ#C5qXDsZr<)JPT~O5v}3C>;#wlSi3_N#qf>di0$MHOqj{fjiXR|Z<Maz_Y2(N23qOt<-(^Pw=SKKbh0WIO%VQr1Z#68UR1e1FU#(XN-zW!#Dzl;KrJcn8L3ZfiCksdae48L`~k_4%(4LfUsYYPdM@h$J60<W-Ab#?6dwlvj^j#F!{>*sxWtLa)bB@vhe39Y%Y=QB+LDrKSl6o1X^21R7R{l%OSvg(D1Ok}mrI0)VyQ-~nzO?_aEjou8=NdKO;I(wfLa+^r~#-^w((sP&8xMV$8PLT;X6R!bzBeId6zb8GWnLuXEoe`Q75T_lv_L<{<?6X`CarQI$!>QaWfFQS<1||Nko2s^c~zh%8>RM0iY^2TQ=QX6tSgA1c-kL{)m<*cnM|YqVWY4P8Q%AXTAaF0&$&r=F~&98OIXo-JJoWOsAI9`JMonr$Uvnfq2fer7TuJhocM7t&kHctSDYl4>g|m<Bu##>BYlSqA!L~%t{qZ+aj;T<anDc#|NO~ulmpaGt2$bTR(?}19u&(ari298Fxaz250)k8XUFFB12@4dM`c+U<6p8Q%tc>42r}vTQ%w#r}NKNky53tqi&($n@Qg2WvdHx)7ZLc216$|7T|+{)iLq_f$O^4154O{<b`HCbZp*L6POzmTXNV4q)KBPW>{(S@2!#PS7B3g@0e-s!&mssR^{HAWIs{NXC(+Q826Ug)qA08Q`jZP?=7$P#zQXcdv=bqoLe$H+Le<px;e|pi-|Jd`>JukR2EQMGpO`rPAMun(}C8d149ak0E`l^ja~}S2D)X8pprFZSrnPi;7>w#tsbOJE3TI8{Eb8?S}N+))x?DZ=99yBK)|l~OM&6@jqqc#&w!+dUm~cNmmM-@4A@mnB}07~D7JZ)&(wd~r@j&SB^vsgNbI*&IOSobSS(`8xDW;m-_5~+91d?TyUxMY$lX;UZdTE(f^oL{UZR=sKWv)%#{YG~MzQ9m9_N>d%c`;jMH&`JPG3t-*xv8bX&7^{vp61$-kIao^YF<cTOH&r9Hv&22QCda$5I~}LCv1lm9rp;1d6dU!88lpR`N(-GZzXT=&V?fbuHne5?+=VVxd5~_;h}s=72^o;^VBCei}(%RT=%Js8{#((2mH-oj%V?N)4>1tnI&oI$V-zge3WBZ0Bz6RS|oD@NzQC6K@{bvl~M0D$~l8(-6kZNF5qqgsI378Lj_mrXlQ9x7?x`Xq2TI5R_q9BLX&UJWcx<TP*bUx(sAQL8bRRKFM6C34bpfU9CkDCNW;nEss06X)+&hiz(0^=x**dyPF>@epo~ZbccPc^FiioZ$9HUoZBie0r5V>BIdhyg2E_Rb@;2;1SnorLS1Xclu2&ko6WI-0UsapI-&0rG$2o~8K-ujew+z^!9Fx}QhB7fot(~g_f)jxfhh;Znsu@B1}0dl?$W)igHqoXv~-aj!YIb>cFM}6lh>OuN+-c%WdZnUP-|UbME?tkD+m)<;Tw=={&}M!?q0F|FU=M;*+jRbf{wnxmKa`t%c75qOyq94riz#9PXthIiScIjNk%1I)_GQT+`uzf9nfGG_D~*ajmX@Ce@xKU1^ZkkqT*PuH^7@97lK-KqL1xEBC%=V<^+EpdQymm2h^hoQ-;77M7v+kzOwN_dXGRil@!`_HXnRrCu<ntNS^HTgfe8KPjRif_jge^_oJ7GWkoqKwX$}J4tyYl38=s0w9f%!yoRJjGEa5^Og6`ZF?Un`yw}ZjsyphJe0=lz`LkRcp@AvQ#AjWfvi2^<uz^o}Z^0bCKr<NVKOG%0(_KA%9LAWic^E&+8l&aH{$_1PR(IvtwBNUI|Lzo0nVLl=c2#?A4##nic8*$Q0MG4z<&kJ}z8BwsgMpvR8rct%LxT7n`O^M_q9dwAyo=wXlb(uqf7heWzq6Q?kdEHS7tTYEp!<Ej)~1L?QJ1VA99?x|0NK=fw=(WPg?#sa4a@3ZK(>`G*!B~y;gi)4mqKGN32x(0jJGqdnH-@f$>}|yNZY&X+kJam>`$UTW<8<C$9f~9Se9|HPc?4W0Ottepsb)!6d=yu(k)k0$kuj%&9M<3mUc=I8w@iTo3=oT5;z;51Z@_=?z>uG_`sSpLq^l(Mb%R*OLwJNL8;9EBOFPT9uPlQ{wmCcYfBA;U*xEHg3-E8xnz7qmk6^+J&Q*@I*`9yF61iH)W!=}l;_B}^e^uc5}|_*);KE%RU|OV^*R@wIt@lO^h4!$gp3uk@oWYfui6-rovRLGMIY)pDUn&om$gB=f!kH0N0V4QTqhJ$+~cz7kZH%+(`gMtzbbtzwwgb~Q=6{L;kKpYM#buC#Bnkj9l`o4NH)Kd-5vD^byI7v2ROr?rs*V^XZk}&*y!-dusGn4A8S@~*2Vnb&npuF55_aA5xO7EIhp%M*W;{D$;@MxjJe6)urKV|fT*)BMkOV0b1U9KU%;fO64~6hJ8oHdzyl7{)CM8%@H&A7R&8*Fy|KybAJ1cLe$OF6$uWE!el<UZ;2hoY6<lxT>&OzK>f{dT<GS}GBABV?t+N`4Ad)a7I>$s^dW^3{3PRx4Al)f0pS#1QneTF4J*v{STcxl-m#_-q@dTdQ_XQs@m|>*?z^rFfeWnf!tYbl{{Am2P0!xR0u~Hx)wuit;q*pMar)+WYcQ;^;UximY1`GiTXKlpFu_DDcbBJud%JZV`2;H?QWiy$c8pMvvXgfG<BbJpuQN2xuPQ`|k(Qr=}9OM>ldBE%}R0uZkcP&bzuey6zWq8@s{kojY=qviL;+66gj|QvdLXIJ`q{=sVa+G+C!kg*K4HpDH=dcIQ1Ivc91g0q>sTXIB;f)N$f~o|u0A$8Kx4R;qv_2=Z%*ZEi;n+N#e6$`hpUXh)mlSTBN}A3EUkt*+08JjWRsv!Tjiv)287eEkx`$-z&67e1Cf(I_Y`5OvA3!K{5J4S$HHaR!C=?+{Ivx~f^$C{SOk&2W>^i1R=?~zmar(`*L_6Gk3ynSU#K@;l9(%$KyI%%A#FKia5NNxI&^Q=@-B>pXx}V3a1hw#u1X^N$jyz}E1s}9(y|AGP9aq7&Z)_=g<Fsk@1WU;2D1o8eP5b(8n5Ead6)RZP#gys9F8WxcF7dg_LGbpq;|c%M(DLDgrEOrn4>xD*KMLwplbgfLPOMB0F1y2}8(!^sfy2n$^D0o%>S1bDx#M^wrt}J(M<sG8*j%I~cf=<BBY@^VO}<vXYW9i^d?Es$9%5;Q@%*Jyz_F7R{-|<yn-YF63Ei)9V0dgh-psmYAliNGOHKISo2@VftaEb<gM3c&)KDOb%-%?Ld5tpg2CQIPheK>YA&!D@xP=T5(CHO^JP9F74uMwDgcV$0N0$76<>`58vbW^$<z?819{EzE`Li}4I$AC`H2e=7o1?rDNbq5aju&J+y)8UdXL}zB=+jHJ-=R-Tlllsfh0e!}&kX5BA@=j%{+WP11B=_~i7c;VLk?dA`VPIni@`WaXu;ztp@V;N(R&D-+IEhq!B1B+?*61@X<_E{!~hQjO7T1=B?tsMM1?s(ZF9#ej1yKQ8^0MDE50W%+QitH@LK5k%jPW1!vJ=)>%acXTezHx{W_hx88Kw$511vRwl0l;`Z^CLei<KgDovL@(d@;v%ODNqTOP6`Wi?O#_oi;kPaN9Z&0i<yKuk7JFS6Y0P>o>C2v2?oks9|eY_6=?9sc0+5SZ&r{0KrSR<CPKB9g_KOk{~udkV^{E`9+;LZvlj_aS0SX|GgZi2kT4Rrvn@<!J`|RP$&VIOl?5eMLH@rB8RgV>`;`Pp7z4y%PK{qTn;NoJ#qQDQD(Ld8y25V(~#bYH6`7#F7CDOKP$=p(LfAPn-kP@K?Arh4$}wq}bQ6+v8N)fu4)%HC)y?_<3B$ASP)g?Evv2!)WSpyh5(`;I?>Wb;!Gb@8}^l*u-^U2XM7M-+?%nWmI6L`yhXp^U8KeUSC4LD20J;47C`OAQDEUE4&JYlKdt5{kKut4O!{wPcRIn{9$bAk`hx;%~h~XX1o<d+Ea>lC#pWfA*iE)E=tCoBmqldhRw~8Uf>lURx`4IATL&o9lZ(*$q3Es3k|}f-&x6$h{55tIrq7g-jN}Ch3XwF+fYaCD>;)~l12lvmR1b%sMyd)L1xmtQD;Qsx8j^R`?N}8r7h@Yf*J)Zbl0NCjQk0ziAK8iO^I13t!39Ua9Ek8K$8om@wjlnmBwk1wx_yXe51Dor8@~zq^)Xi3$PLH$S&~PoN~s1Jx=i}e&ti!-)u63DhB#AMqbVTglXJl+T9U0mGA*aLf($Z_RUw`*Bo7cedt_!NQ`z+u)Y}^Teemq+g}{Nn^+*dbfQ`T*!*DT)kf~Q7Z`<8uoM>sj|#$;+C$`D$2SdklIHHLScNBm)}Hc`nG+;ZVvhZY0&O|SXYex!_G^r&Lckju^j29M6;DIZr`ox6KKj(SNQ_T=;nHcnU<2O@{K?+WWAQJ*b$<UT0b{Op=YnqY_U7@<#-i;gv;rTaZksrR@%nSfr$9(beQ`<}4h{tGGZ>BFrr8Yz?!iH*bzxeK-MGOfgYVjE55zO<>X{_@`Ppr2YkR-<&eONnvXZJ5K?f|&8ni|all-PnhPO)56yCD<oESRMIBP1Ud-!mYh*7Hj@q*SNG6o@veV!zLQUKecy^38jW*QWNk+S}#MeUr2L88!xjLdL9r^&}g<`-71bTwKsN6k_ER{C3@vb`%)9`pCO<v`}N_h?@Y!ZP!1oWZ$r-&?-n>!PufdzQ&wA;tCHEbf}%BK1DyIqAo*$)t>Upl&afGQ{fSIUPq)L~E(!-y^nphCV{YGGbTPcXit}ih7*}s&EE6l^>q9O4-S<X(D^VRD08mb1fiJg8GrK^lb^0N8CdxxbSz*3y4g4``9lr5{IQ|sH)|@Os|5q!n`-ZwqWU-<6Af^FZ0C0<6T;4j*3O4i}kbPWy3)f#u&5^H9s=8*U_F+&YBh0=|!Hv67P*8aLdpBjek8)SDx4CXfxF4`D+vEgdw=#|280j8blZ78J(%+=;5nM_z%k_X5-f{?q7Z-VGh#t%g;LDxgOg=d%!FpDZ!~nynEyNc9NB#{dxcH-XG2}&P8tR7-gPTn?y?6*M#vkh|cD(@?U1bF`JUI5$|l!(Q#hQ6?uxX`yD>U0Z38UF-{VdRz%?RZqWgY8DkJgw&V~jjv=231uJj2h95!V%CIYuUNA%pRnv+be*n)`yl&r6R4AMXITAE&$a>ZK*EwZDlZ{gQi<2>O*+P9yir0PJt8i;1&U<}1ANOE}vU^UPG{MJoFJ_ggV1R+&LCm+Ge~4Cr**__h@$>*=nKyo?2D!O3T-0V_4g-b?Fn`X)4G73;;31?4Ol*T8v|co*eTdY!^Rw2ttCvTmnAn$~;qUX%S%Pyt!`mC4LE-clUf_R8Kt%d1?zr97Wf*Rn`B`l`U0l7oUf}3HEjMj<IOd|P;7IVk2uDVU#w8>X6Od@6ZjsFEPxjM^8#$!#!&2rxZGph_Ce<K3BhMZUZp$tWAYI+IFdEx!F!8wvd;bh6hCNMeL-K?_c2_R>$G50iw4#`S->TI<pe{Qd3*EVJnv3(sT*A$Q#9ejA+(C*mQe*AL1}q<tMp1H6Zu#kLLe%Zc$m;;Q2r4D%ux)~J9AzNFXk&dgd4qzS{Ck)>5fg&Hs--0#qnWT{LSc*lP=w{`Z$BEPf@kQ3V%W#|19V#7=&vafMA8dhT6_6qBK;>Jsku%b)Rhu@r}#9N5&lwM8nA=QFH!zwYz7N5rU-iF7?K2~)BO2)>RVb1@HX<Wi6d*urZb77IEI&V*u1P{1=|wpCdwH%u{#LeLKS*=)~&<LCj)ZrftNWJa&B9)^5j^fTKbTH&Q<ZTlCOo;OzGW+3{sjd2K!4c*X4C)jU=9HcW<DQ<+tMgj5yfGNCtxjrP83LX77LalN{Z#d@Ewb*9>%uXvVjR>ynh?z_;97$c%^4c;Gqe719shESpob3HzB^CmgMHt_?rR0<d1paL8=}!xA{3!xu~dVeVQ<PV~M@LK}S63D}!9*=}<lRUPm5gLGPWn-Md0lXUL>v|&G&#>Zkko3Tnzf(ZEe(kYc~$n9xfjBLx_stmXZb$f8{MNONEX()jOUDThb-)IyC)GRj?b0SauCwY1}vpCDCb2L=O{cS_b8X(=z?=<1;O!1|u0Eb^hJJ50wb*vq=BYt~a!&DrEEiw}^<BDhghH%;f60hkSozwTpG0`oH6W(S{yTA}V4FFN(z_hEcgm?7ca_8K>!#@`mGuXly92gvS5XFeL%^d<K&6mJ}w<5Zh%Jaj!trgfmt5Zxm&292jcT?k$Z0r@ng5<|m0*y#f8F4AB+aBK(@d)MSwST8Dq``twO$`fWUxFJ$x9Sk)`$Z0xG7ON5+nQoxCdFOZUti`A;l&KtpZFG}*E#>k<3huf_RZ$^bR%Yo+^nlhu`FXpDOwTf5_&a--D{1jM5z7rufY)V7}r~#Q<gCLWD5FhznBydKI~r-cbTaw20k(B#PYFqpRWLwkKZ?xs02w57#b8RuIaBGE}S@GD7WT*rv|_-!825P%Mez_98#trhFl=HcOf>?2igoKORJhOoWpXBtsqv|8_Qx`(t?M|LXv<j7+>N%rd`OrqjfhfxEB&d_Fh$!3`|3l?lEpk?{LT{BskOA2TU%-LWk|S)OD9LqP?~H+jn4l4;F;iVQ?@btyDP#16-{brZRYD@0eWGOjTjBCxms&76e{bnY>Y*c+4`+VnhBWlLR|-Up0nEN}>~eEi<+ZWrm;nZPnZ^51$7yiCMK?l}^3`rH0=mtcEXUSZ8P6p?n=eIxC_bXON$VM5vyOCq^<jHU>LeE)KxsW}|RiS*hT~88`xr_b@(qo~HfDDs%LO5kHlDYhbHVSnMbR(!izwd#YHdZB(EOXpgP9R9SXk^iXSB1T?j&NWRg>)LXF7CSP7CF}>1e*05V?<p=(>?nbO#m7mc9b{SU)K1>UbAGsaz`$@jbwv;m2t>_r2w1d#nhA6A(W<BENZfdQNr|_`suEfHo2qYV^Al2yvlVWrBJ<SjL3ZV84ebq8AZ)+pT$JoJJJB`2U6eMi17OWL+2e2H_f}W6^FwF3$h(Fb<{K@{S)+77cObd;-|0LArMGauzgJpE+YHT;JQtaT;))41P#W;oI`P1E<FkXcOddE=~lh<<#LNxt0b8L^0Qb+#9lU~iW@Dz&Dq()NaqT!?pbqd^$z0mduM>WnzlE%u=>J`ifw-}YLa|I?{LcFNh>xtcXTZ+5T9OrM`+*zj>GKrY3mSQDX-1Ucjsi93jNGsUfiJZkIWGC(|0C%?k)6_WuhWU-EzD~ox$G=|Cnni8xh+}mI1J<07N6NA1->OozN)tPa2j{Uywo%L%jzsDXgiEE<HOmHNnmJap#bGF~UmK!UK!&jny$f_7U{IZ)$mv^IEVU?p!gG@nClNzB{O2XYE2A<MG<Xit>4yBe3~3dpm27ZrDZ;FmOTe0Y%58+1Ebmm%;|>hp7sqY~aC}>bEo@$v!UuJ*jcd!tv2%;*JJW;!1O+)xW4;WS%eEk@<G3lT61!VSHHLG}0KhPNp==t&1>@e4yWI53NeQ#Qwk4u*1L0DvJiQ7Ei`W17mmrXNRUxk>AQ*I{U>B);LQcyTP58t`r;%ohnuQ)w<9>YeN54pX&6EKZRsF;?>bd|N9l#FC<9xX6?_^+DI^k>R-h|G`+n|BIK}dh)NWRG0pz&2|`B;ZO^VwgFqzF4HkGIft<}f|bWX(2wq3cY!>QR}k7@BKt6bEUYW*)n3fCU?UHovZZ*^P`id{8}!4Qr~N*IRlLYYvF1d0_lL#!Z2d(VG+Tb77W0gnLAt0~@R(pyYv&fi(Cgp^kRr<dx^4*CZgeWyGd%?v$VA{>7zq>N|(ESLNGgMtK`ayB+hA3I8Ul6BCXgK!)1_uloFjdMGBL1SS84{XgLu0*yny@oP5VPw29m<Mzr$lzbRZ(|ZnelX)rQq)P8om`U*KRE)bVTGWd*ZA*BY!~;yVvme0d#o=^*{8(flRuRc_4>3Mqy??LEGK!|)(N>3klAdqG1%>V&BkCKtUHWl?Ra<+mvS79&9HkzStb>W8eE`%ZkCs>P`*@1x!9<VJ^1iXsxsa@&*z@J|y_iT1hSyrCbeFG&;!uhS|I3Xg>Rr6aBp@N@!$yyVbLc`oe~Ncm{`C9453}r)Z-W>{%3ogNs_GCaxqnh@#tSw1T<*!Zv<LW*{%<v01))*Db3iwXd9$vXE@<%m{)WCNmxPNus8~~x2J;kV_RqZC_7afe6qiOP@S)bB=_P~f+w|c})Q9ORm-0#GElDH9D_?#NT+f@Yw;kGqDeD4tcDdd(8q04k`$C8?-bfgwGi3IJw<@fvNyb7{QFb?ubM^TaNE}=#f)HyD)Zce3_Li&lgZTnOn~@udzhCdez_Ui)$_hY`-9jU80El(KHjM}oJCog5v+-f-_%nJfF(Ko5=;G;bdc#7-E0S^8*9j%_jnf=$An~Q32}INMZJn$RE`JKPrL7{QdZ)6xAUZWH+?u?2ysmUa!vH94Wup!9&@y@%b`=j#GvAk|`6&@_(AX(R#R+n&E9GL|ZxU|RmxE=c!ZrFdgl@$h<=%w12)Z$11^p(d#Xf1!0%H;TtS{fh?}Z?(6O*gVzPSnS1feNz>~f^%$`7c9)_Z1#kNUchhJVj)8898k$a(w+pL*dDKTtzIRX6JpqM$$5vlG_JdmqU%U_YkHYLjp9G%;Z6W)305X_%Rgf9k;Jstjq#!@2Q3eFa<P^=KjkdJ0#qUCI#?NE+Sl*nFwlj$a6q&qlivkO)P$j^$y*^8qcpenIxgw@W>{kL4V#PGGr08m$iY@X;+Dj9V<{ackl7g5(oi(-^5>z{~Hi!c{x4ipMN4&sQ)gyU(3Ga6lhD&QI8hZoz|b{mB9dqdXT-ycIh!8_pDsZO`_u*dlrlHiRP-3TJ6IWgAMjKSg;9%+e||P|d=sEZufjS98C^+kdZKb?5Eq)@!M9qp5s|Z+b}RkGmAB0|m1{$@PhovHsW<8R+>xh^=9loXZw0v=NH99}QWr=sV`ZORA_bpT$_&vGCj#C!zc?<npb0E>3cdiX^@EkB(qLfj8Nu`PZX90pnw%#9=v)4Wj)-0T5W-*7TcJK-H3=*`=Ez=z839wUIBA?!pr-kHmFMCo_b3+~a(s+HXXoiF8h)p_*Y))-4ArAfP#QaMhc0>EtT8XyE_s&TmQSB5m8yykbVq0W8Ak@-C%rKAz^`jv=iK;YVtFx#EYvDWDi`f;j7`IC;#r%=Sf|9l!bY2WmJ&erTMLND$m`-^~{p(>k=I_%ls9AQF*7J4LS0_yyx(xsiQi0F-+0qwjfomK%)x%=*n0qckm-;8#Bn7tCe1HHjEAW7Xl=ZXGm*etP{u<PsP<W;$(U;<E_PsBT<PHtbAUu0SZPh!<62YIGN9p#F)A2gO9_N4oCprUmxj@Ut}}Od(ej8d>Ii5ZqS*Yo7M1RO{6mS)McIhc~`=^Q>1xk%0$LY=)6UP@6G=#HupappT;+R~x+{4o^qWZ}WO}3YZm!9`K{jQM0CvwStmZIIy5a2;IZI6};vN<cA{?@XlqQ$IkwV+4IUA5%>h~|FJUgX0t6Uw|p!z2>G`5$cLHL(GlWprM-ohP}w#}%S#AeK7WjAX>>vrTi)vdXR@d*22keI!NCAJ)aRCLjON8d8n`aCsz@Ht1c)&RDhFlziaS_itLK69F2(hmd?>WujahZXGO&v)A7Ykqlpo5fZOUlm^2NXg21L+A=Qr_g&b|hi_%i#X_n#F9uVS2>99bK{g}<HoXLB-9-ysMlOMGb&BzUjYHQf{cUB4Q!-7#Ts&QtU^>xa6n3o0jqr)G8gfsgQrp|wLKIHhC-#uO?3|2$R4l17f+^C?3&5CfX`IKBC2sv}>S7z@bXlx?d@C<y!EK=ESfRioP~r_|<>h<}8xOEyU8d4^!^;SN!E4MZDMxS_cBWl~Y~y^XOP8l_af7#JxZe_<GJ?98BeW)z47mOb!^{$y}3a_WB&|9G!zgH<YdMW^*XvdNM`;IbWydzD43nj*>&vIxisbqvQ`9`{hOW__?p-2!e$GHge*CPbGut!a(*RZEKtgfI<QtPa4<BBp{}l7h%81E#UYG3)0_7K?LNzpUX&e?=d@|1DkOSY-E>g<-dec-)M)2Kq~noeW*4@cW|rFPT!p{@AqA8iuh(GiW3e0MIgrh7N%Q;>@t+v?_Z`J*0hZczEPrmd;h#8v~SX_rQ*_U4z2RR5l0RptTxU_-4zWu`^gZKwPnWv@5+!613e?_Y?yd_rjExw5l7Eu?2n+UB+H6T!xlwJU;;NaAK96Mv)xsniH#>kGEso1a<2}NS>F171XEPPe6*geB1vQWF>7nu#cG^$^F`vbNVSqe{_q*<x_g2LJPHuVp0<0O}9o@Sp`oHL1;n;v<SO?Zp{-jmxg)Ckrrop0f$}-L?1f)4c72+5Qs#c*BT+{sD$$@NEeul#NbKH%bq$~KPEM+9{9)p2f2O%N<3H)VEzF<`p>fB)8iLL`H$3Hk>o9{hqJKY6WEf#8<q-&Ic{$yL$pS8>@C56qiZQ>|8fXRN#YvLsXG+Jq0*YGs4pU#6qMoHA%7ZD6_=j__HrYz+SOCM{s9Y41d6`I9o1JTG0iBM<tXJac=#S%2EoZu691JYU@wZiz7{bhLkQo>GnpJ}hy_`*PnOQ8n&8s6j*TGmLQVKgqKJ@JF;72|06_iRsI7-dx%lRva7_)Lz5W0`sIc)`xFN-cXuz6?V{Orig=f+TB5wbhG#aB^G_j2tV!xKqr#ePK(|m_z1L|3EkMHDCqeW<Mll@n3T+u`dK${w5eK1IZ;OOiI>Yhy`v0yp>&4C`UA{f+$rha((lbB3{MwlApw#sz=^Wg|5ebqy?V)u_EU2Ak9R=e*^WlgLCZR*3__7q5(i~PNFn#wYBEF~4EtB*nr5TG;0Cv1gi?{d#vq3#XyN>T;u6h2p^2UHd7S8p<UdCR)O(S`vW^hi_!GQf)}Y~RZm*-kJ3ifK|34#TsgkS#7t=PL*{wt6xzmeZPZ1j^vq5hliIx48~kdSAGEoy?9f`MvejMDiU-L~zlYaJ{<=^{)>182$*$yLW4cVQ0Kj-_K^#l^`Cd;m_i)RvnHzr+2l#o}{<uYE$b`sAH?R{g%i?vUeC7j>v6-N_r*>f*uug(oXK_ReXuICSh6lC@WW$kQ<>5Bnjrb_Cj%EBK814EkZ?=Yh&z_Hj`y6f8DE4HsLnt+>DcOzba2G`{>EiWgdSbU`s7_QvRL@Aj_g4JcAH5Hoz-dN|vQ$cYv6o5qkjuu5^^b2S#h!JjS*0_0xFlD=Zb>e9$OqtxEbys%kR5xhYdrI;kjG<n+>$530%c9%E$`WQJ+8um1Vi<lQp$eD>jNT3uC{Lx=x%leF0ikgtE%z5Z!dsVLa>@tR!;@M8k{V-}76#)XlBRTb=WsclYN&{ihx?Lnl`K<w)f@<<GL3MYdlmaLf##0h?n7m`_07K#1gu!LaVF5Y)+A41+$p_J=+Y95p=l7a8Y2n<5sRQz#MPth>b7?{=D)OwVUQD05W#k+y4Q-9Z+o3K0UKrvN=%Rki)!Z%I1MC7lkvt%J?Qu(<v<CKzMls+eASUA=8;Owe8$TYq(-E1`KaQ4*RTb`F@e6S-9jqVxjOLlaHS@LFmSC$Gh;Y=kK@a>?K7}^Xg2{uF6+c5=XnzK#ZV&SaiWU5@XBu-(VIQ>&qlS(IoI*Fy6%CFa<qtL|gWm|dkvAG;f9j490%z6%cPu`6dNqBih(CK^m-!hKB)OV$3p{bOq%C5T?2pN_OCZbSnbrtan8zdD_M60sc<VE>HDN->boD-U&$x3r6I8~WrC3n=h8SCsz)lwT0V^oV6YBt0h3mz_xmb(N_A@iNGyFEInQ-eZpR_h0{Udpzv(6q-nbAb&7`aoCUyr);Ox^7R21rY~#TjSe9|9LN=v?{C+kngZjk&!TsAg!k2H+W^XNXV=r26O_5oss2&NDsXM-KzO=j#LNt?Hn(h`#eX^0Xw|Cg9b^fb^U~)7*Ke&;%{9-vmC9GFTT-5;={V7c-Nng1l)z$FfUIfEyX$IxJqgnguxfTF4)KmdbZE*iy#-bFW{1XvrX^bVG=bKji`X_rF4yw!b1>4Plw&5WV&8WPJN8sYxqe>AMw;Lm<Ar_^75*Y1krsI6`lhecmn$S2ExRfXbkJ@JquwyBe76}3cHQ^Afq99b3&UG^juW!UG^8&1%GF@lL9_8IV5JT0S4s#Im6t$z@{Sgds$r^7$7a2q%E-x%mftY_m7}e)%f*M-$ReP0*30@oi2lh@`L3_Id9vCK3C1Ctnd5&#;=Xpod1$<Ms$J_QCYeLb+?HiGU<XvT}&W$8X62>Dmzh*omR_k&&!7LGAISALriON<;R<Iy{;PP#m2Wg!vS5W?4o<g^GJ|SspyE45c2IO*2{tlAe6?u%8jPN8y|2_5G=umTG8i%y|*5LNd+H^=`J^lg2ratnT;c@T2nvQ2Nc*QCh`x|timRD`k|}!`q1H5?H0FIMgQRBg>3|mU2x55Jbno|&lubar#I_q$xZrMKU-|zJ#T2}&NPjcH#%ge+vq#}s|IHA@D6mWTT1gqdNogXkgNJcka>&xue8Ya)qNOFrnd=?m!lP7YTX6OM((`dORZHget9*}I@|Dzd3%A-`h|o|VwKyN(@584%^K?Ek(rjVWu^WSZ+w)9#sW~h-|@{u(CZLuapq~vC$G0Klt``1A_$qGIz@NAe<XxiMmir-;4B<@MD|Kheea7P7s9j&ao_N@sh8_|K)6#iSriK%WCBkxt?EZ*37J*(N%YamtH|+Ze7+Yjq(O?SF%0VAj0T+co&cq{zy}+(y(aA!2LYk(KAyBqgtv^BbLNgku&QK@w72aCV;SLSW&})r4D-F($}fD>Y9K(6_%|gt)>BJy_e+XO!mc<19&3-Iex=o51`*!ia!6>>vnQz|ObY=snX7Qp(d+AP_phNqOW5DW@X9;h_$zlEzTNxPLno7$o&%l8^oV<D2K2q5-7h)26JT4pMWlsgxay8Re3L%-u2ud$gvc342-nji7}+u$txXF>Dq3Wa=UZ4{Kt567Ek5(y-VP0x90&b+0M0NSnS6<+o0iu=gg)k{fgRLx>oNrOZ55>hL!^XGly(DIKJsLg9G@ZpD!*8PGsbCW%If|$!ewX^W(&c?5TX>ek1K-u#Ol3%Ek1Tc-TUAB0!)MA+4H7!dEWi%-Z`E4F<ts(O}N&o{NZdnnh2v*O@ZTOw-q${mQ4%nw5IWFr*!N1H!b3Fndhc@)`(2umnZN=cX2)6nkWkk<o%m&Y?fH8jdz@!*sUt*NV9)cJ{VDHtIp}TM;AaZvVG@Om!y*l2b>Y>+8c8d#I^jx(90<sY<RTFZ*i7iu?Po6h-KfSclmPMQ9Gi(uBfGJj|^Q_{M6B4!;o5IPUtxw<~M1w^3R&pC#FaaWbcLC2p?c-`i^4f;~?L0R<C-#cM<SmK&yBcrth9jEZ|7?M?03?NIMS8VmlZ5{NlDgCt)%I56Xzl&V==*vuRv;i$}UgECD;nFScbk-gpl-Cg+3|cR)1yc(wN3rYV48i+j<HP@qPEzj2jQV;qeCbn|pR;|?n{<I$~-Bm2XO+#dq}>y<@f+X<SQXOaIFu3ax(G*;wKB#Pz!@L-*{`V9H#qWmd|DCaig#DT9W3?L_5eZ@MB7(-mLmyhlIoyWsPw#zeg`5j+t1!0vfl_;@+&i24=9lR=~ArG-veSQ5Kp6Mjp#U@Ta^KByhHOQT&P6*^1Mu0~Ae-G!&9cVH<s9s?Jr#1H{exbIsIQa?*W;t<!1PDHj36%XGwWJ|i3CE45+}2!DMN>{%bW73U*wk>F`3aw<{qxYW2brWMlze4JzIXKnlp8sttSwLtui{yc5ipIB8%bK6DmY;VM}>f4NW$}oVA0k0xK6=LmMVnBS#)(CO$IX$jhb5}o3t+fLf;{I7UQ`S#>%NZLdvtr=-^98FfjuXo+X1~Pr41apn7Yt8&or;t(^oz#m~O>B^ZeSz`yv?EMi6N4aytzNh?IlV?*B)pYGBe^(bNg?jY9a(fF^)QCU#k!>X7a6{LnlsCPrV;w_hGRg=mbt^%%M6L<oNUFv5x>SS4}PCdcN1cOgm2I64dO)8+K6d%-+{MVpSdg9=MR*-=5J%@u|oR?j!JY9-ZA$Q+PY1WxCy^0BoF9s}MS<$KWom$6l7CGksWKBvT9kN{Tp5;AoN9|!N(GG)PMKE8L2a|U)x@BCx*Ddv?NL#-WtXDD+cp&UeQc9hH&RVQVYohCVpQq~L@ej<{xXS%6fCD+tT+JgJETL5{!bEz4BX9^frvb(bu-Od-(jN;nOO}fl(?rU_zNhB@w^fBfH)y!bq*@#*Ii+tt4bxPVb;d{}Fnub38Osmen9AdB?lmJJv-Pn3N|i2uOEbG8)JanjNM054E%3!<`<Ntya*5`Fz$FL4^*;CAQ#zV{<=7cStc`=Dc;78LVqvuvU?M5NF{<2RzkA(ebPOYkq56)i(QjNqh3smFe<daPV26jvXuR7^>-o!x<)mwiS9Wj?)POpZIEOg!bL~7Vkh$SfK#&^0W#k;}^ZW94AZ_H|7B+*J_?5a1&a4yDrQmL~n$oasM#M`!8eQ8p%2riwRf^IYG2L){&dn*niVj*AQ~g3URV%L(@=mTyA0pEM-YhpfHuXCWLfk)oUxRE?{Ml$wJo1r#3*(9R?wBH|8L=H4l5YF11(DY(z6buuU#-2Q`MN)ky}ED{k_+|d7r%&#lf{f(L4ew=go7r-{i$jIZ_xoUC?tGPk)q(=RxvQ)^ATx$ghFp%SAL6q0YGAMB(;fqG^?f|3xRRA*FyiCB80#G%+uiK`l!ttp@(Q#b?=$Ol2=0a;pO-)Dcpml!81o)iiJvr4bx21SoZL8K(^*K9Fm||amvmnLfQqVC}zwQCXwiQJ@psrz>Q=Zj1YyZ1Zn>;uA!A)i*$WxDRuABK(9~`lM!D)WdLUBdIJqye`LqGS6#bN=e>6Bf69pA4Wkduh?ldJo0ga*Q~YiDyV`#bv<@{9+J8EhVivuqMk?h|h&FIyLRn&Yzs;)7({;}}J-KX`y$^IRp{P2_11mObEV(QPP8UYr@>*Htj_`sGth(uYU{B8o+nLB8l8$fytwC2baF_DQ_KCq<6$muxu+N(}FP)KK#d`!3>%6L7?HPXs5smcRt<755Ig+MAz08);X>u^ay(<Bc@E#<MI`zNoQyk95u7NPC(g!9I-5%7rr36|0|Je;nAq9=;JtBQ<&>)RX!;2TFB-Y+O-G;4_w0oJ+?S5|h?T&K5uG5--g)Us$n}FZ%J|n-_Yh%d84YbT6J<TE*H_xf#j=N{}VnG2#n70&CB?agcVU~oe##_^y6G<L?P0COqa^0_e)63D>wgmU}zu6b^6+I2&(lOtPr&U2JlL8eV#>e2#AA|V=qFL3AZo><WCdm|f!LWg4?&dylHI9$p<*FP{ZH3!V^>F{bWi_RL_m(A#+|hhkQ<St@6jP0g5mLUSjruF;os{0dfNf5PR`9T0L>m|gK#%}`0KCuchtYXTDM-M@ALMJ^Ba&z1is66<z&cilR!GPNgI}6}(P<|Z@=>?BMH8YRH>dVrGYEu8=LAR38?)UjZtafw68|?sp^(<j3$tu96=&CGa>K{?jcjC0v5pIFwmm79^WoPlq>!7s_cdP~P01k{BRg{oA4Z<30PG_fRSe5&geEG4V%Ws<wW5D`XTUq-uncNz9=-uf{n^kshs_J8NF>hsl!fT*Vj;87wf~}tsT{pBsXIS@Xfc#BoSjHA=eD0mXqzR3`Yl4uTD;i*Sqw#+#-#;Vf$Gz<{}Q+Qt+Hq)EtmXo%HS5<#F91i3~Z{GOe{dZX{GL7{7E#im%<EV@o?g9Ch|&YNkDA55C$k8&}TbI9`lPP8>-=Z$C-~Ylxu|UYrN3Jy1eFq#Qk-<g@SPE>xq)3pXkj;Q9dD&nrN4gp>pj4%b5AgZ=kqMrmSywtF>>GwHht&FOFr2s5AU==m~qXB)|urf=`K;bjNxnqeewmdijElHUsBJN2j6dhK80Fpm{aK6VtZy;XKXw27ua1KTk4fu12gi<3Sz-O1TnS`S~$=PB?UwwglEGj)wV)zo@G|*xrCgvb}6yx-`UmOEO!D`=G;7>y6N!{K;&n!SpTxR<SJQ{7m-9A3c|x;cN>DS8mMd`>+}g8xlk$f5L@n!RM`n(SP2JGatyXW1!tHe#by|`_VDIU-4Z4eXkN9bq1DrerXPi(#FGs!jxFm*)#lBfI>xj<o7f?s1!IqEHJ=BVAUWhtFFupgECw10%cqp4cI1=ct1eWh|Fo`azlcqUSNSTko_2_v2hRHG^AKnW5|c<h61a@S62T(tiy>_b2Ln44;8fC79cZ;w}T_poKX^sk?ZMbP3rnPv#djze%&)iO`{D4*k+9%C)HrMg$xCvM=t8D5N2oKbfrd24MxHVzvt)u>I=a<^P5llRJ5bxdaSYg<28P}y!|poRb)VoM-a>25DFOS1dGc?IL@attDm6)v2I`)lwlJwc&S$_t3nnuY0nLJ-L)E6V7}M}#6?aC8w?t5Y5r}MpgJNHE4WN>9kN_8tk@CMqvi-&6%M4^)dJGzQKTXhzj?Y$LcsbomR=YXpw4;XH%<#L1iJ$u$=bTD5;YY;c}}%5qq&gqHfGU#F-3Rm=GD|%wa7{@R@YH4-pY4PO(Mp<whwxoLZ6Agwxu=F%l9+@2m1vPUK5$Y@>dHG9$I772W1+#N*1#Zd|%|QzJ#krJ;l`d8P&D6s3%-L9b92WD&pZb+h8%Arq-MBD?5Lo5e;@Z)PEE%dNp?6^4q<|BNV^tD-ib2(Ha-}9ly7E=AG7qPw;y|qK0wIJSx-)Ft!p`bNtN*WC``Cw?pnA4z&Nw+?xPCZZThp7z=fQ4>}pPL&U&p@GLyP;#`btdP850V@iR!8_d3&`S{JdhQL0XnkO8lWF5j_=o?DjK@Rge>rd1yxSca78DHMall@tt%OP45wCC^9HtEnycdQ8AcN+)sQG%F?Qkx72t5WH?=2=L>8B(&!UQGtszewqSE>M6o<{rPbMMZ*2r3=Pvd(VoAW>&f9rwNZ?lgE+0{dJIv#l~BvMrq6jW43l)W)vVpGq29o0@k&66Mle-WSi}V7Bw@;w|b7BS5Q1|$7HCQYoss}E}^t**n~-)TygV&{p~Vbm{>uRrkTZ7@e7?3*gUHj>Idh=och)Ny1~|*#Zy?+@IC`|m(HOW`TgT$I<v}c4FL!@tG5bAiSt+PbPolD)iRFZEZO2w;zi5%H-$pj7X0`>8@0fuUlIC(g$BMx`N)RhiHf(<U4;GTnB1ZOCX15m8Nfr#Av=}#vH~jF&dbYLf>ctHOu&F$Ci1cdfFplEUo**C@K()N?^jR2?kntV@Lb9~)0>fsXA7IE+i<|{*6A0jO7I_cg}8KEX^eDAK|?S7+F&zGFnkcrTWLXol<=_7MDa9&QL!$kM&(lhn1!Rb-AA6+H#K}wFYN2quNS^Hoh(H4VYQs(HbVFG9;7s+X6DaE-mVy7-T;=Aj`l=3K^z^C8&Il<bk+5VEcVizF##+Ou3Ni{L)J29`=W(r?f-)fY9ss&ozrzUPzKGqt?*unt;jaRe72lO3Qt!nh)KES<?%&>n6XUzvdLy+0hV_ZhcF!0hH<JclbBh`zhJI4<^TYpgj;?xW;4BX^6ILq%gx&p@GvH+jp;g2i9V$%GU}@f&F>kfw8a#_Ev>)SVn!2|%WI2;=#Wu$yaqvyC82J~4fsdgf+4P$MFLkXTNZw68n&;<6aE=mZ!R(<p&3fuSt-el)E58abERy3KZS%Ry3Ea?<W^*~C7z*YuUEl3f`7npJBlc*Y9x6!MlSXoh56fnWJ3^SSwn~0t6$b)B)m91H%X%(&Y((=C?v3ss3ge$_rM#|(NQvlJrat=fBkR7POLap+`ME!aS%*OA)>k2Ev4kfsB~u6TkwUbz2cA_Jt0z#h7}0^x&Xbi0(h<E=Qzq_J%%tytlfmkTz~!h9&p4VS&OKE0W3eIiFGQEGw+QT0-TgS2-N0i$#v9b9D`UuaOP~D#~c%<nYA33-d-4FocZz0waar0!4u(^99QPCC${r_3n1-86Xea)Njf*(7hfIz$3Nz?3O*kEk;Mv;ZllCM<VBWeYsbWax}13#-tw@I0B|*F4#`$|PwDvcw_p>3`xZBzXgWBT1}RywxT-qumeg2m_^*NbzZ2F%Nb^7!z=^l7+HboBwErv5srV{<D?YpK4ZLZV=Z}9<u&s^q)qUx1;<>F)NtS{6Rxb7RD8n#xyGQzw|DX4OJK0NHbwN!|#5C`0M+&%$N733RJY=Hw2oQbs>H^3Bxp=Y?{%beUfATYPt9pG2OPLQ=dwQZWjzMuywnWAPN+Ae>()RX&^d^$t9|E$eP<tC?dju`SG#w`>5)P$)w)M%+Ib{ICU(aL?Z1nV#qY7t75)&Q8(OP5s7!8ATtv+Ed(6a>x+?Oh~rx4T4h5eSx175Vwlq@)2C36<A%M(@8FhzY7n}5GX3t>Gh*{f=Qoibz6xC?LMit=nznAq=;*n<!P+4}3%g-#n8Sn<s(9@eP2QayvXdG;>dfi`v|<MH)-Qid4>&_m7b<2@ofh!%%bmfHzGG^~Z?srIL8$57G&rc-C}Sic^oPhxX7T!g#C|03l>QiKbUgS&Ue`ui&$Ed%g*U2>>|u&X`~um>DogP&>os-GX{>+joYQt<+7KtPMcCHp6F3FhESLKdGo4ndU+1$nDKiZ1`2Ho@C(iX1nj1!mKY7^!zXsl<bR0?`#8nX%JP1!gZCIV(<l4DFO3rz&)M%EL7#5!lgpR9$b!zYt)8!ljMLDnp)a+tYh)ul0i`HymA5R?hV)xoa_AzBGh@lVU@>gkz%Kh+B}<BqU<U!Gm4xphVPSyi}C{pLR|(G^BUw$m3xxfmlo})2@0u$XKkWjY4V?b10*(cF#b(S2)p($M>xw(%puNeV(81aL%`wvH6EVdgjdtP;|g>)|q(LAVOlBf+Xv!Sz2jQ$aj>?pVHyHADgQ+@Q7O3OY31;!C{Nl(ZfcGF{=!19;335BeP>XY5xyh^e<#y>Ksp4swLsLC*&E`n|Cpu9_s8OXJ5!W@TLVG-)>z150bbBu9=z~k-!RzfEpiDOm}1o9p$FoORVc~V^z%1vgbue(g&)kGGgDHAx3rTTX|`Z?O#5tP`MDh5b3}hxU;mM60io;>*`^6PU^(EKf-E1Zj8d{=(Z(ZcCgxFlgPdq@KC56S?CL3sh3Yu9&TsntP%VIS#g2MmgUtRrSWYok=W63eAHkFYxo{C4Bhr&5P4y2mZTcdsgxZb5S1Wr2D$Uua2G@O${S#HljCz2?q{toNI71(9RH4~oJ>v+eeh+g@fJlq26$Qx#)$7t4OxX_7lt3QspXKW+{o1}PP&=YxX^;iak8kF-3;tFRGC=$g@N*E<)?qpts7p$MOD$o6FltN<Dqk(c={(Ds4OvlMGY(>isw-t92}7h{qjILv;2J8x4iDknBx;9^B2$$X~X>iG4U=nsC5tbIwyv?eHfdi;*Eq{@EH&?Jq}1FHf89Nar+Ba)?5ViphtnpC5BCplpvXf61Y(a6B-J5!d@}%cq^_4*($r?L)oE`ObWyFy;S`!$zZksCM*KZ2I1Bn1qA%i0T<ihM3S<f3C``(Lr;R2_XzcypTUAjrBTOqns^)KG%*ykJeBKLVhE!hLCqGw3)keE91X;Uq)~ZA$>LPH5W&sacM!NOQM-q0<b5MYknx`^vISRkC%|ua8X6lf9l|2Jtw16~&vvm;An1J`6BaU>zDXWLUGJq9?W7?vA_7nw4!cwQ>G`}o%#!OBR)7}7TQqV?cbUB!=45Y?|9k7c!!gxWS2Pq&0e32?<Ab&NS-{U}(@oHBk+Q8f;lpA%kQebXFPb#BY0LRtnj;(~T<cTj?gZcz1_@*ch|c%qO>NNY=+ke-f1CB4m~2?jFF58|m&+T}{?#kE&F3J!<nkXKS%#Vw3QgARb4y&xe;;DnsE{$eksF6(kx<SM`Z>-|>0y+T?OTO7V*+*fc&z_|0`n#p(LXIiIN{}6hpoYkNcDX>JZ~!$0H)<IQulPe@^S{i4~2a>kc)u$m<yNt9IO^v;<4S;H`u1|s6ffio&6pFs0SXMdzVw#&2Hqey5>M_FD2^>*sD)y{PnqI=nT#h)_0ez&O#eBV4*-m+}&e#Fjx#jSKj-_$oEwe{zFHEFMjhHVaeToBzi}K-EJppxqcm%<TG$J#R8{mHF^Kc;(8TJxcBCqKDYD1+|-wW0OtgtyalKb6~v8NJx8NA9;iztmWe-(mFQ-7NU%~D>VyI{MCtR6$HTC4(F@ptEA1Zr0590m$TCW*FFN~Uro5N8MgIe(Wy!S9kQ@Ph7+)Q1t>%@lI&rS!9m7R1g$3k8_!?4{t>^^fv<#%UN5n$9rN5-YJFmbXBhrN9nlM^)u%*a*I2oTEAW{*Nd;=p=Dn;J-ku#ag<6u)4JzSVCZP{=4PqMPbe12UOQ?Qnferq`uMb?6~C0w23A4NRHlv;7gEyLhO(WCHe$UL-dwgG(Qx1D59vB#Q<NPK=MDK++>LPmtU-mr^Lhscbv6##&-!ntW0LB5;}^sQF8X#$8MZ;AE9xr&k$nM?iQ19ZzxgL1z{O%B!IVBpRp&wSsNS`JgA_=7~j2uswZ%uqXrCTn6|b&+*9zych&yw37unSBN4QDMT*6AWFh7#NXNk)v0I3i3Nx+wK?5upv$_%$544;M)l1gRKl-z&kHPD|{oDS>M(9Y6ZgtVhhS%4mx<Jk@vUTQ<>o4dj<aGw~1`6m0Qax%jEk$X(#^{#`P+oY%fZ{A;#~aS3HMH?Aj?O^O?okfnCXl4BHX}Sb$nKhynr~J+KmYfSWCEhH~W{|DtxXaRnU_bU+;v8zqk&J~aupp8ho0YluGX6UNx)r`b&q?MXJ<topkus+;FoQ+X9D%YyNS`k;{OIb`eY4`_5q+H<rYOFjltLN9z^E}2`u1FQtFRXIUkvC5_ebyZ?ayMXYTv;({Opo*8FVOru29Zxm|0tXqzV}}l+`XV@k&_3Kh4ni3j5OA%Z&UWDUrVRKKWj3T=Lon3pwF`9S7bXYuF8tR&^WDACJ81)6L-Hphqp}$fU|K%0S7-`EhtD$;BhA$1O*Ck4do|Yys8r6_YPt&oX_}GVzMH3=E2@aR6sjE#En>j))`Vz9+jIjcsx*KW1LNaDh_XV3F{Les&42<<E$DzO+_yGQADU!=gMpCCl^pBbD7hQSS&`Li#>V}Uo;vnj!dgqYoZ1xlHqWyFeCts+7KCy+8L4qyvVXz|ys7Y3-pimJJZVy4rA<<Nqog`JcS$;dKimu$5u+Ux(Fs{=LGGasNckQJ0pdy1F|nlfne=R=)mi%);EnN!cIm9AR&+?EEm|FuJbtD6JYI|r<(NIy@#0X1C?*0y#bQ<dx7U?8QHQTpoqKDCB8>$P7IJrnTv1;4EI#nv9zYAKJfP53h6ENUuTamZ7Ak&$F8C=U4BzwVlxwGCoYk&(kq5{)1A+LwtCd)}=`<V?tmbzqz$W5_%y<YfefH&&Dgu{!hofDj#YAQuhVLGd8R}jzswX-h)bjQ@7xyikoAqaa5tarMH)?kG;jPU(O&5a$Mv2eXtuMi#JwL!756^|R;Lt-O+v+R?s_mDFhgHR3aZ*gW0D#ov%19{4d5PGcZx9srq5Z9?7*c~#%dDa<_Qp9nFVwv%KaW4}I-S2%EH+F9IL?Q5UPk!`C|ZC%_M{%JP*aP4L_&LWcj7#;`HqZ?P(j*E>x{e>b#a73&~ZI12A!)~iOaZrLT7w;7P*`ZI9Ljh)^jz|#t(uZe?G9Kjyd$j6qFTYZY$FJ_y)1Z$G`iS?E)L(c=JYkP}Lf(sbp6-4gF=Ukk$FbWVK|wA&8pTw*+ORZ*LvUP;Ri*w*Rr<;ApbYDR+&v)s2|(IcZ*cH_1(D18Wbt`n{u!HQ0yF1x%k%GdY^ls9yish(9ZimGhyj@BXo`)leusfi)XZzb~;6ln}TrLKKT_Nwm;Vd^GE_jg%3t)#RKpIEhIB%&>5m?olhx-GZ)?<ly3S@tLcYQJMW9P(=J;cn}%}JU(^*k^k$2)jj0U6HH+L!xYCI%bt)g*G1`AX+!+pyRYDyHplS4GF@}F;l5`6UTzk_Q~42Palbhj>#Idn^&@P(%CBGnO?#EhpCxMMl^QUYPMST1{OzEB_d=w^Nj-qRU5(3a#2Et7zW|66T0ID)@#72{{{WJ|;0KiFNh44NrJfIz;U%#tMw4(OaVknb3?sKD1mURs*Jtb}UUQVehI`$(8BG~!c_7-atq^P<FxX3rC{{0fLhUBv<`qFV#Q!nAM<|6LQQcN=;!o{hze6vw7y%>iogL`pC(lSNozt&clh4Te8F=2iZNep@X19PeqPwJAImr+)z6O^WT&WD|@iiD7*;*U$1(uP*s49Ltrcki^EHj<*qiKskEFl7P*c|Xj5!JBt)#uc48zNZ(u(;_75Xf*L<w>Dc9RbA+8{hD<T|$zfXa=R%BJ&M!!xD!1!tn=pR@hM?`{5glGX*&vg#wqIH)rBfISBD(jTX$=7#|Em1kx`pD<eQx(t!!0j~R($N#ZmsG2aFPaZ`lxBPkw6TKER_Cm*`48t_&)b9_WmJLzqRt#UHNL3gEYr*H|XGD_}+*1>utFpmulh9&QFHpPvAO0=L`4|EJ>0q(ARGCYCdfMG0YK!p-59jKis3<>}}k|yJCPIr~v#IXIf%9J1WzppiLUD#YS6P4d#r7YNWCwP%OfQw$|=rKZuUkQ_AVvXN_^Bmk)Ya}(1ZVf)5P56m-QbEo31jc>3rSzTx9nr5U&lz3Aa2Su}Izj&1vpJAMV1hqMDUb?m|Lj!V<OV*c#>*tl5c(`$g}I8f0!m9De#@|Lbmu4?Cgs*zvC~<<Sh?a#@|k>pX~Q9@P7QY;Swd2H-v#3G-YxM+o$E@jv4$W9FoQ>5X+F{D0K#{qMx^*SD&8H)J6KC#Xfol*w0R@N4~N@GtD>YRebbP6y|;!Wbc7;-TV<m(oYFZ^K@s%ls0^xnaL+z$j*`wo;f8LUDuYPqrp$G68=^-PT=$nMckhAevjOzh*)MOx6{cRJbH#fflx(d{v+lF=SCpD?AsGF9@tbJj5Gr+gcD)&4ZZOtH@sPnSRKnS*o>yWF{&gkq-X08B6XWG$>N}NSktkF?X$IhD1Sufc;<YBZ9Yk!4jvru96oS^U&VTJ)_<JF^F!h?=hA#f=k)DKs1eiz5X+n8*W6~UVr4Gt%g6t0It>=jU8{g}&TV+~*Xn_^4<refLazbH(uLsdE+xBvt^~CnVq}C_K2w^9OxZA4^^?aBUT~d2?HiA=<{&O9VV+e8Zpq#R0;uVI92pZuPu8JZ)?TpqglvpYmFx>yx?oD(QF{#5@WZ#kY=TZEBW=vk>r16G{scJy?G<sH8b*<eU-N~iW{bQFU14qD1L}^<7lV%Oszp@)V;DADW(2)>M={19xnoS=<1apdBh(J@t!4G1ZOfqz|Yx9pO()y`nKp-ZKnKX0@1a|Ous|5~B4l66t#^HnQ$=c&LoR3SNw`3i=kNcwyqZ>GZh0-M&By5_||9^zm0NMmyTGtbyX(mGwncVz_QP@P=!@M{m$ic|(r6}?43!;uhom2-tfWs+#?op58Cg`sHt#3!pwSyy>zxcEpp_<??T#rXvw2v1?onToUbHO#C%Df?IvoJyh3n5v_9=`0cJ$Q%;uLRq@?B9J{r*m8}inzf?ipISl!w#970`^s6$FtmLNsS@@V@z?h{$PQsMnh1}&APer0<@LOB9leP<<^uxJav0vA)Bzfd4lw>NUNKzKmd<#{~gX?{Y(GFrreD2xyT5Pa(Bbz>BUWTBH8E-poXsGWM4_!$D{7+{h_q#Q$5`wN_jx}y7O5I2oke+p0xhMpR@BNa^@-Bc2Dn2@7#<b(_eVU@ZA+_>fk~ut@3S+7u_{RTA3Jc{1raCXzA4+5G62PnLm8vAfCZy@u<BrIBmeb<>{ur8%elJBDhz04w=i{UDJNSEuoTK5!p~vlI(wmMeGjQK4w$IF{>_LJmPwAwak*oQb#>!Agtq-#*&ui=*xU(<x6Y}LA;#$VaeL9;N%|!JH<FRskung7q_89&Qbx^MQm8MG?_;v&*&fWubX6L3>I(}Pd7d=w|Ws87<CC5WF=9=;vO~3V29V_J^RFJ06EI;k}!0VDgx!xJ`U_cY&=`|F!q3zEs`8BV*p#O-U{G*3CK+at;eJEZWxxiZR}BdULLlj`Spz}YW%Mtw>-Mf-VpxIQWSx*%h)Liy`)e3<zXeu<)w<^YIHZ`=L2$??;Dsl8OssW!4~GOs+d%8c2^buo^rkKR($T?@L@yDz=O>kN)8J8XRQ+g$+KMU->NfrXP(bGAOEs^6KPHugeL|bUizdyo<TNBT%#Y`F^K2TB%+3(|3T)AGil4Ov&b1Z&F1QoZD(~Li+$3S`_q++AbPl;l1~<YHsH??%e6q{y-gSMLZST*$`MNj$AW@a@K}vtiabnffU-nR{k!3DN-^{Q=%Je^ihmmgo|EVnTY+LDrC$0sfl~A_)1mTm69WpqD=0;g_4*MoA2FH)lpJC-nBs0%nTq<<rxKFHCSQA8n{*GyMz;-g@Jy5ZP5n>Acn5N`SpCu97CW;0%5gforPp=SYhWiU#bz;dd6sbz-A#+;s3bjPGS!ePFdKvOJ?v}BB86O4(BwCH;$4Zag%iUz-v$)dOy4Dn8w}1a9lSEnHd-}{-Am4kd!hOzX1z&}%VB0H-#$&4!>@pWs#y<^R2vHmq$GGp<5};d+}+2`MJ|b~@qM8V&H~7DY`lXr?c}mW`Vqvoz&AWlqo@=|0m1qYx<sO**qggQ&}iNXf8m*LsO)vX(9g*jL1jSPxzXLYl#6&E71D#C7I_sVts<n$VxPj}EX?qp(Jgzi6Hcr=+;phsDP>0%1flBH_Lp&TaY|tE6^53G#%A2Vf<5^PmB4F>;4w>@(8Gy`X4sL%RN1O7!Hzv?V(^AG0e3_X!EJw@7Gtclc2W&cnQuxHBqq7bQ)kHu?%wY;;3_ejmYlE951Nob?VnLo^kj$p$@l@1U6|qMAlMnoAZ8rb*Us?Mgn$xCMvUhC#azl$6z(aR3(HXZABBF_EK$78)VnI3t;fa#Uo@%_gM{EXxxr(-SFj`UkvT&AH?7}1%(OL~4!dya&}Hf&Or~?&R!5X0X~Tp_#GZ7NyiigALM`#*58$cbv{V_NyylG!c~rVOtAhb#BDA?5h0~)|>OfX*hU=3bf(Ev@&{eMY%2vm^ZPD2>X&M|2=pIW7_zizr<&#JbOcX~SDnP<Mf{XA)2&KFryAhSO_-unNYgJ)G{RYILLHEf>uMzmb1JN8fqXc5nmnE8^b(~*g@^FmEM2mSr6hagXE&pM|i!3)_tn_jSXNA&P3sj6P&&R1cp52rPig9;vv)ZeTU03VtVv7yD>KU*^;#x2OfKmtHa7s_U$l~E&=%yyE(591W&+uj+U7Xd8RZQN7II+8Izi8x3K)B0~mdU05kEOJ4!Vm)x$T%xo`mQi}RC3~s`(;yHGc7X;0{{C3Ndp!q;z?sN^Fb&}@?N;7VWEbYsyyHMW<Vgl{Tg7Fv{&dC#K*B^!APYE7Wl{nIHp>$%TGe@of}YF>_rA<XASbmENb(IS1OHQrYN0@wp3s}T&lB}m}*}FYt%)2u`9OPpbJV995ThVgp&<(?P3$j=<Y{jE3D<Ex;}wC{!u!^!?9c)+$K;CCvl9&nY&i3j+j`sFNQu}-pg*B(804Nk(tuL9>X5cf%1|oU*{6{U0G^sABZ)cNn62JYWYy&^&mzG6E%mRA6s3O!dQES-=4UIdWq0UX{v<3sJIyjZ65<=%x_lqCFXeY|3i@0y(Q7?LV#^BbWa?%c#BfAu%0viN90LtT}Ae7hhPL|Emyc>qiDh?Ub|f2xyI(4cjtCTOAW$CQVh&3xd#HHOi{`=nEU}+*_B_KPLh^F7kH@WTIQIl{?RC9{SP6If03B&<aUgFQG-_O+XT+*<3Mi5|13cH1m~Ato5bF%@|A*F%m62Eg-L*8$CYSr%ACC=Q{|n*Ap+;{j1JqZ!(9E3<`ntj{ct){F6~L8NZ|)OBq%mrt$b+OGfB)HI-Dd(Mg%)h43edL`2M`)Zb)Y<y`5$myxV5X1rV+|%is`S-Cme8>e15utu!o%K<aVW7N#;tC03f73()oa##i)%ajN2H1tevL;d|A(cg)lCY>+qfX~HSf@nTjF#kfL538e`E2ktWj&^-V4XlvT}@BJ^EZQyUKN#1BnjR1CoivQ{grWbYV(VKn*IcajM!zEp)ANSBj;0jV7zAMQ_kJM)OJmlQyzWDFZL0Ry&s0q}N<eG0E7c)U1aP?38mEQAe9Oy5KA4*HiNxSc3I|_Ub7)`%)QH0o6L~b;3=k_6>ieiL^8+=0o^MP}>rYp6(=nDMx&LV|~&kJzjdhx6Y8p27G^x+J@Y2T?x3AuwHb?y7i)Z^X6$PKL0ze4@?jH#rAsejb1EAml%+nheGBL&`*q|-w*+AnZ8{ggrcgrdn-KwSIZzpUK3rrM`Yj=aoIQwK%sE#TP%=+}iJg$aVQt0qMPQ8uV)7$_b$ezgLXxX}@0EXcG}K<h(`dtguYLTt;F|1~?@(4b*v5q@~60yiL+xvRG8OLWj2t_&L3SNarXUz((6i)}0mZb)O2*Wt(ArqvLhtXBjO{2*E<HlXdsSMvM^&_2<L5@p2&Y3RbebKoBXS7-h0v@f6<6IBv9Uj$PuU@*`2eJqQV^$#B7r+*&fmhV~8&^D)`opH*ehiw{9;Spa?f>yj%U(SEhs%6L<z7sS1C(qm7*6-~dl0la|E>55v6SB_b$nc?X64Dlmh#b1sbrUXq<6RcYbXU_T^F>Ukx7cY&+>T>AOU`yNV#&9gbv5cB6}5YBTUm32{xg;3vCu~#rU)z=^@oCqR@?bMWm$~sN4=XG>XEezUzl7zAv`*XACdlPxW!pB%pKSEM<!+h;9Z>9>iPs#*4Z>n^X~j!HA5mBbE&$i@bB=hv#fTFFVb8UHn2{uDLu-7d`^MTX+=!2XTCYl9taphzzDJNd6~Vm3AzTV=$qKH)2S;V+c9v~Fv5@r!#ieRk1p!Q@8FA|+~6dX9W|3k5t`yzc&0^>OFVBecj-X}BsZ<30)A_$h*O}%q%2>rw4xtPl;-`hw~GfB4ZP4bK9`wWlQyR!O&TI>3P?gF-l(V9hFaAL!p$C#Lb&EmjU6KLSDO)d{tGRbM9&Dvwl0kZk;ZKu6dbYE^o=FG*?ociBTrP21yB&Q@wEm_)Tf?3pLTa*oUUAe)I@X0R>_V&i5VGE1a~Rrrow{MuVq8N83ux{2ALI{&Y?4r*soigkT;hPuB@^FLdIvgenH%$^^XOSc{Ol1KO#OYkVhfBzqistd4iAb@Od|cTTdBtKT(LM1_k&Igab~00t<!Qu*CPdV}GE9?s}vm$}MD2Y5@cs$u_o#GJ7_B-o*rt^caf}+5MGhjA@*ncbOqG;-`fCG^B75@uShV(lxRN83b)A9!!|Zn5sZ>D?zKmR9)_)T<0<pNA~r~q4u#kx~D=_AUNi~k6aDcAz|3tlPT%ALOUM!`!(c@69dB}AE^~`M>^6WMQ6O*7w+^|Q_;9456Vf6xD^K!enky1;(AOuNFFx(5b7RL<k*DcFC%1nkfW-<D=qN@Kx9vpf~=yHlAE;59hj&r<a#4C7r!R7%?#&Cj9GhXiKcVhepqM_G;8+d4>+L$JL~q*Y_H58SzEuX0U0sNY=={w`5U)N9)kc)7$s&oBuB9mCNQRSy*W(Si6h6-I)wN6yR!FPPfkQeobd{{>WnLsNmTNJ{}$8Sj&xvp*lZmhq?>=)V13$^J3;DV1<)fMpWh@SEvSr0UumG(u<GjmdhB#Mo|M6>Sa{J*it<VjYu>tWlFqW&cwLcLhE&#Fg~L^zOdw0DwEUFJCZISwE$Slq@V^Yw+>B>ko!w!j`6^si_!U_shwl3_hi>&ZIQd<7BvC65+WDBDm;f$vb$7AB6IER8UPgbhX$$L1XP#IM@<8`;JDW-oHxA9JJihN7<jdgX8;5-o9q}#fUZHNO=xy&sNcBmL9?XSH7e7$Z5?MK7?s|hNy)ZyEB0QXy6*S1#`7Qqtjnj%jHT6USDiq>l&cjBNwDL;7FR>mC#-$VQ%&gw#RraF@>Z_cW7(Q`l2IU#2dRG|k423$86jO7rXU2$4*yuw}9(Kaf+e+-j^I3;8@vjwOp}5?kP;35YgHf*G!+yA_++2p6S7}B(i3tU{9YFOV-v(?6VSj5~k;O}9bd>5;;n2^s!D-2$s7<X3>vf~q!>vJT#N{ijUBV+ab+KtNxQZ#Ty-{#Bj%0S3P>R-)kokape(4eI)_^W5N_X3Tc>_N^g}r<imEZXvq*%R)l}2*$VYg0{zz|slFou^4r42Sy*4Vu0ip@HcAybB(zY%7(472?ijWZkqqjkSz!nNz;DgxOmk7TA(vEyMlKS%<*)NssRD5u*qX0b%Vr{JSGeBm@K1j@b^a0ExKZ$BKhAU~^tAWcSqmz1CE1**`Fs28*_ygaAv{Z+`{fVTuh0^^BBvjALi1%f24H2Fqw(a^+?EwHO*Zc6EN{lyh^@1r}xh5^&A`>u*A^a|j98NLlkd#o)?HQl;1{ANLJ$!q0<-2j;5E(DaZKYL0+>tgV2*~diHxCRx-LqvNXna)3@flwz%l)!c*|BF{$q?m}QH}Vm1(lm+sMgol_fx7$L9AhF66TC=%S`d>BJ-@auXTvuX1o;;RJnLZA+Ac}gZ#U$gPWl}DuaInS=FFOP8o2!ADJqWN5Go^fP&=47QA`Hf*3X~UthydcEsr!iVH{I-k)G=HXQjh91!!cl+AV1$s7XNAy?$@2ED8UwqtvpVy{_lo2tdx<&PW#(!CiQ#YN@$o1*HonC%jtiR#yCX>vHwxVTI6$7pe>8T3AB&h3BLiW92eK6$u|)Xa_jcd`y`bEsm5z(Qj+E@e`-j_BIG~M}U%ez$}C`4DXtcn0<{<W9hfkW{^c+_&GH*B53qDojR`a%^~0F7er~*=K|Nza^EYgU>gYOExWY0V~yY!K3X5q+otwr&zK+C0Y$4PfMZGT3b4BkJ*Me^<c+&7hlM+k%-2W@uHN226azlW3jOlCk#?*cqDmLYZ<v3<3u;1mWv(ep=qx8^cXHz@tp>XEN&(nDI<o)UdnA37l}}bR=+(T?)A7YCi%k!up_LzuIUE@h1cVY*u07+mLfKR=1v|W_REN-yo5y~ORj!*-ak*d;Cu{D|>Anj!`;zdauN-HkD!=*?aRKA1300f?)`pd2XS+a?bE!&2kSeU;S|A!%%)>EM!i}^}t7>?>jmSrl7)J1XdC48OPRAc=5Imj-V)zNd+}I{^LFL3(#B5BAyR2%=G$1e{ecniRp=4b`rZW)aozIIfHvJsvWA|i(Q1lLBsx!!YV0#6S;(9cDY=tR&eTX0rH|j=|I=5J#Q4UDJjv^;~5>K6wi<t<3RmXLk(SM{QY)I~Zr5HD_FNxy8^*RwRrKgJJ@Xqt7D2Jd#XQ56%JAb@gUijeN-3b6~Y(;zM=U)MSvht5=AkrOJC8Qdjp@U??^br>Wg-T9(*3_qvQ_9;;3D#5GTgUAx=d7+p(wbbqD?$~J37Z@^Ar$MxejmZ=#<3#bc+bAHkPn>87h*v~h?Al8{qQ`ABOWZNF><5vn;Y5S!yit)SJ(C@p=Aiwgs-i<E>&xoXnVwK1;wher62bl*@Z%IrAH75hsuOC2Zlt-#nSy*mBTS58Tk8tIkahl?i}gViIk2$vAy2U6Rb-^4?!WUIU9l>r;I7lJ~8VDq*9_%9rDSO&$!_WZBwJS2}tV)xk8|tT{x~6D@uApoo|6ie9xJ>WCz9P3XnOf3IjN7UmEO6*u!S?g{<E~cZ)|%|FQ-GWi;xkvMBKtKM<&ckF*l<eV)Jwt<GQSr}B!qC5wl$N@t9c?R1#Yf&3m-py%1klY!KiWY3Ad-GDvJ?;uPnz?xsTel_D$*%zQ9*B1BnKra!dptz(GE{^=Sj4xs=eg9oylVE|=)2^IvZLQ2tWqXPH6|%*1uHn$VbNXu}Hy79YLOGE{!2AiH^L|$8<%w&4(HtxM&Hk?z7a$Hrh1*DnD9%b5@mL)fV-uhBRN>ixfx@s=cm1y;Vzbk-!{8JF%W;om0eJwW6k=Xlw|#6UiOa%Xh@IKwEaMFOInQ$_o+4HTKr_3YF}s3FirbtO#4V!RpHO<FxK8K}95^~X@6T>r2*SfxdHRf4A$phA3ulO{CSz0lGsxdJ3_9QQ4^}pcG)^E7v8!?Tx^4YWz!ExeEEfCI!k%l#;h-fPZEobvpe$F2?ffj3REwas=p;lgL&swp+pja_pzv8rlXpw5##eLQhSN;-T$0iHuLk211xF;}#yWUdpIP%FKAAh$PA_UE4~uXrZ;J;Qq4trkhIZIi&eSM*03C}td}-dPRO9jH%0gz;C@X6{PjXUc)~)BY`ppsrMv=hqU0A>8mceJ;-KwmZuQr<dHRv=sRJ>^Y{la3FhFoWoiYXK(MM?BJqwcv;R~7!#Ee~x;uVV?-*Fr6g0x2=8v97I{|BU_~pdOGtJ;V=A2)?G{v+)&gkpXZHM~h<ZS2%d@IOKLI2#6O)Xknqp7vGv1(_<WEYPzMjt;rzTTpQ4LD1tVHFWd?!7SizV&&uV?I#rG1_S1YX!y}`y+IP8IhZ??uI?E-j`oRiV<F}U2;#x)_;@OGn73(S8ZQQK3)Sw|)?@pU2q;u)75T`4smqh2OA6CTClPNzFp&%jip`U~}BcXFTPx-z2ci!~*yMPqnI@3fobD)6k97N4$>vPT7`NCVZrjra+#wu69R~61r5Q;M9qbjfT5=Wk=c(e`-;l$P~S+;kNmUO7|ygu{5?wAOiz={eKi7Gf0yy;Z_#Yi`S9{G+~l#_vk^N^Y7%f*ne<j9=oox@uZd=1iZSdm9iSFFc?_w=qeA&4=IZ=kw@aUu5EO-SDc%yp!)#HK*Qm*55RyAF47bB+Vo=PY5Z&g7*C4*Qu00*v=fM^x`0UWg!S_SJDNc#xQ5BmC@F)I77cw=X+YLa6r=|E;5q`ye*Pm3n~G=6s2eRf>*VY&##wL^9Vlk~OYq(PcD;e!eRe$ihbU2d|`od1GuP+lx@KZ+m6lz@#`gE@=Fp`ANGU>s$zoq~H^p_2aYBHTomIHTo<c@pgIyYsHt?_92kqiOa#?Ao}2Gp&4iRy%$@F^E}1r6f0|tHJYR#)x+`m)%7`B*=R2+G<<!p5uuPu4ty9jN^KHHcHE=%{}tjX15$9Zh=UI!M~`+x8N}r#!-#=#2`IWUXu4zeEU=;ih3KX$0>cWF{vD}>ngXb`9C1*NH$)_Lv{IK`JWmGw5(nmme|b%5<g~j#L)f8jyDzFCA<u|$=FeIW(CWgq;2P_QRzwXjt{yJW#v_(X+0#CF!M%({F&{<sci&ncl85SsILZbO9PKBy1Nr4Ht-I^n(&>#NsmQJt>5~Q>T)tQoMg`x?@hP{3Sm2m^A#k(0XAsaaY$|}~Zf*7#=0)vnrJjCKIX~3WUlTK4UE^hBPVcUL<H)M*h)If0O<Ev1rxoV64o_oesi;r;^Yz#uZ|)Vb0Bj{?saW<5<keMM1)ycpS|tz;4XxIwT5xXE4as>{9$sXVcJ1%#f|-4Pg`v+C+8oI^g5CM$;WvR~M0?{}MOGd6W7T4`x9BKttHQV%HruXpNE1K=xhAgSvxTLinA~?gEYgD}`bCHFZpc5wDK?lX>M@xmin2RzU!S8m!p#9-8VYz!sgNg!?9;ZlXLGaODl9AH&o6{k<37z)pU?F+f`%CO>E-fuV9vCO|KPy;k}ZU=!NOc4iC~2|-ootzd_()W?TMI%CwKDlPpP_vF~Vpv{s!qRJeS|)&zXeo&Jxu&^7knk*Dm@Yvwy|9ZGNVe*a8ro*m9i`aqmBJ!o~0=I=R@$tqxom)hev02@>Dsl=W5M_7E>tTaX3TNl9X8rEqSxapaPBnQ>*rL6(LAi+mZNCTg<}z!q&nDn3T}W*O5dod?sFFS#%?yUTVAetu|;`>G1VDl!bnJR+hp@%gD_I|pi&>R2fx+)1%Kc7#m^fhbC*&JIBHK&Lah6ABf8F$^Fd6<m)81HN3?(%~lOA!z%Lq`p%-qlk*~+JS?1$@z^hH>0qb>JV0Aa}eMm6gdx;47|3nMd%B4FvR2)^aLXqX{Fg!rEhhP%YkXN+HOI0hL`SckZT;~Qnlu$-YleJih)M!l!!Aq(z&LrKF1jUOg0Y6Xd;paPVy=*4VDtQwv5ef>(WQyfTR->$|kiJQxy86YTRk-CDC=n<s%cQz>o~+UJ{3H$ay%1PpX((YzHtJvy9`kFW|(V_FWX71duAkjlJ(Mh*O>x6Uts1foQqL`nJZVdt2B>H1ei57Gx2bl~n@5OX{c<iUcs;>L&}d&PVa&W!>}h%Z^8-c}vg=xr`2TTDu}+DNT@Z<;5T$ghR!<D3-d`B)<ByDUX)17+1WaTSsTO%-js{WfTG3QlC@Z?Gv;#6o-M3Est#c5U$U=g74a*GwoTIvHf8v?HyaS$$W#{*19;StpZ3`KkHrkzw9yxzE{Nled5DM{B}2`uxXyjU-+;4DJ_qvOAz?6dbm0UMIXCG8lFERD`D+*re{Pyb|cR{<+lzl;3E8w#G2V37XwzhxrI6(jw^qz&DE;xQ34e|A8&iFS2UB>+4T^}S?*oQ<-r_OS<v%t#w*fEb(5xiakyO!xO`~)6y^oPijPV9!BtlXOChNqV=@lJG)}9U;;gcpYmK4w7`g!uwhOTiqHUk0&F;BB@PX8(?n5Pq&Qi+@y#D07CZ{BshmfXHXl>|S9K?m;e1%`z8gPsMFMC`q%O;%Bt?j}39V=KC(cffcE(tK!pMhZ=bHW)v^WAi`O#3MX&vBgyL&_Mw52rfm!b`NuR~P#?vKn;1yT}KFgSQFD+PkD9HN)A5q`Lm}_Qs}$k!0Mb#wL)Q1qeRmk$=1LmW5PbJvnq6)C`cwHp`%S6pkfP;V1_qu80&nZ`&dy#d_^uFMqgKi2691?y-*Om>s%qVPJEL(u}*DpZ{(j2S0nohU0mmRT-$tQtq_05MffYjsV3E$w_b2>=?q$s$Ruf;Z<J~FO^%HhjZIADMkyw9YF%}jZ)okWi%7{&I-Ny)gpXD6Rql6nLjS6;M{3CGia%?*D+Ba62Hw480z2hcW#see55hShsDGBQK_pyQBi4Ui6V9y7|v+oC#CZAYU<u_W6t|)&THK+>QW2a^CH|s!-~{gXPYtH(H$_^=>HtqmiJ;0GM%+ucF5PCPt=awc15EWGYUH7%v#Qum`I)`Hf##u!Mk{`rS5v;KClBt(|bx7=E0>i=KR#`tt!ovvD%_^ZFCHXoWApBS^~{C-BRCJ++s%&O@4BR;eno%IN+H0l92shmWE)A91VxC;4{PQc@T6_Yxo@fORx-IQu1d~%umOSB3}3h>_Ldx>i%p!gJIqp%n8DD5dmPIm*Gb~n{Cp~83!Yjtc-E9%#2vYdl3$=#e>I9A&#SDL9t7Q+yM|&PWF}H3W`uyuM7^NSw4RKrDP}!-LQ4367UZB+l_yw)B(P=ss^@>qKVMVoC-GdmS6k<_0I_CbQ2y3FXK*1pO`UMrx*nJ3<N*`<p5$Rat7hxr%YQrD2uF7P=skV@H(cXI~l(-E`F_4i|ZcPlFtQGg?-~1a7@^#u43M~rJG!>pD&(Md*}p1r&JM2qL9|&DK27il(n%y_h$c*e+$PFYYV^jqSxSgrZ|$s#9IZ-2Vlwjk2;}VhVxWQgfaQ9jiT4<9P{okl%A1@q;Y_L*OBqTM`+jLF^mFM9W*6Yf?~Dd`I$yGFr*Ubp@ES6x#x5w#e`y9^`m|Vskn>0sKN9Iz@nj~-eZOqk1~uKC}Dg<L(s>r;=0eRLl?$K&tr^+7Bsky%bry<%V<*~c8)B8x|b(71*<(%Iw`v=zg@sdRzmV$2Z)fLvIOSsMkXnLu+aO7Ug_r%LbCU&xo37S23B$SAEcfvE;8oBp}ZRI$Ek9V338FMle`q#6#k`HbdD*WWB}!6XWhE=0C-%}QW?_Bi-Z|*>wx7NeSX3@M(bGGo(-Kag~$Yl#Nelt3zc`WWiM{(7X1G8ACN2fr-G?`Tg2Owda``ZSefww@G7}ab=1b+{$>c<!sysOO*Grf>_u5o2|>VB<?XWzZxl$TlRG<2m%vH~5aP@a00X#S?vlhXAq^=f-i8)LB>G)pwOi>(5c}=?R;iB*Fe^Lt2d-Fv<eBR=UDPW70q#UU9l#d@HhkaqIrxk);Bv<UZZJ1G{NU94&{S4;VWa=U1w<@>JJx68k5=G?G$`w2e+>A<Jq!k>o#d^5<??@gDn8H-zbY3gK4XWA<m^yr+QnuB<O5_<v{KC%#dV8k#=t0i`f!;x;S67@kGRpF^)JnBXX(Z{Ri5B=rmOo0Y5OFx!bi1xbFG3da<6NJKjPD+;@f75B^~rdBmDAo1Q?FIxNCZDg@1!=jo7d8U~s}(AvN4|{8FxW;7qs97x`N+Cm}odrCLrFz|^`Mk5m>JqM@|1h9<f-oIuC`XgdNW%jPLnJ|AFiRkEhP7d+z{KJo+4K*YFJ6{F#NBN}2HlgjM=Nk%o|mLd2*pQ4}AV(1=HADT8c!g4F68qTl4OdfrP@1QDTvCo^%HA!<s5)n2VG4tA(!j+)#-|B)Z9PahQc;92wEz8*JKK?2XXxF`~?X2}=JOWQp1$_02)z{<wSI?o<&KB;(ya2Dt2bMs$tHG;Ui~r18`xF&Uwk|fn7OyO4;S1&g$`vix<LS&!@r!5cNCNP}eFEnfYgDX<hTPsyiirqv5pO>q23ypkIr=0dr!A8@pIHC+I(_M;XIBl(%`m$b&2I5I-^5CZV9#-CvP4_`HDy}OFbXsQK?bQvw3VonhF6|qN1KtI)2}m;(6nzQub+!b1T?(0snUYBDUcHei~=rqIr#I}t?=Hpn|i2EHrs<rul$N#MHAQYoPyGhg$X3<kLP^~1%B=^fs^@)+<F{d+yBJHq`Srq%zG&I{m0NljeVOlR)z!)j9(i6=zlJ?dpLl4yLUaJ0o@x!&YghODlaeI7OIT?83Mm2Hq~I3W?GMyJ(vrkPG}+wwG{DZ<P{RyLyKRC0MTF4)ANK~(r2ivj;m#?KXUPCpR&KJ6BIc+Rk1R8wn}HTA#hRIhW5zL-b-{)<q(SXvo@D$h$ucW$a@FUQ4H;4vxGh(w$q1ku;;!D{Y&eT>g7C&o&~Bul%>0B0SL?43Z{4qDZr+tpa^(g#phyV#drRx4$da@rdMtN_?-e1Il!V63Yva{t9jRtX7q@KM4G5kX=nhGwH1UA#RZ&Z!Z=PIg1hKc)7zw9EJI967zieCa>f)%bkF$~KS98yfA%aDwpXM>;3n;HB{Hwinl`x*gUfKOL8J=fP>C#5xl6z!48t+BVuq3B=6qYn07ydoT+rzQ9+h!{!6k3JWQM7TdVuS}OBO;>xkPc0o%}zjPK4_`^`Nsdlfz9~v6vZ^*-h8~>V(+^)~gvg_MmKAj>sc|E?i8cWCi5ibxmNi=O^cu=+4ZJc`4eE|Gwa(p0+j)jn)20{RL*@9m}O8EZ;kZUq1d$g81q8is!3Qy8Cj`tWM&&tEJ?90mPikOas`^w?p8eStYZyo_^RqMG2>VdoH+^?i)xvy&MYL-qpz*ZUDF{CV8KGYBsE=uyS->%6_RTPP~sH^;Q6oVpQ1V*&F;l_!TEKsP?$V)TY)~S47vIkJvoS&l$%Sk0jF`-XF&sbxS5ke(_7Sdq@1i8wbB$x5o*ZtO^}pN>~>gt5>vki5sfQ9e%^508f05W)DXjnwG}R3En#f+Sx(ljN&?14hMqE*9$%om$QNMw6UtDCDZKVRlhVyxg{r}8*S6t#ePh*MUs<ENt{P`&q9i)ODj(~`<o@;WLhUJ%r0nR^Dx!>l`>E1ymq%zKzc?~ZCo|ZmI0#daox`F6iN2)4*c`{)Eo<p2H*ONH(@zwU%JB5{^kCCV@I&|44}ltKC}vrTtyG>l`8r-#nqTX&teuuzDK?24nFb8F|+uI*HmluVWQAhkoVbOQY!8Efl<W<7C1|RmmjQdcO7u6p<f?sxxo0{Myz2KD0cPv)Zz*)RJ%g1@EA9YTj#n99Ui*s0g-xAKmq=J@w^btS>pTOudczfH%Z<^#}~v21+}ZktTCHkd3Aa>dAkf#nlTi7Ag)vh8y6&R=#4F=jvBvKfAsV1^r)?d_C#xYP`pG^JU~&QYoyFjwbAd%Wu>lq;hjsopJ_LVDWUOC$eQarf%pAuU${D}tzgioJmxw%29xyA1l;1`(W(UA{(^2=-M1|Z6_4uw(Am`!?1W*f2DOx+fLLWGeg)hD)e<jnGuDJQ;8W>-4{t(wf$MpcL`Oin7aq3$>M|aE@1znVUsCNgGh@*Tm*|efVg4OK0MSUH<WsCA1Bs~|1ilok;*1jP?UIe+V~2)dhvd>J(L<YeicGfTi^u+;V#cC3VSxqBHwQgW<sk+t?-^cDc`xXcBa8Xqz&EqgqD0JS+&@ECIQv4m`>>`H`Nve5PbS=d-Bs^c^Dc@d%9t6oxGv+^3ZaZnWTm;Cr`A>fw(KZW1e$m9H_ub`Z-FNQO;hLjMo-B$m6a|t5q=oXMsNQCnc1ZvR7cecXtvNaS*HqqlDCGV0=IZBNAR9}THobl#Qj(IU<a_-Ya(s*8$=Ur+m#p+-z$xY-1kDQkYd>*^AqRBte$|ZVYOhZKf9X|SO5S3Nj32*8l#VY00Fwh7|M=&Lfg|1vBYQl0ssI200dcD'
REPLAY_DATA_SHA256 = 'bc464e301feedaf0b10f4c61a03be22190dc05dbb32ad1c8744dce1876e16ab4'
REPLAY_DATA_SIZE = 2017098


In [ ]:
packet_decoder = lzma.LZMADecompressor(memlimit=64 * 1024 * 1024)
assert 0 < REPLAY_DATA_SIZE < 16 * 1024 * 1024
raw_packet = packet_decoder.decompress(
    base64.b85decode(REPLAY_DATA_BASE85), max_length=REPLAY_DATA_SIZE + 1
)
assert packet_decoder.eof and not packet_decoder.unused_data
assert len(raw_packet) == REPLAY_DATA_SIZE
assert hashlib.sha256(raw_packet).hexdigest() == REPLAY_DATA_SHA256
manifest_end = 4 + int.from_bytes(raw_packet[:4], "little")
assert 4 < manifest_end <= 4096
manifest = json.loads(raw_packet[4:manifest_end])
assert sorted(manifest) == [
    "DATA_LICENSE.md",
    "SOURCE_NOTICES.md",
    "continuation_study.json",
    "device_check_cuda_round20.json",
    "frozen_local_summary.json",
    "full_certificate_audit.csv",
    "guard_audit.json",
    "licenses/LICENSE-APACHE-2.0.txt",
    "licenses/LICENSE-LEVY-SOURCES.txt",
    "longer_study.json",
    "matched_public_starts.npz",
    "movement_starts.npz",
    "movement_study.json",
    "rotation_audit.json",
    "source_parser_audit.csv",
    "witness_layout_hashes.json",
    "witness_offsets.npz",
]
offset = manifest_end
for relative_name, byte_count in manifest.items():
    assert isinstance(byte_count, int) and 0 < byte_count <= 4 * 1024 * 1024
    end = offset + byte_count
    assert end <= len(raw_packet)
    destination = ROOT / relative_name
    assert destination.resolve().is_relative_to(ROOT.resolve())
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(raw_packet[offset:end])
    offset = end
assert offset == len(raw_packet)
del raw_packet, packet_decoder
summary = json.loads((ROOT / "frozen_local_summary.json").read_text())


"""Restore exact numerical witnesses relative to the pinned public data."""

import csv
import hashlib
import itertools
import json
from pathlib import Path

import numpy as np
import requests

REFERENCE = {
    "dataset": "prvsiyan/squares-verified-best-of-1105",
    "version": 2,
    "file": "submission.csv",
    "bytes": 26662946,
    "sha256": "00f11d13ec771fc050a2eef78ab3c93ef420c7b2ac983e85d92392b35719cf47",
}


def file_sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def fetch_pinned_reference(path):
    """Stream just the fixed CSV; never read the mutable latest dataset."""
    path = Path(path)
    downloaded = not path.exists()
    if downloaded:
        url = (
            "https://www.kaggle.com/api/v1/datasets/download/"
            f"{REFERENCE['dataset']}/{REFERENCE['file']}"
            f"?datasetVersionNumber={REFERENCE['version']}"
        )
        partial = path.with_suffix(".csv.part")
        with requests.Session() as session:
            session.trust_env = False
            with session.get(url, stream=True, timeout=(20, 60)) as response:
                response.raise_for_status()
                assert response.status_code == 200
                with partial.open("wb") as stream:
                    for block in response.iter_content(262144):
                        stream.write(block)
                        assert stream.tell() <= REFERENCE["bytes"]
        assert partial.stat().st_size == REFERENCE["bytes"]
        assert file_sha256(partial) == REFERENCE["sha256"]
        partial.replace(path)
    assert path.stat().st_size == REFERENCE["bytes"]
    assert file_sha256(path) == REFERENCE["sha256"]
    return {
        **REFERENCE,
        "downloaded": downloaded,
        "streamed": True,
        "network_workers": 1,
    }


def iter_selected_layouts(path, selected):
    """Parse the source layout in square-ID order, one puzzle at a time."""
    with Path(path).open(newline="") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ["id", "n", "square_id", "x", "y", "angle"]
        for count, rows in itertools.groupby(reader, key=lambda row: int(row["n"])):
            if str(count) not in selected:
                continue
            rows = list(rows)
            assert len(rows) == count
            assert all(int(row["square_id"]) == i for i, row in enumerate(rows))
            coordinates = np.asarray(
                [[float(row[key]) for key in ("x", "y", "angle")] for row in rows],
                dtype="<f8",
            )
            assert np.isfinite(coordinates).all()
            yield str(count), coordinates


def restore_layouts(reference_path, offsets_path, expected_hashes):
    """Undo the byte shuffle and add uint64 offsets modulo 2**64."""
    layouts = {}
    with np.load(offsets_path, allow_pickle=False) as offsets:
        assert set(offsets.files) == set(expected_hashes)
        for key, original in iter_selected_layouts(reference_path, expected_hashes):
            packed = offsets[key]
            assert packed.dtype == np.uint8
            assert packed.shape == (8, 3 * int(key))
            residual = (
                np.ascontiguousarray(packed.T).view("<u8").reshape(original.shape)
            )
            restored = (original.view("<u8") + residual).view("<f8")
            assert np.isfinite(restored).all()
            assert (
                hashlib.sha256(restored.tobytes()).hexdigest() == expected_hashes[key]
            )
            layouts[key] = restored
    assert set(layouts) == set(expected_hashes)
    return layouts


def materialize_witnesses(cache, root):
    """Verify the source bytes and every restored layout before export."""
    root = Path(root)
    reference_path = Path(cache) / "prvsiyan_dataset_v2.csv"
    transfer = fetch_pinned_reference(reference_path)
    expected_hashes = json.loads((root / "witness_layout_hashes.json").read_text())
    layouts = restore_layouts(
        reference_path, root / "witness_offsets.npz", expected_hashes
    )
    np.savez_compressed(root / "retained_contact_witnesses.npz", **layouts)
    transfer["layout_count"] = len(layouts)
    (root / "witness_reference_download.json").write_text(
        json.dumps(transfer, indent=2) + "\n"
    )
    print(f"Restored and checked {len(layouts)} exact float64 layouts from Dataset V2.")


materialize_witnesses(CACHE, ROOT)


### Preserve unselected decimal rows

Replace 65 complete layouts. Keep the other 483 layouts and 241,551 decimal rows byte for byte from V30, including the original row IDs, square IDs, and coordinate strings.


In [ ]:
%%writefile decimal_witness_union.py
"""Replace complete layouts while preserving every other checked decimal row."""

import csv
from pathlib import Path

import numpy as np


def replay_public_witnesses(baseline_csv, witness_path, output_csv):
    with np.load(witness_path, allow_pickle=False) as archive:
        witnesses = {int(key): archive[key].copy() for key in archive.files}
    for count, packing in witnesses.items():
        assert packing.shape == (count, 3) and np.isfinite(packing).all()
    rows_written = 0
    replaced = set()
    with Path(baseline_csv).open(newline="") as source, Path(output_csv).open(
        "w", newline=""
    ) as destination:
        reader = csv.DictReader(source)
        assert reader.fieldnames == ["id", "n", "square_id", "x", "y", "angle"]
        writer = csv.DictWriter(destination, fieldnames=reader.fieldnames)
        writer.writeheader()
        for row in reader:
            count = int(row["n"])
            if count in witnesses:
                square = witnesses[count][int(row["square_id"])]
                for key, value in zip(("x", "y", "angle"), square):
                    row[key] = format(float(value), ".17g")
                replaced.add(count)
            writer.writerow(row)
            rows_written += 1
    assert replaced == set(witnesses)
    return rows_written


In [ ]:
from decimal_witness_union import replay_public_witnesses

rows_written = replay_public_witnesses(
    baseline_path, "retained_contact_witnesses.npz", "submission.csv"
)
summary = json.loads(Path("frozen_local_summary.json").read_text())
assert rows_written == summary["rows"]
assert sha256_file("submission.csv") == EXPECTED_CSV_SHA256
preserved_hash = hashlib.sha256()
preserved_count = 0
with Path(baseline_path).open("rb") as before_stream, Path("submission.csv").open(
    "rb"
) as after_stream:
    assert before_stream.readline() == after_stream.readline()
    for before, after in zip(before_stream, after_stream, strict=True):
        if int(before.split(b",")[1]) not in summary["replaced_sizes"]:
            assert before == after
            preserved_hash.update(before)
            preserved_count += 1
assert preserved_count == summary["preserved_rows"]
assert preserved_hash.hexdigest() == summary["unchanged_rows_sha256"]
Path("local_summary.json").write_text(json.dumps(summary, indent=2))
pd.DataFrame(summary["provenance"]).to_csv("packing_provenance.csv", index=False)
display(
    pd.DataFrame(
        [
            {
                "Stage": "Previous own V30: complete Kaggle CPU raw certificate",
                "Score": summary["baseline_score"],
            },
            {
                "Stage": "Credited public union: exact group certificates",
                "Score": summary["public_union_score"],
            },
            {"Stage": "Candidate: complete raw certificate", "Score": summary["score"]},
            {
                "Stage": "Candidate: complete ordinary pandas certificate",
                "Score": summary["pandas_score"],
            },
        ]
    )
    .style.hide(axis="index")
    .format({"Score": "{:.12f}"})
)
print("Whole replacements:", summary["replaced_sizes"])
print("Byte-identical decimal rows:", preserved_count)
display(FileLink("SOURCE_NOTICES.md"))


## 4. Matched packing and contact figures

Compare the guarded public starts with the final witnesses at n=910, 991 and 994. The n=910 and n=994 controls retain their starting layouts. Use the exact score terms and movement histograms to see changes at n=991.


### `contact_diagnostics.py`

Measured before/after geometry and contact-graph changes, without scaling the squares.

In [ ]:
%%writefile contact_diagnostics.py
"""Measured geometry and contact diagnostics for the retained complete packing witnesses."""

import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon
from matplotlib.ticker import MaxNLocator
from packing_geometry import pair_data, side, vertices


def draw_packing(axis, packing, title, extent):
    polygons = [Polygon(points, closed=True) for points in vertices(packing)]
    tilt = np.abs((packing[:, 2] + np.pi / 4) % (np.pi / 2) - np.pi / 4)
    colours = np.where(
        (tilt > 0.001)[:, None],
        np.array([0.08, 0.5, 0.48]),
        np.array([0.7, 0.77, 0.82]),
    )
    axis.add_collection(
        PatchCollection(polygons, facecolor=colours, edgecolor="white", linewidth=0.25)
    )
    axis.set(
        xlim=(-0.1, extent + 0.1),
        ylim=(-0.1, extent + 0.1),
        aspect="equal",
        title=title,
    )
    axis.set_xticks([])
    axis.set_yticks([])


def active_contacts(packing, tolerance=2e-7):
    pairs, projection, support = pair_data(packing)
    gaps = np.abs(projection).max(axis=1) - support
    return {tuple(map(int, pair)) for pair, gap in zip(pairs, gaps) if gap <= tolerance}


def create_diagnostics(before, after, output_prefix="contact"):
    extent = max(side(before), side(after))
    figure, axes = plt.subplots(1, 2, figsize=(11, 5.3), layout="constrained")
    draw_packing(
        axes[0],
        before,
        f"Certified matched start · n={len(before)}\nL={side(before):.9f}",
        extent,
    )
    draw_packing(
        axes[1],
        after,
        f"Certified candidate · n={len(after)}\nL={side(after):.9f}",
        extent,
    )
    figure.savefig(f"{output_prefix}_packings.png", dpi=160)
    plt.show()
    old_contacts = active_contacts(before)
    new_contacts = active_contacts(after)
    displacement = np.linalg.norm(after[:, :2] - before[:, :2], axis=1)
    angle_change = np.abs(
        (after[:, 2] - before[:, 2] + np.pi / 4) % (np.pi / 2) - np.pi / 4
    )
    figure, axes = plt.subplots(1, 3, figsize=(12, 3.6), layout="constrained")
    counts = [
        len(old_contacts & new_contacts),
        len(old_contacts - new_contacts),
        len(new_contacts - old_contacts),
    ]
    axes[0].bar(
        ["Retained", "Released", "New"], counts, color=["#a4b7c6", "#d39b6d", "#14877d"]
    )
    axes[0].set(ylabel="Near-contact pairs", title="Contact graph changes")
    axes[1].hist(displacement, bins=35, color="#14877d")
    axes[1].set(
        xlabel="Centre displacement", ylabel="Squares", title="How far centres moved"
    )
    axes[1].xaxis.set_major_locator(MaxNLocator(nbins=4))
    axes[2].hist(np.degrees(angle_change) * 1000, bins=35, color="#a4b7c6")
    axes[2].set(
        xlabel="Absolute angle change (millidegrees)",
        ylabel="Squares",
        title="How much angles changed",
    )
    figure.savefig(f"{output_prefix}_changes.png", dpi=160)
    plt.show()
    return {
        "diagnostic_contact_tolerance": 2e-7,
        "retained_contact_pairs": counts[0],
        "released_contact_pairs": counts[1],
        "new_contact_pairs": counts[2],
        "max_centre_displacement": float(displacement.max()),
        "max_angle_change_degrees": float(np.degrees(angle_change).max()),
        "side_reduction": side(before) - side(after),
        "score_reduction": (side(before) - side(after)) / math.sqrt(len(before)),
    }


In [ ]:
from contact_diagnostics import create_diagnostics

with np.load("matched_public_starts.npz", allow_pickle=False) as archive:
    baseline_bank = {int(key): archive[key].copy() for key in archive.files}
candidate_rows = {count: [] for count in summary["diagnostics_n"]}
with Path("submission.csv").open(newline="") as stream:
    for row in csv.DictReader(stream):
        count = int(row["n"])
        if count in candidate_rows:
            candidate_rows[count].append(
                [float(row[key]) for key in ("x", "y", "angle")]
            )
candidate_bank = {count: np.asarray(rows) for count, rows in candidate_rows.items()}
assert all(len(packing) == count for count, packing in candidate_bank.items())
diagnostic_rows = []
for count in summary["diagnostics_n"]:
    result = create_diagnostics(
        baseline_bank[count], candidate_bank[count], output_prefix=f"contact_{count}"
    )
    diagnostic_rows.append({"n": count, **result})
Path("contact_diagnostics.json").write_text(json.dumps(diagnostic_rows, indent=2))
display(pd.DataFrame(diagnostic_rows).style.hide(axis="index").format(precision=12))


## 5. EDA: movement, clearance and continuation

The first figure compares matched movement penalties, accepted steps and time, the 19 source parser rejections, how inherited gains concentrate by size, and the cost of positive export clearance. The second figure shows the exact per-group aggregate through the successive CPU budgets. The final full-file certificates are separate.


In [ ]:
"""Summarize the paired movement-penalty trials and the exact export checks."""

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def render_experiment_audit(summary):
    study = json.loads(Path("movement_study.json").read_text())
    trials = pd.DataFrame(study["trials"])
    trials.to_csv("movement_penalty_audit.csv", index=False)
    guarded = pd.read_json("guard_audit.json")
    guarded["guard_cost"] = guarded["score"] - guarded["source_exact_term"]
    guarded["inherited_gain"] = guarded["parent_exact_term"] - guarded["score"]
    guarded.to_csv("positive_guard_audit.csv", index=False)
    admitted = guarded.loc[guarded["retained"]].sort_values(
        "inherited_gain", ascending=False
    )
    concentration = admitted[["n", "inherited_gain"]].reset_index(drop=True)
    concentration["rank"] = np.arange(1, len(concentration) + 1)
    concentration["cumulative_share"] = (
        concentration["inherited_gain"].cumsum() / concentration["inherited_gain"].sum()
    )
    concentration.to_csv("contribution_concentration.csv", index=False)
    parser = pd.read_csv("source_parser_audit.csv")
    rejected = parser[parser["ordinary_pandas_status"] != "ok"]

    figure, panels = plt.subplots(2, 3, figsize=(17, 9), layout="constrained")
    figure.suptitle(
        "Movement regularization and certified whole-layout export", fontsize=19
    )
    first, runtime, parser_plot, source_share, guard_plot, total = panels.ravel()
    paired = trials.pivot(
        index=["n", "seed"], columns="movement_penalty", values="floating_gain"
    )
    positions = np.arange(len(paired))
    for column, (penalty, values) in enumerate(paired.items()):
        first.bar(
            positions + (column - 0.5) * 0.34,
            1e9 * values,
            width=0.34,
            label=f"λ = {penalty:g}",
        )
    first.set_xticks(positions, [f"{n} / {seed}" for n, seed in paired.index])
    first.set(
        title="Paired CPU starts",
        xlabel="n / seed",
        ylabel="Floating side gain / √n × 10⁹",
    )
    first.tick_params(axis="x", rotation=30)
    first.legend()

    for penalty, batch in trials.groupby("movement_penalty"):
        runtime.scatter(
            batch["elapsed_seconds"],
            batch["accepted_steps"],
            label=f"λ = {penalty:g}",
        )
    runtime.set(
        title="CPU budget and accepted steps",
        xlabel="Measured seconds",
        ylabel="Accepted LP steps",
    )
    runtime.legend()

    parser_plot.scatter(rejected["n"], rejected["score_penalty"], color="#a44335")
    parser_plot.set(
        title=f"Dataset V2: {len(rejected)} ordinary-parser rejections",
        xlabel="n; original decimal rows all passed",
        ylabel="Metric penalty",
    )

    source_share.plot(
        np.r_[0, concentration["rank"]],
        np.r_[0, 100 * concentration["cumulative_share"]],
        color="#466b43",
    )
    top = concentration.iloc[0]
    source_share.annotate(
        f"n={int(top['n'])} contributes {100 * top['cumulative_share']:.1f}%",
        xy=(1, 100 * top["cumulative_share"]),
        xytext=(7, 55),
        arrowprops={"arrowstyle": "->", "color": "#466b43"},
    )
    source_share.set(
        title="Concentration of inherited gains",
        xlabel="Whole layouts in descending gain order",
        ylabel="Cumulative share (%)",
        ylim=(0, 105),
    )

    guard_plot.scatter(
        guarded["n"], 1e12 * guarded["guard_cost"], color="#307780", s=20
    )
    guard_plot.axhline(0, color="0.5", linewidth=0.8)
    guard_plot.set(
        title="Cost of positive clearance",
        xlabel="n; exact group certificates",
        ylabel="Guarded term minus source term × 10¹²",
    )

    labels = ["Inherited whole layouts", "Further own CPU"]
    values = 1e9 * np.array(
        [summary["inherited_public_gain"], summary["additional_cpu_gain"]]
    )
    bars = total.barh(labels, values, color=["#789341", "#246699"])
    total.set_xscale("symlog", linthresh=0.1)
    total.bar_label(bars, fmt="%.3g", padding=4)
    total.margins(x=0.2)
    total.set(
        title="Separate exact contributions",
        xlabel="Score reduction × 10⁹ (symmetric log axis)",
    )
    for panel in panels.ravel():
        panel.grid(alpha=0.16)
    figure.savefig("experiment_audit.png", dpi=145)
    display(figure)
    plt.close(figure)

    columns = [
        "n",
        "seed",
        "movement_penalty",
        "elapsed_seconds",
        "accepted_steps",
        "floating_gain",
        "raw_term",
        "pandas_term",
    ]
    display(trials[columns].style.hide(axis="index").format(precision=10))
    display(pd.read_csv("full_certificate_audit.csv").style.hide(axis="index"))

    continuation = json.loads(Path("continuation_study.json").read_text())
    longer = json.loads(Path("longer_study.json").read_text())
    pd.concat(
        [
            pd.DataFrame(continuation["trials"]).assign(stage="Trust continuation"),
            pd.DataFrame(longer["trials"]).assign(stage="Longer continuation"),
        ],
        ignore_index=True,
    ).to_csv("n991_continuation_audit.csv", index=False)
    phases = pd.DataFrame(
        [
            ("Guarded union", summary["public_union_score"]),
            ("Movement controls", study["raw_group_aggregate"]),
            ("Trust continuation", continuation["raw_group_aggregate"]),
            ("Longer continuation", longer["raw_group_aggregate"]),
        ],
        columns=["phase", "score"],
    )
    phases["gap_to_exact_source"] = phases["score"] - summary["source_exact_score"]
    phases.to_csv("continuation_score_path.csv", index=False)
    path_figure, axis = plt.subplots(figsize=(10, 4.2), layout="constrained")
    axis.plot(
        phases["phase"],
        1e9 * phases["gap_to_exact_source"],
        marker="o",
        color="#246699",
    )
    axis.axhline(0, color="#a44335", linestyle="--", label="Fixed Dataset V2 exact raw")
    axis.set(
        title="Which phase closed the gap to the exact source?",
        ylabel="Exact group aggregate minus source × 10⁹",
    )
    axis.legend()
    axis.grid(alpha=0.2)
    path_figure.savefig("continuation_score_path.png", dpi=145)
    display(path_figure)
    plt.close(path_figure)


render_experiment_audit(summary)


### Matched movement controls

Let $z$ be the displacement in units of the current trust radius. The LP minimizes the side step plus $\lambda\sum_i |z_i|$ over centres and angles. The absolute-value slacks make the penalty linear. Both arms use the same feasible set, seeds and starts; the penalty changes which proposal is selected. The exact admission criterion remains a lower certified side.

The first 12 arms compare $\lambda=0$ and $10^{-8}$ on three sizes and two seeds. The positive penalty accepted no moves under these budgets. Only n=991 improved, with seed 0 and the unpenalized objective.


In [ ]:
%%writefile cpu_angle_controls.py
# Derived from our previously published CPU contact module; upstream attribution below.
# New matched controls vary only the angle bound, with common clearance and centre trust.
# Adapted from lightningv08 CUDA Packing Search contact_polish.py (Apache 2.0).
# Changes: CPU-only imports, configurable regularization/trust, local experiment runner.
"""Trust-region sequential LP with free centers/angles and exact SAT admission.

Background: coupled sparse LP refinement in prvsiyan's A Square Deal notebook.
This vectorized constraint builder and experiment runner are implemented here.
"""

import os

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
import argparse, json, math, time
from pathlib import Path
import numpy as np
from scipy.optimize import linprog
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from packing_geometry import side, normalize, repair


def polish_angle_control(
    p,
    rounds=8,
    seed=0,
    max_seconds=12,
    regularization=0.0,
    trust_start=0.04,
    angle_scale=0.4,
    clearance=3e-12,
    minimum_side_gain=1e-11,
):
    assert 0 <= angle_scale <= 0.4
    assert 0 < trust_start <= 0.04
    assert clearance >= 2e-12
    p = normalize(p)
    n = len(p)
    rng = np.random.default_rng(seed)
    at = time.perf_counter()
    step = trust_start
    trace = []
    for it in range(rounds):
        if time.perf_counter() - at > max_seconds:
            break
        L = side(p)
        theta = p[:, 2]
        c = np.cos(theta)
        s = np.sin(theta)
        h = (abs(c) + abs(s)) / 2
        edges = cKDTree(p[:, :2]).query_pairs(1.75, output_type="ndarray")
        i, j = edges.T
        d = p[j, :2] - p[i, :2]
        ux = np.column_stack((c[i], -s[i], c[j], -s[j]))
        uy = np.column_stack((s[i], c[i], s[j], c[j]))
        proj = d[:, 0, None] * ux + d[:, 1, None] * uy
        ranks = abs(proj)
        rank = ranks + rng.uniform(-2e-12, 2e-12, ranks.shape)
        chosen = rank.argmax(1)
        rr = np.arange(len(i))
        sign = np.sign(proj[rr, chosen])
        vx = sign * ux[rr, chosen]
        vy = sign * uy[rr, chosen]
        rotation = -d[:, 0] * vy + d[:, 1] * vx
        delta = theta[j] - theta[i]
        cd = np.cos(delta)
        sd = np.sin(delta)
        rad = (1 + abs(cd) + abs(sd)) / 2
        dr = (-sd * np.sign(cd) + cd * np.sign(sd)) / 2
        dist = abs(proj[rr, chosen])
        own = chosen < 2
        columns = np.column_stack(
            (3 * i, 3 * i + 1, 3 * j, 3 * j + 1, 3 * i + 2, 3 * j + 2)
        )
        values = np.column_stack(
            (
                vx,
                vy,
                -vx,
                -vy,
                -dr - np.where(own, rotation, 0),
                dr - np.where(own, 0, rotation),
            )
        )
        rows = []
        cols = []
        vals = []
        bounds = []
        offset = 0

        def add(cc, vv, bb):
            nonlocal offset
            bb = np.asarray(bb)
            count = len(bb)
            rows.append(np.repeat(np.arange(offset, offset + count), cc.shape[1]))
            cols.append(cc.ravel())
            vals.append(vv.ravel())
            bounds.append(bb)
            offset += count

        add(columns, values, dist - rad - clearance)
        # At relative right angles, both one-sided radius derivatives are needed.
        kink = np.flatnonzero(np.minimum(abs(cd), abs(sd)) < 1e-8)
        for slope in [-0.5, 0.5]:
            vv = values[kink].copy()
            vv[:, 4] = -slope - np.where(own[kink], rotation[kink], 0)
            vv[:, 5] = slope - np.where(own[kink], 0, rotation[kink])
            add(columns[kink], vv, dist[kink] - rad[kink] - clearance)
        k = np.arange(n)
        dh = (-s * np.sign(c) + c * np.sign(s)) / 2
        aligned = np.minimum(abs(c), abs(s)) < 1e-8
        for dim in [0, 1]:
            for upper in [False, True]:
                ids = k
                extra = k[aligned]
                for ids, deriv in [
                    (ids, dh),
                    (extra, np.full(len(extra), -0.5)),
                    (extra, np.full(len(extra), 0.5)),
                ]:
                    cc = np.column_stack(
                        (3 * ids + dim, 3 * ids + 2, np.full(len(ids), 3 * n))
                    )
                    vv = np.column_stack(
                        (
                            np.full(len(ids), 1.0 if upper else -1.0),
                            deriv,
                            np.full(len(ids), -1.0 if upper else 0.0),
                        )
                    )
                    bb = L - p[ids, dim] - h[ids] if upper else p[ids, dim] - h[ids]
                    add(cc, vv, bb)
        k = np.arange(3 * n)
        u = 3 * n + 1 + k
        for sign in [-1.0, 1.0]:
            add(
                np.column_stack((k, u)),
                np.tile([sign, -1.0], (3 * n, 1)),
                np.zeros(3 * n),
            )
        A = coo_matrix(
            (np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
            shape=(offset, 6 * n + 1),
        ).tocsr()
        rhs = np.concatenate(bounds)
        objective = np.zeros(6 * n + 1)
        objective[3 * n] = 1
        objective[3 * n + 1 :] = regularization
        trust = np.tile([step, step, step * angle_scale], n)
        limits = list(zip(-trust, trust)) + [(-step, step)] + [(0, None)] * (3 * n)
        fit = linprog(
            objective,
            A_ub=A,
            b_ub=rhs,
            bounds=limits,
            method="highs",
            options={
                "primal_feasibility_tolerance": 1e-10,
                "dual_feasibility_tolerance": 1e-9,
                "time_limit": 2,
            },
        )
        if not fit.success:
            step *= 0.4
            if step < 2e-7:
                break
            continue
        move = fit.x[: 3 * n].reshape(n, 3)
        winner = None
        for alpha in [1.0, 0.5, 0.25, 0.125, 0.0625, 0.03125]:
            trial = repair(p + alpha * move, clearance=clearance)
            if (
                trial is not None
                and side(trial) < L - minimum_side_gain
                and (winner is None or side(trial) < side(winner))
            ):
                winner = trial
        if winner is None:
            step *= 0.4
            if step < 2e-7:
                break
        else:
            p = winner
            trace.append(dict(iteration=it, before=L, after=side(p), trust=step))
            step = min(trust_start, step * 1.3)
    return p, trace, time.perf_counter() - at


### Continuations from saved winners

Eight matched arms use the certified n=991 winner from the first study. Cross initial centre trust $10^{-4}$ and $10^{-3}$ with scaled and unscaled variables and two seeds. The best unscaled seed 55 run stopped at its 45-second budget. Two longer 60-second arms use its saved result. Angles are bounded by 0.4 times the centre trust in every LP, and the cap is checked between rounds.


### Rigid orientation control

Keeping all internal contacts fixed, sweep the enclosing axes using convex-hull support changes and width/height crossings. The three-square strip control attained $4/\sqrt{2}$ at 45 degrees. The sweep across all 548 explicit sizes found no useful rigid-rotation improvement. The downloadable rotation audit includes every size.


### New CPU control: normalize the LP displacement units

Write a displacement as $δ = r z$ at trust radius $r$. The same linearized inequality $Aδ ≤ b$ becomes $Az ≤ b/r$. The dimensionless trust bound is one, and the proposed displacement is mapped back to physical units before geometry checks. This changes how the solver tolerance applies to a tiny gap. It does not change the final certification or the positive export floor.

The perturbed analytic five-square control reduced the side in both modes and passed exact raw/pandas checks. The unscaled control reproduced the previous core's array exactly. `analytic_control.json` records the remaining error from the known side. This is a functional control, not an optimality claim.

In [ ]:
%%writefile cpu_scaled_trust.py
# Derived from our previously published CPU contact module; upstream attribution below.
# New matched controls vary only the angle bound, with common clearance and centre trust.
# Adapted from lightningv08 CUDA Packing Search contact_polish.py (Apache 2.0).
# Changes: CPU-only imports, configurable regularization/trust, local experiment runner.
"""Trust-region LP with optional normalization of its displacement variables.

At trust radius r, write every displacement as r*z and divide each geometric
right-hand side by r. This leaves the same linearized feasible set while the
LP feasibility tolerance is interpreted at scale r. Numerical results can
differ, so unchanged exact geometry and serialized official metric gates still
admit every retained proposal. The variable-scaling idea is credited to the
readable live contact-polish explanation in Prvsiyan V28; its code was not
executed or copied. Implementation derives from our previous published core.

Trust-region sequential LP with free centers/angles and exact SAT admission.

Background: coupled sparse LP refinement in prvsiyan's A Square Deal notebook.
This vectorized constraint builder and experiment runner are implemented here.
"""

import os

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
import argparse, json, math, time
from pathlib import Path
import numpy as np
from scipy.optimize import linprog
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from packing_geometry import side, normalize, repair


def polish_scaled_trust(
    p,
    rounds=8,
    seed=0,
    max_seconds=12,
    regularization=0.0,
    trust_start=0.04,
    angle_scale=0.4,
    clearance=3e-12,
    minimum_side_gain=1e-11,
    normalize_variables=True,
):
    assert 0 <= angle_scale <= 0.4
    assert 0 < trust_start <= 0.04
    assert clearance >= 2e-12
    p = normalize(p)
    n = len(p)
    rng = np.random.default_rng(seed)
    at = time.perf_counter()
    step = trust_start
    trace = []
    for it in range(rounds):
        if time.perf_counter() - at > max_seconds:
            break
        L = side(p)
        theta = p[:, 2]
        c = np.cos(theta)
        s = np.sin(theta)
        h = (abs(c) + abs(s)) / 2
        edges = cKDTree(p[:, :2]).query_pairs(1.75, output_type="ndarray")
        i, j = edges.T
        d = p[j, :2] - p[i, :2]
        ux = np.column_stack((c[i], -s[i], c[j], -s[j]))
        uy = np.column_stack((s[i], c[i], s[j], c[j]))
        proj = d[:, 0, None] * ux + d[:, 1, None] * uy
        ranks = abs(proj)
        rank = ranks + rng.uniform(-2e-12, 2e-12, ranks.shape)
        chosen = rank.argmax(1)
        rr = np.arange(len(i))
        sign = np.sign(proj[rr, chosen])
        vx = sign * ux[rr, chosen]
        vy = sign * uy[rr, chosen]
        rotation = -d[:, 0] * vy + d[:, 1] * vx
        delta = theta[j] - theta[i]
        cd = np.cos(delta)
        sd = np.sin(delta)
        rad = (1 + abs(cd) + abs(sd)) / 2
        dr = (-sd * np.sign(cd) + cd * np.sign(sd)) / 2
        dist = abs(proj[rr, chosen])
        own = chosen < 2
        columns = np.column_stack(
            (3 * i, 3 * i + 1, 3 * j, 3 * j + 1, 3 * i + 2, 3 * j + 2)
        )
        values = np.column_stack(
            (
                vx,
                vy,
                -vx,
                -vy,
                -dr - np.where(own, rotation, 0),
                dr - np.where(own, 0, rotation),
            )
        )
        rows = []
        cols = []
        vals = []
        bounds = []
        offset = 0

        def add(cc, vv, bb):
            nonlocal offset
            bb = np.asarray(bb)
            count = len(bb)
            rows.append(np.repeat(np.arange(offset, offset + count), cc.shape[1]))
            cols.append(cc.ravel())
            vals.append(vv.ravel())
            bounds.append(bb)
            offset += count

        add(columns, values, dist - rad - clearance)
        # At relative right angles, both one-sided radius derivatives are needed.
        kink = np.flatnonzero(np.minimum(abs(cd), abs(sd)) < 1e-8)
        for slope in [-0.5, 0.5]:
            vv = values[kink].copy()
            vv[:, 4] = -slope - np.where(own[kink], rotation[kink], 0)
            vv[:, 5] = slope - np.where(own[kink], 0, rotation[kink])
            add(columns[kink], vv, dist[kink] - rad[kink] - clearance)
        k = np.arange(n)
        dh = (-s * np.sign(c) + c * np.sign(s)) / 2
        aligned = np.minimum(abs(c), abs(s)) < 1e-8
        for dim in [0, 1]:
            for upper in [False, True]:
                ids = k
                extra = k[aligned]
                for ids, deriv in [
                    (ids, dh),
                    (extra, np.full(len(extra), -0.5)),
                    (extra, np.full(len(extra), 0.5)),
                ]:
                    cc = np.column_stack(
                        (3 * ids + dim, 3 * ids + 2, np.full(len(ids), 3 * n))
                    )
                    vv = np.column_stack(
                        (
                            np.full(len(ids), 1.0 if upper else -1.0),
                            deriv,
                            np.full(len(ids), -1.0 if upper else 0.0),
                        )
                    )
                    bb = L - p[ids, dim] - h[ids] if upper else p[ids, dim] - h[ids]
                    add(cc, vv, bb)
        k = np.arange(3 * n)
        u = 3 * n + 1 + k
        for sign in [-1.0, 1.0]:
            add(
                np.column_stack((k, u)),
                np.tile([sign, -1.0], (3 * n, 1)),
                np.zeros(3 * n),
            )
        A = coo_matrix(
            (np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
            shape=(offset, 6 * n + 1),
        ).tocsr()
        rhs = np.concatenate(bounds)
        objective = np.zeros(6 * n + 1)
        objective[3 * n] = 1
        objective[3 * n + 1 :] = regularization
        trust = np.tile([step, step, step * angle_scale], n)
        limits = list(zip(-trust, trust)) + [(-step, step)] + [(0, None)] * (3 * n)
        if normalize_variables:
            # delta = step*z for centres, angles, box side and absolute slacks.
            rhs = rhs / step
            scaled_trust = np.tile([1.0, 1.0, angle_scale], n)
            limits = (
                list(zip(-scaled_trust, scaled_trust))
                + [(-1.0, 1.0)]
                + [(0, None)] * (3 * n)
            )
        fit = linprog(
            objective,
            A_ub=A,
            b_ub=rhs,
            bounds=limits,
            method="highs",
            options={
                "primal_feasibility_tolerance": 1e-10,
                "dual_feasibility_tolerance": 1e-9,
                "time_limit": 2,
            },
        )
        if not fit.success:
            step *= 0.4
            if step < 2e-7:
                break
            continue
        move = fit.x[: 3 * n].reshape(n, 3)
        if normalize_variables:
            move = move * step
        winner = None
        for alpha in [1.0, 0.5, 0.25, 0.125, 0.0625, 0.03125]:
            trial = repair(p + alpha * move, clearance=clearance)
            if (
                trial is not None
                and side(trial) < L - minimum_side_gain
                and (winner is None or side(trial) < side(winner))
            ):
                winner = trial
        if winner is None:
            step *= 0.4
            if step < 2e-7:
                break
        else:
            p = winner
            trace.append(dict(iteration=it, before=L, after=side(p), trust=step))
            step = min(trust_start, step * 1.3)
    return p, trace, time.perf_counter() - at


In [ ]:
%%writefile replay_movement_controls.py
"""Optionally rerun the 22 CPU arms from their fixed numerical starts."""

import argparse
import json
import math
from pathlib import Path

import numpy as np

from cpu_scaled_trust import polish_scaled_trust
from packing_geometry import side


def run_control(start, config):
    proposal, trace, elapsed = polish_scaled_trust(
        start,
        rounds=config["rounds"],
        seed=config["seed"],
        max_seconds=config["seconds"],
        regularization=config["penalty"],
        trust_start=config["trust"],
        angle_scale=0.4,
        clearance=3e-12,
        minimum_side_gain=1e-11,
        normalize_variables=config["scaled"],
    )
    before, after = side(start), side(proposal)
    return {
        **config,
        "before_side": before,
        "after_side": after,
        "floating_gain": (before - after) / math.sqrt(len(start)),
        "accepted_steps": len(trace),
        "elapsed_seconds": elapsed,
    }


def run_controls(starts_file, output):
    rows = []
    with np.load(starts_file, allow_pickle=False) as starts:
        for count in (910, 991, 994):
            for seed in (0, 55):
                for penalty in (0.0, 1e-8):
                    config = dict(
                        stage="movement penalty",
                        start=str(count),
                        seed=seed,
                        rounds=32,
                        seconds=40,
                        penalty=penalty,
                        trust=1e-3,
                        scaled=True,
                    )
                    rows.append(run_control(starts[str(count)], config))
        for seed in (0, 55):
            for trust in (1e-4, 1e-3):
                for scaled in (False, True):
                    config = dict(
                        stage="trust continuation",
                        start="n991_after_movement",
                        seed=seed,
                        rounds=48,
                        seconds=45,
                        penalty=0.0,
                        trust=trust,
                        scaled=scaled,
                    )
                    rows.append(run_control(starts[config["start"]], config))
        for seed in (55, 0):
            config = dict(
                stage="longer continuation",
                start="n991_after_trust",
                seed=seed,
                rounds=80,
                seconds=60,
                penalty=0.0,
                trust=1e-4,
                scaled=False,
            )
            rows.append(run_control(starts[config["start"]], config))
    assert len(rows) == 22
    Path(output).write_text(json.dumps(rows, indent=2) + "\n")
    return rows


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()
    run_controls(args.input, args.output)


### Exact admission with positive export guards

The finite guard ladder keeps a positive 2e-12 floor and retries1e-11 if needed. Each whole proposed packing must pass raw decimal and ordinary pandas read/write interval checks. Reducing the floor to chase aggregate rounding differences is excluded. Full CSV certification remains a separate final gate.


In [ ]:
%%writefile adaptive_guard.py
"""Repeat the finite, independently certified centre-guard ladder on CPU."""

from pathlib import Path
import csv, io, json, math, sys
import numpy as np
import pandas as pd
from packing_geometry import aabbs, pair_data, side

sys.path.insert(0, str(Path.cwd() / "reference_sources"))
import official_metric as metric

CLEARANCES = (2e-12, 1e-11)
COLUMNS = ["id", "n", "square_id", "x", "y", "angle"]


def encode_rows(count, packing):
    return [
        dict(
            zip(
                COLUMNS,
                [str(i), str(count), str(i), *[format(v, ".17g") for v in square]],
            )
        )
        for i, square in enumerate(packing)
    ]


def ordinary_roundtrip(rows):
    stream = io.StringIO()
    writer = csv.DictWriter(stream, fieldnames=COLUMNS)
    writer.writeheader()
    writer.writerows(rows)
    stream.seek(0)
    return pd.read_csv(
        io.StringIO(pd.read_csv(stream).to_csv(index=False)), dtype=str
    ).to_dict("records")


def guard(packing, clearance):
    packing = packing.copy()
    before = aabbs(packing)[0].min(axis=0)
    packing[:, :2] -= before
    _, projection, support = pair_data(packing)
    scale = (
        max(
            1.0,
            float(
                np.max(
                    (support + clearance)
                    / np.maximum(np.abs(projection).max(axis=1), 1e-30)
                )
            ),
        )
        if len(support)
        else 1.0
    )
    packing[:, :2] *= scale
    after = aabbs(packing)[0].min(axis=0)
    packing[:, :2] -= after
    return packing, {
        "offset_before": before.tolist(),
        "centre_scale": scale,
        "offset_after": after.tolist(),
    }


def certify_group(job):
    count, packing, source, donor_rows = job
    donor_result = metric._evaluate_puzzle_worker((count, donor_rows))
    attempts = []
    for clearance in CLEARANCES:
        guarded, recipe = guard(packing, clearance)
        rows = encode_rows(count, guarded)
        raw = metric._evaluate_puzzle_worker((count, rows))
        roundtrip = metric._evaluate_puzzle_worker((count, ordinary_roundtrip(rows)))
        attempts.append(
            {
                "clearance": clearance,
                "raw_status": raw[3],
                "roundtrip_status": roundtrip[3],
            }
        )
        if raw[3] == roundtrip[3] == "ok":
            return (
                count,
                guarded,
                recipe,
                {
                    "n": count,
                    "source": source,
                    "donor_score": donor_result[1],
                    "donor_status": donor_result[3],
                    "before": side(packing),
                    "after": side(guarded),
                    "score": raw[1],
                    "roundtrip_score": roundtrip[1],
                    "clearance": clearance,
                    "serialization_cost": (side(guarded) - side(packing))
                    / math.sqrt(count),
                    "attempts": attempts,
                },
            )
    raise ValueError(f"No certified guard at n={count}")


def replay_audit(baseline_csv, witness_path, recipe_path, output_path):
    groups = {}
    with Path(baseline_csv).open(newline="") as stream:
        for row in csv.DictReader(stream):
            groups.setdefault(int(row["n"]), []).append(row)
    with np.load(witness_path, allow_pickle=False) as archive:
        witnesses = {int(key): archive[key].copy() for key in archive.files}
    recipe = json.loads(Path(recipe_path).read_text())
    jobs = []
    for text in sorted(recipe, key=int):
        count = int(text)
        rows = sorted(groups[count], key=lambda row: int(row["square_id"]))
        packing = witnesses.get(count)
        if packing is None:
            packing = np.array(
                [[float(row[key]) for key in ("x", "y", "angle")] for row in rows]
            )
        jobs.append((count, packing, "replayed_selection", rows))
    reports = [certify_group(job)[3] for job in jobs]
    Path(output_path).write_text(json.dumps(reports, indent=2))


## 6. Optional real CPU, CUDA and MPS experiments

The matched continuation study uses CPU LP. The separate tensor route preserves real
CPU/CUDA/MPS choices, CPU repair and exact admission. RUN_SEARCH=False retains
the frozen file. CPU initializes no accelerator. A fresh CPU device, gradient
and export check is required. Unchanged CUDA code retains historical real T4
smoke evidence, not a fresh full GPU benchmark. MPS is unused and hardware
untested.


In [ ]:
%%writefile portable_seam_search.py
"""Batched seam-kick SAT pressure search on PyTorch CPU / CUDA / Apple MPS.

Float32 tensors propose layouts; CPU float64 SAT repair and trust-region LP
admit candidates. Unit-square side lengths are constant throughout. No tensor
loss or float32 feasibility result is used as a competition validity test.

Inspired by established soft packing and contact continuation. This independent
portable implementation follows the inherited geometry/contact-search lineage.
"""

import time, math
import numpy as np
from scipy.spatial import cKDTree
from packing_geometry import normalize, side, repair


def choose_device(backend):
    import torch

    if backend not in ("cpu", "cuda", "mps"):
        raise ValueError("Choose cpu, cuda or mps explicitly.")
    # Do not probe or initialize any accelerator in CPU mode.
    if backend == "cuda" and not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA requested but unavailable. Enable a Kaggle NVIDIA GPU or choose cpu."
        )
    if backend == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError(
            "MPS requested but unavailable. MPS needs Apple Silicon/macOS; Kaggle uses cpu or cuda."
        )
    return torch.device(backend)


def tensor_loss(centres, angles, pairs, box):
    import torch

    i, j = pairs[:, 0], pairs[:, 1]
    a = angles[:, i]
    b = angles[:, j]
    dx = centres[:, j, 0] - centres[:, i, 0]
    dy = centres[:, j, 1] - centres[:, i, 1]
    projections = torch.stack(
        (
            dx * torch.cos(a) + dy * torch.sin(a),
            -dx * torch.sin(a) + dy * torch.cos(a),
            dx * torch.cos(b) + dy * torch.sin(b),
            -dx * torch.sin(b) + dy * torch.cos(b),
        ),
        dim=-1,
    )
    radius = (1 + torch.abs(torch.cos(a - b)) + torch.abs(torch.sin(a - b))) * 0.5
    penetration = torch.relu(radius - torch.amax(torch.abs(projections), dim=-1))
    h = (torch.abs(torch.cos(angles)) + torch.abs(torch.sin(angles))) * 0.5
    low = torch.relu(h[:, :, None] - centres)
    high = torch.relu(centres + h[:, :, None] - box)
    return (
        penetration.square().sum(1)
        + low.square().sum((1, 2))
        + high.square().sum((1, 2))
    )


def seam_pressure(
    p, backend="cpu", seed=0, batch=6, steps=240, shrink=0.001, seconds=20
):
    import torch

    device = choose_device(backend)
    if backend == "cpu":
        torch.set_num_threads(2)
    p = normalize(p)
    n = len(p)
    L = side(p)
    rng = np.random.default_rng(seed)
    start = time.perf_counter()
    pair_array = cKDTree(p[:, :2]).query_pairs(2.2, output_type="ndarray")
    # Centre moves are projected to +/-0.25 per coordinate. An omitted pair
    # starts farther than 2.2 > sqrt(2)+2sqrt(2)*0.25 and therefore cannot meet,
    # even if both squares rotate. Final admission independently checks all pairs.
    pairs = torch.tensor(pair_array, dtype=torch.long, device=device)
    original = torch.tensor(p[:, :2], dtype=torch.float32, device=device)
    original_angles = torch.tensor(p[:, 2], dtype=torch.float32, device=device)
    initial = np.repeat(p[None, :, :2], batch, axis=0)
    initial_angles = np.repeat(p[None, :, 2], batch, axis=0)
    box = L - shrink
    for k in range(batch):
        # Opposed tangential motion across one horizontal/vertical seam; a
        # coherent kick can explore different contacts from independent jitter.
        axis = k % 2
        cut = rng.uniform(0.2, 0.8) * L
        halves = np.where(p[:, axis] > cut, 1.0, -1.0)
        initial[k, :, 1 - axis] += (
            halves * rng.choice([-1.0, 1.0]) * rng.uniform(0.0001, 0.0025)
        )
        initial_angles[k] += rng.normal(0, 0.00015, n)
    initial[:, :, :2] = (initial[:, :, :2] - L * 0.5) * (box - 1) / (L - 1) + box * 0.5
    centres = torch.tensor(
        initial, dtype=torch.float32, device=device, requires_grad=True
    )
    angles = torch.tensor(
        initial_angles, dtype=torch.float32, device=device, requires_grad=True
    )
    optimizer = torch.optim.Adam(
        [{"params": [centres], "lr": 0.002}, {"params": [angles], "lr": 0.00012}]
    )
    best_loss = torch.full((batch,), float("inf"), device=device)
    best_c = centres.detach().clone()
    best_a = angles.detach().clone()
    trace = []
    completed = 0
    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)
        loss = tensor_loss(centres, angles, pairs, box)
        if not bool(torch.isfinite(loss).all().item()):
            raise FloatingPointError("Non-finite pressure loss")
        with torch.no_grad():
            better = loss < best_loss
            best_loss = torch.where(better, loss, best_loss)
            best_c = torch.where(better[:, None, None], centres.detach(), best_c)
            best_a = torch.where(better[:, None], angles.detach(), best_a)
        loss.sum().backward()
        optimizer.step()
        with torch.no_grad():
            centres.copy_(
                torch.maximum(torch.minimum(centres, original + 0.25), original - 0.25)
            )
            angles.copy_(
                torch.maximum(
                    torch.minimum(angles, original_angles + 0.05),
                    original_angles - 0.05,
                )
            )
        completed = step + 1
        if step % 40 == 0:
            trace.append({"step": step, "loss": float(best_loss.min().item())})
            if time.perf_counter() - start > seconds:
                break
        if step in [steps // 2, 3 * steps // 4]:
            for group in optimizer.param_groups:
                group["lr"] *= 0.25
    # Transfer only tiny candidate arrays, not autograd state, to CPU.
    cc = best_c.detach().cpu().numpy().astype(float)
    aa = best_a.detach().cpu().numpy().astype(float)
    proposals = []
    for k in range(batch):
        q = repair(np.column_stack((cc[k], aa[k])), clearance=2e-9)
        if q is not None:
            proposals.append(q)
    proposals.sort(key=side)
    report = dict(
        backend=backend,
        batch=batch,
        steps=completed,
        seconds=time.perf_counter() - start,
        trace=trace,
        proposal_sides=[side(q) for q in proposals],
        before=L,
    )
    return proposals, report


if __name__ == "__main__":
    import argparse, json
    from packing_geometry import load_bank, save_bank
    from cpu_contact_polish import polish

    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--backend", choices=["cpu", "cuda", "mps"], default="cpu")
    parser.add_argument("--sizes", nargs="+", type=int, default=[984, 553])
    parser.add_argument("--steps", type=int, default=240)
    parser.add_argument("--batch", type=int, default=6)
    parser.add_argument("--prefix", default="portable_seam")
    args = parser.parse_args()
    bank = load_bank(args.input)
    history = []
    for n in args.sizes:
        initial = side(bank[n])
        proposals, report = seam_pressure(
            bank[n],
            backend=args.backend,
            seed=81173 + n,
            batch=args.batch,
            steps=args.steps,
        )
        for q in proposals[:2]:
            if side(q) > initial + 0.015:
                continue
            q, _, _ = polish(q, rounds=50, max_seconds=5, trust_start=0.015)
            if side(q) < side(bank[n]) - 1e-8:
                bank[n] = q
        report.update(
            n=n, after=side(bank[n]), gain=(initial - side(bank[n])) / math.sqrt(n)
        )
        history.append(report)
        print(json.dumps(report), flush=True)
    save_bank(args.prefix + "_packings.npz", bank)
    from pathlib import Path

    Path(args.prefix + "_history.json").write_text(json.dumps(history, indent=2))


### `portable_experiment.py`

Export only proposals accepted by the unchanged official metric and pandas roundtrip.

In [ ]:
%%writefile portable_experiment.py
"""Recompute using an explicitly chosen tensor device; certify every accepted row.

The tensor proposal stage runs on cpu/cuda/mps. Exact LP polishing, decimal
serialization and the unchanged official metric run on CPU in every mode.
"""

import csv, json, math, sys
from pathlib import Path
from collections import defaultdict
import numpy as np
from packing_geometry import load_bank, side, repair
from portable_seam_search import seam_pressure, choose_device
from cpu_axis_reclosure import reclose
from cpu_contact_polish import polish


def run_experiment(
    input_csv, output_csv, backend="cpu", sizes=(801, 460), batch=6, steps=240
):
    choose_device(backend)  # Fail explicitly; never silently switch devices.
    sys.path.insert(0, str(Path(__file__).resolve().parent / "reference_sources"))
    import official_metric as metric

    bank = load_bank(input_csv)
    rows = defaultdict(list)
    with Path(input_csv).open(newline="") as f:
        for row in csv.DictReader(f):
            rows[int(row["n"])].append(row)
    history = []
    for n in sizes:
        if n not in rows:
            raise ValueError("Choose a submitted non-grid puzzle size.")
        before = metric._evaluate_puzzle_worker((n, rows[n]))
        assert before[3] == "ok"
        proposals, report = seam_pressure(
            bank[n], backend=backend, seed=99173 + n, batch=batch, steps=steps
        )
        q, _, _ = reclose(
            bank[n], seed=397 + n, rounds=6, seconds=3, trust=0.3, tie=0.1
        )
        proposals.append(q)
        best_rows = rows[n]
        best_score = before[1]
        for q in sorted(proposals, key=side)[:3]:
            if side(q) > side(bank[n]) + 0.015:
                continue
            q, _, _ = polish(q, rounds=60, max_seconds=6, trust_start=0.015)
            q = repair(q, clearance=2e-9)
            if q is None:
                continue
            q = np.rint(q * 1e11) / 1e11
            q = repair(q, clearance=2e-9)
            if q is None:
                continue
            candidate = [
                dict(
                    id=str(sid),
                    n=str(n),
                    square_id=str(sid),
                    x=format(x, ".17g"),
                    y=format(y, ".17g"),
                    angle=format(a, ".17g"),
                )
                for sid, (x, y, a) in enumerate(q)
            ]
            result = metric._evaluate_puzzle_worker((n, candidate))
            if result[3] == "ok" and result[1] < best_score - 1e-8:
                # Also reject any candidate that changes validity when parsed
                # with pandas' ordinary float64 parser and serialized again.
                import pandas as pd, io

                buf = io.StringIO()
                pd.read_csv(
                    io.StringIO(pd.DataFrame(candidate).to_csv(index=False))
                ).to_csv(buf, index=False, float_format="%.17g")
                buf.seek(0)
                rt = metric._evaluate_puzzle_worker((n, list(csv.DictReader(buf))))
                if rt[3] != "ok" or abs(rt[1] - result[1]) > 1e-10:
                    continue
                best_rows = candidate
                best_score = result[1]
        rows[n] = best_rows
        report.update(
            n=n,
            official_before=before[1],
            official_after=best_score,
            accepted=best_score < before[1] - 1e-8,
        )
        history.append(report)
    fields = ["id", "n", "square_id", "x", "y", "angle"]
    index = 0
    with Path(output_csv).open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for n in sorted(rows):
            for row in rows[n]:
                row = dict(row)
                row["id"] = str(index)
                writer.writerow(row)
                index += 1
    Path("recompute_history.json").write_text(json.dumps(history, indent=2))
    return history


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--backend", choices=["cpu", "cuda", "mps"], default="cpu")
    parser.add_argument("--sizes", nargs="+", type=int, default=[801, 460])
    parser.add_argument("--batch", type=int, default=6)
    parser.add_argument("--steps", type=int, default=240)
    args = parser.parse_args()
    history = run_experiment(
        args.input,
        args.output,
        backend=args.backend,
        sizes=args.sizes,
        batch=args.batch,
        steps=args.steps,
    )
    print(
        json.dumps(
            [
                dict(
                    n=r["n"],
                    backend=r["backend"],
                    accepted=r["accepted"],
                    before=r["official_before"],
                    after=r["official_after"],
                )
                for r in history
            ],
            indent=2,
        )
    )


### `check_portable_backends.py`

A small gradient and end-to-end export check for the explicitly requested device.

In [ ]:
%%writefile check_portable_backends.py
"""Exercise the requested real tensor device and full small-puzzle export path."""

import argparse, csv, json, tempfile, os, hashlib
from pathlib import Path
import numpy as np
from portable_seam_search import choose_device, tensor_loss, seam_pressure
from portable_experiment import run_experiment


def check(backend):
    import torch

    device = choose_device(backend)
    dtype = torch.float32 if backend == "mps" else torch.float64
    c = torch.tensor(
        [[[0.7, 0.8], [1.5, 1.1], [1.1, 1.8]]],
        dtype=dtype,
        device=device,
        requires_grad=True,
    )
    a = torch.tensor(
        [[0.12, -0.21, 0.37]], dtype=dtype, device=device, requires_grad=True
    )
    pairs = torch.tensor([[0, 1], [0, 2], [1, 2]], dtype=torch.long, device=device)
    loss = tensor_loss(c, a, pairs, 2.8).sum()
    loss.backward()
    assert torch.isfinite(c.grad).all() and torch.isfinite(a.grad).all()
    # Finite differences on a smooth configuration check angle and centre gradients.
    eps = 2e-3 if dtype == torch.float32 else 1e-6
    errors = []
    for tensor, index in [(c, (0, 0, 0)), (a, (0, 1))]:
        analytic = float(tensor.grad[index].item())
        plus = tensor.detach().clone()
        minus = plus.clone()
        plus[index] += eps
        minus[index] -= eps
        fplus = tensor_loss(
            plus if tensor is c else c.detach(),
            a.detach() if tensor is c else plus,
            pairs,
            2.8,
        ).sum()
        fminus = tensor_loss(
            minus if tensor is c else c.detach(),
            a.detach() if tensor is c else minus,
            pairs,
            2.8,
        ).sum()
        err = abs(float((fplus - fminus).item()) / (2 * eps) - analytic)
        errors.append(err)
    assert max(errors) < (3e-3 if dtype == torch.float32 else 1e-5), errors
    root = Path.cwd()
    source = Path("submission.csv")
    with source.open(newline="") as f:
        rows = [r for r in csv.DictReader(f) if int(r["n"]) == 5]
    assert len(rows) == 5
    with tempfile.TemporaryDirectory() as work:
        work = Path(work)
        inp = work / "input.csv"
        with inp.open("w", newline="") as f:
            w = csv.DictWriter(
                f, fieldnames=["id", "n", "square_id", "x", "y", "angle"]
            )
            w.writeheader()
            w.writerows(rows)
        # Keep an audit run separate from any production recomputation history.
        os.chdir(work)
        try:
            history = run_experiment(
                str(inp),
                str(work / "output.csv"),
                backend=backend,
                sizes=[5],
                batch=2,
                steps=24,
            )
        finally:
            os.chdir(root)
    assert history[0]["backend"] == backend and history[0]["steps"] == 24
    result = dict(
        backend=backend,
        actual_tensor_device=str(device),
        torch_version=torch.__version__,
        gradient_max_error=max(errors),
        full_export_history=history,
        cpu_certification=True,
    )
    Path("device_check_" + backend + ".json").write_text(json.dumps(result, indent=2))
    print(json.dumps(result, indent=2))
    return result


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--backend", choices=["cpu", "cuda", "mps"], default="cpu")
    check(p.parse_args().backend)


### `validate_submission.py`

Unchanged official V20 geometry and score functions, wrapped in a sequential
CSV reader. Only one puzzle's decimal rows are retained at a time. Partial
reports resume only with matching full CSV and metric hashes. No worker pool.


In [ ]:
%%writefile validate_submission.py
"""Certify a CSV one puzzle at a time, using the unchanged official metric.

No process pool is created. Partial reports can be resumed only when both the
exact input CSV hash and the official metric hash match the original run.
"""

from pathlib import Path
import argparse
import csv
import hashlib
import importlib.metadata
import itertools
import json
import resource
import sys
import time

METRIC_PATH = Path(__file__).resolve().parent / "reference_sources/official_metric.py"
METRIC_SHA256 = "d269f24b02d5f33fbd2149f76b83414b3b446be4d02776560e04631af3365ccf"


def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(262144), b""):
            digest.update(block)
    return digest.hexdigest()


def peak_memory_mib():
    maximum = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    divisor = 1024 * 1024 if sys.platform == "darwin" else 1024
    return maximum / divisor


def save_report(path, report):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(report, indent=2) + "\n")
    temporary.replace(path)


def validate(input_path, output_path, memory_limit_mib=1536):
    assert file_hash(METRIC_PATH) == METRIC_SHA256
    input_sha256 = file_hash(input_path)
    results = []
    previous_seconds = 0.0
    if output_path.exists():
        previous = json.loads(output_path.read_text())
        assert previous["sha256"] == input_sha256
        assert previous["metric_sha256"] == METRIC_SHA256
        if previous.get("complete"):
            print("Reusing completed certificate for the same CSV and metric hashes.")
            return previous
        results = previous["results"]
        previous_seconds = previous["elapsed_seconds"]
    sys.path.insert(0, str(METRIC_PATH.parent))
    import official_metric as metric

    assert Path(metric.__file__).resolve() == METRIC_PATH.resolve()
    completed = {result[0] for result in results}
    started = time.monotonic()
    row_count = 0
    seen_sizes = set()
    previous_size = 0

    def record(complete=False):
        ordered = sorted(results, key=lambda result: result[0])
        missing = sorted(set(range(1, 1001)) - seen_sizes) if complete else []
        return {
            "file": str(input_path),
            "sha256": input_sha256,
            "metric_sha256": METRIC_SHA256,
            "score": sum(result[1] for result in ordered)
            + sum(metric.infeasible_score(n) for n in missing),
            "rows": row_count,
            "submitted_puzzles": len(seen_sizes),
            "missing_puzzles": len(missing) if complete else None,
            "rejected": [result for result in ordered if result[3] != "ok"],
            "results": ordered,
            "complete": complete,
            "elapsed_seconds": previous_seconds + time.monotonic() - started,
            "peak_rss_mib": peak_memory_mib(),
            "workers": 1,
            "mode": "single process; one puzzle's decimal rows in memory",
            "runtime_versions": {
                name: importlib.metadata.version(name)
                for name in ("numpy", "pandas", "mpmath")
            },
        }

    with input_path.open(newline="") as stream:
        reader = csv.DictReader(stream)
        assert {"n", "square_id", "x", "y", "angle"} <= set(reader.fieldnames or [])
        for size, group in itertools.groupby(reader, key=lambda row: int(row["n"])):
            assert (
                1 <= size <= 1000 and size > previous_size
            ), "CSV groups must be ordered by n"
            previous_size = size
            rows = list(group)
            row_count += len(rows)
            seen_sizes.add(size)
            if size not in completed:
                results.append(metric._evaluate_puzzle_worker((size, rows)))
            del rows
            if len(seen_sizes) % 25 == 0:
                save_report(output_path, record())
            if peak_memory_mib() > memory_limit_mib:
                save_report(output_path, record())
                raise MemoryError(
                    "Own validator exceeded its memory budget; partial certificate saved"
                )
            if len(seen_sizes) % 100 == 0:
                print(
                    f"Validated {len(seen_sizes)} groups; "
                    f"{time.monotonic() - started:.1f}s; "
                    f"peak RSS {peak_memory_mib():.1f}MiB",
                    flush=True,
                )
    assert completed <= seen_sizes
    report = record(complete=True)
    save_report(output_path, report)
    print(
        json.dumps(
            {key: value for key, value in report.items() if key != "results"}, indent=2
        )
    )
    return report


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("input", type=Path)
    parser.add_argument("output", type=Path)
    parser.add_argument("--memory-limit-mib", type=float, default=1536)
    arguments = parser.parse_args()
    result = validate(arguments.input, arguments.output, arguments.memory_limit_mib)
    raise SystemExit(0 if not result["rejected"] else 1)


### `reference_sources/official_metric.py`

Official competition metric V20, archived directly from metric/squares-in-squares and retained verbatim.

In [ ]:
%%writefile reference_sources/official_metric.py
"""Metric for the "Packing N Unit Squares into the Smallest Square" competition.

Participants submit packings of n unit squares (n=1..1000) inside a bounding
square.  Each square is specified by its center (x, y) and rotation angle.
Coordinates must be plain decimal literals, optionally in scientific notation
(e.g. ``2.70710678118654752440`` or ``1.5e-3``).  Symbolic / closed-form
expressions are not accepted -- submit their decimal expansion instead, at as
many digits as you like.

Scoring
-------
For each puzzle n the bounding side length L is measured
translation-invariantly as ``max(max_x - min_x, max_y - min_y)`` over all
corners.  The per-puzzle score is the excess over the area lower bound

    score(n) = L / sqrt(n) - 1

which is 0.0 exactly when the packing achieves the area lower bound (only
possible for perfect squares packed on the grid) and grows above 0 as the
bounding square wastes area.  **Lower is better**; total score is the sum
across n=1..N.

+-------------------+---------------------------------------------------------+
| Condition         | Score                                                   |
+===================+=========================================================+
| feasible packing  | L / sqrt(n) - 1     (>= 0 always)                       |
+-------------------+---------------------------------------------------------+
| infeasible/invalid| L_0 / sqrt(n) - 1   (grid ceiling — no worse than       |
|                   |                      submitting the trivial grid)       |
+-------------------+---------------------------------------------------------+

For reference: ``L_B = sqrt(n)`` is the area lower bound and
``L_0 = ceil(sqrt(n))`` is the trivial grid packing.  The best-known side
length plays no part -- beating a record just yields a smaller excess -- so
this metric reads no baseline values at all.

Feasibility is checked with interval arithmetic (mpmath) at escalating
precision to certify that no two squares overlap.

Kaggle entry point
------------------
``score(solution, submission, row_id_column_name)`` — called by the Kaggle
backend.  ``solution`` is a DataFrame loaded from ``solution.csv``; only its
``n`` column is read, to fix which puzzles are scored (and which absent ones
are charged the grid ceiling).  ``submission`` is the participant CSV with
columns ``n, square_id, x, y, angle``.

As a side effect it writes ``submission_details.csv``, one row per puzzle with
columns ``n`` and ``L``::

    n,L
    1,1.00000000000000000000000000000000000000000000000000
    2,2.00000000000000000000000000000000000000000000000000
    ...

``L`` is the measured bounding side for puzzle n -- not the per-puzzle score --
written with exactly 50 digits after the decimal point.  Those digits are
significant: L is measured in ``L_DPS``-digit mpmath arithmetic straight from
the submitted decimal strings, not from a double.  Puzzles with no accepted L
(absent from the submission, unparseable, failing column validation, or
rejected as infeasible) are written as zero.
"""

import contextlib
import csv
import io
import math
import multiprocessing
import re
from collections import defaultdict
from collections.abc import Callable, Sequence
from dataclasses import dataclass, field
from fractions import Fraction
from pathlib import Path

import mpmath
import pandas as pd

# ---------------------------------------------------------------------------
# interval
# ---------------------------------------------------------------------------

IVal = mpmath.ctx_iv.ivmpf

# Working precision for the (non-interval) measurement of the bounding side L,
# and the number of digits after the decimal point written for it in
# submission_details.csv.  The margin between the two keeps every written digit
# correct: the only rounding is in the trig of the corner computation, tens of
# digits below the last digit reported.
L_DPS = 60
L_DECIMALS = 50


@contextlib.contextmanager
def working_dps(dps: int):
    """Run a block with both mpmath contexts at ``dps`` digits, then restore them.

    ``mpmath.iv`` carries its own precision, independent of ``mpmath.mp``:
    setting only ``mpmath.mp.dps`` leaves every interval operation at the
    default 53 bits, which caps what a contact can be certified to at ~1e-13 no
    matter how far the escalation sequence climbs.  Both contexts are set here
    because the interval work is interleaved with plain ``mp`` arithmetic.

    >>> with working_dps(500):
    ...     width = mpmath.mpf(mpmath.iv.mpf('0.1').delta)
    >>> width < mpmath.mpf(10) ** -500
    True
    """
    old_iv_dps = mpmath.iv.dps
    old_mp_dps = mpmath.mp.dps
    try:
        mpmath.iv.dps = dps
        mpmath.mp.dps = dps
        yield
    finally:
        mpmath.iv.dps = old_iv_dps
        mpmath.mp.dps = old_mp_dps


def make_interval(value: str, dps: int = 50) -> IVal:
    with working_dps(dps):
        return mpmath.iv.mpf(value)


def iv_cos(x: IVal) -> IVal:
    return mpmath.iv.cos(x)


def iv_sin(x: IVal) -> IVal:
    return mpmath.iv.sin(x)


_ESCALATION_SEQUENCE = [50, 100, 200, 500]


def escalate(
    fn: Callable[[int], bool | None],
    max_dps: int = 500,
    start_dps: int = 50,
) -> bool | None:
    for dps in _ESCALATION_SEQUENCE:
        if dps < start_dps:
            continue
        if dps > max_dps:
            break
        result = fn(dps)
        if result is not None:
            return result
    return None


def iv_le(a: IVal, b: IVal) -> bool | None:
    """Certified less-or-equal: True if a <= b, False if a > b, None if undecided.

    Comparing two overlapping intervals is indeterminate.  mpmath 1.3 reports
    that as None, 1.4 raises ValueError; both mean "undecided" here.
    """
    try:
        return a <= b
    except ValueError:
        return None


def iv_min(values: Sequence[IVal]) -> IVal:
    """Interval extension of ``min``: the tightest interval containing the true minimum.

    Comparison operators are useless for this -- two overlapping intervals have
    no determinate order -- so take it endpoint-wise instead.  min of the lower
    bounds is a lower bound on the true min, and min of the upper bounds is an
    upper bound on it, whichever element turns out to be smallest.
    """
    return mpmath.iv.mpf([min(v.a for v in values), min(v.b for v in values)])


def iv_max(values: Sequence[IVal]) -> IVal:
    """Interval extension of ``max``. See :func:`iv_min`."""
    return mpmath.iv.mpf([max(v.a for v in values), max(v.b for v in values)])


# ---------------------------------------------------------------------------
# parser
# ---------------------------------------------------------------------------

# Bounds the cost of parsing one coordinate.  Far above any useful precision
# (L is reported to 50 decimals), so no honest submission comes near it.
_MAX_LITERAL_CHARS = 100

# Coordinate, angle, and sweep bounds.  For n <= 1000 unit squares, valid
# packings have side L <= 32.  Bounding magnitudes prevents float64/mpmath
# discrepancies and precision loss under extreme translation or rotation.
_MAX_COORD = 1e4
_MAX_ANGLE = 100.0
_AABB_PAD = 1e-7

_DECIMAL_RE = re.compile(r'^[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?$')


def _is_decimal(s: str) -> bool:
    return len(s) <= _MAX_LITERAL_CHARS and bool(_DECIMAL_RE.match(s.strip()))


def _check_decimal(s: str) -> str:
    """Return the stripped literal, or raise if it is not a plain decimal.

    >>> _check_decimal(" 2.70710678118654752440 ")
    '2.70710678118654752440'
    >>> _check_decimal("1.5e-3")
    '1.5e-3'
    >>> _check_decimal("1 + sqrt(2)")
    Traceback (most recent call last):
        ...
    ValueError: not a decimal number: '1 + sqrt(2)'
    """
    s = s.strip()
    if not _is_decimal(s):
        shown = s if len(s) <= 40 else s[:40] + '...'
        raise ValueError(f'not a decimal number: {shown!r}')
    return s


def parse_value(s: str, dps: int = 50) -> IVal:
    """Parse a decimal literal into an enclosing interval at ``dps`` digits."""
    return make_interval(_check_decimal(s), dps)


def parse_value_mp(s: str, dps: int = L_DPS) -> mpmath.mpf:
    """Parse a decimal literal into an ``mpf`` at ``dps`` digits.

    >>> mpmath.nstr(parse_value_mp("2.70710678118654752440"), 21, strip_zeros=False)
    '2.70710678118654752440'
    """
    s = _check_decimal(s)
    old_dps = mpmath.mp.dps
    try:
        mpmath.mp.dps = dps
        return mpmath.mpf(s)
    finally:
        mpmath.mp.dps = old_dps


def parse_value_float(s: str) -> float:
    """Parse a decimal literal to float.

    Values too large for a double are rejected rather than becoming ``inf``,
    which would poison every downstream geometry check:

    >>> parse_value_float("3.5")
    3.5
    >>> parse_value_float("-2.5")
    -2.5
    >>> parse_value_float("1.5e-3")
    0.0015
    >>> parse_value_float("1e400")
    Traceback (most recent call last):
        ...
    ValueError: value out of range for a double: '1e400'
    """
    s = _check_decimal(s)
    value = float(s)
    if not math.isfinite(value):
        raise ValueError(f'value out of range for a double: {s!r}')
    return value


# ---------------------------------------------------------------------------
# scoring
# ---------------------------------------------------------------------------


def score_puzzle(n: int, s_certified: float) -> float:
    """Score a single puzzle: L / sqrt(n) - 1 (lower is better).

    Result is >= 0 for any feasible packing (equality only for perfect-square
    grid packings).  The best-known side length s* plays no part: beating a
    record simply yields a smaller excess.

    >>> score_puzzle(4, 2.0)
    0.0
    >>> round(score_puzzle(5, 2.707), 6)
    0.210607
    >>> score_puzzle(4, 4.0)
    1.0
    """
    L_B = math.sqrt(n)
    return s_certified / L_B - 1.0


def infeasible_score(n: int) -> float:
    """Penalty score for an infeasible / invalid submission for puzzle n.

    Equal to submitting the trivial grid packing: L_0 / sqrt(n) - 1.
    """
    return math.ceil(math.sqrt(n)) / math.sqrt(n) - 1.0


# ---------------------------------------------------------------------------
# geometry
# ---------------------------------------------------------------------------


@dataclass
class Square:
    x: float
    y: float
    angle: float
    x_str: str = ''
    y_str: str = ''
    angle_str: str = ''
    _corners_float: list[tuple[float, float]] = field(default_factory=list, repr=False)

    def corners_float(self) -> list[tuple[float, float]]:
        if not self._corners_float:
            self._corners_float = compute_corners_float(self.x, self.y, self.angle)
        return self._corners_float


def compute_corners_float(cx: float, cy: float, angle: float) -> list[tuple[float, float]]:
    """Return the four corners of a unit square centered at (cx, cy) rotated by angle.

    >>> corners = compute_corners_float(0.5, 0.5, 0.0)
    >>> [(round(x, 1), round(y, 1)) for x, y in corners]
    [(1.0, 1.0), (0.0, 1.0), (0.0, 0.0), (1.0, 0.0)]
    """
    c = math.cos(angle)
    s = math.sin(angle)
    h = 0.5
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx + dx1, cy + dy1),
        (cx + dx2, cy + dy2),
        (cx - dx1, cy - dy1),
        (cx - dx2, cy - dy2),
    ]


def compute_corners_mp(cx: mpmath.mpf, cy: mpmath.mpf, angle: mpmath.mpf) -> list[tuple[mpmath.mpf, mpmath.mpf]]:
    """``compute_corners_float`` at the ambient ``mpmath.mp`` precision."""
    c = mpmath.cos(angle)
    s = mpmath.sin(angle)
    h = mpmath.mpf(1) / 2
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx + dx1, cy + dy1),
        (cx + dx2, cy + dy2),
        (cx - dx1, cy - dy1),
        (cx - dx2, cy - dy2),
    ]


def measure_side_mp(squares: list['Square'], dps: int = L_DPS) -> mpmath.mpf:
    """Bounding side ``max(max_x - min_x, max_y - min_y)`` at ``dps`` digits.

    Measured from the submitted decimal strings translated relative to Square 0,
    so coordinates carry high precision without loss due to global translation.

    >>> unit = Square(0.5, 0.5, 0.0, x_str="0.5", y_str="0.5", angle_str="0")
    >>> measure_side_mp([unit])
    mpf('1.0')
    """
    old_dps = mpmath.mp.dps
    try:
        mpmath.mp.dps = dps
        xs: list[mpmath.mpf] = []
        ys: list[mpmath.mpf] = []
        x0 = parse_value_mp(squares[0].x_str, dps) if squares else mpmath.mpf(0)
        y0 = parse_value_mp(squares[0].y_str, dps) if squares else mpmath.mpf(0)
        for sq in squares:
            corners = compute_corners_mp(
                parse_value_mp(sq.x_str, dps) - x0,
                parse_value_mp(sq.y_str, dps) - y0,
                parse_value_mp(sq.angle_str, dps),
            )
            for x, y in corners:
                xs.append(x)
                ys.append(y)
        return max(max(xs) - min(xs), max(ys) - min(ys))
    finally:
        mpmath.mp.dps = old_dps


def compute_corners_interval(cx_iv: IVal, cy_iv: IVal, angle_iv: IVal) -> list[tuple[IVal, IVal]]:
    c = iv_cos(angle_iv)
    s = iv_sin(angle_iv)
    h = mpmath.iv.mpf('0.5')
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx_iv + dx1, cy_iv + dy1),
        (cx_iv + dx2, cy_iv + dy2),
        (cx_iv - dx1, cy_iv - dy1),
        (cx_iv - dx2, cy_iv - dy2),
    ]


def compute_aabb(
    corners: list[tuple[float, float]],
    pad: float = _AABB_PAD,
) -> tuple[float, float, float, float]:
    xs = [c[0] for c in corners]
    ys = [c[1] for c in corners]
    return (min(xs) - pad, min(ys) - pad, max(xs) + pad, max(ys) + pad)


def aabb_overlap(
    a: tuple[float, float, float, float],
    b: tuple[float, float, float, float],
) -> bool:
    return not (a[2] < b[0] or b[2] < a[0] or a[3] < b[1] or b[3] < a[1])


def _project_corners_float(corners: list[tuple[float, float]], ax: float, ay: float) -> tuple[float, float]:
    projs = [ax * cx + ay * cy for cx, cy in corners]
    return (min(projs), max(projs))


def sat_check_float(sq1: Square, sq2: Square, undecided_tol: float = 1e-8) -> str:
    """Check overlap between two unit squares using the Separating Axis Theorem.

    Returns ``"separated"``, ``"overlapping"``, or ``"undecided"``.

    >>> sat_check_float(Square(0.5, 0.5, 0.0), Square(5.0, 5.0, 0.0))
    'separated'
    >>> sat_check_float(Square(0.5, 0.5, 0.0), Square(0.6, 0.6, 0.0))
    'overlapping'
    """
    c1 = sq1.corners_float()
    c2 = sq2.corners_float()

    axes = _get_axes_float(sq1.angle, sq2.angle)

    min_overlap = float('inf')
    for ax, ay in axes:
        min1, max1 = _project_corners_float(c1, ax, ay)
        min2, max2 = _project_corners_float(c2, ax, ay)
        overlap = min(max1, max2) - max(min1, min2)
        if overlap < -undecided_tol:
            return 'separated'
        min_overlap = min(min_overlap, overlap)

    if min_overlap > undecided_tol:
        return 'overlapping'
    return 'undecided'


def _get_axes_float(angle1: float, angle2: float) -> list[tuple[float, float]]:
    c1, s1 = math.cos(angle1), math.sin(angle1)
    c2, s2 = math.cos(angle2), math.sin(angle2)
    return [(c1, s1), (-s1, c1), (c2, s2), (-s2, c2)]


def sat_check_interval(
    sq1_corners: list[tuple[IVal, IVal]],
    sq2_corners: list[tuple[IVal, IVal]],
    angle1_iv: IVal,
    angle2_iv: IVal,
) -> bool | None:
    c1_iv, s1_iv = iv_cos(angle1_iv), iv_sin(angle1_iv)
    c2_iv, s2_iv = iv_cos(angle2_iv), iv_sin(angle2_iv)
    axes = [(c1_iv, s1_iv), (-s1_iv, c1_iv), (c2_iv, s2_iv), (-s2_iv, c2_iv)]

    has_undecided = False
    for ax, ay in axes:
        projs1 = [ax * cx + ay * cy for cx, cy in sq1_corners]
        projs2 = [ax * cx + ay * cy for cx, cy in sq2_corners]

        # Endpoint-wise, via iv_min/iv_max: ordering two overlapping intervals
        # with < is indeterminate, and skipping the projections that compare
        # indeterminately would narrow the span -- which shrinks the overlap and
        # biases the test towards wrongly certifying a pair as separated.
        min1, max1 = iv_min(projs1), iv_max(projs1)
        min2, max2 = iv_min(projs2), iv_max(projs2)

        overlap = iv_min([max1, max2]) - iv_max([min1, min2])
        le_result = iv_le(overlap, mpmath.iv.mpf('0'))
        if le_result is True:
            return True
        if le_result is None:
            has_undecided = True

    if has_undecided:
        return None
    return False


def sort_and_sweep(squares: list[Square]) -> list[tuple[int, int]]:
    n = len(squares)
    if n <= 1:
        return []

    aabbs = []
    for i, sq in enumerate(squares):
        corners = sq.corners_float()
        aabb = compute_aabb(corners)
        aabbs.append((i, aabb))

    aabbs.sort(key=lambda x: x[1][0])

    pairs = []
    for i_idx in range(n):
        i, aabb_i = aabbs[i_idx]
        for j_idx in range(i_idx + 1, n):
            j, aabb_j = aabbs[j_idx]
            if aabb_j[0] > aabb_i[2]:
                break
            if aabb_overlap(aabb_i, aabb_j):
                pairs.append((i, j) if i < j else (j, i))

    return pairs


def verify_puzzle(
    squares: list[Square],
) -> tuple[bool, str]:
    """Check that no two squares overlap, using float SAT then interval arithmetic.

    >>> s1 = Square(0.5, 0.5, 0.0, x_str="0.5", y_str="0.5", angle_str="0")
    >>> s2 = Square(1.5, 0.5, 0.0, x_str="1.5", y_str="0.5", angle_str="0")
    >>> verify_puzzle([s1, s2])
    (True, 'feasible')
    >>> verify_puzzle([Square(0.5, 0.5, 0.0), Square(0.6, 0.6, 0.0)])
    (False, 'Squares 0 and 1 overlap (float check)')
    """
    pairs = sort_and_sweep(squares)
    undecided_pairs = []
    for i, j in pairs:
        result = sat_check_float(squares[i], squares[j])
        if result == 'overlapping':
            return (False, f'Squares {i} and {j} overlap (float check)')
        if result == 'undecided':
            undecided_pairs.append((i, j))

    if undecided_pairs:

        def check_overlap_certified(dps: int) -> bool | None:
            # The whole pair loop runs inside the precision context: parsing at dps
            # digits is not enough, since every later interval operation widens its
            # result to whatever precision mpmath.iv is set to at the time.
            with working_dps(dps):
                still_undecided = []
                for i, j in undecided_pairs:
                    sq_i, sq_j = squares[i], squares[j]
                    cx_i = parse_value(sq_i.x_str, dps)
                    cy_i = parse_value(sq_i.y_str, dps)
                    ai = parse_value(sq_i.angle_str, dps)
                    cx_j = parse_value(sq_j.x_str, dps)
                    cy_j = parse_value(sq_j.y_str, dps)
                    aj = parse_value(sq_j.angle_str, dps)
                    corners_i = compute_corners_interval(cx_i, cy_i, ai)
                    corners_j = compute_corners_interval(cx_j, cy_j, aj)
                    result = sat_check_interval(corners_i, corners_j, ai, aj)
                    if result is False:
                        return False
                    if result is None:
                        still_undecided.append((i, j))
                if still_undecided:
                    return None
                return True

        result = escalate(check_overlap_certified)
        if result is False:
            return (False, 'Squares overlap (certified)')
        if result is None:
            return (False, 'Overlap undecidable at max precision')

    return (True, 'feasible')


# ---------------------------------------------------------------------------
# main / Kaggle entry point
# ---------------------------------------------------------------------------


class ParticipantVisibleError(Exception):
    pass


MAX_N = 1000
KAGGLE_CORES = 4
MIN_PUZZLES_FOR_MP = 4


def _parse_submission_csv(text: str) -> dict[int, list[dict]]:
    reader = csv.DictReader(io.StringIO(text))
    required_cols = {'n', 'square_id', 'x', 'y', 'angle'}
    if not required_cols.issubset(set(reader.fieldnames or [])):
        missing = required_cols - set(reader.fieldnames or [])
        raise ParticipantVisibleError(f'Missing columns: {missing}')

    puzzles: dict[int, list[dict]] = defaultdict(list)
    for row_num, row in enumerate(reader, start=2):
        try:
            n = int(row['n'])
        except (ValueError, TypeError):
            raise ParticipantVisibleError(f'Row {row_num}: invalid n value: {row["n"]!r}')
        if n < 1 or n > MAX_N:
            raise ParticipantVisibleError(f'Row {row_num}: n={n} out of range [1, {MAX_N}]')
        puzzles[n].append(row)

    return dict(puzzles)


def _validate_puzzle(n: int, rows: list[dict]) -> str | None:
    if len(rows) != n:
        return f'n={n}: expected {n} rows, got {len(rows)}'

    square_ids = set()
    for row in rows:
        try:
            sid = int(row['square_id'])
        except (ValueError, TypeError):
            return f'n={n}: invalid square_id: {row["square_id"]!r}'
        square_ids.add(sid)

    expected_ids = set(range(n))
    if square_ids != expected_ids:
        return f'n={n}: square_ids should be 0..{n - 1}, got {sorted(square_ids)}'

    return None


def _build_squares(n: int, rows: list[dict]) -> tuple[list[Square], mpmath.mpf]:
    rows_by_id = {int(r['square_id']): r for r in rows}

    raw_coords = []
    for sid in range(n):
        row = rows_by_id[sid]
        x_str = row['x'].strip()
        y_str = row['y'].strip()
        angle_str = row['angle'].strip()

        try:
            x_f = parse_value_float(x_str)
            y_f = parse_value_float(y_str)
            angle_f = parse_value_float(angle_str)
        except ValueError as e:
            raise ParticipantVisibleError(f'n={n}, square {sid}: cannot parse coordinates: {e}')

        if abs(x_f) > _MAX_COORD or abs(y_f) > _MAX_COORD:
            raise ParticipantVisibleError(f'n={n}, square {sid}: coordinate out of bounds (|x|, |y| must be <= {_MAX_COORD:g})')
        if abs(angle_f) > _MAX_ANGLE:
            raise ParticipantVisibleError(f'n={n}, square {sid}: angle out of bounds (|angle| must be <= {_MAX_ANGLE:g} radians)')

        raw_coords.append((sid, x_str, y_str, angle_str))

    x0_mp = parse_value_mp(raw_coords[0][1], L_DPS)
    y0_mp = parse_value_mp(raw_coords[0][2], L_DPS)
    two_pi = 2 * mpmath.pi

    squares = []
    for sid, x_str, y_str, angle_str in raw_coords:
        x_mp = parse_value_mp(x_str, L_DPS)
        y_mp = parse_value_mp(y_str, L_DPS)
        angle_mp = parse_value_mp(angle_str, L_DPS)

        dx_f = float(x_mp - x0_mp)
        dy_f = float(y_mp - y0_mp)
        angle_reduced_f = float(angle_mp % two_pi)

        squares.append(
            Square(
                x=dx_f,
                y=dy_f,
                angle=angle_reduced_f,
                x_str=x_str,
                y_str=y_str,
                angle_str=angle_str,
            )
        )

    return squares, measure_side_mp(squares)


def evaluate_single_puzzle(n: int, rows: list[dict]) -> tuple[int, float, str | None, str]:
    """Evaluate one puzzle, returning ``(n, score, L, reason)``.

    ``L`` is decimal text with ``L_DECIMALS`` digits after the point, and is
    reported only for a packing that passed the feasibility check: an
    unparseable, malformed, or overlapping puzzle gets None, so no L is
    published for a packing the metric rejected.  Text rather than a float
    because a double cannot hold those digits; the score itself is a double,
    taken from the same high-precision measurement.
    """
    error = _validate_puzzle(n, rows)
    if error:
        return (n, infeasible_score(n), None, error)

    try:
        squares, s_mp = _build_squares(n, rows)
    except ParticipantVisibleError as e:
        return (n, infeasible_score(n), None, str(e))

    feasible, reason = verify_puzzle(squares)
    if not feasible:
        return (n, infeasible_score(n), None, f'infeasible: {reason}')

    L_B = math.sqrt(n)
    if float(s_mp) < L_B - 1e-7:
        return (
            n,
            infeasible_score(n),
            None,
            f'infeasible: L={float(s_mp):.6f} is below area lower bound sqrt({n})={L_B:.6f}',
        )

    puzzle_score = max(0.0, score_puzzle(n, float(s_mp)))
    return (n, puzzle_score, _format_detail(s_mp), 'ok')


def _evaluate_puzzle_worker(args: tuple) -> tuple[int, float, str | None, str]:
    n, rows = args
    try:
        return evaluate_single_puzzle(n, rows)
    except Exception as e:  # noqa: BLE001 - one bad puzzle must not kill the pool
        return (n, infeasible_score(n), None, f'worker error: {e}')


def _evaluate_puzzles_parallel(
    puzzles: dict[int, list[dict]],
    puzzle_ns: set[int],
) -> tuple[float, list[tuple[int, float, str | None, str]]]:
    """Score every n in ``puzzle_ns``; rows for any other n are ignored."""
    work = [(n, rows) for n, rows in sorted(puzzles.items()) if n in puzzle_ns]

    if not work:
        results: list[tuple[int, float, str | None, str]] = []
    elif len(work) < MIN_PUZZLES_FOR_MP:
        results = [evaluate_single_puzzle(n, rows) for n, rows in work]
    else:
        with multiprocessing.Pool(processes=KAGGLE_CORES) as pool:
            results = pool.map(_evaluate_puzzle_worker, work)

    # Any scored n missing from the submission gets the grid-ceiling penalty
    # (lower-is-better metric — otherwise omitting a puzzle would improve the
    # total).
    seen = {n for n, _, _, _ in results}
    for n in puzzle_ns:
        if n not in seen:
            results.append((n, infeasible_score(n), None, 'missing from submission'))
    results.sort(key=lambda r: r[0])

    total = sum(puzzle_score for _, puzzle_score, _, _ in results)
    return total, results


def _format_detail(value: mpmath.mpf) -> str:
    """Decimal text for one per-puzzle L: exactly ``L_DECIMALS`` digits after the point.

    An ``mpf`` is a binary fraction (``man * 2**exp``), so it is converted
    exactly to a ``Fraction`` and rounded once, half-to-even, at the last
    written digit.  ``mpmath.nstr`` counts significant digits rather than
    decimal places, and ``format(mpf, '.50f')`` needs mpmath >= 1.4, so neither
    is usable here.

    >>> _format_detail(mpmath.mpf(2))[:12]
    '2.0000000000'
    >>> len(_format_detail(mpmath.mpf(2)).split('.')[1])
    50
    >>> _format_detail(parse_value_mp('1.41421356237309504880168872420969807856967187537695'))
    '1.41421356237309504880168872420969807856967187537695'
    """
    exact = Fraction(value.man) * Fraction(2) ** value.exp
    scale = 10**L_DECIMALS
    units = round(exact * scale)
    sign = '-' if units < 0 else ''
    whole, frac = divmod(abs(units), scale)
    return f'{sign}{whole}.{frac:0{L_DECIMALS}d}'


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
) -> float:
    """Kaggle entry point. Score a submission against the solution's puzzle set.

    ``solution`` is read for one thing only: the set of n to score.  Everything
    else about a puzzle -- best-known side length, proven-optimal flag, source
    -- is irrelevant to ``L / sqrt(n) - 1``.

    >>> import pandas as pd
    >>> sol = pd.DataFrame({"id": [0], "n": [1]})
    >>> sub = pd.DataFrame({"id": [0], "n": [1], "square_id": [0],
    ...     "x": ["0.5"], "y": ["0.5"], "angle": ["0"]})
    >>> score(sol, sub, "id")  # single unit square, L=1, L/sqrt(1) - 1 = 0.0
    0.0
    """
    solution = solution.drop(columns=[row_id_column_name])
    submission = submission.drop(columns=[row_id_column_name])

    puzzle_ns = {int(n) for n in solution['n']}

    text = submission.to_csv(index=False)
    puzzles = _parse_submission_csv(text)
    total, results = _evaluate_puzzles_parallel(puzzles, puzzle_ns)

    # L is the measured bounding side for puzzle n -- NOT the per-puzzle score
    # L/sqrt(n) - 1.  Puzzles with no accepted L (missing from the submission,
    # unparseable, failing column validation, or rejected as infeasible) keep
    # the historical 0 placeholder: an L measured off an overlapping packing is
    # not a real packing of n squares, and downstream harvesting treats every
    # published L as attainable.  Values arrive here already rendered as decimal
    # text with L_DECIMALS digits after the point; unlike a JSON float they are
    # not re-serialized downstream, so all 50 digits reach the stored file
    # verbatim.
    result_map = {n: L_text for n, _, L_text, _ in results if L_text is not None}
    zero = _format_detail(mpmath.mpf(0))

    output_dir = Path('/kaggle/working')
    if not output_dir.is_dir():
        output_dir = Path('.')
    with open(output_dir / 'submission_details.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['n', 'L'])
        for n in range(1, MAX_N + 1):
            writer.writerow([n, result_map.get(n, zero)])

    return float(total)

## 7. Certify both exact exported files sequentially

Raw and ordinary pandas read/write files are independently checked with the
unmodified official metric. These fresh host reports, CSV hash and CPU check
are saved as outputs. A local certificate is distinct from a competition Complete.
Run All performs no competition submission.


In [ ]:
import mpmath

if not hasattr(mpmath.mpf(1), "man"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "mpmath==1.4.1"], check=True
    )
metric_path = Path("reference_sources/official_metric.py")
metric_bytes = metric_path.read_bytes()
if hashlib.sha256(metric_bytes).hexdigest() != OFFICIAL_METRIC_SHA256:
    assert metric_bytes.endswith(b"\n")
    assert hashlib.sha256(metric_bytes[:-1]).hexdigest() == OFFICIAL_METRIC_SHA256
    metric_path.write_bytes(metric_bytes[:-1])
assert hashlib.sha256(metric_path.read_bytes()).hexdigest() == OFFICIAL_METRIC_SHA256
subprocess.run(
    [sys.executable, "check_portable_backends.py", "--backend", "cpu"], check=True
)
if REPLAY_MOVEMENT_STUDY:
    subprocess.run(
        [
            sys.executable,
            "replay_movement_controls.py",
            "--input",
            "movement_starts.npz",
            "--output",
            "replayed_movement_study.json",
        ],
        check=True,
    )
if RUN_SEARCH:
    subprocess.run(
        [
            sys.executable,
            "portable_experiment.py",
            "--input",
            "submission.csv",
            "--output",
            "recomputed_submission.csv",
            "--backend",
            BACKEND,
            "--sizes",
            *[str(n) for n in SEARCH_SIZES],
            "--batch",
            str(SEARCH_BATCH),
            "--steps",
            str(SEARCH_STEPS),
        ],
        check=True,
    )
    shutil.copyfile("recomputed_submission.csv", "submission.csv")
pd.read_csv("submission.csv").to_csv("float64_roundtrip.csv", index=False)
for csv_path, report_path in [
    ("submission.csv", "validation.json"),
    ("float64_roundtrip.csv", "roundtrip_validation.json"),
]:
    subprocess.run(
        [
            sys.executable,
            "validate_submission.py",
            csv_path,
            report_path,
            "--memory-limit-mib",
            "1536",
        ],
        check=True,
    )
raw = json.loads(Path("validation.json").read_text())
roundtrip = json.loads(Path("roundtrip_validation.json").read_text())
assert raw["complete"] and roundtrip["complete"]
assert not raw["rejected"] and (not roundtrip["rejected"])
assert abs(raw["score"] - roundtrip["score"]) < 1e-10
assert raw["score"] <= EXPECTED_SCORE + 1e-10
if not RUN_SEARCH:
    assert raw["sha256"] == EXPECTED_CSV_SHA256
    assert abs(raw["score"] - EXPECTED_SCORE) < 1e-10
print(f"Freshly verified score: {raw['score']:.12f}")
print("Rejected packings:", len(raw["rejected"]))
print("CSV SHA256:", raw["sha256"])
print(
    "Fresh raw/pandas process peak RSS MiB:",
    raw["peak_rss_mib"],
    roundtrip["peak_rss_mib"],
)
display(FileLink("submission.csv"))
assert max(raw["score"], roundtrip["score"]) < 22.249885341497446


## Sources, devices and limits

The 22 matched and longer CPU controls measure finite solver budgets, with only n=991 retained after search. The final longer arm used seed 55, initial centre trust 1e-4 and the unscaled LP. It accepted 39 steps before the wall-clock cap. The frozen CSV uses the certified coordinates, independent of optional replay timing.

The existing CPU, CUDA and MPS code remains visible. The fresh Run All uses CPU for the functional check and certification. The real T4 smoke test is historical evidence for unchanged CUDA code. MPS is implemented and hardware untested.
